In [1]:
# Controlled augmentation-ablation run selector
RUN_CLEAN_AUG_ABLATION = True
AUG_ABLATION_CONDITION = "synthetic_only"

AUG_ABLATION_CONDITIONS = {
    "no_aug": {
        "use_synthetic_multiparty": False,
        "use_diffusion_aug": False,
    },
    "synthetic_only": {
        "use_synthetic_multiparty": True,
        "use_diffusion_aug": False,
    },
    "diffusion_only": {
        "use_synthetic_multiparty": False,
        "use_diffusion_aug": True,
    },
    "full": {
        "use_synthetic_multiparty": True,
        "use_diffusion_aug": True,
    },
}

if AUG_ABLATION_CONDITION not in AUG_ABLATION_CONDITIONS:
    raise ValueError(
        "Unknown AUG_ABLATION_CONDITION. Choose one of: "
        f"{sorted(AUG_ABLATION_CONDITIONS)}"
    )

if RUN_CLEAN_AUG_ABLATION:
    selected = AUG_ABLATION_CONDITIONS[AUG_ABLATION_CONDITION]

    USE_SYNTH_MULTIPARTY = bool(
        selected["use_synthetic_multiparty"]
    )
    USE_DIFFUSION_AUG = bool(
        selected["use_diffusion_aug"]
    )

    PAPER_READY_ABLATION_BASE_VERSION = (
        "18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2"
    )
    RUN_VERSION_OVERRIDE = (
        f"{PAPER_READY_ABLATION_BASE_VERSION}"
        f"__aug_ablation_{AUG_ABLATION_CONDITION}"
    )

    print("CLEAN AUGMENTATION ABLATION ACTIVE")
    print("Condition:", AUG_ABLATION_CONDITION)
    print("RUN_VERSION_OVERRIDE:", RUN_VERSION_OVERRIDE)
    print("USE_SYNTH_MULTIPARTY:", USE_SYNTH_MULTIPARTY)
    print("USE_DIFFUSION_AUG:", USE_DIFFUSION_AUG)
else:
    print("Normal cache-safe notebook mode.")

CLEAN AUGMENTATION ABLATION ACTIVE
Condition: synthetic_only
RUN_VERSION_OVERRIDE: 18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only
USE_SYNTH_MULTIPARTY: True
USE_DIFFUSION_AUG: False


In [2]:
# Global experiment config
from pathlib import Path
import os, json, time, math, random, warnings, shutil, glob, re
warnings.filterwarnings("ignore")

EXPERIMENT_NAME = "vap_rrfap_text_meta_confirmer_v1"
BASE_RUN_VERSION = "18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2"
AUDIT_BASE_ASSET_RUN_VERSION = str(globals().get(
    "AUDIT_BASE_ASSET_RUN_VERSION",
    "18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1",
))
RUN_VERSION = str(globals().get("RUN_VERSION_OVERRIDE", BASE_RUN_VERSION))

USE_WAVLM_SYS1 = bool(globals().get("USE_WAVLM_SYS1", True))
USE_PYANNOTE_DIA = bool(globals().get("USE_PYANNOTE_DIA", True))
USE_ASR_TEXT = bool(globals().get("USE_ASR_TEXT", True))
USE_QWEN3_LLM = bool(globals().get("USE_QWEN3_LLM", True))
USE_ECAPA_SPK = bool(globals().get("USE_ECAPA_SPK", True))
USE_SYNTH_MULTIPARTY = bool(globals().get("USE_SYNTH_MULTIPARTY", True))
USE_DIFFUSION_AUG = bool(globals().get("USE_DIFFUSION_AUG", True))
USE_TEMPORAL_SYS2 = bool(globals().get("USE_TEMPORAL_SYS2", True))
USE_LIGHTGBM_SYS2 = bool(globals().get("USE_LIGHTGBM_SYS2", True))
USE_META_CONFIRMER = bool(globals().get("USE_META_CONFIRMER", True))

FORCE_REBUILD_SPLITS = bool(globals().get("FORCE_REBUILD_SPLITS", False))
FORCE_REBUILD_REAL_EVENTS = bool(globals().get("FORCE_REBUILD_REAL_EVENTS", False))
FORCE_REGENERATE_SYNTH = bool(globals().get("FORCE_REGENERATE_SYNTH", False))
FORCE_REBUILD_AUGMENTED_AUDIO = bool(globals().get("FORCE_REBUILD_AUGMENTED_AUDIO", False))
FORCE_RERUN_ASR = bool(globals().get("FORCE_RERUN_ASR", False))
FORCE_RERUN_PYANNOTE = bool(globals().get("FORCE_RERUN_PYANNOTE", False))
FORCE_RETRAIN_SYS1 = bool(globals().get("FORCE_RETRAIN_SYS1", False))
FORCE_RETRAIN_QWEN3 = bool(globals().get("FORCE_RETRAIN_QWEN3", False))
FORCE_REBUILD_SYS1_CANDIDATES = bool(globals().get("FORCE_REBUILD_SYS1_CANDIDATES", False))
FORCE_REBUILD_SYS2_FEATURES = bool(globals().get("FORCE_REBUILD_SYS2_FEATURES", False))
FORCE_RETRAIN_TEMPORAL_SYS2 = bool(globals().get("FORCE_RETRAIN_TEMPORAL_SYS2", False))
FORCE_RETRAIN_LIGHTGBM = bool(globals().get("FORCE_RETRAIN_LIGHTGBM", False))
FORCE_RETRAIN_LIGHTGBM_BINARY = bool(globals().get("FORCE_RETRAIN_LIGHTGBM_BINARY", FORCE_RETRAIN_LIGHTGBM))
FORCE_RETRAIN_EVENT_RANKER = bool(globals().get("FORCE_RETRAIN_EVENT_RANKER", False))
FORCE_RETRAIN_RR_FAP = bool(globals().get("FORCE_RETRAIN_RR_FAP", False))
FORCE_RETRAIN_META_CONFIRMER = bool(globals().get("FORCE_RETRAIN_META_CONFIRMER", False))
FORCE_RETUNE_VAL = bool(globals().get("FORCE_RETUNE_VAL", False))
FORCE_REBUILD_TEST_PRED = bool(globals().get("FORCE_REBUILD_TEST_PRED", False))

DEBUG_SMALL_RUN = bool(globals().get("DEBUG_SMALL_RUN", False))
DEBUG_MAX_TRAIN_SESSIONS = int(globals().get("DEBUG_MAX_TRAIN_SESSIONS", 12))
DEBUG_MAX_VAL_SESSIONS = int(globals().get("DEBUG_MAX_VAL_SESSIONS", 4))
DEBUG_MAX_TEST_SESSIONS = int(globals().get("DEBUG_MAX_TEST_SESSIONS", 4))

PAPER_READY_PROTOCOL = bool(globals().get("PAPER_READY_PROTOCOL", True))
EVENT_CONTRACT_VERSION = str(globals().get(
    "EVENT_CONTRACT_VERSION",
    "voxconverse_action_aware_immediate_v1",
))

PRIMARY_EVENT_TIME_DEFINITION = str(globals().get(
    "PRIMARY_EVENT_TIME_DEFINITION",
    "action_aware_boundary",
))

PRIMARY_MIN_GAP_S = float(globals().get("PRIMARY_MIN_GAP_S", -1.0))
PRIMARY_MAX_POSITIVE_GAP_S = float(globals().get(
    "PRIMARY_MAX_POSITIVE_GAP_S",
    1.0,
))
ACTION_SHORT_TURN_S = float(globals().get("ACTION_SHORT_TURN_S", 0.50))
ACTION_OVERLAP_MIN_S = float(globals().get("ACTION_OVERLAP_MIN_S", 0.06))

KEEP_LEGACY_EVENT_TABLES = bool(globals().get("KEEP_LEGACY_EVENT_TABLES", True))
REPORT_TRANSITION_INTERVAL_METRICS = bool(globals().get(
    "REPORT_TRANSITION_INTERVAL_METRICS",
    True,
))
REPORT_LEGACY_MIDPOINT_METRICS = bool(globals().get(
    "REPORT_LEGACY_MIDPOINT_METRICS",
    True,
))

EVENT_CONTRACT_SELECTION_BASIS = str(globals().get(
    "EVENT_CONTRACT_SELECTION_BASIS",
    (
        "audit-informed causal task definition fixed before retraining; "
        "no TEST prediction scores or model performance used"
    ),
))
EVENT_CONTRACT_AUDIT_INCLUDED_TEST_ANNOTATIONS = True
EVENT_CONTRACT_TEST_PREDICTIONS_USED_FOR_SELECTION = False
EVENT_CONTRACT_TEST_USED_FOR_SELECTION = (
    EVENT_CONTRACT_TEST_PREDICTIONS_USED_FOR_SELECTION
)

RUN_LEGACY_TEST_INFORMED_DIAGNOSTICS = bool(
    globals().get(
        "RUN_LEGACY_TEST_INFORMED_DIAGNOSTICS",
        False,
    )
)

TOL = float(globals().get("TOL", 0.50))

STRICT_E2E_TOL = float(globals().get("STRICT_E2E_TOL", 0.50))
DATASET_CALIBRATED_E2E_TOL = float(globals().get("DATASET_CALIBRATED_E2E_TOL", 0.70))
RELAXED_E2E_TOL = float(globals().get("RELAXED_E2E_TOL", 1.00))

BRANCH1_E2E_TOLERANCES = list(globals().get(
    "BRANCH1_E2E_TOLERANCES",
    [
        {"name": "strict", "label": "strict_0p50", "tol": STRICT_E2E_TOL, "description": "Strict event matching, ±0.50 s"},
        {"name": "dataset_calibrated", "label": "dataset_calibrated_0p70", "tol": DATASET_CALIBRATED_E2E_TOL, "description": "Dataset-calibrated event matching, ±0.70 s"},
        {"name": "relaxed", "label": "relaxed_1p00", "tol": RELAXED_E2E_TOL, "description": "Relaxed event matching, ±1.00 s"},
    ],
))
GT_GAP_MIN = float(globals().get("GT_GAP_MIN", PRIMARY_MIN_GAP_S))
GT_GAP_MAX = float(globals().get("GT_GAP_MAX", PRIMARY_MAX_POSITIVE_GAP_S))

TARGET_SYS1_RECALL = float(globals().get("TARGET_SYS1_RECALL", 0.90))
PREFERRED_SYS1_RECALL = float(globals().get("PREFERRED_SYS1_RECALL", 0.95))

TARGET_FINAL_PRECISION = float(globals().get("TARGET_FINAL_PRECISION", 0.70))
TARGET_FINAL_RECALL = float(globals().get("TARGET_FINAL_RECALL", 0.70))
TARGET_FINAL_F1 = float(globals().get("TARGET_FINAL_F1", 0.70))

VAL_MIN_PRECISION_FOR_SELECTION = float(globals().get("VAL_MIN_PRECISION_FOR_SELECTION", 0.55))
VAL_MIN_RECALL_FOR_SELECTION = float(globals().get("VAL_MIN_RECALL_FOR_SELECTION", 0.55))

ACTION_NAMES = ["NA", "NTT", "ITT", "BC", "BI"]
ACTION_TO_ID = {a: i for i, a in enumerate(ACTION_NAMES)}
ID_TO_ACTION = {i: a for a, i in ACTION_TO_ID.items()}

SHIFT_ACTIONS = {"NTT", "ITT"}
HOLD_ACTIONS = {"NA", "BC", "BI"}
SHIFT_ACTION_IDS = sorted([ACTION_TO_ID[a] for a in SHIFT_ACTIONS])
HOLD_ACTION_IDS = sorted([ACTION_TO_ID[a] for a in HOLD_ACTIONS])

WAVLM_MODEL_NAME_PRIMARY = "microsoft/wavlm-large"
WAVLM_MODEL_NAME_FALLBACK = "microsoft/wavlm-base-plus"
WAVLM_MODEL_NAME = str(globals().get("WAVLM_MODEL_NAME", WAVLM_MODEL_NAME_PRIMARY))

QWEN3_MODEL_NAME = str(globals().get("QWEN3_MODEL_NAME", "Qwen/Qwen3-1.7B"))
QWEN3_MAX_LENGTH = int(globals().get("QWEN3_MAX_LENGTH", 512))
QWEN3_LORA_R = int(globals().get("QWEN3_LORA_R", 16))
QWEN3_LORA_ALPHA = int(globals().get("QWEN3_LORA_ALPHA", 32))
QWEN3_EPOCHS = int(globals().get("QWEN3_EPOCHS", 1 if DEBUG_SMALL_RUN else 3))
QWEN3_BATCH_SIZE = int(globals().get("QWEN3_BATCH_SIZE", 4))

PYANNOTE_PIPELINE_NAME = str(globals().get("PYANNOTE_PIPELINE_NAME", "pyannote/speaker-diarization-3.1"))
PYANNOTE_SEGMENTATION_NAME = str(globals().get("PYANNOTE_SEGMENTATION_NAME", "pyannote/segmentation-3.0"))
HF_DIARIZERS_VOXCONVERSE = "diarizers-community/voxconverse"

ASR_MODEL_NAME = str(globals().get("ASR_MODEL_NAME", "medium"))
ASR_COMPUTE_TYPE = str(globals().get("ASR_COMPUTE_TYPE", "float16"))

WEIGHT_REAL_CLEAN = float(globals().get("WEIGHT_REAL_CLEAN", 1.00))
WEIGHT_REAL_DIFFUSION = float(globals().get("WEIGHT_REAL_DIFFUSION", 0.50))
WEIGHT_SYNTH_CLEAN = float(globals().get("WEIGHT_SYNTH_CLEAN", 0.50))
WEIGHT_SYNTH_DIFFUSION = float(globals().get("WEIGHT_SYNTH_DIFFUSION", 0.25))

ADD_MIDTURN_HOLD_NEGATIVES = bool(globals().get(
    "ADD_MIDTURN_HOLD_NEGATIVES",
    True,
))
MIDTURN_NEG_TO_BOUNDARY_RATIO = float(globals().get(
    "MIDTURN_NEG_TO_BOUNDARY_RATIO",
    0.50,
))
MIDTURN_NEG_MAX_PER_SESSION = int(globals().get(
    "MIDTURN_NEG_MAX_PER_SESSION",
    40 if not DEBUG_SMALL_RUN else 8,
))
MIDTURN_NEG_PER_SESSION = MIDTURN_NEG_MAX_PER_SESSION
MIDTURN_NEG_MIN_FROM_EVENT_S = float(globals().get(
    "MIDTURN_NEG_MIN_FROM_EVENT_S",
    1.00,
))
WEIGHT_REAL_MIDTURN = float(globals().get("WEIGHT_REAL_MIDTURN", 0.25))

MIDTURN_HOLD_NEGATIVE_RATIO = float(MIDTURN_NEG_TO_BOUNDARY_RATIO)
MIDTURN_HOLD_MAX_PER_SESSION = int(MIDTURN_NEG_MAX_PER_SESSION)
MIDTURN_HOLD_SAMPLE_WEIGHT = float(WEIGHT_REAL_MIDTURN)
REUSE_AUDIT_COMPATIBLE_BASE_ASSETS = bool(
    globals().get(
        "REUSE_AUDIT_COMPATIBLE_BASE_ASSETS",
        True,
    )
)

SYS1_CLASS_BALANCE_MODE = str(globals().get(
    "SYS1_CLASS_BALANCE_MODE",
    "loss_only",
))
SYS1_USE_WEIGHTED_SAMPLER = bool(globals().get(
    "SYS1_USE_WEIGHTED_SAMPLER",
    False,
))
SYS1_USE_ACTION_CLASS_WEIGHTS = bool(globals().get(
    "SYS1_USE_ACTION_CLASS_WEIGHTS",
    True,
))

SYS1_WAVLM_THR_GRID = list(globals().get(
    "SYS1_WAVLM_THR_GRID",
    [0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.18, 0.22, 0.28, 0.35, 0.45, 0.55]
))

SYS1_QWEN_THR_GRID = list(globals().get(
    "SYS1_QWEN_THR_GRID",
    [0.05, 0.10, 0.15, 0.20, 0.25]
))

SYS1_USE_PYANNOTE_SOURCE_GRID = list(globals().get("SYS1_USE_PYANNOTE_SOURCE_GRID", [True]))
SYS1_USE_ASR_SOURCE_GRID = list(globals().get("SYS1_USE_ASR_SOURCE_GRID", [True]))

SYS1_CAND_MERGE_TOL_GRID = list(globals().get(
    "SYS1_CAND_MERGE_TOL_GRID",
    [0.12, 0.20, 0.25, 0.35]
))

SYS2_SHIFT_THR_GRID = list(globals().get(
    "SYS2_SHIFT_THR_GRID",
    [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
))

FINAL_COOLDOWN_GRID = list(globals().get(
    "FINAL_COOLDOWN_GRID",
    [0.00, 0.10, 0.20, 0.35, 0.50, 0.75]
))

FINAL_MIN_GAP_GRID = list(globals().get(
    "FINAL_MIN_GAP_GRID",
    [0.00, 0.10, 0.20, 0.35, 0.50]
))

SHIFT_TIME_OFFSET_GRID = list(globals().get(
    "SHIFT_TIME_OFFSET_GRID",
    [-0.10, -0.05, 0.0, 0.05, 0.10]
))

VAL_ROBUST_N_FOLDS = int(globals().get("VAL_ROBUST_N_FOLDS", 5))
VAL_ROBUST_TOP_K = int(globals().get("VAL_ROBUST_TOP_K", 100))
VAL_ROBUST_STD_PENALTY = float(globals().get("VAL_ROBUST_STD_PENALTY", 0.50))

USE_LOCAL_TIMING_CORRECTOR = bool(globals().get(
    "USE_LOCAL_TIMING_CORRECTOR",
    True,
))
TIMING_CORRECTOR_MATCH_RADIUS_S = float(globals().get(
    "TIMING_CORRECTOR_MATCH_RADIUS_S",
    1.00,
))
TIMING_CORRECTOR_MAX_ABS_OFFSET_S = float(globals().get(
    "TIMING_CORRECTOR_MAX_ABS_OFFSET_S",
    0.75,
))
TIMING_CORRECTOR_MAX_CANDIDATES_PER_GT = int(globals().get(
    "TIMING_CORRECTOR_MAX_CANDIDATES_PER_GT",
    5,
))
FINAL_TIME_MODE_GRID = list(globals().get(
    "FINAL_TIME_MODE_GRID",
    ["raw", "local_corrected"],
))

TEST_BOOTSTRAP_REPLICATES = int(globals().get(
    "TEST_BOOTSTRAP_REPLICATES",
    1000,
))
TEST_BOOTSTRAP_CONFIDENCE = float(globals().get(
    "TEST_BOOTSTRAP_CONFIDENCE",
    0.95,
))

SYS1_CONTEXT_S = float(globals().get("SYS1_CONTEXT_S", 4.0))
SYS1_HOP_S = float(globals().get("SYS1_HOP_S", 0.10))

WAVLM_TIMELINE_KEEP_THR = float(globals().get("WAVLM_TIMELINE_KEEP_THR", min(SYS1_WAVLM_THR_GRID)))
SYS1_TIMELINE_BATCH_SIZE = int(globals().get("SYS1_TIMELINE_BATCH_SIZE", 32))
SYS1_TIMELINE_LOCAL_MAX_RADIUS = int(globals().get("SYS1_TIMELINE_LOCAL_MAX_RADIUS", 1))

ALLOW_TRAIN_GT_SEED_CANDIDATES = bool(globals().get("ALLOW_TRAIN_GT_SEED_CANDIDATES", True))

SYS1_DENSE_FPS = int(round(1.0 / SYS1_HOP_S))
SYS1_EPOCHS = int(globals().get("SYS1_EPOCHS", 8 if DEBUG_SMALL_RUN else 35))
SYS1_BATCH_SIZE = int(globals().get("SYS1_BATCH_SIZE", 8))
SYS1_MICRO_BS = int(globals().get("SYS1_MICRO_BS", 1))
SYS1_LR = float(globals().get("SYS1_LR", 2e-4))
SYS1_POS_WEIGHT_CAP = float(globals().get("SYS1_POS_WEIGHT_CAP", 12.0))

SYNTH_NUM_MEETINGS = int(globals().get("SYNTH_NUM_MEETINGS", 80 if DEBUG_SMALL_RUN else 6000))
SYNTH_MIN_SPK = int(globals().get("SYNTH_MIN_SPK", 3))
SYNTH_MAX_SPK = int(globals().get("SYNTH_MAX_SPK", 6))
SYNTH_SR = int(globals().get("SYNTH_SR", 16000))
SYNTH_MIN_DURATION = float(globals().get("SYNTH_MIN_DURATION", 8.0))
SYNTH_MAX_DURATION = float(globals().get("SYNTH_MAX_DURATION", 28.0))

SYS2_CONTEXT_S = float(globals().get("SYS2_CONTEXT_S", 4.0))
SYS2_FRAME_HOP_S = float(globals().get("SYS2_FRAME_HOP_S", 0.10))
SYS2_TEMPORAL_EPOCHS = int(globals().get("SYS2_TEMPORAL_EPOCHS", 5 if DEBUG_SMALL_RUN else 45))
SYS2_TEMPORAL_BATCH_SIZE = int(globals().get("SYS2_TEMPORAL_BATCH_SIZE", 256))
SYS2_TEMPORAL_LR = float(globals().get("SYS2_TEMPORAL_LR", 8e-4))
SYS2_TEMPORAL_WEIGHT_DECAY = float(globals().get("SYS2_TEMPORAL_WEIGHT_DECAY", 1e-4))

LIGHTGBM_NUM_BOOST_ROUND = int(globals().get("LIGHTGBM_NUM_BOOST_ROUND", 200 if DEBUG_SMALL_RUN else 1200))
LIGHTGBM_EARLY_STOPPING_ROUNDS = int(globals().get("LIGHTGBM_EARLY_STOPPING_ROUNDS", 30 if DEBUG_SMALL_RUN else 80))

V3_VALIDITY_FIXES = {
    "test_candidates_from_full_timeline": True,
    "qwen3_no_real_val_training": True,
    "qwen3_weighted_training": True,
    "sys1_train_real_synth_diffusion": True,
    "sys2_train_real_synth_diffusion": True,
    "val_test_real_only": True,
    "sys1_high_recall_target_not_score_threshold": True,
    "audit_informed_action_aware_event_time": True,
    "positive_gap_capped_for_primary_task": True,
    "legacy_midpoint_retained_only_for_secondary_reporting": True,
    "midturn_negatives_capped_and_downweighted": True,
    "sys1_balance_not_applied_repeatedly": True,
    "local_timing_corrector_train_val_only": True,
    "robust_session_fold_val_selection": True,
}

print("Experiment:", EXPERIMENT_NAME)
print("Run version:", RUN_VERSION)
print("TOL:", TOL)
print("Event contract:", EVENT_CONTRACT_VERSION)
print("Primary event time:", PRIMARY_EVENT_TIME_DEFINITION)
print("Primary gap range:", (PRIMARY_MIN_GAP_S, PRIMARY_MAX_POSITIVE_GAP_S))
print("Mid-turn negative ratio/weight:", MIDTURN_NEG_TO_BOUNDARY_RATIO, WEIGHT_REAL_MIDTURN)
print("Sys1 class balance mode:", SYS1_CLASS_BALANCE_MODE)
print("Target Sys1 recall floor:", TARGET_SYS1_RECALL, "preferred:", PREFERRED_SYS1_RECALL)
print("Main LLM:", QWEN3_MODEL_NAME)
print("Main WavLM:", WAVLM_MODEL_NAME)
print("DEBUG_SMALL_RUN:", DEBUG_SMALL_RUN)
print("Component switches:", {
    "wavlm": USE_WAVLM_SYS1,
    "pyannote": USE_PYANNOTE_DIA,
    "asr": USE_ASR_TEXT,
    "qwen3": USE_QWEN3_LLM,
    "ecapa": USE_ECAPA_SPK,
    "synth": USE_SYNTH_MULTIPARTY,
    "diffusion": USE_DIFFUSION_AUG,
    "temporal_sys2": USE_TEMPORAL_SYS2,
    "lightgbm": USE_LIGHTGBM_SYS2,
})

Experiment: vap_rrfap_text_meta_confirmer_v1
Run version: 18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only
TOL: 0.5
Event contract: voxconverse_action_aware_immediate_v1
Primary event time: action_aware_boundary
Primary gap range: (-1.0, 1.0)
Mid-turn negative ratio/weight: 0.5 0.25
Sys1 class balance mode: loss_only
Target Sys1 recall floor: 0.9 preferred: 0.95
Main LLM: Qwen/Qwen3-1.7B
Main WavLM: microsoft/wavlm-large
DEBUG_SMALL_RUN: False
Component switches: {'wavlm': True, 'pyannote': True, 'asr': True, 'qwen3': True, 'ecapa': True, 'synth': True, 'diffusion': False, 'temporal_sys2': True, 'lightgbm': True}


In [3]:
from pathlib import Path

FRESH_MACHINE_FULL_RUN = False
CHECKPOINT_REUSE_RUN = True
CACHE_SAFE_RESUME_RUN = False

REUSE_PREVIOUS_SYS1_CHECKPOINT = False
PREVIOUS_SYS1_RUN_VERSION = "16thAug_MULTIMODAL_MULTIPARTY_CONTROLLER_v3"
PREVIOUS_SYS1_REL_DIR = "models/wavlm_sys1_dense_multiscale"
PREVIOUS_SYS1_BEST_FILENAME = "wavlm_dense_multiscale_sys1_best.pt"
PREVIOUS_SYS1_LAST_FILENAME = "wavlm_dense_multiscale_sys1_last.pt"
PREVIOUS_SYS1_HISTORY_FILENAME = "sys1_training_history.csv"
SKIP_FORCE_FRESH_SYS1_CELL10B = True

REUSE_AUDIT_COMPATIBLE_BASE_ASSETS = True
BASE_ASSETS_READ_ONLY = True

FORCE_REBUILD = False
FORCE_TRAIN = False
FORCE_REBUILD_FEATURES = False
FORCE_REBUILD_PYANNOTE = False
FORCE_REBUILD_NS = False

FORCE_REBUILD_NS_AUTO_PROTOTYPES = False
FORCE_REBUILD_NS_AUTO_PREDICTIONS = False
FORCE_REBUILD_NS_AUTO_ORACLE = False

FORCE_REBUILD_SPLITS = False
FORCE_REBUILD_REAL_EVENTS = False

FORCE_REGENERATE_SYNTH = False
FORCE_REBUILD_AUGMENTED_AUDIO = False
FORCE_RERUN_ASR = False
FORCE_RERUN_PYANNOTE = False
FORCE_REBUILD_PYANNOTE = FORCE_RERUN_PYANNOTE

FORCE_RETRAIN_SYS1 = False
RESUME_SYS1_TRAIN = True

FORCE_RETRAIN_QWEN3 = False
QWEN3_RESUME_FROM_CHECKPOINT = True

FORCE_REBUILD_SYS1_CANDIDATES = False
FORCE_REBUILD_SYS2_FEATURES = False

FORCE_RETRAIN_TEMPORAL_SYS2 = False
RESUME_TEMPORAL_SYS2_TRAIN = True

FORCE_RETRAIN_LIGHTGBM = False
FORCE_RETRAIN_LIGHTGBM_BINARY = False
FORCE_RETRAIN_EVENT_RANKER = False
FORCE_RETRAIN_RR_FAP = False
FORCE_RETRAIN_META_CONFIRMER = False
FORCE_RETRAIN_TIMING_CORRECTOR = False

FORCE_RETUNE_VAL = False
FORCE_REBUILD_TEST_PRED = False

FORCE_REBUILD_FEATURES = (
    FORCE_REBUILD_SYS1_CANDIDATES
    or FORCE_REBUILD_SYS2_FEATURES
)

FORCE_TRAIN = (
    FORCE_RETRAIN_SYS1
    or FORCE_RETRAIN_QWEN3
    or FORCE_RETRAIN_TEMPORAL_SYS2
    or FORCE_RETRAIN_LIGHTGBM
    or FORCE_RETRAIN_LIGHTGBM_BINARY
    or FORCE_RETRAIN_EVENT_RANKER
    or FORCE_RETRAIN_RR_FAP
    or FORCE_RETRAIN_META_CONFIRMER
    or FORCE_RETRAIN_TIMING_CORRECTOR
)

FORCE_REBUILD = (
    FORCE_REBUILD_SPLITS
    or FORCE_REBUILD_REAL_EVENTS
    or FORCE_REGENERATE_SYNTH
    or FORCE_REBUILD_AUGMENTED_AUDIO
    or FORCE_RERUN_ASR
    or FORCE_RERUN_PYANNOTE
    or FORCE_REBUILD_SYS1_CANDIDATES
    or FORCE_REBUILD_SYS2_FEATURES
    or FORCE_RETUNE_VAL
    or FORCE_REBUILD_TEST_PRED
    or FORCE_REBUILD_NS
    or FORCE_REBUILD_NS_AUTO_PROTOTYPES
    or FORCE_REBUILD_NS_AUTO_PREDICTIONS
    or FORCE_REBUILD_NS_AUTO_ORACLE
)

SYS2_TEMPORAL_BATCH_SIZE = int(
    globals().get("SYS2_TEMPORAL_BATCH_SIZE", 128)
)
SYS2_TEMPORAL_NUM_WORKERS = int(
    globals().get("SYS2_TEMPORAL_NUM_WORKERS", 0)
)
SYS2_TEMPORAL_PIN_MEMORY = bool(
    globals().get("SYS2_TEMPORAL_PIN_MEMORY", False)
)

USE_TEST_INFORMED_EXPLORATORY_BRANCH = False
ALLOW_TEST_INFORMED_EXPLORATORY_AS_ACTIVE = False

print("Paper-ready checkpoint/cache policy set.")
print("CHECKPOINT_REUSE_RUN:", CHECKPOINT_REUSE_RUN)
print("CACHE_SAFE_RESUME_RUN:", CACHE_SAFE_RESUME_RUN)
print("REUSE_PREVIOUS_SYS1_CHECKPOINT:", REUSE_PREVIOUS_SYS1_CHECKPOINT)
print("REUSE_AUDIT_COMPATIBLE_BASE_ASSETS:", REUSE_AUDIT_COMPATIBLE_BASE_ASSETS)
print("BASE_ASSETS_READ_ONLY:", BASE_ASSETS_READ_ONLY)

print({
    "FORCE_REBUILD_SPLITS": FORCE_REBUILD_SPLITS,
    "FORCE_REBUILD_REAL_EVENTS": FORCE_REBUILD_REAL_EVENTS,
    "FORCE_REGENERATE_SYNTH": FORCE_REGENERATE_SYNTH,
    "FORCE_REBUILD_AUGMENTED_AUDIO": FORCE_REBUILD_AUGMENTED_AUDIO,
    "FORCE_RERUN_ASR": FORCE_RERUN_ASR,
    "FORCE_RERUN_PYANNOTE": FORCE_RERUN_PYANNOTE,
    "FORCE_RETRAIN_SYS1": FORCE_RETRAIN_SYS1,
    "RESUME_SYS1_TRAIN": RESUME_SYS1_TRAIN,
    "FORCE_RETRAIN_QWEN3": FORCE_RETRAIN_QWEN3,
    "QWEN3_RESUME_FROM_CHECKPOINT": QWEN3_RESUME_FROM_CHECKPOINT,
    "FORCE_REBUILD_SYS1_CANDIDATES": FORCE_REBUILD_SYS1_CANDIDATES,
    "FORCE_REBUILD_SYS2_FEATURES": FORCE_REBUILD_SYS2_FEATURES,
    "FORCE_RETRAIN_TEMPORAL_SYS2": FORCE_RETRAIN_TEMPORAL_SYS2,
    "FORCE_RETRAIN_LIGHTGBM": FORCE_RETRAIN_LIGHTGBM,
    "FORCE_RETRAIN_LIGHTGBM_BINARY": FORCE_RETRAIN_LIGHTGBM_BINARY,
    "FORCE_RETRAIN_EVENT_RANKER": FORCE_RETRAIN_EVENT_RANKER,
    "FORCE_RETRAIN_RR_FAP": FORCE_RETRAIN_RR_FAP,
    "FORCE_RETRAIN_META_CONFIRMER": FORCE_RETRAIN_META_CONFIRMER,
    "FORCE_RETRAIN_TIMING_CORRECTOR": FORCE_RETRAIN_TIMING_CORRECTOR,
    "FORCE_RETUNE_VAL": FORCE_RETUNE_VAL,
    "FORCE_REBUILD_TEST_PRED": FORCE_REBUILD_TEST_PRED,
    "USE_TEST_INFORMED_EXPLORATORY_BRANCH": USE_TEST_INFORMED_EXPLORATORY_BRANCH,
    "ALLOW_TEST_INFORMED_EXPLORATORY_AS_ACTIVE": ALLOW_TEST_INFORMED_EXPLORATORY_AS_ACTIVE,
})

Paper-ready checkpoint/cache policy set.
CHECKPOINT_REUSE_RUN: True
CACHE_SAFE_RESUME_RUN: False
REUSE_PREVIOUS_SYS1_CHECKPOINT: False
REUSE_AUDIT_COMPATIBLE_BASE_ASSETS: True
BASE_ASSETS_READ_ONLY: True
{'FORCE_REBUILD_SPLITS': False, 'FORCE_REBUILD_REAL_EVENTS': False, 'FORCE_REGENERATE_SYNTH': False, 'FORCE_REBUILD_AUGMENTED_AUDIO': False, 'FORCE_RERUN_ASR': False, 'FORCE_RERUN_PYANNOTE': False, 'FORCE_RETRAIN_SYS1': False, 'RESUME_SYS1_TRAIN': True, 'FORCE_RETRAIN_QWEN3': False, 'QWEN3_RESUME_FROM_CHECKPOINT': True, 'FORCE_REBUILD_SYS1_CANDIDATES': False, 'FORCE_REBUILD_SYS2_FEATURES': False, 'FORCE_RETRAIN_TEMPORAL_SYS2': False, 'FORCE_RETRAIN_LIGHTGBM': False, 'FORCE_RETRAIN_LIGHTGBM_BINARY': False, 'FORCE_RETRAIN_EVENT_RANKER': False, 'FORCE_RETRAIN_RR_FAP': False, 'FORCE_RETRAIN_META_CONFIRMER': False, 'FORCE_RETRAIN_TIMING_CORRECTOR': False, 'FORCE_RETUNE_VAL': False, 'FORCE_REBUILD_TEST_PRED': False, 'USE_TEST_INFORMED_EXPLORATORY_BRANCH': False, 'ALLOW_TEST_INFORMED_EXPLORAT

In [4]:
if bool(
    globals().get(
        "RUN_CLEAN_AUG_ABLATION",
        False,
    )
):
    CHECKPOINT_REUSE_RUN = True
    CACHE_SAFE_RESUME_RUN = False
    FRESH_MACHINE_FULL_RUN = False

    REUSE_PREVIOUS_SYS1_CHECKPOINT = False
    SKIP_FORCE_FRESH_SYS1_CELL10B = False

    FORCE_REBUILD_SPLITS = False
    FORCE_REBUILD_REAL_EVENTS = False

    FORCE_REGENERATE_SYNTH = False
    FORCE_REBUILD_AUGMENTED_AUDIO = False
    FORCE_RERUN_ASR = False
    FORCE_RERUN_PYANNOTE = False

    FORCE_RETRAIN_SYS1 = False
    RESUME_SYS1_TRAIN = True
    FORCE_RETRAIN_QWEN3 = False
    FORCE_REBUILD_SYS1_CANDIDATES = False
    FORCE_REBUILD_SYS2_FEATURES = False
    FORCE_RETRAIN_TEMPORAL_SYS2 = False
    FORCE_RETRAIN_LIGHTGBM = False
    FORCE_RETRAIN_LIGHTGBM_BINARY = False
    FORCE_RETRAIN_EVENT_RANKER = False
    FORCE_RETRAIN_RR_FAP = False
    FORCE_RETRAIN_META_CONFIRMER = False
    FORCE_RETRAIN_TIMING_CORRECTOR = False
    FORCE_RETUNE_VAL = False
    FORCE_REBUILD_TEST_PRED = False

    FORCE_REBUILD_BRANCH2_FEATURES = False
    FORCE_RETRAIN_BRANCH2_LGBM = False
    FORCE_REBUILD_BRANCH3_TABLE = False
    FORCE_RETRAIN_BRANCH3_LGBM = False
    FORCE_RETRAIN_SYSTEM_FLOOR = False

    print("=" * 80)
    print("PAPER-READY AUGMENTATION CONDITION")
    print("=" * 80)
    print("Condition:", AUG_ABLATION_CONDITION)
    print("Run version:", RUN_VERSION_OVERRIDE)
    print("Missing learned stages train automatically; completed compatible stages resume/load.")
    print(
        "For the final ablation campaign, use streamlined notebooks generated "
        "from this paper-ready full notebook so Branch 2/3 and diagnostics are "
        "not executed unnecessarily."
    )
else:
    print(
        "Cell 2A: normal paper-ready full-system resume policy preserved."
    )

PAPER-READY AUGMENTATION CONDITION
Condition: synthetic_only
Run version: 18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only
Missing learned stages train automatically; completed compatible stages resume/load.
For the final ablation campaign, use streamlined notebooks generated from this paper-ready full notebook so Branch 2/3 and diagnostics are not executed unnecessarily.


In [5]:
# Imports, dependency check, seed, device
import os, sys, json, math, time, random, warnings, shutil, subprocess, importlib.util
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

try:
    import soundfile as sf
except Exception as e:
    raise ImportError("soundfile is required. Install with: pip install soundfile") from e

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
except Exception as e:
    raise ImportError("PyTorch is required for this notebook. Install a CUDA-compatible PyTorch build if using GPU.") from e

try:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        precision_recall_fscore_support,
        accuracy_score,
        balanced_accuracy_score,
        confusion_matrix,
    )
except Exception as e:
    raise ImportError("scikit-learn is required. Install with: pip install scikit-learn") from e

def module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except (ModuleNotFoundError, ValueError, AttributeError):
        return False

REQUIRED_OPTIONAL = {
    "transformers": {
        "modules": ["transformers"],
        "install": "pip install -U transformers accelerate",
        "used_for": "WavLM loading and Qwen3 action classifier",
    },
    "peft": {
        "modules": ["peft"],
        "install": "pip install -U peft",
        "used_for": "Qwen3 LoRA fine-tuning",
    },
    "datasets": {
        "modules": ["datasets"],
        "install": "pip install -U datasets",
        "used_for": "optional Hugging Face diarizers/VoxConverse loading",
    },
    "faster-whisper": {
        "modules": ["faster_whisper"],
        "install": "pip install -U faster-whisper ctranslate2",
        "used_for": "ASR text extraction",
    },
    "pyannote.audio": {
        "modules": ["pyannote.audio"],
        "install": "pip install -U pyannote.audio",
        "used_for": "pyannote diarization/speaker-change candidates",
    },
    "speechbrain": {
        "modules": ["speechbrain"],
        "install": "pip install -U speechbrain",
        "used_for": "ECAPA speaker-similarity features",
    },
    "lightgbm": {
        "modules": ["lightgbm"],
        "install": "pip install -U lightgbm",
        "used_for": "LightGBM Sys2 verifier/calibrator",
    },
    "torchaudio": {
        "modules": ["torchaudio"],
        "install": "pip install -U torchaudio",
        "used_for": "audio loading/resampling fallback for WavLM/Sys1",
    },
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DEVICE = torch.device(DEVICE)

SEED = int(globals().get("SEED", 101))
os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

CUDNN_DETERMINISTIC = bool(globals().get("CUDNN_DETERMINISTIC", False))
torch.backends.cudnn.deterministic = CUDNN_DETERMINISTIC
torch.backends.cudnn.benchmark = not CUDNN_DETERMINISTIC

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("CUDA memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("CUDA available:", torch.cuda.is_available())

print("Seed:", SEED)
print("CUDNN deterministic:", torch.backends.cudnn.deterministic)
print("CUDNN benchmark:", torch.backends.cudnn.benchmark)

missing = []
available = []

for package_name, spec in REQUIRED_OPTIONAL.items():
    ok = all(module_available(m) for m in spec["modules"])
    if ok:
        available.append(package_name)
    else:
        missing.append((package_name, spec["install"], spec["used_for"]))

print("\nAvailable optional packages:")
if available:
    for package_name in available:
        print(f"✅ {package_name}")
else:
    print("None detected.")

if missing:
    print("\nOptional dependencies missing. Enabled branches using them will fail until installed:")
    for package_name, install_cmd, used_for in missing:
        print(f"❌ {package_name}: {install_cmd}  # used for {used_for}")
else:
    print("\nAll optional branch dependencies appear importable.")

def require_module(module_name: str, install_hint: str = ""):
    if not module_available(module_name):
        msg = f"Missing required module for this branch: {module_name}"
        if install_hint:
            msg += f"\nInstall with: {install_hint}"
        raise ImportError(msg)
    return True

Python: 3.11.15
PyTorch: 2.7.0+cu118
Device: cuda
GPU: NVIDIA GeForce RTX 4070
CUDA available: True
CUDA version: 11.8
CUDA memory GB: 12.58
Seed: 101
CUDNN deterministic: False
CUDNN benchmark: True

Available optional packages:
✅ transformers
✅ peft
✅ datasets
✅ faster-whisper
✅ pyannote.audio
✅ speechbrain
✅ lightgbm
✅ torchaudio

All optional branch dependencies appear importable.


In [6]:
# Paths, splits, output directories
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "configs" / "voxconverse").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = Path(
    globals().get(
        "PROJECT_ROOT",
        os.environ.get("PROJECT_ROOT", _find_project_root()),
    )
).expanduser().resolve()

DATA_DIR = PROJECT_ROOT / "data" / "voxconverse"
CONFIG_DIR = PROJECT_ROOT / "configs" / "voxconverse"
RESULTS_DIR = PROJECT_ROOT / "results" / "voxconverse"

VOX_ROOT = Path(
    globals().get(
        "VOX_ROOT",
        os.environ.get("VOX_ROOT", DATA_DIR),
    )
).expanduser().resolve()

AUDIO_DEV = Path(globals().get("AUDIO_DEV", VOX_ROOT / "voxconverse_dev_wav" / "audio")).expanduser().resolve()
AUDIO_TEST = Path(globals().get("AUDIO_TEST", VOX_ROOT / "voxconverse_test_wav" / "audio")).expanduser().resolve()
RTTM_DEV = Path(globals().get("RTTM_DEV", VOX_ROOT / "labels" / "dev")).expanduser().resolve()
RTTM_TEST = Path(globals().get("RTTM_TEST", VOX_ROOT / "labels" / "test")).expanduser().resolve()

SPLITS_DIR = Path(
    globals().get(
        "SPLITS_DIR",
        os.environ.get("VOX_SPLITS_DIR", CONFIG_DIR),
    )
).expanduser().resolve()

RUN_ROOT = Path(
    globals().get(
        "RUN_ROOT",
        os.environ.get("VOX_RUN_ROOT", RESULTS_DIR / RUN_VERSION),
    )
).expanduser().resolve()

CACHE_DIR = RUN_ROOT / "cache"
MODELS_DIR = RUN_ROOT / "models"
PRED_DIR = RUN_ROOT / "predictions"
EVAL_DIR = RUN_ROOT / "eval"
FIG_DIR = RUN_ROOT / "figures"
CONTRACT_DIR = RUN_ROOT / "event_contract"

BASE_ASSET_RUN_ROOT = Path(
    globals().get(
        "BASE_ASSET_RUN_ROOT",
        os.environ.get("VOX_BASE_ASSET_RUN_ROOT", RESULTS_DIR / AUDIT_BASE_ASSET_RUN_VERSION),
    )
).expanduser().resolve()

BASE_SYNTH_ASSET_DIR = BASE_ASSET_RUN_ROOT / "synthetic_multiparty"
BASE_AUG_ASSET_DIR = BASE_ASSET_RUN_ROOT / "augmented_audio"
BASE_ASR_CACHE_DIR = BASE_ASSET_RUN_ROOT / "cache" / "asr"
BASE_PYANNOTE_CACHE_DIR = BASE_ASSET_RUN_ROOT / "cache" / "pyannote"
BASE_ECAPA_CACHE_DIR = BASE_ASSET_RUN_ROOT / "cache" / "ecapa"

SYS1_MODEL_DIR = MODELS_DIR / "wavlm_sys1_dense_multiscale"
QWEN3_MODEL_DIR = MODELS_DIR / "qwen3_action_classifier"
TEMPORAL_SYS2_DIR = MODELS_DIR / "sys2_temporal_verifier"
LGBM_SYS2_DIR = MODELS_DIR / "lightgbm_verifier"
TIMING_CORRECTOR_DIR = MODELS_DIR / "local_timing_corrector"

SYNTH_DIR = RUN_ROOT / "synthetic_multiparty"
AUG_DIR = RUN_ROOT / "augmented_audio"

if (
    bool(globals().get("REUSE_AUDIT_COMPATIBLE_BASE_ASSETS", True))
    and BASE_ASR_CACHE_DIR.exists()
):
    ASR_CACHE_DIR = BASE_ASR_CACHE_DIR
else:
    ASR_CACHE_DIR = CACHE_DIR / "asr"

if (
    bool(globals().get("REUSE_AUDIT_COMPATIBLE_BASE_ASSETS", True))
    and BASE_PYANNOTE_CACHE_DIR.exists()
):
    PYANNOTE_CACHE_DIR = BASE_PYANNOTE_CACHE_DIR
else:
    PYANNOTE_CACHE_DIR = CACHE_DIR / "pyannote"

if (
    bool(globals().get("REUSE_AUDIT_COMPATIBLE_BASE_ASSETS", True))
    and BASE_ECAPA_CACHE_DIR.exists()
):
    ECAPA_CACHE_DIR = BASE_ECAPA_CACHE_DIR
else:
    ECAPA_CACHE_DIR = CACHE_DIR / "ecapa"

ASSET_CACHE_RUN_VERSION = (
    AUDIT_BASE_ASSET_RUN_VERSION
    if ASR_CACHE_DIR == BASE_ASR_CACHE_DIR
    else RUN_VERSION
)

DIRS_TO_CREATE = [
    RUN_ROOT,
    CACHE_DIR,
    MODELS_DIR,
    PRED_DIR,
    EVAL_DIR,
    FIG_DIR,
    CONTRACT_DIR,
    TIMING_CORRECTOR_DIR,
    SPLITS_DIR,
    SYS1_MODEL_DIR,
    QWEN3_MODEL_DIR,
    TEMPORAL_SYS2_DIR,
    LGBM_SYS2_DIR,
    PYANNOTE_CACHE_DIR,
    ASR_CACHE_DIR,
    ECAPA_CACHE_DIR,
    SYNTH_DIR,
    AUG_DIR,
]

for d in DIRS_TO_CREATE:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VOX_ROOT:", VOX_ROOT)
print("AUDIO_DEV:", AUDIO_DEV, "exists=", AUDIO_DEV.exists())
print("AUDIO_TEST:", AUDIO_TEST, "exists=", AUDIO_TEST.exists())
print("RTTM_DEV:", RTTM_DEV, "exists=", RTTM_DEV.exists())
print("RTTM_TEST:", RTTM_TEST, "exists=", RTTM_TEST.exists())
print("SPLITS_DIR:", SPLITS_DIR)
print("RUN_ROOT:", RUN_ROOT)
print("BASE_ASSET_RUN_ROOT:", BASE_ASSET_RUN_ROOT, "exists=", BASE_ASSET_RUN_ROOT.exists())
print("ASR_CACHE_DIR:", ASR_CACHE_DIR)
print("PYANNOTE_CACHE_DIR:", PYANNOTE_CACHE_DIR)
print("ECAPA_CACHE_DIR:", ECAPA_CACHE_DIR)
print("BASE_SYNTH_ASSET_DIR:", BASE_SYNTH_ASSET_DIR)
print("BASE_AUG_ASSET_DIR:", BASE_AUG_ASSET_DIR)

def _session_id_from_path(p: Path) -> str:
    return Path(p).stem

def list_audio_files(audio_dir: Path) -> Dict[str, Path]:
    audio_dir = Path(audio_dir)
    files = {}
    if not audio_dir.exists():
        return files

    valid_exts = {".wav", ".flac", ".mp3", ".m4a"}
    for p in audio_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in valid_exts:
            sid = _session_id_from_path(p)
            if sid not in files or p.suffix.lower() == ".wav":
                files[sid] = p

    return dict(sorted(files.items()))

def list_rttm_files(rttm_dir: Path) -> Dict[str, Path]:
    rttm_dir = Path(rttm_dir)
    files = {}
    if not rttm_dir.exists():
        return files

    for p in rttm_dir.rglob("*.rttm"):
        if p.is_file():
            files[_session_id_from_path(p)] = p

    return dict(sorted(files.items()))

def _load_json_list(path: Path) -> List[str]:
    try:
        obj = json.loads(Path(path).read_text())
        if isinstance(obj, list):
            return [str(x) for x in obj]
    except Exception:
        pass
    return []

def _save_json_list(path: Path, values: List[str]):
    Path(path).write_text(json.dumps(list(values), indent=2))

def _validate_split(train_sessions: List[str], val_sessions: List[str], available_sessions: List[str]) -> Tuple[List[str], List[str], bool]:
    available = set(available_sessions)
    train = sorted([s for s in train_sessions if s in available])
    val = sorted([s for s in val_sessions if s in available])

    overlap = sorted(set(train).intersection(val))
    if overlap:
        print(f"WARNING: removing {len(overlap)} train/val overlaps from VAL.")
        val = sorted([s for s in val if s not in set(train)])

    is_valid = len(train) > 0 and len(val) > 0 and len(set(train).intersection(val)) == 0
    return train, val, is_valid

DEV_AUDIO = list_audio_files(AUDIO_DEV)
TEST_AUDIO = list_audio_files(AUDIO_TEST)
DEV_RTTM = list_rttm_files(RTTM_DEV)
TEST_RTTM = list_rttm_files(RTTM_TEST)

DEV_SESSIONS_ALL = sorted(set(DEV_AUDIO).intersection(DEV_RTTM))
TEST_SESSIONS = sorted(set(TEST_AUDIO).intersection(TEST_RTTM))

print("\nDiscovered files:")
print("DEV audio:", len(DEV_AUDIO), "| DEV RTTM:", len(DEV_RTTM), "| matched DEV sessions:", len(DEV_SESSIONS_ALL))
print("TEST audio:", len(TEST_AUDIO), "| TEST RTTM:", len(TEST_RTTM), "| matched TEST sessions:", len(TEST_SESSIONS))

missing_dev_audio = sorted(set(DEV_RTTM) - set(DEV_AUDIO))
missing_dev_rttm = sorted(set(DEV_AUDIO) - set(DEV_RTTM))
missing_test_audio = sorted(set(TEST_RTTM) - set(TEST_AUDIO))
missing_test_rttm = sorted(set(TEST_AUDIO) - set(TEST_RTTM))

if missing_dev_audio[:5]:
    print("Example DEV RTTM without audio:", missing_dev_audio[:5])
if missing_dev_rttm[:5]:
    print("Example DEV audio without RTTM:", missing_dev_rttm[:5])
if missing_test_audio[:5]:
    print("Example TEST RTTM without audio:", missing_test_audio[:5])
if missing_test_rttm[:5]:
    print("Example TEST audio without RTTM:", missing_test_rttm[:5])

assert len(DEV_SESSIONS_ALL) > 0, (
    f"No matched DEV sessions found.\n"
    f"Check AUDIO_DEV={AUDIO_DEV}\n"
    f"Check RTTM_DEV={RTTM_DEV}"
)
assert len(TEST_SESSIONS) > 0, (
    f"No matched TEST sessions found.\n"
    f"Check AUDIO_TEST={AUDIO_TEST}\n"
    f"Check RTTM_TEST={RTTM_TEST}"
)

train_split_file = SPLITS_DIR / "train_sessions.json"
val_split_file = SPLITS_DIR / "val_sessions.json"

missing_split_files = [str(p) for p in [train_split_file, val_split_file] if not p.exists()]
if missing_split_files:
    raise FileNotFoundError(
        "Frozen VoxConverse TRAIN/VAL split manifests are required. "
        f"Missing: {missing_split_files}"
    )

if FORCE_REBUILD_SPLITS:
    print("FORCE_REBUILD_SPLITS is ignored in the public release; frozen manifests are required.")

loaded_train = _load_json_list(train_split_file)
loaded_val = _load_json_list(val_split_file)
raw_overlap = sorted(set(loaded_train).intersection(loaded_val))
if raw_overlap:
    raise ValueError(f"Frozen TRAIN/VAL manifests overlap: {raw_overlap[:10]}")
if len(loaded_train) != 177 or len(loaded_val) != 39:
    raise ValueError(
        "Frozen VoxConverse TRAIN/VAL manifests must contain exactly "
        f"177 TRAIN and 39 VAL sessions; got {len(loaded_train)} and {len(loaded_val)}."
    )

TRAIN_SESSIONS, VAL_SESSIONS, split_valid = _validate_split(
    loaded_train,
    loaded_val,
    DEV_SESSIONS_ALL,
)
if not split_valid:
    raise ValueError("Frozen VoxConverse TRAIN/VAL split manifests are invalid for the discovered DEV data.")
if len(TRAIN_SESSIONS) != 177 or len(VAL_SESSIONS) != 39:
    raise ValueError(
        "Frozen VoxConverse TRAIN/VAL manifests did not match the discovered DEV sessions: "
        f"{len(TRAIN_SESSIONS)} TRAIN and {len(VAL_SESSIONS)} VAL after validation."
    )

loaded_existing_split = True
print("\nLoaded frozen train/val split from:", SPLITS_DIR)
print("Verified TRAIN sessions:", len(TRAIN_SESSIONS))
print("Verified VAL sessions:", len(VAL_SESSIONS))
print("Verified TRAIN/VAL overlap: 0")

TRAIN_SESSIONS_FULL = list(TRAIN_SESSIONS)
VAL_SESSIONS_FULL = list(VAL_SESSIONS)
TEST_SESSIONS_FULL = list(TEST_SESSIONS)

if DEBUG_SMALL_RUN:
    TRAIN_SESSIONS = TRAIN_SESSIONS[:DEBUG_MAX_TRAIN_SESSIONS]
    VAL_SESSIONS = VAL_SESSIONS[:DEBUG_MAX_VAL_SESSIONS]
    TEST_SESSIONS = TEST_SESSIONS[:DEBUG_MAX_TEST_SESSIONS]
    print("\nDEBUG_SMALL_RUN enabled.")
    print("Active sessions are truncated, but saved split files remain full-size.")

SESSION_SPLIT = {s: "train" for s in TRAIN_SESSIONS}
SESSION_SPLIT.update({s: "val" for s in VAL_SESSIONS})
SESSION_SPLIT.update({s: "test" for s in TEST_SESSIONS})

AUDIO_BY_SPLIT = {
    "train": {s: DEV_AUDIO[s] for s in TRAIN_SESSIONS if s in DEV_AUDIO},
    "val": {s: DEV_AUDIO[s] for s in VAL_SESSIONS if s in DEV_AUDIO},
    "test": {s: TEST_AUDIO[s] for s in TEST_SESSIONS if s in TEST_AUDIO},
}

RTTM_BY_SPLIT = {
    "train": {s: DEV_RTTM[s] for s in TRAIN_SESSIONS if s in DEV_RTTM},
    "val": {s: DEV_RTTM[s] for s in VAL_SESSIONS if s in DEV_RTTM},
    "test": {s: TEST_RTTM[s] for s in TEST_SESSIONS if s in TEST_RTTM},
}

SPLIT_SUMMARY = {
    "run_version": RUN_VERSION,
    "event_contract_version": EVENT_CONTRACT_VERSION,
    "audit_base_asset_run_version": AUDIT_BASE_ASSET_RUN_VERSION,
    "base_asset_run_root": str(BASE_ASSET_RUN_ROOT),
    "asset_cache_run_version": str(ASSET_CACHE_RUN_VERSION),
    "vox_root": str(VOX_ROOT),
    "audio_dev": str(AUDIO_DEV),
    "audio_test": str(AUDIO_TEST),
    "rttm_dev": str(RTTM_DEV),
    "rttm_test": str(RTTM_TEST),
    "splits_dir": str(SPLITS_DIR),
    "loaded_existing_split": loaded_existing_split,
    "debug_small_run": DEBUG_SMALL_RUN,
    "seed": SEED,
    "n_dev_sessions_all": len(DEV_SESSIONS_ALL),
    "n_train_full": len(TRAIN_SESSIONS_FULL),
    "n_val_full": len(VAL_SESSIONS_FULL),
    "n_test_full": len(TEST_SESSIONS_FULL),
    "n_train_active": len(TRAIN_SESSIONS),
    "n_val_active": len(VAL_SESSIONS),
    "n_test_active": len(TEST_SESSIONS),
    "train_sessions_active": TRAIN_SESSIONS,
    "val_sessions_active": VAL_SESSIONS,
    "test_sessions_active": TEST_SESSIONS,
}

split_summary_path = RUN_ROOT / "split_summary.json"
split_summary_path.write_text(json.dumps(SPLIT_SUMMARY, indent=2))

ASSET_PROVENANCE = {
    "run_version": RUN_VERSION,
    "event_contract_version": EVENT_CONTRACT_VERSION,
    "base_asset_run_version": AUDIT_BASE_ASSET_RUN_VERSION,
    "base_asset_run_root": str(BASE_ASSET_RUN_ROOT),
    "synthetic_audio_rttm_source": str(BASE_SYNTH_ASSET_DIR),
    "diffusion_audio_source": str(BASE_AUG_ASSET_DIR),
    "asr_cache_dir": str(ASR_CACHE_DIR),
    "pyannote_cache_dir": str(PYANNOTE_CACHE_DIR),
    "ecapa_cache_dir": str(ECAPA_CACHE_DIR),
    "reused_condition_independent_assets_only": True,
    "old_learned_checkpoints_reused": False,
}
asset_provenance_path = RUN_ROOT / "asset_provenance.json"
asset_provenance_path.write_text(
    json.dumps(ASSET_PROVENANCE, indent=2, sort_keys=True)
)
ASSET_PROVENANCE_PATH = asset_provenance_path

print("\nFinal active split:")
print("Train sessions:", len(TRAIN_SESSIONS), "| full:", len(TRAIN_SESSIONS_FULL))
print("Val sessions:", len(VAL_SESSIONS), "| full:", len(VAL_SESSIONS_FULL))
print("Test sessions:", len(TEST_SESSIONS), "| full:", len(TEST_SESSIONS_FULL))
print("Split summary saved to:", split_summary_path)

assert len(TRAIN_SESSIONS) > 0, "No active TRAIN sessions."
assert len(VAL_SESSIONS) > 0, "No active VAL sessions."
assert len(TEST_SESSIONS) > 0, "No active TEST sessions."

globals()["ASSET_PROVENANCE"] = ASSET_PROVENANCE
globals()["ASSET_PROVENANCE_PATH"] = ASSET_PROVENANCE_PATH

VOX_ROOT: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset
AUDIO_DEV: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/voxconverse_dev_wav/audio exists= True
AUDIO_TEST: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/voxconverse_test_wav/audio exists= True
RTTM_DEV: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/labels/dev exists= True
RTTM_TEST: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/labels/test exists= True
SPLITS_DIR: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/Update_1/splits_multiparty_official
RUN_ROOT: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only
BASE_ASSET_RUN_ROOT: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1 exists= True
ASR_CACHE_DIR: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1/cache/asr
PYANNOTE_CACHE_DIR: <REPOSITORY_ROOT>/Experiment_1/voxconverse-

In [7]:
import hashlib
from collections import defaultdict

EVENT_COLUMNS = [
    "session",
    "time",
    "time_floor_release",
    "time_floor_acquisition",
    "time_midpoint",
    "transition_interval_start",
    "transition_interval_end",
    "transition_width_s",
    "event_time_definition",
    "event_anchor_role",
    "event_contract_version",
    "contract_name",
    "is_primary_contract_event",
    "is_delayed_transition",
    "prev_end",
    "next_start",
    "prev_speaker",
    "next_speaker",
    "gap",
    "overlap",
    "prev_dur",
    "next_dur",
    "action",
    "action_id",
    "label",
    "source_event_type",
]

REAL_EVENT_EXTRA_COLUMNS = [
    "split",
    "audio_path",
    "rttm_path",
    "is_synthetic",
    "is_diffusion_augmented",
    "sample_weight",
]

TURN_COLUMNS = [
    "session",
    "start",
    "end",
    "dur",
    "speaker",
]

def _empty_turns_df() -> pd.DataFrame:
    return pd.DataFrame(columns=TURN_COLUMNS)

def _empty_events_df(
    include_real_extra: bool = False,
) -> pd.DataFrame:
    columns = list(EVENT_COLUMNS)

    if include_real_extra:
        columns += list(REAL_EVENT_EXTRA_COLUMNS)

    return pd.DataFrame(columns=columns)

def _stable_session_seed(
    session: str,
    base_seed: int = 101,
) -> int:
    digest = hashlib.md5(
        str(session).encode("utf-8")
    ).hexdigest()

    return int(digest[:8], 16) + int(base_seed)

def read_rttm_as_turns(
    rttm_path: Path,
) -> pd.DataFrame:
    rttm_path = Path(rttm_path)
    rows = []

    if not rttm_path.exists():
        return _empty_turns_df()

    with rttm_path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        for line in handle:
            parts = line.strip().split()

            if (
                not parts
                or parts[0] != "SPEAKER"
                or len(parts) < 8
            ):
                continue

            try:
                session = str(parts[1])
                start = float(parts[3])
                duration = float(parts[4])
                speaker = str(parts[7])
            except Exception:
                continue

            if (
                not np.isfinite(start)
                or not np.isfinite(duration)
                or duration <= 0
            ):
                continue

            rows.append({
                "session": session,
                "start": float(start),
                "end": float(start + duration),
                "dur": float(duration),
                "speaker": speaker,
            })

    if not rows:
        return _empty_turns_df()

    frame = pd.DataFrame(rows)

    frame = (
        frame
        .sort_values(
            ["start", "end", "speaker"]
        )
        .reset_index(drop=True)
    )

    return frame[TURN_COLUMNS]

def get_session_audio_path(
    session: str,
    split: str,
) -> Path:
    split = str(split).lower()

    if split == "test":
        return Path(TEST_AUDIO[session])

    return Path(DEV_AUDIO[session])

def get_session_rttm_path(
    session: str,
    split: str,
) -> Path:
    split = str(split).lower()

    if split == "test":
        return Path(TEST_RTTM[session])

    return Path(DEV_RTTM[session])

def get_audio_duration(
    path: Path,
) -> float:
    info = sf.info(str(path))

    return (
        float(info.frames)
        / float(info.samplerate)
    )

def classify_action_from_adjacent_turns(
    same_speaker: bool,
    overlap_s: float,
    next_duration_s: float,
    short_turn_s: float = ACTION_SHORT_TURN_S,
    overlap_min_s: float = ACTION_OVERLAP_MIN_S,
) -> str:
    if bool(same_speaker):
        return "NA"

    if float(overlap_s) >= float(overlap_min_s):
        return (
            "BC"
            if float(next_duration_s) <= float(short_turn_s)
            else "ITT"
        )

    return (
        "BI"
        if float(next_duration_s) <= float(short_turn_s)
        else "NTT"
    )

def resolve_event_time(
    action: str,
    prev_end: float,
    next_start: float,
    definition: str = PRIMARY_EVENT_TIME_DEFINITION,
) -> Tuple[float, str]:
    action = str(action)
    definition = str(definition)

    if definition == "floor_release":
        return float(prev_end), "floor_release"

    if definition == "floor_acquisition":
        return float(next_start), "floor_acquisition"

    if definition == "midpoint":
        return (
            float(
                0.5
                * (
                    float(prev_end)
                    + float(next_start)
                )
            ),
            "midpoint",
        )

    if definition == "action_aware_boundary":
        if action in {"ITT", "BC", "BI"}:
            return (
                float(next_start),
                "floor_acquisition",
            )

        return (
            float(prev_end),
            "floor_release",
        )

    raise ValueError(
        "Unknown event-time definition: "
        f"{definition}"
    )

def _event_contract_parameters(
    contract_name: str,
) -> Dict[str, Any]:
    contract_name = str(contract_name)

    if contract_name == "legacy_midpoint":
        return {
            "contract_name": contract_name,
            "event_contract_version": (
                "legacy_midpoint_no_positive_gap_cap"
            ),
            "event_time_definition": "midpoint",
            "gap_min_s": float(PRIMARY_MIN_GAP_S),
            "gap_max_s": None,
            "short_turn_s": float(ACTION_SHORT_TURN_S),
            "overlap_min_s": float(ACTION_OVERLAP_MIN_S),
        }

    if contract_name == "primary":
        return {
            "contract_name": contract_name,
            "event_contract_version": str(
                EVENT_CONTRACT_VERSION
            ),
            "event_time_definition": str(
                PRIMARY_EVENT_TIME_DEFINITION
            ),
            "gap_min_s": float(
                PRIMARY_MIN_GAP_S
            ),
            "gap_max_s": float(
                PRIMARY_MAX_POSITIVE_GAP_S
            ),
            "short_turn_s": float(
                ACTION_SHORT_TURN_S
            ),
            "overlap_min_s": float(
                ACTION_OVERLAP_MIN_S
            ),
        }

    raise ValueError(
        f"Unknown event contract: {contract_name}"
    )

def adjacent_boundary_events(
    turns: pd.DataFrame,
    gap_min: Optional[float] = None,
    gap_max: Optional[float] = None,
    short_turn_s: Optional[float] = None,
    overlap_min_s: Optional[float] = None,
    event_time_definition: Optional[str] = None,
    contract_name: str = "primary",
    keep_out_of_contract: bool = False,
) -> pd.DataFrame:
    if turns is None or len(turns) < 2:
        return _empty_events_df()

    params = _event_contract_parameters(
        contract_name
    )

    if gap_min is None:
        gap_min = params["gap_min_s"]

    if gap_max is None:
        gap_max = params["gap_max_s"]

    if short_turn_s is None:
        short_turn_s = params["short_turn_s"]

    if overlap_min_s is None:
        overlap_min_s = params[
            "overlap_min_s"
        ]

    if event_time_definition is None:
        event_time_definition = params[
            "event_time_definition"
        ]

    frame = turns.copy()

    for column in [
        "start",
        "end",
        "dur",
    ]:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    frame = frame.dropna(
        subset=[
            "start",
            "end",
            "dur",
            "speaker",
        ]
    )

    frame = frame[
        frame["end"] > frame["start"]
    ]

    frame = (
        frame
        .sort_values(
            ["start", "end", "speaker"]
        )
        .reset_index(drop=True)
    )

    if len(frame) < 2:
        return _empty_events_df()

    session = str(
        frame.iloc[0]["session"]
    )

    rows = []

    for index in range(len(frame) - 1):
        previous = frame.iloc[index]
        following = frame.iloc[index + 1]

        prev_speaker = str(
            previous["speaker"]
        )
        next_speaker = str(
            following["speaker"]
        )

        prev_start = float(
            previous["start"]
        )
        prev_end = float(
            previous["end"]
        )
        next_start = float(
            following["start"]
        )
        next_end = float(
            following["end"]
        )

        prev_duration = max(
            0.0,
            prev_end - prev_start,
        )
        next_duration = max(
            0.0,
            next_end - next_start,
        )

        gap = float(
            next_start - prev_end
        )
        overlap = max(
            0.0,
            float(prev_end - next_start),
        )

        same_speaker = (
            prev_speaker == next_speaker
        )

        action = classify_action_from_adjacent_turns(
            same_speaker=same_speaker,
            overlap_s=overlap,
            next_duration_s=next_duration,
            short_turn_s=float(short_turn_s),
            overlap_min_s=float(overlap_min_s),
        )

        within_min = (
            gap >= float(gap_min)
            if gap_min is not None
            else True
        )
        within_max = (
            gap <= float(gap_max)
            if gap_max is not None
            else True
        )

        is_primary = bool(
            within_min and within_max
        )

        if (
            not keep_out_of_contract
            and not is_primary
        ):
            continue

        event_time, anchor_role = (
            resolve_event_time(
                action=action,
                prev_end=prev_end,
                next_start=next_start,
                definition=str(
                    event_time_definition
                ),
            )
        )

        if (
            not np.isfinite(event_time)
            or event_time < 0
        ):
            continue

        interval_start = min(
            prev_end,
            next_start,
        )
        interval_end = max(
            prev_end,
            next_start,
        )

        rows.append({
            "session": session,
            "time": float(event_time),
            "time_floor_release": float(
                prev_end
            ),
            "time_floor_acquisition": float(
                next_start
            ),
            "time_midpoint": float(
                0.5
                * (
                    prev_end
                    + next_start
                )
            ),
            "transition_interval_start": float(
                interval_start
            ),
            "transition_interval_end": float(
                interval_end
            ),
            "transition_width_s": float(
                interval_end
                - interval_start
            ),
            "event_time_definition": str(
                event_time_definition
            ),
            "event_anchor_role": str(
                anchor_role
            ),
            "event_contract_version": str(
                params[
                    "event_contract_version"
                ]
            ),
            "contract_name": str(
                contract_name
            ),
            "is_primary_contract_event": bool(
                is_primary
            ),
            "is_delayed_transition": bool(
                gap_max is not None
                and gap > float(gap_max)
            ),
            "prev_end": float(prev_end),
            "next_start": float(next_start),
            "prev_speaker": prev_speaker,
            "next_speaker": next_speaker,
            "gap": float(gap),
            "overlap": float(overlap),
            "prev_dur": float(
                prev_duration
            ),
            "next_dur": float(
                next_duration
            ),
            "action": str(action),
            "action_id": int(
                ACTION_TO_ID[action]
            ),
            "label": int(
                action in SHIFT_ACTIONS
            ),
            "source_event_type": (
                "adjacent_rttm_boundary"
            ),
        })

    if not rows:
        return _empty_events_df()

    output = pd.DataFrame(rows)

    for column in EVENT_COLUMNS:
        if column not in output.columns:
            output[column] = np.nan

    return (
        output[EVENT_COLUMNS]
        .sort_values(
            ["session", "time"]
        )
        .reset_index(drop=True)
    )

def build_midturn_hold_negatives(
    turns: pd.DataFrame,
    audio_duration: float,
    max_per_session: int,
    min_from_event_s: float = 1.00,
    margin_s: float = 0.50,
    seed: int = 101,
) -> pd.DataFrame:
    if (
        turns is None
        or len(turns) == 0
        or int(max_per_session) <= 0
    ):
        return _empty_events_df()

    frame = turns.copy()

    for column in [
        "start",
        "end",
        "dur",
    ]:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    frame = frame.dropna(
        subset=[
            "start",
            "end",
            "speaker",
        ]
    )

    frame = frame[
        frame["end"] > frame["start"]
    ]

    frame = (
        frame
        .sort_values(
            ["start", "end"]
        )
        .reset_index(drop=True)
    )

    if len(frame) == 0:
        return _empty_events_df()

    session = str(
        frame.iloc[0]["session"]
    )

    rng = random.Random(
        _stable_session_seed(
            session,
            seed,
        )
    )

    boundary_events = (
        adjacent_boundary_events(
            frame,
            contract_name="primary",
        )
    )

    event_times = (
        boundary_events[
            "time"
        ].to_numpy(
            dtype=np.float64
        )
        if len(boundary_events)
        else np.array(
            [],
            dtype=np.float64,
        )
    )

    candidate_turns = []

    for row in frame.itertuples(
        index=False
    ):
        start = float(row.start)
        end = float(row.end)
        duration = end - start

        low = max(
            0.0,
            start + float(margin_s),
        )
        high = min(
            float(audio_duration),
            end - float(margin_s),
        )

        if (
            high <= low
            or duration
            < 2.0 * float(margin_s)
            + 0.20
        ):
            continue

        candidate_turns.append(
            (
                low,
                high,
                str(row.speaker),
            )
        )

    if not candidate_turns:
        return _empty_events_df()

    rows = []
    attempts = 0
    max_attempts = max(
        200,
        int(max_per_session) * 25,
    )

    while (
        len(rows) < int(max_per_session)
        and attempts < max_attempts
    ):
        attempts += 1

        low, high, speaker = rng.choice(
            candidate_turns
        )

        event_time = rng.uniform(
            low,
            high,
        )

        if (
            len(event_times)
            and float(
                np.min(
                    np.abs(
                        event_times
                        - event_time
                    )
                )
            )
            < float(min_from_event_s)
        ):
            continue

        rows.append({
            "session": session,
            "time": float(event_time),
            "time_floor_release": np.nan,
            "time_floor_acquisition": np.nan,
            "time_midpoint": float(
                event_time
            ),
            "transition_interval_start": float(
                event_time
            ),
            "transition_interval_end": float(
                event_time
            ),
            "transition_width_s": 0.0,
            "event_time_definition": (
                "midturn_negative"
            ),
            "event_anchor_role": (
                "within_turn"
            ),
            "event_contract_version": str(
                EVENT_CONTRACT_VERSION
            ),
            "contract_name": "primary",
            "is_primary_contract_event": True,
            "is_delayed_transition": False,
            "prev_end": np.nan,
            "next_start": np.nan,
            "prev_speaker": speaker,
            "next_speaker": speaker,
            "gap": np.nan,
            "overlap": 0.0,
            "prev_dur": np.nan,
            "next_dur": np.nan,
            "action": "NA",
            "action_id": int(
                ACTION_TO_ID["NA"]
            ),
            "label": 0,
            "source_event_type": (
                "midturn_hold_negative"
            ),
        })

    if not rows:
        return _empty_events_df()

    output = pd.DataFrame(rows)

    return (
        output[EVENT_COLUMNS]
        .sort_values(
            ["session", "time"]
        )
        .reset_index(drop=True)
    )

def _real_event_cache_config(
    sessions: List[str],
    split: str,
    cache_name: str,
    add_midturn_negatives: bool,
    contract_name: str,
) -> Dict[str, Any]:
    params = _event_contract_parameters(
        contract_name
    )

    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(
            params[
                "event_contract_version"
            ]
        ),
        "contract_name": str(
            contract_name
        ),
        "cache_name": str(cache_name),
        "split": str(split),
        "sessions": list(
            map(str, sessions)
        ),
        "gap_min_s": params[
            "gap_min_s"
        ],
        "gap_max_s": params[
            "gap_max_s"
        ],
        "event_time_definition": str(
            params[
                "event_time_definition"
            ]
        ),
        "short_turn_s": float(
            params[
                "short_turn_s"
            ]
        ),
        "overlap_min_s": float(
            params[
                "overlap_min_s"
            ]
        ),
        "action_names": list(
            ACTION_NAMES
        ),
        "shift_actions": sorted(
            list(SHIFT_ACTIONS)
        ),
        "hold_actions": sorted(
            list(HOLD_ACTIONS)
        ),
        "add_midturn_negatives": bool(
            add_midturn_negatives
        ),
        "midturn_neg_to_boundary_ratio": float(
            MIDTURN_NEG_TO_BOUNDARY_RATIO
        ),
        "midturn_neg_max_per_session": int(
            MIDTURN_NEG_MAX_PER_SESSION
        ),
        "midturn_neg_min_from_event_s": float(
            MIDTURN_NEG_MIN_FROM_EVENT_S
        ),
        "weight_real_clean": float(
            WEIGHT_REAL_CLEAN
        ),
        "weight_real_midturn": float(
            WEIGHT_REAL_MIDTURN
        ),
        "cache_format": "pickle",
    }

def _cache_config_matches(
    meta_path: Path,
    expected_config: Dict[str, Any],
) -> bool:
    if not Path(meta_path).exists():
        return False

    try:
        previous = json.loads(
            Path(meta_path).read_text()
        )

        return (
            previous
            == expected_config
        )
    except Exception:
        return False

def build_real_events_for_sessions(
    sessions: List[str],
    split: str,
    cache_name: str,
    force: bool = False,
    add_midturn_negatives: bool = False,
    contract_name: str = "primary",
) -> pd.DataFrame:
    cache_path = (
        CACHE_DIR
        / f"{cache_name}_events.pkl"
    )
    meta_path = (
        CACHE_DIR
        / f"{cache_name}_events.meta.json"
    )
    legacy_parquet_path = (
        CACHE_DIR
        / f"{cache_name}_events.parquet"
    )

    expected_config = (
        _real_event_cache_config(
            sessions=sessions,
            split=split,
            cache_name=cache_name,
            add_midturn_negatives=(
                add_midturn_negatives
            ),
            contract_name=contract_name,
        )
    )

    if (
        cache_path.exists()
        and _cache_config_matches(
            meta_path,
            expected_config,
        )
        and not force
    ):
        output = pd.read_pickle(
            cache_path
        )

        for column in (
            EVENT_COLUMNS
            + REAL_EVENT_EXTRA_COLUMNS
        ):
            if column not in output.columns:
                output[column] = np.nan

        return output[
            EVENT_COLUMNS
            + REAL_EVENT_EXTRA_COLUMNS
        ]

    all_rows = []

    for session in tqdm(
        sessions,
        desc=(
            "Build real events: "
            f"{cache_name}"
        ),
    ):
        turns = read_rttm_as_turns(
            get_session_rttm_path(
                session,
                split,
            )
        )

        boundary_events = (
            adjacent_boundary_events(
                turns,
                contract_name=contract_name,
            )
        )

        examples = boundary_events.copy()

        if (
            add_midturn_negatives
            and contract_name == "primary"
            and bool(
                globals().get(
                    "ADD_MIDTURN_HOLD_NEGATIVES",
                    False,
                )
            )
        ):
            try:
                audio_duration = (
                    get_audio_duration(
                        get_session_audio_path(
                            session,
                            split,
                        )
                    )
                )

                ratio_cap = int(
                    math.ceil(
                        float(
                            MIDTURN_NEG_TO_BOUNDARY_RATIO
                        )
                        * max(
                            1,
                            len(
                                boundary_events
                            ),
                        )
                    )
                )

                session_cap = min(
                    int(
                        MIDTURN_NEG_MAX_PER_SESSION
                    ),
                    ratio_cap,
                )

                midturn = (
                    build_midturn_hold_negatives(
                        turns=turns,
                        audio_duration=audio_duration,
                        max_per_session=session_cap,
                        min_from_event_s=float(
                            MIDTURN_NEG_MIN_FROM_EVENT_S
                        ),
                        seed=int(SEED),
                    )
                )

                examples = pd.concat(
                    [
                        examples,
                        midturn,
                    ],
                    ignore_index=True,
                )
            except Exception as exc:
                print(
                    "⚠️ Mid-turn negative "
                    f"sampling failed for "
                    f"{session}: {exc}"
                )

        if len(examples) == 0:
            continue

        examples["split"] = str(
            cache_name
        )
        examples["audio_path"] = str(
            get_session_audio_path(
                session,
                split,
            )
        )
        examples["rttm_path"] = str(
            get_session_rttm_path(
                session,
                split,
            )
        )
        examples["is_synthetic"] = False
        examples[
            "is_diffusion_augmented"
        ] = False

        examples["sample_weight"] = np.where(
            examples[
                "source_event_type"
            ]
            .astype(str)
            .eq(
                "midturn_hold_negative"
            ),
            float(
                WEIGHT_REAL_MIDTURN
            ),
            float(
                WEIGHT_REAL_CLEAN
            ),
        )

        all_rows.append(examples)

    if all_rows:
        output = pd.concat(
            all_rows,
            ignore_index=True,
        )
    else:
        output = _empty_events_df(
            include_real_extra=True
        )

    for column in (
        EVENT_COLUMNS
        + REAL_EVENT_EXTRA_COLUMNS
    ):
        if column not in output.columns:
            output[column] = np.nan

    output = output[
        EVENT_COLUMNS
        + REAL_EVENT_EXTRA_COLUMNS
    ]

    output = (
        output
        .sort_values(
            ["session", "time"]
        )
        .reset_index(drop=True)
    )

    output.to_pickle(
        cache_path
    )

    meta_path.write_text(
        json.dumps(
            expected_config,
            indent=2,
            sort_keys=True,
        )
    )

    try:
        legacy_parquet_path.unlink(
            missing_ok=True
        )
    except Exception:
        pass

    return output

def _primary_events_for_one_session(
    session: str,
    split: str,
) -> pd.DataFrame:
    turns = read_rttm_as_turns(
        get_session_rttm_path(
            session,
            split,
        )
    )

    return adjacent_boundary_events(
        turns,
        contract_name="primary",
    )

def get_gt_shift_times(
    session: str,
    split: str = "test",
) -> np.ndarray:
    events = _primary_events_for_one_session(
        session,
        split,
    )

    if len(events) == 0:
        return np.array(
            [],
            dtype=np.float32,
        )

    return events.loc[
        events["label"] == 1,
        "time",
    ].to_numpy(
        dtype=np.float32
    )

def get_gt_shift_intervals(
    session: str,
    split: str = "test",
) -> np.ndarray:
    events = _primary_events_for_one_session(
        session,
        split,
    )

    events = events[
        events["label"] == 1
    ]

    if len(events) == 0:
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )

    return events[
        [
            "transition_interval_start",
            "transition_interval_end",
        ]
    ].to_numpy(
        dtype=np.float32
    )

def match_event_times(
    pred_times,
    gt_times,
    tol: float = 0.50,
    prefer_min_cost: bool = True,
):
    pred = np.array(
        sorted(
            [
                float(value)
                for value in pred_times
                if np.isfinite(value)
            ]
        ),
        dtype=np.float64,
    )

    gt = np.array(
        sorted(
            [
                float(value)
                for value in gt_times
                if np.isfinite(value)
            ]
        ),
        dtype=np.float64,
    )

    if len(pred) == 0 and len(gt) == 0:
        return {
            "P": 0.0,
            "R": 0.0,
            "F1": 0.0,
            "TP": 0,
            "FP": 0,
            "FN": 0,
            "N_pred": 0,
            "N_gt": 0,
            "matches": [],
        }

    if len(pred) == 0:
        return {
            "P": 0.0,
            "R": 0.0,
            "F1": 0.0,
            "TP": 0,
            "FP": 0,
            "FN": int(len(gt)),
            "N_pred": 0,
            "N_gt": int(len(gt)),
            "matches": [],
        }

    if len(gt) == 0:
        return {
            "P": 0.0,
            "R": 0.0,
            "F1": 0.0,
            "TP": 0,
            "FP": int(len(pred)),
            "FN": 0,
            "N_pred": int(len(pred)),
            "N_gt": 0,
            "matches": [],
        }

    matches = []

    if prefer_min_cost:
        try:
            from scipy.optimize import (
                linear_sum_assignment,
            )

            cost = np.abs(
                pred[:, None]
                - gt[None, :]
            )

            large = float(tol) + 1e6

            masked = np.where(
                cost <= float(tol),
                cost,
                large,
            )

            row_indices, column_indices = (
                linear_sum_assignment(
                    masked
                )
            )

            for (
                row_index,
                column_index,
            ) in zip(
                row_indices,
                column_indices,
            ):
                distance = float(
                    cost[
                        row_index,
                        column_index,
                    ]
                )

                if distance <= float(tol):
                    matches.append(
                        (
                            float(
                                pred[
                                    row_index
                                ]
                            ),
                            float(
                                gt[
                                    column_index
                                ]
                            ),
                            distance,
                        )
                    )
        except Exception:
            matches = []

    if not matches:
        pred_index = 0
        gt_index = 0

        while (
            pred_index < len(pred)
            and gt_index < len(gt)
        ):
            difference = (
                pred[pred_index]
                - gt[gt_index]
            )

            if (
                abs(difference)
                <= float(tol)
            ):
                matches.append(
                    (
                        float(
                            pred[
                                pred_index
                            ]
                        ),
                        float(
                            gt[
                                gt_index
                            ]
                        ),
                        float(
                            abs(
                                difference
                            )
                        ),
                    )
                )
                pred_index += 1
                gt_index += 1
            elif (
                pred[pred_index]
                < gt[gt_index]
                - float(tol)
            ):
                pred_index += 1
            else:
                gt_index += 1

    true_positive = len(matches)
    false_positive = (
        len(pred)
        - true_positive
    )
    false_negative = (
        len(gt)
        - true_positive
    )

    precision = (
        true_positive
        / max(
            1,
            true_positive
            + false_positive,
        )
    )
    recall = (
        true_positive
        / max(
            1,
            true_positive
            + false_negative,
        )
    )
    f1 = (
        2.0
        * precision
        * recall
        / max(
            1e-12,
            precision
            + recall,
        )
    )

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        "TP": int(
            true_positive
        ),
        "FP": int(
            false_positive
        ),
        "FN": int(
            false_negative
        ),
        "N_pred": int(
            len(pred)
        ),
        "N_gt": int(
            len(gt)
        ),
        "matches": matches,
    }

def _distance_to_interval(
    point: float,
    start: float,
    end: float,
) -> float:
    raw_start = float(start)
    raw_end = float(end)
    start = min(raw_start, raw_end)
    end = max(raw_start, raw_end)
    point = float(point)

    if point < start:
        return start - point

    if point > end:
        return point - end

    return 0.0

def match_event_times_to_intervals(
    pred_times,
    gt_intervals,
    tol: float = 0.50,
):
    pred = np.array(
        sorted(
            [
                float(value)
                for value in pred_times
                if np.isfinite(value)
            ]
        ),
        dtype=np.float64,
    )

    intervals = np.asarray(
        gt_intervals,
        dtype=np.float64,
    ).reshape(-1, 2)

    intervals = intervals[
        np.isfinite(
            intervals
        ).all(axis=1)
    ]

    if len(pred) == 0 or len(intervals) == 0:
        true_positive = 0
        false_positive = len(pred)
        false_negative = len(intervals)
        matches = []
    else:
        cost = np.zeros(
            (
                len(pred),
                len(intervals),
            ),
            dtype=np.float64,
        )

        for pred_index, point in enumerate(
            pred
        ):
            for (
                interval_index,
                interval,
            ) in enumerate(intervals):
                cost[
                    pred_index,
                    interval_index,
                ] = _distance_to_interval(
                    point,
                    interval[0],
                    interval[1],
                )

        try:
            from scipy.optimize import (
                linear_sum_assignment,
            )

            large = float(tol) + 1e6

            masked = np.where(
                cost <= float(tol),
                cost,
                large,
            )

            row_indices, column_indices = (
                linear_sum_assignment(
                    masked
                )
            )

            matches = []

            for (
                row_index,
                column_index,
            ) in zip(
                row_indices,
                column_indices,
            ):
                distance = float(
                    cost[
                        row_index,
                        column_index,
                    ]
                )

                if distance <= float(tol):
                    matches.append(
                        (
                            float(
                                pred[
                                    row_index
                                ]
                            ),
                            float(
                                intervals[
                                    column_index,
                                    0,
                                ]
                            ),
                            float(
                                intervals[
                                    column_index,
                                    1,
                                ]
                            ),
                            distance,
                        )
                    )
        except Exception:
            candidates = []

            for pred_index in range(
                len(pred)
            ):
                for interval_index in range(
                    len(intervals)
                ):
                    distance = float(
                        cost[
                            pred_index,
                            interval_index,
                        ]
                    )

                    if distance <= float(tol):
                        candidates.append(
                            (
                                distance,
                                pred_index,
                                interval_index,
                            )
                        )

            candidates.sort()
            used_pred = set()
            used_interval = set()
            matches = []

            for (
                distance,
                pred_index,
                interval_index,
            ) in candidates:
                if (
                    pred_index in used_pred
                    or interval_index
                    in used_interval
                ):
                    continue

                used_pred.add(
                    pred_index
                )
                used_interval.add(
                    interval_index
                )

                matches.append(
                    (
                        float(
                            pred[
                                pred_index
                            ]
                        ),
                        float(
                            intervals[
                                interval_index,
                                0,
                            ]
                        ),
                        float(
                            intervals[
                                interval_index,
                                1,
                            ]
                        ),
                        float(distance),
                    )
                )

        true_positive = len(matches)
        false_positive = (
            len(pred)
            - true_positive
        )
        false_negative = (
            len(intervals)
            - true_positive
        )

    precision = (
        true_positive
        / max(
            1,
            true_positive
            + false_positive,
        )
    )
    recall = (
        true_positive
        / max(
            1,
            true_positive
            + false_negative,
        )
    )
    f1 = (
        2.0
        * precision
        * recall
        / max(
            1e-12,
            precision
            + recall,
        )
    )

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        "TP": int(
            true_positive
        ),
        "FP": int(
            false_positive
        ),
        "FN": int(
            false_negative
        ),
        "N_pred": int(
            len(pred)
        ),
        "N_gt": int(
            len(intervals)
        ),
        "matches": matches,
    }

def event_metrics_from_predictions(
    pred_df: pd.DataFrame,
    sessions: List[str],
    split: str = "test",
    tol: float = 0.50,
) -> Dict[str, Any]:
    if pred_df is None or len(pred_df) == 0:
        pred_df = pd.DataFrame(
            columns=[
                "session",
                "time",
            ]
        )

    predictions = pred_df.copy()
    predictions["session"] = (
        predictions["session"]
        .astype(str)
    )
    predictions["time"] = pd.to_numeric(
        predictions["time"],
        errors="coerce",
    )
    predictions = predictions.dropna(
        subset=["time"]
    )
    predictions = (
        predictions
        .drop_duplicates(
            subset=[
                "session",
                "time",
            ]
        )
        .reset_index(drop=True)
    )

    total = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "N_pred": 0,
        "N_gt": 0,
    }
    per_session = []

    for session in sessions:
        pred_times = predictions.loc[
            predictions["session"]
            .eq(str(session)),
            "time",
        ].to_numpy(
            dtype=np.float32
        )

        gt_times = get_gt_shift_times(
            session,
            split=split,
        )

        metrics = match_event_times(
            pred_times,
            gt_times,
            tol=float(tol),
            prefer_min_cost=True,
        )

        for key in total:
            total[key] += int(
                metrics[key]
            )

        per_session.append({
            "session": str(session),
            **{
                key: value
                for key, value
                in metrics.items()
                if key != "matches"
            },
        })

    true_positive = total["TP"]
    false_positive = total["FP"]
    false_negative = total["FN"]

    precision = (
        true_positive
        / max(
            1,
            true_positive
            + false_positive,
        )
    )
    recall = (
        true_positive
        / max(
            1,
            true_positive
            + false_negative,
        )
    )
    f1 = (
        2.0
        * precision
        * recall
        / max(
            1e-12,
            precision
            + recall,
        )
    )

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        **total,
        "per_session": pd.DataFrame(
            per_session
        ),
    }

def interval_metrics_from_predictions(
    pred_df: pd.DataFrame,
    sessions: List[str],
    split: str = "test",
    tol: float = 0.50,
) -> Dict[str, Any]:
    if pred_df is None or len(pred_df) == 0:
        pred_df = pd.DataFrame(
            columns=[
                "session",
                "time",
            ]
        )

    predictions = pred_df.copy()
    predictions["session"] = (
        predictions["session"]
        .astype(str)
    )
    predictions["time"] = pd.to_numeric(
        predictions["time"],
        errors="coerce",
    )
    predictions = predictions.dropna(
        subset=[
            "session",
            "time",
        ]
    )
    predictions = predictions.drop_duplicates(
        [
            "session",
            "time",
        ]
    )

    total = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "N_pred": 0,
        "N_gt": 0,
    }
    per_session = []

    for session in sessions:
        pred_times = predictions.loc[
            predictions["session"]
            .eq(str(session)),
            "time",
        ].to_numpy(
            dtype=np.float32
        )

        intervals = get_gt_shift_intervals(
            session,
            split=split,
        )

        metrics = (
            match_event_times_to_intervals(
                pred_times,
                intervals,
                tol=float(tol),
            )
        )

        for key in total:
            total[key] += int(
                metrics[key]
            )

        per_session.append({
            "session": str(session),
            **{
                key: value
                for key, value
                in metrics.items()
                if key != "matches"
            },
        })

    true_positive = total["TP"]
    false_positive = total["FP"]
    false_negative = total["FN"]

    precision = (
        true_positive
        / max(
            1,
            true_positive
            + false_positive,
        )
    )
    recall = (
        true_positive
        / max(
            1,
            true_positive
            + false_negative,
        )
    )
    f1 = (
        2.0
        * precision
        * recall
        / max(
            1e-12,
            precision
            + recall,
        )
    )

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        **total,
        "per_session": pd.DataFrame(
            per_session
        ),
    }

for parquet_path in CACHE_DIR.glob(
    "*_events.parquet"
):
    try:
        parquet_path.unlink()
        print(
            "Removed legacy parquet "
            "event cache:",
            parquet_path,
        )
    except Exception as exc:
        print(
            "Could not remove legacy "
            "parquet cache:",
            parquet_path,
            exc,
        )

REAL_TRAIN_BOUNDARY_EVENTS = (
    build_real_events_for_sessions(
        TRAIN_SESSIONS,
        "dev",
        "train_primary_boundaries",
        FORCE_REBUILD_REAL_EVENTS,
        add_midturn_negatives=False,
        contract_name="primary",
    )
)

REAL_TRAIN_EVENTS = (
    build_real_events_for_sessions(
        TRAIN_SESSIONS,
        "dev",
        "train_primary",
        FORCE_REBUILD_REAL_EVENTS,
        add_midturn_negatives=bool(
            ADD_MIDTURN_HOLD_NEGATIVES
        ),
        contract_name="primary",
    )
)

REAL_VAL_EVENTS = (
    build_real_events_for_sessions(
        VAL_SESSIONS,
        "dev",
        "val_primary",
        FORCE_REBUILD_REAL_EVENTS,
        add_midturn_negatives=False,
        contract_name="primary",
    )
)

REAL_TEST_EVENTS = (
    build_real_events_for_sessions(
        TEST_SESSIONS,
        "test",
        "test_primary",
        FORCE_REBUILD_REAL_EVENTS,
        add_midturn_negatives=False,
        contract_name="primary",
    )
)

if KEEP_LEGACY_EVENT_TABLES:
    REAL_TRAIN_EVENTS_LEGACY = (
        build_real_events_for_sessions(
            TRAIN_SESSIONS,
            "dev",
            "train_legacy_midpoint",
            FORCE_REBUILD_REAL_EVENTS,
            add_midturn_negatives=False,
            contract_name=(
                "legacy_midpoint"
            ),
        )
    )

    REAL_VAL_EVENTS_LEGACY = (
        build_real_events_for_sessions(
            VAL_SESSIONS,
            "dev",
            "val_legacy_midpoint",
            FORCE_REBUILD_REAL_EVENTS,
            add_midturn_negatives=False,
            contract_name=(
                "legacy_midpoint"
            ),
        )
    )

    REAL_TEST_EVENTS_LEGACY = (
        build_real_events_for_sessions(
            TEST_SESSIONS,
            "test",
            "test_legacy_midpoint",
            FORCE_REBUILD_REAL_EVENTS,
            add_midturn_negatives=False,
            contract_name=(
                "legacy_midpoint"
            ),
        )
    )
else:
    REAL_TRAIN_EVENTS_LEGACY = (
        _empty_events_df(
            include_real_extra=True
        )
    )
    REAL_VAL_EVENTS_LEGACY = (
        _empty_events_df(
            include_real_extra=True
        )
    )
    REAL_TEST_EVENTS_LEGACY = (
        _empty_events_df(
            include_real_extra=True
        )
    )

DELAYED_TRAIN_BOUNDARIES = (
    REAL_TRAIN_EVENTS_LEGACY[
        pd.to_numeric(
            REAL_TRAIN_EVENTS_LEGACY[
                "gap"
            ],
            errors="coerce",
        )
        > float(
            PRIMARY_MAX_POSITIVE_GAP_S
        )
    ]
    .copy()
)

DELAYED_VAL_BOUNDARIES = (
    REAL_VAL_EVENTS_LEGACY[
        pd.to_numeric(
            REAL_VAL_EVENTS_LEGACY[
                "gap"
            ],
            errors="coerce",
        )
        > float(
            PRIMARY_MAX_POSITIVE_GAP_S
        )
    ]
    .copy()
)

DELAYED_TEST_BOUNDARIES = (
    REAL_TEST_EVENTS_LEGACY[
        pd.to_numeric(
            REAL_TEST_EVENTS_LEGACY[
                "gap"
            ],
            errors="coerce",
        )
        > float(
            PRIMARY_MAX_POSITIVE_GAP_S
        )
    ]
    .copy()
)

def _event_summary_row(
    split: str,
    contract: str,
    frame: pd.DataFrame,
) -> Dict[str, Any]:
    frame = (
        frame.copy()
        if frame is not None
        else pd.DataFrame()
    )

    if len(frame) == 0:
        return {
            "split": split,
            "contract": contract,
            "rows": 0,
            "boundary_rows": 0,
            "midturn_rows": 0,
            "shifts": 0,
            "holds": 0,
            "shift_rate": np.nan,
            "weighted_shift_rate": np.nan,
        }

    labels = pd.to_numeric(
        frame["label"],
        errors="coerce",
    ).fillna(0)

    weights = pd.to_numeric(
        frame.get(
            "sample_weight",
            1.0,
        ),
        errors="coerce",
    ).fillna(1.0)

    source = frame[
        "source_event_type"
    ].astype(str)

    weighted_shift_rate = (
        float(
            np.sum(
                weights
                * labels
            )
            / max(
                1e-12,
                np.sum(weights),
            )
        )
    )

    return {
        "split": str(split),
        "contract": str(contract),
        "rows": int(len(frame)),
        "boundary_rows": int(
            source.eq(
                "adjacent_rttm_boundary"
            ).sum()
        ),
        "midturn_rows": int(
            source.eq(
                "midturn_hold_negative"
            ).sum()
        ),
        "shifts": int(
            labels.sum()
        ),
        "holds": int(
            len(frame)
            - labels.sum()
        ),
        "shift_rate": float(
            labels.mean()
        ),
        "weighted_shift_rate": (
            weighted_shift_rate
        ),
    }

EVENT_CONTRACT_SUMMARY_DF = pd.DataFrame([
    _event_summary_row(
        "train",
        "primary_training",
        REAL_TRAIN_EVENTS,
    ),
    _event_summary_row(
        "train",
        "primary_boundaries",
        REAL_TRAIN_BOUNDARY_EVENTS,
    ),
    _event_summary_row(
        "val",
        "primary",
        REAL_VAL_EVENTS,
    ),
    _event_summary_row(
        "test",
        "primary",
        REAL_TEST_EVENTS,
    ),
    _event_summary_row(
        "train",
        "legacy_midpoint",
        REAL_TRAIN_EVENTS_LEGACY,
    ),
    _event_summary_row(
        "val",
        "legacy_midpoint",
        REAL_VAL_EVENTS_LEGACY,
    ),
    _event_summary_row(
        "test",
        "legacy_midpoint",
        REAL_TEST_EVENTS_LEGACY,
    ),
])

EVENT_CONTRACT_SUMMARY_PATH = (
    CONTRACT_DIR
    / "event_contract_summary.csv"
)

EVENT_CONTRACT_SUMMARY_DF.to_csv(
    EVENT_CONTRACT_SUMMARY_PATH,
    index=False,
)

DELAYED_EVENT_SUMMARY_DF = pd.DataFrame([
    {
        "split": "train",
        "delayed_boundaries": int(
            len(
                DELAYED_TRAIN_BOUNDARIES
            )
        ),
        "delayed_shifts": int(
            pd.to_numeric(
                DELAYED_TRAIN_BOUNDARIES[
                    "label"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        ),
    },
    {
        "split": "val",
        "delayed_boundaries": int(
            len(
                DELAYED_VAL_BOUNDARIES
            )
        ),
        "delayed_shifts": int(
            pd.to_numeric(
                DELAYED_VAL_BOUNDARIES[
                    "label"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        ),
    },
    {
        "split": "test",
        "delayed_boundaries": int(
            len(
                DELAYED_TEST_BOUNDARIES
            )
        ),
        "delayed_shifts": int(
            pd.to_numeric(
                DELAYED_TEST_BOUNDARIES[
                    "label"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        ),
    },
])

DELAYED_EVENT_SUMMARY_PATH = (
    CONTRACT_DIR
    / "delayed_event_summary.csv"
)

DELAYED_EVENT_SUMMARY_DF.to_csv(
    DELAYED_EVENT_SUMMARY_PATH,
    index=False,
)

EVENT_CONTRACT = {
    "run_version": str(RUN_VERSION),
    "event_contract_version": str(
        EVENT_CONTRACT_VERSION
    ),
    "primary_event_time_definition": str(
        PRIMARY_EVENT_TIME_DEFINITION
    ),
    "primary_min_gap_s": float(
        PRIMARY_MIN_GAP_S
    ),
    "primary_max_positive_gap_s": float(
        PRIMARY_MAX_POSITIVE_GAP_S
    ),
    "short_turn_s": float(
        ACTION_SHORT_TURN_S
    ),
    "overlap_min_s": float(
        ACTION_OVERLAP_MIN_S
    ),
    "shift_actions": sorted(
        list(SHIFT_ACTIONS)
    ),
    "hold_actions": sorted(
        list(HOLD_ACTIONS)
    ),
    "midturn_neg_to_boundary_ratio": float(
        MIDTURN_NEG_TO_BOUNDARY_RATIO
    ),
    "midturn_neg_max_per_session": int(
        MIDTURN_NEG_MAX_PER_SESSION
    ),
    "midturn_negative_weight": float(
        WEIGHT_REAL_MIDTURN
    ),
    "selection_basis": str(
        EVENT_CONTRACT_SELECTION_BASIS
    ),
    "test_used_for_selection": False,
    "legacy_midpoint_retained": bool(
        KEEP_LEGACY_EVENT_TABLES
    ),
    "secondary_interval_reporting": bool(
        REPORT_TRANSITION_INTERVAL_METRICS
    ),
}

EVENT_CONTRACT_PATH = (
    CONTRACT_DIR
    / "event_contract.json"
)

EVENT_CONTRACT_PATH.write_text(
    json.dumps(
        EVENT_CONTRACT,
        indent=2,
        sort_keys=True,
    )
)

def _print_event_summary(
    name: str,
    frame: pd.DataFrame,
) -> None:
    if frame is None or len(frame) == 0:
        print(
            name,
            (0, 0),
            {},
        )
        return

    action_counts = (
        frame["action"]
        .value_counts(
            dropna=False
        )
        .to_dict()
    )
    source_counts = (
        frame[
            "source_event_type"
        ]
        .value_counts(
            dropna=False
        )
        .to_dict()
    )

    print(
        name,
        frame.shape,
    )
    print(
        "  action:",
        action_counts,
    )
    print(
        "  source:",
        source_counts,
    )
    print(
        "  shift_rate:",
        round(
            float(
                pd.to_numeric(
                    frame["label"],
                    errors="coerce",
                )
                .fillna(0)
                .mean()
            ),
            4,
        ),
    )

_print_event_summary(
    "REAL_TRAIN_EVENTS",
    REAL_TRAIN_EVENTS,
)
_print_event_summary(
    "REAL_VAL_EVENTS",
    REAL_VAL_EVENTS,
)
_print_event_summary(
    "REAL_TEST_EVENTS",
    REAL_TEST_EVENTS,
)

print(
    "\nPaper-ready event contract:"
)
print(
    json.dumps(
        EVENT_CONTRACT,
        indent=2,
        sort_keys=True,
    )
)

display(
    EVENT_CONTRACT_SUMMARY_DF
)

display(
    DELAYED_EVENT_SUMMARY_DF
)

globals()["REAL_TRAIN_BOUNDARY_EVENTS"] = (
    REAL_TRAIN_BOUNDARY_EVENTS
)
globals()["REAL_TRAIN_EVENTS"] = (
    REAL_TRAIN_EVENTS
)
globals()["REAL_VAL_EVENTS"] = (
    REAL_VAL_EVENTS
)
globals()["REAL_TEST_EVENTS"] = (
    REAL_TEST_EVENTS
)
globals()["REAL_TRAIN_EVENTS_LEGACY"] = (
    REAL_TRAIN_EVENTS_LEGACY
)
globals()["REAL_VAL_EVENTS_LEGACY"] = (
    REAL_VAL_EVENTS_LEGACY
)
globals()["REAL_TEST_EVENTS_LEGACY"] = (
    REAL_TEST_EVENTS_LEGACY
)
globals()["DELAYED_TRAIN_BOUNDARIES"] = (
    DELAYED_TRAIN_BOUNDARIES
)
globals()["DELAYED_VAL_BOUNDARIES"] = (
    DELAYED_VAL_BOUNDARIES
)
globals()["DELAYED_TEST_BOUNDARIES"] = (
    DELAYED_TEST_BOUNDARIES
)
globals()["EVENT_CONTRACT"] = (
    EVENT_CONTRACT
)
globals()["EVENT_CONTRACT_PATH"] = (
    EVENT_CONTRACT_PATH
)
globals()["EVENT_CONTRACT_SUMMARY_DF"] = (
    EVENT_CONTRACT_SUMMARY_DF
)
globals()["DELAYED_EVENT_SUMMARY_DF"] = (
    DELAYED_EVENT_SUMMARY_DF
)

REAL_TRAIN_EVENTS (5675, 32)
  action: {'NA': 3384, 'NTT': 1684, 'ITT': 513, 'BI': 62, 'BC': 32}
  source: {'adjacent_rttm_boundary': 3805, 'midturn_hold_negative': 1870}
  shift_rate: 0.3871
REAL_VAL_EVENTS (1109, 32)
  action: {'NA': 617, 'NTT': 338, 'ITT': 129, 'BI': 17, 'BC': 8}
  source: {'adjacent_rttm_boundary': 1109}
  shift_rate: 0.4211
REAL_TEST_EVENTS (11201, 32)
  action: {'NTT': 4575, 'NA': 4417, 'ITT': 1775, 'BI': 300, 'BC': 134}
  source: {'adjacent_rttm_boundary': 11201}
  shift_rate: 0.5669

Paper-ready event contract:
{
  "event_contract_version": "voxconverse_action_aware_immediate_v1",
  "hold_actions": [
    "BC",
    "BI",
    "NA"
  ],
  "legacy_midpoint_retained": true,
  "midturn_neg_max_per_session": 40,
  "midturn_neg_to_boundary_ratio": 0.5,
  "midturn_negative_weight": 0.25,
  "overlap_min_s": 0.06,
  "primary_event_time_definition": "action_aware_boundary",
  "primary_max_positive_gap_s": 1.0,
  "primary_min_gap_s": -1.0,
  "run_version": "18thAug_VAP_RRFA

,split,contract,rows,boundary_rows,midturn_rows,shifts,holds,shift_rate,weighted_shift_rate
0,train,primary_training,5675,3805,1870,2197,3478,0.387137,0.514219
1,train,primary_boundaries,3805,3805,0,2197,1608,0.577398,0.577398
2,val,primary,1109,1109,0,467,642,0.421100,0.421100
3,test,primary,11201,11201,0,6350,4851,0.566914,0.566914
4,train,legacy_midpoint,5351,5351,0,2782,2569,0.519903,0.519903
5,val,legacy_midpoint,1612,1612,0,621,991,0.385236,0.385236
6,test,legacy_midpoint,17164,17164,0,8420,8744,0.490562,0.490562


,split,delayed_boundaries,delayed_shifts
0,train,1546,585
1,val,503,154
2,test,5963,2070


In [8]:
assert PAPER_READY_PROTOCOL is True
assert EVENT_CONTRACT_TEST_USED_FOR_SELECTION is False
assert PRIMARY_EVENT_TIME_DEFINITION in {
    "action_aware_boundary",
    "floor_release",
    "floor_acquisition",
}
assert np.isfinite(PRIMARY_MAX_POSITIVE_GAP_S)
assert PRIMARY_MAX_POSITIVE_GAP_S > 0
assert REUSE_PREVIOUS_SYS1_CHECKPOINT is False

required_primary_columns = {
    "time",
    "time_floor_release",
    "time_floor_acquisition",
    "time_midpoint",
    "transition_interval_start",
    "transition_interval_end",
    "event_time_definition",
    "event_contract_version",
    "is_primary_contract_event",
    "gap",
    "action",
    "label",
}

for name, frame in {
    "REAL_TRAIN_EVENTS": REAL_TRAIN_EVENTS,
    "REAL_VAL_EVENTS": REAL_VAL_EVENTS,
    "REAL_TEST_EVENTS": REAL_TEST_EVENTS,
}.items():
    missing = sorted(
        required_primary_columns
        - set(frame.columns)
    )

    if missing:
        raise KeyError(
            f"{name} is missing event-contract columns: {missing}"
        )

    boundary = frame[
        frame["source_event_type"]
        .astype(str)
        .eq("adjacent_rttm_boundary")
    ]

    if len(boundary):
        observed_definitions = set(
            boundary[
                "event_time_definition"
            ]
            .dropna()
            .astype(str)
            .unique()
        )

        assert observed_definitions == {
            PRIMARY_EVENT_TIME_DEFINITION
        }, (
            f"{name} contains event-time definitions "
            f"{observed_definitions}, expected "
            f"{PRIMARY_EVENT_TIME_DEFINITION!r}."
        )

        assert (
            pd.to_numeric(
                boundary["gap"],
                errors="coerce",
            )
            <= float(
                PRIMARY_MAX_POSITIVE_GAP_S
            )
            + 1e-9
        ).all(), (
            f"{name} contains a positive gap above the "
            "primary immediate-transition cap."
        )

def _action_anchor_check(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    boundary = frame[
        frame["source_event_type"]
        .astype(str)
        .eq("adjacent_rttm_boundary")
    ].copy()

    if len(boundary) == 0:
        return pd.DataFrame()

    boundary["expected_time"] = np.where(
        boundary["action"]
        .astype(str)
        .isin(["ITT", "BC", "BI"]),
        pd.to_numeric(
            boundary[
                "time_floor_acquisition"
            ],
            errors="coerce",
        ),
        pd.to_numeric(
            boundary[
                "time_floor_release"
            ],
            errors="coerce",
        ),
    )

    boundary["anchor_abs_error_s"] = (
        pd.to_numeric(
            boundary["time"],
            errors="coerce",
        )
        - boundary["expected_time"]
    ).abs()

    return (
        boundary
        .groupby("action")
        .agg(
            rows=("session", "count"),
            maximum_anchor_error_s=(
                "anchor_abs_error_s",
                "max",
            ),
            mean_gap_s=("gap", "mean"),
            median_gap_s=("gap", "median"),
        )
        .reset_index()
    )

ACTION_ANCHOR_CHECK_DF = pd.concat(
    [
        _action_anchor_check(
            REAL_TRAIN_BOUNDARY_EVENTS
        ).assign(split="train"),
        _action_anchor_check(
            REAL_VAL_EVENTS
        ).assign(split="val"),
        _action_anchor_check(
            REAL_TEST_EVENTS
        ).assign(split="test"),
    ],
    ignore_index=True,
)

if len(ACTION_ANCHOR_CHECK_DF):
    maximum_error = float(
        pd.to_numeric(
            ACTION_ANCHOR_CHECK_DF[
                "maximum_anchor_error_s"
            ],
            errors="coerce",
        )
        .fillna(0)
        .max()
    )

    assert maximum_error <= 1e-8, (
        "Action-aware timestamp construction failed: "
        f"maximum error={maximum_error}"
    )

ACTION_ANCHOR_CHECK_PATH = (
    CONTRACT_DIR
    / "action_anchor_check.csv"
)

ACTION_ANCHOR_CHECK_DF.to_csv(
    ACTION_ANCHOR_CHECK_PATH,
    index=False,
)

contract_compare_rows = []

for split, primary, legacy in [
    (
        "train",
        REAL_TRAIN_BOUNDARY_EVENTS,
        REAL_TRAIN_EVENTS_LEGACY,
    ),
    (
        "val",
        REAL_VAL_EVENTS,
        REAL_VAL_EVENTS_LEGACY,
    ),
    (
        "test",
        REAL_TEST_EVENTS,
        REAL_TEST_EVENTS_LEGACY,
    ),
]:
    primary_shift = primary[
        pd.to_numeric(
            primary["label"],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
        .eq(1)
    ].copy()

    legacy_shift = legacy[
        pd.to_numeric(
            legacy["label"],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
        .eq(1)
    ].copy()

    contract_compare_rows.append({
        "split": split,
        "primary_boundaries": int(
            len(primary)
        ),
        "primary_shifts": int(
            len(primary_shift)
        ),
        "legacy_boundaries": int(
            len(legacy)
        ),
        "legacy_shifts": int(
            len(legacy_shift)
        ),
        "excluded_delayed_boundaries": int(
            len(legacy)
            - len(primary)
        ),
        "excluded_delayed_shifts": int(
            len(legacy_shift)
            - len(primary_shift)
        ),
        "primary_shift_rate": (
            float(
                primary["label"].mean()
            )
            if len(primary)
            else np.nan
        ),
        "legacy_shift_rate": (
            float(
                legacy["label"].mean()
            )
            if len(legacy)
            else np.nan
        ),
    })

EVENT_CONTRACT_COMPARISON_DF = pd.DataFrame(
    contract_compare_rows
)

EVENT_CONTRACT_COMPARISON_PATH = (
    CONTRACT_DIR
    / "primary_vs_legacy_event_contract.csv"
)

EVENT_CONTRACT_COMPARISON_DF.to_csv(
    EVENT_CONTRACT_COMPARISON_PATH,
    index=False,
)

train_boundary_count = int(
    REAL_TRAIN_EVENTS[
        "source_event_type"
    ]
    .astype(str)
    .eq("adjacent_rttm_boundary")
    .sum()
)

train_midturn_count = int(
    REAL_TRAIN_EVENTS[
        "source_event_type"
    ]
    .astype(str)
    .eq("midturn_hold_negative")
    .sum()
)

allowed_midturn = int(
    math.ceil(
        float(
            MIDTURN_NEG_TO_BOUNDARY_RATIO
        )
        * max(
            1,
            train_boundary_count,
        )
    )
)

assert train_midturn_count <= allowed_midturn, (
    f"Mid-turn negatives={train_midturn_count} exceed "
    f"ratio cap={allowed_midturn}."
)

PAPER_READY_EXECUTION_GUARD = {
    "run_version": RUN_VERSION,
    "event_contract_version": EVENT_CONTRACT_VERSION,
    "primary_event_time_definition": PRIMARY_EVENT_TIME_DEFINITION,
    "primary_gap_range_s": [
        float(PRIMARY_MIN_GAP_S),
        float(
            PRIMARY_MAX_POSITIVE_GAP_S
        ),
    ],
    "audit_included_test_annotations": bool(
        EVENT_CONTRACT_AUDIT_INCLUDED_TEST_ANNOTATIONS
    ),
    "test_predictions_or_performance_used_for_contract_selection": bool(
        EVENT_CONTRACT_TEST_PREDICTIONS_USED_FOR_SELECTION
    ),
    "contract_selection_basis": str(
        EVENT_CONTRACT_SELECTION_BASIS
    ),
    "previous_sys1_checkpoint_reused": False,
    "midturn_negative_rows": train_midturn_count,
    "boundary_rows": train_boundary_count,
    "midturn_negative_weight": float(
        WEIGHT_REAL_MIDTURN
    ),
    "sys1_class_balance_mode": str(
        SYS1_CLASS_BALANCE_MODE
    ),
    "shared_base_assets": bool(
        REUSE_AUDIT_COMPATIBLE_BASE_ASSETS
    ),
    "base_asset_run_root": str(
        BASE_ASSET_RUN_ROOT
    ),
}

PAPER_READY_EXECUTION_GUARD_PATH = (
    CONTRACT_DIR
    / "paper_ready_execution_guard.json"
)

PAPER_READY_EXECUTION_GUARD_PATH.write_text(
    json.dumps(
        PAPER_READY_EXECUTION_GUARD,
        indent=2,
        sort_keys=True,
    )
)

display(EVENT_CONTRACT_COMPARISON_DF)
display(ACTION_ANCHOR_CHECK_DF)

print(
    "✅ Paper-ready event contract is internally consistent."
)
print(
    "The legacy midpoint contract is retained only for secondary reporting."
)
print(
    "The audit inspected annotations in DEV and TEST, but no TEST model "
    "prediction, score, threshold, or performance result selected this "
    "contract or any learned operating point."
)

globals()["EVENT_CONTRACT_COMPARISON_DF"] = (
    EVENT_CONTRACT_COMPARISON_DF
)
globals()["EVENT_CONTRACT_COMPARISON_PATH"] = (
    EVENT_CONTRACT_COMPARISON_PATH
)
globals()["ACTION_ANCHOR_CHECK_DF"] = (
    ACTION_ANCHOR_CHECK_DF
)
globals()["PAPER_READY_EXECUTION_GUARD"] = (
    PAPER_READY_EXECUTION_GUARD
)
globals()["PAPER_READY_EXECUTION_GUARD_PATH"] = (
    PAPER_READY_EXECUTION_GUARD_PATH
)

,split,primary_boundaries,primary_shifts,legacy_boundaries,legacy_shifts,excluded_delayed_boundaries,excluded_delayed_shifts,primary_shift_rate,legacy_shift_rate
0,train,3805,2197,5351,2782,1546,585,0.577398,0.519903
1,val,1109,467,1612,621,503,154,0.421100,0.385236
2,test,11201,6350,17164,8420,5963,2070,0.566914,0.490562


,action,rows,maximum_anchor_error_s,mean_gap_s,median_gap_s,split
0,BC,32,0.0,-0.468750,-0.380,train
1,BI,62,0.0,0.261290,0.200,train
2,ITT,513,0.0,-0.403353,-0.360,train
3,NA,1514,0.0,0.485020,0.520,train
4,NTT,1684,0.0,0.274489,0.200,train
5,BC,8,0.0,-0.445000,-0.380,val
6,BI,17,0.0,0.174118,0.120,val
7,ITT,129,0.0,-0.418605,-0.360,val
8,NA,617,0.0,0.519741,0.560,val
9,NTT,338,0.0,0.287692,0.200,val


✅ Paper-ready event contract is internally consistent.
The legacy midpoint contract is retained only for secondary reporting.
The audit inspected annotations in DEV and TEST, but no TEST model prediction, score, threshold, or performance result selected this contract or any learned operating point.


In [9]:
# Audio loading, energy, and augmentation helpers
TARGET_SR = int(globals().get("TARGET_SR", 16000))

def _to_float32_mono(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)

    if x.ndim == 0:
        x = x.reshape(1)

    if x.ndim > 1:
        x = np.mean(x, axis=1)

    x = x.astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x

def _peak_limit(x: np.ndarray, peak_limit: float = 0.99) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    if len(x) == 0:
        return x.astype(np.float32)

    peak = float(np.max(np.abs(x)))
    if peak > peak_limit and peak > 0:
        x = x / peak * peak_limit

    return x.astype(np.float32)

def _resample_np(x: np.ndarray, orig_sr: int, target_sr: int = TARGET_SR) -> np.ndarray:
    if int(orig_sr) == int(target_sr):
        return x.astype(np.float32)

    try:
        import torchaudio.functional as AF

        xt = torch.from_numpy(np.asarray(x, dtype=np.float32)).view(1, -1)
        yt = AF.resample(xt, orig_freq=int(orig_sr), new_freq=int(target_sr))
        return yt.view(-1).cpu().numpy().astype(np.float32)

    except Exception:
        try:
            import librosa

            y = librosa.resample(
                np.asarray(x, dtype=np.float32),
                orig_sr=int(orig_sr),
                target_sr=int(target_sr),
            )
            return np.asarray(y, dtype=np.float32)

        except Exception as e:
            raise ImportError(
                "Audio resampling requires torchaudio or librosa. "
                "Install with: pip install torchaudio  OR  pip install librosa"
            ) from e

def read_audio_mono(
    path: Path,
    start_s: Optional[float] = None,
    duration_s: Optional[float] = None,
    sr: int = TARGET_SR,
) -> Tuple[np.ndarray, int]:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Audio file not found: {path}")

    if start_s is None and duration_s is None:
        x, fs = sf.read(str(path), dtype="float32", always_2d=False)
    else:
        info = sf.info(str(path))
        fs0 = int(info.samplerate)

        start_s = 0.0 if start_s is None else float(start_s)
        start_frame = max(0, int(round(start_s * fs0)))

        if duration_s is None:
            frames = -1
        else:
            frames = max(1, int(round(float(duration_s) * fs0)))

        x, fs = sf.read(
            str(path),
            start=start_frame,
            frames=frames,
            dtype="float32",
            always_2d=False,
        )

    x = _to_float32_mono(x)

    if int(fs) != int(sr):
        x = _resample_np(x, orig_sr=int(fs), target_sr=int(sr))
        fs = int(sr)

    return x.astype(np.float32), int(fs)

def write_audio(path: Path, x: np.ndarray, sr: int = TARGET_SR):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    x = _peak_limit(np.asarray(x, dtype=np.float32), peak_limit=0.99)
    sf.write(str(path), x, int(sr))

def slice_or_pad(
    x: np.ndarray,
    center_s: float,
    total_s: float,
    sr: int = TARGET_SR,
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)

    n = int(round(float(total_s) * int(sr)))
    n = max(1, n)

    center = int(round(float(center_s) * int(sr)))
    start = center - n // 2
    end = start + n

    out = np.zeros(n, dtype=np.float32)

    src_start = max(0, start)
    src_end = min(len(x), end)

    dst_start = src_start - start
    dst_end = dst_start + max(0, src_end - src_start)

    if src_end > src_start and dst_end > dst_start:
        out[dst_start:dst_end] = x[src_start:src_end]

    return out.astype(np.float32)

def rms(x: np.ndarray, eps: float = 1e-8) -> float:
    x = np.asarray(x, dtype=np.float32)

    if len(x) == 0:
        return 0.0

    return float(np.sqrt(np.mean(x * x) + eps))

def frame_rms_trace(
    x: np.ndarray,
    sr: int = TARGET_SR,
    hop_s: float = 0.10,
    win_s: float = 0.20,
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)

    hop = max(1, int(round(float(hop_s) * int(sr))))
    win = max(1, int(round(float(win_s) * int(sr))))

    if len(x) == 0:
        return np.zeros(1, dtype=np.float32)

    n_frames = max(1, 1 + int(math.floor(max(0, len(x) - 1) / hop)))

    vals = []
    for i in range(n_frames):
        c = i * hop
        a = max(0, c - win // 2)
        b = min(len(x), c + win // 2)

        if b <= a:
            vals.append(0.0)
        else:
            vals.append(rms(x[a:b]))

    vals = np.asarray(vals, dtype=np.float32)
    return np.log1p(vals * 100.0).astype(np.float32)

def add_noise_snr(
    x: np.ndarray,
    rng: np.random.Generator,
    snr_db: float,
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)

    sig = rms(x)
    if sig < 1e-7:
        return x.astype(np.float32)

    noise_rms = sig / (10 ** (float(snr_db) / 20.0))
    noise = rng.normal(0.0, noise_rms, size=x.shape).astype(np.float32)

    return (x + noise).astype(np.float32)

def apply_gain(
    x: np.ndarray,
    rng: np.random.Generator,
    gain_db_range=(-4, 4),
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)

    gain_db = float(rng.uniform(float(gain_db_range[0]), float(gain_db_range[1])))
    y = x * (10 ** (gain_db / 20.0))

    return y.astype(np.float32)

def bedmix(
    x: np.ndarray,
    bed: np.ndarray,
    rng: np.random.Generator,
    snr_db: float = 18.0,
) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    bed = np.asarray(bed, dtype=np.float32)

    if len(x) == 0 or len(bed) == 0:
        return x.astype(np.float32)

    if len(bed) < len(x):
        reps = int(np.ceil(len(x) / max(1, len(bed))))
        bed = np.tile(bed, reps)

    max_start = max(1, len(bed) - len(x) + 1)
    start = int(rng.integers(0, max_start))

    b = bed[start:start + len(x)].astype(np.float32)

    x_rms = rms(x)
    b_rms = rms(b)

    if b_rms < 1e-8 or x_rms < 1e-8:
        return x.astype(np.float32)

    b = b / b_rms * (x_rms / (10 ** (float(snr_db) / 20.0)))
    y = x + b

    return y.astype(np.float32)

def safe_audio_augment(
    x: np.ndarray,
    rng: np.random.Generator,
    bed_paths: Optional[List[Path]] = None,
    p_bedmix: float = 0.35,
) -> np.ndarray:
    y = np.asarray(x, dtype=np.float32).copy()

    if len(y) == 0:
        return y.astype(np.float32)

    if rng.random() < 0.75:
        y = apply_gain(y, rng)

    if rng.random() < 0.55:
        y = add_noise_snr(y, rng, float(rng.uniform(18, 32)))

    bed_paths = list(bed_paths or [])
    if bed_paths and rng.random() < float(p_bedmix):
        try:
            idx = int(rng.integers(0, len(bed_paths)))
            bp = Path(bed_paths[idx])

            bed, _ = read_audio_mono(bp, sr=TARGET_SR)
            y = bedmix(y, bed, rng, snr_db=float(rng.uniform(12, 26)))

        except Exception:
            pass

    return _peak_limit(y, peak_limit=0.99)

def find_bedmix_files(bed_dir: Path) -> List[Path]:
    bed_dir = Path(bed_dir)

    if not bed_dir.exists():
        return []

    valid_exts = {".wav", ".flac", ".mp3", ".m4a"}
    files = []

    for p in bed_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in valid_exts:
            files.append(p)

    return sorted(files)

DIFF_BEDS_DIR = Path(globals().get("DIFF_BEDS_DIR", VOX_ROOT / "diffusion_beds_16k")).expanduser()
DIFF_BEDS_DIR.mkdir(parents=True, exist_ok=True)

DIFF_BED_PATHS = find_bedmix_files(DIFF_BEDS_DIR)

print("TARGET_SR:", TARGET_SR)
print("Diffusion/bedmix bed dir:", DIFF_BEDS_DIR)
print("Available bed files:", len(DIFF_BED_PATHS))

if len(DIFF_BED_PATHS) == 0 and USE_DIFFUSION_AUG:
    print("⚠️ USE_DIFFUSION_AUG=True but no bed files were found.")
    print("   Add .wav/.flac/.mp3/.m4a bed files to:", DIFF_BEDS_DIR)

TARGET_SR: 16000
Diffusion/bedmix bed dir: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/diffusion_beds_16k
Available bed files: 96


In [10]:
SYNTH_AUDIO_DIR = SYNTH_DIR / "audio"
SYNTH_RTTM_DIR = SYNTH_DIR / "rttm"
SYNTH_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_RTTM_DIR.mkdir(parents=True, exist_ok=True)

SYNTH_EVENTS_PATH = SYNTH_DIR / "synthetic_events.pkl"
SYNTH_MANIFEST_PATH = SYNTH_DIR / "synthetic_manifest.pkl"
SYNTH_META_PATH = SYNTH_DIR / "synthetic_generation_meta.json"

BASE_SYNTH_EVENTS_PATH = (
    BASE_SYNTH_ASSET_DIR
    / "synthetic_events.pkl"
)
BASE_SYNTH_MANIFEST_PATH = (
    BASE_SYNTH_ASSET_DIR
    / "synthetic_manifest.pkl"
)
BASE_SYNTH_META_PATH = (
    BASE_SYNTH_ASSET_DIR
    / "synthetic_generation_meta.json"
)

SPEECH_POOL_PATH = CACHE_DIR / f"speech_pool_train_{'debug' if DEBUG_SMALL_RUN else 'full'}.pkl"
SPEECH_POOL_META_PATH = CACHE_DIR / f"speech_pool_train_{'debug' if DEBUG_SMALL_RUN else 'full'}.meta.json"

SYNTH_EXTRA_COLUMNS = [
    "split", "audio_path", "rttm_path",
    "is_synthetic", "is_diffusion_augmented",
    "sample_weight", "synthetic_action_type",
]

DIARIZERS_STYLE_SYNTH_CONFIG = {
    "num_meetings": int(SYNTH_NUM_MEETINGS),
    "nb_speakers_per_meeting": [int(SYNTH_MIN_SPK), int(SYNTH_MAX_SPK)],
    "segments_per_meeting": list(globals().get(
        "SYNTH_SEGMENTS_PER_MEETING",
        [10 if DEBUG_SMALL_RUN else 18, 18 if DEBUG_SMALL_RUN else 42],
    )),
    "meeting_duration": [float(SYNTH_MIN_DURATION), float(SYNTH_MAX_DURATION)],
    "overlap_proba": float(globals().get("SYNTH_OVERLAP_PROBA", 0.32)),
    "overlap_length": list(globals().get("SYNTH_OVERLAP_LENGTH", [0.08, 0.55])),
    "silence_proba": float(globals().get("SYNTH_SILENCE_PROBA", 0.24)),
    "silence_duration": list(globals().get("SYNTH_SILENCE_DURATION", [0.05, 1.30])),
    "short_utterance_proba": float(globals().get("SYNTH_SHORT_UTT_PROBA", 0.18)),
    "brief_interjection_proba": float(globals().get("SYNTH_BRIEF_INTERJECTION_PROBA", 0.09)),
    "backchannel_proba_given_overlap": float(globals().get("SYNTH_BACKCHANNEL_PROBA_GIVEN_OVERLAP", 0.38)),
    "same_speaker_proba": float(globals().get("SYNTH_SAME_SPEAKER_PROBA", 0.20)),
    "normalize": bool(globals().get("SYNTH_NORMALIZE", True)),
    "augment": False,
    "sample_rate": int(SYNTH_SR),
    "cache_format": "pickle",
}

def _synthetic_generation_config() -> Dict[str, Any]:
    return {
        "run_version": RUN_VERSION,
        "event_contract_version": EVENT_CONTRACT_VERSION,
        "primary_event_time_definition": PRIMARY_EVENT_TIME_DEFINITION,
        "primary_min_gap_s": float(PRIMARY_MIN_GAP_S),
        "primary_max_positive_gap_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "short_turn_s": float(ACTION_SHORT_TURN_S),
        "overlap_min_s": float(ACTION_OVERLAP_MIN_S),
        "base_synthetic_asset_dir": str(BASE_SYNTH_ASSET_DIR),
        "seed": int(SEED),
        "diarizers_style_config": DIARIZERS_STYLE_SYNTH_CONFIG,
        "action_names": list(ACTION_NAMES),
        "shift_actions": sorted(list(SHIFT_ACTIONS)),
        "hold_actions": sorted(list(HOLD_ACTIONS)),
        "weight_synth_clean": float(WEIGHT_SYNTH_CLEAN),
        "speech_pool_train_sessions": list(map(str, TRAIN_SESSIONS)),
    }

def _meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False
    try:
        return json.loads(Path(path).read_text()) == expected
    except Exception:
        return False

def _clean_legacy_synth_parquet():
    legacy_paths = [
        SYNTH_DIR / "synthetic_events.parquet",
        SYNTH_DIR / "synthetic_manifest.parquet",
        CACHE_DIR / "speech_pool_3000.parquet",
        CACHE_DIR / "speech_pool_20000.parquet",
    ]
    for p in legacy_paths:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet cache:", p)
        except Exception as e:
            print("Could not remove legacy parquet cache:", p, e)

def _speech_pool_config(max_segments: int) -> Dict[str, Any]:
    return {
        "run_version": RUN_VERSION,
        "seed": int(SEED),
        "max_segments": int(max_segments),
        "train_sessions": list(map(str, TRAIN_SESSIONS)),
        "min_segment_dur": 0.45,
        "max_segment_dur": 8.0,
        "cache_format": "pickle",
    }

def extract_real_speech_pool(max_segments: int = 5000, force: bool = False) -> pd.DataFrame:
    expected = _speech_pool_config(max_segments)

    if SPEECH_POOL_PATH.exists() and _meta_matches(SPEECH_POOL_META_PATH, expected) and not force:
        pool = pd.read_pickle(SPEECH_POOL_PATH)
        return pool

    rows = []

    for sid in tqdm(TRAIN_SESSIONS, desc="Build real speech pool"):
        turns = read_rttm_as_turns(get_session_rttm_path(sid, "dev"))
        audio_path = get_session_audio_path(sid, "dev")

        for _, r in turns.iterrows():
            dur = float(r.end - r.start)
            if 0.45 <= dur <= 8.0:
                rows.append({
                    "session": sid,
                    "audio_path": str(audio_path),
                    "speaker": str(r.speaker),
                    "profile_id": f"{sid}::{str(r.speaker)}",
                    "start": float(r.start),
                    "end": float(r.end),
                    "dur": dur,
                })

    pool = pd.DataFrame(rows)

    if len(pool) > max_segments:
        pool = pool.sample(max_segments, random_state=SEED).reset_index(drop=True)

    pool.to_pickle(SPEECH_POOL_PATH)
    SPEECH_POOL_META_PATH.write_text(json.dumps(expected, indent=2, sort_keys=True))

    return pool

def build_speaker_profile_index(speech_pool: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    if speech_pool is None or len(speech_pool) == 0:
        return {}

    groups = {}
    for profile_id, g in speech_pool.groupby("profile_id"):
        if len(g) > 0:
            groups[str(profile_id)] = g.reset_index(drop=True)

    return groups

def _choose_profile_ids(
    profile_index: Dict[str, pd.DataFrame],
    n_speakers: int,
    rng: np.random.Generator,
) -> List[str]:
    keys = sorted(profile_index.keys())
    assert len(keys) > 0, "No speaker profiles available."

    if len(keys) >= n_speakers:
        idx = rng.choice(len(keys), size=n_speakers, replace=False)
    else:
        idx = rng.choice(len(keys), size=n_speakers, replace=True)

    return [keys[int(i)] for i in idx]

def _sample_pool_segment_for_profile(
    profile_df: pd.DataFrame,
    rng: np.random.Generator,
    target_dur: Optional[float] = None,
) -> np.ndarray:
    row = profile_df.iloc[int(rng.integers(0, len(profile_df)))]

    source_dur = float(row.end - row.start)
    dur = source_dur if target_dur is None else min(float(target_dur), source_dur)
    dur = max(0.05, dur)

    x, _ = read_audio_mono(
        Path(row.audio_path),
        start_s=float(row.start),
        duration_s=dur,
        sr=SYNTH_SR,
    )

    return x.astype(np.float32)

def _write_rttm(session: str, rows: List[Dict[str, Any]], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    rows = sorted(rows, key=lambda r: (float(r["start"]), float(r["end"]), str(r["speaker"])))

    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            start = max(0.0, float(r["start"]))
            dur = max(0.01, float(r["end"] - r["start"]))
            spk = str(r["speaker"])
            f.write(f"SPEAKER {session} 1 {start:.3f} {dur:.3f} <NA> <NA> {spk} <NA> <NA>\n")

def _normalise_segment(seg: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    seg = np.asarray(seg, dtype=np.float32)
    if len(seg) == 0:
        return seg.astype(np.float32)

    seg = np.nan_to_num(seg)
    peak = float(np.max(np.abs(seg))) if len(seg) else 0.0

    if peak > 1e-5:
        seg = seg / peak * float(rng.uniform(0.35, 0.75))

    return seg.astype(np.float32)

def _event_from_synthetic_boundary(
    sid: str,
    audio_path: Path,
    rttm_path: Path,
    prev_end: float,
    next_start: float,
    prev_spk: str,
    next_spk: str,
    prev_dur: float,
    next_dur: float,
    action: str,
) -> Dict[str, Any]:
    gap = float(next_start - prev_end)
    overlap = max(
        0.0,
        float(prev_end - next_start),
    )

    action = str(action)

    if action not in ACTION_TO_ID:
        action = "NA"

    event_time, anchor_role = (
        resolve_event_time(
            action=action,
            prev_end=float(prev_end),
            next_start=float(next_start),
            definition=str(
                PRIMARY_EVENT_TIME_DEFINITION
            ),
        )
    )

    interval_start = min(
        float(prev_end),
        float(next_start),
    )
    interval_end = max(
        float(prev_end),
        float(next_start),
    )

    is_primary = bool(
        gap >= float(PRIMARY_MIN_GAP_S)
        and gap
        <= float(
            PRIMARY_MAX_POSITIVE_GAP_S
        )
    )

    return {
        "session": sid,
        "time": float(
            max(
                0.0,
                event_time,
            )
        ),
        "time_floor_release": float(
            prev_end
        ),
        "time_floor_acquisition": float(
            next_start
        ),
        "time_midpoint": float(
            0.5
            * (
                float(prev_end)
                + float(next_start)
            )
        ),
        "transition_interval_start": float(
            interval_start
        ),
        "transition_interval_end": float(
            interval_end
        ),
        "transition_width_s": float(
            interval_end
            - interval_start
        ),
        "event_time_definition": str(
            PRIMARY_EVENT_TIME_DEFINITION
        ),
        "event_anchor_role": str(
            anchor_role
        ),
        "event_contract_version": str(
            EVENT_CONTRACT_VERSION
        ),
        "contract_name": "primary",
        "is_primary_contract_event": bool(
            is_primary
        ),
        "is_delayed_transition": bool(
            gap
            > float(
                PRIMARY_MAX_POSITIVE_GAP_S
            )
        ),
        "prev_end": float(prev_end),
        "next_start": float(next_start),
        "prev_speaker": str(prev_spk),
        "next_speaker": str(next_spk),
        "gap": float(gap),
        "overlap": float(overlap),
        "prev_dur": float(prev_dur),
        "next_dur": float(next_dur),
        "action": action,
        "action_id": int(
            ACTION_TO_ID[action]
        ),
        "label": int(
            action in SHIFT_ACTIONS
        ),
        "source_event_type": (
            "synthetic_diarizers_style_action_boundary"
        ),
        "split": "synth_train",
        "audio_path": str(audio_path),
        "rttm_path": str(rttm_path),
        "is_synthetic": True,
        "is_diffusion_augmented": False,
        "sample_weight": float(
            WEIGHT_SYNTH_CLEAN
        ),
        "synthetic_action_type": action,
    }

def _choose_next_speaker(
    speakers: List[str],
    current_floor_spk: str,
    action: str,
    rng: np.random.Generator,
) -> str:
    if action in {"NA", "PAUSE"}:
        return current_floor_spk

    choices = [s for s in speakers if s != current_floor_spk]
    if not choices:
        return current_floor_spk

    return str(rng.choice(choices))

def _sample_action_scenario(rng: np.random.Generator) -> str:
    cfg = DIARIZERS_STYLE_SYNTH_CONFIG

    same_speaker_proba = float(cfg["same_speaker_proba"])
    overlap_proba = float(cfg["overlap_proba"])
    silence_proba = float(cfg["silence_proba"])
    short_proba = float(cfg["short_utterance_proba"])
    bi_proba = float(cfg["brief_interjection_proba"])
    bc_given_overlap = float(cfg["backchannel_proba_given_overlap"])

    u = float(rng.random())

    if u < same_speaker_proba:
        return "NA"

    if u < same_speaker_proba + silence_proba:
        return "PAUSE"

    if u < same_speaker_proba + silence_proba + overlap_proba:
        return "BC" if float(rng.random()) < bc_given_overlap else "ITT"

    if float(rng.random()) < bi_proba or float(rng.random()) < short_proba * 0.35:
        return "BI"

    return "NTT"

def generate_synthetic_meeting(
    idx: int,
    speech_pool: pd.DataFrame,
    profile_index: Dict[str, pd.DataFrame],
    rng: np.random.Generator,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    cfg = DIARIZERS_STYLE_SYNTH_CONFIG

    sid = f"synth_{idx:06d}"

    min_spk, max_spk = cfg["nb_speakers_per_meeting"]
    n_spk = int(rng.integers(int(min_spk), int(max_spk) + 1))

    seg_min, seg_max = cfg["segments_per_meeting"]
    n_segments = int(rng.integers(int(seg_min), int(seg_max) + 1))

    dur_min, dur_max = cfg["meeting_duration"]
    target_duration = float(rng.uniform(float(dur_min), float(dur_max)))

    speakers = [f"spk_{i:02d}" for i in range(n_spk)]
    profile_ids = _choose_profile_ids(profile_index, n_spk, rng)
    speaker_to_profile = {spk: pid for spk, pid in zip(speakers, profile_ids)}

    timeline_len = int(round((target_duration + 6.0) * SYNTH_SR))
    timeline = np.zeros(max(1, timeline_len), dtype=np.float32)

    audio_path = SYNTH_AUDIO_DIR / f"{sid}.wav"
    rttm_path = SYNTH_RTTM_DIR / f"{sid}.rttm"

    rttm_rows = []
    event_rows = []

    current_floor_spk = str(rng.choice(speakers))
    floor_end = 0.0
    floor_dur = 0.0
    cur_t = 0.0

    overlap_min, overlap_max = cfg["overlap_length"]
    silence_min, silence_max = cfg["silence_duration"]

    for seg_idx in range(n_segments):
        if cur_t >= target_duration:
            break

        if seg_idx == 0:
            action_scenario = "NA"
            next_spk = current_floor_spk
            gap = 0.0
        else:
            action_scenario = _sample_action_scenario(rng)
            next_spk = _choose_next_speaker(speakers, current_floor_spk, action_scenario, rng)

            if action_scenario in {"ITT", "BC"}:
                gap = -float(rng.uniform(float(overlap_min), float(overlap_max)))
            elif action_scenario == "PAUSE":
                gap = float(rng.uniform(max(0.30, float(silence_min)), float(silence_max)))
            elif action_scenario == "BI":
                gap = float(rng.uniform(0.02, min(0.45, float(silence_max))))
            else:
                gap = float(rng.uniform(float(silence_min), min(0.60, float(silence_max))))

        if action_scenario in {"BI", "BC"}:
            seg_dur = float(rng.uniform(0.20, 0.55))
        else:
            seg_dur = float(rng.uniform(0.65, 3.20))

        profile_df = profile_index[speaker_to_profile[next_spk]]
        seg = _sample_pool_segment_for_profile(profile_df, rng, target_dur=seg_dur)

        if len(seg) < int(0.12 * SYNTH_SR):
            continue

        if bool(cfg["normalize"]):
            seg = _normalise_segment(seg, rng)

        if seg_idx == 0:
            start_t = 0.0
        else:
            start_t = max(0.0, floor_end + gap)

        a = int(round(start_t * SYNTH_SR))
        if a >= len(timeline):
            break

        b = min(len(timeline), a + len(seg))
        if b <= a:
            continue

        seg_use = seg[: b - a]
        timeline[a:b] += seg_use

        actual_end = start_t + len(seg_use) / SYNTH_SR
        actual_dur = max(0.0, actual_end - start_t)

        rttm_rows.append({
            "start": float(start_t),
            "end": float(actual_end),
            "speaker": str(next_spk),
        })

        if seg_idx > 0:
            if action_scenario == "PAUSE":
                action = "NA"
            else:
                action = action_scenario if action_scenario in ACTION_TO_ID else "NA"

            event_rows.append(
                _event_from_synthetic_boundary(
                    sid=sid,
                    audio_path=audio_path,
                    rttm_path=rttm_path,
                    prev_end=float(floor_end),
                    next_start=float(start_t),
                    prev_spk=str(current_floor_spk),
                    next_spk=str(next_spk),
                    prev_dur=float(floor_dur),
                    next_dur=float(actual_dur),
                    action=action,
                )
            )

        if action_scenario in {"NTT", "ITT"}:
            current_floor_spk = str(next_spk)
            floor_end = float(actual_end)
            floor_dur = float(actual_dur)
            cur_t = max(cur_t + 0.05, float(actual_end))

        elif action_scenario in {"NA", "PAUSE"}:
            current_floor_spk = str(next_spk)
            floor_end = float(actual_end)
            floor_dur = float(actual_dur)
            cur_t = max(cur_t + 0.05, float(actual_end))

        elif action_scenario in {"BC", "BI"}:
            floor_end = max(float(floor_end), float(actual_end))
            floor_dur = max(float(floor_dur), float(actual_dur))
            cur_t = max(cur_t + 0.05, floor_end)

        else:
            floor_end = float(actual_end)
            floor_dur = float(actual_dur)
            cur_t = max(cur_t + 0.05, float(actual_end))

    timeline = _peak_limit(timeline, peak_limit=0.98)

    write_audio(audio_path, timeline, SYNTH_SR)
    _write_rttm(sid, rttm_rows, rttm_path)

    manifest = {
        "session": sid,
        "audio_path": str(audio_path),
        "rttm_path": str(rttm_path),
        "duration": float(len(timeline) / SYNTH_SR),
        "target_duration": float(target_duration),
        "n_events": int(len(event_rows)),
        "n_turns": int(len(rttm_rows)),
        "n_speakers": int(n_spk),
        "segments_per_meeting": int(n_segments),
        "speaker_profile_map": json.dumps(speaker_to_profile, sort_keys=True),
        "synthetic_style": "huggingface_diarizers_style_plus_action_labels",
    }

    return manifest, event_rows

def _ensure_synth_schema(
    df: pd.DataFrame,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(
            columns=(
                EVENT_COLUMNS
                + SYNTH_EXTRA_COLUMNS
            )
        )

    frame = df.copy()

    for column in [
        "prev_end",
        "next_start",
        "prev_dur",
        "next_dur",
        "gap",
        "overlap",
    ]:
        if column not in frame.columns:
            frame[column] = np.nan

        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    frame["session"] = (
        frame["session"]
        .astype(str)
    )

    frame["action"] = (
        frame.get(
            "action",
            "NA",
        )
        .fillna("NA")
        .astype(str)
    )

    frame.loc[
        ~frame["action"].isin(
            ACTION_NAMES
        ),
        "action",
    ] = "NA"

    valid_boundary = (
        frame["prev_end"].notna()
        & frame["next_start"].notna()
    )

    if "gap" not in frame.columns:
        frame["gap"] = (
            frame["next_start"]
            - frame["prev_end"]
        )
    else:
        frame["gap"] = frame[
            "gap"
        ].fillna(
            frame["next_start"]
            - frame["prev_end"]
        )

    frame["overlap"] = (
        pd.to_numeric(
            frame["overlap"],
            errors="coerce",
        )
        .fillna(
            (
                frame["prev_end"]
                - frame["next_start"]
            )
            .clip(lower=0.0)
        )
    )

    frame[
        "time_floor_release"
    ] = frame["prev_end"]

    frame[
        "time_floor_acquisition"
    ] = frame["next_start"]

    frame["time_midpoint"] = (
        0.5
        * (
            frame["prev_end"]
            + frame["next_start"]
        )
    )

    frame[
        "transition_interval_start"
    ] = np.minimum(
        frame["prev_end"],
        frame["next_start"],
    )

    frame[
        "transition_interval_end"
    ] = np.maximum(
        frame["prev_end"],
        frame["next_start"],
    )

    frame["transition_width_s"] = (
        frame[
            "transition_interval_end"
        ]
        - frame[
            "transition_interval_start"
        ]
    )

    resolved_times = []
    anchor_roles = []

    for row in frame.itertuples(
        index=False
    ):
        if (
            not np.isfinite(
                float(row.prev_end)
            )
            or not np.isfinite(
                float(row.next_start)
            )
        ):
            resolved_times.append(
                np.nan
            )
            anchor_roles.append(
                "unknown"
            )
            continue

        event_time, anchor_role = (
            resolve_event_time(
                action=str(row.action),
                prev_end=float(
                    row.prev_end
                ),
                next_start=float(
                    row.next_start
                ),
                definition=str(
                    PRIMARY_EVENT_TIME_DEFINITION
                ),
            )
        )

        resolved_times.append(
            float(event_time)
        )
        anchor_roles.append(
            str(anchor_role)
        )

    frame["time"] = resolved_times
    frame[
        "event_anchor_role"
    ] = anchor_roles
    frame[
        "event_time_definition"
    ] = str(
        PRIMARY_EVENT_TIME_DEFINITION
    )
    frame[
        "event_contract_version"
    ] = str(
        EVENT_CONTRACT_VERSION
    )
    frame["contract_name"] = "primary"

    frame[
        "is_primary_contract_event"
    ] = (
        valid_boundary
        & frame["gap"].ge(
            float(
                PRIMARY_MIN_GAP_S
            )
        )
        & frame["gap"].le(
            float(
                PRIMARY_MAX_POSITIVE_GAP_S
            )
        )
    )

    frame[
        "is_delayed_transition"
    ] = frame["gap"].gt(
        float(
            PRIMARY_MAX_POSITIVE_GAP_S
        )
    )

    frame = frame[
        frame[
            "is_primary_contract_event"
        ]
    ].copy()

    frame["action_id"] = (
        frame["action"]
        .map(ACTION_TO_ID)
        .astype(int)
    )

    frame["label"] = (
        frame["action"]
        .isin(SHIFT_ACTIONS)
        .astype(int)
    )

    frame[
        "source_event_type"
    ] = (
        frame.get(
            "source_event_type",
            "synthetic_diarizers_style_action_boundary",
        )
        .fillna(
            "synthetic_diarizers_style_action_boundary"
        )
        .astype(str)
    )

    frame["split"] = (
        "synth_train"
    )
    frame["is_synthetic"] = True
    frame[
        "is_diffusion_augmented"
    ] = False
    frame["sample_weight"] = float(
        WEIGHT_SYNTH_CLEAN
    )
    frame[
        "synthetic_action_type"
    ] = frame["action"]

    for column in (
        EVENT_COLUMNS
        + SYNTH_EXTRA_COLUMNS
    ):
        if column not in frame.columns:
            frame[column] = np.nan

    frame["time"] = pd.to_numeric(
        frame["time"],
        errors="coerce",
    )

    frame = frame.dropna(
        subset=[
            "session",
            "time",
        ]
    )

    return (
        frame[
            EVENT_COLUMNS
            + SYNTH_EXTRA_COLUMNS
        ]
        .sort_values(
            ["session", "time"]
        )
        .reset_index(drop=True)
    )

def _load_and_relabel_base_synthetic_assets():
    if not (
        BASE_SYNTH_EVENTS_PATH.exists()
        and BASE_SYNTH_MANIFEST_PATH.exists()
    ):
        return None, None

    base_events = pd.read_pickle(
        BASE_SYNTH_EVENTS_PATH
    )
    base_manifest = pd.read_pickle(
        BASE_SYNTH_MANIFEST_PATH
    )

    relabelled = _ensure_synth_schema(
        base_events
    )

    return relabelled, base_manifest

_clean_legacy_synth_parquet()

if USE_SYNTH_MULTIPARTY:
    expected_meta = (
        _synthetic_generation_config()
    )

    if (
        SYNTH_EVENTS_PATH.exists()
        and SYNTH_MANIFEST_PATH.exists()
        and _meta_matches(
            SYNTH_META_PATH,
            expected_meta,
        )
        and not FORCE_REGENERATE_SYNTH
    ):
        SYNTH_EVENTS = pd.read_pickle(
            SYNTH_EVENTS_PATH
        )
        SYNTH_MANIFEST = pd.read_pickle(
            SYNTH_MANIFEST_PATH
        )
        SYNTH_EVENTS = (
            _ensure_synth_schema(
                SYNTH_EVENTS
            )
        )

        print(
            "Loaded local paper-ready "
            "synthetic event tables:",
            SYNTH_EVENTS.shape,
            SYNTH_MANIFEST.shape,
        )

    elif (
        bool(
            REUSE_AUDIT_COMPATIBLE_BASE_ASSETS
        )
        and not FORCE_REGENERATE_SYNTH
    ):
        (
            base_events,
            base_manifest,
        ) = (
            _load_and_relabel_base_synthetic_assets()
        )

        if (
            base_events is not None
            and base_manifest is not None
        ):
            SYNTH_EVENTS = base_events
            SYNTH_MANIFEST = (
                base_manifest
            )

            SYNTH_MANIFEST.to_pickle(
                SYNTH_MANIFEST_PATH
            )
            SYNTH_EVENTS.to_pickle(
                SYNTH_EVENTS_PATH
            )
            SYNTH_META_PATH.write_text(
                json.dumps(
                    expected_meta,
                    indent=2,
                    sort_keys=True,
                )
            )

            print(
                "Reused completed synthetic "
                "audio/RTTM assets and rebuilt "
                "only the paper-ready event table:",
                SYNTH_EVENTS.shape,
                SYNTH_MANIFEST.shape,
            )
        else:
            raise FileNotFoundError(
                "Base synthetic assets were requested "
                "but synthetic_events.pkl and "
                "synthetic_manifest.pkl were not found in "
                f"{BASE_SYNTH_ASSET_DIR}. "
                "Set FORCE_REGENERATE_SYNTH=True only if "
                "you intentionally want to recreate the "
                "6,000 meetings."
            )

    else:
        speech_pool = (
            extract_real_speech_pool(
                max_segments=(
                    3000
                    if DEBUG_SMALL_RUN
                    else 20000
                ),
                force=(
                    FORCE_REGENERATE_SYNTH
                ),
            )
        )

        assert len(speech_pool) > 0, (
            "No real speech pool segments "
            "available for synthetic generation."
        )

        profile_index = (
            build_speaker_profile_index(
                speech_pool
            )
        )

        assert len(profile_index) > 0, (
            "No speaker profiles available "
            "for synthetic generation."
        )

        rng = np.random.default_rng(
            SEED + 77
        )

        manifest_rows = []
        event_rows = []

        for meeting_index in tqdm(
            range(
                SYNTH_NUM_MEETINGS
            ),
            desc=(
                "Generate HF-diarizers-style "
                "synthetic meetings"
            ),
        ):
            manifest, events = (
                generate_synthetic_meeting(
                    meeting_index,
                    speech_pool,
                    profile_index,
                    rng,
                )
            )
            manifest_rows.append(
                manifest
            )
            event_rows.extend(
                events
            )

        SYNTH_MANIFEST = pd.DataFrame(
            manifest_rows
        )
        SYNTH_EVENTS = (
            _ensure_synth_schema(
                pd.DataFrame(
                    event_rows
                )
            )
        )

        SYNTH_MANIFEST.to_pickle(
            SYNTH_MANIFEST_PATH
        )
        SYNTH_EVENTS.to_pickle(
            SYNTH_EVENTS_PATH
        )
        SYNTH_META_PATH.write_text(
            json.dumps(
                expected_meta,
                indent=2,
                sort_keys=True,
            )
        )

        print(
            "Generated synthetic data:",
            SYNTH_EVENTS.shape,
            SYNTH_MANIFEST.shape,
        )

else:
    SYNTH_EVENTS = pd.DataFrame(
        columns=(
            EVENT_COLUMNS
            + SYNTH_EXTRA_COLUMNS
        )
    )
    SYNTH_MANIFEST = pd.DataFrame(
        columns=[
            "session",
            "audio_path",
            "rttm_path",
            "duration",
            "target_duration",
            "n_events",
            "n_turns",
            "n_speakers",
            "segments_per_meeting",
            "speaker_profile_map",
            "synthetic_style",
        ]
    )

print("SYNTH_EVENTS:", SYNTH_EVENTS.shape)
print("SYNTH_MANIFEST:", SYNTH_MANIFEST.shape)
print("Synthetic style:", "huggingface_diarizers_style_plus_action_labels")

if len(SYNTH_EVENTS):
    display(SYNTH_EVENTS["action"].value_counts().to_frame("count"))
    print("Synthetic shift rate:", round(float(SYNTH_EVENTS["label"].mean()), 4))
    print("Synthetic source types:", SYNTH_EVENTS["source_event_type"].value_counts().to_dict())

Loaded local paper-ready synthetic event tables: (62312, 33) (6000, 11)
SYNTH_EVENTS: (62312, 33)
SYNTH_MANIFEST: (6000, 11)
Synthetic style: huggingface_diarizers_style_plus_action_labels


,count
action,
NA,24790
NTT,13795
ITT,13139
BC,8240
BI,2348


Synthetic shift rate: 0.4322
Synthetic source types: {'synthetic_diarizers_style_action_boundary': 62312}


In [11]:
REAL_AUG_EVENTS_PATH = AUG_DIR / "real_train_augmented_events.pkl"
SYNTH_AUG_EVENTS_PATH = AUG_DIR / "synth_augmented_events.pkl"
AUG_MANIFEST_PATH = AUG_DIR / "augmentation_manifest.pkl"
AUG_META_PATH = AUG_DIR / "augmentation_meta.json"

BASE_AUG_MANIFEST_PATH = BASE_AUG_ASSET_DIR / "augmentation_manifest.pkl"

def _augmentation_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_min_s": float(PRIMARY_MIN_GAP_S),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "midturn_negative_ratio": float(MIDTURN_HOLD_NEGATIVE_RATIO),
        "midturn_negative_max_per_session": int(MIDTURN_HOLD_MAX_PER_SESSION),
        "midturn_negative_weight": float(MIDTURN_HOLD_SAMPLE_WEIGHT),
        "seed": int(SEED),
        "use_diffusion_aug": bool(USE_DIFFUSION_AUG),
        "weight_real_diffusion": float(WEIGHT_REAL_DIFFUSION),
        "weight_synth_diffusion": float(WEIGHT_SYNTH_DIFFUSION),
        "n_diff_beds": int(len(DIFF_BED_PATHS)),
        "diff_beds_dir": str(DIFF_BEDS_DIR),
        "base_aug_manifest": str(BASE_AUG_MANIFEST_PATH),
        "reuse_base_augmented_audio": bool(
            AUG_ABLATION_REUSE_BASE_ASSETS
            if "AUG_ABLATION_REUSE_BASE_ASSETS" in globals()
            else True
        ),
        "debug_small_run": bool(DEBUG_SMALL_RUN),
        "debug_max_train_sessions": int(DEBUG_MAX_TRAIN_SESSIONS),
        "synth_debug_max_sessions": int(40 if DEBUG_SMALL_RUN else -1),
        "cache_format": "pickle",
    }

def _aug_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    path = Path(path)
    if not path.exists():
        return False
    try:
        return json.loads(path.read_text()) == expected
    except Exception:
        return False

def _clean_legacy_aug_parquet() -> None:
    for path in [
        AUG_DIR / "real_train_augmented_events.parquet",
        AUG_DIR / "synth_augmented_events.parquet",
        AUG_DIR / "augmentation_manifest.parquet",
    ]:
        try:
            if path.exists():
                path.unlink()
                print("Removed legacy parquet augmentation cache:", path)
        except Exception as exc:
            print("Could not remove legacy augmentation cache:", path, exc)

def make_augmented_copy(
    audio_path: Path,
    out_path: Path,
    rng: np.random.Generator,
) -> Path:
    audio_path = Path(audio_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    waveform, _ = read_audio_mono(audio_path, sr=TARGET_SR)
    augmented = safe_audio_augment(
        waveform,
        rng,
        DIFF_BED_PATHS,
        p_bedmix=0.55 if len(DIFF_BED_PATHS) else 0.0,
    )
    write_audio(out_path, augmented, TARGET_SR)
    return out_path

def _ensure_augmented_event_schema(
    frame: pd.DataFrame,
    weight: float,
) -> pd.DataFrame:
    expected_columns = list(
        dict.fromkeys(
            EVENT_COLUMNS
            + REAL_EVENT_EXTRA_COLUMNS
            + SYNTH_EXTRA_COLUMNS
            + ["orig_session"]
        )
    )

    if frame is None or len(frame) == 0:
        return pd.DataFrame(columns=expected_columns)

    frame = frame.copy()

    for column in expected_columns:
        if column not in frame.columns:
            frame[column] = np.nan

    frame["session"] = frame["session"].astype(str)
    frame["orig_session"] = frame["orig_session"].astype(str)
    frame["time"] = pd.to_numeric(frame["time"], errors="coerce")
    frame = frame.dropna(subset=["session", "time"]).copy()

    frame["is_diffusion_augmented"] = True
    frame["sample_weight"] = float(weight)
    frame["event_contract_version"] = str(EVENT_CONTRACT_VERSION)
    frame["event_time_definition"] = str(PRIMARY_EVENT_TIME_DEFINITION)

    if "is_synthetic" in frame.columns:
        frame["is_synthetic"] = (
            frame["is_synthetic"].fillna(False).astype(bool)
        )

    frame["action"] = frame["action"].fillna("NA").astype(str)
    frame.loc[~frame["action"].isin(ACTION_NAMES), "action"] = "NA"
    frame["action_id"] = frame["action"].map(ACTION_TO_ID).astype(int)
    frame["label"] = frame["action"].isin(SHIFT_ACTIONS).astype(int)

    return (
        frame[expected_columns]
        .sort_values(["session", "time"])
        .reset_index(drop=True)
    )

def _rebuild_events_from_existing_manifest(
    base_events: pd.DataFrame,
    manifest: pd.DataFrame,
    kind: str,
    weight: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if base_events is None or len(base_events) == 0:
        return _ensure_augmented_event_schema(pd.DataFrame(), weight), pd.DataFrame()

    required_manifest_columns = {"session", "orig_session", "audio_path"}
    missing = sorted(required_manifest_columns - set(manifest.columns))
    if missing:
        raise KeyError(
            "Base augmentation manifest is missing required columns: "
            f"{missing}"
        )

    selected = manifest.copy()
    if "kind" in selected.columns:
        selected = selected[selected["kind"].astype(str).eq(str(kind))].copy()
    elif "is_synthetic" in selected.columns:
        expected_synth = str(kind).startswith("synth")
        selected = selected[
            selected["is_synthetic"].fillna(False).astype(bool).eq(expected_synth)
        ].copy()

    event_source = base_events.copy()
    event_source["session"] = event_source["session"].astype(str)

    event_parts = []
    retained_manifest_rows = []

    for manifest_row in selected.itertuples(index=False):
        original_session = str(getattr(manifest_row, "orig_session"))
        augmented_session = str(getattr(manifest_row, "session"))
        augmented_audio = Path(str(getattr(manifest_row, "audio_path")))

        if not augmented_audio.exists():
            continue

        rows = event_source[
            event_source["session"].astype(str).eq(original_session)
        ].copy()
        if len(rows) == 0:
            continue

        rows["orig_session"] = original_session
        rows["session"] = augmented_session
        rows["audio_path"] = str(augmented_audio)
        rows["is_diffusion_augmented"] = True
        rows["sample_weight"] = float(weight)
        event_parts.append(rows)

        retained = {
            column: getattr(manifest_row, column)
            for column in selected.columns
            if hasattr(manifest_row, column)
        }
        retained["event_contract_version"] = str(EVENT_CONTRACT_VERSION)
        retained["events_rebuilt_from_current_contract"] = True
        retained["n_events"] = int(len(rows))
        retained_manifest_rows.append(retained)

    rebuilt = (
        pd.concat(event_parts, ignore_index=True)
        if event_parts
        else pd.DataFrame()
    )
    rebuilt_manifest = pd.DataFrame(retained_manifest_rows)

    return (
        _ensure_augmented_event_schema(rebuilt, weight),
        rebuilt_manifest,
    )

def build_augmented_events(
    events_df: pd.DataFrame,
    out_subdir: str,
    weight: float,
    max_sessions: Optional[int] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if events_df is None or len(events_df) == 0:
        return _ensure_augmented_event_schema(pd.DataFrame(), weight), pd.DataFrame()

    rng = np.random.default_rng(SEED + 202)
    out_audio_dir = AUG_DIR / out_subdir / "audio"
    out_audio_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows: List[Dict[str, Any]] = []
    augmented_event_parts: List[pd.DataFrame] = []

    source = events_df.copy()
    source["session"] = source["session"].astype(str)

    sessions = sorted(source["session"].unique())
    if max_sessions is not None:
        sessions = sessions[: int(max_sessions)]

    for session_id in tqdm(
        sessions,
        desc=f"Build augmented audio: {out_subdir}",
    ):
        rows = source[source["session"].eq(session_id)].copy()
        if len(rows) == 0:
            continue

        if "audio_path" not in rows.columns or pd.isna(rows.iloc[0]["audio_path"]):
            print(f"Missing audio_path for {session_id}; skipping.")
            continue

        source_audio = Path(str(rows.iloc[0]["audio_path"]))
        if not source_audio.exists():
            print(f"Source audio not found for {session_id}: {source_audio}")
            continue

        augmented_session = f"{session_id}__diff"
        output_audio = out_audio_dir / f"{augmented_session}.wav"

        if not output_audio.exists() or FORCE_REBUILD_AUGMENTED_AUDIO:
            make_augmented_copy(source_audio, output_audio, rng)

        rows["orig_session"] = rows["session"].astype(str)
        rows["session"] = augmented_session
        rows["audio_path"] = str(output_audio)
        rows["is_diffusion_augmented"] = True
        rows["sample_weight"] = float(weight)
        augmented_event_parts.append(rows)

        manifest_rows.append({
            "session": augmented_session,
            "orig_session": session_id,
            "src_audio": str(source_audio),
            "audio_path": str(output_audio),
            "weight": float(weight),
            "kind": str(out_subdir),
            "n_events": int(len(rows)),
            "is_synthetic": bool(
                rows["is_synthetic"].iloc[0]
                if "is_synthetic" in rows.columns
                else False
            ),
            "event_contract_version": str(EVENT_CONTRACT_VERSION),
            "events_rebuilt_from_current_contract": True,
        })

    augmented = (
        pd.concat(augmented_event_parts, ignore_index=True)
        if augmented_event_parts
        else pd.DataFrame()
    )

    return (
        _ensure_augmented_event_schema(augmented, weight),
        pd.DataFrame(manifest_rows),
    )

_clean_legacy_aug_parquet()
expected_aug_meta = _augmentation_config()

if USE_DIFFUSION_AUG:
    local_cache_ready = (
        REAL_AUG_EVENTS_PATH.exists()
        and SYNTH_AUG_EVENTS_PATH.exists()
        and AUG_MANIFEST_PATH.exists()
        and _aug_meta_matches(AUG_META_PATH, expected_aug_meta)
        and not FORCE_REBUILD_AUGMENTED_AUDIO
    )

    if local_cache_ready:
        REAL_AUG_EVENTS = _ensure_augmented_event_schema(
            pd.read_pickle(REAL_AUG_EVENTS_PATH),
            WEIGHT_REAL_DIFFUSION,
        )
        SYNTH_AUG_EVENTS = _ensure_augmented_event_schema(
            pd.read_pickle(SYNTH_AUG_EVENTS_PATH),
            WEIGHT_SYNTH_DIFFUSION,
        )
        AUG_MANIFEST = pd.read_pickle(AUG_MANIFEST_PATH)
        print(
            "Loaded paper-ready augmentation event caches:",
            REAL_AUG_EVENTS.shape,
            SYNTH_AUG_EVENTS.shape,
        )
    else:
        reused_base_manifest = False

        if (
            BASE_AUG_MANIFEST_PATH.exists()
            and not FORCE_REBUILD_AUGMENTED_AUDIO
        ):
            base_manifest = pd.read_pickle(BASE_AUG_MANIFEST_PATH)

            REAL_AUG_EVENTS, real_manifest = (
                _rebuild_events_from_existing_manifest(
                    REAL_TRAIN_EVENTS,
                    base_manifest,
                    kind="real_diffusion",
                    weight=WEIGHT_REAL_DIFFUSION,
                )
            )
            SYNTH_AUG_EVENTS, synth_manifest = (
                _rebuild_events_from_existing_manifest(
                    SYNTH_EVENTS,
                    base_manifest,
                    kind="synth_diffusion",
                    weight=WEIGHT_SYNTH_DIFFUSION,
                )
            )

            reused_base_manifest = (
                len(REAL_AUG_EVENTS) > 0
                or len(SYNTH_AUG_EVENTS) > 0
            )

            if reused_base_manifest:
                AUG_MANIFEST = pd.concat(
                    [real_manifest, synth_manifest],
                    ignore_index=True,
                )
                print(
                    "Reused base augmented waveforms and rebuilt labels:",
                    REAL_AUG_EVENTS.shape,
                    SYNTH_AUG_EVENTS.shape,
                )

        if not reused_base_manifest:
            max_real_aug = (
                DEBUG_MAX_TRAIN_SESSIONS
                if DEBUG_SMALL_RUN
                else None
            )
            max_synth_aug = 40 if DEBUG_SMALL_RUN else None

            REAL_AUG_EVENTS, real_manifest = build_augmented_events(
                REAL_TRAIN_EVENTS,
                "real_diffusion",
                WEIGHT_REAL_DIFFUSION,
                max_sessions=max_real_aug,
            )
            SYNTH_AUG_EVENTS, synth_manifest = build_augmented_events(
                SYNTH_EVENTS,
                "synth_diffusion",
                WEIGHT_SYNTH_DIFFUSION,
                max_sessions=max_synth_aug,
            )
            AUG_MANIFEST = pd.concat(
                [real_manifest, synth_manifest],
                ignore_index=True,
            )

        REAL_AUG_EVENTS.to_pickle(REAL_AUG_EVENTS_PATH)
        SYNTH_AUG_EVENTS.to_pickle(SYNTH_AUG_EVENTS_PATH)
        AUG_MANIFEST.to_pickle(AUG_MANIFEST_PATH)
        AUG_META_PATH.write_text(
            json.dumps(expected_aug_meta, indent=2, sort_keys=True)
        )
else:
    REAL_AUG_EVENTS = _ensure_augmented_event_schema(
        pd.DataFrame(),
        WEIGHT_REAL_DIFFUSION,
    )
    SYNTH_AUG_EVENTS = _ensure_augmented_event_schema(
        pd.DataFrame(),
        WEIGHT_SYNTH_DIFFUSION,
    )
    AUG_MANIFEST = pd.DataFrame()

TRAIN_EVENTS_MIXED = pd.concat(
    [
        REAL_TRAIN_EVENTS,
        SYNTH_EVENTS,
        REAL_AUG_EVENTS,
        SYNTH_AUG_EVENTS,
    ],
    ignore_index=True,
)

if len(TRAIN_EVENTS_MIXED):
    TRAIN_EVENTS_MIXED = (
        TRAIN_EVENTS_MIXED
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

print("REAL_TRAIN_EVENTS:", REAL_TRAIN_EVENTS.shape)
print("SYNTH_EVENTS:", SYNTH_EVENTS.shape)
print("REAL_AUG_EVENTS:", REAL_AUG_EVENTS.shape)
print("SYNTH_AUG_EVENTS:", SYNTH_AUG_EVENTS.shape)
print("AUG_MANIFEST:", AUG_MANIFEST.shape)
print("TRAIN_EVENTS_MIXED:", TRAIN_EVENTS_MIXED.shape)

if len(TRAIN_EVENTS_MIXED):
    display(
        TRAIN_EVENTS_MIXED
        .groupby(
            [
                "is_synthetic",
                "is_diffusion_augmented",
                "action",
            ],
            dropna=False,
        )
        .agg(
            rows=("session", "size"),
            weighted_rows=("sample_weight", "sum"),
        )
        .reset_index()
        .sort_values(
            [
                "is_synthetic",
                "is_diffusion_augmented",
                "action",
            ]
        )
    )

    print(
        "Mixed TRAIN raw SHIFT rate:",
        round(float(TRAIN_EVENTS_MIXED["label"].mean()), 4),
    )
    weighted_shift_rate = float(
        np.average(
            TRAIN_EVENTS_MIXED["label"].astype(float),
            weights=TRAIN_EVENTS_MIXED["sample_weight"].astype(float),
        )
    )
    print(
        "Mixed TRAIN weighted SHIFT rate:",
        round(weighted_shift_rate, 4),
    )
    print("Mixed TRAIN sample-weight summary:")
    display(
        TRAIN_EVENTS_MIXED["sample_weight"]
        .describe()
        .to_frame("sample_weight")
    )

globals()["REAL_AUG_EVENTS"] = REAL_AUG_EVENTS
globals()["SYNTH_AUG_EVENTS"] = SYNTH_AUG_EVENTS
globals()["AUG_MANIFEST"] = AUG_MANIFEST
globals()["TRAIN_EVENTS_MIXED"] = TRAIN_EVENTS_MIXED

REAL_TRAIN_EVENTS: (5675, 32)
SYNTH_EVENTS: (62312, 33)
REAL_AUG_EVENTS: (0, 34)
SYNTH_AUG_EVENTS: (0, 34)
AUG_MANIFEST: (0, 0)
TRAIN_EVENTS_MIXED: (67987, 34)


,is_synthetic,is_diffusion_augmented,action,rows,weighted_rows
0,False,False,BC,32,32.0
1,False,False,BI,62,62.0
2,False,False,ITT,513,513.0
3,False,False,NA,3384,1981.5
4,False,False,NTT,1684,1684.0
5,True,False,BC,8240,4120.0
6,True,False,BI,2348,1174.0
7,True,False,ITT,13139,6569.5
8,True,False,NA,24790,12395.0
9,True,False,NTT,13795,6897.5


Mixed TRAIN raw SHIFT rate: 0.4285
Mixed TRAIN weighted SHIFT rate: 0.4421
Mixed TRAIN sample-weight summary:


,sample_weight
count,67987.0
unique,3.0
top,0.5
freq,62312.0


In [12]:
# Run/load ASR text for real + synthetic training/VAL/TEST audio
ASR_SEGMENTS_PATH = ASR_CACHE_DIR / "asr_segments_all.pkl"
ASR_META_PATH = ASR_CACHE_DIR / "asr_segments_all.meta.json"

def _asr_config() -> Dict[str, Any]:
    return {
        "run_version": str(ASSET_CACHE_RUN_VERSION),
        "seed": int(SEED),
        "use_asr_text": bool(USE_ASR_TEXT),
        "asr_model_name": str(ASR_MODEL_NAME),
        "asr_compute_type": str(ASR_COMPUTE_TYPE),
        "debug_small_run": bool(DEBUG_SMALL_RUN),
        "debug_manifest_limit": 60 if DEBUG_SMALL_RUN else -1,
        "include_real_train": True,
        "include_real_val": True,
        "include_real_test": True,
        "include_synthetic": bool(USE_SYNTH_MULTIPARTY),
        "copy_augmented_from_original": True,
        "cache_format": "pickle",
    }

def _asr_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False
    try:
        return json.loads(Path(path).read_text()) == expected
    except Exception:
        return False

def _clean_legacy_asr_parquet():
    for p in [
        ASR_CACHE_DIR / "asr_segments_all.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet ASR cache:", p)
        except Exception as e:
            print("Could not remove legacy parquet ASR cache:", p, e)

def collect_audio_manifest_for_asr() -> pd.DataFrame:
    rows = []

    for sid in TRAIN_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "train",
            "is_synthetic": False,
        })

    for sid in VAL_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "val",
            "is_synthetic": False,
        })

    for sid in TEST_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "test")),
            "split": "test",
            "is_synthetic": False,
        })

    if USE_SYNTH_MULTIPARTY and SYNTH_MANIFEST is not None and len(SYNTH_MANIFEST):
        synth_rows = (
            SYNTH_MANIFEST[["session", "audio_path"]]
            .copy()
            .assign(split="synth_train", is_synthetic=True)
            .to_dict("records")
        )
        rows.extend(synth_rows)

    manifest = pd.DataFrame(rows)

    if len(manifest) == 0:
        return pd.DataFrame(columns=["session", "audio_path", "split", "is_synthetic"])

    manifest["session"] = manifest["session"].astype(str)
    manifest["audio_path"] = manifest["audio_path"].astype(str)
    manifest = manifest.drop_duplicates(subset=["session"]).reset_index(drop=True)

    exists_mask = manifest["audio_path"].map(lambda p: Path(p).exists())
    if not bool(exists_mask.all()):
        missing = manifest.loc[~exists_mask, ["session", "audio_path"]].head(10)
        print("⚠️ Some ASR audio files do not exist. First examples:")
        display(missing)

    manifest = manifest.loc[exists_mask].reset_index(drop=True)
    return manifest

def run_faster_whisper_asr(manifest: pd.DataFrame) -> pd.DataFrame:
    try:
        from faster_whisper import WhisperModel
    except Exception as e:
        raise ImportError(
            "faster-whisper is required. Install with: pip install -U faster-whisper ctranslate2"
        ) from e

    if manifest is None or len(manifest) == 0:
        return pd.DataFrame(columns=[
            "session", "start", "end", "text",
            "asr_model", "language", "language_probability",
            "split", "is_synthetic",
        ])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = ASR_COMPUTE_TYPE if device == "cuda" else "int8"

    print("Loading faster-whisper:", ASR_MODEL_NAME, "| device:", device, "| compute_type:", compute_type)
    model = WhisperModel(ASR_MODEL_NAME, device=device, compute_type=compute_type)

    rows = []

    for _, r in tqdm(manifest.iterrows(), total=len(manifest), desc="ASR"):
        sid = str(r.session)
        audio_path = str(r.audio_path)
        split = str(r.get("split", "unknown"))
        is_synth = bool(r.get("is_synthetic", False))

        try:
            segments, info = model.transcribe(
                audio_path,
                vad_filter=True,
                beam_size=1,
                condition_on_previous_text=True,
            )

            language = str(getattr(info, "language", ""))
            language_probability = float(getattr(info, "language_probability", np.nan))

            for seg in segments:
                text = str(seg.text).strip()
                if not text:
                    continue

                rows.append({
                    "session": sid,
                    "start": float(seg.start),
                    "end": float(seg.end),
                    "text": text,
                    "asr_model": str(ASR_MODEL_NAME),
                    "language": language,
                    "language_probability": language_probability,
                    "split": split,
                    "is_synthetic": is_synth,
                })

        except Exception as e:
            print("ASR failed:", sid, e)

    out = pd.DataFrame(rows)

    if len(out) == 0:
        out = pd.DataFrame(columns=[
            "session", "start", "end", "text",
            "asr_model", "language", "language_probability",
            "split", "is_synthetic",
        ])

    return out.sort_values(["session", "start", "end"]).reset_index(drop=True)

def _ensure_asr_schema(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "session", "start", "end", "text",
        "asr_model", "language", "language_probability",
        "split", "is_synthetic", "orig_session",
    ]

    if df is None or len(df) == 0:
        return pd.DataFrame(columns=cols)

    df = df.copy()

    for col in cols:
        if col not in df.columns:
            df[col] = np.nan

    df["session"] = df["session"].astype(str)
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")
    df["text"] = df["text"].fillna("").astype(str)

    df = df.dropna(subset=["session", "start", "end"])
    df = df[df["end"] >= df["start"]].reset_index(drop=True)

    return df[cols].sort_values(["session", "start", "end"]).reset_index(drop=True)

def copy_asr_to_augmented_sessions(asr_df: pd.DataFrame, aug_manifest: pd.DataFrame) -> pd.DataFrame:
    asr_df = _ensure_asr_schema(asr_df)

    if aug_manifest is None or len(aug_manifest) == 0 or len(asr_df) == 0:
        return asr_df

    copies = []

    for _, r in aug_manifest.iterrows():
        orig = str(r.get("orig_session", ""))
        aug = str(r.get("session", ""))

        if not orig or not aug:
            continue

        seg = asr_df[asr_df["session"].astype(str) == orig].copy()
        if len(seg) == 0:
            continue

        seg["orig_session"] = orig
        seg["session"] = aug

        if "kind" in r:
            seg["split"] = str(r["kind"])

        copies.append(seg)

    if copies:
        out = pd.concat([asr_df] + copies, ignore_index=True)
    else:
        out = asr_df

    out = _ensure_asr_schema(out)
    out = out.drop_duplicates(["session", "start", "end", "text"]).reset_index(drop=True)

    return out

_clean_legacy_asr_parquet()

if USE_ASR_TEXT:
    expected_asr_meta = _asr_config()

    if ASR_SEGMENTS_PATH.exists() and _asr_meta_matches(ASR_META_PATH, expected_asr_meta) and not FORCE_RERUN_ASR:
        ASR_SEGMENTS = pd.read_pickle(ASR_SEGMENTS_PATH)
        ASR_SEGMENTS = _ensure_asr_schema(ASR_SEGMENTS)
        print("Loaded ASR segments:", ASR_SEGMENTS.shape)

    else:
        asr_manifest = collect_audio_manifest_for_asr()

        if DEBUG_SMALL_RUN:
            asr_manifest = asr_manifest.head(60).reset_index(drop=True)

        print("ASR manifest:", asr_manifest.shape)
        ASR_SEGMENTS = run_faster_whisper_asr(asr_manifest)
        ASR_SEGMENTS = _ensure_asr_schema(ASR_SEGMENTS)

        ASR_SEGMENTS = copy_asr_to_augmented_sessions(ASR_SEGMENTS, AUG_MANIFEST)

        ASR_SEGMENTS.to_pickle(ASR_SEGMENTS_PATH)
        ASR_META_PATH.write_text(json.dumps(expected_asr_meta, indent=2, sort_keys=True))

else:
    ASR_SEGMENTS = pd.DataFrame(columns=[
        "session", "start", "end", "text",
        "asr_model", "language", "language_probability",
        "split", "is_synthetic", "orig_session",
    ])

if USE_ASR_TEXT and len(AUG_MANIFEST) and len(ASR_SEGMENTS):
    before_n = len(ASR_SEGMENTS)
    ASR_SEGMENTS = copy_asr_to_augmented_sessions(ASR_SEGMENTS, AUG_MANIFEST)
    after_n = len(ASR_SEGMENTS)
    if after_n > before_n:
        ASR_SEGMENTS.to_pickle(ASR_SEGMENTS_PATH)
        print(f"Added ASR copies for augmented sessions: +{after_n - before_n}")

print("ASR segments:", ASR_SEGMENTS.shape)

if len(ASR_SEGMENTS):
    display(ASR_SEGMENTS.head())

def asr_context_for_candidate(
    session: str,
    tau: float,
    left_s: float = 6.0,
    right_s: float = 4.0,
) -> Dict[str, Any]:
    if ASR_SEGMENTS is None or len(ASR_SEGMENTS) == 0:
        return {
            "asr_left": "",
            "asr_right": "",
            "asr_boundary_dist": 999.0,
            "asr_has_left": 0,
            "asr_has_right": 0,
        }

    tau = float(tau)
    seg = ASR_SEGMENTS[ASR_SEGMENTS["session"].astype(str) == str(session)]

    if len(seg) == 0:
        return {
            "asr_left": "",
            "asr_right": "",
            "asr_boundary_dist": 999.0,
            "asr_has_left": 0,
            "asr_has_right": 0,
        }

    seg = seg.copy()
    seg["start"] = pd.to_numeric(seg["start"], errors="coerce")
    seg["end"] = pd.to_numeric(seg["end"], errors="coerce")
    seg = seg.dropna(subset=["start", "end"])

    left = seg[(seg["end"] <= tau) & (seg["end"] >= tau - float(left_s))].sort_values("end")
    right = seg[(seg["start"] >= tau) & (seg["start"] <= tau + float(right_s))].sort_values("start")

    boundary_times = np.r_[
        seg["start"].to_numpy(dtype=float),
        seg["end"].to_numpy(dtype=float),
    ]

    bdist = float(np.min(np.abs(boundary_times - tau))) if len(boundary_times) else 999.0

    left_text = " ".join(left["text"].astype(str).tolist())[-800:]
    right_text = " ".join(right["text"].astype(str).tolist())[:800]

    return {
        "asr_left": left_text,
        "asr_right": right_text,
        "asr_boundary_dist": bdist,
        "asr_has_left": int(len(left) > 0),
        "asr_has_right": int(len(right) > 0),
    }

Loaded ASR segments: (115399, 10)
ASR segments: (115399, 10)


,session,start,end,text,asr_model,language,language_probability,split,is_synthetic,orig_session
0,abjxc,0.05,4.51,CNN has decided not to run a Donald Trump endo...,medium,en,1.0,train,False,NaN
1,abjxc,4.51,6.93,network decided it does not meet their standards.,medium,en,1.0,train,False,NaN
2,abjxc,8.51,13.35,The 30-second advertisement paid for by Trump'...,medium,en,1.0,train,False,NaN
3,abjxc,13.35,17.51,toward Ukraine while he served as vice preside...,medium,en,1.0,train,False,NaN
4,abjxc,17.51,22.51,"personalities Don Lemon, Chris Cuomo and Jim A...",medium,en,1.0,train,False,NaN


In [13]:
PYANNOTE_TURNS_PATH = PYANNOTE_CACHE_DIR / "pyannote_turns_all.pkl"
PYANNOTE_CANDS_PATH = PYANNOTE_CACHE_DIR / "pyannote_candidates_all.pkl"
PYANNOTE_META_PATH = PYANNOTE_CACHE_DIR / "pyannote_cache_meta.json"

PYANNOTE_TURN_COLUMNS = [
    "session", "start", "end", "speaker", "source", "split",
    "is_synthetic", "orig_session",
]

PYANNOTE_CAND_COLUMNS = [
    "session", "time",
    "source_pyannote",
    "source_pyannote_change",
    "source_pyannote_overlap",
    "pyannote_candidate_type",
    "pyannote_gap",
    "pyannote_prev_speaker",
    "pyannote_next_speaker",
    "pyannote_change_score",
    "pyannote_overlap_score",
    "split",
    "is_synthetic",
    "orig_session",
]

def _pyannote_config() -> Dict[str, Any]:
    return {
        "run_version": str(ASSET_CACHE_RUN_VERSION),
        "seed": int(SEED),
        "use_pyannote_dia": bool(USE_PYANNOTE_DIA),
        "pyannote_pipeline_name": str(PYANNOTE_PIPELINE_NAME),
        "debug_small_run": bool(DEBUG_SMALL_RUN),
        "debug_manifest_limit": 60 if DEBUG_SMALL_RUN else -1,
        "include_real_train": True,
        "include_real_val": True,
        "include_real_test": True,
        "include_synthetic": bool(USE_SYNTH_MULTIPARTY),
        "copy_augmented_from_original": True,
        "candidate_types": ["speaker_change", "overlap_start", "overlap_end"],
        "cache_format": "pickle",
    }

def _pyannote_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False
    try:
        return json.loads(Path(path).read_text()) == expected
    except Exception:
        return False

def _clean_legacy_pyannote_parquet():
    for p in [
        PYANNOTE_CACHE_DIR / "pyannote_turns_all.parquet",
        PYANNOTE_CACHE_DIR / "pyannote_candidates_all.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet pyannote cache:", p)
        except Exception as e:
            print("Could not remove legacy parquet pyannote cache:", p, e)

def _empty_pyannote_turns() -> pd.DataFrame:
    return pd.DataFrame(columns=PYANNOTE_TURN_COLUMNS)

def _empty_pyannote_candidates() -> pd.DataFrame:
    return pd.DataFrame(columns=PYANNOTE_CAND_COLUMNS)

def _ensure_pyannote_turn_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return _empty_pyannote_turns()

    df = df.copy()

    for col in PYANNOTE_TURN_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df["session"] = df["session"].astype(str)
    df["speaker"] = df["speaker"].astype(str)
    df["source"] = df["source"].fillna("pyannote").astype(str)
    df["split"] = df["split"].fillna("unknown").astype(str)
    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["orig_session"] = df["orig_session"].fillna(np.nan)

    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"] = pd.to_numeric(df["end"], errors="coerce")

    df = df.dropna(subset=["session", "start", "end", "speaker"])
    df = df[df["end"] > df["start"]].reset_index(drop=True)

    return (
        df[PYANNOTE_TURN_COLUMNS]
        .drop_duplicates()
        .sort_values(["session", "start", "end", "speaker"])
        .reset_index(drop=True)
    )

def _ensure_pyannote_candidate_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return _empty_pyannote_candidates()

    df = df.copy()

    for col in PYANNOTE_CAND_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df["session"] = df["session"].astype(str)
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df = df.dropna(subset=["session", "time"])
    df = df[df["time"] >= 0].reset_index(drop=True)

    df["source_pyannote"] = pd.to_numeric(df["source_pyannote"], errors="coerce").fillna(1).astype(int)
    df["source_pyannote_change"] = pd.to_numeric(df["source_pyannote_change"], errors="coerce").fillna(0).astype(int)
    df["source_pyannote_overlap"] = pd.to_numeric(df["source_pyannote_overlap"], errors="coerce").fillna(0).astype(int)

    df["pyannote_candidate_type"] = df["pyannote_candidate_type"].fillna("unknown").astype(str)
    df["pyannote_prev_speaker"] = df["pyannote_prev_speaker"].fillna("").astype(str)
    df["pyannote_next_speaker"] = df["pyannote_next_speaker"].fillna("").astype(str)
    df["split"] = df["split"].fillna("unknown").astype(str)
    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["orig_session"] = df["orig_session"].fillna(np.nan)

    df["pyannote_change_score"] = pd.to_numeric(df["pyannote_change_score"], errors="coerce").fillna(0.0)
    df["pyannote_overlap_score"] = pd.to_numeric(df["pyannote_overlap_score"], errors="coerce").fillna(0.0)
    df["pyannote_gap"] = pd.to_numeric(df["pyannote_gap"], errors="coerce")

    return (
        df[PYANNOTE_CAND_COLUMNS]
        .drop_duplicates(subset=["session", "time", "pyannote_candidate_type"])
        .sort_values(["session", "time", "pyannote_candidate_type"])
        .reset_index(drop=True)
    )

def collect_audio_manifest_for_diarization() -> pd.DataFrame:
    rows = []

    for sid in TRAIN_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "train",
            "is_synthetic": False,
        })

    for sid in VAL_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "val",
            "is_synthetic": False,
        })

    for sid in TEST_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "test")),
            "split": "test",
            "is_synthetic": False,
        })

    if USE_SYNTH_MULTIPARTY and SYNTH_MANIFEST is not None and len(SYNTH_MANIFEST):
        rows.extend(
            SYNTH_MANIFEST[["session", "audio_path"]]
            .copy()
            .assign(split="synth_train", is_synthetic=True)
            .to_dict("records")
        )

    manifest = pd.DataFrame(rows)

    if len(manifest) == 0:
        return pd.DataFrame(columns=["session", "audio_path", "split", "is_synthetic"])

    manifest["session"] = manifest["session"].astype(str)
    manifest["audio_path"] = manifest["audio_path"].astype(str)
    manifest["split"] = manifest["split"].astype(str)
    manifest["is_synthetic"] = manifest["is_synthetic"].fillna(False).astype(bool)

    manifest = manifest.drop_duplicates(subset=["session"]).reset_index(drop=True)

    exists_mask = manifest["audio_path"].map(lambda p: Path(p).exists())

    if not bool(exists_mask.all()):
        missing = manifest.loc[~exists_mask, ["session", "audio_path"]].head(10)
        print("⚠️ Some pyannote audio files do not exist. First examples:")
        display(missing)

    manifest = manifest.loc[exists_mask].reset_index(drop=True)

    return manifest

def _load_pyannote_pipeline():
    import inspect

    try:
        import pyannote.audio
        from pyannote.audio import Pipeline
    except Exception as e:
        raise ImportError(
            "pyannote.audio import failed. This is an environment/dependency issue, "
            "not a notebook logic issue. Check pyannote.audio, torch, lightning, scipy, "
            "torchmetrics, transformers, and huggingface_hub versions."
        ) from e

    hf_token = os.environ.get("HF_TOKEN", globals().get("HF_TOKEN", None))

    if hf_token is None or str(hf_token).strip() == "":
        raise RuntimeError(
            "HF_TOKEN is not set. Set it before running pyannote. Also make sure you have "
            "accepted the Hugging Face conditions for the pyannote pipeline you are using."
        )

    hf_token = str(hf_token).strip()

    print("Loading pyannote pipeline:", PYANNOTE_PIPELINE_NAME)
    print("pyannote.audio version:", getattr(pyannote.audio, "__version__", "unknown"))

    try:
        sig = inspect.signature(Pipeline.from_pretrained)
        params = set(sig.parameters.keys())
        print("Pipeline.from_pretrained parameters:", sorted(params))
    except Exception as e:
        print("Could not inspect Pipeline.from_pretrained signature:", repr(e))
        params = set()

    pipe = None
    first_error = None

    try:
        if "token" in params:
            pipe = Pipeline.from_pretrained(
                PYANNOTE_PIPELINE_NAME,
                token=hf_token,
            )

        elif "use_auth_token" in params:
            pipe = Pipeline.from_pretrained(
                PYANNOTE_PIPELINE_NAME,
                use_auth_token=hf_token,
            )

        else:
            raise TypeError("No token/use_auth_token parameter found in inspected signature.")

    except Exception as e:
        first_error = e
        print("First pyannote load attempt failed:")
        print(repr(e))

    if pipe is None:
        try:
            print("Trying explicit token=... load.")
            pipe = Pipeline.from_pretrained(
                PYANNOTE_PIPELINE_NAME,
                token=hf_token,
            )
        except Exception as e:
            print("Explicit token=... load failed:")
            print(repr(e))

    if pipe is None:
        try:
            print("Trying explicit use_auth_token=... load.")
            pipe = Pipeline.from_pretrained(
                PYANNOTE_PIPELINE_NAME,
                use_auth_token=hf_token,
            )
        except Exception as e:
            print("Explicit use_auth_token=... load failed:")
            print(repr(e))

    if pipe is None:
        try:
            from huggingface_hub import login
            login(token=hf_token, add_to_git_credential=False)
            print("Logged into Hugging Face through huggingface_hub.login(...).")
        except Exception as e:
            print("Hugging Face login fallback failed:")
            print(repr(e))

        try:
            print("Trying Pipeline.from_pretrained(...) without token keyword after global login.")
            pipe = Pipeline.from_pretrained(PYANNOTE_PIPELINE_NAME)
        except Exception as e:
            raise RuntimeError(
                "Could not load pyannote pipeline after all authentication attempts.\n"
                f"Pipeline name: {PYANNOTE_PIPELINE_NAME}\n"
                f"First error: {repr(first_error)}\n"
                f"Final error: {repr(e)}\n\n"
                "Check that your pyannote.audio version matches the selected pipeline:\n"
                "  - pyannote/speaker-diarization-community-1 usually expects the newer token=... API.\n"
                "  - pyannote/speaker-diarization-3.1 usually expects the legacy use_auth_token=... API.\n"
                "Also confirm that your HF_TOKEN has access and that you accepted the model conditions."
            ) from e

    if torch.cuda.is_available():
        try:
            pipe.to(torch.device("cuda"))
            print("pyannote pipeline moved to CUDA.")
        except Exception as e:
            print("Could not move pyannote pipeline to CUDA; continuing on default device.")
            print(repr(e))
    else:
        print("pyannote pipeline running on CPU.")

    return pipe

def _extract_annotation_from_pyannote_output(output):
    if output is None:
        return None

    if hasattr(output, "itertracks"):
        return output

    for attr in ["speaker_diarization", "exclusive_speaker_diarization", "diarization"]:
        if hasattr(output, attr):
            ann = getattr(output, attr)
            if ann is not None and hasattr(ann, "itertracks"):
                return ann

    if isinstance(output, dict):
        for key in ["speaker_diarization", "exclusive_speaker_diarization", "diarization"]:
            ann = output.get(key, None)
            if ann is not None and hasattr(ann, "itertracks"):
                return ann

    return None

def _candidate_rows_from_pyannote_turns(
    sid: str,
    turns: List[Dict[str, Any]],
    split: str,
    is_synthetic: bool,
) -> List[Dict[str, Any]]:
    cand_rows = []

    if not turns:
        return cand_rows

    tdf = pd.DataFrame(turns)

    if len(tdf) == 0:
        return cand_rows

    tdf["start"] = pd.to_numeric(tdf["start"], errors="coerce")
    tdf["end"] = pd.to_numeric(tdf["end"], errors="coerce")
    tdf = tdf.dropna(subset=["start", "end", "speaker"])
    tdf = tdf[tdf["end"] > tdf["start"]].sort_values(["start", "end", "speaker"]).reset_index(drop=True)

    if len(tdf) >= 2:
        for i in range(len(tdf) - 1):
            a = tdf.iloc[i]
            b = tdf.iloc[i + 1]

            a_spk = str(a.speaker)
            b_spk = str(b.speaker)
            gap = float(b.start - a.end)

            if a_spk != b_spk:
                tau = 0.5 * (float(a.end) + float(b.start))

                if np.isfinite(tau) and tau >= 0:
                    cand_rows.append({
                        "session": sid,
                        "time": float(tau),
                        "source_pyannote": 1,
                        "source_pyannote_change": 1,
                        "source_pyannote_overlap": 0,
                        "pyannote_candidate_type": "speaker_change",
                        "pyannote_gap": gap,
                        "pyannote_prev_speaker": a_spk,
                        "pyannote_next_speaker": b_spk,
                        "pyannote_change_score": 1.0,
                        "pyannote_overlap_score": max(0.0, min(1.0, -gap / 0.75)) if gap < 0 else 0.0,
                        "split": split,
                        "is_synthetic": bool(is_synthetic),
                        "orig_session": np.nan,
                    })

    overlap_rows = []

    for i in range(len(tdf)):
        a = tdf.iloc[i]

        for j in range(i + 1, len(tdf)):
            b = tdf.iloc[j]

            if float(b.start) >= float(a.end):
                break

            if str(a.speaker) == str(b.speaker):
                continue

            ov_start = max(float(a.start), float(b.start))
            ov_end = min(float(a.end), float(b.end))
            ov_dur = ov_end - ov_start

            if ov_dur < 0.05:
                continue

            overlap_rows.append((ov_start, ov_end, str(a.speaker), str(b.speaker), ov_dur))

    for ov_start, ov_end, spk_a, spk_b, ov_dur in overlap_rows:
        for cand_type, tau in [("overlap_start", ov_start), ("overlap_end", ov_end)]:
            cand_rows.append({
                "session": sid,
                "time": float(max(0.0, tau)),
                "source_pyannote": 1,
                "source_pyannote_change": 0,
                "source_pyannote_overlap": 1,
                "pyannote_candidate_type": cand_type,
                "pyannote_gap": -float(ov_dur),
                "pyannote_prev_speaker": spk_a,
                "pyannote_next_speaker": spk_b,
                "pyannote_change_score": 0.0,
                "pyannote_overlap_score": max(0.0, min(1.0, ov_dur / 0.75)),
                "split": split,
                "is_synthetic": bool(is_synthetic),
                "orig_session": np.nan,
            })

    return cand_rows

def run_pyannote_pipeline(manifest: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if manifest is None or len(manifest) == 0:
        return _empty_pyannote_turns(), _empty_pyannote_candidates()

    pipe = _load_pyannote_pipeline()

    turn_rows = []
    cand_rows = []
    failed_rows = []

    for _, r in tqdm(manifest.iterrows(), total=len(manifest), desc="pyannote diarization"):
        sid = str(r.session)
        audio_path = str(r.audio_path)
        split = str(r.get("split", "unknown"))
        is_synthetic = bool(r.get("is_synthetic", False))

        try:
            if not Path(audio_path).exists():
                raise FileNotFoundError(f"Audio path does not exist: {audio_path}")

            wav_np, sr = read_audio_mono(Path(audio_path), sr=TARGET_SR)

            if wav_np is None or len(wav_np) == 0:
                raise RuntimeError(f"Empty waveform after loading: {audio_path}")

            wav_np = np.asarray(wav_np, dtype=np.float32)
            wav_np = np.nan_to_num(wav_np, nan=0.0, posinf=0.0, neginf=0.0)

            wav_tensor = torch.from_numpy(wav_np).float().unsqueeze(0)

            diar_input = {
                "waveform": wav_tensor,
                "sample_rate": int(sr),
            }

            diar_output = pipe(diar_input)
            diar = _extract_annotation_from_pyannote_output(diar_output)

            if diar is None:
                print("pyannote produced unsupported output for:", sid, type(diar_output))
                continue

            turns = []

            for turn, _, spk in diar.itertracks(yield_label=True):
                start = float(turn.start)
                end = float(turn.end)

                if not np.isfinite(start) or not np.isfinite(end) or end <= start:
                    continue

                turns.append({
                    "session": sid,
                    "start": start,
                    "end": end,
                    "speaker": str(spk),
                    "source": "pyannote",
                    "split": split,
                    "is_synthetic": bool(is_synthetic),
                    "orig_session": np.nan,
                })

            turn_rows.extend(turns)

            cand_rows.extend(
                _candidate_rows_from_pyannote_turns(
                    sid=sid,
                    turns=turns,
                    split=split,
                    is_synthetic=is_synthetic,
                )
            )

        except Exception as e:
            failed_rows.append({
                "session": sid,
                "audio_path": audio_path,
                "split": split,
                "error": repr(e),
            })

            if len(failed_rows) <= 20:
                print("pyannote failed:", sid, e)
            elif len(failed_rows) == 21:
                print("Further pyannote failures suppressed. See failed_rows summary below.")

    turns_df = _ensure_pyannote_turn_schema(pd.DataFrame(turn_rows))
    cands_df = _ensure_pyannote_candidate_schema(pd.DataFrame(cand_rows))

    if failed_rows:
        PYANNOTE_FAILED_PATH = PYANNOTE_CACHE_DIR / "pyannote_failed_sessions.csv"
        pd.DataFrame(failed_rows).to_csv(PYANNOTE_FAILED_PATH, index=False)
        print(f"⚠️ pyannote failed for {len(failed_rows)} / {len(manifest)} sessions.")
        print("Saved pyannote failure log:", PYANNOTE_FAILED_PATH)

        fail_df = pd.DataFrame(failed_rows)
        display(fail_df["error"].value_counts().head(10).to_frame("count"))

    print("pyannote successful sessions with turns:", turns_df["session"].nunique() if len(turns_df) else 0)
    print("pyannote successful sessions with candidates:", cands_df["session"].nunique() if len(cands_df) else 0)

    return turns_df, cands_df

def copy_pyannote_to_augmented_sessions(
    turns_df: pd.DataFrame,
    cands_df: pd.DataFrame,
    aug_manifest: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    turns_df = _ensure_pyannote_turn_schema(turns_df)
    cands_df = _ensure_pyannote_candidate_schema(cands_df)

    if aug_manifest is None or len(aug_manifest) == 0:
        return turns_df, cands_df

    turn_copies = []
    cand_copies = []

    for _, r in aug_manifest.iterrows():
        orig = str(r.get("orig_session", ""))
        aug = str(r.get("session", ""))

        if not orig or not aug:
            continue

        t = turns_df[turns_df["session"].astype(str) == orig].copy()

        if len(t):
            t["orig_session"] = orig
            t["session"] = aug

            if "kind" in r:
                t["split"] = str(r["kind"])

            turn_copies.append(t)

        c = cands_df[cands_df["session"].astype(str) == orig].copy()

        if len(c):
            c["orig_session"] = orig
            c["session"] = aug

            if "kind" in r:
                c["split"] = str(r["kind"])

            cand_copies.append(c)

    if turn_copies:
        turns_df = pd.concat([turns_df] + turn_copies, ignore_index=True)

    if cand_copies:
        cands_df = pd.concat([cands_df] + cand_copies, ignore_index=True)

    turns_df = _ensure_pyannote_turn_schema(turns_df).drop_duplicates().reset_index(drop=True)
    cands_df = _ensure_pyannote_candidate_schema(cands_df).drop_duplicates().reset_index(drop=True)

    return turns_df, cands_df

_clean_legacy_pyannote_parquet()

if USE_PYANNOTE_DIA:
    expected_pyannote_meta = _pyannote_config()

    if (
        PYANNOTE_TURNS_PATH.exists()
        and PYANNOTE_CANDS_PATH.exists()
        and _pyannote_meta_matches(PYANNOTE_META_PATH, expected_pyannote_meta)
        and not FORCE_RERUN_PYANNOTE
    ):
        PYANNOTE_TURNS = pd.read_pickle(PYANNOTE_TURNS_PATH)
        PYANNOTE_CANDIDATES = pd.read_pickle(PYANNOTE_CANDS_PATH)

        PYANNOTE_TURNS = _ensure_pyannote_turn_schema(PYANNOTE_TURNS)
        PYANNOTE_CANDIDATES = _ensure_pyannote_candidate_schema(PYANNOTE_CANDIDATES)

        print("Loaded pyannote:", PYANNOTE_TURNS.shape, PYANNOTE_CANDIDATES.shape)

    else:
        dia_manifest = collect_audio_manifest_for_diarization()

        if DEBUG_SMALL_RUN:
            dia_manifest = dia_manifest.head(60).reset_index(drop=True)

        print("pyannote manifest:", dia_manifest.shape)

        PYANNOTE_TURNS, PYANNOTE_CANDIDATES = run_pyannote_pipeline(dia_manifest)

        PYANNOTE_TURNS, PYANNOTE_CANDIDATES = copy_pyannote_to_augmented_sessions(
            PYANNOTE_TURNS,
            PYANNOTE_CANDIDATES,
            AUG_MANIFEST,
        )

        PYANNOTE_TURNS.to_pickle(PYANNOTE_TURNS_PATH)
        PYANNOTE_CANDIDATES.to_pickle(PYANNOTE_CANDS_PATH)
        PYANNOTE_META_PATH.write_text(json.dumps(expected_pyannote_meta, indent=2, sort_keys=True))

else:
    PYANNOTE_TURNS = _empty_pyannote_turns()
    PYANNOTE_CANDIDATES = _empty_pyannote_candidates()

if USE_PYANNOTE_DIA and len(AUG_MANIFEST) and (len(PYANNOTE_TURNS) or len(PYANNOTE_CANDIDATES)):
    before_turns = len(PYANNOTE_TURNS)
    before_cands = len(PYANNOTE_CANDIDATES)

    PYANNOTE_TURNS, PYANNOTE_CANDIDATES = copy_pyannote_to_augmented_sessions(
        PYANNOTE_TURNS,
        PYANNOTE_CANDIDATES,
        AUG_MANIFEST,
    )

    after_turns = len(PYANNOTE_TURNS)
    after_cands = len(PYANNOTE_CANDIDATES)

    if after_turns > before_turns or after_cands > before_cands:
        PYANNOTE_TURNS.to_pickle(PYANNOTE_TURNS_PATH)
        PYANNOTE_CANDIDATES.to_pickle(PYANNOTE_CANDS_PATH)
        print(
            f"Added pyannote copies for augmented sessions: "
            f"turns +{after_turns - before_turns}, cands +{after_cands - before_cands}"
        )

print("PYANNOTE_TURNS:", PYANNOTE_TURNS.shape)
print("PYANNOTE_CANDIDATES:", PYANNOTE_CANDIDATES.shape)

if len(PYANNOTE_CANDIDATES):
    display(PYANNOTE_CANDIDATES["pyannote_candidate_type"].value_counts().to_frame("count"))

def pyannote_features(session: str, tau: float) -> Dict[str, float]:
    tau = float(tau)

    if PYANNOTE_CANDIDATES is not None and len(PYANNOTE_CANDIDATES):
        cands = PYANNOTE_CANDIDATES[PYANNOTE_CANDIDATES["session"].astype(str) == str(session)]
    else:
        cands = pd.DataFrame()

    if len(cands):
        cand_times = cands["time"].to_numpy(dtype=float)
        dist = float(np.min(np.abs(cand_times - tau)))

        change_cands = cands[cands["source_pyannote_change"].astype(int) == 1]

        if len(change_cands):
            change_dist = float(np.min(np.abs(change_cands["time"].to_numpy(dtype=float) - tau)))
        else:
            change_dist = 999.0

        overlap_cands = cands[cands["source_pyannote_overlap"].astype(int) == 1]

        if len(overlap_cands):
            overlap_dist = float(np.min(np.abs(overlap_cands["time"].to_numpy(dtype=float) - tau)))
        else:
            overlap_dist = 999.0

    else:
        dist = 999.0
        change_dist = 999.0
        overlap_dist = 999.0

    if PYANNOTE_TURNS is not None and len(PYANNOTE_TURNS):
        turns = PYANNOTE_TURNS[PYANNOTE_TURNS["session"].astype(str) == str(session)].copy()
    else:
        turns = pd.DataFrame()

    if len(turns):
        turns["start"] = pd.to_numeric(turns["start"], errors="coerce")
        turns["end"] = pd.to_numeric(turns["end"], errors="coerce")
        turns = turns.dropna(subset=["start", "end", "speaker"])

        before = turns[(turns["start"] <= tau) & (turns["end"] >= tau - 0.5)]
        after = turns[(turns["start"] <= tau + 0.5) & (turns["end"] >= tau)]

        active_before = int(before["speaker"].nunique())
        active_after = int(after["speaker"].nunique())

        local = turns[(turns["start"] <= tau + 0.25) & (turns["end"] >= tau - 0.25)]
        local_active = int(local["speaker"].nunique())
        local_overlap_flag = int(local_active >= 2)

    else:
        active_before = 0
        active_after = 0
        local_active = 0
        local_overlap_flag = 0

    return {
        "pyannote_change_dist": change_dist,
        "pyannote_any_candidate_dist": dist,
        "pyannote_overlap_dist": overlap_dist,
        "pyannote_change_score": float(max(0.0, 1.0 - change_dist / 0.75)) if np.isfinite(change_dist) else 0.0,
        "pyannote_overlap_score": float(max(0.0, 1.0 - overlap_dist / 0.75)) if np.isfinite(overlap_dist) else 0.0,
        "pyannote_active_before": active_before,
        "pyannote_active_after": active_after,
        "pyannote_active_delta": active_after - active_before,
        "pyannote_local_active": local_active,
        "pyannote_local_overlap_flag": local_overlap_flag,
    }

Loaded pyannote: (119746, 8) (145059, 14)
PYANNOTE_TURNS: (119746, 8)
PYANNOTE_CANDIDATES: (145059, 14)


,count
pyannote_candidate_type,
speaker_change,74471
overlap_start,35294
overlap_end,35294


In [14]:
# ECAPA speaker embeddings and before/after similarity
import atexit
import gc

from collections import OrderedDict
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import pandas as pd
import torch

ECAPA_FEATURES_PATH = ECAPA_CACHE_DIR / "ecapa_candidate_features.pkl"

ECAPA_EMB_CACHE_PATH = (
    ECAPA_CACHE_DIR
    / "ecapa_segment_embedding_cache.pkl"
)

ECAPA_BOUNDED_CACHE_PATH = (
    ECAPA_CACHE_DIR
    / "ecapa_segment_embedding_cache_bounded.pkl"
)

ECAPA_MODEL_SOURCE = str(
    globals().get(
        "ECAPA_MODEL_SOURCE",
        "speechbrain/spkrec-ecapa-voxceleb",
    )
)

ECAPA_LEFT_S = float(
    globals().get("ECAPA_LEFT_S", 1.25)
)

ECAPA_RIGHT_S = float(
    globals().get("ECAPA_RIGHT_S", 1.25)
)

ECAPA_GUARD_S = float(
    globals().get("ECAPA_GUARD_S", 0.08)
)

ECAPA_MIN_SEG_S = float(
    globals().get("ECAPA_MIN_SEG_S", 0.25)
)

ECAPA_MAX_IN_MEMORY_EMBEDDINGS = int(
    globals().get(
        "ECAPA_MAX_IN_MEMORY_EMBEDDINGS",
        20_000,
    )
)

ECAPA_CACHE_AUTOSAVE_EVERY = int(
    globals().get(
        "ECAPA_CACHE_AUTOSAVE_EVERY",
        2_000,
    )
)

ECAPA_LOAD_BOUNDED_CACHE = bool(
    globals().get(
        "ECAPA_LOAD_BOUNDED_CACHE",
        True,
    )
)

ECAPA_LOAD_LEGACY_CACHE = bool(
    globals().get(
        "ECAPA_LOAD_LEGACY_CACHE",
        False,
    )
)

ECAPA_PERSIST_BOUNDED_CACHE = bool(
    globals().get(
        "ECAPA_PERSIST_BOUNDED_CACHE",
        True,
    )
)

ECAPA_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ECAPA_EMB_CACHE = OrderedDict()
_ECAPA_NEW_EMBEDDINGS_SINCE_SAVE = 0

def _clean_ecapa_embedding(
    value,
) -> Optional[np.ndarray]:
    try:
        arr = np.asarray(
            value,
            dtype=np.float32,
        ).reshape(-1)

        if arr.size == 0:
            return None

        if not np.isfinite(arr).all():
            return None

        norm = float(
            np.linalg.norm(arr)
        )

        if not np.isfinite(norm) or norm < 1e-8:
            return None

        arr = arr / norm

        return arr.astype(
            np.float32,
            copy=False,
        )

    except Exception:
        return None

def _trim_ecapa_cache():
    max_entries = max(
        0,
        int(ECAPA_MAX_IN_MEMORY_EMBEDDINGS),
    )

    if max_entries == 0:
        ECAPA_EMB_CACHE.clear()
        return

    while len(ECAPA_EMB_CACHE) > max_entries:
        ECAPA_EMB_CACHE.popitem(
            last=False,
        )

def _load_ecapa_embedding_cache(
    cache_path: Path,
    label: str,
) -> int:
    cache_path = Path(cache_path)

    if not cache_path.exists():
        print(
            f"{label} ECAPA cache not found:",
            cache_path,
        )
        return 0

    try:
        loaded = pd.read_pickle(
            cache_path
        )

        if not isinstance(loaded, dict):
            print(
                f"{label} ECAPA cache is not a dictionary."
            )
            return 0

        cleaned = OrderedDict()

        for key, value in loaded.items():
            emb = _clean_ecapa_embedding(
                value
            )

            if emb is not None:
                cleaned[str(key)] = emb

        ECAPA_EMB_CACHE.clear()
        ECAPA_EMB_CACHE.update(cleaned)

        _trim_ecapa_cache()

        del loaded
        del cleaned
        gc.collect()

        print(
            f"Loaded {label} ECAPA cache:",
            len(ECAPA_EMB_CACHE),
            "entries",
        )

        return len(ECAPA_EMB_CACHE)

    except Exception as exc:
        print(
            f"Could not load {label} ECAPA cache:",
            repr(exc),
        )

        ECAPA_EMB_CACHE.clear()
        gc.collect()

        return 0

if not bool(
    globals().get(
        "FORCE_REBUILD_SYS2_FEATURES",
        False,
    )
):
    if (
        ECAPA_LOAD_BOUNDED_CACHE
        and ECAPA_BOUNDED_CACHE_PATH.exists()
    ):
        _load_ecapa_embedding_cache(
            ECAPA_BOUNDED_CACHE_PATH,
            label="bounded",
        )

    elif (
        ECAPA_LOAD_LEGACY_CACHE
        and ECAPA_EMB_CACHE_PATH.exists()
    ):
        print(
            "WARNING: loading the legacy monolithic ECAPA cache."
        )
        print(
            "This may consume several gigabytes of memory."
        )

        _load_ecapa_embedding_cache(
            ECAPA_EMB_CACHE_PATH,
            label="legacy",
        )

else:
    print(
        "FORCE_REBUILD_SYS2_FEATURES=True:"
        " starting with an empty ECAPA segment cache."
    )

def _ecapa_cache_get(
    key: str,
) -> Optional[np.ndarray]:
    key = str(key)

    if key not in ECAPA_EMB_CACHE:
        return None

    value = ECAPA_EMB_CACHE.pop(
        key
    )

    ECAPA_EMB_CACHE[key] = value

    return value

def _ecapa_cache_put(
    key: str,
    embedding: np.ndarray,
):
    global _ECAPA_NEW_EMBEDDINGS_SINCE_SAVE

    emb = _clean_ecapa_embedding(
        embedding
    )

    if emb is None:
        return

    key = str(key)

    if key in ECAPA_EMB_CACHE:
        ECAPA_EMB_CACHE.pop(
            key,
            None,
        )

    ECAPA_EMB_CACHE[key] = emb

    _trim_ecapa_cache()

    _ECAPA_NEW_EMBEDDINGS_SINCE_SAVE += 1

def _patch_hf_hub_use_auth_token_compat():
    try:
        import inspect
        import huggingface_hub

        original = (
            huggingface_hub.hf_hub_download
        )

        if getattr(
            original,
            "_ecapa_use_auth_token_patch",
            False,
        ):
            return True

        try:
            parameters = set(
                inspect.signature(
                    original
                ).parameters.keys()
            )
        except Exception:
            parameters = set()

        if "use_auth_token" in parameters:
            return True

        def hf_hub_download_compat(
            *args,
            use_auth_token=None,
            token=None,
            **kwargs,
        ):
            if (
                token is None
                and use_auth_token not in [None, False]
            ):
                token = use_auth_token

            return original(
                *args,
                token=token,
                **kwargs,
            )

        hf_hub_download_compat._ecapa_use_auth_token_patch = True
        hf_hub_download_compat._ecapa_original_hf_hub_download = original

        huggingface_hub.hf_hub_download = (
            hf_hub_download_compat
        )

        for module_name in [
            "speechbrain.utils.fetching",
            "speechbrain.pretrained.fetching",
        ]:
            try:
                module = __import__(
                    module_name,
                    fromlist=["hf_hub_download"],
                )

                setattr(
                    module,
                    "hf_hub_download",
                    hf_hub_download_compat,
                )

            except Exception:
                pass

        print(
            "Applied HF Hub compatibility patch:"
            " use_auth_token -> token"
        )

        return True

    except Exception as exc:
        print(
            "Could not apply HF Hub compatibility patch:",
            repr(exc),
        )

        return False

def _resample_waveform_if_needed(
    wav: np.ndarray,
    orig_sr: int,
    target_sr: int = TARGET_SR,
) -> Optional[np.ndarray]:
    wav = np.asarray(
        wav,
        dtype=np.float32,
    )

    if int(orig_sr) == int(target_sr):
        return wav.astype(
            np.float32,
            copy=False,
        )

    try:
        import torchaudio.functional as AF

        wav_t = torch.as_tensor(
            wav,
            dtype=torch.float32,
        ).unsqueeze(0)

        output = AF.resample(
            wav_t,
            orig_freq=int(orig_sr),
            new_freq=int(target_sr),
        )

        output = (
            output
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
        )

        return output.astype(
            np.float32,
            copy=False,
        )

    except Exception as exc:
        print(
            f"ECAPA resample failed: "
            f"{orig_sr} -> {target_sr}; "
            f"returning None. Error: {exc}"
        )

        return None

def _load_speechbrain_encoder_classifier_class():
    try:
        from speechbrain.inference.speaker import EncoderClassifier

        return (
            EncoderClassifier,
            "speechbrain.inference.speaker.EncoderClassifier",
        )

    except Exception as error_1:
        try:
            from speechbrain.inference.classifiers import EncoderClassifier

            return (
                EncoderClassifier,
                "speechbrain.inference.classifiers.EncoderClassifier",
            )

        except Exception as error_2:
            try:
                from speechbrain.pretrained import EncoderClassifier

                return (
                    EncoderClassifier,
                    "speechbrain.pretrained.EncoderClassifier",
                )

            except Exception as error_3:
                raise ImportError(
                    "Could not import SpeechBrain EncoderClassifier "
                    "from any supported path.\n"
                    f"speechbrain.inference.speaker: {repr(error_1)}\n"
                    f"speechbrain.inference.classifiers: {repr(error_2)}\n"
                    f"speechbrain.pretrained: {repr(error_3)}"
                ) from error_3

def _try_load_ecapa_from_hparams(
    EncoderClassifier,
    savedir: Path,
):
    savedir = Path(savedir)

    savedir.mkdir(
        parents=True,
        exist_ok=True,
    )

    base_kwargs = {
        "source": ECAPA_MODEL_SOURCE,
        "savedir": str(savedir),
        "run_opts": {
            "device": DEVICE,
        },
    }

    attempts = [
        {
            **base_kwargs,
            "pymodule_file": None,
        },
        base_kwargs.copy(),
    ]

    last_error = None

    for attempt_index, kwargs in enumerate(
        attempts,
        start=1,
    ):
        try:
            shown_kwargs = {
                key: value
                for key, value in kwargs.items()
                if key != "run_opts"
            }

            print(
                f"Trying ECAPA load attempt {attempt_index}:",
                shown_kwargs,
            )

            return EncoderClassifier.from_hparams(
                **kwargs
            )

        except TypeError as exc:
            last_error = exc

            if "pymodule_file" in kwargs:
                fallback_kwargs = kwargs.copy()

                fallback_kwargs.pop(
                    "pymodule_file",
                    None,
                )

                try:
                    print(
                        "Retrying ECAPA load without pymodule_file."
                    )

                    return EncoderClassifier.from_hparams(
                        **fallback_kwargs
                    )

                except Exception as fallback_exc:
                    last_error = fallback_exc

        except Exception as exc:
            last_error = exc

            print(
                f"ECAPA load attempt {attempt_index} failed:",
                repr(exc),
            )

    raise RuntimeError(
        "All ECAPA model-load attempts failed. "
        f"Last error: {repr(last_error)}"
    ) from last_error

class ECAPAWrapper:

    def __init__(self):
        self.model = None
        self.available = False

        if not USE_ECAPA_SPK:
            print(
                "ECAPA disabled by USE_ECAPA_SPK=False"
            )
            return

        _patch_hf_hub_use_auth_token_compat()

        savedir = (
            ECAPA_CACHE_DIR
            / "speechbrain_ecapa_model"
        )

        try:
            (
                EncoderClassifier,
                import_name,
            ) = _load_speechbrain_encoder_classifier_class()

            print(
                "Using:",
                import_name,
            )

            self.model = _try_load_ecapa_from_hparams(
                EncoderClassifier=EncoderClassifier,
                savedir=savedir,
            )

            self.available = True

            print(
                "Loaded ECAPA:",
                ECAPA_MODEL_SOURCE,
            )

        except Exception as exc:
            print(
                "ECAPA unavailable."
            )

            print(
                "  error:",
                repr(exc),
            )

            self.available = False

    @torch.inference_mode()
    def embed_waveform(
        self,
        wav: np.ndarray,
        sr: int = TARGET_SR,
    ) -> Optional[np.ndarray]:
        if not self.available:
            return None

        wav = np.asarray(
            wav,
            dtype=np.float32,
        )

        wav = np.nan_to_num(
            wav,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        if int(sr) != int(TARGET_SR):
            wav = _resample_waveform_if_needed(
                wav,
                orig_sr=int(sr),
                target_sr=int(TARGET_SR),
            )

            if wav is None:
                return None

            sr = TARGET_SR

        if len(wav) < int(
            ECAPA_MIN_SEG_S * sr
        ):
            return None

        try:
            tensor = torch.as_tensor(
                wav,
                dtype=torch.float32,
                device=TORCH_DEVICE,
            ).unsqueeze(0)

            embedding = self.model.encode_batch(
                tensor
            )

            embedding = (
                embedding
                .squeeze()
                .detach()
                .float()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            return _clean_ecapa_embedding(
                embedding
            )

        except Exception:
            return None

ECAPA = ECAPAWrapper()

def _ecapa_cache_key(
    audio_path: Path,
    start_s: float,
    duration_s: float,
) -> str:
    try:
        audio_path = (
            Path(audio_path)
            .expanduser()
            .resolve()
        )

    except Exception:
        audio_path = Path(audio_path)

    return (
        f"{audio_path}"
        f"::{float(start_s):.3f}"
        f"::{float(duration_s):.3f}"
    )

def save_ecapa_embedding_cache(
    quiet: bool = False,
):
    global _ECAPA_NEW_EMBEDDINGS_SINCE_SAVE

    if not ECAPA_PERSIST_BOUNDED_CACHE:
        return

    try:
        ECAPA_CACHE_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        serialisable = dict(
            ECAPA_EMB_CACHE
        )

        temporary_path = (
            ECAPA_BOUNDED_CACHE_PATH
            .with_suffix(".tmp.pkl")
        )

        pd.to_pickle(
            serialisable,
            temporary_path,
        )

        temporary_path.replace(
            ECAPA_BOUNDED_CACHE_PATH
        )

        _ECAPA_NEW_EMBEDDINGS_SINCE_SAVE = 0

        if not quiet:
            print(
                "Saved bounded ECAPA cache:",
                len(serialisable),
                "->",
                ECAPA_BOUNDED_CACHE_PATH,
            )

        del serialisable

    except Exception as exc:
        if not quiet:
            print(
                "Could not save bounded ECAPA cache:",
                repr(exc),
            )

def maybe_save_ecapa_embedding_cache():
    if ECAPA_CACHE_AUTOSAVE_EVERY <= 0:
        return

    if (
        _ECAPA_NEW_EMBEDDINGS_SINCE_SAVE
        >= ECAPA_CACHE_AUTOSAVE_EVERY
    ):
        save_ecapa_embedding_cache(
            quiet=True
        )

def ecapa_embed_segment(
    audio_path: Path,
    start_s: float,
    duration_s: float,
    use_cache: bool = True,
) -> Optional[np.ndarray]:
    if not ECAPA.available:
        return None

    audio_path = Path(
        audio_path
    )

    if not audio_path.exists():
        return None

    start_s = max(
        0.0,
        float(start_s),
    )

    duration_s = max(
        float(ECAPA_MIN_SEG_S),
        float(duration_s),
    )

    key = _ecapa_cache_key(
        audio_path,
        start_s,
        duration_s,
    )

    if use_cache:
        cached_embedding = _ecapa_cache_get(
            key
        )

        if cached_embedding is not None:
            return cached_embedding

    try:
        wav, sr = read_audio_mono(
            audio_path,
            start_s=start_s,
            duration_s=duration_s,
            sr=TARGET_SR,
        )

        if wav is None or len(wav) == 0:
            return None

        embedding = ECAPA.embed_waveform(
            wav,
            sr=sr,
        )

        if (
            use_cache
            and embedding is not None
        ):
            _ecapa_cache_put(
                key,
                embedding,
            )

            maybe_save_ecapa_embedding_cache()

        return embedding

    except Exception:
        return None

def ecapa_same_similarity(
    audio_path: Path,
    tau: float,
    left_s: float = ECAPA_LEFT_S,
    right_s: float = ECAPA_RIGHT_S,
    guard_s: float = ECAPA_GUARD_S,
) -> Dict[str, float]:
    default = {
        "ecapa_cos": 0.0,
        "ecapa_available": 0,
        "ecapa_left_rms": 0.0,
        "ecapa_right_rms": 0.0,
    }

    if not ECAPA.available:
        return default

    try:
        audio_path = Path(
            audio_path
        )

        if not audio_path.exists():
            return default

        tau = float(tau)
        left_s = float(left_s)
        right_s = float(right_s)
        guard_s = float(guard_s)

        left_start = max(
            0.0,
            tau - guard_s - left_s,
        )

        right_start = max(
            0.0,
            tau + guard_s,
        )

        left, _ = read_audio_mono(
            audio_path,
            start_s=left_start,
            duration_s=left_s,
            sr=TARGET_SR,
        )

        right, _ = read_audio_mono(
            audio_path,
            start_s=right_start,
            duration_s=right_s,
            sr=TARGET_SR,
        )

        left_rms = rms(
            left
        )

        right_rms = rms(
            right
        )

        embedding_left = ecapa_embed_segment(
            audio_path,
            start_s=left_start,
            duration_s=left_s,
            use_cache=True,
        )

        embedding_right = ecapa_embed_segment(
            audio_path,
            start_s=right_start,
            duration_s=right_s,
            use_cache=True,
        )

        if (
            embedding_left is None
            or embedding_right is None
        ):
            return {
                "ecapa_cos": 0.0,
                "ecapa_available": 0,
                "ecapa_left_rms": float(left_rms),
                "ecapa_right_rms": float(right_rms),
            }

        cosine = float(
            np.dot(
                embedding_left,
                embedding_right,
            )
        )

        if not np.isfinite(cosine):
            cosine = 0.0

        return {
            "ecapa_cos": cosine,
            "ecapa_available": 1,
            "ecapa_left_rms": float(left_rms),
            "ecapa_right_rms": float(right_rms),
        }

    except Exception:
        return default

_previous_handler = globals().get(
    "_ECAPA_ATEXIT_HANDLER",
    None,
)

if _previous_handler is not None:
    try:
        atexit.unregister(
            _previous_handler
        )
    except Exception:
        pass

def _save_ecapa_cache_at_exit():
    save_ecapa_embedding_cache(
        quiet=True
    )

_ECAPA_ATEXIT_HANDLER = (
    _save_ecapa_cache_at_exit
)

atexit.register(
    _ECAPA_ATEXIT_HANDLER
)

print(
    "ECAPA available:",
    ECAPA.available,
)

print(
    "ECAPA model source:",
    ECAPA_MODEL_SOURCE,
)

print(
    "ECAPA candidate-feature cache:",
    ECAPA_FEATURES_PATH,
)

print(
    "Legacy ECAPA segment cache:",
    ECAPA_EMB_CACHE_PATH,
)

print(
    "Legacy cache loaded:",
    ECAPA_LOAD_LEGACY_CACHE,
)

print(
    "Bounded ECAPA cache:",
    ECAPA_BOUNDED_CACHE_PATH,
)

print(
    "Bounded in-memory cache size:",
    len(ECAPA_EMB_CACHE),
)

print(
    "Maximum in-memory embeddings:",
    ECAPA_MAX_IN_MEMORY_EMBEDDINGS,
)

print(
    "ECAPA left/right windows:",
    ECAPA_LEFT_S,
    ECAPA_RIGHT_S,
)

print(
    "ECAPA guard:",
    ECAPA_GUARD_S,
)

print(
    "Protocol note:"
    " ecapa_same_similarity uses post-decision audio and is excluded"
    " from the strict past-only Branch 2 and Branch 3 feature sets."
)

Loaded bounded ECAPA cache: 20000 entries


Applied HF Hub compatibility patch: use_auth_token -> token
Using: speechbrain.inference.speaker.EncoderClassifier
Trying ECAPA load attempt 1: {'source': 'speechbrain/spkrec-ecapa-voxceleb', 'savedir': '<REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1/cache/ecapa/speechbrain_ecapa_model', 'pymodule_file': None}


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Retrying ECAPA load without pymodule_file.


Loaded ECAPA: speechbrain/spkrec-ecapa-voxceleb
ECAPA available: True
ECAPA model source: speechbrain/spkrec-ecapa-voxceleb
ECAPA candidate-feature cache: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1/cache/ecapa/ecapa_candidate_features.pkl
Legacy ECAPA segment cache: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1/cache/ecapa/ecapa_segment_embedding_cache.pkl
Legacy cache loaded: False
Bounded ECAPA cache: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_v1/cache/ecapa/ecapa_segment_embedding_cache_bounded.pkl
Bounded in-memory cache size: 20000
Maximum in-memory embeddings: 20000
ECAPA left/right windows: 1.25 1.25
ECAPA guard: 0.08
Protocol note: ecapa_same_similarity uses post-decision audio and is excluded from the strict past-only Branch 2 and Branch 3 feature sets.


In [15]:
from pathlib import Path

assert "SYS1_MODEL_DIR" in globals(), (
    "SYS1_MODEL_DIR is not defined. Run the path/configuration cells first."
)

SYS1_MODEL_DIR = Path(SYS1_MODEL_DIR)
SYS1_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if bool(globals().get("REUSE_PREVIOUS_SYS1_CHECKPOINT", False)):
    raise RuntimeError(
        "REUSE_PREVIOUS_SYS1_CHECKPOINT=True is incompatible with the "
        f"paper-ready event contract {EVENT_CONTRACT_VERSION!r}. "
        "Set it to False. The old checkpoint was trained on different event "
        "timestamps, long-gap rules, and HOLD-negative exposure."
    )

FORCE_RETRAIN_SYS1 = bool(globals().get("FORCE_RETRAIN_SYS1", False))
RESUME_SYS1_TRAIN = bool(globals().get("RESUME_SYS1_TRAIN", True))
AUTO_LOAD_LATEST_SYS1 = True
SYS1_FORCE_FRESH_TRAIN = bool(FORCE_RETRAIN_SYS1)
SYS1_LOAD_PATH = None
EXPECTED_SYS1_CKPT = (
    SYS1_MODEL_DIR
    / "wavlm_dense_multiscale_sys1_best.pt"
)

print("=" * 80)
print("PAPER-READY SYS1 CHECKPOINT POLICY")
print("=" * 80)
print("Event contract:", EVENT_CONTRACT_VERSION)
print("Legacy Sys1 reuse:", False)
print("Active Sys1 directory:", SYS1_MODEL_DIR)
print("Resume same-run checkpoint:", RESUME_SYS1_TRAIN)
print("Force retrain:", FORCE_RETRAIN_SYS1)
print(
    "Condition-independent ASR/pyannote/ECAPA/synthetic/diffusion assets "
    "remain reusable; the learned Sys1 head is contract-dependent."
)

PAPER-READY SYS1 CHECKPOINT POLICY
Event contract: voxconverse_action_aware_immediate_v1
Legacy Sys1 reuse: False
Active Sys1 directory: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/wavlm_sys1_dense_multiscale
Resume same-run checkpoint: True
Force retrain: False
Condition-independent ASR/pyannote/ECAPA/synthetic/diffusion assets remain reusable; the learned Sys1 head is contract-dependent.


In [16]:
# WavLM Sys1 dense multi-scale TCN model, dataset, training, scoring
SYS1_CKPT_PATH = SYS1_MODEL_DIR / "wavlm_dense_multiscale_sys1_best.pt"
SYS1_LAST_CKPT_PATH = SYS1_MODEL_DIR / "wavlm_dense_multiscale_sys1_last.pt"
SYS1_HISTORY_PATH = SYS1_MODEL_DIR / "sys1_training_history.csv"
SYS1_TRAIN_CACHE = CACHE_DIR / "sys1_train_examples.pkl"
WAVLM_CANDIDATES_PATH = CACHE_DIR / "wavlm_sys1_candidates_all.pkl"

RESUME_SYS1_TRAIN = bool(globals().get("RESUME_SYS1_TRAIN", True))
SYS1_EARLY_STOP_PATIENCE = int(globals().get("SYS1_EARLY_STOP_PATIENCE", 6))
SYS1_CE_WEIGHT = float(globals().get("SYS1_CE_WEIGHT", 0.35))
SYS1_GRAD_CLIP = float(globals().get("SYS1_GRAD_CLIP", 5.0))
SYS1_WEIGHT_DECAY = float(globals().get("SYS1_WEIGHT_DECAY", 1e-3))
SYS1_USE_WEIGHTED_SAMPLER = bool(globals().get("SYS1_USE_WEIGHTED_SAMPLER", False))
SYS1_CLASS_BALANCE_MODE = str(globals().get("SYS1_CLASS_BALANCE_MODE", "loss_only"))
SYS1_USE_ACTION_CLASS_WEIGHTS = bool(globals().get("SYS1_USE_ACTION_CLASS_WEIGHTS", True))

def require_module(module_name: str, install_hint: str = ""):
    import importlib.util

    if importlib.util.find_spec(module_name) is None:
        msg = f"Missing required module: {module_name}"

        if install_hint:
            msg += f"\nInstall with: {install_hint}"

        raise ImportError(msg)

def read_centered_audio_window(
    audio_path: Path,
    center_s: float,
    total_s: float,
    sr: int = TARGET_SR,
) -> np.ndarray:
    audio_path = Path(audio_path)
    total_s = float(total_s)
    center_s = float(center_s)

    n = max(1, int(round(total_s * sr)))
    start_s = center_s - total_s / 2.0
    end_s = start_s + total_s

    read_start_s = max(0.0, start_s)
    read_duration_s = max(0.01, end_s - read_start_s)

    try:
        x_part, _ = read_audio_mono(
            audio_path,
            start_s=read_start_s,
            duration_s=read_duration_s,
            sr=sr,
        )
    except Exception:
        x_part = np.zeros(0, dtype=np.float32)

    out = np.zeros(n, dtype=np.float32)

    dst_start = max(0, int(round((read_start_s - start_s) * sr)))
    usable = min(len(x_part), n - dst_start)

    if usable > 0:
        out[dst_start:dst_start + usable] = x_part[:usable]

    return out.astype(np.float32)

def _ensure_sys1_training_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=[
            "session", "time", "audio_path", "label", "action", "action_id",
            "sample_weight", "is_synthetic", "is_diffusion_augmented", "split",
        ])

    df = df.copy()

    required_defaults = {
        "session": "",
        "time": np.nan,
        "audio_path": "",
        "label": 0,
        "action": "NA",
        "action_id": ACTION_TO_ID["NA"],
        "sample_weight": 1.0,
        "is_synthetic": False,
        "is_diffusion_augmented": False,
        "split": "unknown",
    }

    for col, default in required_defaults.items():
        if col not in df.columns:
            df[col] = default

    df["session"] = df["session"].astype(str)
    df["audio_path"] = df["audio_path"].astype(str)
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

    df["action"] = df["action"].fillna("NA").astype(str)
    df.loc[~df["action"].isin(ACTION_NAMES), "action"] = "NA"
    df["action_id"] = df["action"].map(ACTION_TO_ID).fillna(ACTION_TO_ID["NA"]).astype(int)

    df["sample_weight"] = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(1.0).astype(float)
    df["sample_weight"] = df["sample_weight"].clip(lower=1e-6)

    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["is_diffusion_augmented"] = df["is_diffusion_augmented"].fillna(False).astype(bool)

    df = df.dropna(subset=["time"])
    df = df[df["audio_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

    return df

class Sys1WindowDataset(Dataset):
    def __init__(
        self,
        events_df: pd.DataFrame,
        context_s: float = SYS1_CONTEXT_S,
        augment: bool = False,
        max_rows: Optional[int] = None,
    ):
        df = _ensure_sys1_training_schema(events_df)

        if max_rows is not None and len(df) > max_rows:
            df = df.sample(int(max_rows), random_state=SEED).reset_index(drop=True)

        self.df = df.reset_index(drop=True)
        self.context_s = float(context_s)
        self.augment = bool(augment)
        self.rng = np.random.default_rng(SEED + (123 if augment else 0))
        self.n_samples = int(round(context_s * TARGET_SR))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[int(idx)]

        x = read_centered_audio_window(
            Path(r.audio_path),
            center_s=float(r.time),
            total_s=self.context_s,
            sr=TARGET_SR,
        )

        if self.augment:
            x = safe_audio_augment(
                x,
                self.rng,
                DIFF_BED_PATHS,
                p_bedmix=0.30 if len(DIFF_BED_PATHS) else 0.0,
            )

        y = int(r.label)
        action_id = int(r.action_id)
        sw = float(r.get("sample_weight", 1.0))

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(action_id, dtype=torch.long),
            torch.tensor(sw, dtype=torch.float32),
        )

class DenseMultiScaleTCNHead(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden: int = 256,
        n_actions: int = 5,
        dropout: float = 0.15,
    ):
        super().__init__()

        self.proj = nn.Conv1d(in_dim, hidden, kernel_size=1)

        self.blocks = nn.ModuleList()

        for dilation in [1, 2, 4, 8, 16]:
            self.blocks.append(
                nn.Sequential(
                    nn.Conv1d(
                        hidden,
                        hidden,
                        kernel_size=3,
                        padding=dilation,
                        dilation=dilation,
                    ),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Conv1d(hidden, hidden, kernel_size=1),
                    nn.GELU(),
                    nn.Dropout(dropout),
                )
            )

        self.boundary = nn.Conv1d(hidden, 1, kernel_size=1)
        self.action = nn.Conv1d(hidden, n_actions, kernel_size=1)

    def forward(self, feats):
        x = feats.transpose(1, 2)
        h = self.proj(x)

        for block in self.blocks:
            h = h + block(h)

        boundary_logits = self.boundary(h).squeeze(1)
        action_logits = self.action(h).transpose(1, 2)

        return boundary_logits, action_logits

class WavLMSys1Dense(nn.Module):
    def __init__(self, model_name: str = WAVLM_MODEL_NAME, n_actions: int = 5):
        super().__init__()

        require_module("transformers", "pip install -U transformers accelerate")

        from transformers import WavLMModel, AutoFeatureExtractor

        self.model_name = str(model_name)
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.wavlm = WavLMModel.from_pretrained(model_name)

        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.wavlm.eval()

        self.head = DenseMultiScaleTCNHead(
            self.wavlm.config.hidden_size,
            hidden=256,
            n_actions=n_actions,
        )

    def train(self, mode: bool = True):
        super().train(mode)

        self.wavlm.eval()

        return self

    def forward(self, wav):
        with torch.no_grad():
            out = self.wavlm(wav, output_hidden_states=False)
            feats = out.last_hidden_state

        return self.head(feats)

def center_targets_for_wavlm(
    n_frames: int,
    labels: torch.Tensor,
    action_ids: torch.Tensor,
    device,
) -> Tuple[torch.Tensor, torch.Tensor]:
    B = int(labels.shape[0])

    y = torch.zeros((B, n_frames), dtype=torch.float32, device=device)
    a = torch.zeros((B, n_frames), dtype=torch.long, device=device)

    c = n_frames // 2
    radius = max(1, int(round(0.20 * n_frames / max(1e-6, SYS1_CONTEXT_S))))

    for i in range(B):
        if labels[i] > 0.5:
            lo = max(0, c - radius)
            hi = min(n_frames, c + radius + 1)
            y[i, lo:hi] = 1.0

        a[i, :] = action_ids[i]

    return y, a

def build_sys1_sampler(
    train_df: pd.DataFrame,
) -> Optional[WeightedRandomSampler]:
    if (
        not SYS1_USE_WEIGHTED_SAMPLER
        or SYS1_CLASS_BALANCE_MODE != "sampler_only"
    ):
        return None

    frame = _ensure_sys1_training_schema(train_df)
    if len(frame) == 0:
        return None

    labels = frame["label"].astype(int).to_numpy()
    source_weight = frame["sample_weight"].astype(float).to_numpy()

    weighted_count = {
        label: max(
            1e-6,
            float(source_weight[labels == label].sum()),
        )
        for label in [0, 1]
    }
    total = sum(weighted_count.values())
    class_weight = {
        label: total / (2.0 * weighted_count[label])
        for label in [0, 1]
    }

    weights = np.asarray(
        [
            source_weight[index] * class_weight[int(labels[index])]
            for index in range(len(labels))
        ],
        dtype=np.float64,
    )
    weights = np.nan_to_num(
        weights,
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    )
    weights = np.maximum(weights, 1e-6)

    return WeightedRandomSampler(
        weights=torch.as_tensor(
            weights,
            dtype=torch.double,
        ),
        num_samples=len(weights),
        replacement=True,
    )

def compute_sys1_losses(
    boundary_logits: torch.Tensor,
    action_logits: torch.Tensor,
    y_scalar: torch.Tensor,
    action_id: torch.Tensor,
    sw: torch.Tensor,
    pos_weight: torch.Tensor,
    action_class_weight: Optional[torch.Tensor],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    y_dense, a_dense = center_targets_for_wavlm(
        boundary_logits.shape[1],
        y_scalar,
        action_id,
        boundary_logits.device,
    )

    bce = F.binary_cross_entropy_with_logits(
        boundary_logits,
        y_dense,
        pos_weight=pos_weight,
        reduction="none",
    ).mean(dim=1)

    ce = F.cross_entropy(
        action_logits.reshape(-1, len(ACTION_NAMES)),
        a_dense.reshape(-1),
        weight=action_class_weight,
        reduction="none",
    ).reshape(action_logits.shape[:2]).mean(dim=1)

    clean_sw = torch.nan_to_num(
        sw.float(),
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    ).clamp_min(1e-6)
    clean_sw = clean_sw / clean_sw.mean().clamp_min(1e-6)

    combined = bce + SYS1_CE_WEIGHT * ce
    loss = (combined * clean_sw).mean()

    return (
        loss,
        (bce * clean_sw).mean(),
        (ce * clean_sw).mean(),
    )

def _make_sys1_checkpoint(
    model: WavLMSys1Dense,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val: float,
    bad_epochs: int,
    train_loss: float,
    val_loss: float,
    is_best: bool,
) -> Dict[str, Any]:
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "model_name": WAVLM_MODEL_NAME,
        "epoch": int(epoch),
        "best_val": float(best_val),
        "bad_epochs": int(bad_epochs),
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "action_names": list(ACTION_NAMES),
        "sys1_context_s": float(SYS1_CONTEXT_S),
        "sys1_hop_s": float(SYS1_HOP_S),
        "sys1_ce_weight": float(SYS1_CE_WEIGHT),
        "sys1_lr": float(SYS1_LR),
        "sys1_pos_weight_cap": float(SYS1_POS_WEIGHT_CAP),
        "sys1_class_balance_mode": str(SYS1_CLASS_BALANCE_MODE),
        "sys1_use_weighted_sampler": bool(SYS1_USE_WEIGHTED_SAMPLER),
        "sys1_use_action_class_weights": bool(SYS1_USE_ACTION_CLASS_WEIGHTS),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "midturn_negative_ratio": float(MIDTURN_HOLD_NEGATIVE_RATIO),
        "midturn_negative_max_per_session": int(MIDTURN_HOLD_MAX_PER_SESSION),
        "midturn_negative_weight": float(MIDTURN_HOLD_SAMPLE_WEIGHT),
        "is_best": bool(is_best),
        "run_version": RUN_VERSION,
        "saved_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

def train_sys1_wavlm(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    resume: bool = RESUME_SYS1_TRAIN,
) -> WavLMSys1Dense:
    train_df = _ensure_sys1_training_schema(train_df)
    val_df = _ensure_sys1_training_schema(val_df)

    assert len(train_df) > 0, "Sys1 train_df is empty."
    assert len(val_df) > 0, "Sys1 val_df is empty."

    max_train = 2000 if DEBUG_SMALL_RUN else None
    max_val = 600 if DEBUG_SMALL_RUN else None

    train_ds = Sys1WindowDataset(train_df, augment=False, max_rows=max_train)
    val_ds = Sys1WindowDataset(val_df, augment=False, max_rows=max_val)

    sampler = build_sys1_sampler(train_ds.df)

    train_loader = DataLoader(
        train_ds,
        batch_size=SYS1_BATCH_SIZE,
        shuffle=(sampler is None),
        sampler=sampler,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=SYS1_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

    model = WavLMSys1Dense(WAVLM_MODEL_NAME, n_actions=len(ACTION_NAMES)).to(DEVICE)

    train_labels_np = train_ds.df["label"].astype(int).to_numpy()
    train_source_weight_np = (
        pd.to_numeric(
            train_ds.df["sample_weight"],
            errors="coerce",
        )
        .fillna(1.0)
        .clip(lower=1e-6)
        .to_numpy(dtype=np.float64)
    )

    weighted_pos = max(
        1e-6,
        float(
            train_source_weight_np[
                train_labels_np == 1
            ].sum()
        ),
    )
    weighted_neg = max(
        1e-6,
        float(
            train_source_weight_np[
                train_labels_np == 0
            ].sum()
        ),
    )

    if SYS1_CLASS_BALANCE_MODE == "loss_only":
        positive_ratio = min(
            float(SYS1_POS_WEIGHT_CAP),
            weighted_neg / weighted_pos,
        )
    else:
        positive_ratio = 1.0

    pos_weight = torch.tensor(
        [positive_ratio],
        dtype=torch.float32,
        device=DEVICE,
    )

    action_class_weight = None
    if SYS1_USE_ACTION_CLASS_WEIGHTS:
        action_ids_np = (
            train_ds.df["action_id"]
            .astype(int)
            .to_numpy()
        )
        effective_action_counts = np.asarray(
            [
                max(
                    1e-6,
                    float(
                        train_source_weight_np[
                            action_ids_np == action_index
                        ].sum()
                    ),
                )
                for action_index in range(
                    len(ACTION_NAMES)
                )
            ],
            dtype=np.float64,
        )
        action_weights_np = (
            effective_action_counts.sum()
            / (
                len(ACTION_NAMES)
                * effective_action_counts
            )
        )
        action_weights_np = np.clip(
            action_weights_np,
            0.25,
            4.0,
        )
        action_class_weight = torch.tensor(
            action_weights_np,
            dtype=torch.float32,
            device=DEVICE,
        )

    print(
        "Sys1 balance policy:",
        SYS1_CLASS_BALANCE_MODE,
        "| weighted SHIFT/HOLD:",
        round(weighted_pos, 2),
        "/",
        round(weighted_neg, 2),
        "| pos_weight:",
        round(float(pos_weight.item()), 4),
    )
    if action_class_weight is not None:
        print(
            "Sys1 action class weights:",
            {
                action: round(
                    float(action_class_weight[index].item()),
                    4,
                )
                for index, action in enumerate(ACTION_NAMES)
            },
        )

    optimizer = torch.optim.AdamW(
        model.head.parameters(),
        lr=SYS1_LR,
        weight_decay=SYS1_WEIGHT_DECAY,
    )

    start_epoch = 1
    best_val = float("inf")
    bad_epochs = 0

    if resume and SYS1_LAST_CKPT_PATH.exists():
        print("Resuming Sys1 training from last checkpoint:", SYS1_LAST_CKPT_PATH)
        ckpt = torch.load(SYS1_LAST_CKPT_PATH, map_location=DEVICE)

        checkpoint_contract = str(
            ckpt.get("event_contract_version", "")
        )
        if checkpoint_contract != str(EVENT_CONTRACT_VERSION):
            raise RuntimeError(
                "Refusing to resume an incompatible Sys1 checkpoint. "
                f"checkpoint_contract={checkpoint_contract!r}, "
                f"current_contract={EVENT_CONTRACT_VERSION!r}. "
                "Delete only this run's Sys1 checkpoint or set "
                "FORCE_RETRAIN_SYS1=True."
            )

        ckpt_model_name = ckpt.get("model_name", WAVLM_MODEL_NAME)
        if ckpt_model_name != WAVLM_MODEL_NAME:
            print(f"⚠️ Checkpoint model_name={ckpt_model_name}, current WAVLM_MODEL_NAME={WAVLM_MODEL_NAME}")

        model.load_state_dict(ckpt["model"])

        if "optimizer" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer"])

            for state in optimizer.state.values():
                for k, v in state.items():
                    if torch.is_tensor(v):
                        state[k] = v.to(DEVICE)

        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_val = float(ckpt.get("best_val", ckpt.get("val_loss", float("inf"))))
        bad_epochs = int(ckpt.get("bad_epochs", 0))

        print(f"Resume starts at epoch {start_epoch}; best_val={best_val:.4f}; bad_epochs={bad_epochs}")

    losses_history = []

    if SYS1_HISTORY_PATH.exists() and resume and start_epoch > 1:
        try:
            old_hist = pd.read_csv(SYS1_HISTORY_PATH)
            losses_history = old_hist.to_dict("records")
        except Exception:
            losses_history = []

    for epoch in range(start_epoch, SYS1_EPOCHS + 1):
        model.train()
        model.wavlm.eval()
        model.head.train()

        train_losses = []

        for wav, y_scalar, action_id, sw in tqdm(train_loader, desc=f"Sys1 epoch {epoch}", leave=False):
            wav = wav.to(DEVICE, non_blocking=True)
            y_scalar = y_scalar.to(DEVICE, non_blocking=True)
            action_id = action_id.to(DEVICE, non_blocking=True)
            sw = sw.to(DEVICE, non_blocking=True)

            boundary_logits, action_logits = model(wav)

            loss, bce_loss, ce_loss = compute_sys1_losses(
                boundary_logits,
                action_logits,
                y_scalar,
                action_id,
                sw,
                pos_weight,
                action_class_weight,
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.head.parameters(), SYS1_GRAD_CLIP)
            optimizer.step()

            train_losses.append(float(loss.detach().cpu()))

        model.eval()

        val_losses = []

        with torch.no_grad():
            for wav, y_scalar, action_id, sw in val_loader:
                wav = wav.to(DEVICE, non_blocking=True)
                y_scalar = y_scalar.to(DEVICE, non_blocking=True)
                action_id = action_id.to(DEVICE, non_blocking=True)
                sw = sw.to(DEVICE, non_blocking=True)

                boundary_logits, action_logits = model(wav)

                loss, bce_loss, ce_loss = compute_sys1_losses(
                    boundary_logits,
                    action_logits,
                    y_scalar,
                    action_id,
                    sw,
                    pos_weight,
                    action_class_weight,
                )

                val_losses.append(float(loss.detach().cpu()))

        train_loss = float(np.mean(train_losses)) if train_losses else 999.0
        val_loss = float(np.mean(val_losses)) if val_losses else 999.0

        improved = val_loss < best_val

        if improved:
            best_val = val_loss
            bad_epochs = 0
        else:
            bad_epochs += 1

        SYS1_MODEL_DIR.mkdir(parents=True, exist_ok=True)

        last_state = _make_sys1_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_val=best_val,
            bad_epochs=bad_epochs,
            train_loss=train_loss,
            val_loss=val_loss,
            is_best=improved,
        )

        torch.save(last_state, SYS1_LAST_CKPT_PATH)

        if improved:
            torch.save(last_state, SYS1_CKPT_PATH)

        losses_history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val": best_val,
            "bad_epochs": bad_epochs,
            "improved": improved,
        })

        pd.DataFrame(losses_history).to_csv(SYS1_HISTORY_PATH, index=False)

        print(
            f"Sys1 epoch {epoch:03d}: "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f} "
            f"best_val={best_val:.4f} "
            f"bad={bad_epochs}/{SYS1_EARLY_STOP_PATIENCE}"
        )

        if bad_epochs >= SYS1_EARLY_STOP_PATIENCE:
            print("Sys1 early stop.")
            break

    if SYS1_CKPT_PATH.exists():
        best_ckpt = torch.load(SYS1_CKPT_PATH, map_location=DEVICE)
        model.load_state_dict(best_ckpt["model"])
        print("Loaded best Sys1 state from:", SYS1_CKPT_PATH)

    model.eval()
    return model

def load_sys1_model(load_last: bool = False) -> Optional[WavLMSys1Dense]:
    path = SYS1_LAST_CKPT_PATH if load_last else SYS1_CKPT_PATH

    if not path.exists():
        return None

    ckpt = torch.load(path, map_location=DEVICE)

    checkpoint_contract = str(
        ckpt.get("event_contract_version", "")
    )
    if checkpoint_contract != str(EVENT_CONTRACT_VERSION):
        raise RuntimeError(
            "Refusing to load an incompatible Sys1 checkpoint: "
            f"{path}. checkpoint_contract={checkpoint_contract!r}, "
            f"current_contract={EVENT_CONTRACT_VERSION!r}."
        )

    model = WavLMSys1Dense(
        ckpt.get("model_name", WAVLM_MODEL_NAME),
        n_actions=len(ACTION_NAMES),
    ).to(DEVICE)

    model.load_state_dict(ckpt["model"])
    model.eval()

    print("Loaded Sys1 checkpoint:", path)
    print("  epoch:", ckpt.get("epoch", None))
    print("  val_loss:", ckpt.get("val_loss", None))
    print("  best_val:", ckpt.get("best_val", None))

    return model

@torch.no_grad()
def score_sys1_candidates_for_events(
    model: WavLMSys1Dense,
    events_df: pd.DataFrame,
    desc: str,
) -> pd.DataFrame:
    events_df = _ensure_sys1_training_schema(events_df)

    if model is None or len(events_df) == 0:
        return pd.DataFrame()

    ds = Sys1WindowDataset(events_df, augment=False)

    loader = DataLoader(
        ds,
        batch_size=SYS1_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
    )

    rows = []
    model.eval()
    offset = 0

    for wav, y_scalar, action_id, sw in tqdm(loader, desc=desc):
        wav = wav.to(DEVICE, non_blocking=True)

        boundary_logits, action_logits = model(wav)

        probs = torch.sigmoid(boundary_logits).detach().cpu().numpy()
        actp = torch.softmax(
            action_logits[:, action_logits.shape[1] // 2, :],
            dim=-1,
        ).detach().cpu().numpy()

        center = probs.shape[1] // 2
        scores = probs.max(axis=1)
        center_scores = probs[:, center]

        for j in range(len(scores)):
            r = events_df.iloc[offset + j].to_dict()

            row = {
                "session": str(r["session"]),
                "time": float(r["time"]),
                "source_wavlm": 1,
                "wavlm_score": float(scores[j]),
                "wavlm_center_score": float(center_scores[j]),
                "label": int(r.get("label", -1)),
                "action": r.get("action", "NA"),
                "action_id": int(r.get("action_id", 0)),
                "split": r.get("split", desc),
                "audio_path": r.get("audio_path", None),
                "is_synthetic": bool(r.get("is_synthetic", False)),
                "is_diffusion_augmented": bool(r.get("is_diffusion_augmented", False)),
                "sample_weight": float(r.get("sample_weight", 1.0)),
            }

            for k, a in enumerate(ACTION_NAMES):
                row[f"wavlm_p_{a}"] = float(actp[j, k])

            rows.append(row)

        offset += len(scores)

    return pd.DataFrame(rows)

if USE_WAVLM_SYS1:
    if SYS1_CKPT_PATH.exists() and not FORCE_RETRAIN_SYS1:
        SYS1_MODEL = load_sys1_model(load_last=False)

    else:
        train_sys1_df = TRAIN_EVENTS_MIXED.copy()
        val_sys1_df = REAL_VAL_EVENTS.copy()

        SYS1_MODEL = train_sys1_wavlm(
            train_sys1_df,
            val_sys1_df,
            resume=RESUME_SYS1_TRAIN,
        )

else:
    SYS1_MODEL = None

print("Sys1 model:", type(SYS1_MODEL).__name__ if SYS1_MODEL is not None else None)
print("Sys1 best checkpoint:", SYS1_CKPT_PATH, "exists=", SYS1_CKPT_PATH.exists())
print("Sys1 last checkpoint:", SYS1_LAST_CKPT_PATH, "exists=", SYS1_LAST_CKPT_PATH.exists())

Loaded Sys1 checkpoint: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/wavlm_sys1_dense_multiscale/wavlm_dense_multiscale_sys1_best.pt
  epoch: 8
  val_loss: 0.29097983087647517
  best_val: 0.29097983087647517
Sys1 model: WavLMSys1Dense
Sys1 best checkpoint: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/wavlm_sys1_dense_multiscale/wavlm_dense_multiscale_sys1_best.pt exists= True
Sys1 last checkpoint: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/wavlm_sys1_dense_multiscale/wavlm_dense_multiscale_sys1_last.pt exists= True


In [17]:
# Score/load WavLM Sys1 full-timeline candidates for TRAIN/VAL/TEST
WAVLM_CANDIDATES_PATH = CACHE_DIR / "wavlm_sys1_full_timeline_candidates_all.pkl"
WAVLM_CANDIDATES_META_PATH = CACHE_DIR / "wavlm_sys1_full_timeline_candidates_all.meta.json"

WAVLM_CAND_COLUMNS = [
    "session", "time", "split", "audio_path",
    "source_wavlm",
    "wavlm_score", "wavlm_center_score",
    "label", "action", "action_id",
    "is_synthetic", "is_diffusion_augmented", "sample_weight",
] + [f"wavlm_p_{a}" for a in ACTION_NAMES]

def _file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _wavlm_candidate_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "seed": int(SEED),
        "wavlm_model_name": str(WAVLM_MODEL_NAME),
        "sys1_best_checkpoint": _file_fingerprint(SYS1_CKPT_PATH),
        "sys1_context_s": float(SYS1_CONTEXT_S),
        "sys1_hop_s": float(SYS1_HOP_S),
        "wavlm_timeline_keep_thr": float(WAVLM_TIMELINE_KEEP_THR),
        "sys1_timeline_batch_size": int(SYS1_TIMELINE_BATCH_SIZE),
        "sys1_timeline_local_max_radius": int(SYS1_TIMELINE_LOCAL_MAX_RADIUS),
        "debug_small_run": bool(DEBUG_SMALL_RUN),
        "debug_max_train_sessions": int(DEBUG_MAX_TRAIN_SESSIONS),
        "include_train": True,
        "include_val": True,
        "include_test": True,
        "include_synthetic": bool(USE_SYNTH_MULTIPARTY),
        "include_diffusion_augmented": bool(USE_DIFFUSION_AUG),
        "cache_format": "pickle",
    }

def _wavlm_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        old = json.loads(Path(path).read_text())
        return old == expected

    except Exception:
        return False

def _clean_legacy_wavlm_parquet():
    for p in [
        CACHE_DIR / "wavlm_sys1_candidates_all.parquet",
        CACHE_DIR / "wavlm_sys1_full_timeline_candidates_all.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet WavLM candidate cache:", p)

        except Exception as e:
            print("Could not remove legacy parquet WavLM candidate cache:", p, e)

def _empty_wavlm_candidates() -> pd.DataFrame:
    return pd.DataFrame(columns=WAVLM_CAND_COLUMNS)

def _ensure_wavlm_candidate_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return _empty_wavlm_candidates()

    df = df.copy()

    for col in WAVLM_CAND_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df["session"] = df["session"].astype(str)
    df["split"] = df["split"].fillna("unknown").astype(str)
    df["audio_path"] = df["audio_path"].fillna("").astype(str)

    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df = df.dropna(subset=["session", "time"])
    df = df[df["time"] >= 0].reset_index(drop=True)

    df["source_wavlm"] = pd.to_numeric(df["source_wavlm"], errors="coerce").fillna(1).astype(int)
    df["wavlm_score"] = pd.to_numeric(df["wavlm_score"], errors="coerce").fillna(0.0)
    df["wavlm_center_score"] = pd.to_numeric(df["wavlm_center_score"], errors="coerce").fillna(0.0)

    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(-1).astype(int)

    df["action"] = df["action"].fillna("NA").astype(str)
    df.loc[~df["action"].isin(ACTION_NAMES), "action"] = "NA"
    df["action_id"] = df["action"].map(ACTION_TO_ID).fillna(ACTION_TO_ID["NA"]).astype(int)

    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["is_diffusion_augmented"] = df["is_diffusion_augmented"].fillna(False).astype(bool)

    df["sample_weight"] = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(1.0).astype(float)
    df["sample_weight"] = df["sample_weight"].clip(lower=1e-6)

    for a in ACTION_NAMES:
        col = f"wavlm_p_{a}"
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return (
        df[WAVLM_CAND_COLUMNS]
        .drop_duplicates(subset=["session", "time", "split"])
        .sort_values(["split", "session", "time"])
        .reset_index(drop=True)
    )

def _infer_train_weight_from_session(session: str) -> float:
    session = str(session)

    is_synth = session.startswith("synth_")
    is_diff = "__diff" in session

    if is_synth and is_diff:
        return float(WEIGHT_SYNTH_DIFFUSION)

    if is_synth:
        return float(WEIGHT_SYNTH_CLEAN)

    if is_diff:
        return float(WEIGHT_REAL_DIFFUSION)

    return float(WEIGHT_REAL_CLEAN)

def collect_wavlm_timeline_manifest() -> pd.DataFrame:
    rows = []

    for sid in TRAIN_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "train",
            "orig_session": np.nan,
            "is_synthetic": False,
            "is_diffusion_augmented": False,
        })

    for sid in VAL_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "dev")),
            "split": "val",
            "orig_session": np.nan,
            "is_synthetic": False,
            "is_diffusion_augmented": False,
        })

    for sid in TEST_SESSIONS:
        rows.append({
            "session": str(sid),
            "audio_path": str(get_session_audio_path(sid, "test")),
            "split": "test",
            "orig_session": np.nan,
            "is_synthetic": False,
            "is_diffusion_augmented": False,
        })

    if USE_SYNTH_MULTIPARTY and SYNTH_MANIFEST is not None and len(SYNTH_MANIFEST):
        synth_rows = (
            SYNTH_MANIFEST[["session", "audio_path"]]
            .copy()
            .assign(
                split="synth_train",
                orig_session=np.nan,
                is_synthetic=True,
                is_diffusion_augmented=False,
            )
            .to_dict("records")
        )
        rows.extend(synth_rows)

    if USE_DIFFUSION_AUG and AUG_MANIFEST is not None and len(AUG_MANIFEST):
        for _, r in AUG_MANIFEST.iterrows():
            kind = str(r.get("kind", ""))
            sid = str(r.get("session", ""))
            audio_path = str(r.get("audio_path", ""))
            orig_session = str(r.get("orig_session", ""))

            if not sid or not audio_path:
                continue

            split = "synth_diffusion" if "synth" in kind else "real_diffusion"

            rows.append({
                "session": sid,
                "audio_path": audio_path,
                "split": split,
                "orig_session": orig_session,
                "is_synthetic": bool("synth" in kind or sid.startswith("synth_")),
                "is_diffusion_augmented": True,
            })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return pd.DataFrame(columns=[
            "session", "audio_path", "split", "orig_session",
            "is_synthetic", "is_diffusion_augmented",
        ])

    out["session"] = out["session"].astype(str)
    out["audio_path"] = out["audio_path"].astype(str)
    out["split"] = out["split"].astype(str)
    out = out.drop_duplicates(subset=["session"]).reset_index(drop=True)

    exists_mask = out["audio_path"].map(lambda p: Path(p).exists())

    if not bool(exists_mask.all()):
        missing = out.loc[~exists_mask, ["session", "audio_path"]].head(10)
        print("⚠️ Some WavLM timeline audio files do not exist. First examples:")
        display(missing)

    out = out.loc[exists_mask].reset_index(drop=True)

    if DEBUG_SMALL_RUN and len(out):
        out = (
            out.groupby("split", group_keys=False)
            .head(max(2, DEBUG_MAX_TRAIN_SESSIONS))
            .reset_index(drop=True)
        )

    return out

def find_local_peak_indices(
    scores: np.ndarray,
    keep_thr: float,
    radius: int = 1,
) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)

    if len(scores) == 0:
        return np.array([], dtype=int)

    keep = scores >= float(keep_thr)

    if radius <= 0:
        return np.where(keep)[0]

    peaks = []

    for i, s in enumerate(scores):
        if not keep[i]:
            continue

        lo = max(0, i - int(radius))
        hi = min(len(scores), i + int(radius) + 1)

        local = scores[lo:hi]

        if s >= np.max(local):
            left_ok = (i == 0) or (s > scores[i - 1])
            right_ok = (i == len(scores) - 1) or (s >= scores[i + 1])

            if left_ok and right_ok:
                peaks.append(i)

    return np.array(peaks, dtype=int)

@torch.no_grad()
def score_sys1_full_timeline_session(
    model: WavLMSys1Dense,
    session: str,
    audio_path: Path,
    split_name: str,
    hop_s: float = SYS1_HOP_S,
    context_s: float = SYS1_CONTEXT_S,
    keep_thr: float = WAVLM_TIMELINE_KEEP_THR,
) -> pd.DataFrame:
    if model is None:
        return _empty_wavlm_candidates()

    session = str(session)
    split_name = str(split_name)
    audio_path = Path(audio_path)

    if not audio_path.exists():
        return _empty_wavlm_candidates()

    try:
        x, sr = read_audio_mono(audio_path, sr=TARGET_SR)

    except Exception as e:
        print(f"⚠️ Could not read audio for WavLM timeline {session}: {e}")
        return _empty_wavlm_candidates()

    dur = len(x) / float(sr) if sr > 0 else 0.0

    if dur <= 0:
        return _empty_wavlm_candidates()

    times = np.arange(0.0, dur + 1e-6, float(hop_s), dtype=np.float32)

    if DEBUG_SMALL_RUN and len(times) > 1500:
        times = times[:1500]

    all_scores = []
    all_center_scores = []
    all_action_probs = []

    model.eval()

    batch_size = max(1, int(SYS1_TIMELINE_BATCH_SIZE))
    i = 0

    while i < len(times):
        bt = times[i:i + batch_size]

        try:
            batch = np.stack([
                slice_or_pad(x, float(t), float(context_s), sr)
                for t in bt
            ]).astype(np.float32)

            wav = torch.tensor(batch, dtype=torch.float32, device=DEVICE)

            boundary_logits, action_logits = model(wav)

            probs = torch.sigmoid(boundary_logits).detach().cpu().numpy()
            center = probs.shape[1] // 2

            action_probs = torch.softmax(
                action_logits[:, center, :],
                dim=-1,
            ).detach().cpu().numpy()

            all_scores.append(probs.max(axis=1))
            all_center_scores.append(probs[:, center])
            all_action_probs.append(action_probs)

            i += batch_size

        except RuntimeError as e:
            if "out of memory" in str(e).lower() and batch_size > 1 and DEVICE == "cuda":
                torch.cuda.empty_cache()
                batch_size = max(1, batch_size // 2)
                print(f"⚠️ CUDA OOM in WavLM timeline {session}; reducing batch_size to {batch_size}")
                continue

            raise e

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    if not all_scores:
        return _empty_wavlm_candidates()

    scores = np.concatenate(all_scores)
    center_scores = np.concatenate(all_center_scores)
    action_probs = np.vstack(all_action_probs)

    peak_idx = find_local_peak_indices(
        scores,
        keep_thr=float(keep_thr),
        radius=int(SYS1_TIMELINE_LOCAL_MAX_RADIUS),
    )

    rows = []

    for j in peak_idx:
        sess_str = str(session)

        row = {
            "session": sess_str,
            "time": float(times[j]),
            "split": split_name,
            "audio_path": str(audio_path),
            "source_wavlm": 1,
            "wavlm_score": float(scores[j]),
            "wavlm_center_score": float(center_scores[j]),
            "label": -1,
            "action": "NA",
            "action_id": int(ACTION_TO_ID["NA"]),
            "is_synthetic": bool(sess_str.startswith("synth_")),
            "is_diffusion_augmented": bool("__diff" in sess_str),
            "sample_weight": _infer_train_weight_from_session(sess_str),
        }

        for k, a in enumerate(ACTION_NAMES):
            row[f"wavlm_p_{a}"] = float(action_probs[j, k])

        rows.append(row)

    return _ensure_wavlm_candidate_schema(pd.DataFrame(rows))

def score_sys1_full_timeline_manifest(
    model: WavLMSys1Dense,
    manifest: pd.DataFrame,
) -> pd.DataFrame:
    if model is None or manifest is None or len(manifest) == 0:
        return _empty_wavlm_candidates()

    frames = []

    for _, r in tqdm(manifest.iterrows(), total=len(manifest), desc="WavLM full-timeline sessions"):
        df = score_sys1_full_timeline_session(
            model=model,
            session=str(r.session),
            audio_path=Path(r.audio_path),
            split_name=str(r.split),
        )

        if len(df):
            frames.append(df)

    if not frames:
        return _empty_wavlm_candidates()

    return _ensure_wavlm_candidate_schema(pd.concat(frames, ignore_index=True))

_clean_legacy_wavlm_parquet()

if USE_WAVLM_SYS1 and SYS1_MODEL is not None:
    expected_wavlm_meta = _wavlm_candidate_config()

    if (
        WAVLM_CANDIDATES_PATH.exists()
        and _wavlm_meta_matches(WAVLM_CANDIDATES_META_PATH, expected_wavlm_meta)
        and not FORCE_REBUILD_SYS1_CANDIDATES
    ):
        WAVLM_CANDIDATES_ALL = pd.read_pickle(WAVLM_CANDIDATES_PATH)
        WAVLM_CANDIDATES_ALL = _ensure_wavlm_candidate_schema(WAVLM_CANDIDATES_ALL)

        print("Loaded WavLM full-timeline candidates:", WAVLM_CANDIDATES_ALL.shape)

    else:
        wavlm_manifest = collect_wavlm_timeline_manifest()

        print("WavLM timeline manifest:", wavlm_manifest.shape)

        if len(wavlm_manifest):
            display(wavlm_manifest.groupby("split").size().to_frame("n_audio_files"))

        WAVLM_CANDIDATES_ALL = score_sys1_full_timeline_manifest(
            SYS1_MODEL,
            wavlm_manifest,
        )

        WAVLM_CANDIDATES_ALL.to_pickle(WAVLM_CANDIDATES_PATH)
        WAVLM_CANDIDATES_META_PATH.write_text(json.dumps(expected_wavlm_meta, indent=2, sort_keys=True))

        print("Saved WavLM full-timeline candidates:", WAVLM_CANDIDATES_PATH)

else:
    WAVLM_CANDIDATES_ALL = _empty_wavlm_candidates()

print("WAVLM_CANDIDATES_ALL:", WAVLM_CANDIDATES_ALL.shape)

if len(WAVLM_CANDIDATES_ALL):
    display(WAVLM_CANDIDATES_ALL.head())
    display(WAVLM_CANDIDATES_ALL.groupby("split").size().to_frame("wavlm_candidate_count"))

    print("WavLM candidate score summary:")
    display(WAVLM_CANDIDATES_ALL["wavlm_score"].describe().to_frame("wavlm_score"))

Loaded WavLM full-timeline candidates: (574564, 18)
WAVLM_CANDIDATES_ALL: (574564, 18)


,session,time,split,audio_path,source_wavlm,wavlm_score,wavlm_center_score,label,action,action_id,is_synthetic,is_diffusion_augmented,sample_weight,wavlm_p_NA,wavlm_p_NTT,wavlm_p_ITT,wavlm_p_BC,wavlm_p_BI
0,synth_000000,0.0,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0.625229,2.389844e-01,-1,NA,0,True,False,0.5,0.175247,0.009619,0.686777,0.126965,0.001393
1,synth_000000,0.2,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0.368246,1.520933e-01,-1,NA,0,True,False,0.5,0.669130,0.195304,0.110006,0.024303,0.001257
2,synth_000000,0.4,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0.197140,4.885994e-02,-1,NA,0,True,False,0.5,0.291277,0.100523,0.570787,0.034066,0.003348
3,synth_000000,1.4,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0.925097,1.182156e-07,-1,NA,0,True,False,0.5,0.250314,0.706248,0.018161,0.017386,0.007891
4,synth_000000,1.8,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0.868545,3.914137e-03,-1,NA,0,True,False,0.5,0.014233,0.015893,0.476671,0.471850,0.021354


,wavlm_candidate_count
split,
synth_train,245457
test,238383
train,74186
val,16538


WavLM candidate score summary:


,wavlm_score
count,574564.000000
mean,0.483552
std,0.335890
min,0.030000
25%,0.139561
50%,0.469958
75%,0.816694
max,0.999519


In [18]:
import os
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from typing import Dict, Any, List, Optional
from IPython.display import display
from tqdm.auto import tqdm

QWEN3_TRAIN_DATA_PATH = QWEN3_MODEL_DIR / "qwen3_action_train_data.pkl"
QWEN3_TRAIN_META_PATH = QWEN3_MODEL_DIR / "qwen3_action_train_data.meta.json"

QWEN3_CANDIDATE_INPUT_PATH = CACHE_DIR / "qwen3_candidate_input_pre_qwen.pkl"
QWEN3_CANDIDATE_INPUT_META_PATH = CACHE_DIR / "qwen3_candidate_input_pre_qwen.meta.json"

QWEN3_SCORES_PATH = CACHE_DIR / "qwen3_action_scores_on_candidates_all.pkl"
QWEN3_SCORES_META_PATH = CACHE_DIR / "qwen3_action_scores_on_candidates_all.meta.json"

QWEN3_RESUME_FROM_CHECKPOINT = bool(globals().get("QWEN3_RESUME_FROM_CHECKPOINT", True))
QWEN3_GRADIENT_ACCUMULATION_STEPS = int(globals().get("QWEN3_GRADIENT_ACCUMULATION_STEPS", 4))
QWEN3_LEARNING_RATE = float(globals().get("QWEN3_LEARNING_RATE", 5e-5))
QWEN3_LEARNING_RATE = min(QWEN3_LEARNING_RATE, 5e-5)
QWEN3_WEIGHT_DECAY = float(globals().get("QWEN3_WEIGHT_DECAY", 0.01))
QWEN3_LORA_DROPOUT = float(globals().get("QWEN3_LORA_DROPOUT", 0.05))
QWEN3_GRADIENT_CHECKPOINTING = bool(globals().get("QWEN3_GRADIENT_CHECKPOINTING", True))
QWEN3_TRAIN_INTERNAL_VAL_FRAC = float(globals().get("QWEN3_TRAIN_INTERNAL_VAL_FRAC", 0.08))
QWEN3_SCORE_BATCH_SIZE = int(globals().get("QWEN3_SCORE_BATCH_SIZE", QWEN3_BATCH_SIZE))
QWEN3_TARGET_MODULES = list(globals().get(
    "QWEN3_TARGET_MODULES",
    ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
QWEN3_MODULES_TO_SAVE_CANDIDATES = list(globals().get(
    "QWEN3_MODULES_TO_SAVE_CANDIDATES",
    ["score", "classifier", "classification_head"],
))

QWEN3_TRAINING_LOGIC_VERSION = "qwen3_weighted_loss_labels_checked_bf16_fp32_stable_v3"

QWEN_SCORE_COLUMNS = [f"qwen_p_{a}" for a in ACTION_NAMES] + [
    "qwen_shift_score",
    "qwen_backchannel_score",
    "qwen_interruption_score",
]

def _is_cuda_device() -> bool:
    return str(DEVICE).lower().startswith("cuda")

def _qwen_use_bf16() -> bool:
    try:
        return bool(
            _is_cuda_device()
            and torch.cuda.is_available()
            and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        )
    except Exception:
        return False

def _qwen_model_dtype():
    if _qwen_use_bf16():
        return torch.bfloat16
    return torch.float32

def _hf_token_kwargs() -> Dict[str, Any]:
    tok = os.environ.get("HF_TOKEN", globals().get("HF_TOKEN", None))

    if tok is None or str(tok).strip() == "":
        return {}

    return {"token": str(tok).strip()}

def _qwen_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _qwen_model_fingerprint() -> Dict[str, Any]:
    files = [
        QWEN3_MODEL_DIR / "adapter_model.safetensors",
        QWEN3_MODEL_DIR / "adapter_model.bin",
        QWEN3_MODEL_DIR / "adapter_config.json",
        QWEN3_MODEL_DIR / "model.safetensors",
        QWEN3_MODEL_DIR / "pytorch_model.bin",
        QWEN3_MODEL_DIR / "config.json",
        QWEN3_MODEL_DIR / "qwen3_training_config.json",
    ]

    return {
        str(p.name): _qwen_file_fingerprint(p)
        for p in files
    }

def _qwen_train_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_min_s": float(PRIMARY_MIN_GAP_S),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "midturn_negative_ratio": float(MIDTURN_HOLD_NEGATIVE_RATIO),
        "midturn_negative_max_per_session": int(MIDTURN_HOLD_MAX_PER_SESSION),
        "midturn_negative_weight": float(MIDTURN_HOLD_SAMPLE_WEIGHT),
        "training_logic_version": str(QWEN3_TRAINING_LOGIC_VERSION),
        "seed": int(SEED),
        "qwen3_model_name": str(QWEN3_MODEL_NAME),
        "qwen3_max_length": int(QWEN3_MAX_LENGTH),
        "qwen3_lora_r": int(QWEN3_LORA_R),
        "qwen3_lora_alpha": int(QWEN3_LORA_ALPHA),
        "qwen3_lora_dropout": float(QWEN3_LORA_DROPOUT),
        "qwen3_epochs": int(QWEN3_EPOCHS),
        "qwen3_batch_size": int(QWEN3_BATCH_SIZE),
        "qwen3_grad_accum": int(QWEN3_GRADIENT_ACCUMULATION_STEPS),
        "qwen3_learning_rate": float(QWEN3_LEARNING_RATE),
        "qwen3_weight_decay": float(QWEN3_WEIGHT_DECAY),
        "qwen3_use_bf16": bool(_qwen_use_bf16()),
        "qwen3_force_no_fp16": True,
        "qwen3_internal_val_frac": float(QWEN3_TRAIN_INTERNAL_VAL_FRAC),
        "qwen3_target_modules": list(QWEN3_TARGET_MODULES),
        "qwen3_modules_to_save_candidates": list(QWEN3_MODULES_TO_SAVE_CANDIDATES),
        "action_names": list(ACTION_NAMES),
        "train_sources": {
            "real_train": True,
            "synthetic": bool(len(SYNTH_EVENTS)),
            "real_diffusion": bool(len(REAL_AUG_EVENTS)),
            "synthetic_diffusion": bool(len(SYNTH_AUG_EVENTS)),
        },
        "no_real_val_leakage": True,
        "weighted_loss": True,
        "labels_column_required": True,
        "sample_weight_column_required": True,
        "remove_unused_columns": False,
        "cache_format": "pickle",
    }

def _qwen_candidate_input_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_min_s": float(PRIMARY_MIN_GAP_S),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "midturn_negative_ratio": float(MIDTURN_HOLD_NEGATIVE_RATIO),
        "midturn_negative_max_per_session": int(MIDTURN_HOLD_MAX_PER_SESSION),
        "midturn_negative_weight": float(MIDTURN_HOLD_SAMPLE_WEIGHT),
        "seed": int(SEED),
        "use_wavlm": bool(USE_WAVLM_SYS1),
        "use_pyannote": bool(USE_PYANNOTE_DIA),
        "use_asr": bool(USE_ASR_TEXT),
        "allow_train_gt_seed_candidates": bool(ALLOW_TRAIN_GT_SEED_CANDIDATES),
        "merge_tol": float(max(SYS1_CAND_MERGE_TOL_GRID)),
        "n_wavlm_candidates": int(len(WAVLM_CANDIDATES_ALL)) if "WAVLM_CANDIDATES_ALL" in globals() else 0,
        "n_pyannote_candidates": int(len(PYANNOTE_CANDIDATES)) if "PYANNOTE_CANDIDATES" in globals() else 0,
        "n_asr_segments": int(len(ASR_SEGMENTS)) if "ASR_SEGMENTS" in globals() else 0,
        "n_synth_events": int(len(SYNTH_EVENTS)) if "SYNTH_EVENTS" in globals() else 0,
        "n_real_aug_events": int(len(REAL_AUG_EVENTS)) if "REAL_AUG_EVENTS" in globals() else 0,
        "n_synth_aug_events": int(len(SYNTH_AUG_EVENTS)) if "SYNTH_AUG_EVENTS" in globals() else 0,
        "wavlm_candidates_path": str(WAVLM_CANDIDATES_PATH) if "WAVLM_CANDIDATES_PATH" in globals() else None,
        "pyannote_candidates_path": str(PYANNOTE_CANDS_PATH) if "PYANNOTE_CANDS_PATH" in globals() else None,
        "asr_segments_path": str(ASR_SEGMENTS_PATH) if "ASR_SEGMENTS_PATH" in globals() else None,
        "cache_format": "pickle",
    }

def _qwen_score_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "seed": int(SEED),
        "qwen3_model_name": str(QWEN3_MODEL_NAME),
        "qwen3_model_dir": str(QWEN3_MODEL_DIR),
        "qwen3_model_fingerprint": _qwen_model_fingerprint(),
        "qwen3_max_length": int(QWEN3_MAX_LENGTH),
        "score_batch_size": int(QWEN3_SCORE_BATCH_SIZE),
        "candidate_input_meta": _qwen_candidate_input_config(),
        "candidate_input_fingerprint": _qwen_file_fingerprint(QWEN3_CANDIDATE_INPUT_PATH),
        "cache_format": "pickle",
    }

def _json_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        return json.loads(Path(path).read_text()) == expected

    except Exception:
        return False

def _clean_legacy_qwen_parquet():
    for p in [
        QWEN3_MODEL_DIR / "qwen3_action_train_data.parquet",
        CACHE_DIR / "qwen3_candidate_input_pre_qwen.parquet",
        CACHE_DIR / "qwen3_action_scores_on_candidates_all.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet Qwen3 cache:", p)

        except Exception as e:
            print("Could not remove legacy parquet Qwen3 cache:", p, e)

def make_asr_boundary_candidates_from_sessions(sessions: List[str], split_name: str) -> pd.DataFrame:
    if not USE_ASR_TEXT or ASR_SEGMENTS is None or len(ASR_SEGMENTS) == 0 or not sessions:
        return pd.DataFrame()

    rows = []
    ss = set(map(str, sessions))

    segs = ASR_SEGMENTS[ASR_SEGMENTS["session"].astype(str).isin(ss)].copy()

    if len(segs) == 0:
        return pd.DataFrame()

    for _, s in segs.iterrows():
        sid = str(s.session)

        for tcol in ["start", "end"]:
            try:
                tau = float(s[tcol])

            except Exception:
                continue

            if not np.isfinite(tau) or tau < 0:
                continue

            rows.append({
                "session": sid,
                "time": tau,
                "split": split_name,
                "source_asr": 1,
                "asr_boundary_score": 1.0,
            })

    return pd.DataFrame(rows)

def add_labels_to_candidates(
    cand: pd.DataFrame,
    real_events: pd.DataFrame,
    split_name: str,
    tol: float = TOL,
) -> pd.DataFrame:
    cand = cand.copy()

    if len(cand) == 0:
        return cand

    ev = real_events.copy()

    labels = []
    actions = []
    action_ids = []
    dts = []

    for _, r in cand.iterrows():
        sid = str(r.session)
        tau = float(r.time)

        e = ev[ev["session"].astype(str) == sid]

        if len(e) == 0:
            labels.append(-1)
            actions.append("NA")
            action_ids.append(ACTION_TO_ID["NA"])
            dts.append(np.nan)
            continue

        dt = np.abs(e["time"].to_numpy(dtype=float) - tau)
        j = int(np.argmin(dt))

        if dt[j] <= float(tol):
            er = e.iloc[j]
            action = str(er.action)

            if action not in ACTION_TO_ID:
                action = "NA"

            labels.append(int(er.label))
            actions.append(action)
            action_ids.append(int(ACTION_TO_ID[action]))
            dts.append(float(tau - er.time))

        else:
            labels.append(0)
            actions.append("NA")
            action_ids.append(ACTION_TO_ID["NA"])
            dts.append(float(np.min(dt)))

    cand["label"] = labels
    cand["action"] = actions
    cand["action_id"] = action_ids
    cand["dt_to_event"] = dts
    cand["split"] = split_name

    return cand

def _candidate_default_columns() -> Dict[str, Any]:
    defaults = {
        "source_wavlm": 0,
        "source_pyannote": 0,
        "source_asr": 0,
        "source_qwen3": 0,
        "source_synthetic": 0,
        "source_train_gt_seed": 0,
        "source_pyannote_change": 0,
        "source_pyannote_overlap": 0,
        "wavlm_score": 0.0,
        "wavlm_center_score": 0.0,
        "pyannote_change_score": 0.0,
        "pyannote_overlap_score": 0.0,
        "asr_boundary_score": 0.0,
        "qwen_shift_score": 0.0,
        "sample_weight": 1.0,
        "label": -1,
        "action": "NA",
        "action_id": ACTION_TO_ID["NA"],
        "is_synthetic": False,
        "is_diffusion_augmented": False,
    }

    for a in ACTION_NAMES:
        defaults[f"wavlm_p_{a}"] = 0.0
        defaults[f"qwen_p_{a}"] = 0.0

    return defaults

def _choose_merged_action(sub: pd.DataFrame) -> str:
    if sub is None or len(sub) == 0 or "action" not in sub.columns:
        return "NA"

    acts = sub["action"].dropna().astype(str)
    acts = [a for a in acts if a in ACTION_TO_ID]

    if not acts:
        return "NA"

    for a in ["NTT", "ITT"]:
        if a in acts:
            return a

    non_na = [a for a in acts if a != "NA"]

    if non_na:
        return pd.Series(non_na).value_counts().index[0]

    return "NA"

def _choose_merged_label(sub: pd.DataFrame, action: str) -> int:
    if action in SHIFT_ACTIONS:
        return 1

    if "label" in sub.columns:
        labels = pd.to_numeric(sub["label"], errors="coerce").dropna().astype(int)

        if len(labels):
            if int((labels == 1).any()):
                return 1
            if int((labels == 0).any()):
                return 0

    return 0

def merge_candidate_sources(frames: List[pd.DataFrame], merge_tol: float = 0.25) -> pd.DataFrame:
    cols_defaults = _candidate_default_columns()

    frames2 = []

    for f in frames:
        if f is None or len(f) == 0:
            continue

        x = f.copy()

        for c, v in cols_defaults.items():
            if c not in x.columns:
                x[c] = v

        if "session" not in x.columns or "time" not in x.columns:
            continue

        frames2.append(x)

    if not frames2:
        return pd.DataFrame()

    allc = pd.concat(frames2, ignore_index=True)

    allc["session"] = allc["session"].astype(str)
    allc["time"] = pd.to_numeric(allc["time"], errors="coerce")
    allc = allc.dropna(subset=["session", "time"])
    allc = allc[allc["time"] >= 0].sort_values(["session", "time"]).reset_index(drop=True)

    out_rows = []
    score_cols = [
        "wavlm_score",
        "pyannote_change_score",
        "pyannote_overlap_score",
        "asr_boundary_score",
        "qwen_shift_score",
    ]

    for sid, g in allc.groupby("session"):
        g = g.sort_values("time").reset_index(drop=True)
        used = np.zeros(len(g), dtype=bool)
        times = g["time"].to_numpy(dtype=float)

        for i in range(len(g)):
            if used[i]:
                continue

            tau = float(times[i])
            mask = np.abs(times - tau) <= float(merge_tol)
            idxs = np.where(mask & (~used))[0]

            if len(idxs) == 0:
                continue

            sub = g.iloc[idxs].copy()
            used[idxs] = True

            for sc in score_cols:
                if sc not in sub.columns:
                    sub[sc] = 0.0

            merge_scores = sub[score_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
            weights = np.maximum(1e-6, merge_scores.max(axis=1).to_numpy(dtype=float) + 1e-3)

            action = _choose_merged_action(sub)
            label = _choose_merged_label(sub, action)

            row = {
                "session": sid,
                "time": float(np.average(sub["time"].astype(float), weights=weights)),
                "action": action,
                "action_id": int(ACTION_TO_ID[action]),
                "label": int(label),
            }

            if "split" in sub.columns:
                non_null = sub["split"].dropna().astype(str)
                row["split"] = non_null.iloc[0] if len(non_null) else "unknown"
            else:
                row["split"] = "unknown"

            if "audio_path" in sub.columns:
                ap = sub["audio_path"].dropna().astype(str)
                ap = ap[ap != ""]
                row["audio_path"] = ap.iloc[0] if len(ap) else None
            else:
                row["audio_path"] = None

            for c, default in cols_defaults.items():
                if c in {"action", "action_id", "label"}:
                    continue

                if c.startswith("source_"):
                    row[c] = int(pd.to_numeric(sub[c], errors="coerce").fillna(0).max())

                elif c in ["is_synthetic", "is_diffusion_augmented"]:
                    row[c] = bool(sub[c].fillna(False).astype(bool).max())

                else:
                    row[c] = float(pd.to_numeric(sub[c], errors="coerce").fillna(float(default)).max())

            for a in ACTION_NAMES:
                for prefix_name in ["qwen_p", "wavlm_p", "temporal_p", "lgbm_p"]:
                    col = f"{prefix_name}_{a}"

                    if col in sub.columns:
                        row[col] = float(pd.to_numeric(sub[col], errors="coerce").fillna(0).max())

            out_rows.append(row)

    out = pd.DataFrame(out_rows)

    if len(out):
        out = out.sort_values(["split", "session", "time"]).reset_index(drop=True)

    return out

def build_qwen3_text(row: pd.Series) -> str:
    ctx = (
        asr_context_for_candidate(str(row.session), float(row.time))
        if USE_ASR_TEXT
        else {"asr_left": "", "asr_right": "", "asr_boundary_dist": 999.0}
    )

    pyf = (
        pyannote_features(str(row.session), float(row.time))
        if USE_PYANNOTE_DIA
        else {}
    )

    wavlm_score = float(row.get("wavlm_score", 0.0)) if pd.notna(row.get("wavlm_score", 0.0)) else 0.0
    asr_dist = float(ctx.get("asr_boundary_dist", 999.0))
    pyn_dist = float(pyf.get("pyannote_change_dist", 999.0))
    pyn_ov = float(pyf.get("pyannote_overlap_score", 0.0))

    return (
        "Task: classify the conversational turn-taking action at the candidate boundary.\n"
        "Return one action class only through the classifier head.\n"
        "Actions:\n"
        "NA = no action / hold / same-speaker continuation.\n"
        "NTT = normal turn transfer.\n"
        "ITT = interruption turn transfer.\n"
        "BC = backchannel or overlap continuation without floor transfer.\n"
        "BI = brief interjection.\n"
        "Binary mapping: SHIFT = NTT or ITT. HOLD = NA, BC, or BI.\n"
        f"Candidate time: {float(row.time):.3f}\n"
        f"WavLM boundary score: {wavlm_score:.4f}\n"
        f"ASR boundary distance: {asr_dist:.3f}\n"
        f"Pyannote change distance: {pyn_dist:.3f}\n"
        f"Pyannote overlap score: {pyn_ov:.3f}\n"
        f"Left ASR context: {ctx.get('asr_left', '')}\n"
        f"Right ASR context: {ctx.get('asr_right', '')}\n"
    )

def _ensure_qwen_training_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["text", "label_id", "sample_weight", "action"])

    df = df.copy()

    for col, default in {
        "action": "NA",
        "action_id": ACTION_TO_ID["NA"],
        "sample_weight": 1.0,
        "session": "",
        "time": 0.0,
    }.items():
        if col not in df.columns:
            df[col] = default

    df["action"] = df["action"].fillna("NA").astype(str)
    df.loc[~df["action"].isin(ACTION_NAMES), "action"] = "NA"
    df["action_id"] = df["action"].map(ACTION_TO_ID).astype(int)
    df["sample_weight"] = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(1.0).astype(float)
    df["sample_weight"] = df["sample_weight"].clip(lower=1e-6)
    df["label_id"] = df["action_id"].astype(int)

    return df

def build_qwen3_training_dataframe(force: bool = False) -> pd.DataFrame:
    expected = _qwen_train_config()

    if QWEN3_TRAIN_DATA_PATH.exists() and _json_meta_matches(QWEN3_TRAIN_META_PATH, expected) and not force:
        df = pd.read_pickle(QWEN3_TRAIN_DATA_PATH)
        return _ensure_qwen_training_schema(df)

    frames = [REAL_TRAIN_EVENTS]

    if len(SYNTH_EVENTS):
        frames.append(SYNTH_EVENTS)

    if len(REAL_AUG_EVENTS):
        frames.append(REAL_AUG_EVENTS)

    if len(SYNTH_AUG_EVENTS):
        frames.append(SYNTH_AUG_EVENTS)

    df = pd.concat(frames, ignore_index=True)
    df = _ensure_qwen_training_schema(df)

    if DEBUG_SMALL_RUN and len(df) > 3000:
        df = df.sample(3000, random_state=SEED).reset_index(drop=True)

    df = df.copy()
    df["text"] = [
        build_qwen3_text(r)
        for _, r in tqdm(df.iterrows(), total=len(df), desc="Build Qwen3 train texts")
    ]

    df.to_pickle(QWEN3_TRAIN_DATA_PATH)
    QWEN3_TRAIN_META_PATH.write_text(json.dumps(expected, indent=2, sort_keys=True))

    return df

class WeightedTrainerMixin:

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if "labels" not in inputs:
            raise RuntimeError(
                f"Qwen3 compute_loss received no labels. Input keys: {list(inputs.keys())}"
            )

        labels = inputs.pop("labels")
        weights = inputs.pop("sample_weight", None)

        if not torch.is_tensor(labels):
            labels = torch.tensor(labels, device=model.device)

        labels = labels.to(model.device).long()

        if labels.numel() == 0:
            raise RuntimeError("Qwen3 compute_loss received an empty labels tensor.")

        if int(labels.min().item()) < 0 or int(labels.max().item()) >= len(ACTION_NAMES):
            raise RuntimeError(
                f"Qwen3 labels out of range. min={int(labels.min().item())}, max={int(labels.max().item())}"
            )

        outputs = model(**inputs)
        logits = outputs.logits.float()

        if not torch.isfinite(logits).all():
            raise FloatingPointError("Qwen3 logits contain NaN/Inf before loss computation.")

        loss_vec = F.cross_entropy(logits, labels, reduction="none")

        if weights is not None:
            if not torch.is_tensor(weights):
                weights = torch.tensor(weights, device=loss_vec.device)

            weights = weights.to(loss_vec.device).float()
            weights = torch.nan_to_num(weights, nan=1.0, posinf=1.0, neginf=1.0)
            weights = torch.clamp(weights, min=1e-6)

            if weights.shape[0] != loss_vec.shape[0]:
                raise RuntimeError(
                    f"Qwen3 sample_weight shape mismatch: weights={tuple(weights.shape)}, loss_vec={tuple(loss_vec.shape)}"
                )

            loss_vec = loss_vec * weights

        loss = loss_vec.mean()

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Qwen3 loss is not finite: {loss}")

        return (loss, outputs) if return_outputs else loss

def _latest_trainer_checkpoint(trainer_dir: Path) -> Optional[str]:
    trainer_dir = Path(trainer_dir)

    if not trainer_dir.exists():
        return None

    checkpoints = sorted(
        trainer_dir.glob("checkpoint-*"),
        key=lambda p: int(str(p.name).split("-")[-1]) if str(p.name).split("-")[-1].isdigit() else -1,
    )

    if not checkpoints:
        return None

    return str(checkpoints[-1])

def _resolve_lora_modules_to_save(model) -> Optional[List[str]]:
    module_names = [name for name, _ in model.named_modules()]
    leaf_names = set()

    for name in module_names:
        if "." in name:
            leaf_names.add(name.split(".")[-1])
        else:
            leaf_names.add(name)

    modules_to_save = [
        m for m in QWEN3_MODULES_TO_SAVE_CANDIDATES
        if m in leaf_names or any(n.endswith("." + m) for n in module_names)
    ]

    if not modules_to_save:
        print("⚠️ No obvious sequence-classification head found for modules_to_save.")
        print("   Candidate names checked:", QWEN3_MODULES_TO_SAVE_CANDIDATES)
        return None

    print("Qwen3 LoRA modules_to_save:", modules_to_save)
    return modules_to_save

def _check_qwen_tokenized_dataset(ds_tok):
    print("Qwen3 tokenized train columns:", ds_tok["train"].column_names)
    print("Qwen3 tokenized val columns:", ds_tok["test"].column_names)

    assert "labels" in ds_tok["train"].column_names, "Qwen3 train dataset is missing labels."
    assert "labels" in ds_tok["test"].column_names, "Qwen3 val dataset is missing labels."
    assert "sample_weight" in ds_tok["train"].column_names, "Qwen3 train dataset is missing sample_weight."
    assert "sample_weight" in ds_tok["test"].column_names, "Qwen3 val dataset is missing sample_weight."

    _train_labels = np.asarray(ds_tok["train"]["labels"], dtype=int)
    _val_labels = np.asarray(ds_tok["test"]["labels"], dtype=int)

    _train_weights = np.asarray(ds_tok["train"]["sample_weight"], dtype=float)
    _val_weights = np.asarray(ds_tok["test"]["sample_weight"], dtype=float)

    print("Qwen3 train label counts:", dict(zip(*np.unique(_train_labels, return_counts=True))))
    print("Qwen3 val label counts:", dict(zip(*np.unique(_val_labels, return_counts=True))))

    print("Qwen3 train sample_weight finite:", bool(np.isfinite(_train_weights).all()))
    print("Qwen3 val sample_weight finite:", bool(np.isfinite(_val_weights).all()))

    assert len(_train_labels) > 0, "Qwen3 train labels are empty."
    assert len(_val_labels) > 0, "Qwen3 val labels are empty."
    assert set(np.unique(_train_labels)).issubset(set(range(len(ACTION_NAMES)))), "Bad Qwen3 train labels."
    assert set(np.unique(_val_labels)).issubset(set(range(len(ACTION_NAMES)))), "Bad Qwen3 val labels."
    assert np.isfinite(_train_weights).all(), "Qwen3 train sample_weight contains NaN/Inf."
    assert np.isfinite(_val_weights).all(), "Qwen3 val sample_weight contains NaN/Inf."
    assert np.all(_train_weights > 0), "Qwen3 train sample_weight must be positive."
    assert np.all(_val_weights > 0), "Qwen3 val sample_weight must be positive."

def train_qwen3_action_classifier(train_df: pd.DataFrame):
    try:
        from datasets import Dataset as HFDataset, DatasetDict
        from sklearn.model_selection import train_test_split as sk_train_test_split
        from transformers import (
            AutoTokenizer,
            AutoModelForSequenceClassification,
            TrainingArguments,
            Trainer,
            DataCollatorWithPadding,
        )
        from peft import LoraConfig, get_peft_model, TaskType

    except Exception as e:
        raise ImportError(
            "Qwen3 training requires transformers, datasets, peft, accelerate.\n"
            "Install with: pip install -U transformers datasets peft accelerate"
        ) from e

    train_df = _ensure_qwen_training_schema(train_df)

    assert len(train_df) > 0, "Qwen3 train dataframe is empty."
    assert "text" in train_df.columns, "Qwen3 train dataframe missing text column."
    assert "label_id" in train_df.columns, "Qwen3 train dataframe missing label_id column."
    assert "sample_weight" in train_df.columns, "Qwen3 train dataframe missing sample_weight column."

    train_df["label_id"] = pd.to_numeric(train_df["label_id"], errors="coerce").fillna(-1).astype(int)
    train_df["sample_weight"] = pd.to_numeric(train_df["sample_weight"], errors="coerce").fillna(1.0).astype(float)

    assert set(train_df["label_id"].unique()).issubset(set(range(len(ACTION_NAMES)))), \
        f"Qwen3 train dataframe has invalid label_id values: {sorted(train_df['label_id'].unique().tolist())}"
    assert np.isfinite(train_df["sample_weight"].to_numpy(dtype=float)).all(), \
        "Qwen3 train dataframe sample_weight contains NaN/Inf."
    assert (train_df["sample_weight"] > 0).all(), \
        "Qwen3 train dataframe sample_weight must be positive."

    hf_kwargs = _hf_token_kwargs()

    tokenizer = AutoTokenizer.from_pretrained(
        QWEN3_MODEL_NAME,
        trust_remote_code=True,
        **hf_kwargs,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_load_dtype = _qwen_model_dtype()
    print("Qwen3 model load dtype:", model_load_dtype)

    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            QWEN3_MODEL_NAME,
            num_labels=len(ACTION_NAMES),
            id2label=ID_TO_ACTION,
            label2id=ACTION_TO_ID,
            trust_remote_code=True,
            dtype=model_load_dtype,
            **hf_kwargs,
        )
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(
            QWEN3_MODEL_NAME,
            num_labels=len(ACTION_NAMES),
            id2label=ID_TO_ACTION,
            label2id=ACTION_TO_ID,
            trust_remote_code=True,
            torch_dtype=model_load_dtype,
            **hf_kwargs,
        )

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    if QWEN3_GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable()

            if hasattr(model.config, "use_cache"):
                model.config.use_cache = False

        except Exception:
            pass

    modules_to_save = _resolve_lora_modules_to_save(model)

    lora = LoraConfig(
        r=QWEN3_LORA_R,
        lora_alpha=QWEN3_LORA_ALPHA,
        target_modules=QWEN3_TARGET_MODULES,
        lora_dropout=QWEN3_LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=modules_to_save,
    )

    model = get_peft_model(model, lora)

    n_trainable_cast = 0

    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.float()
            n_trainable_cast += 1

    print(f"Qwen3 trainable parameters cast to FP32 tensors: {n_trainable_cast}")

    try:
        model.print_trainable_parameters()

    except Exception:
        pass

    def tok(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            max_length=QWEN3_MAX_LENGTH,
        )

        enc["labels"] = [int(x) for x in batch["label_id"]]

        enc["sample_weight"] = [float(x) for x in batch["sample_weight"]]

        return enc

    base_ds = HFDataset.from_pandas(
        train_df[["text", "label_id", "sample_weight"]].reset_index(drop=True),
        preserve_index=False,
    )

    labels_np = train_df["label_id"].to_numpy(dtype=int)
    idx_all = np.arange(len(train_df), dtype=int)

    label_counts = np.bincount(labels_np, minlength=len(ACTION_NAMES))
    nonzero_counts = label_counts[label_counts > 0]

    can_stratify = (
        len(np.unique(labels_np)) > 1
        and len(nonzero_counts) > 1
        and int(nonzero_counts.min()) >= 2
    )

    try:
        train_idx, val_idx = sk_train_test_split(
            idx_all,
            test_size=float(QWEN3_TRAIN_INTERNAL_VAL_FRAC),
            random_state=int(SEED),
            shuffle=True,
            stratify=labels_np if can_stratify else None,
        )

    except ValueError as e:
        print("⚠️ Stratified Qwen3 split failed; falling back to random split:", e)

        rng = np.random.default_rng(int(SEED))
        idx_perm = rng.permutation(idx_all)

        n_val = max(1, int(round(len(idx_perm) * float(QWEN3_TRAIN_INTERNAL_VAL_FRAC))))
        val_idx = idx_perm[:n_val]
        train_idx = idx_perm[n_val:]

    ds = DatasetDict({
        "train": base_ds.select(train_idx.tolist()),
        "test": base_ds.select(val_idx.tolist()),
    })

    print("Qwen3 split sizes:", {"train": len(ds["train"]), "val": len(ds["test"])})
    print("Qwen3 split stratified:", bool(can_stratify))

    ds_tok = ds.map(
        tok,
        batched=True,
        remove_columns=ds["train"].column_names,
        desc="Tokenize Qwen3 action data",
    )

    _check_qwen_tokenized_dataset(ds_tok)

    WeightedTrainer = type("WeightedTrainer", (WeightedTrainerMixin, Trainer), {})

    trainer_dir = QWEN3_MODEL_DIR / "trainer"

    common_args = dict(
        output_dir=str(trainer_dir),
        per_device_train_batch_size=QWEN3_BATCH_SIZE,
        per_device_eval_batch_size=QWEN3_BATCH_SIZE,
        gradient_accumulation_steps=QWEN3_GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=QWEN3_EPOCHS,
        learning_rate=QWEN3_LEARNING_RATE,
        weight_decay=QWEN3_WEIGHT_DECAY,
        fp16=False,
        bf16=_qwen_use_bf16(),
        max_grad_norm=1.0,
        logging_nan_inf_filter=False,
        save_strategy="epoch",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to=[],
        remove_unused_columns=False,
        label_names=["labels"],
        save_total_limit=3,
    )

    try:
        args = TrainingArguments(
            **common_args,
            eval_strategy="epoch",
        )

    except TypeError:
        args = TrainingArguments(
            **common_args,
            evaluation_strategy="epoch",
        )

    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["test"],
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    try:
        trainer = WeightedTrainer(
            **trainer_kwargs,
            processing_class=tokenizer,
        )

    except TypeError:
        trainer = WeightedTrainer(
            **trainer_kwargs,
            tokenizer=tokenizer,
        )

    resume_ckpt = None

    if QWEN3_RESUME_FROM_CHECKPOINT:
        resume_ckpt = _latest_trainer_checkpoint(trainer_dir)

        if resume_ckpt:
            print("Resuming Qwen3 trainer from:", resume_ckpt)

    trainer.train(resume_from_checkpoint=resume_ckpt)

    train_log = trainer.state.log_history if hasattr(trainer, "state") else []
    loss_values = []

    for row in train_log:
        if "loss" in row:
            try:
                loss_values.append(float(row["loss"]))
            except Exception:
                pass
        if "eval_loss" in row:
            try:
                loss_values.append(float(row["eval_loss"]))
            except Exception:
                pass

    if loss_values:
        finite_losses = [x for x in loss_values if np.isfinite(x)]
        assert len(finite_losses) > 0, "Qwen3 training produced no finite train/eval losses."

        if all(abs(x) < 1e-12 for x in finite_losses):
            raise RuntimeError(
                "Qwen3 training produced only zero losses. This indicates labels/loss are still broken."
            )

    QWEN3_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(QWEN3_MODEL_DIR))
    tokenizer.save_pretrained(str(QWEN3_MODEL_DIR))
    (QWEN3_MODEL_DIR / "action_names.json").write_text(json.dumps(ACTION_NAMES, indent=2))
    (QWEN3_MODEL_DIR / "qwen3_training_config.json").write_text(json.dumps(_qwen_train_config(), indent=2, sort_keys=True))

    return trainer.model, tokenizer

def _load_qwen3_for_scoring():
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        from peft import PeftModel

    except Exception as e:
        raise ImportError(
            "Qwen3 scoring requires transformers and peft.\n"
            "Install with: pip install -U transformers peft accelerate"
        ) from e

    hf_kwargs = _hf_token_kwargs()

    tokenizer = AutoTokenizer.from_pretrained(
        QWEN3_MODEL_DIR if QWEN3_MODEL_DIR.exists() else QWEN3_MODEL_NAME,
        trust_remote_code=True,
        **hf_kwargs,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_load_dtype = _qwen_model_dtype()
    print("Qwen3 model load dtype:", model_load_dtype)

    adapter_config = QWEN3_MODEL_DIR / "adapter_config.json"

    if adapter_config.exists():
        try:
            base = AutoModelForSequenceClassification.from_pretrained(
                QWEN3_MODEL_NAME,
                num_labels=len(ACTION_NAMES),
                id2label=ID_TO_ACTION,
                label2id=ACTION_TO_ID,
                trust_remote_code=True,
                dtype=model_load_dtype,
                **hf_kwargs,
            )
        except TypeError:
            base = AutoModelForSequenceClassification.from_pretrained(
                QWEN3_MODEL_NAME,
                num_labels=len(ACTION_NAMES),
                id2label=ID_TO_ACTION,
                label2id=ACTION_TO_ID,
                trust_remote_code=True,
                torch_dtype=model_load_dtype,
                **hf_kwargs,
            )

        if getattr(base.config, "pad_token_id", None) is None:
            base.config.pad_token_id = tokenizer.pad_token_id

        model = PeftModel.from_pretrained(
            base,
            str(QWEN3_MODEL_DIR),
            is_trainable=False,
        )

    else:
        try:
            model = AutoModelForSequenceClassification.from_pretrained(
                QWEN3_MODEL_DIR,
                num_labels=len(ACTION_NAMES),
                id2label=ID_TO_ACTION,
                label2id=ACTION_TO_ID,
                trust_remote_code=True,
                dtype=model_load_dtype,
                **hf_kwargs,
            )
        except TypeError:
            model = AutoModelForSequenceClassification.from_pretrained(
                QWEN3_MODEL_DIR,
                num_labels=len(ACTION_NAMES),
                id2label=ID_TO_ACTION,
                label2id=ACTION_TO_ID,
                trust_remote_code=True,
                torch_dtype=model_load_dtype,
                **hf_kwargs,
            )

        if getattr(model.config, "pad_token_id", None) is None:
            model.config.pad_token_id = tokenizer.pad_token_id

    model.to(DEVICE)
    model.eval()

    return model, tokenizer

@torch.no_grad()
def score_qwen3_dataframe(
    df: pd.DataFrame,
    out_path: Path,
    batch_size: int = QWEN3_SCORE_BATCH_SIZE,
) -> pd.DataFrame:
    expected = _qwen_score_config()
    out_path = Path(out_path)
    meta_path = QWEN3_SCORES_META_PATH

    if (
        out_path.exists()
        and _json_meta_matches(meta_path, expected)
        and not FORCE_REBUILD_SYS1_CANDIDATES
        and not FORCE_RETRAIN_QWEN3
    ):
        out = pd.read_pickle(out_path)
        print("Loaded Qwen3 candidate scores:", out.shape)
        return out

    if df is None or len(df) == 0:
        out = pd.DataFrame()
        out.to_pickle(out_path)
        meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True))
        return out

    model, tokenizer = _load_qwen3_for_scoring()

    work = df.copy().reset_index(drop=True)

    if "text" not in work.columns:
        work["text"] = [
            build_qwen3_text(r)
            for _, r in tqdm(work.iterrows(), total=len(work), desc="Build Qwen3 score texts")
        ]

    base_cols = [
        "session",
        "time",
        "split",
        "audio_path",
        "source_wavlm",
        "source_pyannote",
        "source_asr",
        "source_qwen3",
        "source_synthetic",
        "source_train_gt_seed",
        "wavlm_score",
        "wavlm_center_score",
        "pyannote_change_score",
        "pyannote_overlap_score",
        "asr_boundary_score",
        "label",
        "action",
        "action_id",
        "is_synthetic",
        "is_diffusion_augmented",
        "sample_weight",
    ]

    base_cols = [c for c in base_cols if c in work.columns]

    rows = []
    batch_size = max(1, int(batch_size))

    for start in tqdm(range(0, len(work), batch_size), desc="Qwen3 score candidates"):
        end = min(len(work), start + batch_size)
        batch_texts = work.loc[start:end - 1, "text"].astype(str).tolist()

        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=QWEN3_MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )

        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits.float()

        if not torch.isfinite(logits).all():
            raise FloatingPointError("Qwen3 scoring logits contain NaN/Inf.")

        probs = torch.softmax(logits, dim=-1).detach().float().cpu().numpy()

        for j in range(probs.shape[0]):
            src = work.iloc[start + j]
            row = {c: src.get(c, np.nan) for c in base_cols}

            for k, a in enumerate(ACTION_NAMES):
                row[f"qwen_p_{a}"] = float(probs[j, k])

            row["qwen_shift_score"] = float(
                row.get("qwen_p_NTT", 0.0) + row.get("qwen_p_ITT", 0.0)
            )
            row["qwen_backchannel_score"] = float(row.get("qwen_p_BC", 0.0))
            row["qwen_interruption_score"] = float(row.get("qwen_p_ITT", 0.0))
            row["source_qwen3"] = 1

            rows.append(row)

    out = pd.DataFrame(rows)

    if len(out):
        out["session"] = out["session"].astype(str)
        out["time"] = pd.to_numeric(out["time"], errors="coerce")
        out = out.dropna(subset=["session", "time"])
        out = out[out["time"] >= 0].copy()

        if "split" not in out.columns:
            out["split"] = "unknown"

        out["split"] = out["split"].fillna("unknown").astype(str)

        for c in QWEN_SCORE_COLUMNS:
            if c not in out.columns:
                out[c] = 0.0
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

        out = out.sort_values(["split", "session", "time"]).reset_index(drop=True)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True))

    print("Saved Qwen3 candidate scores:", out_path)
    return out

def build_pre_qwen_candidate_input(force: bool = False) -> pd.DataFrame:
    expected = _qwen_candidate_input_config()

    if (
        QWEN3_CANDIDATE_INPUT_PATH.exists()
        and _json_meta_matches(QWEN3_CANDIDATE_INPUT_META_PATH, expected)
        and not force
    ):
        return pd.read_pickle(QWEN3_CANDIDATE_INPUT_PATH)

    frames = []

    if "WAVLM_CANDIDATES_ALL" in globals() and len(WAVLM_CANDIDATES_ALL):
        frames.append(WAVLM_CANDIDATES_ALL.assign(source_wavlm=1))

    if "PYANNOTE_CANDIDATES" in globals() and len(PYANNOTE_CANDIDATES):
        frames.append(PYANNOTE_CANDIDATES.assign(source_pyannote=1))

    if USE_ASR_TEXT:
        for split_name, sessions in [
            ("train", TRAIN_SESSIONS),
            ("val", VAL_SESSIONS),
            ("test", TEST_SESSIONS),
            ("synth_train", sorted(SYNTH_EVENTS["session"].astype(str).unique()) if len(SYNTH_EVENTS) else []),
            ("real_diffusion", sorted(REAL_AUG_EVENTS["session"].astype(str).unique()) if len(REAL_AUG_EVENTS) else []),
            ("synth_diffusion", sorted(SYNTH_AUG_EVENTS["session"].astype(str).unique()) if len(SYNTH_AUG_EVENTS) else []),
        ]:
            b = make_asr_boundary_candidates_from_sessions(list(sessions), split_name)

            if len(b):
                frames.append(b)

    if ALLOW_TRAIN_GT_SEED_CANDIDATES:
        seed_cols = ["session", "time", "audio_path", "label", "action", "action_id", "sample_weight"]
        seeds = REAL_TRAIN_EVENTS[seed_cols].copy().assign(split="train", source_train_gt_seed=1)
        frames.append(seeds)

    if len(SYNTH_EVENTS):
        frames.append(
            SYNTH_EVENTS[["session", "time", "audio_path", "label", "action", "action_id", "sample_weight"]]
            .copy()
            .assign(split="synth_train", source_synthetic=1)
        )

    if len(REAL_AUG_EVENTS):
        frames.append(
            REAL_AUG_EVENTS[["session", "time", "audio_path", "label", "action", "action_id", "sample_weight"]]
            .copy()
            .assign(split="real_diffusion", source_train_gt_seed=1)
        )

    if len(SYNTH_AUG_EVENTS):
        frames.append(
            SYNTH_AUG_EVENTS[["session", "time", "audio_path", "label", "action", "action_id", "sample_weight"]]
            .copy()
            .assign(split="synth_diffusion", source_synthetic=1)
        )

    out = merge_candidate_sources(
        frames,
        merge_tol=max(SYS1_CAND_MERGE_TOL_GRID),
    ) if frames else pd.DataFrame()

    if len(out):
        out = out.sort_values(["split", "session", "time"]).reset_index(drop=True)

    out.to_pickle(QWEN3_CANDIDATE_INPUT_PATH)
    QWEN3_CANDIDATE_INPUT_META_PATH.write_text(json.dumps(expected, indent=2, sort_keys=True))

    return out

_clean_legacy_qwen_parquet()

if USE_QWEN3_LLM:
    model_exists = (
        (QWEN3_MODEL_DIR / "adapter_config.json").exists()
        or (QWEN3_MODEL_DIR / "config.json").exists()
    )

    if model_exists and not FORCE_RETRAIN_QWEN3:
        print("Qwen3 classifier exists:", QWEN3_MODEL_DIR)

    else:
        QWEN3_TRAIN_DF = build_qwen3_training_dataframe(force=FORCE_RETRAIN_QWEN3)

        display(QWEN3_TRAIN_DF["action"].value_counts().to_frame("qwen_train_count"))
        display(QWEN3_TRAIN_DF["sample_weight"].describe().to_frame("qwen_sample_weight"))

        train_qwen3_action_classifier(QWEN3_TRAIN_DF)

    QWEN3_CANDIDATE_INPUT = build_pre_qwen_candidate_input(
        force=FORCE_REBUILD_SYS1_CANDIDATES,
    )

    QWEN3_SCORES_ALL = score_qwen3_dataframe(
        QWEN3_CANDIDATE_INPUT,
        QWEN3_SCORES_PATH,
        batch_size=QWEN3_SCORE_BATCH_SIZE,
    )

else:
    QWEN3_CANDIDATE_INPUT = build_pre_qwen_candidate_input(
        force=FORCE_REBUILD_SYS1_CANDIDATES,
    )
    QWEN3_SCORES_ALL = pd.DataFrame()

print("QWEN3_CANDIDATE_INPUT:", QWEN3_CANDIDATE_INPUT.shape)
print("QWEN3_SCORES_ALL:", QWEN3_SCORES_ALL.shape)

if len(QWEN3_CANDIDATE_INPUT):
    display(QWEN3_CANDIDATE_INPUT.groupby("split").size().to_frame("pre_qwen_candidates"))

if len(QWEN3_SCORES_ALL):
    display(QWEN3_SCORES_ALL.head())
    display(QWEN3_SCORES_ALL.groupby("split").size().to_frame("qwen_scored_candidates"))

    print("Qwen shift-score summary:")
    display(QWEN3_SCORES_ALL["qwen_shift_score"].describe().to_frame("qwen_shift_score"))

Qwen3 classifier exists: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/qwen3_action_classifier


Loaded Qwen3 candidate scores: (468068, 29)
QWEN3_CANDIDATE_INPUT: (468068, 34)
QWEN3_SCORES_ALL: (468068, 29)


,pre_qwen_candidates
split,
synth_train,209091
test,183233
train,62238
val,13506


,session,time,split,audio_path,source_wavlm,source_pyannote,source_asr,source_qwen3,source_synthetic,source_train_gt_seed,...,is_diffusion_augmented,sample_weight,qwen_p_NA,qwen_p_NTT,qwen_p_ITT,qwen_p_BC,qwen_p_BI,qwen_shift_score,qwen_backchannel_score,qwen_interruption_score
0,synth_000000,0.036990,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,1,1,0,0,...,False,1.0,0.019876,0.028472,0.504671,0.428309,0.018672,0.533143,0.428309,0.504671
1,synth_000000,0.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,1,0,0,...,False,0.5,0.026082,0.053308,0.651971,0.241728,0.026910,0.705280,0.241728,0.651971
2,synth_000000,1.026594,synth_train,NaN,0,1,0,1,0,0,...,False,1.0,0.292136,0.254308,0.234280,0.137995,0.081282,0.488588,0.137995,0.234280
3,synth_000000,1.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,1,0,0,...,False,0.5,0.457601,0.450506,0.039751,0.031879,0.020263,0.490257,0.031879,0.039751
4,synth_000000,1.989819,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,1,0,1,0,0,...,False,1.0,0.290735,0.186981,0.270994,0.190111,0.061180,0.457975,0.190111,0.270994


,qwen_scored_candidates
split,
synth_train,209091
test,183233
train,62238
val,13506


Qwen shift-score summary:


,qwen_shift_score
count,4.680680e+05
mean,2.453116e-01
std,2.343263e-01
min,5.918191e-10
25%,7.896338e-03
50%,1.956758e-01
75%,4.364129e-01
max,9.965262e-01


In [19]:
# Build multimodal Sys1 candidate union with Qwen3 action hints
SYS1_UNION_PATH = CACHE_DIR / "multimodal_sys1_candidate_union_all_v3.pkl"
SYS1_UNION_META_PATH = CACHE_DIR / "multimodal_sys1_candidate_union_all_v3.meta.json"

def _sys1_union_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _sys1_union_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "seed": int(SEED),
        "tol": float(TOL),
        "merge_tol": float(max(SYS1_CAND_MERGE_TOL_GRID)),
        "use_wavlm": bool(USE_WAVLM_SYS1),
        "use_pyannote": bool(USE_PYANNOTE_DIA),
        "use_asr": bool(USE_ASR_TEXT),
        "use_qwen3": bool(USE_QWEN3_LLM),
        "allow_train_gt_seed_candidates": bool(ALLOW_TRAIN_GT_SEED_CANDIDATES),
        "n_pre_qwen_candidates": int(len(QWEN3_CANDIDATE_INPUT)) if "QWEN3_CANDIDATE_INPUT" in globals() else 0,
        "n_qwen_scores": int(len(QWEN3_SCORES_ALL)) if "QWEN3_SCORES_ALL" in globals() else 0,
        "n_wavlm_candidates": int(len(WAVLM_CANDIDATES_ALL)) if "WAVLM_CANDIDATES_ALL" in globals() else 0,
        "n_pyannote_candidates": int(len(PYANNOTE_CANDIDATES)) if "PYANNOTE_CANDIDATES" in globals() else 0,
        "n_asr_segments": int(len(ASR_SEGMENTS)) if "ASR_SEGMENTS" in globals() else 0,
        "qwen_candidate_input_fingerprint": _sys1_union_file_fingerprint(QWEN3_CANDIDATE_INPUT_PATH) if "QWEN3_CANDIDATE_INPUT_PATH" in globals() else None,
        "qwen_scores_fingerprint": _sys1_union_file_fingerprint(QWEN3_SCORES_PATH) if "QWEN3_SCORES_PATH" in globals() else None,
        "wavlm_candidates_fingerprint": _sys1_union_file_fingerprint(WAVLM_CANDIDATES_PATH) if "WAVLM_CANDIDATES_PATH" in globals() else None,
        "pyannote_candidates_fingerprint": _sys1_union_file_fingerprint(PYANNOTE_CANDS_PATH) if "PYANNOTE_CANDS_PATH" in globals() else None,
        "asr_segments_fingerprint": _sys1_union_file_fingerprint(ASR_SEGMENTS_PATH) if "ASR_SEGMENTS_PATH" in globals() else None,
        "test_labels_attached": False,
        "cache_format": "pickle",
    }

def _sys1_union_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        return json.loads(Path(path).read_text()) == expected

    except Exception:
        return False

def _clean_legacy_sys1_union_parquet():
    for p in [
        CACHE_DIR / "multimodal_sys1_candidate_union_all_v3.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet Sys1 union cache:", p)

        except Exception as e:
            print("Could not remove legacy parquet Sys1 union cache:", p, e)

def split_event_table_for_union(split_name: str) -> pd.DataFrame:
    split_name = str(split_name)

    if split_name == "train":
        return REAL_TRAIN_EVENTS

    if split_name == "val":
        return REAL_VAL_EVENTS

    if split_name == "synth_train":
        return SYNTH_EVENTS

    if split_name == "real_diffusion":
        return REAL_AUG_EVENTS

    if split_name == "synth_diffusion":
        return SYNTH_AUG_EVENTS

    return pd.DataFrame()

def _ensure_sys1_union_schema(df: pd.DataFrame) -> pd.DataFrame:
    base_cols = [
        "session", "time", "split", "audio_path",
        "source_wavlm",
        "source_pyannote",
        "source_pyannote_change",
        "source_pyannote_overlap",
        "source_asr",
        "source_qwen3",
        "source_synthetic",
        "source_train_gt_seed",
        "wavlm_score",
        "wavlm_center_score",
        "pyannote_change_score",
        "pyannote_overlap_score",
        "pyannote_any_candidate_dist",
        "pyannote_overlap_dist",
        "asr_boundary_score",
        "qwen_shift_score",
        "qwen_backchannel_score",
        "qwen_interruption_score",
        "label",
        "action",
        "action_id",
        "dt_to_event",
        "is_synthetic",
        "is_diffusion_augmented",
        "sample_weight",
    ]

    prob_cols = []
    for a in ACTION_NAMES:
        prob_cols.extend([
            f"wavlm_p_{a}",
            f"qwen_p_{a}",
        ])

    cols = list(dict.fromkeys(base_cols + prob_cols))

    if df is None or len(df) == 0:
        return pd.DataFrame(columns=cols)

    df = df.copy()

    defaults = {
        "session": "",
        "time": np.nan,
        "split": "unknown",
        "audio_path": None,
        "source_wavlm": 0,
        "source_pyannote": 0,
        "source_pyannote_change": 0,
        "source_pyannote_overlap": 0,
        "source_asr": 0,
        "source_qwen3": 0,
        "source_synthetic": 0,
        "source_train_gt_seed": 0,
        "wavlm_score": 0.0,
        "wavlm_center_score": 0.0,
        "pyannote_change_score": 0.0,
        "pyannote_overlap_score": 0.0,
        "pyannote_any_candidate_dist": 999.0,
        "pyannote_overlap_dist": 999.0,
        "asr_boundary_score": 0.0,
        "qwen_shift_score": 0.0,
        "qwen_backchannel_score": 0.0,
        "qwen_interruption_score": 0.0,
        "label": -1,
        "action": "NA",
        "action_id": ACTION_TO_ID["NA"],
        "dt_to_event": np.nan,
        "is_synthetic": False,
        "is_diffusion_augmented": False,
        "sample_weight": 1.0,
    }

    for a in ACTION_NAMES:
        defaults[f"wavlm_p_{a}"] = 0.0
        defaults[f"qwen_p_{a}"] = 0.0

    for c, v in defaults.items():
        if c not in df.columns:
            df[c] = v

    df["session"] = df["session"].astype(str)
    df["split"] = df["split"].fillna("unknown").astype(str)
    df["audio_path"] = df["audio_path"].fillna("").astype(str)

    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df = df.dropna(subset=["session", "time"])
    df = df[df["time"] >= 0].reset_index(drop=True)

    source_cols = [c for c in cols if c.startswith("source_")]
    for c in source_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    numeric_cols = [
        "wavlm_score",
        "wavlm_center_score",
        "pyannote_change_score",
        "pyannote_overlap_score",
        "pyannote_any_candidate_dist",
        "pyannote_overlap_dist",
        "asr_boundary_score",
        "qwen_shift_score",
        "qwen_backchannel_score",
        "qwen_interruption_score",
        "dt_to_event",
        "sample_weight",
    ] + prob_cols

    for c in numeric_cols:
        if c not in df.columns:
            df[c] = defaults.get(c, 0.0)

        if c == "dt_to_event":
            df[c] = pd.to_numeric(df[c], errors="coerce")
        elif c in {"pyannote_any_candidate_dist", "pyannote_overlap_dist"}:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(999.0)
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    df["sample_weight"] = df["sample_weight"].clip(lower=1e-6)

    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(-1).astype(int)

    df["action"] = df["action"].fillna("NA").astype(str)
    df.loc[~df["action"].isin(ACTION_NAMES), "action"] = "NA"
    df["action_id"] = df["action"].map(ACTION_TO_ID).fillna(ACTION_TO_ID["NA"]).astype(int)

    df.loc[df["action"].isin(SHIFT_ACTIONS), "label"] = 1
    df.loc[df["action"].isin(HOLD_ACTIONS) & (df["label"] != -1), "label"] = 0

    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["is_diffusion_augmented"] = df["is_diffusion_augmented"].fillna(False).astype(bool)

    return (
        df[cols]
        .drop_duplicates(subset=["session", "time", "split"])
        .sort_values(["split", "session", "time"])
        .reset_index(drop=True)
    )

def _attach_labels_to_non_test_splits(union_df: pd.DataFrame) -> pd.DataFrame:
    if union_df is None or len(union_df) == 0:
        return _ensure_sys1_union_schema(union_df)

    union_df = _ensure_sys1_union_schema(union_df)

    labeled_parts = []
    label_splits = ["train", "val", "synth_train", "real_diffusion", "synth_diffusion"]

    for split_name in label_splits:
        ev = split_event_table_for_union(split_name)

        if ev is None or len(ev) == 0:
            continue

        part = union_df[union_df["split"].astype(str) == split_name].copy()

        if len(part) == 0:
            sessions = set(ev["session"].astype(str).unique())
            part = union_df[union_df["session"].astype(str).isin(sessions)].copy()

        if len(part) == 0:
            continue

        part = add_labels_to_candidates(
            part,
            ev,
            split_name,
            tol=TOL,
        )

        labeled_parts.append(part)

    labeled_splits = set(label_splits)
    unlabeled = union_df[~union_df["split"].astype(str).isin(labeled_splits)].copy()

    if len(unlabeled):
        unlabeled["label"] = -1
        unlabeled["action"] = "NA"
        unlabeled["action_id"] = ACTION_TO_ID["NA"]
        unlabeled["dt_to_event"] = np.nan
        labeled_parts.append(unlabeled)

    if not labeled_parts:
        out = union_df.copy()
    else:
        out = pd.concat(labeled_parts, ignore_index=True)

    return _ensure_sys1_union_schema(out)

_clean_legacy_sys1_union_parquet()

expected_sys1_union_meta = _sys1_union_config()

if (
    SYS1_UNION_PATH.exists()
    and _sys1_union_meta_matches(SYS1_UNION_META_PATH, expected_sys1_union_meta)
    and not FORCE_REBUILD_SYS1_CANDIDATES
):
    SYS1_UNION_ALL = pd.read_pickle(SYS1_UNION_PATH)
    SYS1_UNION_ALL = _ensure_sys1_union_schema(SYS1_UNION_ALL)

    print("Loaded Sys1 union:", SYS1_UNION_ALL.shape)

else:
    frames = []

    if (
        "QWEN3_CANDIDATE_INPUT" in globals()
        and QWEN3_CANDIDATE_INPUT is not None
        and len(QWEN3_CANDIDATE_INPUT)
    ):
        frames.append(QWEN3_CANDIDATE_INPUT)

    if (
        "QWEN3_SCORES_ALL" in globals()
        and QWEN3_SCORES_ALL is not None
        and len(QWEN3_SCORES_ALL)
    ):
        q_cols = [
            "session",
            "time",
            "split",
            "audio_path",
            "qwen_shift_score",
            "qwen_backchannel_score",
            "qwen_interruption_score",
        ] + [f"qwen_p_{a}" for a in ACTION_NAMES]

        q_cols = [c for c in q_cols if c in QWEN3_SCORES_ALL.columns]

        q = QWEN3_SCORES_ALL[q_cols].copy()
        q["source_qwen3"] = 1
        frames.append(q)

    if frames:
        SYS1_UNION_ALL = merge_candidate_sources(
            frames,
            merge_tol=max(SYS1_CAND_MERGE_TOL_GRID),
        )
    else:
        SYS1_UNION_ALL = pd.DataFrame()

    SYS1_UNION_ALL = _attach_labels_to_non_test_splits(SYS1_UNION_ALL)

    SYS1_UNION_ALL.to_pickle(SYS1_UNION_PATH)
    SYS1_UNION_META_PATH.write_text(json.dumps(expected_sys1_union_meta, indent=2, sort_keys=True))

    print("Saved Sys1 union:", SYS1_UNION_PATH)

print("SYS1_UNION_ALL:", SYS1_UNION_ALL.shape)

if len(SYS1_UNION_ALL):
    display(SYS1_UNION_ALL.head())
    display(SYS1_UNION_ALL.groupby("split").size().to_frame("candidate_count"))

    source_cols = [c for c in SYS1_UNION_ALL.columns if c.startswith("source_")]
    if source_cols:
        display(SYS1_UNION_ALL[source_cols].sum().to_frame("source_count"))

    print("Label summary by split:")
    display(
        SYS1_UNION_ALL
        .groupby("split")["label"]
        .value_counts(dropna=False)
        .to_frame("n")
    )

    if "qwen_shift_score" in SYS1_UNION_ALL.columns:
        print("Qwen shift-score summary:")
        display(SYS1_UNION_ALL["qwen_shift_score"].describe().to_frame("qwen_shift_score"))

    print("TEST leakage check:")
    if "test" in set(SYS1_UNION_ALL["split"].astype(str)):
        test_label_counts = SYS1_UNION_ALL.loc[
            SYS1_UNION_ALL["split"].astype(str) == "test",
            "label"
        ].value_counts(dropna=False).to_dict()
        print("  TEST label counts:", test_label_counts)
        assert set(test_label_counts.keys()).issubset({-1}), "TEST labels should remain -1 in Cell 14."
    else:
        print("  No TEST rows in SYS1_UNION_ALL.")

Loaded Sys1 union: (425139, 39)
SYS1_UNION_ALL: (425139, 39)


,session,time,split,audio_path,source_wavlm,source_pyannote,source_pyannote_change,source_pyannote_overlap,source_asr,source_qwen3,...,wavlm_p_NA,qwen_p_NA,wavlm_p_NTT,qwen_p_NTT,wavlm_p_ITT,qwen_p_ITT,wavlm_p_BC,qwen_p_BC,wavlm_p_BI,qwen_p_BI
0,synth_000000,0.036990,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,1,1,...,0.669130,0.019876,0.195304,0.028472,0.686777,0.504671,0.126965,0.428309,0.001393,0.018672
1,synth_000000,0.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,0,1,...,0.291277,0.026082,0.100523,0.053308,0.570787,0.651971,0.034066,0.241728,0.003348,0.026910
2,synth_000000,1.026594,synth_train,,0,1,1,0,0,1,...,0.000000,0.292136,0.000000,0.254308,0.000000,0.234280,0.000000,0.137995,0.000000,0.081282
3,synth_000000,1.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,0,1,...,0.250314,0.457601,0.706248,0.450506,0.018161,0.039751,0.017386,0.031879,0.007891,0.020263
4,synth_000000,1.989819,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,1,1,0,0,1,...,0.014233,0.290735,0.015893,0.186981,0.718837,0.270994,0.471850,0.190111,0.030430,0.061180


,candidate_count
split,
synth_train,186015
test,169181
train,57421
val,12522


,source_count
source_wavlm,372532
source_pyannote,91569
source_pyannote_change,59950
source_pyannote_overlap,51811
source_asr,129326
source_qwen3,425139
source_synthetic,56925
source_train_gt_seed,5559


Label summary by split:


n
split       label        
synth_train  0     145453
             1      40547
            -1         15
test        -1     169181
train        0      53886
             1       3535
val          0      11728
             1        734
            -1         60

Qwen shift-score summary:


,qwen_shift_score
count,4.251390e+05
mean,2.465050e-01
std,2.351809e-01
min,5.918191e-10
25%,1.423783e-02
50%,1.943595e-01
75%,4.352125e-01
max,9.965262e-01


TEST leakage check:
  TEST label counts: {-1: 169181}


In [20]:
# Tune/evaluate Sys1 candidate recall on real VAL
import hashlib
import json
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

SYS1_LOCKED_PATH = Path(RUN_ROOT) / "sys1_locked_settings.json"
SYS1_TUNE_PATH = Path(RUN_ROOT) / "sys1_val_recall_sweep.csv"

SYS1_TUNING_PROTOCOL_VERSION = str(
    globals().get(
        "SYS1_TUNING_PROTOCOL_VERSION",
        "sys1_val_candidate_oracle_v3_canonical_float64",
    )
)
SYS1_MATCHING_PROTOCOL = "float64_one_to_one_minimum_cost_assignment_v1"

def _as_bool_grid(value, default):
    if value is None:
        value = default
    if isinstance(value, (bool, np.bool_)):
        value = [bool(value)]
    elif not isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        value = [value]

    out = []
    for item in value:
        if isinstance(item, str):
            parsed = item.strip().lower() in {"1", "true", "yes", "y", "on"}
        else:
            parsed = bool(item)
        if parsed not in out:
            out.append(parsed)
    return out or list(default)

SYS1_USE_PYANNOTE_SOURCE_GRID = _as_bool_grid(
    globals().get("SYS1_USE_PYANNOTE_SOURCE_GRID", [True, False]),
    [True, False],
)
SYS1_USE_ASR_SOURCE_GRID = _as_bool_grid(
    globals().get("SYS1_USE_ASR_SOURCE_GRID", [True, False]),
    [True, False],
)

def _norm_session_id_15(value) -> str:
    return Path(str(value)).stem

def _safe_numeric_array(df: pd.DataFrame, col: str, default: float = 0.0) -> np.ndarray:
    if df is None or len(df) == 0:
        return np.array([], dtype=np.float64)
    if col not in df.columns:
        return np.full(len(df), float(default), dtype=np.float64)
    return (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(float(default))
        .to_numpy(dtype=np.float64)
    )

def _safe_int_array(df: pd.DataFrame, col: str, default: int = 0) -> np.ndarray:
    if df is None or len(df) == 0:
        return np.array([], dtype=np.int64)
    if col not in df.columns:
        return np.full(len(df), int(default), dtype=np.int64)
    return (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(int(default))
        .astype(int)
        .to_numpy(dtype=np.int64)
    )

def _clear_test_labels(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df

    out = df.copy()
    if "split" not in out.columns:
        return out

    test_mask = out["split"].fillna("unknown").astype(str).eq("test")
    if not test_mask.any():
        return out

    out.loc[test_mask, "label"] = -1
    out.loc[test_mask, "action"] = "NA"

    if "ACTION_TO_ID" in globals() and "NA" in ACTION_TO_ID:
        out.loc[test_mask, "action_id"] = int(ACTION_TO_ID["NA"])
    else:
        out.loc[test_mask, "action_id"] = -1

    if "dt_to_event" not in out.columns:
        out["dt_to_event"] = np.nan
    out.loc[test_mask, "dt_to_event"] = np.nan

    return out

def _prepare_candidate_times_15(cand_df: pd.DataFrame) -> pd.DataFrame:
    if cand_df is None or len(cand_df) == 0:
        return pd.DataFrame(columns=["session", "time"])

    if not {"session", "time"}.issubset(cand_df.columns):
        raise ValueError("Candidate table must contain session and time columns.")

    out = cand_df[["session", "time"]].copy()
    out["session"] = out["session"].astype(str).map(_norm_session_id_15)
    out["time"] = pd.to_numeric(out["time"], errors="coerce").astype("float64")
    out = out.dropna(subset=["session", "time"])
    out = out[out["time"] >= 0.0].copy()
    out["_time_key"] = out["time"].round(9)
    out = (
        out.drop_duplicates(["session", "_time_key"], keep="first")
        .drop(columns=["_time_key"])
        .sort_values(["session", "time"])
        .reset_index(drop=True)
    )
    return out

def _prepare_gt_shift_events_15(events_df: pd.DataFrame) -> pd.DataFrame:
    if events_df is None or len(events_df) == 0:
        return pd.DataFrame(columns=["session", "time"])

    if not {"session", "time"}.issubset(events_df.columns):
        raise ValueError("GT event table must contain session and time columns.")

    out = events_df.copy()
    out["session"] = out["session"].astype(str).map(_norm_session_id_15)
    out["time"] = pd.to_numeric(out["time"], errors="coerce").astype("float64")

    if "label" in out.columns:
        out["label"] = (
            pd.to_numeric(out["label"], errors="coerce")
            .fillna(0)
            .astype(int)
        )
        out = out[out["label"].eq(1)].copy()

    if "source_event_type" in out.columns:
        adjacent = out["source_event_type"].astype(str).eq("adjacent_rttm_boundary")
        if adjacent.any():
            out = out[adjacent].copy()

    out = out.dropna(subset=["session", "time"])
    out["_time_key"] = out["time"].round(9)
    out = (
        out.drop_duplicates(["session", "_time_key"], keep="first")
        .drop(columns=["_time_key"])
        .sort_values(["session", "time"])
        .reset_index(drop=True)
    )
    return out

def _canonical_match_indices_15(
    pred_times,
    gt_times,
    tol: float,
) -> List[Tuple[int, int, float]]:
    pred = np.asarray(pred_times, dtype=np.float64).reshape(-1)
    gt = np.asarray(gt_times, dtype=np.float64).reshape(-1)

    if len(pred) == 0 or len(gt) == 0:
        return []

    cost = np.abs(pred[:, None] - gt[None, :])
    tol = float(tol)

    try:
        from scipy.optimize import linear_sum_assignment

        invalid_cost = 1e9
        row_ind, col_ind = linear_sum_assignment(
            np.where(cost <= tol, cost, invalid_cost)
        )
        pairs = [
            (int(i), int(j), float(cost[i, j]))
            for i, j in zip(row_ind, col_ind)
            if cost[i, j] <= tol
        ]
        return sorted(pairs, key=lambda z: (z[0], z[1]))

    except Exception:
        candidates = []
        for i in range(len(pred)):
            for j in np.where(cost[i] <= tol)[0]:
                candidates.append((float(cost[i, j]), int(i), int(j)))

        candidates.sort(key=lambda z: (z[0], z[1], z[2]))
        used_pred, used_gt, pairs = set(), set(), []

        for error, i, j in candidates:
            if i in used_pred or j in used_gt:
                continue
            used_pred.add(i)
            used_gt.add(j)
            pairs.append((i, j, error))

        return sorted(pairs, key=lambda z: (z[0], z[1]))

def candidate_oracle_metrics(
    cand_df: pd.DataFrame,
    events_df: pd.DataFrame,
    tol: Optional[float] = None,
) -> Dict[str, Any]:
    tol = float(TOL if tol is None else tol)

    pred = _prepare_candidate_times_15(cand_df)
    gt = _prepare_gt_shift_events_15(events_df)

    sessions = sorted(set(pred["session"]).union(set(gt["session"])))

    total = {"TP": 0, "FP": 0, "FN": 0, "N_pred": 0, "N_gt": 0}

    for sid in sessions:
        pred_times = pred.loc[pred["session"].eq(sid), "time"].to_numpy(np.float64)
        gt_times = gt.loc[gt["session"].eq(sid), "time"].to_numpy(np.float64)

        pairs = _canonical_match_indices_15(pred_times, gt_times, tol=tol)
        tp = int(len(pairs))
        n_pred = int(len(pred_times))
        n_gt = int(len(gt_times))

        total["TP"] += tp
        total["FP"] += n_pred - tp
        total["FN"] += n_gt - tp
        total["N_pred"] += n_pred
        total["N_gt"] += n_gt

    tp, fp, fn = total["TP"], total["FP"], total["FN"]
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2.0 * precision * recall / max(1e-12, precision + recall)

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        **total,
    }

def filter_sys1_candidates(
    df: pd.DataFrame,
    wavlm_thr: float = 0.10,
    qwen_thr: float = 0.10,
    use_pyannote: bool = True,
    use_asr: bool = True,
    merge_tol: float = 0.25,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    x = df.copy()

    if not {"session", "time"}.issubset(x.columns):
        raise ValueError("Sys1 candidate table must contain session and time columns.")

    if "split" not in x.columns:
        x["split"] = "unknown"

    x["split"] = x["split"].fillna("unknown").astype(str)
    x["session"] = x["session"].astype(str).map(_norm_session_id_15)
    x["time"] = pd.to_numeric(x["time"], errors="coerce")
    x = x.dropna(subset=["session", "time"]).reset_index(drop=True)

    if len(x) == 0:
        return x

    keep = np.zeros(len(x), dtype=bool)

    keep |= (
        (_safe_int_array(x, "source_wavlm", 0) == 1)
        & (_safe_numeric_array(x, "wavlm_score", 0.0) >= float(wavlm_thr))
    )

    keep |= (
        (_safe_int_array(x, "source_qwen3", 0) == 1)
        & (_safe_numeric_array(x, "qwen_shift_score", 0.0) >= float(qwen_thr))
    )

    if bool(use_pyannote):
        keep |= (_safe_int_array(x, "source_pyannote", 0) == 1)

    if bool(use_asr):
        keep |= (_safe_int_array(x, "source_asr", 0) == 1)

    train_like = x["split"].isin(
        ["train", "synth_train", "real_diffusion", "synth_diffusion"]
    ).to_numpy()

    keep |= train_like & (_safe_int_array(x, "source_train_gt_seed", 0) == 1)
    keep |= train_like & (_safe_int_array(x, "source_synthetic", 0) == 1)

    out = x.loc[keep].copy()

    if len(out) == 0:
        return out

    out = merge_candidate_sources([out], merge_tol=float(merge_tol))

    if len(out):
        out["session"] = out["session"].astype(str).map(_norm_session_id_15)
        out["time"] = pd.to_numeric(out["time"], errors="coerce")
        out = (
            out.dropna(subset=["session", "time"])
            .sort_values(["split", "session", "time"])
            .reset_index(drop=True)
        )
        out = _clear_test_labels(out)

    return out

def filter_sys1_candidates_by_split(
    df: pd.DataFrame,
    wavlm_thr: float = 0.10,
    qwen_thr: float = 0.10,
    use_pyannote: bool = True,
    use_asr: bool = True,
    merge_tol: float = 0.25,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    x = df.copy()

    if "split" not in x.columns:
        x["split"] = "unknown"

    x["split"] = x["split"].fillna("unknown").astype(str)

    parts = []

    for split_name, part in x.groupby("split", sort=False):
        filtered = filter_sys1_candidates(
            part.copy(),
            wavlm_thr=float(wavlm_thr),
            qwen_thr=float(qwen_thr),
            use_pyannote=bool(use_pyannote),
            use_asr=bool(use_asr),
            merge_tol=float(merge_tol),
        )

        if len(filtered):
            filtered["split"] = str(split_name)

            if str(split_name) == "test":
                filtered = _clear_test_labels(filtered)

            parts.append(filtered)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)
    out = out.sort_values(["split", "session", "time"]).reset_index(drop=True)

    return _clear_test_labels(out)

def _json_safe_dict(values: Dict[str, Any]) -> Dict[str, Any]:
    out = {}

    for key, value in values.items():
        if isinstance(value, (np.floating, float)):
            out[key] = float(value)
        elif isinstance(value, (np.integer, int)):
            out[key] = int(value)
        elif isinstance(value, (np.bool_, bool)):
            out[key] = bool(value)
        else:
            out[key] = value

    return out

def _sys1_tuning_signature_payload() -> Dict[str, Any]:
    return {
        "protocol_version": SYS1_TUNING_PROTOCOL_VERSION,
        "matching_protocol": SYS1_MATCHING_PROTOCOL,
        "tol": float(TOL),
        "wavlm_grid": [float(x) for x in SYS1_WAVLM_THR_GRID],
        "qwen_grid": [float(x) for x in SYS1_QWEN_THR_GRID],
        "use_pyannote_grid": [bool(x) for x in SYS1_USE_PYANNOTE_SOURCE_GRID],
        "use_asr_grid": [bool(x) for x in SYS1_USE_ASR_SOURCE_GRID],
        "merge_tol_grid": [float(x) for x in SYS1_CAND_MERGE_TOL_GRID],
        "target_recall": float(TARGET_SYS1_RECALL),
        "preferred_recall": float(PREFERRED_SYS1_RECALL),
    }

SYS1_TUNING_SIGNATURE_PAYLOAD = _sys1_tuning_signature_payload()
SYS1_TUNING_SIGNATURE = hashlib.sha256(
    json.dumps(
        SYS1_TUNING_SIGNATURE_PAYLOAD,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

def _select_sys1_setting(tune_df: pd.DataFrame) -> pd.Series:
    assert tune_df is not None and len(tune_df) > 0, "Empty Sys1 tuning dataframe."

    frame = tune_df.copy()
    preferred = frame[frame["R"] >= float(PREFERRED_SYS1_RECALL)]
    target = frame[frame["R"] >= float(TARGET_SYS1_RECALL)]

    tail = ["wavlm_thr", "qwen_thr", "use_pyannote", "use_asr", "merge_tol"]

    if len(preferred):
        pool = preferred.copy()
        reason = f"reached preferred recall >= {PREFERRED_SYS1_RECALL:.3f}"
        sort_cols = ["P", "F1", "R", "N_pred", *tail]
        ascending = [False, False, False, True, True, True, False, False, True]

    elif len(target):
        pool = target.copy()
        reason = f"reached target recall >= {TARGET_SYS1_RECALL:.3f}"
        sort_cols = ["P", "F1", "R", "N_pred", *tail]
        ascending = [False, False, False, True, True, True, False, False, True]

    else:
        pool = frame.copy()
        reason = "no setting reached recall floor; selected highest recall"
        sort_cols = ["R", "P", "F1", "N_pred", *tail]
        ascending = [False, False, False, True, True, True, False, False, True]

    pool = pool.sort_values(sort_cols, ascending=ascending, kind="mergesort")
    best = pool.iloc[0].copy()
    best["selection_reason"] = reason

    return best

def _sys1_locked_is_compatible(values: Dict[str, Any]) -> bool:
    required = {
        "wavlm_thr",
        "qwen_thr",
        "use_pyannote",
        "use_asr",
        "merge_tol",
        "tol",
        "locked_on",
        "protocol_version",
        "matching_protocol",
        "tuning_signature",
    }

    if not isinstance(values, dict):
        return False

    if not required.issubset(values.keys()):
        return False

    if str(values.get("locked_on", "")) != "real_val_candidate_oracle":
        return False

    if str(values.get("protocol_version", "")) != SYS1_TUNING_PROTOCOL_VERSION:
        return False

    if str(values.get("matching_protocol", "")) != SYS1_MATCHING_PROTOCOL:
        return False

    if str(values.get("tuning_signature", "")) != SYS1_TUNING_SIGNATURE:
        return False

    try:
        return abs(float(values.get("tol", -999.0)) - float(TOL)) <= 1e-9
    except Exception:
        return False

SYS1_LOCKED = None
SYS1_TUNE_DF = pd.DataFrame()

if SYS1_LOCKED_PATH.exists() and not bool(FORCE_RETUNE_VAL):
    try:
        loaded_locked = json.loads(SYS1_LOCKED_PATH.read_text())
    except Exception as exc:
        print("Could not read existing Sys1 lock:", repr(exc))
        loaded_locked = None

    if _sys1_locked_is_compatible(loaded_locked):
        SYS1_LOCKED = loaded_locked
        print("Loaded compatible locked Sys1 settings:")
        print(json.dumps(SYS1_LOCKED, indent=2, sort_keys=True))

        if SYS1_TUNE_PATH.exists():
            try:
                SYS1_TUNE_DF = pd.read_csv(SYS1_TUNE_PATH)
            except Exception as exc:
                print("Could not load prior Sys1 sweep:", repr(exc))
                SYS1_TUNE_DF = pd.DataFrame()

    else:
        print("Existing Sys1 lock is stale/incompatible with the canonical protocol.")
        print("Retuning Sys1 candidate settings on real VAL only.")
        FORCE_RETUNE_VAL = True

if SYS1_LOCKED is None or bool(FORCE_RETUNE_VAL):
    val_union = SYS1_UNION_ALL[
        SYS1_UNION_ALL["split"].astype(str).eq("val")
    ].copy()

    assert len(val_union) > 0, (
        "No VAL candidates found in SYS1_UNION_ALL. Check Cells 12–14."
    )

    results = []

    for wav_thr in SYS1_WAVLM_THR_GRID:
        for q_thr in SYS1_QWEN_THR_GRID:
            for use_pyannote in SYS1_USE_PYANNOTE_SOURCE_GRID:
                for use_asr in SYS1_USE_ASR_SOURCE_GRID:
                    for merge_tol in SYS1_CAND_MERGE_TOL_GRID:
                        candidates = filter_sys1_candidates(
                            val_union,
                            wavlm_thr=float(wav_thr),
                            qwen_thr=float(q_thr),
                            use_pyannote=bool(use_pyannote),
                            use_asr=bool(use_asr),
                            merge_tol=float(merge_tol),
                        )

                        metrics = candidate_oracle_metrics(
                            candidates,
                            REAL_VAL_EVENTS,
                            tol=float(TOL),
                        )

                        results.append(
                            {
                                "wavlm_thr": float(wav_thr),
                                "qwen_thr": float(q_thr),
                                "use_pyannote": bool(use_pyannote),
                                "use_asr": bool(use_asr),
                                "merge_tol": float(merge_tol),
                                **metrics,
                            }
                        )

    SYS1_TUNE_DF = pd.DataFrame(results)

    assert len(SYS1_TUNE_DF) > 0, "Sys1 tuning produced no rows."

    SYS1_TUNE_DF = (
        SYS1_TUNE_DF.sort_values(
            [
                "R",
                "P",
                "F1",
                "N_pred",
                "wavlm_thr",
                "qwen_thr",
                "use_pyannote",
                "use_asr",
                "merge_tol",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                True,
                True,
                False,
                False,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    best = _select_sys1_setting(SYS1_TUNE_DF)

    SYS1_LOCKED = _json_safe_dict(best.to_dict())
    SYS1_LOCKED.update(
        {
            "target_sys1_recall": float(TARGET_SYS1_RECALL),
            "preferred_sys1_recall": float(PREFERRED_SYS1_RECALL),
            "tol": float(TOL),
            "locked_on": "real_val_candidate_oracle",
            "test_not_used_for_selection": True,
            "protocol_version": SYS1_TUNING_PROTOCOL_VERSION,
            "matching_protocol": SYS1_MATCHING_PROTOCOL,
            "tuning_signature": SYS1_TUNING_SIGNATURE,
            "tuning_signature_payload": SYS1_TUNING_SIGNATURE_PAYLOAD,
        }
    )

    SYS1_LOCKED_PATH.write_text(
        json.dumps(SYS1_LOCKED, indent=2, sort_keys=True)
    )
    SYS1_TUNE_DF.to_csv(SYS1_TUNE_PATH, index=False)

    print("Saved Sys1 locked settings:", SYS1_LOCKED_PATH)
    print("Saved Sys1 VAL recall sweep:", SYS1_TUNE_PATH)
    display(SYS1_TUNE_DF.head(10))

assert isinstance(SYS1_LOCKED, dict), "SYS1_LOCKED was not created or loaded."

print("SYS1_LOCKED:")
print(json.dumps(SYS1_LOCKED, indent=2, sort_keys=True))

SYS1_FILTERED_ALL = filter_sys1_candidates_by_split(
    SYS1_UNION_ALL,
    wavlm_thr=float(SYS1_LOCKED.get("wavlm_thr", 0.10)),
    qwen_thr=float(SYS1_LOCKED.get("qwen_thr", 0.10)),
    use_pyannote=bool(SYS1_LOCKED.get("use_pyannote", True)),
    use_asr=bool(SYS1_LOCKED.get("use_asr", True)),
    merge_tol=float(SYS1_LOCKED.get("merge_tol", 0.25)),
)

SYS1_FILTERED_ALL = _clear_test_labels(SYS1_FILTERED_ALL)

SYS1_FILTERED_VAL = SYS1_FILTERED_ALL[
    SYS1_FILTERED_ALL["split"].astype(str).eq("val")
].copy()

SYS1_FILTERED_TEST = SYS1_FILTERED_ALL[
    SYS1_FILTERED_ALL["split"].astype(str).eq("test")
].copy()

VAL_SYS1_ORACLE = candidate_oracle_metrics(
    SYS1_FILTERED_VAL,
    REAL_VAL_EVENTS,
    tol=float(TOL),
)

print("SYS1 tuning protocol:", SYS1_TUNING_PROTOCOL_VERSION)
print("SYS1 matching protocol:", SYS1_MATCHING_PROTOCOL)
print("SYS1 tuning signature:", SYS1_TUNING_SIGNATURE)
print("SYS1_FILTERED_ALL:", SYS1_FILTERED_ALL.shape)
print("SYS1_FILTERED_VAL:", SYS1_FILTERED_VAL.shape)
print("SYS1_FILTERED_TEST:", SYS1_FILTERED_TEST.shape)
print("VAL_SYS1_ORACLE:", VAL_SYS1_ORACLE)

if len(SYS1_FILTERED_ALL):
    display(
        SYS1_FILTERED_ALL.groupby("split").size().to_frame(
            "filtered_candidate_count"
        )
    )

if len(SYS1_FILTERED_VAL):
    print("VAL filtered source counts:")
    val_source_cols = [
        c for c in SYS1_FILTERED_VAL.columns if c.startswith("source_")
    ]
    if val_source_cols:
        display(
            SYS1_FILTERED_VAL[val_source_cols]
            .sum()
            .to_frame("val_source_count")
        )

if len(SYS1_FILTERED_TEST):
    print("TEST filtered source counts:")
    test_source_cols = [
        c for c in SYS1_FILTERED_TEST.columns if c.startswith("source_")
    ]
    if test_source_cols:
        display(
            SYS1_FILTERED_TEST[test_source_cols]
            .sum()
            .to_frame("test_source_count")
        )

    print("TEST filtered label check — should be only -1:")
    test_label_counts = (
        SYS1_FILTERED_TEST["label"]
        .value_counts(dropna=False)
        .to_frame("test_label_count")
    )
    display(test_label_counts)

    assert set(test_label_counts.index).issubset({-1}), (
        "TEST labels should remain -1 after Sys1 filtering."
    )

print(
    "✅ Cell 15 complete: Sys1 candidate settings are VAL-locked "
    "with canonical float64 event matching."
)

Loaded compatible locked Sys1 settings:
{
  "F1": 0.10859513930053348,
  "FN": 9,
  "FP": 7510,
  "N_gt": 467,
  "N_pred": 7968,
  "P": 0.05747991967871486,
  "R": 0.9807280513918629,
  "TP": 458,
  "locked_on": "real_val_candidate_oracle",
  "matching_protocol": "float64_one_to_one_minimum_cost_assignment_v1",
  "merge_tol": 0.12,
  "preferred_sys1_recall": 0.95,
  "protocol_version": "sys1_val_candidate_oracle_v3_canonical_float64",
  "qwen_thr": 0.25,
  "selection_reason": "reached preferred recall >= 0.950",
  "target_sys1_recall": 0.9,
  "test_not_used_for_selection": true,
  "tol": 0.5,
  "tuning_signature": "9fdbb2f02e7a03c7e99e493344dfac368c78a210e8e27d23352c2a7d02a65106",
  "tuning_signature_payload": {
    "matching_protocol": "float64_one_to_one_minimum_cost_assignment_v1",
    "merge_tol_grid": [
      0.12,
      0.2,
      0.25,
      0.35
    ],
    "preferred_recall": 0.95,
    "protocol_version": "sys1_val_candidate_oracle_v3_canonical_float64",
    "qwen_grid": [
    

SYS1 tuning protocol: sys1_val_candidate_oracle_v3_canonical_float64
SYS1 matching protocol: float64_one_to_one_minimum_cost_assignment_v1
SYS1 tuning signature: 9fdbb2f02e7a03c7e99e493344dfac368c78a210e8e27d23352c2a7d02a65106
SYS1_FILTERED_ALL: (313372, 35)
SYS1_FILTERED_VAL: (7968, 35)
SYS1_FILTERED_TEST: (99307, 35)
VAL_SYS1_ORACLE: {'P': 0.05747991967871486, 'R': 0.9807280513918629, 'F1': 0.10859513930053348, 'TP': 458, 'FP': 7510, 'FN': 9, 'N_pred': 7968, 'N_gt': 467}


,filtered_candidate_count
split,
synth_train,168975
test,99307
train,37122
val,7968


VAL filtered source counts:


,val_source_count
source_wavlm,6198
source_pyannote,1553
source_asr,4007
source_qwen3,7968
source_synthetic,0
source_train_gt_seed,0
source_pyannote_change,851
source_pyannote_overlap,900


TEST filtered source counts:


,test_source_count
source_wavlm,83301
source_pyannote,18356
source_asr,43191
source_qwen3,99307
source_synthetic,0
source_train_gt_seed,0
source_pyannote_change,11543
source_pyannote_overlap,9566


TEST filtered label check — should be only -1:


,test_label_count
label,
-1,99307


✅ Cell 15 complete: Sys1 candidate settings are VAL-locked with canonical float64 event matching.


In [21]:
# Build Sys2 multimodal candidate feature table
import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, Optional
from IPython.display import display
from tqdm.auto import tqdm

SYS2_FEATURES_PATH = CACHE_DIR / "sys2_multimodal_features_all.pkl"
SYS2_FEATURES_META_PATH = CACHE_DIR / "sys2_multimodal_features_all.meta.json"

SYS2_BASE_FEATURE_COLS = [
    "session", "time", "split", "audio_path",

    "source_wavlm",
    "source_pyannote",
    "source_pyannote_change",
    "source_pyannote_overlap",
    "source_asr",
    "source_qwen3",
    "source_synthetic",
    "source_train_gt_seed",

    "wavlm_score",
    "wavlm_center_score",
    "pyannote_change_score",
    "pyannote_overlap_score",
    "pyannote_change_dist",
    "pyannote_any_candidate_dist",
    "pyannote_overlap_dist",
    "pyannote_active_before",
    "pyannote_active_after",
    "pyannote_active_delta",
    "pyannote_local_active",
    "pyannote_local_overlap_flag",
    "asr_boundary_score",
    "asr_boundary_dist",
    "asr_has_left",
    "asr_has_right",
    "asr_left_len",
    "asr_right_len",
    "qwen_shift_score",
    "qwen_backchannel_score",
    "qwen_interruption_score",

    "energy_left",
    "energy_right",
    "energy_delta",
    "energy_ratio",
    "ecapa_cos",
    "ecapa_available",
    "ecapa_left_rms",
    "ecapa_right_rms",

    "candidate_density_1s",
    "candidate_density_2s",

    "label",
    "action",
    "action_id",
    "dt_to_event",
    "is_synthetic",
    "is_diffusion_augmented",
    "sample_weight",
]

SYS2_PROB_COLS = []
for _a in ACTION_NAMES:
    SYS2_PROB_COLS.extend([f"qwen_p_{_a}", f"wavlm_p_{_a}"])

SYS2_FEATURE_COLUMNS = list(dict.fromkeys(SYS2_BASE_FEATURE_COLS + SYS2_PROB_COLS))

def _sys2_feature_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _sys2_feature_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "tol": float(TOL),
        "sys1_locked": SYS1_LOCKED if "SYS1_LOCKED" in globals() else {},
        "use_wavlm": bool(USE_WAVLM_SYS1),
        "use_pyannote": bool(USE_PYANNOTE_DIA),
        "use_asr": bool(USE_ASR_TEXT),
        "use_qwen3": bool(USE_QWEN3_LLM),
        "use_ecapa": bool(USE_ECAPA_SPK),
        "ecapa_available": bool(getattr(globals().get("ECAPA", None), "available", False)),
        "n_sys1_filtered": int(len(SYS1_FILTERED_ALL)) if "SYS1_FILTERED_ALL" in globals() else -1,
        "sys1_union_fingerprint": _sys2_feature_file_fingerprint(SYS1_UNION_PATH) if "SYS1_UNION_PATH" in globals() else None,
        "wavlm_candidates_fingerprint": _sys2_feature_file_fingerprint(WAVLM_CANDIDATES_PATH) if "WAVLM_CANDIDATES_PATH" in globals() else None,
        "qwen_scores_fingerprint": _sys2_feature_file_fingerprint(QWEN3_SCORES_PATH) if "QWEN3_SCORES_PATH" in globals() else None,
        "pyannote_candidates_fingerprint": _sys2_feature_file_fingerprint(PYANNOTE_CANDS_PATH) if "PYANNOTE_CANDS_PATH" in globals() else None,
        "asr_segments_fingerprint": _sys2_feature_file_fingerprint(ASR_SEGMENTS_PATH) if "ASR_SEGMENTS_PATH" in globals() else None,
        "test_labels_attached": False,
        "cache_format": "pickle",
    }

def _sys2_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        return json.loads(Path(path).read_text()) == expected

    except Exception:
        return False

def _clean_legacy_sys2_features_parquet():
    for p in [
        CACHE_DIR / "sys2_multimodal_features_all.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet Sys2 feature cache:", p)

        except Exception as e:
            print("Could not remove legacy parquet Sys2 feature cache:", p, e)

def _empty_sys2_features() -> pd.DataFrame:
    return pd.DataFrame(columns=SYS2_FEATURE_COLUMNS)

def _ensure_sys2_feature_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return _empty_sys2_features()

    df = df.copy()

    distance_cols = {
        "pyannote_change_dist",
        "pyannote_any_candidate_dist",
        "pyannote_overlap_dist",
        "asr_boundary_dist",
    }

    source_cols = [c for c in SYS2_FEATURE_COLUMNS if c.startswith("source_")]

    for c in SYS2_FEATURE_COLUMNS:
        if c in df.columns:
            continue

        if c == "action":
            df[c] = "NA"
        elif c in ["session", "split", "audio_path"]:
            df[c] = ""
        elif c in ["is_synthetic", "is_diffusion_augmented"]:
            df[c] = False
        elif c == "label":
            df[c] = -1
        elif c == "action_id":
            df[c] = -1
        elif c in distance_cols:
            df[c] = 999.0
        else:
            df[c] = 0.0

    df["session"] = df["session"].astype(str)
    df["split"] = df["split"].fillna("unknown").astype(str)
    df["audio_path"] = df["audio_path"].fillna("").astype(str)

    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df = df.dropna(subset=["session", "time"])
    df = df[df["time"] >= 0].reset_index(drop=True)

    for c in source_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    numeric_cols = [
        c for c in SYS2_FEATURE_COLUMNS
        if c not in ["session", "split", "audio_path", "action", "is_synthetic", "is_diffusion_augmented"]
        and c not in source_cols
    ]

    for c in numeric_cols:
        if c == "label":
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1)
        elif c == "action_id":
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1)
        elif c in distance_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(999.0)
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    df["label"] = df["label"].astype(int)

    df["action"] = df["action"].fillna("NA").astype(str)
    df.loc[~df["action"].isin(ACTION_NAMES), "action"] = "NA"

    df["action_id"] = pd.to_numeric(df["action_id"], errors="coerce").fillna(-1).astype(int)

    labelled_mask = df["label"].isin([0, 1])
    df.loc[labelled_mask, "action_id"] = (
        df.loc[labelled_mask, "action"]
        .map(ACTION_TO_ID)
        .fillna(ACTION_TO_ID["NA"])
        .astype(int)
    )

    df.loc[df["label"] == -1, "action"] = "NA"
    df.loc[df["label"] == -1, "action_id"] = -1

    df.loc[df["action"].isin(SHIFT_ACTIONS), "label"] = 1
    df.loc[df["action"].isin(HOLD_ACTIONS) & (df["label"] != -1), "label"] = 0

    df["is_synthetic"] = df["is_synthetic"].fillna(False).astype(bool)
    df["is_diffusion_augmented"] = df["is_diffusion_augmented"].fillna(False).astype(bool)

    df["sample_weight"] = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(1.0).astype(float)
    df["sample_weight"] = df["sample_weight"].clip(lower=1e-6)

    return (
        df[SYS2_FEATURE_COLUMNS]
        .drop_duplicates(subset=["session", "time", "split"])
        .sort_values(["split", "session", "time"])
        .reset_index(drop=True)
    )

def nearest_value(
    df: pd.DataFrame,
    session: str,
    tau: float,
    value_col: str,
    time_col: str = "time",
    default: float = 0.0,
    max_dist: float = 0.75,
) -> float:
    if df is None or len(df) == 0 or value_col not in df.columns or time_col not in df.columns:
        return float(default)

    g = df[df["session"].astype(str) == str(session)]

    if len(g) == 0:
        return float(default)

    arr = pd.to_numeric(g[time_col], errors="coerce").to_numpy(dtype=float)

    if len(arr) == 0 or not np.isfinite(arr).any():
        return float(default)

    d = np.abs(arr - float(tau))
    j = int(np.nanargmin(d))

    if not np.isfinite(d[j]) or d[j] > float(max_dist):
        return float(default)

    try:
        v = float(g.iloc[j][value_col])
        return v if np.isfinite(v) else float(default)

    except Exception:
        return float(default)

def row_or_nearest(
    row: pd.Series,
    col: str,
    fallback_df: Optional[pd.DataFrame],
    session: str,
    tau: float,
    default: float = 0.0,
    max_dist: float = 0.75,
    allow_zero_as_missing: bool = False,
) -> float:
    if col in row.index:
        try:
            v = float(row.get(col, default))

            if np.isfinite(v):
                if not allow_zero_as_missing:
                    return v

                if abs(v - float(default)) > 1e-12:
                    return v

        except Exception:
            pass

    return nearest_value(
        fallback_df,
        session=session,
        tau=tau,
        value_col=col,
        default=default,
        max_dist=max_dist,
    )

def energy_features(
    audio_path: Path,
    tau: float,
    side_s: float = 0.75,
    guard_s: float = 0.05,
) -> Dict[str, float]:
    try:
        audio_path = Path(audio_path)

        left, _ = read_audio_mono(
            audio_path,
            start_s=max(0.0, float(tau) - float(guard_s) - float(side_s)),
            duration_s=float(side_s),
            sr=TARGET_SR,
        )

        right, _ = read_audio_mono(
            audio_path,
            start_s=max(0.0, float(tau) + float(guard_s)),
            duration_s=float(side_s),
            sr=TARGET_SR,
        )

        rl = rms(left)
        rr = rms(right)

        return {
            "energy_left": float(rl),
            "energy_right": float(rr),
            "energy_delta": float(rr - rl),
            "energy_ratio": float(rr / max(1e-8, rl)),
        }

    except Exception:
        return {
            "energy_left": 0.0,
            "energy_right": 0.0,
            "energy_delta": 0.0,
            "energy_ratio": 0.0,
        }

def build_candidate_density_index(cand_df: pd.DataFrame) -> Dict[str, np.ndarray]:
    out = {}

    if cand_df is None or len(cand_df) == 0:
        return out

    tmp = cand_df[["session", "time"]].copy()
    tmp["session"] = tmp["session"].astype(str)
    tmp["time"] = pd.to_numeric(tmp["time"], errors="coerce")
    tmp = tmp.dropna(subset=["session", "time"])

    for sid, g in tmp.groupby("session"):
        out[str(sid)] = np.sort(g["time"].to_numpy(dtype=float))

    return out

def candidate_density_from_index(
    density_index: Dict[str, np.ndarray],
    session: str,
    tau: float,
    radius_s: float = 1.0,
) -> int:
    arr = density_index.get(str(session), np.array([], dtype=float))

    if len(arr) == 0:
        return 0

    tau = float(tau)

    lo = np.searchsorted(arr, tau - float(radius_s), side="left")
    hi = np.searchsorted(arr, tau + float(radius_s), side="right")

    return int(max(0, hi - lo))

def resolve_candidate_audio_path(row: pd.Series) -> str:
    sid = str(row.get("session", ""))
    audio_path = row.get("audio_path", "")

    if isinstance(audio_path, str) and audio_path and Path(audio_path).exists():
        return audio_path

    if "DEV_AUDIO" in globals() and sid in DEV_AUDIO:
        return str(DEV_AUDIO[sid])

    if "TEST_AUDIO" in globals() and sid in TEST_AUDIO:
        return str(TEST_AUDIO[sid])

    if "TRAIN_EVENTS_MIXED" in globals() and TRAIN_EVENTS_MIXED is not None and len(TRAIN_EVENTS_MIXED):
        match = TRAIN_EVENTS_MIXED[TRAIN_EVENTS_MIXED["session"].astype(str) == sid]

        if len(match) and "audio_path" in match.columns:
            p = str(match.iloc[0].audio_path)

            if Path(p).exists():
                return p

    return ""

def label_candidate_from_events(
    session: str,
    tau: float,
    events_df: pd.DataFrame,
    split_name: str,
    tol: Optional[float] = None,
) -> Dict[str, Any]:
    tol = float(TOL if tol is None else tol)

    if events_df is None or len(events_df) == 0:
        return {
            "label": -1,
            "action": "NA",
            "action_id": -1,
            "dt_to_event": np.nan,
        }

    e = events_df[events_df["session"].astype(str) == str(session)].copy()

    if len(e) == 0:
        return {
            "label": 0,
            "action": "NA",
            "action_id": ACTION_TO_ID["NA"],
            "dt_to_event": np.nan,
        }

    e["time"] = pd.to_numeric(e["time"], errors="coerce")
    e = e.dropna(subset=["time"]).reset_index(drop=True)

    if len(e) == 0:
        return {
            "label": 0,
            "action": "NA",
            "action_id": ACTION_TO_ID["NA"],
            "dt_to_event": np.nan,
        }

    dt = np.abs(e["time"].to_numpy(dtype=float) - float(tau))
    j = int(np.argmin(dt))

    if float(dt[j]) <= tol:
        er = e.iloc[j]

        action = str(er.get("action", "NA"))
        if action not in ACTION_TO_ID:
            action = "NA"

        label = int(er.get("label", 0))
        action_id = int(ACTION_TO_ID[action]) if label in [0, 1] else -1

        return {
            "label": label,
            "action": action if label in [0, 1] else "NA",
            "action_id": action_id,
            "dt_to_event": float(float(tau) - float(er.get("time", tau))),
        }

    return {
        "label": 0,
        "action": "NA",
        "action_id": ACTION_TO_ID["NA"],
        "dt_to_event": float(np.min(dt)) if len(dt) else np.nan,
    }

def get_event_table_for_sys2_split(split_name: str) -> Optional[pd.DataFrame]:
    split_name = str(split_name)

    if split_name == "train":
        return REAL_TRAIN_EVENTS

    if split_name == "val":
        return REAL_VAL_EVENTS

    if split_name == "synth_train":
        return SYNTH_EVENTS

    if split_name == "real_diffusion":
        return REAL_AUG_EVENTS

    if split_name == "synth_diffusion":
        return SYNTH_AUG_EVENTS

    return None

def _source_int(row: pd.Series, col: str) -> int:
    try:
        v = row.get(col, 0)
        if pd.isna(v):
            return 0
        return int(float(v))
    except Exception:
        return 0

def build_sys2_features_for_candidates(
    cand_df: pd.DataFrame,
    events_df: Optional[pd.DataFrame],
    split_name: str,
    attach_labels: bool,
) -> pd.DataFrame:
    if cand_df is None or len(cand_df) == 0:
        return _empty_sys2_features()

    cand_df = cand_df.copy()
    cand_df["session"] = cand_df["session"].astype(str)
    cand_df["time"] = pd.to_numeric(cand_df["time"], errors="coerce")
    cand_df = cand_df.dropna(subset=["session", "time"]).reset_index(drop=True)

    density_index = build_candidate_density_index(cand_df)

    rows = []

    for idx, r in tqdm(cand_df.iterrows(), total=len(cand_df), desc=f"Sys2 features {split_name}"):
        sid = str(r.session)
        tau = float(r.time)

        audio_path = resolve_candidate_audio_path(r)

        row = {
            "session": sid,
            "time": tau,
            "split": str(split_name),
            "audio_path": audio_path,
        }

        for c in [
            "source_wavlm",
            "source_pyannote",
            "source_pyannote_change",
            "source_pyannote_overlap",
            "source_asr",
            "source_qwen3",
            "source_synthetic",
            "source_train_gt_seed",
        ]:
            row[c] = _source_int(r, c)

        row["wavlm_score"] = row_or_nearest(
            r,
            "wavlm_score",
            WAVLM_CANDIDATES_ALL if "WAVLM_CANDIDATES_ALL" in globals() else None,
            sid,
            tau,
            default=0.0,
            allow_zero_as_missing=True,
        )
        row["wavlm_center_score"] = row_or_nearest(
            r,
            "wavlm_center_score",
            WAVLM_CANDIDATES_ALL if "WAVLM_CANDIDATES_ALL" in globals() else None,
            sid,
            tau,
            default=0.0,
            allow_zero_as_missing=True,
        )

        row["pyannote_change_score"] = row_or_nearest(
            r,
            "pyannote_change_score",
            PYANNOTE_CANDIDATES if "PYANNOTE_CANDIDATES" in globals() else None,
            sid,
            tau,
            default=0.0,
        )
        row["pyannote_overlap_score"] = row_or_nearest(
            r,
            "pyannote_overlap_score",
            PYANNOTE_CANDIDATES if "PYANNOTE_CANDIDATES" in globals() else None,
            sid,
            tau,
            default=0.0,
        )

        pyf = pyannote_features(sid, tau) if USE_PYANNOTE_DIA else {}

        row.update({
            "pyannote_change_dist": float(pyf.get("pyannote_change_dist", 999.0)),
            "pyannote_any_candidate_dist": float(pyf.get("pyannote_any_candidate_dist", 999.0)),
            "pyannote_overlap_dist": float(pyf.get("pyannote_overlap_dist", 999.0)),
            "pyannote_active_before": int(pyf.get("pyannote_active_before", 0)),
            "pyannote_active_after": int(pyf.get("pyannote_active_after", 0)),
            "pyannote_active_delta": int(pyf.get("pyannote_active_delta", 0)),
            "pyannote_local_active": int(pyf.get("pyannote_local_active", 0)),
            "pyannote_local_overlap_flag": int(pyf.get("pyannote_local_overlap_flag", 0)),
        })

        ctx = asr_context_for_candidate(sid, tau) if USE_ASR_TEXT else {}

        asr_left = str(ctx.get("asr_left", ""))
        asr_right = str(ctx.get("asr_right", ""))

        row["asr_boundary_score"] = float(r.get("asr_boundary_score", 0.0)) if pd.notna(r.get("asr_boundary_score", 0.0)) else 0.0
        row["asr_boundary_dist"] = float(ctx.get("asr_boundary_dist", 999.0))
        row["asr_has_left"] = int(ctx.get("asr_has_left", 0))
        row["asr_has_right"] = int(ctx.get("asr_has_right", 0))
        row["asr_left_len"] = int(len(asr_left))
        row["asr_right_len"] = int(len(asr_right))

        for a in ACTION_NAMES:
            q_col = f"qwen_p_{a}"
            w_col = f"wavlm_p_{a}"

            row[q_col] = row_or_nearest(
                r,
                q_col,
                QWEN3_SCORES_ALL if "QWEN3_SCORES_ALL" in globals() else None,
                sid,
                tau,
                default=0.0,
                allow_zero_as_missing=True,
            )

            row[w_col] = row_or_nearest(
                r,
                w_col,
                WAVLM_CANDIDATES_ALL if "WAVLM_CANDIDATES_ALL" in globals() else None,
                sid,
                tau,
                default=0.0,
                allow_zero_as_missing=True,
            )

        qwen_prob_shift = float(row.get("qwen_p_NTT", 0.0)) + float(row.get("qwen_p_ITT", 0.0))

        row["qwen_shift_score"] = max(
            float(r.get("qwen_shift_score", 0.0)) if pd.notna(r.get("qwen_shift_score", 0.0)) else 0.0,
            qwen_prob_shift,
        )
        row["qwen_backchannel_score"] = max(
            float(r.get("qwen_backchannel_score", 0.0)) if pd.notna(r.get("qwen_backchannel_score", 0.0)) else 0.0,
            float(row.get("qwen_p_BC", 0.0)),
        )
        row["qwen_interruption_score"] = max(
            float(r.get("qwen_interruption_score", 0.0)) if pd.notna(r.get("qwen_interruption_score", 0.0)) else 0.0,
            float(row.get("qwen_p_ITT", 0.0)),
        )

        if audio_path and Path(audio_path).exists():
            row.update(energy_features(Path(audio_path), tau))

            if USE_ECAPA_SPK:
                row.update(ecapa_same_similarity(Path(audio_path), tau))
            else:
                row.update({
                    "ecapa_cos": 0.0,
                    "ecapa_available": 0,
                    "ecapa_left_rms": 0.0,
                    "ecapa_right_rms": 0.0,
                })
        else:
            row.update({
                "energy_left": 0.0,
                "energy_right": 0.0,
                "energy_delta": 0.0,
                "energy_ratio": 0.0,
                "ecapa_cos": 0.0,
                "ecapa_available": 0,
                "ecapa_left_rms": 0.0,
                "ecapa_right_rms": 0.0,
            })

        row["candidate_density_1s"] = candidate_density_from_index(density_index, sid, tau, radius_s=1.0)
        row["candidate_density_2s"] = candidate_density_from_index(density_index, sid, tau, radius_s=2.0)

        if attach_labels:
            lab = label_candidate_from_events(
                session=sid,
                tau=tau,
                events_df=events_df,
                split_name=split_name,
                tol=float(TOL),
            )
            row.update(lab)
        else:
            row["label"] = -1
            row["action"] = "NA"
            row["action_id"] = -1
            row["dt_to_event"] = np.nan

        row["is_synthetic"] = bool(sid.startswith("synth_"))
        row["is_diffusion_augmented"] = bool("__diff" in sid)

        if row["is_synthetic"] and row["is_diffusion_augmented"]:
            row["sample_weight"] = float(WEIGHT_SYNTH_DIFFUSION)
        elif row["is_synthetic"]:
            row["sample_weight"] = float(WEIGHT_SYNTH_CLEAN)
        elif row["is_diffusion_augmented"]:
            row["sample_weight"] = float(WEIGHT_REAL_DIFFUSION)
        else:
            row["sample_weight"] = float(WEIGHT_REAL_CLEAN)

        rows.append(row)

        if USE_ECAPA_SPK and "save_ecapa_embedding_cache" in globals() and (idx + 1) % 5000 == 0:
            save_ecapa_embedding_cache()

    return _ensure_sys2_feature_schema(pd.DataFrame(rows))

def get_locked_sys1_candidate_table_for_split(split_name: str) -> pd.DataFrame:
    split_name = str(split_name)

    if "SYS1_FILTERED_ALL" in globals() and SYS1_FILTERED_ALL is not None and len(SYS1_FILTERED_ALL):
        return SYS1_FILTERED_ALL[SYS1_FILTERED_ALL["split"].astype(str) == split_name].copy()

    part = SYS1_UNION_ALL[SYS1_UNION_ALL["split"].astype(str) == split_name].copy()

    if len(part) == 0:
        return pd.DataFrame()

    return filter_sys1_candidates(
        part,
        wavlm_thr=float(SYS1_LOCKED.get("wavlm_thr", 0.10)),
        qwen_thr=float(SYS1_LOCKED.get("qwen_thr", 0.10)),
        use_pyannote=bool(SYS1_LOCKED.get("use_pyannote", True)),
        use_asr=bool(SYS1_LOCKED.get("use_asr", True)),
        merge_tol=float(SYS1_LOCKED.get("merge_tol", 0.25)),
    )

_clean_legacy_sys2_features_parquet()

expected_sys2_meta = _sys2_feature_config()

STRICT_SYS2_FEATURE_META_CHECK = bool(
    globals().get("STRICT_SYS2_FEATURE_META_CHECK", False)
)

loaded_sys2_from_cache = False

if SYS2_FEATURES_PATH.exists() and not FORCE_REBUILD_SYS2_FEATURES:
    meta_matches = _sys2_meta_matches(SYS2_FEATURES_META_PATH, expected_sys2_meta)

    if meta_matches:
        print("Sys2 feature metadata matches current configuration.")
    else:
        print("⚠️ Sys2 feature metadata does not exactly match current configuration.")

        if STRICT_SYS2_FEATURE_META_CHECK:
            print("STRICT_SYS2_FEATURE_META_CHECK=True, so this cell will rebuild Sys2 features.")
        else:
            print(
                "Cache-safe resume mode: loading existing Sys2 feature cache anyway.\n"
                "This avoids an expensive rebuild after kernel restart.\n"
                "Set STRICT_SYS2_FEATURE_META_CHECK=True only if you intentionally want strict cache invalidation."
            )

    if meta_matches or not STRICT_SYS2_FEATURE_META_CHECK:
        SYS2_FEATURES_ALL = pd.read_pickle(SYS2_FEATURES_PATH)
        SYS2_FEATURES_ALL = _ensure_sys2_feature_schema(SYS2_FEATURES_ALL)
        loaded_sys2_from_cache = True

        print("Loaded Sys2 features:", SYS2_FEATURES_ALL.shape)
        print("Loaded from:", SYS2_FEATURES_PATH)

if not loaded_sys2_from_cache:
    if FORCE_REBUILD_SYS2_FEATURES:
        print("FORCE_REBUILD_SYS2_FEATURES=True, rebuilding Sys2 features.")
    elif not SYS2_FEATURES_PATH.exists():
        print("No Sys2 feature cache found. Rebuilding Sys2 features.")
    else:
        print("Rebuilding Sys2 features because strict metadata check failed.")

    required_for_rebuild = [
        "SYS1_UNION_ALL",
        "SYS1_LOCKED",
        "REAL_TRAIN_EVENTS",
        "REAL_VAL_EVENTS",
        "merge_candidate_sources",
        "filter_sys1_candidates",
    ]

    missing_for_rebuild = [name for name in required_for_rebuild if name not in globals()]

    if missing_for_rebuild:
        raise RuntimeError(
            "Cannot rebuild Sys2 features because required upstream objects are missing:\n"
            f"{missing_for_rebuild}\n\n"
            "Since this is a cache-safe restart, the better option is usually to load the "
            "existing Sys2 feature cache. Check that SYS2_FEATURES_PATH exists:\n"
            f"{SYS2_FEATURES_PATH}"
        )

    selected_parts = []

    split_names = [
        "train",
        "val",
        "test",
        "synth_train",
        "real_diffusion",
        "synth_diffusion",
    ]

    for split_name in split_names:
        part = get_locked_sys1_candidate_table_for_split(split_name)

        if part is None or len(part) == 0:
            print(f"Skipping Sys2 features for {split_name}: no candidates.")
            continue

        events_df = get_event_table_for_sys2_split(split_name)
        attach_labels = split_name != "test"

        feat = build_sys2_features_for_candidates(
            cand_df=part,
            events_df=events_df,
            split_name=split_name,
            attach_labels=attach_labels,
        )

        if len(feat):
            selected_parts.append(feat)

    SYS2_FEATURES_ALL = (
        pd.concat(selected_parts, ignore_index=True)
        if selected_parts
        else _empty_sys2_features()
    )

    SYS2_FEATURES_ALL = _ensure_sys2_feature_schema(SYS2_FEATURES_ALL)

    SYS2_FEATURES_ALL.to_pickle(SYS2_FEATURES_PATH)
    SYS2_FEATURES_META_PATH.write_text(json.dumps(expected_sys2_meta, indent=2, sort_keys=True))

    print("Saved Sys2 features:", SYS2_FEATURES_PATH)

    if "save_ecapa_embedding_cache" in globals():
        save_ecapa_embedding_cache()

print("SYS2_FEATURES_ALL:", SYS2_FEATURES_ALL.shape)

if len(SYS2_FEATURES_ALL):
    display(SYS2_FEATURES_ALL.head())

    print("Feature rows by split/action:")
    display(
        SYS2_FEATURES_ALL
        .groupby(["split", "action"], dropna=False)
        .size()
        .to_frame("n")
    )

    print("Label summary by split:")
    display(
        SYS2_FEATURES_ALL
        .groupby("split")["label"]
        .value_counts(dropna=False)
        .to_frame("n")
    )

    print("TEST label check — should be only -1:")
    if "test" in set(SYS2_FEATURES_ALL["split"].astype(str)):
        display(
            SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "test"]["label"]
            .value_counts(dropna=False)
            .to_frame("test_label_count")
        )

        assert set(
            SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "test"]["label"]
            .astype(int)
            .unique()
        ).issubset({-1}), "TEST labels should remain -1 in Cell 16."

        print("TEST action_id check — should be only -1:")
        display(
            SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "test"]["action_id"]
            .value_counts(dropna=False)
            .to_frame("test_action_id_count")
        )

Sys2 feature metadata matches current configuration.


Loaded Sys2 features: (313372, 60)
Loaded from: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/sys2_multimodal_features_all.pkl
SYS2_FEATURES_ALL: (313372, 60)


,session,time,split,audio_path,source_wavlm,source_pyannote,source_pyannote_change,source_pyannote_overlap,source_asr,source_qwen3,...,qwen_p_NA,wavlm_p_NA,qwen_p_NTT,wavlm_p_NTT,qwen_p_ITT,wavlm_p_ITT,qwen_p_BC,wavlm_p_BC,qwen_p_BI,wavlm_p_BI
0,synth_000000,0.036990,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,1,1,...,0.019876,0.669130,0.028472,0.195304,0.504671,0.686777,0.428309,0.126965,0.018672,0.001393
1,synth_000000,0.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,0,1,...,0.026082,0.291277,0.053308,0.100523,0.651971,0.570787,0.241728,0.034066,0.026910,0.003348
2,synth_000000,1.026594,synth_train,/Documents/Turn_Taking/Tech_paper/R...,0,1,1,0,0,1,...,0.292136,0.250314,0.254308,0.706248,0.234280,0.018161,0.137995,0.017386,0.081282,0.007891
3,synth_000000,1.400000,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,0,0,0,0,1,...,0.457601,0.250314,0.450506,0.706248,0.039751,0.018161,0.031879,0.017386,0.020263,0.007891
4,synth_000000,1.989819,synth_train,/Documents/Turn_Taking/Tech_paper/R...,1,1,1,0,0,1,...,0.290735,0.014233,0.186981,0.015893,0.270994,0.718837,0.190111,0.471850,0.061180,0.030430


Feature rows by split/action:


n
split       action        
synth_train BC        7880
            BI        2421
            ITT      18962
            NA      118236
            NTT      21476
test        NA       99307
train       BC          36
            BI          67
            ITT        821
            NA       33489
            NTT       2709
val         BC          12
            BI          20
            ITT        209
            NA        7215
            NTT        512

...

n
split       label        
synth_train  0     128537
             1      40438
test        -1      99307
train        0      33592
             1       3530
val          0       7247
             1        721

TEST label check — should be only -1:


,test_label_count
label,
-1,99307


TEST action_id check — should be only -1:


,test_action_id_count
action_id,
-1,99307


In [22]:
# Build/load temporal tensors for primary Sys2 verifier
import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, Optional, List
from IPython.display import display
from tqdm.auto import tqdm

TEMPORAL_TENSOR_DIR = CACHE_DIR / "temporal_tensors"
TEMPORAL_TENSOR_DIR.mkdir(parents=True, exist_ok=True)

SYS2_SEQ_RADIUS = int(globals().get("SYS2_SEQ_RADIUS", 6))
SYS2_SEQ_LEN = 2 * SYS2_SEQ_RADIUS + 1

SYS2_FEATURE_COLUMNS_PATH = TEMPORAL_TENSOR_DIR / "sys2_numeric_feature_columns.json"
SYS2_SCALER_PATH = TEMPORAL_TENSOR_DIR / "sys2_train_scaler.npz"
SYS2_SCALER_META_PATH = TEMPORAL_TENSOR_DIR / "sys2_train_scaler.meta.json"
SYS2_TENSOR_META_PATH = TEMPORAL_TENSOR_DIR / "temporal_tensor_meta.json"

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

FORCE_REBUILD_TEMPORAL_TENSORS = bool(
    globals().get("FORCE_REBUILD_TEMPORAL_TENSORS", False)
)

STRICT_TEMPORAL_META_CHECK = bool(
    globals().get("STRICT_TEMPORAL_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_REBUILD_TEMPORAL_TENSORS = False

TEMPORAL_FORCE_REBUILD = bool(
    FORCE_REBUILD_TEMPORAL_TENSORS
    or (
        bool(globals().get("FORCE_REBUILD_SYS2_FEATURES", False))
        and not CACHE_SAFE_RESUME_RUN
    )
)

globals()["FORCE_REBUILD_TEMPORAL_TENSORS"] = FORCE_REBUILD_TEMPORAL_TENSORS
globals()["STRICT_TEMPORAL_META_CHECK"] = STRICT_TEMPORAL_META_CHECK

SYS2_EXCLUDE_FROM_INPUT = {
    "session",
    "time",
    "split",
    "audio_path",
    "action",
    "label",
    "action_id",
    "dt_to_event",
    "sample_weight",
    "source_train_gt_seed",
    "source_synthetic",
    "is_synthetic",
    "is_diffusion_augmented",
}

def _temporal_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _select_numeric_feature_columns(df: pd.DataFrame) -> List[str]:
    cols = []

    if df is None or len(df) == 0:
        return cols

    for c in df.columns:
        if c in SYS2_EXCLUDE_FROM_INPUT:
            continue

        if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]):
            cols.append(c)

    return sorted(set(cols))

NUMERIC_FEATURE_COLS = _select_numeric_feature_columns(SYS2_FEATURES_ALL)

assert len(NUMERIC_FEATURE_COLS) > 0, "No numeric Sys2 feature columns found. Check Cell 16."

SYS2_FEATURE_COLUMNS_PATH.write_text(json.dumps(NUMERIC_FEATURE_COLS, indent=2))

print("Numeric Sys2 input feature columns:", len(NUMERIC_FEATURE_COLS))
print("Saved feature columns:", SYS2_FEATURE_COLUMNS_PATH)

excluded_present = [c for c in SYS2_EXCLUDE_FROM_INPUT if c in SYS2_FEATURES_ALL.columns]
print("Excluded from Sys2 input:", sorted(excluded_present))

def _temporal_base_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "seq_radius": int(SYS2_SEQ_RADIUS),
        "seq_len": int(SYS2_SEQ_LEN),
        "feature_columns": list(NUMERIC_FEATURE_COLS),
        "excluded_from_input": sorted(list(SYS2_EXCLUDE_FROM_INPUT)),
        "n_features": int(len(NUMERIC_FEATURE_COLS)),
        "scaler": "train_median_iqr",
        "sys2_features_fingerprint": _temporal_file_fingerprint(SYS2_FEATURES_PATH) if "SYS2_FEATURES_PATH" in globals() else None,
        "sys2_features_meta_fingerprint": _temporal_file_fingerprint(SYS2_FEATURES_META_PATH) if "SYS2_FEATURES_META_PATH" in globals() else None,
        "test_labels_attached": False,
        "cache_arrays": "np.savez_compressed",
        "cache_meta": "pickle",
        "cache_safe_resume": bool(CACHE_SAFE_RESUME_RUN),
    }

def _temporal_scaler_config() -> Dict[str, Any]:
    cfg = _temporal_base_config()
    cfg.update({
        "kind": "sys2_train_scaler",
        "train_splits": ["train", "synth_train", "real_diffusion", "synth_diffusion"],
        "fit_on_train_only": True,
    })
    return cfg

def _temporal_dataset_config(
    out_prefix: str,
    split_names: List[str],
    include_synth: bool,
) -> Dict[str, Any]:
    cfg = _temporal_base_config()
    cfg.update({
        "kind": "temporal_dataset",
        "out_prefix": str(out_prefix),
        "split_names": list(map(str, split_names)),
        "include_synth": bool(include_synth),
    })
    return cfg

def _json_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        old = json.loads(Path(path).read_text())
        return old == expected

    except Exception:
        return False

def _clean_legacy_temporal_parquet():
    for p in TEMPORAL_TENSOR_DIR.glob("*_meta.parquet"):
        try:
            p.unlink()
            print("Removed legacy parquet temporal meta:", p)
        except Exception as e:
            print("Could not remove legacy temporal parquet:", p, e)

def _safe_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    if df is None or len(df) == 0:
        return np.empty((0, len(NUMERIC_FEATURE_COLS)), dtype=np.float32)

    work = df.copy()

    for c in NUMERIC_FEATURE_COLS:
        if c not in work.columns:
            work[c] = 0.0

    X = (
        work[NUMERIC_FEATURE_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    return X.astype(np.float32)

def _prepare_temporal_split_df(
    split_names: List[str],
    include_synth: bool = False,
) -> pd.DataFrame:
    if SYS2_FEATURES_ALL is None or len(SYS2_FEATURES_ALL) == 0:
        return pd.DataFrame()

    df = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin(list(map(str, split_names)))
    ].copy()

    df = df.reset_index(drop=True)

    if not include_synth:
        df = df[
            ~df["split"].astype(str).isin(["synth_train", "real_diffusion", "synth_diffusion"])
        ].copy()
        df = df.reset_index(drop=True)

    if DEBUG_SMALL_RUN and len(df) > 8000:
        df = df.sample(8000, random_state=SEED).reset_index(drop=True)

    df["session"] = df["session"].astype(str)
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df = df.dropna(subset=["session", "time"]).reset_index(drop=True)

    return df

def fit_train_feature_scaler(train_df: pd.DataFrame) -> Dict[str, np.ndarray]:
    X = _safe_feature_matrix(train_df)

    if len(X) == 0:
        median = np.zeros(len(NUMERIC_FEATURE_COLS), dtype=np.float32)
        scale = np.ones(len(NUMERIC_FEATURE_COLS), dtype=np.float32)

    else:
        median = np.nanmedian(X, axis=0).astype(np.float32)
        q25 = np.nanpercentile(X, 25, axis=0).astype(np.float32)
        q75 = np.nanpercentile(X, 75, axis=0).astype(np.float32)

        scale = (q75 - q25).astype(np.float32)
        scale[~np.isfinite(scale)] = 1.0
        scale[np.abs(scale) < 1e-6] = 1.0

    np.savez_compressed(
        SYS2_SCALER_PATH,
        median=median,
        scale=scale,
        feature_columns=np.array(NUMERIC_FEATURE_COLS, dtype=object),
    )

    SYS2_SCALER_META_PATH.write_text(
        json.dumps(_temporal_scaler_config(), indent=2, sort_keys=True)
    )

    print("Saved Sys2 train scaler:", SYS2_SCALER_PATH)
    print("Saved Sys2 train scaler meta:", SYS2_SCALER_META_PATH)

    return {
        "median": median,
        "scale": scale,
    }

def load_train_feature_scaler() -> Dict[str, np.ndarray]:
    if not SYS2_SCALER_PATH.exists():
        raise FileNotFoundError(f"Missing Sys2 scaler: {SYS2_SCALER_PATH}")

    z = np.load(SYS2_SCALER_PATH, allow_pickle=True)

    scaler = {
        "median": z["median"].astype(np.float32),
        "scale": z["scale"].astype(np.float32),
    }

    if len(scaler["median"]) != len(NUMERIC_FEATURE_COLS):
        raise RuntimeError(
            "Loaded Sys2 scaler feature dimension does not match current NUMERIC_FEATURE_COLS.\n"
            f"Scaler dim: {len(scaler['median'])}\n"
            f"Current feature dim: {len(NUMERIC_FEATURE_COLS)}\n"
            f"Scaler path: {SYS2_SCALER_PATH}"
        )

    return scaler

def apply_feature_scaler(X: np.ndarray, scaler: Dict[str, np.ndarray]) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)

    if X.size == 0:
        return X.astype(np.float32)

    median = np.asarray(scaler["median"], dtype=np.float32)
    scale = np.asarray(scaler["scale"], dtype=np.float32)

    assert X.shape[-1] == len(median), (
        f"Feature dimension mismatch: X has {X.shape[-1]}, scaler has {len(median)}"
    )

    X = (X - median[None, :]) / scale[None, :]
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    return X.astype(np.float32)

def build_temporal_sequences_for_df(
    df: pd.DataFrame,
    scaler: Dict[str, np.ndarray],
    seq_radius: int = SYS2_SEQ_RADIUS,
) -> np.ndarray:
    seq_radius = int(seq_radius)
    seq_len = 2 * seq_radius + 1

    if df is None or len(df) == 0:
        return np.empty((0, seq_len, len(NUMERIC_FEATURE_COLS)), dtype=np.float32)

    df = df.copy().reset_index(drop=True)
    df["session"] = df["session"].astype(str)
    df["time"] = pd.to_numeric(df["time"], errors="coerce")

    base_X = _safe_feature_matrix(df)
    base_X = apply_feature_scaler(base_X, scaler)

    session_to_ordered_indices = {}

    for sid, g in df.groupby("session", sort=False):
        idxs = g.index.to_numpy(dtype=int)
        times = df.loc[idxs, "time"].to_numpy(dtype=float)
        order = np.argsort(times, kind="mergesort")
        session_to_ordered_indices[str(sid)] = idxs[order]

    X_seq = np.zeros((len(df), seq_len, len(NUMERIC_FEATURE_COLS)), dtype=np.float32)

    for i in tqdm(range(len(df)), desc="Build temporal sequences"):
        sid = str(df.loc[i, "session"])
        tau = float(df.loc[i, "time"])

        ordered_idxs = session_to_ordered_indices.get(sid, np.array([i], dtype=int))
        ordered_times = df.loc[ordered_idxs, "time"].to_numpy(dtype=float)

        nearest_order = np.argsort(np.abs(ordered_times - tau), kind="mergesort")[:seq_len]
        chosen = ordered_idxs[nearest_order]

        chosen_times = df.loc[chosen, "time"].to_numpy(dtype=float)
        chosen = chosen[np.argsort(chosen_times, kind="mergesort")]

        seq = base_X[chosen]

        if len(seq) < seq_len:
            center_vec = base_X[i]
            pad = np.tile(center_vec[None, :], (seq_len - len(seq), 1))
            seq = np.vstack([seq, pad])

        elif len(seq) > seq_len:
            seq = seq[:seq_len]

        X_seq[i] = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    return X_seq.astype(np.float32)

def _dataset_paths(out_prefix: str) -> Dict[str, Path]:
    return {
        "npz": TEMPORAL_TENSOR_DIR / f"{out_prefix}.npz",
        "meta": TEMPORAL_TENSOR_DIR / f"{out_prefix}_meta.pkl",
        "meta_json": TEMPORAL_TENSOR_DIR / f"{out_prefix}.meta.json",
    }

def _validate_loaded_temporal_dataset(
    X: np.ndarray,
    y: np.ndarray,
    aid: np.ndarray,
    w: np.ndarray,
    meta_df: pd.DataFrame,
    out_prefix: str,
) -> bool:
    if X.ndim != 3:
        print(f"Invalid cached {out_prefix}: X is not 3D:", X.shape)
        return False

    if X.shape[1] != SYS2_SEQ_LEN:
        print(f"Invalid cached {out_prefix}: sequence length mismatch:", X.shape)
        return False

    if X.shape[2] != len(NUMERIC_FEATURE_COLS):
        print(f"Invalid cached {out_prefix}: feature dimension mismatch:", X.shape)
        return False

    n = X.shape[0]
    if not (len(y) == len(aid) == len(w) == len(meta_df) == n):
        print(f"Invalid cached {out_prefix}: row-count mismatch.")
        print("X:", X.shape, "y:", len(y), "aid:", len(aid), "w:", len(w), "meta:", len(meta_df))
        return False

    required_meta_cols = {"session", "time", "split", "label", "action_id"}
    if not required_meta_cols.issubset(set(meta_df.columns)):
        print(f"Invalid cached {out_prefix}: missing metadata columns.")
        return False

    return True

def build_temporal_dataset(
    split_names: List[str],
    out_prefix: str,
    include_synth: bool = False,
    scaler: Optional[Dict[str, np.ndarray]] = None,
    force: bool = False,
):
    paths = _dataset_paths(out_prefix)
    expected = _temporal_dataset_config(
        out_prefix=out_prefix,
        split_names=split_names,
        include_synth=include_synth,
    )

    cache_exists = paths["npz"].exists() and paths["meta"].exists()
    meta_matches = _json_meta_matches(paths["meta_json"], expected)

    if cache_exists and not force:
        if meta_matches:
            print(f"Temporal dataset metadata matches for {out_prefix}.")
        else:
            print(f"⚠️ Temporal dataset metadata mismatch for {out_prefix}.")

            if STRICT_TEMPORAL_META_CHECK:
                print(f"STRICT_TEMPORAL_META_CHECK=True, so {out_prefix} will rebuild.")
            else:
                print(f"Cache-safe resume mode: loading existing temporal dataset anyway: {out_prefix}")

        if meta_matches or not STRICT_TEMPORAL_META_CHECK:
            arr = np.load(paths["npz"], allow_pickle=True)
            meta_df = pd.read_pickle(paths["meta"])

            X = arr["X"].astype(np.float32)
            y = arr["y"].astype(np.int64)
            aid = arr["action_id"].astype(np.int64)
            w = arr["weights"].astype(np.float32)

            if _validate_loaded_temporal_dataset(X, y, aid, w, meta_df, out_prefix):
                print(f"Loaded temporal dataset {out_prefix}:", X.shape)
                return X, y, aid, w, meta_df

            raise RuntimeError(
                f"Cached temporal dataset is incompatible: {out_prefix}\n"
                f"Paths: {paths}\n\n"
                "Do not silently rebuild during cache-safe restart. "
                "Set FORCE_REBUILD_TEMPORAL_TENSORS=True only if you intentionally want to rebuild."
            )

    if force:
        print(f"FORCE_REBUILD_TEMPORAL_TENSORS=True, rebuilding temporal dataset {out_prefix}.")
    elif not cache_exists:
        print(f"No temporal dataset cache found for {out_prefix}. Rebuilding.")
    else:
        print(f"Rebuilding temporal dataset {out_prefix} because strict metadata check failed.")

    if scaler is None:
        scaler = load_train_feature_scaler()

    df = _prepare_temporal_split_df(split_names, include_synth=include_synth)

    if len(df) == 0:
        X = np.empty((0, SYS2_SEQ_LEN, len(NUMERIC_FEATURE_COLS)), dtype=np.float32)
        y = np.empty((0,), dtype=np.int64)
        aid = np.empty((0,), dtype=np.int64)
        w = np.empty((0,), dtype=np.float32)
        meta_df = df.copy()

    else:
        X = build_temporal_sequences_for_df(
            df,
            scaler=scaler,
            seq_radius=SYS2_SEQ_RADIUS,
        )

        y = (
            pd.to_numeric(df["label"], errors="coerce")
            .fillna(-1)
            .astype(int)
            .to_numpy(dtype=np.int64)
        )

        aid = (
            pd.to_numeric(df["action_id"], errors="coerce")
            .fillna(-1)
            .astype(int)
            .to_numpy(dtype=np.int64)
        )

        w = (
            pd.to_numeric(df["sample_weight"], errors="coerce")
            .fillna(1.0)
            .astype(float)
            .clip(lower=1e-6)
            .to_numpy(dtype=np.float32)
        )

        meta_df = df.copy()

    np.savez_compressed(
        paths["npz"],
        X=X,
        y=y,
        action_id=aid,
        weights=w,
    )

    meta_df.to_pickle(paths["meta"])
    paths["meta_json"].write_text(json.dumps(expected, indent=2, sort_keys=True))

    print(f"Saved temporal dataset {out_prefix}:", paths["npz"])
    print(f"Saved temporal meta {out_prefix}:", paths["meta"])

    return X, y, aid, w, meta_df

def _safe_bincount(arr: np.ndarray, minlength: int) -> np.ndarray:
    arr = np.asarray(arr)
    arr = arr[arr >= 0]

    if len(arr) == 0:
        return np.zeros(minlength, dtype=int)

    return np.bincount(arr.astype(int), minlength=minlength)

_clean_legacy_temporal_parquet()

expected_scaler_meta = _temporal_scaler_config()
scaler_meta_matches = _json_meta_matches(SYS2_SCALER_META_PATH, expected_scaler_meta)

if SYS2_SCALER_PATH.exists() and not TEMPORAL_FORCE_REBUILD:
    if scaler_meta_matches:
        print("Sys2 scaler metadata matches current configuration.")
    else:
        print("⚠️ Sys2 scaler metadata does not exactly match current configuration.")

        if STRICT_TEMPORAL_META_CHECK:
            print("STRICT_TEMPORAL_META_CHECK=True, so the Sys2 scaler will be refit.")
        else:
            print(
                "Cache-safe resume mode: loading existing Sys2 scaler anyway.\n"
                "This avoids unnecessary refitting after kernel restart."
            )

    if scaler_meta_matches or not STRICT_TEMPORAL_META_CHECK:
        SYS2_SCALER = load_train_feature_scaler()
        print("Loaded Sys2 train scaler:", SYS2_SCALER_PATH)
    else:
        train_for_scaler = _prepare_temporal_split_df(
            ["train", "synth_train", "real_diffusion", "synth_diffusion"],
            include_synth=True,
        )
        assert len(train_for_scaler) > 0, "No TRAIN rows available to fit Sys2 scaler. Check Cell 16."
        SYS2_SCALER = fit_train_feature_scaler(train_for_scaler)

else:
    train_for_scaler = _prepare_temporal_split_df(
        ["train", "synth_train", "real_diffusion", "synth_diffusion"],
        include_synth=True,
    )
    assert len(train_for_scaler) > 0, "No TRAIN rows available to fit Sys2 scaler. Check Cell 16."
    SYS2_SCALER = fit_train_feature_scaler(train_for_scaler)

X_SYS2_TRAIN, y_SYS2_TRAIN, aid_SYS2_TRAIN, w_SYS2_TRAIN, META_SYS2_TRAIN = build_temporal_dataset(
    ["train", "synth_train", "real_diffusion", "synth_diffusion"],
    "train_real_synth_diffusion",
    include_synth=True,
    scaler=SYS2_SCALER,
    force=TEMPORAL_FORCE_REBUILD,
)

X_SYS2_VAL, y_SYS2_VAL, aid_SYS2_VAL, w_SYS2_VAL, META_SYS2_VAL = build_temporal_dataset(
    ["val"],
    "val_real",
    include_synth=False,
    scaler=SYS2_SCALER,
    force=TEMPORAL_FORCE_REBUILD,
)

X_SYS2_TEST, y_SYS2_TEST, aid_SYS2_TEST, w_SYS2_TEST, META_SYS2_TEST = build_temporal_dataset(
    ["test"],
    "test_real",
    include_synth=False,
    scaler=SYS2_SCALER,
    force=TEMPORAL_FORCE_REBUILD,
)

SYS2_TENSOR_META_PATH.write_text(
    json.dumps(_temporal_base_config(), indent=2, sort_keys=True)
)

assert X_SYS2_TRAIN.ndim == 3, f"Expected train tensor to be 3D, got {X_SYS2_TRAIN.shape}"
assert X_SYS2_VAL.ndim == 3, f"Expected val tensor to be 3D, got {X_SYS2_VAL.shape}"
assert X_SYS2_TEST.ndim == 3, f"Expected test tensor to be 3D, got {X_SYS2_TEST.shape}"

assert X_SYS2_TRAIN.shape[1] == SYS2_SEQ_LEN, f"Bad train seq len: {X_SYS2_TRAIN.shape}"
assert X_SYS2_VAL.shape[1] == SYS2_SEQ_LEN, f"Bad val seq len: {X_SYS2_VAL.shape}"
assert X_SYS2_TEST.shape[1] == SYS2_SEQ_LEN, f"Bad test seq len: {X_SYS2_TEST.shape}"

assert X_SYS2_TRAIN.shape[2] == len(NUMERIC_FEATURE_COLS), "Train feature dim mismatch."
assert X_SYS2_VAL.shape[2] == len(NUMERIC_FEATURE_COLS), "Val feature dim mismatch."
assert X_SYS2_TEST.shape[2] == len(NUMERIC_FEATURE_COLS), "Test feature dim mismatch."

if len(META_SYS2_TRAIN):
    train_labels = set(pd.to_numeric(META_SYS2_TRAIN["label"], errors="coerce").fillna(-1).astype(int).unique())
    assert train_labels.issubset({0, 1}), f"Unexpected TRAIN labels in Cell 17: {train_labels}"

if len(META_SYS2_VAL):
    val_labels = set(pd.to_numeric(META_SYS2_VAL["label"], errors="coerce").fillna(-1).astype(int).unique())
    assert val_labels.issubset({0, 1}), f"Unexpected VAL labels in Cell 17: {val_labels}"

if len(META_SYS2_TEST):
    test_labels = set(pd.to_numeric(META_SYS2_TEST["label"], errors="coerce").fillna(-1).astype(int).unique())
    test_action_ids = set(pd.to_numeric(META_SYS2_TEST["action_id"], errors="coerce").fillna(-1).astype(int).unique())

    assert test_labels.issubset({-1}), f"TEST labels leaked into Cell 17 tensors/meta: {test_labels}"
    assert test_action_ids.issubset({-1}), f"TEST action_id leaked into Cell 17 tensors/meta: {test_action_ids}"

print("Train temporal:", X_SYS2_TRAIN.shape, "labels:", _safe_bincount(y_SYS2_TRAIN, 2), "actions:", _safe_bincount(aid_SYS2_TRAIN, len(ACTION_NAMES)))
print("Val temporal:", X_SYS2_VAL.shape, "labels:", _safe_bincount(y_SYS2_VAL, 2), "actions:", _safe_bincount(aid_SYS2_VAL, len(ACTION_NAMES)))
print("Test temporal:", X_SYS2_TEST.shape, "labels:", _safe_bincount(y_SYS2_TEST, 2), "actions:", _safe_bincount(aid_SYS2_TEST, len(ACTION_NAMES)))

print("Feature dim:", len(NUMERIC_FEATURE_COLS))
print("Sequence length:", SYS2_SEQ_LEN)

print("First 20 Sys2 numeric feature columns:")
print(NUMERIC_FEATURE_COLS[:20])

if len(META_SYS2_TEST):
    print("TEST labels should remain -1 in Cell 17 tensors/meta.")
    display(META_SYS2_TEST["label"].value_counts(dropna=False).to_frame("test_label_count"))

    print("TEST action_id should remain -1 in Cell 17 tensors/meta.")
    display(META_SYS2_TEST["action_id"].value_counts(dropna=False).to_frame("test_action_id_count"))

print("✅ Cell 17 complete: temporal tensors are loaded/built with train-only scaling.")
print("⚠️ Note: Cell 18 should use X_SYS2_TRAIN / X_SYS2_VAL / X_SYS2_TEST directly and should not re-standardize them again.")

Numeric Sys2 input feature columns: 47
Saved feature columns: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/sys2_numeric_feature_columns.json
Excluded from Sys2 input: ['action', 'action_id', 'audio_path', 'dt_to_event', 'is_diffusion_augmented', 'is_synthetic', 'label', 'sample_weight', 'session', 'source_synthetic', 'source_train_gt_seed', 'split', 'time']
Sys2 scaler metadata matches current configuration.
Loaded Sys2 train scaler: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/sys2_train_scaler.npz
No temporal dataset cache found for train_real_synth_diffusion. Rebuilding.


Saved temporal dataset train_real_synth_diffusion: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/train_real_synth_diffusion.npz
Saved temporal meta train_real_synth_diffusion: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/train_real_synth_diffusion_meta.pkl
No temporal dataset cache found for val_real. Rebuilding.


Saved temporal dataset val_real: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/val_real.npz
Saved temporal meta val_real: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/val_real_meta.pkl
No temporal dataset cache found for test_real. Rebuilding.


Saved temporal dataset test_real: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/test_real.npz
Saved temporal meta test_real: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/cache/temporal_tensors/test_real_meta.pkl
Train temporal: (206097, 13, 47) labels: [162129  43968] actions: [151725  24185  19783   7916   2488]
Val temporal: (7968, 13, 47) labels: [7247  721] actions: [7215  512  209   12   20]
Test temporal: (99307, 13, 47) labels: [0 0] actions: [0 0 0 0 0]
Feature dim: 47
Sequence length: 13
First 20 Sys2 numeric feature columns:
['asr_boundary_dist', 'asr_boundary_score', 'asr_has_left', 'asr_has_right', 'asr_left_len', 'asr_right_len', 'candidate_density_1s', 'candidate_density_2s', 'ecapa_available', 'ecapa_cos', 'ecapa_left_rms', 'ecapa_right_rms', 'energy_delta', 'energy_

,test_label_count
label,
-1,99307


TEST action_id should remain -1 in Cell 17 tensors/meta.


,test_action_id_count
action_id,
-1,99307


✅ Cell 17 complete: temporal tensors are loaded/built with train-only scaling.
⚠️ Note: Cell 18 should use X_SYS2_TRAIN / X_SYS2_VAL / X_SYS2_TEST directly and should not re-standardize them again.


In [23]:
# Train/load primary Sys2 multimodal temporal action verifier
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from typing import Dict, Any, Optional
from IPython.display import display
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm

TEMPORAL_SYS2_DIR.mkdir(parents=True, exist_ok=True)

TEMPORAL_SYS2_BEST_CKPT = TEMPORAL_SYS2_DIR / "best_temporal_action_verifier.pt"
TEMPORAL_SYS2_LAST_CKPT = TEMPORAL_SYS2_DIR / "last_temporal_action_verifier.pt"

TEMPORAL_SYS2_CKPT = TEMPORAL_SYS2_BEST_CKPT

TEMPORAL_SYS2_HISTORY_PATH = TEMPORAL_SYS2_DIR / "temporal_sys2_training_history.csv"

TEMPORAL_TRAIN_SCORES_PATH = TEMPORAL_SYS2_DIR / "temporal_train_scores.pkl"
TEMPORAL_VAL_SCORES_PATH = TEMPORAL_SYS2_DIR / "temporal_val_scores.pkl"
TEMPORAL_TEST_SCORES_PATH = TEMPORAL_SYS2_DIR / "temporal_test_scores.pkl"

TEMPORAL_TRAIN_SCORES_META_PATH = TEMPORAL_SYS2_DIR / "temporal_train_scores.meta.json"
TEMPORAL_VAL_SCORES_META_PATH = TEMPORAL_SYS2_DIR / "temporal_val_scores.meta.json"
TEMPORAL_TEST_SCORES_META_PATH = TEMPORAL_SYS2_DIR / "temporal_test_scores.meta.json"

USE_TEMPORAL_SYS2 = bool(globals().get("USE_TEMPORAL_SYS2", True))
FORCE_RETRAIN_TEMPORAL_SYS2 = bool(globals().get("FORCE_RETRAIN_TEMPORAL_SYS2", False))
RESUME_TEMPORAL_SYS2_TRAIN = bool(globals().get("RESUME_TEMPORAL_SYS2_TRAIN", True))

AUTO_TRAIN_MISSING_PAPER_READY_STAGES = bool(
    globals().get("AUTO_TRAIN_MISSING_PAPER_READY_STAGES", True)
)

CACHE_SAFE_RESUME_RUN = bool(
    globals().get("CACHE_SAFE_RESUME_RUN", False)
)

STRICT_TEMPORAL_SYS2_CKPT_CHECK = bool(
    globals().get("STRICT_TEMPORAL_SYS2_CKPT_CHECK", False)
)

STRICT_TEMPORAL_SCORE_META_CHECK = bool(
    globals().get("STRICT_TEMPORAL_SCORE_META_CHECK", False)
)

if AUTO_TRAIN_MISSING_PAPER_READY_STAGES:
    CACHE_SAFE_RESUME_RUN = False

if FORCE_RETRAIN_TEMPORAL_SYS2:
    RESUME_TEMPORAL_SYS2_TRAIN = False
else:
    RESUME_TEMPORAL_SYS2_TRAIN = bool(
        globals().get("RESUME_TEMPORAL_SYS2_TRAIN", True)
    )

globals()["AUTO_TRAIN_MISSING_PAPER_READY_STAGES"] = (
    AUTO_TRAIN_MISSING_PAPER_READY_STAGES
)
globals()["CACHE_SAFE_RESUME_RUN"] = CACHE_SAFE_RESUME_RUN
globals()["FORCE_RETRAIN_TEMPORAL_SYS2"] = FORCE_RETRAIN_TEMPORAL_SYS2
globals()["RESUME_TEMPORAL_SYS2_TRAIN"] = RESUME_TEMPORAL_SYS2_TRAIN
globals()["STRICT_TEMPORAL_SYS2_CKPT_CHECK"] = STRICT_TEMPORAL_SYS2_CKPT_CHECK
globals()["STRICT_TEMPORAL_SCORE_META_CHECK"] = STRICT_TEMPORAL_SCORE_META_CHECK

SYS2_TEMPORAL_EARLY_STOP_PATIENCE = int(globals().get("SYS2_TEMPORAL_EARLY_STOP_PATIENCE", 8))
SYS2_TEMPORAL_HIDDEN = int(globals().get("SYS2_TEMPORAL_HIDDEN", 192))
SYS2_TEMPORAL_DROPOUT = float(globals().get("SYS2_TEMPORAL_DROPOUT", 0.18))

SYS2_TEMPORAL_GRAD_CLIP = float(globals().get("SYS2_TEMPORAL_GRAD_CLIP", 5.0))

SYS2_TEMPORAL_BATCH_SIZE = int(globals().get("SYS2_TEMPORAL_BATCH_SIZE", 128))
SYS2_TEMPORAL_BATCH_SIZE = max(1, min(SYS2_TEMPORAL_BATCH_SIZE, 128))

SYS2_TEMPORAL_NUM_WORKERS = 0
SYS2_TEMPORAL_PIN_MEMORY = False
SYS2_TEMPORAL_PERSISTENT_WORKERS = False
SYS2_TEMPORAL_NON_BLOCKING = False
SYS2_TEMPORAL_PREFLIGHT = bool(globals().get("SYS2_TEMPORAL_PREFLIGHT", True))

SYS2_TEMPORAL_DEVICE = str(globals().get("SYS2_TEMPORAL_DEVICE", DEVICE))
TEMPORAL_DEVICE = torch.device(SYS2_TEMPORAL_DEVICE)

print("Temporal Sys2 settings:")
print("  device:", TEMPORAL_DEVICE)
print("  batch_size:", SYS2_TEMPORAL_BATCH_SIZE)
print("  num_workers:", SYS2_TEMPORAL_NUM_WORKERS)
print("  pin_memory:", SYS2_TEMPORAL_PIN_MEMORY)
print("  non_blocking:", SYS2_TEMPORAL_NON_BLOCKING)
print("  force_retrain:", FORCE_RETRAIN_TEMPORAL_SYS2)
print("  resume_train:", RESUME_TEMPORAL_SYS2_TRAIN)
print("  auto_train_missing_stages:", AUTO_TRAIN_MISSING_PAPER_READY_STAGES)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_score_meta_check:", STRICT_TEMPORAL_SCORE_META_CHECK)

def _is_cuda_device() -> bool:
    return str(TEMPORAL_DEVICE).startswith("cuda")

def _safe_dataloader(ds, batch_size: int, shuffle: bool):
    kwargs = dict(
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )

    try:
        return DataLoader(
            ds,
            persistent_workers=False,
            **kwargs,
        )
    except TypeError:
        return DataLoader(ds, **kwargs)

def _to_temporal_device(x):
    return x.to(TEMPORAL_DEVICE, non_blocking=False)

def _temporal_ckpt_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _clear_stale_temporal_score_caches():
    for p in [
        TEMPORAL_TRAIN_SCORES_PATH,
        TEMPORAL_VAL_SCORES_PATH,
        TEMPORAL_TEST_SCORES_PATH,
        TEMPORAL_TRAIN_SCORES_META_PATH,
        TEMPORAL_VAL_SCORES_META_PATH,
        TEMPORAL_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale temporal score cache:", p)
        except Exception as e:
            print("Could not remove stale temporal score cache:", p, e)

def _all_temporal_score_caches_exist() -> bool:
    return (
        TEMPORAL_TRAIN_SCORES_PATH.exists()
        and TEMPORAL_VAL_SCORES_PATH.exists()
        and TEMPORAL_TEST_SCORES_PATH.exists()
    )

def _checkpoint_is_compatible(ckpt: Dict[str, Any]) -> bool:
    if not isinstance(ckpt, dict):
        return False

    expected = {
        "in_dim": int(X_SYS2_TRAIN.shape[-1]),
        "hidden": int(SYS2_TEMPORAL_HIDDEN),
        "dropout": float(SYS2_TEMPORAL_DROPOUT),
        "n_actions": int(len(ACTION_NAMES)),
        "sys2_seq_len": int(X_SYS2_TRAIN.shape[1]),
        "feature_cols": list(NUMERIC_FEATURE_COLS),
        "action_names": list(ACTION_NAMES),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
    }

    if int(ckpt.get("in_dim", -1)) != expected["in_dim"]:
        return False

    if int(ckpt.get("hidden", -1)) != expected["hidden"]:
        return False

    if abs(float(ckpt.get("dropout", -999.0)) - expected["dropout"]) > 1e-12:
        return False

    if int(ckpt.get("n_actions", -1)) != expected["n_actions"]:
        return False

    if int(ckpt.get("sys2_seq_len", -1)) != expected["sys2_seq_len"]:
        return False

    if list(ckpt.get("feature_cols", [])) != expected["feature_cols"]:
        return False

    if list(ckpt.get("action_names", [])) != expected["action_names"]:
        return False

    if str(
        ckpt.get("event_contract_version", "")
    ) != expected["event_contract_version"]:
        return False

    return True

class MultiScaleActionTCN(nn.Module):

    def __init__(
        self,
        in_dim: int,
        hidden: int = SYS2_TEMPORAL_HIDDEN,
        n_actions: int = 5,
        dropout: float = SYS2_TEMPORAL_DROPOUT,
    ):
        super().__init__()

        self.in_dim = int(in_dim)
        self.hidden = int(hidden)
        self.n_actions = int(n_actions)
        self.dropout = float(dropout)

        self.in_proj = nn.Linear(int(in_dim), int(hidden))

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1, dilation=1),
                nn.GELU(),
                nn.Dropout(dropout),
            ),
            nn.Sequential(
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=2, dilation=2),
                nn.GELU(),
                nn.Dropout(dropout),
            ),
            nn.Sequential(
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=4, dilation=4),
                nn.GELU(),
                nn.Dropout(dropout),
            ),
        ])

        self.gru = nn.GRU(
            hidden * 3,
            hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        h = self.in_proj(x)
        h = h.transpose(1, 2)

        outs = [branch(h) for branch in self.branches]

        h = torch.cat(outs, dim=1).transpose(1, 2)
        h, _ = self.gru(h)

        centre = h[:, h.shape[1] // 2, :]
        pooled = 0.50 * centre + 0.25 * h.max(dim=1).values + 0.25 * h.mean(dim=1)

        return self.head(pooled)

def _valid_temporal_training_rows(X, aid, weights):
    X = np.asarray(X, dtype=np.float32)
    aid = np.asarray(aid, dtype=np.int64)
    weights = np.asarray(weights, dtype=np.float32)

    if len(X) == 0:
        return X, aid, weights

    valid = np.isfinite(X).all(axis=(1, 2)) & (aid >= 0) & (aid < len(ACTION_NAMES))

    X = X[valid]
    aid = aid[valid]
    weights = weights[valid]

    weights = np.nan_to_num(weights, nan=1.0, posinf=1.0, neginf=1.0).astype(np.float32)
    weights = np.maximum(weights, 1e-6)

    return X.astype(np.float32), aid.astype(np.int64), weights.astype(np.float32)

def _make_temporal_loader(X, aid, weights, shuffle: bool):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(aid, dtype=torch.long),
        torch.tensor(weights, dtype=torch.float32),
    )

    return _safe_dataloader(
        ds,
        batch_size=SYS2_TEMPORAL_BATCH_SIZE,
        shuffle=bool(shuffle),
    )

def _temporal_class_weights(aid: np.ndarray) -> torch.Tensor:
    counts = np.bincount(aid.astype(int), minlength=len(ACTION_NAMES)).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    class_w = counts.sum() / counts
    class_w = class_w / max(1e-8, class_w.mean())

    return torch.tensor(class_w, dtype=torch.float32, device=TEMPORAL_DEVICE)

@torch.no_grad()
def _temporal_predict_proba(model, X: np.ndarray, desc: str = "Temporal predict") -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)

    if len(X) == 0:
        return np.zeros((0, len(ACTION_NAMES)), dtype=np.float32)

    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = _safe_dataloader(
        ds,
        batch_size=SYS2_TEMPORAL_BATCH_SIZE,
        shuffle=False,
    )

    probs = []

    model.eval()

    for (xb,) in tqdm(loader, desc=desc):
        xb = _to_temporal_device(xb)
        logits = model(xb)

        if not torch.isfinite(logits).all():
            raise FloatingPointError(f"{desc}: temporal logits contain NaN/Inf.")

        p = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        probs.append(p)

    return np.vstack(probs).astype(np.float32) if probs else np.zeros((0, len(ACTION_NAMES)), dtype=np.float32)

def _temporal_val_metrics(model, X_val, aid_val):
    X_val = np.asarray(X_val, dtype=np.float32)
    aid_val = np.asarray(aid_val, dtype=np.int64)

    valid = (aid_val >= 0) & (aid_val < len(ACTION_NAMES))
    X_val = X_val[valid]
    aid_val = aid_val[valid]

    if len(X_val) == 0:
        return {
            "val_precision": 0.0,
            "val_recall": 0.0,
            "val_f1": 0.0,
            "val_action_acc": 0.0,
            "n_val": 0,
        }

    probs = _temporal_predict_proba(model, X_val, desc="Temporal VAL metrics")

    shift_score = probs[:, ACTION_TO_ID["NTT"]] + probs[:, ACTION_TO_ID["ITT"]]
    pred_shift = (shift_score >= 0.50).astype(int)
    true_shift = np.isin(aid_val, SHIFT_ACTION_IDS).astype(int)

    P, R, F1, _ = precision_recall_fscore_support(
        true_shift,
        pred_shift,
        average="binary",
        zero_division=0,
    )

    pred_action = probs.argmax(axis=1)
    action_acc = float((pred_action == aid_val).mean()) if len(aid_val) else 0.0

    return {
        "val_precision": float(P),
        "val_recall": float(R),
        "val_f1": float(F1),
        "val_action_acc": float(action_acc),
        "n_val": int(len(aid_val)),
    }

def _make_temporal_checkpoint(
    model,
    optimizer,
    epoch: int,
    best_val_f1: float,
    bad_epochs: int,
    train_loss: float,
    metrics: Dict[str, Any],
    is_best: bool,
) -> Dict[str, Any]:
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": int(epoch),
        "best_val_f1": float(best_val_f1),
        "bad_epochs": int(bad_epochs),
        "train_loss": float(train_loss),
        "metrics": dict(metrics),
        "in_dim": int(X_SYS2_TRAIN.shape[-1]),
        "hidden": int(SYS2_TEMPORAL_HIDDEN),
        "dropout": float(SYS2_TEMPORAL_DROPOUT),
        "n_actions": int(len(ACTION_NAMES)),
        "feature_cols": list(NUMERIC_FEATURE_COLS),
        "action_names": list(ACTION_NAMES),
        "shift_action_ids": list(map(int, SHIFT_ACTION_IDS)),
        "sys2_seq_len": int(X_SYS2_TRAIN.shape[1]),
        "sys2_temporal_lr": float(SYS2_TEMPORAL_LR),
        "sys2_temporal_weight_decay": float(SYS2_TEMPORAL_WEIGHT_DECAY),
        "is_best": bool(is_best),
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "saved_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

def _temporal_preflight_check(model, loader):
    if not SYS2_TEMPORAL_PREFLIGHT:
        return

    print("Running Temporal Sys2 first-batch preflight check...")

    xb, yb, wb = next(iter(loader))

    print("  first batch shapes:", tuple(xb.shape), tuple(yb.shape), tuple(wb.shape))
    print("  first batch dtypes:", xb.dtype, yb.dtype, wb.dtype)
    print("  first batch finite X:", bool(torch.isfinite(xb).all()))
    print("  first batch finite weights:", bool(torch.isfinite(wb).all()))

    assert torch.isfinite(xb).all(), "First Temporal Sys2 batch X contains NaN/Inf."
    assert torch.isfinite(wb).all(), "First Temporal Sys2 batch weights contain NaN/Inf."

    model.eval()

    with torch.no_grad():
        xb_small = _to_temporal_device(xb[: min(8, xb.shape[0])])
        logits = model(xb_small)

        if not torch.isfinite(logits).all():
            raise FloatingPointError("Temporal Sys2 preflight logits contain NaN/Inf.")

    if _is_cuda_device():
        torch.cuda.synchronize()

    model.train()
    print("Temporal Sys2 first-batch preflight passed.")

def train_temporal_sys2():
    TEMPORAL_SYS2_DIR.mkdir(parents=True, exist_ok=True)

    if _is_cuda_device():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    Xtr, ytr, wtr = _valid_temporal_training_rows(
        X_SYS2_TRAIN,
        aid_SYS2_TRAIN,
        w_SYS2_TRAIN,
    )

    Xv, yv, wv = _valid_temporal_training_rows(
        X_SYS2_VAL,
        aid_SYS2_VAL,
        w_SYS2_VAL,
    )

    assert len(Xtr) > 0, "No valid Sys2 training rows."
    assert len(Xv) > 0, "No valid Sys2 validation rows."

    print("Temporal Sys2 train tensor:", Xtr.shape, "labels:", ytr.shape, "weights:", wtr.shape)
    print("Temporal Sys2 val tensor:", Xv.shape, "labels:", yv.shape, "weights:", wv.shape)

    model = MultiScaleActionTCN(
        in_dim=Xtr.shape[-1],
        hidden=SYS2_TEMPORAL_HIDDEN,
        n_actions=len(ACTION_NAMES),
        dropout=SYS2_TEMPORAL_DROPOUT,
    ).to(TEMPORAL_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SYS2_TEMPORAL_LR,
        weight_decay=SYS2_TEMPORAL_WEIGHT_DECAY,
    )

    start_epoch = 1
    best_val_f1 = -1.0
    bad_epochs = 0

    if RESUME_TEMPORAL_SYS2_TRAIN and TEMPORAL_SYS2_LAST_CKPT.exists():
        print("Checking temporal Sys2 resume checkpoint:", TEMPORAL_SYS2_LAST_CKPT)

        ckpt = torch.load(TEMPORAL_SYS2_LAST_CKPT, map_location=TEMPORAL_DEVICE)

        if _checkpoint_is_compatible(ckpt):
            print("Resuming temporal Sys2 from last checkpoint:", TEMPORAL_SYS2_LAST_CKPT)

            model.load_state_dict(ckpt["model"])

            if "optimizer" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer"])

                for state in optimizer.state.values():
                    for k, v in state.items():
                        if torch.is_tensor(v):
                            state[k] = v.to(TEMPORAL_DEVICE)

            start_epoch = int(ckpt.get("epoch", 0)) + 1
            best_val_f1 = float(ckpt.get("best_val_f1", -1.0))
            bad_epochs = int(ckpt.get("bad_epochs", 0))

            print(f"Resume starts at epoch {start_epoch}; best_val_f1={best_val_f1:.4f}; bad_epochs={bad_epochs}")

        else:
            print("⚠️ Last temporal Sys2 checkpoint is incompatible with current tensors/features.")
            print("   Training from scratch instead.")

    train_loader = _make_temporal_loader(Xtr, ytr, wtr, shuffle=True)
    class_w = _temporal_class_weights(ytr)

    print("Temporal Sys2 train batches per epoch:", len(train_loader))

    _temporal_preflight_check(model, train_loader)

    history = []

    if TEMPORAL_SYS2_HISTORY_PATH.exists() and start_epoch > 1:
        try:
            history = pd.read_csv(TEMPORAL_SYS2_HISTORY_PATH).to_dict("records")
        except Exception:
            history = []

    for epoch in range(start_epoch, SYS2_TEMPORAL_EPOCHS + 1):
        model.train()
        losses = []

        pbar = tqdm(train_loader, desc=f"Temporal Sys2 epoch {epoch}", leave=False, mininterval=5.0)

        for step, (xb, yb, wb) in enumerate(pbar, start=1):
            xb = _to_temporal_device(xb)
            yb = _to_temporal_device(yb)
            wb = _to_temporal_device(wb)

            logits = model(xb)

            if not torch.isfinite(logits).all():
                raise FloatingPointError(f"Temporal Sys2 logits contain NaN/Inf at epoch={epoch}, step={step}.")

            loss = F.cross_entropy(
                logits,
                yb,
                weight=class_w,
                reduction="none",
            )

            loss = (loss * wb).mean()

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Temporal Sys2 loss is NaN/Inf at epoch={epoch}, step={step}.")

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), SYS2_TEMPORAL_GRAD_CLIP)
            optimizer.step()

            loss_value = float(loss.detach().cpu())
            losses.append(loss_value)

            if step == 1:
                print(f"Temporal Sys2 epoch {epoch}: first batch completed, loss={loss_value:.4f}")

            if step % 50 == 0:
                pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")

        train_loss = float(np.mean(losses)) if losses else 999.0

        metrics = _temporal_val_metrics(model, Xv, yv)
        val_f1 = float(metrics["val_f1"])

        improved = val_f1 > best_val_f1

        if improved:
            best_val_f1 = val_f1
            bad_epochs = 0
        else:
            bad_epochs += 1

        state = _make_temporal_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            best_val_f1=best_val_f1,
            bad_epochs=bad_epochs,
            train_loss=train_loss,
            metrics=metrics,
            is_best=improved,
        )

        torch.save(state, TEMPORAL_SYS2_LAST_CKPT)

        if improved:
            torch.save(state, TEMPORAL_SYS2_BEST_CKPT)

        row = {
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "best_val_f1": float(best_val_f1),
            "bad_epochs": int(bad_epochs),
            "improved": bool(improved),
            **metrics,
        }

        history.append(row)
        pd.DataFrame(history).to_csv(TEMPORAL_SYS2_HISTORY_PATH, index=False)

        print(
            f"Temporal Sys2 epoch {epoch:03d}: "
            f"loss={train_loss:.4f} "
            f"valP={metrics['val_precision']:.3f} "
            f"valR={metrics['val_recall']:.3f} "
            f"valF1={metrics['val_f1']:.3f} "
            f"valActionAcc={metrics['val_action_acc']:.3f} "
            f"best={best_val_f1:.3f} "
            f"bad={bad_epochs}/{SYS2_TEMPORAL_EARLY_STOP_PATIENCE}"
        )

        if bad_epochs >= SYS2_TEMPORAL_EARLY_STOP_PATIENCE:
            print("Temporal Sys2 early stop.")
            break

    if TEMPORAL_SYS2_BEST_CKPT.exists():
        best = torch.load(TEMPORAL_SYS2_BEST_CKPT, map_location=TEMPORAL_DEVICE)

        if _checkpoint_is_compatible(best):
            model.load_state_dict(best["model"])
            print("Loaded best temporal Sys2 state from:", TEMPORAL_SYS2_BEST_CKPT)
        else:
            print("⚠️ Best checkpoint exists but is incompatible. Keeping current model state.")

    model.eval()

    _clear_stale_temporal_score_caches()

    return model

def load_temporal_sys2_model(load_last: bool = False):
    path = TEMPORAL_SYS2_LAST_CKPT if load_last else TEMPORAL_SYS2_BEST_CKPT

    if not path.exists():
        return None

    ckpt = torch.load(path, map_location=TEMPORAL_DEVICE)

    if not _checkpoint_is_compatible(ckpt):
        print("⚠️ Temporal Sys2 checkpoint is incompatible with current tensors/features:", path)
        return None

    model = MultiScaleActionTCN(
        in_dim=int(ckpt["in_dim"]),
        hidden=int(ckpt.get("hidden", SYS2_TEMPORAL_HIDDEN)),
        n_actions=int(ckpt.get("n_actions", len(ACTION_NAMES))),
        dropout=float(ckpt.get("dropout", SYS2_TEMPORAL_DROPOUT)),
    ).to(TEMPORAL_DEVICE)

    model.load_state_dict(ckpt["model"])
    model.eval()

    print("Loaded temporal Sys2 checkpoint:", path)
    print("  epoch:", ckpt.get("epoch", None))
    print("  best_val_f1:", ckpt.get("best_val_f1", None))
    print("  metrics:", ckpt.get("metrics", None))

    return model

def _temporal_score_config(split_name: str, X: np.ndarray, meta: pd.DataFrame) -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "split": str(split_name),
        "best_ckpt_fingerprint": _temporal_ckpt_fingerprint(TEMPORAL_SYS2_BEST_CKPT),
        "n_actions": int(len(ACTION_NAMES)),
        "action_names": list(ACTION_NAMES),
        "feature_cols": list(NUMERIC_FEATURE_COLS),
        "x_shape": list(np.asarray(X).shape),
        "n_meta_rows": int(len(meta)) if meta is not None else 0,
        "sys2_seq_len": int(SYS2_SEQ_LEN),
        "cache_format": "pickle",
    }

def _temporal_score_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        return json.loads(Path(path).read_text()) == expected

    except Exception:
        return False

def _validate_temporal_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "temporal_shift_score",
        "temporal_pred_action",
        "temporal_pred_action_id",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"Temporal {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

@torch.no_grad()
def temporal_scores(
    model,
    X: np.ndarray,
    meta: pd.DataFrame,
    out_path: Path,
    meta_path: Path,
    split_name: str,
):
    expected = _temporal_score_config(split_name, X, meta)

    if out_path.exists() and not FORCE_RETRAIN_TEMPORAL_SYS2:
        meta_matches = _temporal_score_meta_matches(meta_path, expected)

        if meta_matches:
            print(f"Temporal score metadata matches for {split_name}.")
        else:
            print(f"⚠️ Temporal score metadata mismatch for {split_name}.")

            if STRICT_TEMPORAL_SCORE_META_CHECK:
                print(f"STRICT_TEMPORAL_SCORE_META_CHECK=True, so {split_name} scores will be recomputed.")
            else:
                print(
                    f"Cache-safe resume mode: loading existing temporal {split_name} scores anyway.\n"
                    f"Loaded from: {out_path}"
                )

        if meta_matches or not STRICT_TEMPORAL_SCORE_META_CHECK:
            cached = pd.read_pickle(out_path)

            if not _validate_temporal_score_cache(cached, split_name):
                raise RuntimeError(
                    f"Cached temporal {split_name} scores are incompatible:\n"
                    f"{out_path}\n\n"
                    "Set FORCE_RETRAIN_TEMPORAL_SYS2=True only if you intentionally want to retrain/rescore."
                )

            print(f"Loaded temporal {split_name} scores:", cached.shape)
            return cached

    if model is None:
        if USE_TEMPORAL_SYS2 and CACHE_SAFE_RESUME_RUN:
            raise FileNotFoundError(
                f"No compatible temporal Sys2 model is loaded and no usable cached temporal {split_name} scores were found.\n"
                f"Expected score cache: {out_path}\n\n"
                "In cache-safe resume mode, this cell will not train/rescore automatically.\n"
                "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_TEMPORAL_SYS2=True only if you intentionally want to rebuild this."
            )
        return pd.DataFrame()

    if X is None or meta is None or len(X) == 0 or len(meta) == 0:
        return pd.DataFrame()

    X = np.asarray(X, dtype=np.float32)

    probs = _temporal_predict_proba(model, X, desc=f"Temporal score {split_name}")

    cols = ["session", "time", "split", "label", "action", "action_id", "audio_path"]
    cols = [c for c in cols if c in meta.columns]

    out = meta[cols].copy().reset_index(drop=True)

    if len(out) != len(probs):
        raise ValueError(f"Temporal score length mismatch: meta={len(out)} probs={len(probs)}")

    for k, a in enumerate(ACTION_NAMES):
        out[f"temporal_p_{a}"] = probs[:, k]

    out["temporal_shift_score"] = out["temporal_p_NTT"] + out["temporal_p_ITT"]
    out["temporal_backchannel_score"] = out["temporal_p_BC"]
    out["temporal_interruption_score"] = out["temporal_p_ITT"]
    out["temporal_pred_action_id"] = probs.argmax(axis=1)
    out["temporal_pred_action"] = out["temporal_pred_action_id"].map(ID_TO_ACTION)

    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True))

    print(f"Saved temporal {split_name} scores:", out_path)

    return out

if USE_TEMPORAL_SYS2:
    TEMPORAL_SYS2_MODEL = None

    best_exists = TEMPORAL_SYS2_BEST_CKPT.exists()
    last_exists = TEMPORAL_SYS2_LAST_CKPT.exists()
    score_caches_exist = _all_temporal_score_caches_exist()

    print("Temporal Sys2 artifact state:")
    print("  best checkpoint exists:", best_exists)
    print("  last checkpoint exists:", last_exists)
    print("  all score caches exist:", score_caches_exist)

    if FORCE_RETRAIN_TEMPORAL_SYS2:
        print(
            "FORCE_RETRAIN_TEMPORAL_SYS2=True; "
            "training Temporal Sys2 from a deliberately fresh state."
        )
        TEMPORAL_SYS2_MODEL = train_temporal_sys2()

    elif best_exists:
        TEMPORAL_SYS2_MODEL = load_temporal_sys2_model(
            load_last=False
        )

        if TEMPORAL_SYS2_MODEL is None:
            print(
                "⚠️ Existing best Temporal Sys2 checkpoint is incompatible "
                "with the current event contract/features."
            )

            if AUTO_TRAIN_MISSING_PAPER_READY_STAGES:
                print(
                    "Training/resuming the current paper-ready Temporal Sys2 "
                    "stage instead of stopping."
                )
                TEMPORAL_SYS2_MODEL = train_temporal_sys2()

            elif score_caches_exist and not STRICT_TEMPORAL_SCORE_META_CHECK:
                print(
                    "Using existing score caches in score-cache-only mode."
                )

            else:
                raise RuntimeError(
                    "The existing Temporal Sys2 checkpoint is incompatible and "
                    "automatic missing-stage training is disabled."
                )

    else:
        if AUTO_TRAIN_MISSING_PAPER_READY_STAGES:
            if last_exists and RESUME_TEMPORAL_SYS2_TRAIN:
                print(
                    "No best Temporal Sys2 checkpoint yet. "
                    "Resuming the interrupted same-run last checkpoint."
                )
            else:
                print(
                    "No Temporal Sys2 checkpoint exists for this new "
                    "paper-ready event contract. Training it now."
                )

            TEMPORAL_SYS2_MODEL = train_temporal_sys2()

        elif score_caches_exist and not STRICT_TEMPORAL_SCORE_META_CHECK:
            print(
                "No checkpoint exists, but complete score caches exist; "
                "continuing in score-cache-only mode."
            )

        else:
            raise FileNotFoundError(
                "No compatible Temporal Sys2 checkpoint or complete score "
                "cache set exists, and automatic missing-stage training is "
                "disabled."
            )

else:
    TEMPORAL_SYS2_MODEL = None

TEMPORAL_TRAIN_SCORES = (
    temporal_scores(
        TEMPORAL_SYS2_MODEL,
        X_SYS2_TRAIN,
        META_SYS2_TRAIN,
        TEMPORAL_TRAIN_SCORES_PATH,
        TEMPORAL_TRAIN_SCORES_META_PATH,
        "train",
    )
    if USE_TEMPORAL_SYS2
    else pd.DataFrame()
)

TEMPORAL_VAL_SCORES = (
    temporal_scores(
        TEMPORAL_SYS2_MODEL,
        X_SYS2_VAL,
        META_SYS2_VAL,
        TEMPORAL_VAL_SCORES_PATH,
        TEMPORAL_VAL_SCORES_META_PATH,
        "val",
    )
    if USE_TEMPORAL_SYS2
    else pd.DataFrame()
)

TEMPORAL_TEST_SCORES = (
    temporal_scores(
        TEMPORAL_SYS2_MODEL,
        X_SYS2_TEST,
        META_SYS2_TEST,
        TEMPORAL_TEST_SCORES_PATH,
        TEMPORAL_TEST_SCORES_META_PATH,
        "test",
    )
    if USE_TEMPORAL_SYS2
    else pd.DataFrame()
)

globals()["TEMPORAL_SYS2_MODEL"] = TEMPORAL_SYS2_MODEL
globals()["TEMPORAL_TRAIN_SCORES"] = TEMPORAL_TRAIN_SCORES
globals()["TEMPORAL_VAL_SCORES"] = TEMPORAL_VAL_SCORES
globals()["TEMPORAL_TEST_SCORES"] = TEMPORAL_TEST_SCORES
globals()["TEMPORAL_SYS2_CKPT"] = TEMPORAL_SYS2_CKPT

print("TEMPORAL_TRAIN_SCORES:", TEMPORAL_TRAIN_SCORES.shape)
print("TEMPORAL_VAL_SCORES:", TEMPORAL_VAL_SCORES.shape)
print("TEMPORAL_TEST_SCORES:", TEMPORAL_TEST_SCORES.shape)

if len(TEMPORAL_VAL_SCORES):
    display(TEMPORAL_VAL_SCORES.head())

    val_known = TEMPORAL_VAL_SCORES[
        TEMPORAL_VAL_SCORES["label"].astype(int).isin([0, 1])
    ].copy()

    if len(val_known):
        val_pred = (val_known["temporal_shift_score"].astype(float) >= 0.50).astype(int).to_numpy()
        val_true = val_known["label"].astype(int).to_numpy()

        P, R, F1, _ = precision_recall_fscore_support(
            val_true,
            val_pred,
            average="binary",
            zero_division=0,
        )

        print(f"Temporal Sys2 VAL row-level @0.50: P={P:.4f} R={R:.4f} F1={F1:.4f}")

if len(TEMPORAL_TEST_SCORES):
    print("TEST temporal label check — should be only -1:")
    display(TEMPORAL_TEST_SCORES["label"].value_counts(dropna=False).to_frame("test_label_count"))

    test_labels = set(
        pd.to_numeric(TEMPORAL_TEST_SCORES["label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
    )
    assert test_labels.issubset({-1}), f"TEST labels leaked into temporal scores: {test_labels}"

print("✅ Cell 18 complete: temporal Sys2 verifier loaded/trained and scores loaded/generated.")

Temporal Sys2 settings:
  device: cuda
  batch_size: 128
  num_workers: 0
  pin_memory: False
  non_blocking: False
  force_retrain: False
  resume_train: True
  auto_train_missing_stages: True
  cache_safe_resume: False
  strict_score_meta_check: False
Temporal Sys2 artifact state:
  best checkpoint exists: False
  last checkpoint exists: False
  all score caches exist: False
No Temporal Sys2 checkpoint exists for this new paper-ready event contract. Training it now.


Temporal Sys2 train tensor: (206097, 13, 47) labels: (206097,) weights: (206097,)
Temporal Sys2 val tensor: (7968, 13, 47) labels: (7968,) weights: (7968,)
Temporal Sys2 train batches per epoch: 1611
Running Temporal Sys2 first-batch preflight check...
  first batch shapes: (128, 13, 47) (128,) (128,)
  first batch dtypes: torch.float32 torch.int64 torch.float32
  first batch finite X: True
  first batch finite weights: True


Temporal Sys2 first-batch preflight passed.


Temporal Sys2 epoch 1: first batch completed, loss=0.1528


Temporal Sys2 epoch 001: loss=0.1346 valP=0.232 valR=0.591 valF1=0.333 valActionAcc=0.749 best=0.333 bad=0/8


...

Saved temporal test scores: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/sys2_temporal_verifier/temporal_test_scores.pkl
TEMPORAL_TRAIN_SCORES: (206097, 17)
TEMPORAL_VAL_SCORES: (7968, 17)
TEMPORAL_TEST_SCORES: (99307, 17)


,session,time,split,label,action,action_id,audio_path,temporal_p_NA,temporal_p_NTT,temporal_p_ITT,temporal_p_BC,temporal_p_BI,temporal_shift_score,temporal_backchannel_score,temporal_interruption_score,temporal_pred_action_id,temporal_pred_action
0,ahnss,0.090504,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.900523,0.069170,0.017835,0.010297,0.002175,0.087005,0.010297,0.017835,0,NA
1,ahnss,1.331692,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.900523,0.069170,0.017835,0.010297,0.002175,0.087005,0.010297,0.017835,0,NA
2,ahnss,9.000000,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.900523,0.069170,0.017835,0.010297,0.002175,0.087005,0.010297,0.017835,0,NA
3,ahnss,17.429841,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.649657,0.330209,0.012265,0.007541,0.000328,0.342473,0.007541,0.012265,0,NA
4,ahnss,18.300001,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.649657,0.330209,0.012265,0.007541,0.000328,0.342473,0.007541,0.012265,0,NA


Temporal Sys2 VAL row-level @0.50: P=0.2688 R=0.6200 F1=0.3750
TEST temporal label check — should be only -1:


,test_label_count
label,
-1,99307


✅ Cell 18 complete: temporal Sys2 verifier loaded/trained and scores loaded/generated.


In [24]:
# Train/load secondary LightGBM action verifier/calibrator
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, Optional
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support

LGBM_SYS2_DIR.mkdir(parents=True, exist_ok=True)

LGBM_MODEL_PATH = LGBM_SYS2_DIR / "lightgbm_action_verifier.txt"
LGBM_MODEL_META_PATH = LGBM_SYS2_DIR / "lightgbm_action_verifier.meta.json"
LGBM_FEATURE_COLS_PATH = LGBM_SYS2_DIR / "feature_cols.json"
LGBM_PARAMS_PATH = LGBM_SYS2_DIR / "lightgbm_params.json"
LGBM_IMPORTANCE_PATH = LGBM_SYS2_DIR / "lightgbm_feature_importance.csv"

LGBM_TRAIN_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_train_scores.pkl"
LGBM_VAL_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_val_scores.pkl"
LGBM_TEST_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_test_scores.pkl"

LGBM_TRAIN_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_train_scores.meta.json"
LGBM_VAL_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_val_scores.meta.json"
LGBM_TEST_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_test_scores.meta.json"

LGBM_TRAIN_HISTORY_PATH = LGBM_SYS2_DIR / "lightgbm_eval_history.json"

USE_LIGHTGBM_SYS2 = bool(globals().get("USE_LIGHTGBM_SYS2", True))
FORCE_RETRAIN_LIGHTGBM = bool(globals().get("FORCE_RETRAIN_LIGHTGBM", False))

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

STRICT_LGBM_MODEL_META_CHECK = bool(
    globals().get("STRICT_LGBM_MODEL_META_CHECK", False)
)

STRICT_LGBM_SCORE_META_CHECK = bool(
    globals().get("STRICT_LGBM_SCORE_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_RETRAIN_LIGHTGBM = False

globals()["FORCE_RETRAIN_LIGHTGBM"] = FORCE_RETRAIN_LIGHTGBM
globals()["STRICT_LGBM_MODEL_META_CHECK"] = STRICT_LGBM_MODEL_META_CHECK
globals()["STRICT_LGBM_SCORE_META_CHECK"] = STRICT_LGBM_SCORE_META_CHECK

LGBM_EXCLUDE_COLS = {
    "session",
    "time",
    "split",
    "audio_path",
    "action",
    "label",
    "action_id",
    "dt_to_event",
    "sample_weight",
}

FEATURE_COLS_LGBM = [
    c for c in NUMERIC_FEATURE_COLS
    if c not in LGBM_EXCLUDE_COLS and c in SYS2_FEATURES_ALL.columns
]

FEATURE_COLS_LGBM = sorted(set(FEATURE_COLS_LGBM))

assert len(FEATURE_COLS_LGBM) > 0, "No LightGBM feature columns found. Check Cell 16/17."

LGBM_NUM_BOOST_ROUND = int(globals().get("LGBM_NUM_BOOST_ROUND", 2500))
LGBM_EARLY_STOPPING_ROUNDS = int(globals().get("LGBM_EARLY_STOPPING_ROUNDS", 100))
LGBM_LEARNING_RATE = float(globals().get("LGBM_LEARNING_RATE", 0.025))
LGBM_NUM_LEAVES = int(globals().get("LGBM_NUM_LEAVES", 63))
LGBM_MIN_DATA_IN_LEAF = int(globals().get("LGBM_MIN_DATA_IN_LEAF", 25))
LGBM_FEATURE_FRACTION = float(globals().get("LGBM_FEATURE_FRACTION", 0.85))
LGBM_BAGGING_FRACTION = float(globals().get("LGBM_BAGGING_FRACTION", 0.85))
LGBM_BAGGING_FREQ = int(globals().get("LGBM_BAGGING_FREQ", 1))
LGBM_LAMBDA_L2 = float(globals().get("LGBM_LAMBDA_L2", 2.0))
LGBM_LOG_EVERY = int(globals().get("LGBM_LOG_EVERY", 100))
LGBM_USE_CLASS_BALANCE = bool(globals().get("LGBM_USE_CLASS_BALANCE", True))

print("LightGBM Sys2 settings:")
print("  use_lightgbm:", USE_LIGHTGBM_SYS2)
print("  force_retrain:", FORCE_RETRAIN_LIGHTGBM)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_model_meta_check:", STRICT_LGBM_MODEL_META_CHECK)
print("  strict_score_meta_check:", STRICT_LGBM_SCORE_META_CHECK)
print("  n_features:", len(FEATURE_COLS_LGBM))

def _lgbm_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _json_safe_obj(obj):
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {str(k): _json_safe_obj(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [_json_safe_obj(v) for v in obj]

    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _lgbm_params() -> Dict[str, Any]:
    return {
        "objective": "multiclass",
        "num_class": int(len(ACTION_NAMES)),
        "metric": "multi_logloss",
        "learning_rate": float(LGBM_LEARNING_RATE),
        "num_leaves": int(LGBM_NUM_LEAVES),
        "min_data_in_leaf": int(LGBM_MIN_DATA_IN_LEAF),
        "feature_fraction": float(LGBM_FEATURE_FRACTION),
        "bagging_fraction": float(LGBM_BAGGING_FRACTION),
        "bagging_freq": int(LGBM_BAGGING_FREQ),
        "lambda_l2": float(LGBM_LAMBDA_L2),
        "verbosity": -1,
        "seed": int(SEED),
        "feature_pre_filter": False,
        "force_col_wise": True,
    }

def _lgbm_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "action_names": list(ACTION_NAMES),
        "train_splits": ["train", "synth_train", "real_diffusion", "synth_diffusion"],
        "val_split": "val",
        "test_split": "test",
        "objective": "multiclass",
        "num_class": int(len(ACTION_NAMES)),
        "num_boost_round": int(LGBM_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(LGBM_EARLY_STOPPING_ROUNDS),
        "learning_rate": float(LGBM_LEARNING_RATE),
        "num_leaves": int(LGBM_NUM_LEAVES),
        "min_data_in_leaf": int(LGBM_MIN_DATA_IN_LEAF),
        "feature_fraction": float(LGBM_FEATURE_FRACTION),
        "bagging_fraction": float(LGBM_BAGGING_FRACTION),
        "bagging_freq": int(LGBM_BAGGING_FREQ),
        "lambda_l2": float(LGBM_LAMBDA_L2),
        "use_class_balance": bool(LGBM_USE_CLASS_BALANCE),
        "sys2_features_fingerprint": _lgbm_file_fingerprint(SYS2_FEATURES_PATH) if "SYS2_FEATURES_PATH" in globals() else None,
        "sys2_features_meta_fingerprint": _lgbm_file_fingerprint(SYS2_FEATURES_META_PATH) if "SYS2_FEATURES_META_PATH" in globals() else None,
        "test_labels_attached": False,
        "cache_format": "pickle",
    }

def _lgbm_core_model_config() -> Dict[str, Any]:
    return {
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "action_names": list(ACTION_NAMES),
        "objective": "multiclass",
        "num_class": int(len(ACTION_NAMES)),
    }

def _lgbm_score_config(split_name: str, df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "split": str(split_name),
        "model_fingerprint": _lgbm_file_fingerprint(LGBM_MODEL_PATH),
        "model_meta_fingerprint": _lgbm_file_fingerprint(LGBM_MODEL_META_PATH),
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "n_rows": int(len(df)) if df is not None else 0,
        "action_names": list(ACTION_NAMES),
        "cache_format": "pickle",
    }

def _json_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        old = json.loads(Path(path).read_text())
        return old == expected

    except Exception:
        return False

def _lgbm_model_meta_is_compatible() -> bool:
    if not LGBM_MODEL_META_PATH.exists():
        print("⚠️ Missing LightGBM model meta:", LGBM_MODEL_META_PATH)
        return False

    try:
        saved_meta = json.loads(LGBM_MODEL_META_PATH.read_text())
    except Exception as e:
        print("⚠️ Could not read LightGBM model meta:", e)
        return False

    if STRICT_LGBM_MODEL_META_CHECK:
        expected = _lgbm_config()

        for k, expected_v in expected.items():
            saved_v = saved_meta.get(k, None)

            if saved_v != expected_v:
                print(f"⚠️ LightGBM strict model meta mismatch at key: {k}")
                return False

        return True

    expected_core = _lgbm_core_model_config()

    for k, expected_v in expected_core.items():
        saved_v = saved_meta.get(k, None)

        if saved_v != expected_v:
            print(f"⚠️ LightGBM core model meta mismatch at key: {k}")
            print("  expected:", expected_v)
            print("  saved:   ", saved_v)
            return False

    return True

def _clean_legacy_lgbm_parquet():
    for p in [
        LGBM_SYS2_DIR / "lightgbm_train_scores.parquet",
        LGBM_SYS2_DIR / "lightgbm_val_scores.parquet",
        LGBM_SYS2_DIR / "lightgbm_test_scores.parquet",
    ]:
        try:
            if p.exists():
                p.unlink()
                print("Removed legacy parquet LightGBM score cache:", p)

        except Exception as e:
            print("Could not remove legacy parquet LightGBM score cache:", p, e)

def _clear_stale_lgbm_score_caches():
    for p in [
        LGBM_TRAIN_SCORES_PATH,
        LGBM_VAL_SCORES_PATH,
        LGBM_TEST_SCORES_PATH,
        LGBM_TRAIN_SCORES_META_PATH,
        LGBM_VAL_SCORES_META_PATH,
        LGBM_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale LightGBM score cache:", p)

        except Exception as e:
            print("Could not remove stale LightGBM score cache:", p, e)

def _all_lgbm_score_caches_exist() -> bool:
    return (
        LGBM_TRAIN_SCORES_PATH.exists()
        and LGBM_VAL_SCORES_PATH.exists()
        and LGBM_TEST_SCORES_PATH.exists()
    )

def _prepare_lgbm_frame(df: pd.DataFrame, require_labels: bool) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()

    for c in FEATURE_COLS_LGBM:
        if c not in out.columns:
            out[c] = 0.0

    for c in FEATURE_COLS_LGBM:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out[FEATURE_COLS_LGBM] = (
        out[FEATURE_COLS_LGBM]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    if "action_id" not in out.columns:
        out["action_id"] = -1

    out["action_id"] = pd.to_numeric(out["action_id"], errors="coerce").fillna(-1).astype(int)

    if "label" not in out.columns:
        out["label"] = -1

    out["label"] = pd.to_numeric(out["label"], errors="coerce").fillna(-1).astype(int)

    if "action" not in out.columns:
        out["action"] = "NA"

    out["action"] = out["action"].fillna("NA").astype(str)

    if "sample_weight" not in out.columns:
        out["sample_weight"] = 1.0

    out["sample_weight"] = pd.to_numeric(out["sample_weight"], errors="coerce").fillna(1.0).astype(float)
    out["sample_weight"] = np.maximum(out["sample_weight"].to_numpy(dtype=float), 1e-6)

    if require_labels:
        out = out[out["action_id"].between(0, len(ACTION_NAMES) - 1)].copy()

    return out.reset_index(drop=True)

def _make_lgbm_weights(train_df: pd.DataFrame) -> np.ndarray:
    base_w = train_df["sample_weight"].astype(float).to_numpy(dtype=np.float32)

    if not LGBM_USE_CLASS_BALANCE:
        return base_w.astype(np.float32)

    y = train_df["action_id"].astype(int).to_numpy()
    counts = np.bincount(y, minlength=len(ACTION_NAMES)).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    class_w = counts.sum() / counts
    class_w = class_w / max(1e-8, class_w.mean())

    w = base_w * class_w[y]
    w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
    w = np.maximum(w, 1e-6)

    return w.astype(np.float32)

def train_lightgbm_verifier():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    LGBM_SYS2_DIR.mkdir(parents=True, exist_ok=True)

    tr = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin(["train", "synth_train", "real_diffusion", "synth_diffusion"])
    ].copy()

    va = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str) == "val"
    ].copy()

    tr = _prepare_lgbm_frame(tr, require_labels=True)
    va = _prepare_lgbm_frame(va, require_labels=True)

    assert len(tr) > 0, "No valid LightGBM training rows."
    assert len(va) > 0, "No valid LightGBM validation rows."

    Xtr = tr[FEATURE_COLS_LGBM]
    ytr = tr["action_id"].astype(int)
    wtr = _make_lgbm_weights(tr)

    Xva = va[FEATURE_COLS_LGBM]
    yva = va["action_id"].astype(int)
    wva = va["sample_weight"].astype(float).to_numpy(dtype=np.float32)

    dtr = lgb.Dataset(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_name=FEATURE_COLS_LGBM,
        free_raw_data=False,
    )

    dva = lgb.Dataset(
        Xva,
        label=yva,
        weight=wva,
        reference=dtr,
        feature_name=FEATURE_COLS_LGBM,
        free_raw_data=False,
    )

    params = _lgbm_params()
    evals_result = {}

    model = lgb.train(
        params,
        dtr,
        num_boost_round=int(LGBM_NUM_BOOST_ROUND),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=int(LGBM_EARLY_STOPPING_ROUNDS),
                first_metric_only=True,
                verbose=True,
            ),
            lgb.log_evaluation(period=int(LGBM_LOG_EVERY)),
            lgb.record_evaluation(evals_result),
        ],
    )

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        model.save_model(str(LGBM_MODEL_PATH), num_iteration=best_iter)
    else:
        model.save_model(str(LGBM_MODEL_PATH))

    model_meta = _lgbm_config()
    model_meta.update({
        "best_iteration": int(best_iter),
        "best_score": _json_safe_obj(getattr(model, "best_score", {})),
        "trained_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    })

    LGBM_MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2, sort_keys=True))
    LGBM_FEATURE_COLS_PATH.write_text(json.dumps(FEATURE_COLS_LGBM, indent=2))
    LGBM_PARAMS_PATH.write_text(json.dumps(params, indent=2, sort_keys=True))
    LGBM_TRAIN_HISTORY_PATH.write_text(json.dumps(_json_safe_obj(evals_result), indent=2, sort_keys=True))

    importance_df = pd.DataFrame({
        "feature": FEATURE_COLS_LGBM,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)

    importance_df.to_csv(LGBM_IMPORTANCE_PATH, index=False)

    print("Saved LightGBM model:", LGBM_MODEL_PATH)
    print("Saved LightGBM model meta:", LGBM_MODEL_META_PATH)
    print("Saved LightGBM feature cols:", LGBM_FEATURE_COLS_PATH)
    print("Saved LightGBM feature importance:", LGBM_IMPORTANCE_PATH)
    print("Best iteration:", best_iter)

    _clear_stale_lgbm_score_caches()

    return model

def load_lightgbm_model():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    if not LGBM_MODEL_PATH.exists():
        return None

    if not _lgbm_model_meta_is_compatible():
        print("⚠️ Existing LightGBM model is incompatible with current config.")
        return None

    model = lgb.Booster(model_file=str(LGBM_MODEL_PATH))

    print("Loaded LightGBM:", LGBM_MODEL_PATH)

    return model

def _validate_lgbm_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "lgbm_shift_score",
        "lgbm_pred_action",
        "lgbm_pred_action_id",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"LightGBM {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

def lgbm_scores(
    model,
    df: pd.DataFrame,
    out_path: Path,
    meta_path: Path,
    split_name: str,
) -> pd.DataFrame:
    expected = _lgbm_score_config(split_name, df)

    if out_path.exists() and not FORCE_RETRAIN_LIGHTGBM:
        meta_matches = _json_meta_matches(meta_path, expected)

        if meta_matches:
            print(f"LightGBM score metadata matches for {split_name}.")
        else:
            print(f"⚠️ LightGBM score metadata mismatch for {split_name}.")

            if STRICT_LGBM_SCORE_META_CHECK:
                print(f"STRICT_LGBM_SCORE_META_CHECK=True, so {split_name} scores will be recomputed.")
            else:
                print(
                    f"Cache-safe resume mode: loading existing LightGBM {split_name} scores anyway.\n"
                    f"Loaded from: {out_path}"
                )

        if meta_matches or not STRICT_LGBM_SCORE_META_CHECK:
            cached = pd.read_pickle(out_path)

            if not _validate_lgbm_score_cache(cached, split_name):
                raise RuntimeError(
                    f"Cached LightGBM {split_name} scores are incompatible:\n"
                    f"{out_path}\n\n"
                    "Set FORCE_RETRAIN_LIGHTGBM=True only if you intentionally want to retrain/rescore."
                )

            print(f"Loaded LightGBM {split_name} scores:", cached.shape)
            return cached

    if model is None:
        if USE_LIGHTGBM_SYS2 and CACHE_SAFE_RESUME_RUN:
            raise FileNotFoundError(
                f"No compatible LightGBM model is loaded and no usable cached LightGBM {split_name} scores were found.\n"
                f"Expected score cache: {out_path}\n\n"
                "In cache-safe resume mode, this cell will not train/rescore automatically.\n"
                "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM=True only if you intentionally want to rebuild this."
            )
        return pd.DataFrame()

    if df is None or len(df) == 0:
        return pd.DataFrame()

    work = _prepare_lgbm_frame(df, require_labels=False)

    if len(work) == 0:
        return pd.DataFrame()

    X = work[FEATURE_COLS_LGBM]

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        probs = model.predict(X, num_iteration=best_iter)
    else:
        probs = model.predict(X)

    probs = np.asarray(probs, dtype=np.float32)

    if probs.ndim == 1:
        probs = probs.reshape(-1, len(ACTION_NAMES))

    base_cols = ["session", "time", "split", "label", "action", "action_id", "audio_path"]
    base_cols = [c for c in base_cols if c in work.columns]

    out = work[base_cols].copy().reset_index(drop=True)

    if len(out) != len(probs):
        raise ValueError(f"LightGBM score length mismatch: rows={len(out)} probs={len(probs)}")

    for k, a in enumerate(ACTION_NAMES):
        out[f"lgbm_p_{a}"] = probs[:, k]

    out["lgbm_shift_score"] = out["lgbm_p_NTT"] + out["lgbm_p_ITT"]
    out["lgbm_backchannel_score"] = out["lgbm_p_BC"]
    out["lgbm_interruption_score"] = out["lgbm_p_ITT"]
    out["lgbm_pred_action_id"] = probs.argmax(axis=1)
    out["lgbm_pred_action"] = out["lgbm_pred_action_id"].map(ID_TO_ACTION)

    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True))

    print(f"Saved LightGBM {split_name} scores:", out_path)

    return out

_clean_legacy_lgbm_parquet()

if USE_LIGHTGBM_SYS2:
    LGBM_MODEL = None

    try:
        if LGBM_MODEL_PATH.exists() and not FORCE_RETRAIN_LIGHTGBM:
            LGBM_MODEL = load_lightgbm_model()

            if LGBM_MODEL is None:
                msg = (
                    "LightGBM model exists but is incompatible with the current features/config:\n"
                    f"{LGBM_MODEL_PATH}\n\n"
                    "In cache-safe resume mode, this cell will not retrain automatically."
                )

                if CACHE_SAFE_RESUME_RUN and _all_lgbm_score_caches_exist():
                    print("⚠️", msg)
                    print("LightGBM score caches exist, so continuing in score-cache-only mode.")
                elif CACHE_SAFE_RESUME_RUN:
                    raise RuntimeError(
                        msg
                        + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM=True only if you intentionally want to retrain."
                    )
                else:
                    print("No compatible LightGBM model found. Training a new one.")
                    LGBM_MODEL = train_lightgbm_verifier()

        elif FORCE_RETRAIN_LIGHTGBM:
            print("FORCE_RETRAIN_LIGHTGBM=True, training LightGBM.")
            LGBM_MODEL = train_lightgbm_verifier()

        else:
            msg = (
                "No LightGBM model found:\n"
                f"{LGBM_MODEL_PATH}\n\n"
                "In cache-safe resume mode, this cell will not train automatically."
            )

            if CACHE_SAFE_RESUME_RUN and _all_lgbm_score_caches_exist():
                print("⚠️", msg)
                print("LightGBM score caches exist, so continuing in score-cache-only mode.")
            elif CACHE_SAFE_RESUME_RUN:
                raise FileNotFoundError(
                    msg
                    + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM=True only if you intentionally want to train."
                )
            else:
                LGBM_MODEL = train_lightgbm_verifier()

    except Exception as e:
        if CACHE_SAFE_RESUME_RUN and _all_lgbm_score_caches_exist():
            print("⚠️ LightGBM model load/train branch unavailable:", e)
            print("LightGBM score caches exist, so continuing in score-cache-only mode.")
            LGBM_MODEL = None
        else:
            raise

else:
    LGBM_MODEL = None

LGBM_TRAIN_SCORES = (
    lgbm_scores(
        LGBM_MODEL,
        SYS2_FEATURES_ALL[
            SYS2_FEATURES_ALL["split"].astype(str).isin(["train", "synth_train", "real_diffusion", "synth_diffusion"])
        ].copy(),
        LGBM_TRAIN_SCORES_PATH,
        LGBM_TRAIN_SCORES_META_PATH,
        "train",
    )
    if USE_LIGHTGBM_SYS2
    else pd.DataFrame()
)

LGBM_VAL_SCORES = (
    lgbm_scores(
        LGBM_MODEL,
        SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "val"].copy(),
        LGBM_VAL_SCORES_PATH,
        LGBM_VAL_SCORES_META_PATH,
        "val",
    )
    if USE_LIGHTGBM_SYS2
    else pd.DataFrame()
)

LGBM_TEST_SCORES = (
    lgbm_scores(
        LGBM_MODEL,
        SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "test"].copy(),
        LGBM_TEST_SCORES_PATH,
        LGBM_TEST_SCORES_META_PATH,
        "test",
    )
    if USE_LIGHTGBM_SYS2
    else pd.DataFrame()
)

globals()["LGBM_MODEL"] = LGBM_MODEL
globals()["LGBM_TRAIN_SCORES"] = LGBM_TRAIN_SCORES
globals()["LGBM_VAL_SCORES"] = LGBM_VAL_SCORES
globals()["LGBM_TEST_SCORES"] = LGBM_TEST_SCORES
globals()["FEATURE_COLS_LGBM"] = FEATURE_COLS_LGBM

print("LGBM_TRAIN_SCORES:", LGBM_TRAIN_SCORES.shape)
print("LGBM_VAL_SCORES:", LGBM_VAL_SCORES.shape)
print("LGBM_TEST_SCORES:", LGBM_TEST_SCORES.shape)

if len(LGBM_VAL_SCORES):
    display(LGBM_VAL_SCORES.head())

    val_known = LGBM_VAL_SCORES[
        LGBM_VAL_SCORES["label"].astype(int).isin([0, 1])
    ].copy()

    if len(val_known):
        val_pred = (val_known["lgbm_shift_score"].astype(float) >= 0.50).astype(int).to_numpy()
        val_true = val_known["label"].astype(int).to_numpy()

        P, R, F1, _ = precision_recall_fscore_support(
            val_true,
            val_pred,
            average="binary",
            zero_division=0,
        )

        print(f"LightGBM Sys2 VAL row-level @0.50: P={P:.4f} R={R:.4f} F1={F1:.4f}")

if len(LGBM_TEST_SCORES):
    print("TEST LightGBM label check — should be only -1:")
    display(LGBM_TEST_SCORES["label"].value_counts(dropna=False).to_frame("test_label_count"))

    test_labels = set(
        pd.to_numeric(LGBM_TEST_SCORES["label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
    )

    assert test_labels.issubset({-1}), f"TEST labels leaked into LightGBM scores: {test_labels}"

if LGBM_IMPORTANCE_PATH.exists():
    print("Top LightGBM features by gain:")
    display(pd.read_csv(LGBM_IMPORTANCE_PATH).head(20))

print("✅ Cell 19 complete: LightGBM Sys2 verifier loaded/trained and scores loaded/generated.")

LightGBM Sys2 settings:
  use_lightgbm: True
  force_retrain: False
  cache_safe_resume: False
  strict_model_meta_check: False
  strict_score_meta_check: False
  n_features: 47


Training until validation scores don't improve for 100 rounds


[100]	train's multi_logloss: 0.645717	val's multi_logloss: 0.399988


[200]	train's multi_logloss: 0.533092	val's multi_logloss: 0.339237


[300]	train's multi_logloss: 0.471457	val's multi_logloss: 0.320682


...

LightGBM Sys2 VAL row-level @0.50: P=0.5231 R=0.6283 F1=0.5709
TEST LightGBM label check — should be only -1:


,test_label_count
label,
-1,99307


Top LightGBM features by gain:


,feature,importance_gain,importance_split
0,wavlm_p_ITT,157834.473198,32709
1,wavlm_p_BI,148817.155037,33856
2,wavlm_p_BC,105926.757816,32531
3,wavlm_p_NTT,101676.175766,30061
4,ecapa_cos,78186.147028,35925
5,wavlm_score,60443.832395,33711
6,wavlm_p_NA,50939.577222,30537
7,energy_right,34148.882700,28903
8,wavlm_center_score,33814.606105,33452
9,qwen_backchannel_score,30078.610826,25684


✅ Cell 19 complete: LightGBM Sys2 verifier loaded/trained and scores loaded/generated.


In [25]:
# Train/load binary LightGBM SHIFT/HOLD verifier
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score

assert "LGBM_SYS2_DIR" in globals(), "Run Cell 19 first so LGBM_SYS2_DIR is available."
assert "FEATURE_COLS_LGBM" in globals(), "Run Cell 19 first so FEATURE_COLS_LGBM is available."
assert "SYS2_FEATURES_ALL" in globals(), "Run Cell 16 first so SYS2_FEATURES_ALL is available."
assert "LGBM_TRAIN_SCORES" in globals(), "Run Cell 19 first so LGBM_TRAIN_SCORES exists."
assert "LGBM_VAL_SCORES" in globals(), "Run Cell 19 first so LGBM_VAL_SCORES exists."
assert "LGBM_TEST_SCORES" in globals(), "Run Cell 19 first so LGBM_TEST_SCORES exists."
assert "_prepare_lgbm_frame" in globals(), "Run Cell 19 first so _prepare_lgbm_frame exists."
assert "_lgbm_file_fingerprint" in globals(), "Run Cell 19 first so _lgbm_file_fingerprint exists."
assert "_json_meta_matches" in globals(), "Run Cell 19 first so _json_meta_matches exists."

LGBM_BINARY_VARIANT = "binary_shift_hold_v1"

LGBM_BINARY_MODEL_PATH = LGBM_SYS2_DIR / "lightgbm_binary_shift_verifier.txt"
LGBM_BINARY_MODEL_META_PATH = LGBM_SYS2_DIR / "lightgbm_binary_shift_verifier.meta.json"
LGBM_BINARY_PARAMS_PATH = LGBM_SYS2_DIR / "lightgbm_binary_shift_params.json"
LGBM_BINARY_IMPORTANCE_PATH = LGBM_SYS2_DIR / "lightgbm_binary_shift_feature_importance.csv"
LGBM_BINARY_HISTORY_PATH = LGBM_SYS2_DIR / "lightgbm_binary_shift_eval_history.json"

LGBM_BINARY_TRAIN_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_binary_train_scores.pkl"
LGBM_BINARY_VAL_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_binary_val_scores.pkl"
LGBM_BINARY_TEST_SCORES_PATH = LGBM_SYS2_DIR / "lightgbm_binary_test_scores.pkl"

LGBM_BINARY_TRAIN_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_binary_train_scores.meta.json"
LGBM_BINARY_VAL_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_binary_val_scores.meta.json"
LGBM_BINARY_TEST_SCORES_META_PATH = LGBM_SYS2_DIR / "lightgbm_binary_test_scores.meta.json"

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

FORCE_RETRAIN_LIGHTGBM_BINARY = bool(
    globals().get(
        "FORCE_RETRAIN_LIGHTGBM_BINARY",
        globals().get("FORCE_RETRAIN_LIGHTGBM", False),
    )
)

STRICT_LGBM_BINARY_MODEL_META_CHECK = bool(
    globals().get("STRICT_LGBM_BINARY_MODEL_META_CHECK", False)
)

STRICT_LGBM_BINARY_SCORE_META_CHECK = bool(
    globals().get("STRICT_LGBM_BINARY_SCORE_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_RETRAIN_LIGHTGBM_BINARY = False

globals()["FORCE_RETRAIN_LIGHTGBM_BINARY"] = FORCE_RETRAIN_LIGHTGBM_BINARY
globals()["STRICT_LGBM_BINARY_MODEL_META_CHECK"] = STRICT_LGBM_BINARY_MODEL_META_CHECK
globals()["STRICT_LGBM_BINARY_SCORE_META_CHECK"] = STRICT_LGBM_BINARY_SCORE_META_CHECK

LGBM_BINARY_NUM_BOOST_ROUND = int(globals().get("LGBM_BINARY_NUM_BOOST_ROUND", 4000))
LGBM_BINARY_EARLY_STOPPING_ROUNDS = int(globals().get("LGBM_BINARY_EARLY_STOPPING_ROUNDS", 200))
LGBM_BINARY_LEARNING_RATE = float(globals().get("LGBM_BINARY_LEARNING_RATE", 0.020))
LGBM_BINARY_NUM_LEAVES = int(globals().get("LGBM_BINARY_NUM_LEAVES", max(31, int(globals().get("LGBM_NUM_LEAVES", 63)))))
LGBM_BINARY_MIN_DATA_IN_LEAF = int(globals().get("LGBM_BINARY_MIN_DATA_IN_LEAF", 20))
LGBM_BINARY_FEATURE_FRACTION = float(globals().get("LGBM_BINARY_FEATURE_FRACTION", 0.90))
LGBM_BINARY_BAGGING_FRACTION = float(globals().get("LGBM_BINARY_BAGGING_FRACTION", 0.85))
LGBM_BINARY_BAGGING_FREQ = int(globals().get("LGBM_BINARY_BAGGING_FREQ", 1))
LGBM_BINARY_LAMBDA_L2 = float(globals().get("LGBM_BINARY_LAMBDA_L2", 2.0))
LGBM_BINARY_LOG_EVERY = int(globals().get("LGBM_BINARY_LOG_EVERY", 100))
LGBM_BINARY_USE_CLASS_BALANCE = bool(globals().get("LGBM_BINARY_USE_CLASS_BALANCE", True))
LGBM_BINARY_POS_WEIGHT_MULT = float(globals().get("LGBM_BINARY_POS_WEIGHT_MULT", 1.15))

print("Binary LightGBM Sys2 settings:")
print("  force_retrain:", FORCE_RETRAIN_LIGHTGBM_BINARY)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_model_meta_check:", STRICT_LGBM_BINARY_MODEL_META_CHECK)
print("  strict_score_meta_check:", STRICT_LGBM_BINARY_SCORE_META_CHECK)
print("  n_features:", len(FEATURE_COLS_LGBM))

def _json_safe_binary_obj(obj):
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _json_safe_binary_obj(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_json_safe_binary_obj(v) for v in obj]
    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _binary_params() -> Dict[str, Any]:
    return {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "learning_rate": float(LGBM_BINARY_LEARNING_RATE),
        "num_leaves": int(LGBM_BINARY_NUM_LEAVES),
        "min_data_in_leaf": int(LGBM_BINARY_MIN_DATA_IN_LEAF),
        "feature_fraction": float(LGBM_BINARY_FEATURE_FRACTION),
        "bagging_fraction": float(LGBM_BINARY_BAGGING_FRACTION),
        "bagging_freq": int(LGBM_BINARY_BAGGING_FREQ),
        "lambda_l2": float(LGBM_BINARY_LAMBDA_L2),
        "verbosity": -1,
        "seed": int(SEED),
        "feature_pre_filter": False,
        "force_col_wise": True,
    }

def _binary_model_config() -> Dict[str, Any]:
    return {
        "variant": LGBM_BINARY_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "target": "label_binary_shift_hold",
        "positive_class": "SHIFT",
        "negative_class": "HOLD",
        "train_splits": ["train", "synth_train", "real_diffusion", "synth_diffusion"],
        "val_split": "val",
        "test_split": "test",
        "objective": "binary",
        "num_boost_round": int(LGBM_BINARY_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(LGBM_BINARY_EARLY_STOPPING_ROUNDS),
        "learning_rate": float(LGBM_BINARY_LEARNING_RATE),
        "num_leaves": int(LGBM_BINARY_NUM_LEAVES),
        "min_data_in_leaf": int(LGBM_BINARY_MIN_DATA_IN_LEAF),
        "feature_fraction": float(LGBM_BINARY_FEATURE_FRACTION),
        "bagging_fraction": float(LGBM_BINARY_BAGGING_FRACTION),
        "bagging_freq": int(LGBM_BINARY_BAGGING_FREQ),
        "lambda_l2": float(LGBM_BINARY_LAMBDA_L2),
        "use_class_balance": bool(LGBM_BINARY_USE_CLASS_BALANCE),
        "pos_weight_mult": float(LGBM_BINARY_POS_WEIGHT_MULT),
        "sys2_features_fingerprint": _lgbm_file_fingerprint(SYS2_FEATURES_PATH) if "SYS2_FEATURES_PATH" in globals() else None,
        "sys2_features_meta_fingerprint": _lgbm_file_fingerprint(SYS2_FEATURES_META_PATH) if "SYS2_FEATURES_META_PATH" in globals() else None,
        "test_labels_attached": False,
        "cache_format": "pickle",
    }

def _binary_core_model_config() -> Dict[str, Any]:
    return {
        "variant": LGBM_BINARY_VARIANT,
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "target": "label_binary_shift_hold",
        "positive_class": "SHIFT",
        "negative_class": "HOLD",
        "objective": "binary",
    }

def _binary_score_config(split_name: str, df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "variant": LGBM_BINARY_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "split": str(split_name),
        "model_fingerprint": _lgbm_file_fingerprint(LGBM_BINARY_MODEL_PATH),
        "model_meta_fingerprint": _lgbm_file_fingerprint(LGBM_BINARY_MODEL_META_PATH),
        "feature_cols": list(FEATURE_COLS_LGBM),
        "n_features": int(len(FEATURE_COLS_LGBM)),
        "n_rows": int(len(df)) if df is not None else 0,
        "target": "label_binary_shift_hold",
        "cache_format": "pickle",
    }

def _binary_model_meta_is_compatible() -> bool:
    if not LGBM_BINARY_MODEL_META_PATH.exists():
        print("⚠️ Missing binary LightGBM model meta:", LGBM_BINARY_MODEL_META_PATH)
        return False

    try:
        saved_meta = json.loads(LGBM_BINARY_MODEL_META_PATH.read_text())
    except Exception as e:
        print("⚠️ Could not read binary LightGBM model meta:", e)
        return False

    if STRICT_LGBM_BINARY_MODEL_META_CHECK:
        expected = _binary_model_config()

        for k, expected_v in expected.items():
            if saved_meta.get(k, None) != expected_v:
                print(f"⚠️ Binary LightGBM strict model meta mismatch at key: {k}")
                return False

        return True

    expected_core = _binary_core_model_config()

    for k, expected_v in expected_core.items():
        saved_v = saved_meta.get(k, None)

        if saved_v != expected_v:
            print(f"⚠️ Binary LightGBM core model meta mismatch at key: {k}")
            print("  expected:", expected_v)
            print("  saved:   ", saved_v)
            return False

    return True

def _prepare_lgbm_binary_frame(df: pd.DataFrame, require_labels: bool) -> pd.DataFrame:
    out = _prepare_lgbm_frame(df, require_labels=False)

    if out is None or len(out) == 0:
        return pd.DataFrame()

    if "label" not in out.columns:
        out["label"] = -1

    out["label"] = pd.to_numeric(out["label"], errors="coerce").fillna(-1).astype(int)

    if require_labels:
        out = out[out["label"].isin([0, 1])].copy()

    return out.reset_index(drop=True)

def _make_binary_weights(train_df: pd.DataFrame) -> np.ndarray:
    base_w = (
        pd.to_numeric(train_df["sample_weight"], errors="coerce")
        .fillna(1.0)
        .astype(float)
        .to_numpy(dtype=np.float32)
    )
    base_w = np.maximum(base_w, 1e-6)

    if not LGBM_BINARY_USE_CLASS_BALANCE:
        return base_w.astype(np.float32)

    y = train_df["label"].astype(int).to_numpy()
    counts = np.bincount(y, minlength=2).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    class_w = counts.sum() / counts
    class_w = class_w / max(1e-8, class_w.mean())

    class_w[1] *= float(LGBM_BINARY_POS_WEIGHT_MULT)

    w = base_w * class_w[y]
    w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
    w = np.maximum(w, 1e-6)

    return w.astype(np.float32)

def _clear_stale_binary_lgbm_score_caches():
    for p in [
        LGBM_BINARY_TRAIN_SCORES_PATH,
        LGBM_BINARY_VAL_SCORES_PATH,
        LGBM_BINARY_TEST_SCORES_PATH,
        LGBM_BINARY_TRAIN_SCORES_META_PATH,
        LGBM_BINARY_VAL_SCORES_META_PATH,
        LGBM_BINARY_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale binary LightGBM score cache:", p)
        except Exception as e:
            print("Could not remove stale binary LightGBM score cache:", p, e)

def _all_binary_lgbm_score_caches_exist() -> bool:
    return (
        LGBM_BINARY_TRAIN_SCORES_PATH.exists()
        and LGBM_BINARY_VAL_SCORES_PATH.exists()
        and LGBM_BINARY_TEST_SCORES_PATH.exists()
    )

def train_lightgbm_binary_shift_verifier():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    tr = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin(["train", "synth_train", "real_diffusion", "synth_diffusion"])
    ].copy()

    va = SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "val"].copy()

    tr = _prepare_lgbm_binary_frame(tr, require_labels=True)
    va = _prepare_lgbm_binary_frame(va, require_labels=True)

    assert len(tr) > 0, "No valid binary LightGBM training rows."
    assert len(va) > 0, "No valid binary LightGBM validation rows."

    Xtr = tr[FEATURE_COLS_LGBM]
    ytr = tr["label"].astype(int)
    wtr = _make_binary_weights(tr)

    Xva = va[FEATURE_COLS_LGBM]
    yva = va["label"].astype(int)
    wva = va["sample_weight"].astype(float).to_numpy(dtype=np.float32)

    dtr = lgb.Dataset(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_name=FEATURE_COLS_LGBM,
        free_raw_data=False,
    )

    dva = lgb.Dataset(
        Xva,
        label=yva,
        weight=wva,
        reference=dtr,
        feature_name=FEATURE_COLS_LGBM,
        free_raw_data=False,
    )

    params = _binary_params()
    evals_result = {}

    model = lgb.train(
        params,
        dtr,
        num_boost_round=int(LGBM_BINARY_NUM_BOOST_ROUND),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=int(LGBM_BINARY_EARLY_STOPPING_ROUNDS),
                first_metric_only=True,
                verbose=True,
            ),
            lgb.log_evaluation(period=int(LGBM_BINARY_LOG_EVERY)),
            lgb.record_evaluation(evals_result),
        ],
    )

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        model.save_model(str(LGBM_BINARY_MODEL_PATH), num_iteration=best_iter)
    else:
        model.save_model(str(LGBM_BINARY_MODEL_PATH))

    model_meta = _binary_model_config()
    model_meta.update({
        "best_iteration": int(best_iter),
        "best_score": _json_safe_binary_obj(getattr(model, "best_score", {})),
        "trained_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    })

    LGBM_BINARY_MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2, sort_keys=True))
    LGBM_BINARY_PARAMS_PATH.write_text(json.dumps(params, indent=2, sort_keys=True))
    LGBM_BINARY_HISTORY_PATH.write_text(json.dumps(_json_safe_binary_obj(evals_result), indent=2, sort_keys=True))

    importance_df = pd.DataFrame({
        "feature": FEATURE_COLS_LGBM,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)

    importance_df.to_csv(LGBM_BINARY_IMPORTANCE_PATH, index=False)

    print("Saved binary LightGBM model:", LGBM_BINARY_MODEL_PATH)
    print("Saved binary LightGBM model meta:", LGBM_BINARY_MODEL_META_PATH)
    print("Saved binary LightGBM feature importance:", LGBM_BINARY_IMPORTANCE_PATH)
    print("Binary best iteration:", best_iter)

    _clear_stale_binary_lgbm_score_caches()

    return model

def load_lightgbm_binary_shift_model():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    if not LGBM_BINARY_MODEL_PATH.exists():
        return None

    if not _binary_model_meta_is_compatible():
        print("⚠️ Existing binary LightGBM model is incompatible with current config.")
        return None

    model = lgb.Booster(model_file=str(LGBM_BINARY_MODEL_PATH))
    print("Loaded binary LightGBM:", LGBM_BINARY_MODEL_PATH)
    return model

def _validate_lgbm_binary_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "lgbm_binary_shift_score",
        "lgbm_binary_hold_score",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"Binary LightGBM {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

def lgbm_binary_scores(
    model,
    df: pd.DataFrame,
    out_path: Path,
    meta_path: Path,
    split_name: str,
) -> pd.DataFrame:
    expected = _binary_score_config(split_name, df)

    if out_path.exists() and not FORCE_RETRAIN_LIGHTGBM_BINARY:
        meta_matches = _json_meta_matches(meta_path, expected)

        if meta_matches:
            print(f"Binary LightGBM score metadata matches for {split_name}.")
        else:
            print(f"⚠️ Binary LightGBM score metadata mismatch for {split_name}.")

            if STRICT_LGBM_BINARY_SCORE_META_CHECK:
                print(f"STRICT_LGBM_BINARY_SCORE_META_CHECK=True, so {split_name} binary scores will be recomputed.")
            else:
                print(
                    f"Cache-safe resume mode: loading existing binary LightGBM {split_name} scores anyway.\n"
                    f"Loaded from: {out_path}"
                )

        if meta_matches or not STRICT_LGBM_BINARY_SCORE_META_CHECK:
            cached = pd.read_pickle(out_path)

            if not _validate_lgbm_binary_score_cache(cached, split_name):
                raise RuntimeError(
                    f"Cached binary LightGBM {split_name} scores are incompatible:\n"
                    f"{out_path}\n\n"
                    "Set FORCE_RETRAIN_LIGHTGBM_BINARY=True only if you intentionally want to retrain/rescore."
                )

            print(f"Loaded binary LightGBM {split_name} scores:", cached.shape)
            return cached

    if model is None:
        if CACHE_SAFE_RESUME_RUN:
            raise FileNotFoundError(
                f"No compatible binary LightGBM model is loaded and no usable cached binary {split_name} scores were found.\n"
                f"Expected score cache: {out_path}\n\n"
                "In cache-safe resume mode, this cell will not train/rescore automatically.\n"
                "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM_BINARY=True only if you intentionally want to rebuild this."
            )
        return pd.DataFrame()

    if df is None or len(df) == 0:
        return pd.DataFrame()

    work = _prepare_lgbm_binary_frame(df, require_labels=False)

    if len(work) == 0:
        return pd.DataFrame()

    X = work[FEATURE_COLS_LGBM]
    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        prob_shift = model.predict(X, num_iteration=best_iter)
    else:
        prob_shift = model.predict(X)

    prob_shift = np.asarray(prob_shift, dtype=np.float32).reshape(-1)
    prob_shift = np.clip(prob_shift, 0.0, 1.0)

    base_cols = ["session", "time", "split", "label", "action", "action_id", "audio_path"]
    base_cols = [c for c in base_cols if c in work.columns]

    out = work[base_cols].copy().reset_index(drop=True)

    if len(out) != len(prob_shift):
        raise ValueError(f"Binary LightGBM score length mismatch: rows={len(out)} probs={len(prob_shift)}")

    out["lgbm_binary_shift_score"] = prob_shift
    out["lgbm_binary_hold_score"] = 1.0 - prob_shift

    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True))

    print(f"Saved binary LightGBM {split_name} scores:", out_path)

    return out

def _merge_binary_scores_into_lgbm_scores(main_df: pd.DataFrame, bin_df: pd.DataFrame) -> pd.DataFrame:
    if main_df is None or len(main_df) == 0:
        return pd.DataFrame()

    out = main_df.copy().reset_index(drop=True)

    for c in [
        "lgbm_binary_shift_score",
        "lgbm_binary_hold_score",
        "lgbm_best_shift_score",
    ]:
        if c in out.columns:
            out = out.drop(columns=[c])

    if bin_df is None or len(bin_df) == 0 or "lgbm_binary_shift_score" not in bin_df.columns:
        out["lgbm_binary_shift_score"] = pd.to_numeric(out.get("lgbm_shift_score", 0.0), errors="coerce").fillna(0.0)
        out["lgbm_binary_hold_score"] = 1.0 - out["lgbm_binary_shift_score"]
        out["lgbm_best_shift_score"] = out["lgbm_binary_shift_score"]
        return out

    b = bin_df.copy().reset_index(drop=True)

    aligned = (
        len(out) == len(b)
        and "session" in out.columns
        and "session" in b.columns
        and "time" in out.columns
        and "time" in b.columns
        and out["session"].astype(str).equals(b["session"].astype(str))
        and np.allclose(
            pd.to_numeric(out["time"], errors="coerce").to_numpy(dtype=float),
            pd.to_numeric(b["time"], errors="coerce").to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-6,
            equal_nan=False,
        )
    )

    if aligned:
        out["lgbm_binary_shift_score"] = pd.to_numeric(
            b["lgbm_binary_shift_score"],
            errors="coerce",
        ).fillna(0.0).to_numpy(dtype=np.float32)

        out["lgbm_binary_hold_score"] = pd.to_numeric(
            b.get("lgbm_binary_hold_score", 1.0 - b["lgbm_binary_shift_score"]),
            errors="coerce",
        ).fillna(0.0).to_numpy(dtype=np.float32)

    else:
        key_cols = ["session", "time", "split"]
        join_cols = [c for c in key_cols if c in out.columns and c in b.columns]

        b_small = b[join_cols + ["lgbm_binary_shift_score", "lgbm_binary_hold_score"]].copy()
        out = out.merge(b_small, on=join_cols, how="left")

    out["lgbm_binary_shift_score"] = pd.to_numeric(
        out["lgbm_binary_shift_score"],
        errors="coerce",
    ).fillna(pd.to_numeric(out.get("lgbm_shift_score", 0.0), errors="coerce")).fillna(0.0)

    out["lgbm_binary_shift_score"] = out["lgbm_binary_shift_score"].clip(0.0, 1.0)

    out["lgbm_binary_hold_score"] = pd.to_numeric(
        out.get("lgbm_binary_hold_score", 1.0 - out["lgbm_binary_shift_score"]),
        errors="coerce",
    ).fillna(1.0 - out["lgbm_binary_shift_score"]).clip(0.0, 1.0)

    if "lgbm_shift_score" not in out.columns:
        out["lgbm_shift_score"] = out["lgbm_binary_shift_score"]

    out["lgbm_best_shift_score"] = out["lgbm_binary_shift_score"]

    return out

if FORCE_RETRAIN_LIGHTGBM_BINARY and not CACHE_SAFE_RESUME_RUN:
    for p in [
        LGBM_BINARY_MODEL_PATH,
        LGBM_BINARY_MODEL_META_PATH,
        LGBM_BINARY_PARAMS_PATH,
        LGBM_BINARY_IMPORTANCE_PATH,
        LGBM_BINARY_HISTORY_PATH,
        LGBM_BINARY_TRAIN_SCORES_PATH,
        LGBM_BINARY_VAL_SCORES_PATH,
        LGBM_BINARY_TEST_SCORES_PATH,
        LGBM_BINARY_TRAIN_SCORES_META_PATH,
        LGBM_BINARY_VAL_SCORES_META_PATH,
        LGBM_BINARY_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale binary LightGBM artifact:", p)
        except Exception as e:
            print("Could not remove binary LightGBM artifact:", p, e)

LGBM_BINARY_MODEL = None

if LGBM_BINARY_MODEL_PATH.exists() and not FORCE_RETRAIN_LIGHTGBM_BINARY:
    LGBM_BINARY_MODEL = load_lightgbm_binary_shift_model()

    if LGBM_BINARY_MODEL is None:
        msg = (
            "Binary LightGBM model exists but is incompatible with the current features/config:\n"
            f"{LGBM_BINARY_MODEL_PATH}\n\n"
            "In cache-safe resume mode, this cell will not retrain automatically."
        )

        if CACHE_SAFE_RESUME_RUN and _all_binary_lgbm_score_caches_exist():
            print("⚠️", msg)
            print("Binary LightGBM score caches exist, so continuing in score-cache-only mode.")
        elif CACHE_SAFE_RESUME_RUN:
            raise RuntimeError(
                msg
                + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM_BINARY=True only if you intentionally want to retrain."
            )
        else:
            print("No compatible binary LightGBM model found. Training a new one.")
            LGBM_BINARY_MODEL = train_lightgbm_binary_shift_verifier()

elif FORCE_RETRAIN_LIGHTGBM_BINARY:
    print("FORCE_RETRAIN_LIGHTGBM_BINARY=True, training binary LightGBM.")
    LGBM_BINARY_MODEL = train_lightgbm_binary_shift_verifier()

else:
    msg = (
        "No binary LightGBM model found:\n"
        f"{LGBM_BINARY_MODEL_PATH}\n\n"
        "In cache-safe resume mode, this cell will not train automatically."
    )

    if CACHE_SAFE_RESUME_RUN and _all_binary_lgbm_score_caches_exist():
        print("⚠️", msg)
        print("Binary LightGBM score caches exist, so continuing in score-cache-only mode.")
    elif CACHE_SAFE_RESUME_RUN:
        raise FileNotFoundError(
            msg
            + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_LIGHTGBM_BINARY=True only if you intentionally want to train."
        )
    else:
        LGBM_BINARY_MODEL = train_lightgbm_binary_shift_verifier()

LGBM_BINARY_TRAIN_SCORES = lgbm_binary_scores(
    LGBM_BINARY_MODEL,
    SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin(["train", "synth_train", "real_diffusion", "synth_diffusion"])
    ].copy(),
    LGBM_BINARY_TRAIN_SCORES_PATH,
    LGBM_BINARY_TRAIN_SCORES_META_PATH,
    "train",
)

LGBM_BINARY_VAL_SCORES = lgbm_binary_scores(
    LGBM_BINARY_MODEL,
    SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "val"].copy(),
    LGBM_BINARY_VAL_SCORES_PATH,
    LGBM_BINARY_VAL_SCORES_META_PATH,
    "val",
)

LGBM_BINARY_TEST_SCORES = lgbm_binary_scores(
    LGBM_BINARY_MODEL,
    SYS2_FEATURES_ALL[SYS2_FEATURES_ALL["split"].astype(str) == "test"].copy(),
    LGBM_BINARY_TEST_SCORES_PATH,
    LGBM_BINARY_TEST_SCORES_META_PATH,
    "test",
)

LGBM_TRAIN_SCORES = _merge_binary_scores_into_lgbm_scores(LGBM_TRAIN_SCORES, LGBM_BINARY_TRAIN_SCORES)
LGBM_VAL_SCORES = _merge_binary_scores_into_lgbm_scores(LGBM_VAL_SCORES, LGBM_BINARY_VAL_SCORES)
LGBM_TEST_SCORES = _merge_binary_scores_into_lgbm_scores(LGBM_TEST_SCORES, LGBM_BINARY_TEST_SCORES)

LGBM_TRAIN_SCORES.to_pickle(LGBM_TRAIN_SCORES_PATH)
LGBM_VAL_SCORES.to_pickle(LGBM_VAL_SCORES_PATH)
LGBM_TEST_SCORES.to_pickle(LGBM_TEST_SCORES_PATH)

LGBM_HAS_BINARY_SHIFT_SCORE = True

globals()["LGBM_BINARY_MODEL"] = LGBM_BINARY_MODEL
globals()["LGBM_BINARY_TRAIN_SCORES"] = LGBM_BINARY_TRAIN_SCORES
globals()["LGBM_BINARY_VAL_SCORES"] = LGBM_BINARY_VAL_SCORES
globals()["LGBM_BINARY_TEST_SCORES"] = LGBM_BINARY_TEST_SCORES
globals()["LGBM_TRAIN_SCORES"] = LGBM_TRAIN_SCORES
globals()["LGBM_VAL_SCORES"] = LGBM_VAL_SCORES
globals()["LGBM_TEST_SCORES"] = LGBM_TEST_SCORES
globals()["LGBM_HAS_BINARY_SHIFT_SCORE"] = LGBM_HAS_BINARY_SHIFT_SCORE

print("LGBM_BINARY_TRAIN_SCORES:", LGBM_BINARY_TRAIN_SCORES.shape)
print("LGBM_BINARY_VAL_SCORES:", LGBM_BINARY_VAL_SCORES.shape)
print("LGBM_BINARY_TEST_SCORES:", LGBM_BINARY_TEST_SCORES.shape)
print("Augmented LGBM_VAL_SCORES columns include binary:", "lgbm_binary_shift_score" in LGBM_VAL_SCORES.columns)

if len(LGBM_VAL_SCORES):
    val_known = LGBM_VAL_SCORES[LGBM_VAL_SCORES["label"].astype(int).isin([0, 1])].copy()

    if len(val_known):
        y_true = val_known["label"].astype(int).to_numpy()
        y_score = (
            pd.to_numeric(val_known["lgbm_binary_shift_score"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        y_pred = (y_score >= 0.50).astype(int)

        P, R, F1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        try:
            ap = float(average_precision_score(y_true, y_score))
        except Exception:
            ap = float("nan")

        try:
            auc = float(roc_auc_score(y_true, y_score))
        except Exception:
            auc = float("nan")

        print(
            f"Binary LightGBM Sys2 VAL row-level @0.50: "
            f"P={P:.4f} R={R:.4f} F1={F1:.4f} AP={ap:.4f} AUC={auc:.4f}"
        )

    display(LGBM_VAL_SCORES.head())

if len(LGBM_TEST_SCORES):
    print("TEST binary LightGBM label check — should be only -1:")
    display(LGBM_TEST_SCORES["label"].value_counts(dropna=False).to_frame("test_label_count"))

    test_labels = set(
        pd.to_numeric(LGBM_TEST_SCORES["label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
    )

    assert test_labels.issubset({-1}), f"TEST labels leaked into binary LightGBM scores: {test_labels}"

if LGBM_BINARY_IMPORTANCE_PATH.exists():
    print("Top binary LightGBM features by gain:")
    display(pd.read_csv(LGBM_BINARY_IMPORTANCE_PATH).head(20))

print("✅ Cell 19B complete: binary LightGBM SHIFT/HOLD verifier loaded/scored and merged.")

Binary LightGBM Sys2 settings:
  force_retrain: False
  cache_safe_resume: False
  strict_model_meta_check: False
  strict_score_meta_check: False
  n_features: 47


Training until validation scores don't improve for 200 rounds


[100]	train's binary_logloss: 0.377666	train's auc: 0.924776	val's binary_logloss: 0.291999	val's auc: 0.925582


[200]	train's binary_logloss: 0.327749	train's auc: 0.934422	val's binary_logloss: 0.235979	val's auc: 0.930492


[300]	train's binary_logloss: 0.307622	train's auc: 0.940339	val's binary_logloss: 0.219544	val's auc: 0.933752


...

TEST binary LightGBM label check — should be only -1:


,test_label_count
label,
-1,99307


Top binary LightGBM features by gain:


,feature,importance_gain,importance_split
0,ecapa_cos,512129.671570,9985
1,wavlm_score,272856.404417,9751
2,wavlm_p_NA,135442.174383,9206
3,wavlm_p_BI,90777.893851,9522
4,wavlm_center_score,88539.110536,9542
5,wavlm_p_ITT,79271.797355,9634
6,ecapa_right_rms,74404.039198,8243
7,wavlm_p_NTT,63155.347036,8532
8,energy_right,62873.304025,7517
9,pyannote_any_candidate_dist,54887.942272,6395


✅ Cell 19B complete: binary LightGBM SHIFT/HOLD verifier loaded/scored and merged.


In [26]:
# Event-level local candidate ranker / selector
import json
import time
import math
import gc
import os
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, List, Optional
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score

assert "LGBM_SYS2_DIR" in globals(), "Run Cell 19 first so LGBM_SYS2_DIR is available."
assert "FEATURE_COLS_LGBM" in globals(), "Run Cell 19 first so FEATURE_COLS_LGBM is available."
assert "SYS2_FEATURES_ALL" in globals(), "Run Cell 16 first so SYS2_FEATURES_ALL is available."
assert "LGBM_TRAIN_SCORES" in globals(), "Run Cell 19B first so LGBM_TRAIN_SCORES is available."
assert "LGBM_VAL_SCORES" in globals(), "Run Cell 19B first so LGBM_VAL_SCORES is available."
assert "LGBM_TEST_SCORES" in globals(), "Run Cell 19B first so LGBM_TEST_SCORES is available."
assert "TOL" in globals(), "Need TOL."

RANKER_VARIANT = "event_local_ranker_v2_no_gt_distance_leak"

RANKER_MODEL_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker.txt"
RANKER_MODEL_META_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker.meta.json"
RANKER_PARAMS_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker_params.json"
RANKER_IMPORTANCE_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker_feature_importance.csv"
RANKER_HISTORY_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker_eval_history.json"
RANKER_FEATURE_COLS_PATH = LGBM_SYS2_DIR / "lightgbm_event_local_ranker_feature_cols.json"

RANKER_TRAIN_SCORES_PATH = LGBM_SYS2_DIR / "ranker_train_scores.pkl"
RANKER_VAL_SCORES_PATH = LGBM_SYS2_DIR / "ranker_val_scores.pkl"
RANKER_TEST_SCORES_PATH = LGBM_SYS2_DIR / "ranker_test_scores.pkl"

RANKER_TRAIN_SCORES_META_PATH = LGBM_SYS2_DIR / "ranker_train_scores.meta.json"
RANKER_VAL_SCORES_META_PATH = LGBM_SYS2_DIR / "ranker_val_scores.meta.json"
RANKER_TEST_SCORES_META_PATH = LGBM_SYS2_DIR / "ranker_test_scores.meta.json"

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

FORCE_RETRAIN_EVENT_RANKER = bool(
    globals().get("FORCE_RETRAIN_EVENT_RANKER", False)
)

STRICT_RANKER_MODEL_META_CHECK = bool(
    globals().get("STRICT_RANKER_MODEL_META_CHECK", False)
)

STRICT_RANKER_SCORE_META_CHECK = bool(
    globals().get("STRICT_RANKER_SCORE_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_RETRAIN_EVENT_RANKER = False

globals()["FORCE_RETRAIN_EVENT_RANKER"] = FORCE_RETRAIN_EVENT_RANKER
globals()["STRICT_RANKER_MODEL_META_CHECK"] = STRICT_RANKER_MODEL_META_CHECK
globals()["STRICT_RANKER_SCORE_META_CHECK"] = STRICT_RANKER_SCORE_META_CHECK

RANKER_POS_WINDOW_S = float(globals().get("RANKER_POS_WINDOW_S", float(TOL)))
RANKER_LOCAL_CLUSTER_GAP_S = float(globals().get("RANKER_LOCAL_CLUSTER_GAP_S", max(0.35, float(TOL))))
RANKER_ATTACH_SCORE_MAX_DIST_S = float(globals().get("RANKER_ATTACH_SCORE_MAX_DIST_S", 0.05))

RANKER_NUM_BOOST_ROUND = int(globals().get("RANKER_NUM_BOOST_ROUND", 5000))
RANKER_EARLY_STOPPING_ROUNDS = int(globals().get("RANKER_EARLY_STOPPING_ROUNDS", 250))
RANKER_LEARNING_RATE = float(globals().get("RANKER_LEARNING_RATE", 0.015))
RANKER_NUM_LEAVES = int(globals().get("RANKER_NUM_LEAVES", 63))
RANKER_MIN_DATA_IN_LEAF = int(globals().get("RANKER_MIN_DATA_IN_LEAF", 15))
RANKER_FEATURE_FRACTION = float(globals().get("RANKER_FEATURE_FRACTION", 0.90))
RANKER_BAGGING_FRACTION = float(globals().get("RANKER_BAGGING_FRACTION", 0.85))
RANKER_BAGGING_FREQ = int(globals().get("RANKER_BAGGING_FREQ", 1))
RANKER_LAMBDA_L2 = float(globals().get("RANKER_LAMBDA_L2", 2.0))
RANKER_POS_WEIGHT_MULT = float(globals().get("RANKER_POS_WEIGHT_MULT", 1.30))
RANKER_LOG_EVERY = int(globals().get("RANKER_LOG_EVERY", 100))

RANKER_MAX_TRAIN_ROWS = int(
    globals().get("RANKER_MAX_TRAIN_ROWS", 300_000)
)
RANKER_NEGATIVES_PER_POSITIVE = int(
    globals().get("RANKER_NEGATIVES_PER_POSITIVE", 10)
)
RANKER_MAX_NEGATIVES_PER_SESSION = int(
    globals().get("RANKER_MAX_NEGATIVES_PER_SESSION", 250)
)
RANKER_MIN_NEGATIVES_PER_SESSION = int(
    globals().get("RANKER_MIN_NEGATIVES_PER_SESSION", 12)
)
RANKER_PREDICT_CHUNK_ROWS = int(
    globals().get("RANKER_PREDICT_CHUNK_ROWS", 40_000)
)
RANKER_MAX_BIN = int(
    globals().get("RANKER_MAX_BIN", 127)
)
RANKER_NUM_THREADS = int(
    globals().get(
        "RANKER_NUM_THREADS",
        max(1, min(8, (os.cpu_count() or 4) // 2)),
    )
)

RANKER_MAX_TRAIN_ROWS = max(10_000, RANKER_MAX_TRAIN_ROWS)
RANKER_NEGATIVES_PER_POSITIVE = max(1, RANKER_NEGATIVES_PER_POSITIVE)
RANKER_MAX_NEGATIVES_PER_SESSION = max(
    RANKER_MIN_NEGATIVES_PER_SESSION,
    RANKER_MAX_NEGATIVES_PER_SESSION,
)
RANKER_PREDICT_CHUNK_ROWS = max(2_000, RANKER_PREDICT_CHUNK_ROWS)
RANKER_MAX_BIN = max(31, RANKER_MAX_BIN)
RANKER_NUM_THREADS = max(1, RANKER_NUM_THREADS)

def _ranker_release_memory():
    gc.collect()
    try:
        if "torch" in globals() and torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

_ranker_release_memory()

RANKER_TRAIN_SPLITS = list(globals().get(
    "RANKER_TRAIN_SPLITS",
    ["train", "synth_train", "real_diffusion", "synth_diffusion"],
))

RANKER_VAL_SPLITS = ["val"]
RANKER_TEST_SPLITS = ["test"]

RANKER_EXCLUDED_FEATURE_COLS = {
    "ranker_label",
    "ranker_target_source",
    "ranker_nearest_gt_dist",
}

print("Event local ranker settings:")
print("  variant:", RANKER_VARIANT)
print("  force_retrain:", FORCE_RETRAIN_EVENT_RANKER)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_model_meta_check:", STRICT_RANKER_MODEL_META_CHECK)
print("  strict_score_meta_check:", STRICT_RANKER_SCORE_META_CHECK)
print("  max_train_rows:", RANKER_MAX_TRAIN_ROWS)
print("  negatives_per_positive:", RANKER_NEGATIVES_PER_POSITIVE)
print("  max_negatives_per_session:", RANKER_MAX_NEGATIVES_PER_SESSION)
print("  predict_chunk_rows:", RANKER_PREDICT_CHUNK_ROWS)
print("  LightGBM threads:", RANKER_NUM_THREADS)

def _ranker_json_safe(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return v if math.isfinite(v) else None
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _ranker_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_ranker_json_safe(v) for v in obj]
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _ranker_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {"exists": False, "path": str(path), "size": None, "mtime": None}

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _ranker_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        old = json.loads(Path(path).read_text())
        return old == expected
    except Exception:
        return False

def _norm_sid(x) -> str:
    return Path(str(x)).stem

def _prepare_ranker_base_frame(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()

    if "session" not in out.columns:
        raise ValueError("Ranker input is missing session column.")
    if "time" not in out.columns:
        raise ValueError("Ranker input is missing time column.")
    if "split" not in out.columns:
        raise ValueError("Ranker input is missing split column.")

    out["session"] = out["session"].astype(str)
    out["session_norm"] = out["session"].map(_norm_sid)
    out["time"] = pd.to_numeric(out["time"], errors="coerce")
    out["split"] = out["split"].astype(str)

    if "label" not in out.columns:
        out["label"] = -1

    out["label"] = pd.to_numeric(out["label"], errors="coerce").fillna(-1).astype(int)

    if "sample_weight" not in out.columns:
        out["sample_weight"] = 1.0

    out["sample_weight"] = pd.to_numeric(
        out["sample_weight"],
        errors="coerce",
    ).fillna(1.0).clip(lower=1e-6)

    out = out.dropna(subset=["session", "session_norm", "time", "split"]).reset_index(drop=True)

    for c in FEATURE_COLS_LGBM:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = (
            pd.to_numeric(out[c], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

    return out

def _concat_available_score_tables(names: List[str]) -> pd.DataFrame:
    parts = []

    for name in names:
        obj = globals().get(name, None)
        if isinstance(obj, pd.DataFrame) and len(obj):
            parts.append(obj.copy())

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    if "session" in out.columns:
        out["session"] = out["session"].astype(str)
        out["session_norm"] = out["session"].map(_norm_sid)

    if "time" in out.columns:
        out["time"] = pd.to_numeric(out["time"], errors="coerce")

    return out.dropna(subset=["session_norm", "time"]).reset_index(drop=True)

def _ranker_attach_nearest_score(
    base_df: pd.DataFrame,
    score_df: pd.DataFrame,
    score_col: str,
    out_col: str,
    max_dist: float = RANKER_ATTACH_SCORE_MAX_DIST_S,
) -> pd.DataFrame:
    out = base_df.copy()
    out = out.drop(columns=[out_col], errors="ignore")

    if score_df is None or len(score_df) == 0 or score_col not in score_df.columns:
        out[out_col] = np.nan
        return out

    if "session_norm" not in out.columns:
        out["session_norm"] = out["session"].astype(str).map(_norm_sid)

    work = score_df[["session", "time", score_col]].copy()
    work["session_norm"] = work["session"].astype(str).map(_norm_sid)
    work["time"] = pd.to_numeric(work["time"], errors="coerce")
    work[score_col] = pd.to_numeric(work[score_col], errors="coerce")
    work = (
        work.dropna(subset=["session_norm", "time", score_col])
        .drop_duplicates(["session_norm", "time"], keep="last")
        .reset_index(drop=True)
    )

    if len(work) == 0:
        out[out_col] = np.nan
        return out

    aligned = (
        len(out) == len(work)
        and out["session_norm"].astype(str).reset_index(drop=True).equals(
            work["session_norm"].astype(str).reset_index(drop=True)
        )
        and np.allclose(
            pd.to_numeric(out["time"], errors="coerce").to_numpy(dtype=float),
            work["time"].to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-6,
            equal_nan=False,
        )
    )

    if aligned:
        out[out_col] = work[score_col].to_numpy(dtype=np.float32)
        del work
        return out

    score_lookup = {
        str(sid): group[["time", score_col]].sort_values("time").reset_index(drop=True)
        for sid, group in work.groupby("session_norm", sort=False)
    }
    del work

    out["_ranker_row_id"] = np.arange(len(out), dtype=np.int64)
    merged_parts = []

    for sid, group in out.groupby("session_norm", sort=False):
        left = group.sort_values("time").copy()
        right = score_lookup.get(str(sid))

        if right is None or len(right) == 0:
            left[out_col] = np.nan
        else:
            left = pd.merge_asof(
                left,
                right,
                on="time",
                direction="nearest",
                tolerance=float(max_dist),
            ).rename(columns={score_col: out_col})

        merged_parts.append(left)

    result = (
        pd.concat(merged_parts, ignore_index=True)
        .sort_values("_ranker_row_id")
        .drop(columns=["_ranker_row_id"], errors="ignore")
        .reset_index(drop=True)
    )

    if out_col not in result.columns:
        result[out_col] = np.nan

    del out, merged_parts, score_lookup
    _ranker_release_memory()
    return result

def _collect_event_tables_for_splits(split_names: List[str]) -> pd.DataFrame:
    mapping = {
        "train": ["REAL_TRAIN_EVENTS"],
        "val": ["REAL_VAL_EVENTS"],
        "test": ["REAL_TEST_EVENTS"],
        "synth_train": ["SYNTH_EVENTS"],
        "real_diffusion": ["REAL_AUG_EVENTS"],
        "synth_diffusion": ["SYNTH_AUG_EVENTS"],
    }

    parts = []

    for split_name in split_names:
        for global_name in mapping.get(str(split_name), []):
            obj = globals().get(global_name, None)
            if isinstance(obj, pd.DataFrame) and len(obj):
                tmp = obj.copy()
                tmp["source_event_table"] = global_name
                tmp["ranker_event_split"] = str(split_name)
                parts.append(tmp)

    if not parts:
        return pd.DataFrame(columns=["session", "session_norm", "time", "label"])

    ev = pd.concat(parts, ignore_index=True)

    if "session" not in ev.columns or "time" not in ev.columns:
        return pd.DataFrame(columns=["session", "session_norm", "time", "label"])

    ev["session"] = ev["session"].astype(str)
    ev["session_norm"] = ev["session"].map(_norm_sid)
    ev["time"] = pd.to_numeric(ev["time"], errors="coerce")

    if "label" not in ev.columns:
        ev["label"] = 1

    ev["label"] = pd.to_numeric(ev["label"], errors="coerce").fillna(0).astype(int)

    ev = ev.dropna(subset=["session_norm", "time"])
    ev = ev[ev["label"] == 1].copy()
    ev = ev.sort_values(["session_norm", "time"]).reset_index(drop=True)

    return ev

def _assign_event_level_ranker_targets(
    cand_df: pd.DataFrame,
    split_names: List[str],
) -> pd.DataFrame:
    out = cand_df.copy()
    out["ranker_label"] = np.int8(0)
    out["ranker_nearest_gt_dist"] = np.nan
    out["ranker_target_source"] = "event_best_candidate"

    ev = _collect_event_tables_for_splits(split_names)
    event_lookup = {
        str(sid): np.sort(group["time"].to_numpy(dtype=np.float64))
        for sid, group in ev.groupby("session_norm", sort=False)
    }

    selected_idx = []
    matched_event_count = 0

    for sid, group in out.groupby("session_norm", sort=False):
        gt_times = event_lookup.get(str(sid))
        if gt_times is None or len(gt_times) == 0:
            continue

        ordered = group.sort_values("time")
        cand_times = ordered["time"].to_numpy(dtype=np.float64)
        cand_indices = ordered.index.to_numpy(dtype=np.int64)

        if len(cand_times) == 0:
            continue

        pos = np.searchsorted(gt_times, cand_times, side="left")
        left_pos = np.clip(pos - 1, 0, len(gt_times) - 1)
        right_pos = np.clip(pos, 0, len(gt_times) - 1)
        left_dist = np.abs(cand_times - gt_times[left_pos])
        right_dist = np.abs(cand_times - gt_times[right_pos])
        nearest_dist = np.minimum(left_dist, right_dist).astype(np.float32)
        out.loc[cand_indices, "ranker_nearest_gt_dist"] = nearest_dist

        cpos = np.searchsorted(cand_times, gt_times, side="left")
        cleft = np.clip(cpos - 1, 0, len(cand_times) - 1)
        cright = np.clip(cpos, 0, len(cand_times) - 1)
        dleft = np.abs(gt_times - cand_times[cleft])
        dright = np.abs(gt_times - cand_times[cright])
        choose_right = dright < dleft
        best_local = np.where(choose_right, cright, cleft)
        best_dist = np.where(choose_right, dright, dleft)

        valid = best_dist <= float(RANKER_POS_WINDOW_S)
        if np.any(valid):
            selected_idx.extend(cand_indices[best_local[valid]].tolist())
            matched_event_count += int(np.sum(valid))

    if selected_idx:
        selected_unique = np.unique(np.asarray(selected_idx, dtype=np.int64))
        out.loc[selected_unique, "ranker_label"] = np.int8(1)
        print(
            f"Ranker target from event tables: positives={int(out['ranker_label'].sum())}, "
            f"matched_events={matched_event_count}, rows={len(out)}"
        )
    elif "label" in out.columns and out["label"].isin([0, 1]).any():
        out["ranker_label"] = (
            pd.to_numeric(out["label"], errors="coerce")
            .fillna(0)
            .clip(0, 1)
            .astype(np.int8)
        )
        out["ranker_target_source"] = "fallback_candidate_label"
        print(
            "⚠️ Ranker event-table target assignment found no positives. "
            "Falling back to candidate label."
        )
    else:
        out["ranker_target_source"] = "unavailable"

    del ev, event_lookup, selected_idx
    _ranker_release_memory()
    return out

def _add_ranker_local_features(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()
    out = out.sort_values(["session_norm", "time"]).reset_index(drop=True)

    for c in [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "temporal_shift_score",
        "lgbm_best_shift_score",
    ]:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["lgbm_binary_shift_score"] = (
        out["lgbm_binary_shift_score"]
        .fillna(out["lgbm_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["lgbm_shift_score"] = (
        out["lgbm_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["temporal_shift_score"] = (
        out["temporal_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["lgbm_shift_score"])
        .fillna(0.0)
    )

    out["lgbm_best_shift_score"] = (
        out["lgbm_best_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["lgbm_shift_score"])
        .fillna(0.0)
    )

    rows = []

    for sid, g in out.groupby("session_norm", sort=False):
        g = g.sort_values("time").copy()
        times = g["time"].to_numpy(dtype=float)

        if len(g) == 0:
            continue

        gaps = np.diff(times)
        cluster_ids = np.zeros(len(g), dtype=int)

        if len(gaps):
            cluster_ids[1:] = np.cumsum(gaps > float(RANKER_LOCAL_CLUSTER_GAP_S))

        g["ranker_cluster_id"] = cluster_ids

        prev_gap = np.full(len(g), 999.0, dtype=np.float32)
        next_gap = np.full(len(g), 999.0, dtype=np.float32)

        if len(g) > 1:
            prev_gap[1:] = np.diff(times)
            next_gap[:-1] = np.diff(times)

        g["ranker_prev_gap_s"] = np.clip(prev_gap, 0.0, 999.0)
        g["ranker_next_gap_s"] = np.clip(next_gap, 0.0, 999.0)
        g["ranker_min_neighbor_gap_s"] = np.minimum(g["ranker_prev_gap_s"], g["ranker_next_gap_s"])

        cluster_parts = []

        for cid, cg in g.groupby("ranker_cluster_id", sort=False):
            cg = cg.copy()

            cluster_size = len(cg)
            cluster_span = float(cg["time"].max() - cg["time"].min()) if cluster_size else 0.0

            cg["ranker_cluster_size"] = int(cluster_size)
            cg["ranker_cluster_span_s"] = float(cluster_span)
            cg["ranker_cluster_pos_frac"] = (
                np.arange(cluster_size, dtype=np.float32) / max(1.0, float(cluster_size - 1))
                if cluster_size > 1
                else 0.0
            )

            for score_col in [
                "lgbm_binary_shift_score",
                "lgbm_shift_score",
                "temporal_shift_score",
                "lgbm_best_shift_score",
            ]:
                vals = pd.to_numeric(cg[score_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)

                local_max = float(np.max(vals)) if len(vals) else 0.0
                local_mean = float(np.mean(vals)) if len(vals) else 0.0
                local_std = float(np.std(vals)) if len(vals) else 0.0

                cg[f"ranker_{score_col}_local_max"] = local_max
                cg[f"ranker_{score_col}_local_mean"] = local_mean
                cg[f"ranker_{score_col}_local_std"] = local_std
                cg[f"ranker_{score_col}_minus_local_max"] = vals - local_max
                cg[f"ranker_{score_col}_minus_local_mean"] = vals - local_mean

                cg[f"ranker_{score_col}_rank_desc"] = (
                    pd.Series(vals).rank(method="first", ascending=False).to_numpy(dtype=np.float32)
                )

                cg[f"ranker_{score_col}_rank_frac"] = (
                    (cg[f"ranker_{score_col}_rank_desc"].astype(float) - 1.0)
                    / max(1.0, float(cluster_size - 1))
                )

            cluster_parts.append(cg)

        rows.append(pd.concat(cluster_parts, ignore_index=False))

    out = pd.concat(rows, ignore_index=False).sort_index().reset_index(drop=True)

    out["ranker_binary_minus_multiclass"] = out["lgbm_binary_shift_score"] - out["lgbm_shift_score"]
    out["ranker_binary_minus_temporal"] = out["lgbm_binary_shift_score"] - out["temporal_shift_score"]
    out["ranker_multiclass_minus_temporal"] = out["lgbm_shift_score"] - out["temporal_shift_score"]

    out["ranker_score_mean_3way"] = (
        out["lgbm_binary_shift_score"] + out["lgbm_shift_score"] + out["temporal_shift_score"]
    ) / 3.0

    out["ranker_score_max_3way"] = np.maximum.reduce([
        out["lgbm_binary_shift_score"].to_numpy(dtype=float),
        out["lgbm_shift_score"].to_numpy(dtype=float),
        out["temporal_shift_score"].to_numpy(dtype=float),
    ])

    out["ranker_score_min_3way"] = np.minimum.reduce([
        out["lgbm_binary_shift_score"].to_numpy(dtype=float),
        out["lgbm_shift_score"].to_numpy(dtype=float),
        out["temporal_shift_score"].to_numpy(dtype=float),
    ])

    return out

def _build_ranker_frame(
    split_names: List[str],
    include_target: bool,
) -> pd.DataFrame:
    part = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin([str(s) for s in split_names])
    ].copy()

    part = _prepare_ranker_base_frame(part)

    if len(part) == 0:
        return pd.DataFrame()

    split_set = {str(x) for x in split_names}

    if split_set.issubset(set(RANKER_TRAIN_SPLITS)):
        lgbm_names = ["LGBM_TRAIN_SCORES"]
        temporal_names = ["TEMPORAL_TRAIN_SCORES"]
    elif split_set.issubset(set(RANKER_VAL_SPLITS)):
        lgbm_names = ["LGBM_VAL_SCORES"]
        temporal_names = ["TEMPORAL_VAL_SCORES"]
    elif split_set.issubset(set(RANKER_TEST_SPLITS)):
        lgbm_names = ["LGBM_TEST_SCORES"]
        temporal_names = ["TEMPORAL_TEST_SCORES"]
    else:
        lgbm_names = [
            "LGBM_TRAIN_SCORES",
            "LGBM_VAL_SCORES",
            "LGBM_TEST_SCORES",
        ]
        temporal_names = [
            "TEMPORAL_TRAIN_SCORES",
            "TEMPORAL_VAL_SCORES",
            "TEMPORAL_TEST_SCORES",
        ]

    lgbm_scores_all = _concat_available_score_tables(lgbm_names)
    temporal_scores_all = _concat_available_score_tables(temporal_names)

    for score_col in ["lgbm_shift_score", "lgbm_binary_shift_score", "lgbm_best_shift_score"]:
        part = _ranker_attach_nearest_score(
            part,
            lgbm_scores_all,
            score_col=score_col,
            out_col=score_col,
            max_dist=RANKER_ATTACH_SCORE_MAX_DIST_S,
        )

    part = _ranker_attach_nearest_score(
        part,
        temporal_scores_all,
        score_col="temporal_shift_score",
        out_col="temporal_shift_score",
        max_dist=RANKER_ATTACH_SCORE_MAX_DIST_S,
    )

    part = _add_ranker_local_features(part)

    if include_target:
        part = _assign_event_level_ranker_targets(part, split_names=split_names)
    else:
        part["ranker_label"] = -1
        part["ranker_target_source"] = "test_unlabelled"
        part["ranker_nearest_gt_dist"] = np.nan

    return part.reset_index(drop=True)

def _sample_ranker_training_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    work = df.copy()
    work["_ranker_hardness"] = (
        pd.to_numeric(
            work.get(
                "lgbm_binary_shift_score",
                work.get("lgbm_shift_score", 0.0),
            ),
            errors="coerce",
        )
        .fillna(0.0)
        .astype(np.float32)
    )

    positives = work[work["ranker_label"].astype(int).eq(1)].copy()
    selected_negative_indices = []

    for sid, group in work.groupby("session_norm", sort=False):
        neg = group[group["ranker_label"].astype(int).eq(0)]
        if len(neg) == 0:
            continue

        n_pos = int(group["ranker_label"].astype(int).sum())
        keep_n = max(
            int(RANKER_MIN_NEGATIVES_PER_SESSION),
            int(n_pos * RANKER_NEGATIVES_PER_POSITIVE),
        )
        keep_n = min(
            int(RANKER_MAX_NEGATIVES_PER_SESSION),
            int(keep_n),
            int(len(neg)),
        )

        chosen = (
            neg.sort_values(
                ["_ranker_hardness", "time"],
                ascending=[False, True],
            )
            .head(keep_n)
            .index
            .tolist()
        )
        selected_negative_indices.extend(chosen)

    negatives = work.loc[
        np.unique(np.asarray(selected_negative_indices, dtype=np.int64))
    ].copy() if selected_negative_indices else work.iloc[0:0].copy()

    sampled = pd.concat([positives, negatives], ignore_index=True)

    if len(sampled) > int(RANKER_MAX_TRAIN_ROWS):
        positive_rows = sampled[sampled["ranker_label"].astype(int).eq(1)].copy()
        negative_rows = sampled[sampled["ranker_label"].astype(int).eq(0)].copy()

        negative_budget = max(
            0,
            int(RANKER_MAX_TRAIN_ROWS) - len(positive_rows),
        )
        negative_rows = (
            negative_rows.sort_values(
                ["_ranker_hardness", "session_norm", "time"],
                ascending=[False, True, True],
            )
            .head(negative_budget)
        )
        sampled = pd.concat(
            [positive_rows, negative_rows],
            ignore_index=True,
        )

    sampled = (
        sampled.drop(columns=["_ranker_hardness"], errors="ignore")
        .sort_values(["session_norm", "time"])
        .reset_index(drop=True)
    )

    print(
        "Ranker TRAIN memory-bounded sample:",
        {
            "full_rows": int(len(work)),
            "sampled_rows": int(len(sampled)),
            "positives": int(sampled["ranker_label"].sum()),
            "negatives": int((sampled["ranker_label"] == 0).sum()),
        },
    )

    del work, positives, negatives, selected_negative_indices
    _ranker_release_memory()
    return sampled

def _infer_ranker_feature_cols(df: pd.DataFrame) -> List[str]:
    extra_prefixes = [
        "ranker_",
    ]

    explicit_cols = [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "lgbm_best_shift_score",
        "temporal_shift_score",
    ]

    cols = list(FEATURE_COLS_LGBM)

    for c in explicit_cols:
        if c in df.columns and c not in cols:
            cols.append(c)

    for c in df.columns:
        c = str(c)

        if c in RANKER_EXCLUDED_FEATURE_COLS:
            continue

        if any(c.startswith(p) for p in extra_prefixes):
            if c not in cols:
                cols.append(c)

    numeric_cols = []

    for c in cols:
        if c not in df.columns:
            continue

        if str(c) in RANKER_EXCLUDED_FEATURE_COLS:
            continue

        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            numeric_cols.append(c)

    leaked = sorted(set(numeric_cols).intersection(RANKER_EXCLUDED_FEATURE_COLS))
    assert not leaked, f"Leaky/non-deployable ranker features included: {leaked}"

    return numeric_cols

def _ranker_params() -> Dict[str, Any]:
    return {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "learning_rate": float(RANKER_LEARNING_RATE),
        "num_leaves": int(RANKER_NUM_LEAVES),
        "min_data_in_leaf": int(RANKER_MIN_DATA_IN_LEAF),
        "feature_fraction": float(RANKER_FEATURE_FRACTION),
        "bagging_fraction": float(RANKER_BAGGING_FRACTION),
        "bagging_freq": int(RANKER_BAGGING_FREQ),
        "lambda_l2": float(RANKER_LAMBDA_L2),
        "verbosity": -1,
        "seed": int(SEED),
        "feature_pre_filter": False,
        "force_col_wise": True,
        "max_bin": int(RANKER_MAX_BIN),
        "num_threads": int(RANKER_NUM_THREADS),
    }

def _ranker_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RANKER_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "target": "event_level_best_candidate",
        "train_splits": [str(x) for x in RANKER_TRAIN_SPLITS],
        "val_splits": [str(x) for x in RANKER_VAL_SPLITS],
        "test_splits": [str(x) for x in RANKER_TEST_SPLITS],
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RANKER_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "pos_window_s": float(RANKER_POS_WINDOW_S),
        "local_cluster_gap_s": float(RANKER_LOCAL_CLUSTER_GAP_S),
        "attach_score_max_dist_s": float(RANKER_ATTACH_SCORE_MAX_DIST_S),
        "num_boost_round": int(RANKER_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(RANKER_EARLY_STOPPING_ROUNDS),
        "learning_rate": float(RANKER_LEARNING_RATE),
        "num_leaves": int(RANKER_NUM_LEAVES),
        "min_data_in_leaf": int(RANKER_MIN_DATA_IN_LEAF),
        "feature_fraction": float(RANKER_FEATURE_FRACTION),
        "bagging_fraction": float(RANKER_BAGGING_FRACTION),
        "bagging_freq": int(RANKER_BAGGING_FREQ),
        "lambda_l2": float(RANKER_LAMBDA_L2),
        "pos_weight_mult": float(RANKER_POS_WEIGHT_MULT),
        "sys2_features_rows": int(len(SYS2_FEATURES_ALL)),
        "test_labels_attached": False,
    }

def _ranker_core_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RANKER_VARIANT,
        "target": "event_level_best_candidate",
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RANKER_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "pos_window_s": float(RANKER_POS_WINDOW_S),
        "local_cluster_gap_s": float(RANKER_LOCAL_CLUSTER_GAP_S),
        "attach_score_max_dist_s": float(RANKER_ATTACH_SCORE_MAX_DIST_S),
    }

def _make_ranker_weights(df: pd.DataFrame) -> np.ndarray:
    y = (
        pd.to_numeric(df["ranker_label"], errors="coerce")
        .fillna(0)
        .astype(int)
        .clip(0, 1)
        .to_numpy()
    )

    if "sample_weight" in df.columns:
        base = (
            pd.to_numeric(df["sample_weight"], errors="coerce")
            .fillna(1.0)
            .to_numpy(dtype=np.float32)
        )
    else:
        base = np.ones(len(df), dtype=np.float32)

    base = np.maximum(base, 1e-6)

    counts = np.bincount(y, minlength=2).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    class_w = counts.sum() / counts
    class_w = class_w / max(1e-8, float(class_w.mean()))
    class_w[1] *= float(RANKER_POS_WEIGHT_MULT)

    w = base * class_w[y]
    w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
    w = np.maximum(w, 1e-6)

    return w.astype(np.float32)

def _clear_stale_ranker_score_caches():
    for p in [
        RANKER_TRAIN_SCORES_PATH,
        RANKER_VAL_SCORES_PATH,
        RANKER_TEST_SCORES_PATH,
        RANKER_TRAIN_SCORES_META_PATH,
        RANKER_VAL_SCORES_META_PATH,
        RANKER_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale ranker score cache:", p)
        except Exception as e:
            print("Could not remove stale ranker score cache:", p, e)

def _load_ranker_feature_cols_from_file_or_meta() -> Optional[List[str]]:
    if RANKER_FEATURE_COLS_PATH.exists():
        try:
            cols = json.loads(RANKER_FEATURE_COLS_PATH.read_text())
            leaked = sorted(set(cols).intersection(RANKER_EXCLUDED_FEATURE_COLS))
            if leaked:
                print("⚠️ Ranker feature file contains leaky features:", leaked)
                return None
            return list(cols)
        except Exception as e:
            print("Could not load ranker feature columns file:", e)

    for p in [
        RANKER_VAL_SCORES_META_PATH,
        RANKER_TEST_SCORES_META_PATH,
        RANKER_TRAIN_SCORES_META_PATH,
    ]:
        if not p.exists():
            continue

        try:
            meta = json.loads(p.read_text())
            cols = list(meta.get("feature_cols", []))
            leaked = sorted(set(cols).intersection(RANKER_EXCLUDED_FEATURE_COLS))
            if leaked:
                print("⚠️ Ranker score meta contains leaky features:", leaked)
                return None
            if cols:
                return cols
        except Exception:
            continue

    return None

def _ranker_score_cache_core_compatible(meta_path: Path, feature_cols: List[str]) -> bool:
    if not Path(meta_path).exists():
        return False

    try:
        meta = json.loads(Path(meta_path).read_text())
    except Exception:
        return False

    if meta.get("variant", None) != RANKER_VARIANT:
        return False

    if meta.get("target", None) != "event_level_best_candidate":
        return False

    saved_cols = list(meta.get("feature_cols", []))

    if saved_cols != list(feature_cols):
        return False

    leaked = sorted(set(saved_cols).intersection(RANKER_EXCLUDED_FEATURE_COLS))
    if leaked:
        return False

    saved_excluded = set(meta.get("excluded_feature_cols", []))
    if not set(RANKER_EXCLUDED_FEATURE_COLS).issubset(saved_excluded):
        return False

    return True

def _all_ranker_score_caches_usable(feature_cols: Optional[List[str]]) -> bool:
    if not feature_cols:
        return False

    checks = [
        (RANKER_TRAIN_SCORES_PATH, RANKER_TRAIN_SCORES_META_PATH),
        (RANKER_VAL_SCORES_PATH, RANKER_VAL_SCORES_META_PATH),
        (RANKER_TEST_SCORES_PATH, RANKER_TEST_SCORES_META_PATH),
    ]

    for data_path, meta_path in checks:
        if not Path(data_path).exists():
            return False
        if not _ranker_score_cache_core_compatible(meta_path, feature_cols):
            return False

    return True

def train_event_local_ranker():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError(
            "LightGBM is required. Install with: pip install -U lightgbm"
        ) from e

    _ranker_release_memory()

    train_df_full = _build_ranker_frame(
        RANKER_TRAIN_SPLITS,
        include_target=True,
    )
    assert len(train_df_full) > 0, "No ranker training rows."
    assert int(train_df_full["ranker_label"].sum()) > 0, (
        "Ranker train has zero positives."
    )

    train_df = _sample_ranker_training_rows(train_df_full)
    del train_df_full
    _ranker_release_memory()

    val_df = _build_ranker_frame(
        RANKER_VAL_SPLITS,
        include_target=True,
    )

    assert len(val_df) > 0, "No ranker validation rows."
    assert int(val_df["ranker_label"].sum()) > 0, (
        "Ranker VAL has zero positives."
    )

    feature_cols = _infer_ranker_feature_cols(train_df)
    assert len(feature_cols) > 0, "No usable ranker feature columns."

    leaked = sorted(
        set(feature_cols).intersection(
            RANKER_EXCLUDED_FEATURE_COLS
        )
    )
    assert not leaked, (
        f"Leaky/non-deployable ranker features included: {leaked}"
    )

    Xtr = np.empty(
        (len(train_df), len(feature_cols)),
        dtype=np.float32,
    )
    Xva = np.empty(
        (len(val_df), len(feature_cols)),
        dtype=np.float32,
    )

    for j, col in enumerate(feature_cols):
        Xtr[:, j] = (
            pd.to_numeric(train_df[col], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .to_numpy(dtype=np.float32)
        )
        Xva[:, j] = (
            pd.to_numeric(val_df[col], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .to_numpy(dtype=np.float32)
        )

    ytr = train_df["ranker_label"].astype(np.int8).to_numpy()
    yva = val_df["ranker_label"].astype(np.int8).to_numpy()
    wtr = _make_ranker_weights(train_df)
    wva = _make_ranker_weights(val_df)

    train_stats = {
        "train_rows": int(len(train_df)),
        "train_positive_rows": int(train_df["ranker_label"].sum()),
        "val_rows": int(len(val_df)),
        "val_positive_rows": int(val_df["ranker_label"].sum()),
        "train_target_source_counts": _ranker_json_safe(
            train_df["ranker_target_source"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "val_target_source_counts": _ranker_json_safe(
            val_df["ranker_target_source"]
            .value_counts(dropna=False)
            .to_dict()
        ),
    }

    params = _ranker_params()

    dtr = lgb.Dataset(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_name=feature_cols,
        params=params,
        free_raw_data=True,
    )
    dva = lgb.Dataset(
        Xva,
        label=yva,
        weight=wva,
        reference=dtr,
        feature_name=feature_cols,
        params=params,
        free_raw_data=True,
    )

    dtr.construct()
    dva.construct()

    del (
        train_df,
        val_df,
        Xtr,
        Xva,
        ytr,
        yva,
        wtr,
        wva,
    )
    _ranker_release_memory()

    evals_result = {}

    model = lgb.train(
        params,
        dtr,
        num_boost_round=int(RANKER_NUM_BOOST_ROUND),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=int(
                    RANKER_EARLY_STOPPING_ROUNDS
                ),
                first_metric_only=True,
                verbose=True,
            ),
            lgb.log_evaluation(
                period=int(RANKER_LOG_EVERY)
            ),
            lgb.record_evaluation(evals_result),
        ],
    )

    best_iter = int(
        getattr(model, "best_iteration", 0) or 0
    )

    if best_iter > 0:
        model.save_model(
            str(RANKER_MODEL_PATH),
            num_iteration=best_iter,
        )
    else:
        model.save_model(str(RANKER_MODEL_PATH))

    meta = _ranker_model_config(feature_cols)
    meta.update({
        "best_iteration": int(best_iter),
        "best_score": _ranker_json_safe(
            getattr(model, "best_score", {})
        ),
        "trained_time": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        "memory_bounded_training": True,
        "max_train_rows": int(
            RANKER_MAX_TRAIN_ROWS
        ),
        "negatives_per_positive": int(
            RANKER_NEGATIVES_PER_POSITIVE
        ),
        "max_negatives_per_session": int(
            RANKER_MAX_NEGATIVES_PER_SESSION
        ),
        **train_stats,
    })

    RANKER_MODEL_META_PATH.write_text(
        json.dumps(
            meta,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
    )
    RANKER_PARAMS_PATH.write_text(
        json.dumps(
            params,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
    )
    RANKER_HISTORY_PATH.write_text(
        json.dumps(
            _ranker_json_safe(evals_result),
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
    )
    RANKER_FEATURE_COLS_PATH.write_text(
        json.dumps(
            feature_cols,
            indent=2,
            sort_keys=True,
        )
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance_gain": model.feature_importance(
            importance_type="gain"
        ),
        "importance_split": model.feature_importance(
            importance_type="split"
        ),
    }).sort_values(
        "importance_gain",
        ascending=False,
    )
    imp.to_csv(
        RANKER_IMPORTANCE_PATH,
        index=False,
    )

    print("Saved ranker model:", RANKER_MODEL_PATH)
    print("Saved ranker meta:", RANKER_MODEL_META_PATH)
    print("Saved ranker feature columns:", RANKER_FEATURE_COLS_PATH)
    print("Saved ranker feature importance:", RANKER_IMPORTANCE_PATH)
    print("Ranker best iteration:", best_iter)

    del dtr, dva, imp, evals_result
    _ranker_release_memory()
    _clear_stale_ranker_score_caches()

    return model, feature_cols

def load_event_local_ranker():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    if not RANKER_MODEL_PATH.exists() or not RANKER_MODEL_META_PATH.exists() or not RANKER_FEATURE_COLS_PATH.exists():
        return None, None

    try:
        feature_cols = json.loads(RANKER_FEATURE_COLS_PATH.read_text())

        leaked = sorted(set(feature_cols).intersection(RANKER_EXCLUDED_FEATURE_COLS))
        if leaked:
            print(f"⚠️ Existing ranker contains leaky/non-deployable features: {leaked}")
            return None, None

        saved_meta = json.loads(RANKER_MODEL_META_PATH.read_text())

        if STRICT_RANKER_MODEL_META_CHECK:
            expected = _ranker_model_config(feature_cols)
            comparable_keys = [
                "variant",
                "run_version",
                "seed",
                "target",
                "train_splits",
                "val_splits",
                "test_splits",
                "feature_cols",
                "excluded_feature_cols",
                "n_features",
                "pos_window_s",
                "local_cluster_gap_s",
                "attach_score_max_dist_s",
                "learning_rate",
                "num_leaves",
                "min_data_in_leaf",
                "feature_fraction",
                "bagging_fraction",
                "bagging_freq",
                "lambda_l2",
                "pos_weight_mult",
            ]
        else:
            expected = _ranker_core_model_config(feature_cols)
            comparable_keys = list(expected.keys())

        for k in comparable_keys:
            if saved_meta.get(k, None) != expected.get(k, None):
                print(f"⚠️ Existing ranker meta mismatch at key: {k}")
                print("  expected:", expected.get(k, None))
                print("  saved:   ", saved_meta.get(k, None))
                return None, None

    except Exception as e:
        print("⚠️ Could not read ranker meta/features:", e)
        return None, None

    model = lgb.Booster(model_file=str(RANKER_MODEL_PATH))
    print("Loaded event local ranker:", RANKER_MODEL_PATH)

    return model, feature_cols

def _ranker_score_config(split_name: str, df: pd.DataFrame, feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RANKER_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "split": str(split_name),
        "model_fingerprint": _ranker_file_fingerprint(RANKER_MODEL_PATH),
        "model_meta_fingerprint": _ranker_file_fingerprint(RANKER_MODEL_META_PATH),
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RANKER_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "n_rows": int(len(df)) if df is not None else 0,
        "target": "event_level_best_candidate",
        "cache_format": "pickle",
    }

def _validate_ranker_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "ranker_shift_score",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"Ranker {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

def ranker_scores(
    model,
    feature_cols: List[str],
    split_names: List[str],
    out_path: Path,
    meta_path: Path,
    split_name: str,
    include_target: bool,
) -> pd.DataFrame:
    expected = _ranker_score_config(
        split_name,
        SYS2_FEATURES_ALL,
        feature_cols,
    )

    if out_path.exists() and not FORCE_RETRAIN_EVENT_RANKER:
        meta_matches = _ranker_meta_matches(
            meta_path,
            expected,
        )
        core_compatible = _ranker_score_cache_core_compatible(
            meta_path,
            feature_cols,
        )

        if meta_matches:
            print(
                f"Ranker score metadata matches for {split_name}."
            )
        elif not core_compatible:
            print(
                f"Ranker {split_name} cache is not core-compatible "
                "and will be rebuilt."
            )
        elif STRICT_RANKER_SCORE_META_CHECK:
            print(
                f"Strict metadata check requested; rebuilding "
                f"{split_name} ranker scores."
            )
        else:
            print(
                f"Loading compatible existing ranker {split_name} "
                f"scores from: {out_path}"
            )

        if meta_matches or (
            core_compatible
            and not STRICT_RANKER_SCORE_META_CHECK
        ):
            cached = pd.read_pickle(out_path)

            if not _validate_ranker_score_cache(
                cached,
                split_name,
            ):
                raise RuntimeError(
                    f"Cached ranker {split_name} scores are incompatible: "
                    f"{out_path}"
                )

            print(
                f"Loaded ranker {split_name} scores:",
                cached.shape,
            )
            return cached

    leaked = sorted(
        set(feature_cols).intersection(
            RANKER_EXCLUDED_FEATURE_COLS
        )
    )
    assert not leaked, (
        f"Leaky ranker scoring features: {leaked}"
    )

    if model is None:
        raise FileNotFoundError(
            f"No event-ranker model is available for {split_name} scoring."
        )

    work = _build_ranker_frame(
        split_names,
        include_target=include_target,
    )

    if work is None or len(work) == 0:
        return pd.DataFrame()

    best_iter = int(
        getattr(model, "best_iteration", 0) or 0
    )
    score = np.empty(len(work), dtype=np.float32)

    for start in range(
        0,
        len(work),
        int(RANKER_PREDICT_CHUNK_ROWS),
    ):
        stop = min(
            len(work),
            start + int(RANKER_PREDICT_CHUNK_ROWS),
        )

        X_chunk = np.empty(
            (stop - start, len(feature_cols)),
            dtype=np.float32,
        )

        chunk = work.iloc[start:stop]

        for j, col in enumerate(feature_cols):
            if col in chunk.columns:
                X_chunk[:, j] = (
                    pd.to_numeric(
                        chunk[col],
                        errors="coerce",
                    )
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                    .to_numpy(dtype=np.float32)
                )
            else:
                X_chunk[:, j] = 0.0

        if best_iter > 0:
            pred = model.predict(
                X_chunk,
                num_iteration=best_iter,
            )
        else:
            pred = model.predict(X_chunk)

        score[start:stop] = np.clip(
            np.asarray(
                pred,
                dtype=np.float32,
            ).reshape(-1),
            0.0,
            1.0,
        )

        del X_chunk, chunk, pred
        if start and start % (
            int(RANKER_PREDICT_CHUNK_ROWS) * 5
        ) == 0:
            _ranker_release_memory()

    base_cols = [
        "session",
        "time",
        "split",
        "label",
        "action",
        "action_id",
        "audio_path",
        "ranker_label",
        "ranker_target_source",
        "ranker_nearest_gt_dist",
        "ranker_cluster_id",
        "ranker_cluster_size",
        "ranker_cluster_span_s",
        "lgbm_binary_shift_score",
        "lgbm_shift_score",
        "temporal_shift_score",
    ]
    base_cols = [
        col for col in base_cols
        if col in work.columns
    ]

    out = (
        work[base_cols]
        .copy()
        .reset_index(drop=True)
    )
    out["ranker_shift_score"] = score

    out.to_pickle(out_path)
    meta_path.write_text(
        json.dumps(
            expected,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
    )

    print(
        f"Saved ranker {split_name} scores:",
        out_path,
        out.shape,
    )

    del work, score
    _ranker_release_memory()
    return out

def _merge_ranker_scores_into_lgbm_scores(main_df: pd.DataFrame, ranker_df: pd.DataFrame) -> pd.DataFrame:
    if main_df is None or len(main_df) == 0:
        return pd.DataFrame()

    out = main_df.copy().reset_index(drop=True)

    for c in ["ranker_shift_score"]:
        if c in out.columns:
            out = out.drop(columns=[c])

    if ranker_df is None or len(ranker_df) == 0 or "ranker_shift_score" not in ranker_df.columns:
        out["ranker_shift_score"] = pd.to_numeric(
            out.get("lgbm_binary_shift_score", out.get("lgbm_shift_score", 0.0)),
            errors="coerce",
        ).fillna(0.0)
        return out

    b = ranker_df.copy().reset_index(drop=True)

    aligned = (
        len(out) == len(b)
        and "session" in out.columns
        and "session" in b.columns
        and "time" in out.columns
        and "time" in b.columns
        and out["session"].astype(str).map(_norm_sid).equals(b["session"].astype(str).map(_norm_sid))
        and np.allclose(
            pd.to_numeric(out["time"], errors="coerce").to_numpy(dtype=float),
            pd.to_numeric(b["time"], errors="coerce").to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-6,
            equal_nan=False,
        )
    )

    if aligned:
        out["ranker_shift_score"] = pd.to_numeric(
            b["ranker_shift_score"],
            errors="coerce",
        ).fillna(0.0).to_numpy(dtype=np.float32)

    else:
        out["session_norm_tmp"] = out["session"].astype(str).map(_norm_sid)
        b["session_norm_tmp"] = b["session"].astype(str).map(_norm_sid)

        b_small = b[["session_norm_tmp", "time", "ranker_shift_score"]].copy()
        b_small["time"] = pd.to_numeric(b_small["time"], errors="coerce")

        out = out.merge(
            b_small,
            on=["session_norm_tmp", "time"],
            how="left",
        ).drop(columns=["session_norm_tmp"], errors="ignore")

    out["ranker_shift_score"] = (
        pd.to_numeric(out["ranker_shift_score"], errors="coerce")
        .fillna(pd.to_numeric(out.get("lgbm_binary_shift_score", out.get("lgbm_shift_score", 0.0)), errors="coerce"))
        .fillna(0.0)
        .clip(0.0, 1.0)
    )

    return out

if FORCE_RETRAIN_EVENT_RANKER and not CACHE_SAFE_RESUME_RUN:
    for p in [
        RANKER_MODEL_PATH,
        RANKER_MODEL_META_PATH,
        RANKER_PARAMS_PATH,
        RANKER_IMPORTANCE_PATH,
        RANKER_HISTORY_PATH,
        RANKER_FEATURE_COLS_PATH,
        RANKER_TRAIN_SCORES_PATH,
        RANKER_VAL_SCORES_PATH,
        RANKER_TEST_SCORES_PATH,
        RANKER_TRAIN_SCORES_META_PATH,
        RANKER_VAL_SCORES_META_PATH,
        RANKER_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale ranker artifact:", p)
        except Exception as e:
            print("Could not remove ranker artifact:", p, e)

EVENT_RANKER_MODEL = None
EVENT_RANKER_FEATURE_COLS = None

if RANKER_MODEL_PATH.exists() and not FORCE_RETRAIN_EVENT_RANKER:
    EVENT_RANKER_MODEL, EVENT_RANKER_FEATURE_COLS = load_event_local_ranker()

    if EVENT_RANKER_MODEL is None:
        cache_feature_cols = _load_ranker_feature_cols_from_file_or_meta()

        msg = (
            "Event local ranker model exists but is incompatible with the current v2 no-leak configuration:\n"
            f"{RANKER_MODEL_PATH}\n\n"
            "In cache-safe resume mode, this cell will not retrain automatically."
        )

        if CACHE_SAFE_RESUME_RUN and _all_ranker_score_caches_usable(cache_feature_cols):
            print("⚠️", msg)
            print("Compatible v2 ranker score caches exist, so continuing in score-cache-only mode.")
            EVENT_RANKER_FEATURE_COLS = cache_feature_cols
        elif CACHE_SAFE_RESUME_RUN:
            raise RuntimeError(
                msg
                + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_EVENT_RANKER=True only if you intentionally want to retrain."
            )
        else:
            print("No compatible event local ranker found. Training a new one.")
            EVENT_RANKER_MODEL, EVENT_RANKER_FEATURE_COLS = train_event_local_ranker()

elif FORCE_RETRAIN_EVENT_RANKER:
    print("FORCE_RETRAIN_EVENT_RANKER=True, training event local ranker.")
    EVENT_RANKER_MODEL, EVENT_RANKER_FEATURE_COLS = train_event_local_ranker()

else:
    cache_feature_cols = _load_ranker_feature_cols_from_file_or_meta()

    msg = (
        "No event local ranker model found:\n"
        f"{RANKER_MODEL_PATH}\n\n"
        "In cache-safe resume mode, this cell will not train automatically."
    )

    if CACHE_SAFE_RESUME_RUN and _all_ranker_score_caches_usable(cache_feature_cols):
        print("⚠️", msg)
        print("Compatible v2 ranker score caches exist, so continuing in score-cache-only mode.")
        EVENT_RANKER_FEATURE_COLS = cache_feature_cols
    elif CACHE_SAFE_RESUME_RUN:
        raise FileNotFoundError(
            msg
            + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_EVENT_RANKER=True only if you intentionally want to train."
        )
    else:
        EVENT_RANKER_MODEL, EVENT_RANKER_FEATURE_COLS = train_event_local_ranker()

assert EVENT_RANKER_FEATURE_COLS is not None and len(EVENT_RANKER_FEATURE_COLS) > 0, (
    "EVENT_RANKER_FEATURE_COLS could not be resolved."
)

leaked_final = sorted(set(EVENT_RANKER_FEATURE_COLS).intersection(RANKER_EXCLUDED_FEATURE_COLS))
assert not leaked_final, f"Leaky/non-deployable ranker features survived final feature list: {leaked_final}"

RANKER_TRAIN_SCORES = ranker_scores(
    EVENT_RANKER_MODEL,
    EVENT_RANKER_FEATURE_COLS,
    RANKER_TRAIN_SPLITS,
    RANKER_TRAIN_SCORES_PATH,
    RANKER_TRAIN_SCORES_META_PATH,
    "train",
    include_target=True,
)

RANKER_VAL_SCORES = ranker_scores(
    EVENT_RANKER_MODEL,
    EVENT_RANKER_FEATURE_COLS,
    RANKER_VAL_SPLITS,
    RANKER_VAL_SCORES_PATH,
    RANKER_VAL_SCORES_META_PATH,
    "val",
    include_target=True,
)

RANKER_TEST_SCORES = ranker_scores(
    EVENT_RANKER_MODEL,
    EVENT_RANKER_FEATURE_COLS,
    RANKER_TEST_SPLITS,
    RANKER_TEST_SCORES_PATH,
    RANKER_TEST_SCORES_META_PATH,
    "test",
    include_target=False,
)

LGBM_TRAIN_SCORES = _merge_ranker_scores_into_lgbm_scores(LGBM_TRAIN_SCORES, RANKER_TRAIN_SCORES)
LGBM_VAL_SCORES = _merge_ranker_scores_into_lgbm_scores(LGBM_VAL_SCORES, RANKER_VAL_SCORES)
LGBM_TEST_SCORES = _merge_ranker_scores_into_lgbm_scores(LGBM_TEST_SCORES, RANKER_TEST_SCORES)

LGBM_TRAIN_SCORES.to_pickle(LGBM_TRAIN_SCORES_PATH)
LGBM_VAL_SCORES.to_pickle(LGBM_VAL_SCORES_PATH)
LGBM_TEST_SCORES.to_pickle(LGBM_TEST_SCORES_PATH)

LGBM_HAS_EVENT_RANKER_SCORE = True

print("RANKER_TRAIN_SCORES:", RANKER_TRAIN_SCORES.shape)
print("RANKER_VAL_SCORES:", RANKER_VAL_SCORES.shape)
print("RANKER_TEST_SCORES:", RANKER_TEST_SCORES.shape)
print("Augmented LGBM_VAL_SCORES includes ranker:", "ranker_shift_score" in LGBM_VAL_SCORES.columns)
print("Ranker feature count:", len(EVENT_RANKER_FEATURE_COLS))
print("Excluded from ranker features:", sorted(RANKER_EXCLUDED_FEATURE_COLS))
print("ranker_nearest_gt_dist used as feature:", "ranker_nearest_gt_dist" in EVENT_RANKER_FEATURE_COLS)

assert "ranker_nearest_gt_dist" not in EVENT_RANKER_FEATURE_COLS, (
    "ranker_nearest_gt_dist is a GT-derived diagnostic column and must not be used as a ranker feature."
)

if len(RANKER_VAL_SCORES):
    val_known = RANKER_VAL_SCORES[
        RANKER_VAL_SCORES["ranker_label"].astype(int).isin([0, 1])
    ].copy()

    if len(val_known) and int(val_known["ranker_label"].sum()) > 0:
        y_true = val_known["ranker_label"].astype(int).to_numpy()
        y_score = (
            pd.to_numeric(val_known["ranker_shift_score"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        y_pred = (y_score >= 0.50).astype(int)

        P, R, F1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        try:
            ap = float(average_precision_score(y_true, y_score))
        except Exception:
            ap = float("nan")

        try:
            auc = float(roc_auc_score(y_true, y_score))
        except Exception:
            auc = float("nan")

        print(f"Event local ranker VAL row-level @0.50: P={P:.4f} R={R:.4f} F1={F1:.4f} AP={ap:.4f} AUC={auc:.4f}")

    print("Ranker VAL target source counts:")
    display(RANKER_VAL_SCORES["ranker_target_source"].value_counts(dropna=False).to_frame("n"))

    display(RANKER_VAL_SCORES.head())

if len(RANKER_TEST_SCORES):
    print("TEST ranker label check — should be only -1:")
    display(RANKER_TEST_SCORES["ranker_label"].value_counts(dropna=False).to_frame("test_ranker_label_count"))

    test_labels = set(
        pd.to_numeric(RANKER_TEST_SCORES["ranker_label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
    )

    assert test_labels.issubset({-1}), f"TEST labels leaked into event ranker scores: {test_labels}"

if RANKER_IMPORTANCE_PATH.exists():
    print("Top event local ranker features by gain:")
    imp_df = pd.read_csv(RANKER_IMPORTANCE_PATH)
    display(imp_df.head(25))

    leaked_importance = set(imp_df["feature"].astype(str)).intersection(RANKER_EXCLUDED_FEATURE_COLS)
    assert not leaked_importance, f"Leaky features found in ranker importance table: {sorted(leaked_importance)}"

globals()["EVENT_RANKER_MODEL"] = EVENT_RANKER_MODEL
globals()["EVENT_RANKER_FEATURE_COLS"] = EVENT_RANKER_FEATURE_COLS
globals()["RANKER_TRAIN_SCORES"] = RANKER_TRAIN_SCORES
globals()["RANKER_VAL_SCORES"] = RANKER_VAL_SCORES
globals()["RANKER_TEST_SCORES"] = RANKER_TEST_SCORES
globals()["RANKER_MODEL_PATH"] = RANKER_MODEL_PATH
globals()["RANKER_MODEL_META_PATH"] = RANKER_MODEL_META_PATH
globals()["RANKER_PARAMS_PATH"] = RANKER_PARAMS_PATH
globals()["RANKER_HISTORY_PATH"] = RANKER_HISTORY_PATH
globals()["RANKER_IMPORTANCE_PATH"] = RANKER_IMPORTANCE_PATH
globals()["RANKER_FEATURE_COLS_PATH"] = RANKER_FEATURE_COLS_PATH
globals()["RANKER_TRAIN_SCORES_PATH"] = RANKER_TRAIN_SCORES_PATH
globals()["RANKER_VAL_SCORES_PATH"] = RANKER_VAL_SCORES_PATH
globals()["RANKER_TEST_SCORES_PATH"] = RANKER_TEST_SCORES_PATH
globals()["LGBM_TRAIN_SCORES"] = LGBM_TRAIN_SCORES
globals()["LGBM_VAL_SCORES"] = LGBM_VAL_SCORES
globals()["LGBM_TEST_SCORES"] = LGBM_TEST_SCORES
globals()["LGBM_HAS_EVENT_RANKER_SCORE"] = LGBM_HAS_EVENT_RANKER_SCORE

RANKER_TRAIN_SCORE_ROWS = int(len(RANKER_TRAIN_SCORES))
globals()["RANKER_TRAIN_SCORE_ROWS"] = RANKER_TRAIN_SCORE_ROWS

try:
    EVENT_RANKER_MODEL = None
    globals()["EVENT_RANKER_MODEL"] = None
except Exception:
    pass

_ranker_release_memory()

print(
    "✅ Cell 19C complete: memory-safe event-level local candidate ranker "
    "loaded/trained, scored in chunks, and merged without GT-distance leakage."
)

Event local ranker settings:
  variant: event_local_ranker_v2_no_gt_distance_leak
  force_retrain: False
  cache_safe_resume: False
  strict_model_meta_check: False
  strict_score_meta_check: False
  max_train_rows: 300000
  negatives_per_positive: 10
  max_negatives_per_session: 250
  predict_chunk_rows: 40000
  LightGBM threads: 8


Ranker target from event tables: positives=28716, matched_events=29124, rows=206097


Ranker TRAIN memory-bounded sample: {'full_rows': 206097, 'sampled_rows': 182687, 'positives': 28716, 'negatives': 153971}


Ranker target from event tables: positives=457, matched_events=458, rows=7968


Training until validation scores don't improve for 250 rounds


...

TEST ranker label check — should be only -1:


,test_ranker_label_count
ranker_label,
-1,99307


Top event local ranker features by gain:


,feature,importance_gain,importance_split
0,lgbm_binary_shift_score,879977.205944,158
1,lgbm_best_shift_score,70539.146904,10
2,lgbm_shift_score,32713.897689,352
3,ranker_lgbm_binary_shift_score_minus_local_max,22664.490655,152
4,wavlm_center_score,17534.708432,331
5,wavlm_score,16518.173324,308
6,ranker_binary_minus_multiclass,4647.144936,106
7,wavlm_p_NTT,4206.932961,170
8,ranker_lgbm_binary_shift_score_local_max,3690.274028,74
9,ecapa_cos,2993.373763,173


✅ Cell 19C complete: memory-safe event-level local candidate ranker loaded/trained, scored in chunks, and merged without GT-distance leakage.


In [27]:
# Role-Relative Future Activity Projection (RR-FAP)
import json
import time
import math
import numpy as np
import pandas as pd

import gc

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

from pathlib import Path
from typing import Dict, Any, List, Optional
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score

assert "LGBM_SYS2_DIR" in globals(), "Run Cell 19 first so LGBM_SYS2_DIR is available."
assert "FEATURE_COLS_LGBM" in globals(), "Run Cell 19 first so FEATURE_COLS_LGBM is available."
assert "SYS2_FEATURES_ALL" in globals(), "Run Cell 16 first so SYS2_FEATURES_ALL is available."
assert "LGBM_TRAIN_SCORES" in globals(), "Run Cell 19B/19C first so LGBM_TRAIN_SCORES is available."
assert "LGBM_VAL_SCORES" in globals(), "Run Cell 19B/19C first so LGBM_VAL_SCORES is available."
assert "LGBM_TEST_SCORES" in globals(), "Run Cell 19B/19C first so LGBM_TEST_SCORES is available."
assert "TOL" in globals(), "Need TOL."

RR_FAP_VARIANT = "rr_fap_v1_audio_text_role_relative_projection"

RR_FAP_DIR = LGBM_SYS2_DIR / "rr_fap_role_relative_projection"
RR_FAP_DIR.mkdir(parents=True, exist_ok=True)

RR_FAP_MODEL_PATH = RR_FAP_DIR / "lightgbm_rr_fap_role_shift_selector.txt"
RR_FAP_MODEL_META_PATH = RR_FAP_DIR / "lightgbm_rr_fap_role_shift_selector.meta.json"
RR_FAP_PARAMS_PATH = RR_FAP_DIR / "lightgbm_rr_fap_params.json"
RR_FAP_FEATURE_COLS_PATH = RR_FAP_DIR / "lightgbm_rr_fap_feature_cols.json"
RR_FAP_IMPORTANCE_PATH = RR_FAP_DIR / "lightgbm_rr_fap_feature_importance.csv"
RR_FAP_HISTORY_PATH = RR_FAP_DIR / "lightgbm_rr_fap_eval_history.json"

RR_FAP_TRAIN_SCORES_PATH = RR_FAP_DIR / "rr_fap_train_scores.pkl"
RR_FAP_VAL_SCORES_PATH = RR_FAP_DIR / "rr_fap_val_scores.pkl"
RR_FAP_TEST_SCORES_PATH = RR_FAP_DIR / "rr_fap_test_scores.pkl"

RR_FAP_TRAIN_SCORES_META_PATH = RR_FAP_DIR / "rr_fap_train_scores.meta.json"
RR_FAP_VAL_SCORES_META_PATH = RR_FAP_DIR / "rr_fap_val_scores.meta.json"
RR_FAP_TEST_SCORES_META_PATH = RR_FAP_DIR / "rr_fap_test_scores.meta.json"

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

FORCE_RETRAIN_RR_FAP = bool(globals().get("FORCE_RETRAIN_RR_FAP", False))

STRICT_RR_FAP_MODEL_META_CHECK = bool(
    globals().get("STRICT_RR_FAP_MODEL_META_CHECK", False)
)

STRICT_RR_FAP_SCORE_META_CHECK = bool(
    globals().get("STRICT_RR_FAP_SCORE_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_RETRAIN_RR_FAP = False

globals()["FORCE_RETRAIN_RR_FAP"] = FORCE_RETRAIN_RR_FAP
globals()["STRICT_RR_FAP_MODEL_META_CHECK"] = STRICT_RR_FAP_MODEL_META_CHECK
globals()["STRICT_RR_FAP_SCORE_META_CHECK"] = STRICT_RR_FAP_SCORE_META_CHECK

RR_FAP_POS_WINDOW_S = float(globals().get("RR_FAP_POS_WINDOW_S", float(TOL)))
RR_FAP_CLUSTER_GAP_S = float(globals().get("RR_FAP_CLUSTER_GAP_S", max(0.75, float(TOL))))
RR_FAP_ATTACH_SCORE_MAX_DIST_S = float(globals().get("RR_FAP_ATTACH_SCORE_MAX_DIST_S", 0.05))

RR_FAP_TRAIN_SPLITS = list(globals().get(
    "RR_FAP_TRAIN_SPLITS",
    ["train", "synth_train", "real_diffusion", "synth_diffusion"],
))
RR_FAP_VAL_SPLITS = ["val"]
RR_FAP_TEST_SPLITS = ["test"]

RR_FAP_NUM_BOOST_ROUND = int(globals().get("RR_FAP_NUM_BOOST_ROUND", 5000))
RR_FAP_EARLY_STOPPING_ROUNDS = int(globals().get("RR_FAP_EARLY_STOPPING_ROUNDS", 250))
RR_FAP_LEARNING_RATE = float(globals().get("RR_FAP_LEARNING_RATE", 0.015))
RR_FAP_NUM_LEAVES = int(globals().get("RR_FAP_NUM_LEAVES", 63))
RR_FAP_MIN_DATA_IN_LEAF = int(globals().get("RR_FAP_MIN_DATA_IN_LEAF", 15))
RR_FAP_FEATURE_FRACTION = float(globals().get("RR_FAP_FEATURE_FRACTION", 0.92))
RR_FAP_BAGGING_FRACTION = float(globals().get("RR_FAP_BAGGING_FRACTION", 0.85))
RR_FAP_BAGGING_FREQ = int(globals().get("RR_FAP_BAGGING_FREQ", 1))
RR_FAP_LAMBDA_L2 = float(globals().get("RR_FAP_LAMBDA_L2", 2.0))
RR_FAP_POS_WEIGHT_MULT = float(globals().get("RR_FAP_POS_WEIGHT_MULT", 1.30))
RR_FAP_LOG_EVERY = int(globals().get("RR_FAP_LOG_EVERY", 100))

RR_FAP_EXCLUDED_FEATURE_COLS = {
    "rr_label",
    "rr_target_source",
    "rr_nearest_gt_dist",
    "rr_gt_action",
    "rr_gt_prev_speaker",
    "rr_gt_next_speaker",
    "rr_gt_gap",
    "rr_gt_overlap",
    "rr_gt_prev_dur",
    "rr_gt_next_dur",
    "rr_cluster_id",
    "session_norm",
}

print("RR-FAP settings:")
print("  variant:", RR_FAP_VARIANT)
print("  force_retrain:", FORCE_RETRAIN_RR_FAP)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_model_meta_check:", STRICT_RR_FAP_MODEL_META_CHECK)
print("  strict_score_meta_check:", STRICT_RR_FAP_SCORE_META_CHECK)

def _rr_json_safe(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return v if math.isfinite(v) else None
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _rr_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_rr_json_safe(v) for v in obj]
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _rr_norm_sid(x) -> str:
    return Path(str(x)).stem

def _rr_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {"exists": False, "path": str(path), "size": None, "mtime": None}

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _rr_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        return json.loads(Path(path).read_text()) == expected
    except Exception:
        return False

def _rr_num(df: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if df is None or len(df) == 0:
        return pd.Series(dtype=float)

    if col not in df.columns:
        return pd.Series(np.full(len(df), float(default)), index=df.index, dtype=float)

    return (
        pd.to_numeric(df[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(float(default))
        .astype(float)
    )

def _rr_clip01(x) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
    return np.clip(arr, 0.0, 1.0)

def _rr_concat_score_tables(names: List[str]) -> pd.DataFrame:
    parts = []

    for name in names:
        obj = globals().get(name, None)

        if isinstance(obj, pd.DataFrame) and len(obj):
            tmp = obj.copy()

            if "session" in tmp.columns:
                tmp["session"] = tmp["session"].astype(str)
                tmp["session_norm"] = tmp["session"].map(_rr_norm_sid)

            if "time" in tmp.columns:
                tmp["time"] = pd.to_numeric(tmp["time"], errors="coerce")

            parts.append(tmp)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    return out.dropna(subset=["session_norm", "time"]).reset_index(drop=True)

def _rr_score_table_names_for_splits(split_names: List[str], family: str) -> List[str]:
    split_set = {str(s) for s in split_names}

    train_like = {
        "train",
        "synth_train",
        "real_diffusion",
        "synth_diffusion",
        "aug_train",
    }

    names = []

    if family == "lgbm":
        if split_set.intersection(train_like):
            names.append("LGBM_TRAIN_SCORES")
        if "val" in split_set:
            names.append("LGBM_VAL_SCORES")
        if "test" in split_set:
            names.append("LGBM_TEST_SCORES")

    elif family == "temporal":
        if split_set.intersection(train_like):
            names.append("TEMPORAL_TRAIN_SCORES")
        if "val" in split_set:
            names.append("TEMPORAL_VAL_SCORES")
        if "test" in split_set:
            names.append("TEMPORAL_TEST_SCORES")

    return list(dict.fromkeys(names))

def _rr_attach_nearest_score(
    base_df: pd.DataFrame,
    score_df: pd.DataFrame,
    score_col: str,
    out_col: str,
    max_dist: float = RR_FAP_ATTACH_SCORE_MAX_DIST_S,
) -> pd.DataFrame:
    out = base_df.copy()

    if score_df is None or len(score_df) == 0 or score_col not in score_df.columns:
        out[out_col] = np.nan
        return out

    if "session_norm" not in out.columns:
        out["session_norm"] = out["session"].astype(str).map(_rr_norm_sid)

    work = score_df.copy()

    if "session_norm" not in work.columns:
        if "session" not in work.columns:
            out[out_col] = np.nan
            return out
        work["session_norm"] = work["session"].astype(str).map(_rr_norm_sid)

    work["time"] = pd.to_numeric(work["time"], errors="coerce")
    work[score_col] = pd.to_numeric(work[score_col], errors="coerce")
    work = work.dropna(subset=["session_norm", "time", score_col]).reset_index(drop=True)

    if len(work) == 0:
        out[out_col] = np.nan
        return out

    out = out.drop(columns=[out_col], errors="ignore")

    out["_rr_row_id"] = np.arange(len(out), dtype=int)
    merged_parts = []

    for sid, g in out.groupby("session_norm", sort=False):
        left = g.sort_values("time").copy()
        right = work[work["session_norm"] == str(sid)].sort_values("time").copy()

        if len(right) == 0:
            left[out_col] = np.nan
            merged_parts.append(left)
            continue

        merged = pd.merge_asof(
            left,
            right[["time", score_col]],
            on="time",
            direction="nearest",
            tolerance=float(max_dist),
        ).rename(columns={score_col: out_col})

        merged_parts.append(merged)

    out = pd.concat(merged_parts, ignore_index=True)
    out = (
        out
        .sort_values("_rr_row_id")
        .drop(columns=["_rr_row_id"], errors="ignore")
        .reset_index(drop=True)
    )

    if out_col not in out.columns:
        out[out_col] = np.nan

    return out

def _rr_collect_event_tables(split_names: List[str]) -> pd.DataFrame:
    mapping = {
        "train": ["REAL_TRAIN_EVENTS"],
        "val": ["REAL_VAL_EVENTS"],
        "test": ["REAL_TEST_EVENTS"],
        "synth_train": ["SYNTH_EVENTS"],
        "real_diffusion": ["REAL_AUG_EVENTS"],
        "synth_diffusion": ["SYNTH_AUG_EVENTS"],
    }

    parts = []

    for split_name in split_names:
        for global_name in mapping.get(str(split_name), []):
            obj = globals().get(global_name, None)

            if isinstance(obj, pd.DataFrame) and len(obj):
                tmp = obj.copy()
                tmp["rr_event_table"] = global_name
                tmp["rr_event_split"] = str(split_name)
                parts.append(tmp)

    if not parts:
        return pd.DataFrame(columns=["session", "session_norm", "time", "label"])

    ev = pd.concat(parts, ignore_index=True)

    if "session" not in ev.columns or "time" not in ev.columns:
        return pd.DataFrame(columns=["session", "session_norm", "time", "label"])

    ev["session"] = ev["session"].astype(str)
    ev["session_norm"] = ev["session"].map(_rr_norm_sid)
    ev["time"] = pd.to_numeric(ev["time"], errors="coerce")

    if "label" not in ev.columns:
        ev["label"] = 1

    ev["label"] = pd.to_numeric(ev["label"], errors="coerce").fillna(0).astype(int)
    ev = ev.dropna(subset=["session_norm", "time"])
    ev = ev[ev["label"] == 1].copy()
    ev = ev.sort_values(["session_norm", "time"]).reset_index(drop=True)

    return ev

def _rr_prepare_base_frame(split_names: List[str]) -> pd.DataFrame:
    part = SYS2_FEATURES_ALL[
        SYS2_FEATURES_ALL["split"].astype(str).isin([str(s) for s in split_names])
    ].copy()

    if part is None or len(part) == 0:
        return pd.DataFrame()

    part["session"] = part["session"].astype(str)
    part["session_norm"] = part["session"].map(_rr_norm_sid)
    part["time"] = pd.to_numeric(part["time"], errors="coerce")
    part["split"] = part["split"].astype(str)

    if "label" not in part.columns:
        part["label"] = -1

    part["label"] = pd.to_numeric(part["label"], errors="coerce").fillna(-1).astype(int)

    if "sample_weight" not in part.columns:
        part["sample_weight"] = 1.0

    part["sample_weight"] = (
        pd.to_numeric(part["sample_weight"], errors="coerce")
        .fillna(1.0)
        .clip(lower=1e-6)
    )

    part = part.dropna(subset=["session_norm", "time"]).reset_index(drop=True)

    lgbm_scores_all = _rr_concat_score_tables(
        _rr_score_table_names_for_splits(split_names, family="lgbm")
    )

    temporal_scores_all = _rr_concat_score_tables(
        _rr_score_table_names_for_splits(split_names, family="temporal")
    )

    for score_col in [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "lgbm_best_shift_score",
        "ranker_shift_score",
    ]:
        part = _rr_attach_nearest_score(
            part,
            lgbm_scores_all,
            score_col=score_col,
            out_col=score_col,
            max_dist=RR_FAP_ATTACH_SCORE_MAX_DIST_S,
        )

    part = _rr_attach_nearest_score(
        part,
        temporal_scores_all,
        score_col="temporal_shift_score",
        out_col="temporal_shift_score",
        max_dist=RR_FAP_ATTACH_SCORE_MAX_DIST_S,
    )

    for c in [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "lgbm_best_shift_score",
        "ranker_shift_score",
        "temporal_shift_score",
    ]:
        if c not in part.columns:
            part[c] = np.nan
        part[c] = pd.to_numeric(part[c], errors="coerce")

    part["lgbm_binary_shift_score"] = (
        part["lgbm_binary_shift_score"]
        .fillna(part["lgbm_shift_score"])
        .fillna(part["temporal_shift_score"])
        .fillna(0.0)
    )

    part["lgbm_shift_score"] = (
        part["lgbm_shift_score"]
        .fillna(part["lgbm_binary_shift_score"])
        .fillna(part["temporal_shift_score"])
        .fillna(0.0)
    )

    part["temporal_shift_score"] = (
        part["temporal_shift_score"]
        .fillna(part["lgbm_binary_shift_score"])
        .fillna(part["lgbm_shift_score"])
        .fillna(0.0)
    )

    part["ranker_shift_score"] = (
        part["ranker_shift_score"]
        .fillna(part["lgbm_binary_shift_score"])
        .fillna(part["lgbm_shift_score"])
        .fillna(part["temporal_shift_score"])
        .fillna(0.0)
    )

    part["lgbm_best_shift_score"] = (
        part["lgbm_best_shift_score"]
        .fillna(part["lgbm_binary_shift_score"])
        .fillna(part["lgbm_shift_score"])
        .fillna(0.0)
    )

    return part.reset_index(drop=True)

def _rr_add_role_relative_features(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()
    out = out.sort_values(["session_norm", "time"]).reset_index(drop=True)

    qwen_shift = _rr_clip01(
        _rr_num(out, "qwen_shift_score", 0.0)
        + _rr_num(out, "qwen_p_NTT", 0.0)
        + _rr_num(out, "qwen_p_ITT", 0.0)
    )

    qwen_hold = _rr_clip01(
        _rr_num(out, "qwen_p_NA", 0.0)
        + _rr_num(out, "qwen_p_BC", 0.0)
        + _rr_num(out, "qwen_p_BI", 0.0)
    )

    qwen_overlap = _rr_clip01(
        _rr_num(out, "qwen_backchannel_score", 0.0)
        + _rr_num(out, "qwen_interruption_score", 0.0)
        + _rr_num(out, "qwen_p_BC", 0.0)
        + _rr_num(out, "qwen_p_ITT", 0.0)
    )

    wavlm_shift = _rr_clip01(
        _rr_num(out, "wavlm_score", 0.0)
        + _rr_num(out, "wavlm_center_score", 0.0)
        + _rr_num(out, "wavlm_p_NTT", 0.0)
        + _rr_num(out, "wavlm_p_ITT", 0.0)
    )

    py_change = np.maximum.reduce([
        _rr_clip01(_rr_num(out, "pyannote_change_score", 0.0)),
        _rr_clip01(1.0 - (_rr_num(out, "pyannote_change_dist", 999.0) / 0.50)),
        _rr_clip01(_rr_num(out, "source_pyannote_change", 0.0)),
    ])

    py_overlap = np.maximum.reduce([
        _rr_clip01(_rr_num(out, "pyannote_overlap_score", 0.0)),
        _rr_clip01(1.0 - (_rr_num(out, "pyannote_overlap_dist", 999.0) / 0.50)),
        _rr_clip01(_rr_num(out, "pyannote_local_overlap_flag", 0.0)),
        _rr_clip01(_rr_num(out, "source_pyannote_overlap", 0.0)),
    ])

    asr_boundary = np.maximum(
        _rr_clip01(_rr_num(out, "asr_boundary_score", 0.0)),
        _rr_clip01(1.0 - (_rr_num(out, "asr_boundary_dist", 999.0) / 0.60)),
    )

    ecapa_avail = _rr_clip01(_rr_num(out, "ecapa_available", 0.0))
    ecapa_same = _rr_clip01(_rr_num(out, "ecapa_cos", 0.0))
    ecapa_change = _rr_clip01((1.0 - ecapa_same) * ecapa_avail)

    active_before = _rr_clip01(_rr_num(out, "pyannote_active_before", 0.0) / 3.0)
    active_after = _rr_clip01(_rr_num(out, "pyannote_active_after", 0.0) / 3.0)
    active_delta = _rr_num(out, "pyannote_active_delta", 0.0)

    energy_left = np.log1p(np.maximum(0.0, _rr_num(out, "energy_left", 0.0)))
    energy_right = np.log1p(np.maximum(0.0, _rr_num(out, "energy_right", 0.0)))
    energy_delta = np.tanh(_rr_num(out, "energy_delta", 0.0))
    energy_ratio = np.tanh(_rr_num(out, "energy_ratio", 0.0) - 1.0)

    current_activity = _rr_clip01(
        (active_before + _rr_clip01(energy_left) + _rr_clip01(_rr_num(out, "asr_has_left", 0.0))) / 3.0
    )

    next_activity = _rr_clip01(
        (active_after + _rr_clip01(energy_right) + _rr_clip01(_rr_num(out, "asr_has_right", 0.0))) / 3.0
    )

    current_drop = _rr_clip01(
        np.maximum(0.0, active_before - active_after)
        + np.maximum(0.0, -energy_delta)
        + np.maximum(0.0, -energy_ratio)
    )

    next_enters = _rr_clip01(
        np.maximum(0.0, active_after - active_before)
        + np.maximum(0.0, energy_delta)
        + np.maximum(0.0, energy_ratio)
        + qwen_shift
        + py_change
    )

    role_switch_proxy = _rr_clip01(
        0.22 * qwen_shift
        + 0.22 * wavlm_shift
        + 0.20 * py_change
        + 0.16 * ecapa_change
        + 0.12 * asr_boundary
        + 0.08 * _rr_clip01(_rr_num(out, "lgbm_binary_shift_score", 0.0))
    )

    same_holder_continue_proxy = _rr_clip01(
        0.30 * qwen_hold
        + 0.25 * ecapa_same
        + 0.20 * _rr_clip01(
            _rr_num(out, "lgbm_binary_shift_score", 0.0) * 0.0
            + (1.0 - _rr_num(out, "lgbm_binary_shift_score", 0.0))
        )
        + 0.15 * _rr_clip01(1.0 - py_change)
        + 0.10 * current_activity
    )

    source_agree = (
        (qwen_shift >= 0.25).astype(int)
        + (wavlm_shift >= 0.10).astype(int)
        + (py_change >= 0.25).astype(int)
        + (asr_boundary >= 0.25).astype(int)
        + (ecapa_change >= 0.40).astype(int)
    )

    out["rr_qwen_shift_proxy"] = qwen_shift
    out["rr_qwen_hold_proxy"] = qwen_hold
    out["rr_qwen_overlap_proxy"] = qwen_overlap
    out["rr_wavlm_shift_proxy"] = wavlm_shift
    out["rr_pyannote_change_proxy"] = py_change
    out["rr_pyannote_overlap_proxy"] = py_overlap
    out["rr_asr_boundary_proxy"] = asr_boundary
    out["rr_ecapa_change_proxy"] = ecapa_change
    out["rr_ecapa_same_proxy"] = ecapa_same
    out["rr_current_activity_proxy"] = current_activity
    out["rr_next_activity_proxy"] = next_activity
    out["rr_current_drop_proxy"] = current_drop
    out["rr_next_enters_proxy"] = next_enters
    out["rr_role_switch_proxy"] = role_switch_proxy
    out["rr_same_holder_continue_proxy"] = same_holder_continue_proxy
    out["rr_overlap_future_proxy"] = _rr_clip01(0.60 * py_overlap + 0.40 * qwen_overlap)
    out["rr_source_agree_count"] = source_agree.astype(np.float32)
    out["rr_source_agree_frac"] = source_agree.astype(np.float32) / 5.0
    out["rr_text_audio_agreement"] = _rr_clip01(1.0 - np.abs(qwen_shift - wavlm_shift))
    out["rr_role_switch_minus_continue"] = role_switch_proxy - same_holder_continue_proxy
    out["rr_next_minus_current_activity"] = next_activity - current_activity
    out["rr_lgbm_binary_score"] = _rr_clip01(_rr_num(out, "lgbm_binary_shift_score", 0.0))
    out["rr_ranker_score"] = _rr_clip01(_rr_num(out, "ranker_shift_score", 0.0))
    out["rr_temporal_score"] = _rr_clip01(_rr_num(out, "temporal_shift_score", 0.0))
    out["rr_three_model_mean"] = (
        out["rr_lgbm_binary_score"] + out["rr_ranker_score"] + out["rr_temporal_score"]
    ) / 3.0
    out["rr_role_model_mean"] = (
        out["rr_role_switch_proxy"] + out["rr_lgbm_binary_score"] + out["rr_ranker_score"]
    ) / 3.0

    rows = []

    score_cols = [
        "rr_role_switch_proxy",
        "rr_lgbm_binary_score",
        "rr_ranker_score",
        "rr_temporal_score",
        "rr_qwen_shift_proxy",
        "rr_pyannote_change_proxy",
        "rr_asr_boundary_proxy",
        "rr_ecapa_change_proxy",
    ]

    for sid, g in out.groupby("session_norm", sort=False):
        g = g.sort_values("time").copy()
        times = g["time"].to_numpy(dtype=float)

        if len(g) == 0:
            continue

        gaps = np.diff(times)
        cluster_ids = np.zeros(len(g), dtype=int)

        if len(gaps):
            cluster_ids[1:] = np.cumsum(gaps > float(RR_FAP_CLUSTER_GAP_S))

        g["rr_cluster_id"] = cluster_ids

        prev_gap = np.full(len(g), 999.0, dtype=np.float32)
        next_gap = np.full(len(g), 999.0, dtype=np.float32)

        if len(g) > 1:
            prev_gap[1:] = np.diff(times)
            next_gap[:-1] = np.diff(times)

        g["rr_prev_gap_s"] = np.clip(prev_gap, 0.0, 999.0)
        g["rr_next_gap_s"] = np.clip(next_gap, 0.0, 999.0)
        g["rr_min_neighbor_gap_s"] = np.minimum(g["rr_prev_gap_s"], g["rr_next_gap_s"])

        cluster_parts = []

        for cid, cg in g.groupby("rr_cluster_id", sort=False):
            cg = cg.copy()
            n = len(cg)
            span = float(cg["time"].max() - cg["time"].min()) if n else 0.0

            cg["rr_cluster_size"] = int(n)
            cg["rr_cluster_span_s"] = float(span)
            cg["rr_cluster_pos_frac"] = (
                np.arange(n, dtype=np.float32) / max(1.0, float(n - 1))
                if n > 1
                else 0.0
            )

            for score_col in score_cols:
                vals = pd.to_numeric(cg[score_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)

                local_max = float(np.max(vals)) if len(vals) else 0.0
                local_mean = float(np.mean(vals)) if len(vals) else 0.0
                local_std = float(np.std(vals)) if len(vals) else 0.0

                cg[f"{score_col}_local_max"] = local_max
                cg[f"{score_col}_local_mean"] = local_mean
                cg[f"{score_col}_local_std"] = local_std
                cg[f"{score_col}_minus_local_max"] = vals - local_max
                cg[f"{score_col}_minus_local_mean"] = vals - local_mean
                cg[f"{score_col}_rank_desc"] = (
                    pd.Series(vals).rank(method="first", ascending=False).to_numpy(dtype=np.float32)
                )
                cg[f"{score_col}_rank_frac"] = (
                    (cg[f"{score_col}_rank_desc"].astype(float) - 1.0)
                    / max(1.0, float(n - 1))
                )

            cluster_parts.append(cg)

        rows.append(pd.concat(cluster_parts, ignore_index=False))

    out = pd.concat(rows, ignore_index=False).sort_index().reset_index(drop=True)

    return out

def _rr_assign_targets(df: pd.DataFrame, split_names: List[str]) -> pd.DataFrame:
    out = df.copy()
    out["rr_label"] = 0
    out["rr_nearest_gt_dist"] = np.nan
    out["rr_target_source"] = "role_relative_best_candidate"

    for c in [
        "rr_gt_action",
        "rr_gt_prev_speaker",
        "rr_gt_next_speaker",
    ]:
        out[c] = ""

    for c in [
        "rr_gt_gap",
        "rr_gt_overlap",
        "rr_gt_prev_dur",
        "rr_gt_next_dur",
    ]:
        out[c] = np.nan

    ev = _rr_collect_event_tables(split_names)

    selected_idx = set()
    matched_event_count = 0

    if len(ev):
        for sid, g_ev in ev.groupby("session_norm", sort=False):
            g_c = out[out["session_norm"] == str(sid)].copy()

            if len(g_c) == 0:
                continue

            cand_times = g_c["time"].to_numpy(dtype=float)
            cand_indices = g_c.index.to_numpy(dtype=int)

            for _, er in g_ev.iterrows():
                gt_t = float(er.time)
                d = np.abs(cand_times - gt_t)

                if len(d) == 0 or not np.isfinite(d).any():
                    continue

                j = int(np.nanargmin(d))
                best_d = float(d[j])

                if best_d <= float(RR_FAP_POS_WINDOW_S):
                    idx = int(cand_indices[j])
                    selected_idx.add(idx)
                    matched_event_count += 1

                    old_d = out.at[idx, "rr_nearest_gt_dist"]

                    if pd.isna(old_d) or best_d < float(old_d):
                        out.at[idx, "rr_nearest_gt_dist"] = best_d
                        out.at[idx, "rr_gt_action"] = str(er.get("action", ""))
                        out.at[idx, "rr_gt_prev_speaker"] = str(er.get("prev_speaker", ""))
                        out.at[idx, "rr_gt_next_speaker"] = str(er.get("next_speaker", ""))

                        out.at[idx, "rr_gt_gap"] = (
                            float(er.get("gap", np.nan))
                            if pd.notna(er.get("gap", np.nan))
                            else np.nan
                        )

                        out.at[idx, "rr_gt_overlap"] = (
                            float(er.get("overlap", np.nan))
                            if pd.notna(er.get("overlap", np.nan))
                            else np.nan
                        )

                        out.at[idx, "rr_gt_prev_dur"] = (
                            float(er.get("prev_dur", np.nan))
                            if pd.notna(er.get("prev_dur", np.nan))
                            else np.nan
                        )

                        out.at[idx, "rr_gt_next_dur"] = (
                            float(er.get("next_dur", np.nan))
                            if pd.notna(er.get("next_dur", np.nan))
                            else np.nan
                        )

        if selected_idx:
            out.loc[list(selected_idx), "rr_label"] = 1

            for sid, g in out.groupby("session_norm", sort=False):
                gt_times = ev.loc[ev["session_norm"] == str(sid), "time"].to_numpy(dtype=float)

                if len(gt_times) == 0:
                    continue

                cand_times = g["time"].to_numpy(dtype=float)
                dmin = np.min(np.abs(cand_times[:, None] - gt_times[None, :]), axis=1)
                out.loc[g.index, "rr_nearest_gt_dist"] = dmin

            print(
                f"RR-FAP target from event tables: positives={int(out['rr_label'].sum())}, "
                f"matched_events={matched_event_count}, rows={len(out)}"
            )

        else:
            if "label" in out.columns and out["label"].isin([0, 1]).any():
                out["rr_label"] = out["label"].astype(int).clip(0, 1)
                out["rr_target_source"] = "fallback_candidate_label"
                print("⚠️ RR-FAP target assignment found no event positives. Falling back to candidate label.")
            else:
                out["rr_label"] = 0
                out["rr_target_source"] = "unavailable"

    else:
        if "label" in out.columns and out["label"].isin([0, 1]).any():
            out["rr_label"] = out["label"].astype(int).clip(0, 1)
            out["rr_target_source"] = "fallback_candidate_label_no_event_table"
        else:
            out["rr_label"] = 0
            out["rr_target_source"] = "unavailable"

    out["rr_label"] = pd.to_numeric(out["rr_label"], errors="coerce").fillna(0).astype(int)

    return out

def _build_rr_fap_frame(split_names: List[str], include_target: bool) -> pd.DataFrame:
    frame = _rr_prepare_base_frame(split_names)

    if frame is None or len(frame) == 0:
        return pd.DataFrame()

    frame = _rr_add_role_relative_features(frame)

    if include_target:
        frame = _rr_assign_targets(frame, split_names=split_names)

    else:
        frame["rr_label"] = -1
        frame["rr_target_source"] = "test_unlabelled"
        frame["rr_nearest_gt_dist"] = np.nan
        frame["rr_gt_action"] = ""
        frame["rr_gt_prev_speaker"] = ""
        frame["rr_gt_next_speaker"] = ""
        frame["rr_gt_gap"] = np.nan
        frame["rr_gt_overlap"] = np.nan
        frame["rr_gt_prev_dur"] = np.nan
        frame["rr_gt_next_dur"] = np.nan

    return frame.reset_index(drop=True)

def _infer_rr_fap_feature_cols(df: pd.DataFrame) -> List[str]:
    explicit_cols = [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "lgbm_best_shift_score",
        "ranker_shift_score",
        "temporal_shift_score",
    ]

    cols = [
        c for c in FEATURE_COLS_LGBM
        if str(c) not in RR_FAP_EXCLUDED_FEATURE_COLS
    ]

    for c in explicit_cols:
        if c in df.columns and c not in cols:
            cols.append(c)

    for c in df.columns:
        c = str(c)

        if c in RR_FAP_EXCLUDED_FEATURE_COLS:
            continue

        if c.startswith("rr_") and c not in cols:
            cols.append(c)

    numeric_cols = []

    for c in cols:
        if c not in df.columns:
            continue

        if str(c) in RR_FAP_EXCLUDED_FEATURE_COLS:
            continue

        s = pd.to_numeric(df[c], errors="coerce")

        if s.notna().any():
            numeric_cols.append(c)

    leaked = sorted(set(numeric_cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))
    assert not leaked, f"Leaky/non-deployable RR-FAP features included: {leaked}"

    return numeric_cols

def _rr_fap_params() -> Dict[str, Any]:
    return {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "learning_rate": float(RR_FAP_LEARNING_RATE),
        "num_leaves": int(RR_FAP_NUM_LEAVES),
        "min_data_in_leaf": int(RR_FAP_MIN_DATA_IN_LEAF),
        "feature_fraction": float(RR_FAP_FEATURE_FRACTION),
        "bagging_fraction": float(RR_FAP_BAGGING_FRACTION),
        "bagging_freq": int(RR_FAP_BAGGING_FREQ),
        "lambda_l2": float(RR_FAP_LAMBDA_L2),
        "verbosity": -1,
        "seed": int(SEED),
        "feature_pre_filter": False,
        "force_col_wise": True,
    }

def _rr_fap_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RR_FAP_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "target": "role_relative_future_activity_best_shift_candidate",
        "train_splits": [str(x) for x in RR_FAP_TRAIN_SPLITS],
        "val_splits": [str(x) for x in RR_FAP_VAL_SPLITS],
        "test_splits": [str(x) for x in RR_FAP_TEST_SPLITS],
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RR_FAP_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "pos_window_s": float(RR_FAP_POS_WINDOW_S),
        "cluster_gap_s": float(RR_FAP_CLUSTER_GAP_S),
        "attach_score_max_dist_s": float(RR_FAP_ATTACH_SCORE_MAX_DIST_S),
        "num_boost_round": int(RR_FAP_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(RR_FAP_EARLY_STOPPING_ROUNDS),
        "learning_rate": float(RR_FAP_LEARNING_RATE),
        "num_leaves": int(RR_FAP_NUM_LEAVES),
        "min_data_in_leaf": int(RR_FAP_MIN_DATA_IN_LEAF),
        "feature_fraction": float(RR_FAP_FEATURE_FRACTION),
        "bagging_fraction": float(RR_FAP_BAGGING_FRACTION),
        "bagging_freq": int(RR_FAP_BAGGING_FREQ),
        "lambda_l2": float(RR_FAP_LAMBDA_L2),
        "pos_weight_mult": float(RR_FAP_POS_WEIGHT_MULT),
        "sys2_features_rows": int(len(SYS2_FEATURES_ALL)),
        "test_labels_attached": False,
    }

def _rr_fap_core_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RR_FAP_VARIANT,
        "target": "role_relative_future_activity_best_shift_candidate",
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RR_FAP_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "pos_window_s": float(RR_FAP_POS_WINDOW_S),
        "cluster_gap_s": float(RR_FAP_CLUSTER_GAP_S),
        "attach_score_max_dist_s": float(RR_FAP_ATTACH_SCORE_MAX_DIST_S),
    }

def _rr_fap_score_config(split_name: str, feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": RR_FAP_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "split": str(split_name),
        "model_fingerprint": _rr_file_fingerprint(RR_FAP_MODEL_PATH),
        "model_meta_fingerprint": _rr_file_fingerprint(RR_FAP_MODEL_META_PATH),
        "feature_cols": list(feature_cols),
        "excluded_feature_cols": sorted(list(RR_FAP_EXCLUDED_FEATURE_COLS)),
        "n_features": int(len(feature_cols)),
        "target": "role_relative_future_activity_best_shift_candidate",
        "cache_format": "pickle",
    }

def _rr_fap_weights(df: pd.DataFrame) -> np.ndarray:
    y = (
        pd.to_numeric(df["rr_label"], errors="coerce")
        .fillna(0)
        .astype(int)
        .clip(0, 1)
        .to_numpy()
    )

    base = _rr_num(df, "sample_weight", 1.0).to_numpy(dtype=np.float32)
    base = np.maximum(base, 1e-6)

    counts = np.bincount(y, minlength=2).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    class_w = counts.sum() / counts
    class_w = class_w / max(1e-8, float(class_w.mean()))
    class_w[1] *= float(RR_FAP_POS_WEIGHT_MULT)

    w = base * class_w[y]
    w = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)

    return np.maximum(w, 1e-6).astype(np.float32)

def _clear_stale_rr_fap_score_caches():
    for p in [
        RR_FAP_TRAIN_SCORES_PATH,
        RR_FAP_VAL_SCORES_PATH,
        RR_FAP_TEST_SCORES_PATH,
        RR_FAP_TRAIN_SCORES_META_PATH,
        RR_FAP_VAL_SCORES_META_PATH,
        RR_FAP_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale RR-FAP score cache:", p)
        except Exception as e:
            print("Could not remove stale RR-FAP score cache:", p, e)

def _load_rr_fap_feature_cols_from_file_or_meta() -> Optional[List[str]]:
    if RR_FAP_FEATURE_COLS_PATH.exists():
        try:
            cols = json.loads(RR_FAP_FEATURE_COLS_PATH.read_text())
            leaked = sorted(set(cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))

            if leaked:
                print("⚠️ RR-FAP feature file contains leaky features:", leaked)
                return None

            return list(cols)

        except Exception as e:
            print("Could not load RR-FAP feature columns file:", e)

    for p in [
        RR_FAP_VAL_SCORES_META_PATH,
        RR_FAP_TEST_SCORES_META_PATH,
        RR_FAP_TRAIN_SCORES_META_PATH,
    ]:
        if not p.exists():
            continue

        try:
            meta = json.loads(p.read_text())
            cols = list(meta.get("feature_cols", []))
            leaked = sorted(set(cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))

            if leaked:
                print("⚠️ RR-FAP score meta contains leaky features:", leaked)
                return None

            if cols:
                return cols

        except Exception:
            continue

    return None

def _rr_fap_score_cache_core_compatible(meta_path: Path, feature_cols: List[str]) -> bool:
    if not Path(meta_path).exists():
        return False

    try:
        meta = json.loads(Path(meta_path).read_text())
    except Exception:
        return False

    if meta.get("variant", None) != RR_FAP_VARIANT:
        return False

    if meta.get("target", None) != "role_relative_future_activity_best_shift_candidate":
        return False

    saved_cols = list(meta.get("feature_cols", []))

    if saved_cols != list(feature_cols):
        return False

    leaked = sorted(set(saved_cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))

    if leaked:
        return False

    saved_excluded = set(meta.get("excluded_feature_cols", []))

    if not set(RR_FAP_EXCLUDED_FEATURE_COLS).issubset(saved_excluded):
        return False

    return True

def _all_rr_fap_score_caches_usable(feature_cols: Optional[List[str]]) -> bool:
    if not feature_cols:
        return False

    checks = [
        (RR_FAP_TRAIN_SCORES_PATH, RR_FAP_TRAIN_SCORES_META_PATH),
        (RR_FAP_VAL_SCORES_PATH, RR_FAP_VAL_SCORES_META_PATH),
        (RR_FAP_TEST_SCORES_PATH, RR_FAP_TEST_SCORES_META_PATH),
    ]

    for data_path, meta_path in checks:
        if not Path(data_path).exists():
            return False

        if not _rr_fap_score_cache_core_compatible(meta_path, feature_cols):
            return False

    return True

def train_rr_fap_model():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    train_df = _build_rr_fap_frame(RR_FAP_TRAIN_SPLITS, include_target=True)
    val_df = _build_rr_fap_frame(RR_FAP_VAL_SPLITS, include_target=True)

    assert len(train_df) > 0, "No RR-FAP training rows."
    assert len(val_df) > 0, "No RR-FAP validation rows."
    assert int(train_df["rr_label"].sum()) > 0, "RR-FAP train has zero positives."
    assert int(val_df["rr_label"].sum()) > 0, "RR-FAP VAL has zero positives."

    feature_cols = _infer_rr_fap_feature_cols(train_df)

    assert len(feature_cols) > 0, "No usable RR-FAP feature columns."

    leaked = sorted(set(feature_cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))
    assert not leaked, f"Leaky/non-deployable RR-FAP features included: {leaked}"

    for c in feature_cols:
        train_df[c] = (
            pd.to_numeric(train_df[c], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )
        val_df[c] = (
            pd.to_numeric(val_df[c], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

    Xtr = train_df[feature_cols]
    ytr = train_df["rr_label"].astype(int)
    wtr = _rr_fap_weights(train_df)

    Xva = val_df[feature_cols]
    yva = val_df["rr_label"].astype(int)
    wva = _rr_fap_weights(val_df)

    dtr = lgb.Dataset(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_name=feature_cols,
        free_raw_data=False,
    )

    dva = lgb.Dataset(
        Xva,
        label=yva,
        weight=wva,
        reference=dtr,
        feature_name=feature_cols,
        free_raw_data=False,
    )

    params = _rr_fap_params()
    evals_result = {}

    model = lgb.train(
        params,
        dtr,
        num_boost_round=int(RR_FAP_NUM_BOOST_ROUND),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=int(RR_FAP_EARLY_STOPPING_ROUNDS),
                first_metric_only=True,
                verbose=True,
            ),
            lgb.log_evaluation(period=int(RR_FAP_LOG_EVERY)),
            lgb.record_evaluation(evals_result),
        ],
    )

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        model.save_model(str(RR_FAP_MODEL_PATH), num_iteration=best_iter)
    else:
        model.save_model(str(RR_FAP_MODEL_PATH))

    meta = _rr_fap_model_config(feature_cols)
    meta.update({
        "best_iteration": int(best_iter),
        "best_score": _rr_json_safe(getattr(model, "best_score", {})),
        "trained_time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "train_rows": int(len(train_df)),
        "train_positive_rows": int(train_df["rr_label"].sum()),
        "val_rows": int(len(val_df)),
        "val_positive_rows": int(val_df["rr_label"].sum()),
        "train_target_source_counts": _rr_json_safe(
            train_df["rr_target_source"].value_counts(dropna=False).to_dict()
        ),
        "val_target_source_counts": _rr_json_safe(
            val_df["rr_target_source"].value_counts(dropna=False).to_dict()
        ),
    })

    RR_FAP_MODEL_META_PATH.write_text(json.dumps(meta, indent=2, sort_keys=True, allow_nan=False))
    RR_FAP_PARAMS_PATH.write_text(json.dumps(params, indent=2, sort_keys=True, allow_nan=False))
    RR_FAP_HISTORY_PATH.write_text(json.dumps(_rr_json_safe(evals_result), indent=2, sort_keys=True, allow_nan=False))
    RR_FAP_FEATURE_COLS_PATH.write_text(json.dumps(feature_cols, indent=2, sort_keys=True))

    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)

    imp.to_csv(RR_FAP_IMPORTANCE_PATH, index=False)

    print("Saved RR-FAP model:", RR_FAP_MODEL_PATH)
    print("Saved RR-FAP meta:", RR_FAP_MODEL_META_PATH)
    print("Saved RR-FAP feature columns:", RR_FAP_FEATURE_COLS_PATH)
    print("Saved RR-FAP feature importance:", RR_FAP_IMPORTANCE_PATH)
    print("RR-FAP best iteration:", best_iter)
    print("Excluded RR-FAP non-deployable features:", sorted(RR_FAP_EXCLUDED_FEATURE_COLS))

    _clear_stale_rr_fap_score_caches()

    return model, feature_cols

def load_rr_fap_model():
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    if not RR_FAP_MODEL_PATH.exists() or not RR_FAP_MODEL_META_PATH.exists() or not RR_FAP_FEATURE_COLS_PATH.exists():
        return None, None

    try:
        feature_cols = json.loads(RR_FAP_FEATURE_COLS_PATH.read_text())

        leaked = sorted(set(feature_cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))
        if leaked:
            print(f"⚠️ Existing RR-FAP model contains leaky/non-deployable features: {leaked}")
            return None, None

        saved_meta = json.loads(RR_FAP_MODEL_META_PATH.read_text())

        if STRICT_RR_FAP_MODEL_META_CHECK:
            expected = _rr_fap_model_config(feature_cols)
            comparable_keys = [
                "variant",
                "run_version",
                "seed",
                "target",
                "train_splits",
                "val_splits",
                "test_splits",
                "feature_cols",
                "excluded_feature_cols",
                "n_features",
                "pos_window_s",
                "cluster_gap_s",
                "attach_score_max_dist_s",
                "learning_rate",
                "num_leaves",
                "min_data_in_leaf",
                "feature_fraction",
                "bagging_fraction",
                "bagging_freq",
                "lambda_l2",
                "pos_weight_mult",
            ]
        else:
            expected = _rr_fap_core_model_config(feature_cols)
            comparable_keys = list(expected.keys())

        for k in comparable_keys:
            if saved_meta.get(k, None) != expected.get(k, None):
                print(f"⚠️ Existing RR-FAP meta mismatch at key: {k}")
                print("  expected:", expected.get(k, None))
                print("  saved:   ", saved_meta.get(k, None))
                return None, None

    except Exception as e:
        print("⚠️ Could not read RR-FAP meta/features:", e)
        return None, None

    model = lgb.Booster(model_file=str(RR_FAP_MODEL_PATH))
    print("Loaded RR-FAP model:", RR_FAP_MODEL_PATH)

    return model, feature_cols

def _validate_rr_fap_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "rr_role_shift_score",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"RR-FAP {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

def rr_fap_scores(
    model,
    feature_cols: List[str],
    split_names: List[str],
    out_path: Path,
    meta_path: Path,
    split_name: str,
    include_target: bool,
) -> pd.DataFrame:
    expected = _rr_fap_score_config(split_name, feature_cols)

    if out_path.exists() and not FORCE_RETRAIN_RR_FAP:
        meta_matches = _rr_meta_matches(meta_path, expected)
        core_compatible = _rr_fap_score_cache_core_compatible(meta_path, feature_cols)

        if meta_matches:
            print(f"RR-FAP score metadata matches for {split_name}.")
        else:
            print(f"⚠️ RR-FAP score metadata mismatch for {split_name}.")

            if not core_compatible:
                print(
                    f"RR-FAP {split_name} cache is not core-compatible with the current configuration. "
                    "It will not be loaded."
                )
            elif STRICT_RR_FAP_SCORE_META_CHECK:
                print(f"STRICT_RR_FAP_SCORE_META_CHECK=True, so {split_name} RR-FAP scores will be recomputed.")
            else:
                print(
                    f"Cache-safe resume mode: loading existing compatible RR-FAP {split_name} scores anyway.\n"
                    f"Loaded from: {out_path}"
                )

        if meta_matches or (core_compatible and not STRICT_RR_FAP_SCORE_META_CHECK):
            cached = pd.read_pickle(out_path)

            if not _validate_rr_fap_score_cache(cached, split_name):
                raise RuntimeError(
                    f"Cached RR-FAP {split_name} scores are incompatible:\n"
                    f"{out_path}\n\n"
                    "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_RR_FAP=True only if you intentionally want to rebuild."
                )

            print(f"Loaded RR-FAP {split_name} scores:", cached.shape)
            return cached

    leaked = sorted(set(feature_cols).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))
    assert not leaked, f"Leaky/non-deployable RR-FAP features included for scoring: {leaked}"

    if model is None:
        if CACHE_SAFE_RESUME_RUN:
            raise FileNotFoundError(
                f"No compatible RR-FAP model is loaded and no usable cached RR-FAP {split_name} scores were found.\n"
                f"Expected score cache: {out_path}\n\n"
                "In cache-safe resume mode, this cell will not train/rescore automatically.\n"
                "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_RR_FAP=True only if you intentionally want to rebuild this."
            )
        return pd.DataFrame()

    work = _build_rr_fap_frame(split_names, include_target=include_target)

    if work is None or len(work) == 0:
        return pd.DataFrame()

    for c in feature_cols:
        if c not in work.columns:
            work[c] = 0.0

        work[c] = (
            pd.to_numeric(work[c], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

    X = work[feature_cols]
    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        score = model.predict(X, num_iteration=best_iter)
    else:
        score = model.predict(X)

    score = np.asarray(score, dtype=np.float32).reshape(-1)
    score = np.clip(score, 0.0, 1.0)

    base_cols = [
        "session",
        "time",
        "split",
        "label",
        "action",
        "action_id",
        "audio_path",
        "rr_label",
        "rr_target_source",
        "rr_nearest_gt_dist",
        "rr_gt_action",
        "rr_cluster_size",
        "rr_cluster_span_s",
        "rr_cluster_pos_frac",
        "rr_role_switch_proxy",
        "rr_same_holder_continue_proxy",
        "rr_next_enters_proxy",
        "rr_current_drop_proxy",
        "rr_overlap_future_proxy",
        "rr_source_agree_count",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "temporal_shift_score",
    ]

    base_cols = [c for c in base_cols if c in work.columns]

    out = work[base_cols].copy().reset_index(drop=True)
    out["rr_role_shift_score"] = score

    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(expected, indent=2, sort_keys=True, allow_nan=False))

    print(f"Saved RR-FAP {split_name} scores:", out_path)

    return out

def _merge_rr_fap_scores_into_lgbm_scores(main_df: pd.DataFrame, rr_df: pd.DataFrame) -> pd.DataFrame:
    if main_df is None or len(main_df) == 0:
        return pd.DataFrame()

    out = main_df.copy().reset_index(drop=True)
    out = out.drop(columns=["rr_role_shift_score"], errors="ignore")

    if rr_df is None or len(rr_df) == 0 or "rr_role_shift_score" not in rr_df.columns:
        out["rr_role_shift_score"] = pd.to_numeric(
            out.get(
                "ranker_shift_score",
                out.get("lgbm_binary_shift_score", out.get("lgbm_shift_score", 0.0)),
            ),
            errors="coerce",
        ).fillna(0.0)

        return out

    b = rr_df.copy().reset_index(drop=True)

    aligned = (
        len(out) == len(b)
        and "session" in out.columns
        and "session" in b.columns
        and "time" in out.columns
        and "time" in b.columns
        and out["session"].astype(str).map(_rr_norm_sid).equals(b["session"].astype(str).map(_rr_norm_sid))
        and np.allclose(
            pd.to_numeric(out["time"], errors="coerce").to_numpy(dtype=float),
            pd.to_numeric(b["time"], errors="coerce").to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-6,
            equal_nan=False,
        )
    )

    if aligned:
        out["rr_role_shift_score"] = pd.to_numeric(
            b["rr_role_shift_score"],
            errors="coerce",
        ).fillna(0.0).to_numpy(dtype=np.float32)

    else:
        out["session_norm_tmp"] = out["session"].astype(str).map(_rr_norm_sid)
        b["session_norm_tmp"] = b["session"].astype(str).map(_rr_norm_sid)

        b_small = b[["session_norm_tmp", "time", "rr_role_shift_score"]].copy()
        b_small["time"] = pd.to_numeric(b_small["time"], errors="coerce")

        out = out.merge(
            b_small,
            on=["session_norm_tmp", "time"],
            how="left",
        ).drop(columns=["session_norm_tmp"], errors="ignore")

    out["rr_role_shift_score"] = (
        pd.to_numeric(out["rr_role_shift_score"], errors="coerce")
        .fillna(pd.to_numeric(out.get("ranker_shift_score", out.get("lgbm_binary_shift_score", 0.0)), errors="coerce"))
        .fillna(0.0)
        .clip(0.0, 1.0)
    )

    return out

if FORCE_RETRAIN_RR_FAP and not CACHE_SAFE_RESUME_RUN:
    for p in [
        RR_FAP_MODEL_PATH,
        RR_FAP_MODEL_META_PATH,
        RR_FAP_PARAMS_PATH,
        RR_FAP_FEATURE_COLS_PATH,
        RR_FAP_IMPORTANCE_PATH,
        RR_FAP_HISTORY_PATH,
        RR_FAP_TRAIN_SCORES_PATH,
        RR_FAP_VAL_SCORES_PATH,
        RR_FAP_TEST_SCORES_PATH,
        RR_FAP_TRAIN_SCORES_META_PATH,
        RR_FAP_VAL_SCORES_META_PATH,
        RR_FAP_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale RR-FAP artifact:", p)
        except Exception as e:
            print("Could not remove RR-FAP artifact:", p, e)

RR_FAP_MODEL = None
RR_FAP_FEATURE_COLS = None

if RR_FAP_MODEL_PATH.exists() and not FORCE_RETRAIN_RR_FAP:
    RR_FAP_MODEL, RR_FAP_FEATURE_COLS = load_rr_fap_model()

    if RR_FAP_MODEL is None:
        cache_feature_cols = _load_rr_fap_feature_cols_from_file_or_meta()

        msg = (
            "RR-FAP model exists but is incompatible with the current configuration:\n"
            f"{RR_FAP_MODEL_PATH}\n\n"
            "In cache-safe resume mode, this cell will not retrain automatically."
        )

        if CACHE_SAFE_RESUME_RUN and _all_rr_fap_score_caches_usable(cache_feature_cols):
            print("⚠️", msg)
            print("Compatible RR-FAP score caches exist, so continuing in score-cache-only mode.")
            RR_FAP_FEATURE_COLS = cache_feature_cols

        elif CACHE_SAFE_RESUME_RUN:
            raise RuntimeError(
                msg
                + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_RR_FAP=True only if you intentionally want to retrain."
            )

        else:
            print("No compatible RR-FAP model found. Training a new one.")
            RR_FAP_MODEL, RR_FAP_FEATURE_COLS = train_rr_fap_model()

elif FORCE_RETRAIN_RR_FAP:
    print("FORCE_RETRAIN_RR_FAP=True, training RR-FAP.")
    RR_FAP_MODEL, RR_FAP_FEATURE_COLS = train_rr_fap_model()

else:
    cache_feature_cols = _load_rr_fap_feature_cols_from_file_or_meta()

    msg = (
        "No RR-FAP model found:\n"
        f"{RR_FAP_MODEL_PATH}\n\n"
        "In cache-safe resume mode, this cell will not train automatically."
    )

    if CACHE_SAFE_RESUME_RUN and _all_rr_fap_score_caches_usable(cache_feature_cols):
        print("⚠️", msg)
        print("Compatible RR-FAP score caches exist, so continuing in score-cache-only mode.")
        RR_FAP_FEATURE_COLS = cache_feature_cols

    elif CACHE_SAFE_RESUME_RUN:
        raise FileNotFoundError(
            msg
            + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_RR_FAP=True only if you intentionally want to train."
        )

    else:
        RR_FAP_MODEL, RR_FAP_FEATURE_COLS = train_rr_fap_model()

assert RR_FAP_FEATURE_COLS is not None and len(RR_FAP_FEATURE_COLS) > 0, (
    "RR_FAP_FEATURE_COLS could not be resolved."
)

leaked_final = sorted(set(RR_FAP_FEATURE_COLS).intersection(RR_FAP_EXCLUDED_FEATURE_COLS))
assert not leaked_final, f"Leaky/non-deployable RR-FAP features survived final feature list: {leaked_final}"

RR_FAP_TRAIN_SCORES = rr_fap_scores(
    RR_FAP_MODEL,
    RR_FAP_FEATURE_COLS,
    RR_FAP_TRAIN_SPLITS,
    RR_FAP_TRAIN_SCORES_PATH,
    RR_FAP_TRAIN_SCORES_META_PATH,
    "train",
    include_target=True,
)

RR_FAP_VAL_SCORES = rr_fap_scores(
    RR_FAP_MODEL,
    RR_FAP_FEATURE_COLS,
    RR_FAP_VAL_SPLITS,
    RR_FAP_VAL_SCORES_PATH,
    RR_FAP_VAL_SCORES_META_PATH,
    "val",
    include_target=True,
)

RR_FAP_TEST_SCORES = rr_fap_scores(
    RR_FAP_MODEL,
    RR_FAP_FEATURE_COLS,
    RR_FAP_TEST_SPLITS,
    RR_FAP_TEST_SCORES_PATH,
    RR_FAP_TEST_SCORES_META_PATH,
    "test",
    include_target=False,
)

LGBM_TRAIN_SCORES = _merge_rr_fap_scores_into_lgbm_scores(LGBM_TRAIN_SCORES, RR_FAP_TRAIN_SCORES)
LGBM_VAL_SCORES = _merge_rr_fap_scores_into_lgbm_scores(LGBM_VAL_SCORES, RR_FAP_VAL_SCORES)
LGBM_TEST_SCORES = _merge_rr_fap_scores_into_lgbm_scores(LGBM_TEST_SCORES, RR_FAP_TEST_SCORES)

LGBM_TRAIN_SCORES.to_pickle(LGBM_TRAIN_SCORES_PATH)
LGBM_VAL_SCORES.to_pickle(LGBM_VAL_SCORES_PATH)
LGBM_TEST_SCORES.to_pickle(LGBM_TEST_SCORES_PATH)

LGBM_HAS_RR_FAP_SCORE = True

print("RR_FAP_TRAIN_SCORES:", RR_FAP_TRAIN_SCORES.shape)
print("RR_FAP_VAL_SCORES:", RR_FAP_VAL_SCORES.shape)
print("RR_FAP_TEST_SCORES:", RR_FAP_TEST_SCORES.shape)
print("Augmented LGBM_VAL_SCORES includes RR-FAP:", "rr_role_shift_score" in LGBM_VAL_SCORES.columns)
print("RR-FAP feature count:", len(RR_FAP_FEATURE_COLS))
print("Excluded from RR-FAP features:", sorted(RR_FAP_EXCLUDED_FEATURE_COLS))
print("rr_nearest_gt_dist used as feature:", "rr_nearest_gt_dist" in RR_FAP_FEATURE_COLS)

assert "rr_nearest_gt_dist" not in RR_FAP_FEATURE_COLS, (
    "rr_nearest_gt_dist is GT-derived and must not be used as a feature."
)

if len(RR_FAP_VAL_SCORES):
    val_known = RR_FAP_VAL_SCORES[
        RR_FAP_VAL_SCORES["rr_label"].astype(int).isin([0, 1])
    ].copy()

    if len(val_known) and int(val_known["rr_label"].sum()) > 0:
        y_true = val_known["rr_label"].astype(int).to_numpy()
        y_score = (
            pd.to_numeric(val_known["rr_role_shift_score"], errors="coerce")
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        y_pred = (y_score >= 0.50).astype(int)

        P, R, F1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        try:
            ap = float(average_precision_score(y_true, y_score))
        except Exception:
            ap = float("nan")

        try:
            auc = float(roc_auc_score(y_true, y_score))
        except Exception:
            auc = float("nan")

        print(
            f"RR-FAP VAL row-level @0.50: "
            f"P={P:.4f} R={R:.4f} F1={F1:.4f} AP={ap:.4f} AUC={auc:.4f}"
        )

    print("RR-FAP VAL target source counts:")
    display(RR_FAP_VAL_SCORES["rr_target_source"].value_counts(dropna=False).to_frame("n"))
    display(RR_FAP_VAL_SCORES.head())

if len(RR_FAP_TEST_SCORES):
    print("TEST RR-FAP label check — should be only -1:")
    display(RR_FAP_TEST_SCORES["rr_label"].value_counts(dropna=False).to_frame("test_rr_label_count"))

    test_labels = set(
        pd.to_numeric(RR_FAP_TEST_SCORES["rr_label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
    )

    assert test_labels.issubset({-1}), f"TEST labels leaked into RR-FAP scores: {test_labels}"

if RR_FAP_IMPORTANCE_PATH.exists():
    print("Top RR-FAP features by gain:")
    imp_df = pd.read_csv(RR_FAP_IMPORTANCE_PATH)
    display(imp_df.head(25))

    leaked_importance = set(imp_df["feature"].astype(str)).intersection(RR_FAP_EXCLUDED_FEATURE_COLS)
    assert not leaked_importance, f"Leaky features found in RR-FAP importance table: {sorted(leaked_importance)}"

globals()["RR_FAP_MODEL"] = RR_FAP_MODEL
globals()["RR_FAP_FEATURE_COLS"] = RR_FAP_FEATURE_COLS
globals()["RR_FAP_TRAIN_SCORES"] = RR_FAP_TRAIN_SCORES
globals()["RR_FAP_VAL_SCORES"] = RR_FAP_VAL_SCORES
globals()["RR_FAP_TEST_SCORES"] = RR_FAP_TEST_SCORES
globals()["RR_FAP_MODEL_PATH"] = RR_FAP_MODEL_PATH
globals()["RR_FAP_MODEL_META_PATH"] = RR_FAP_MODEL_META_PATH
globals()["RR_FAP_PARAMS_PATH"] = RR_FAP_PARAMS_PATH
globals()["RR_FAP_HISTORY_PATH"] = RR_FAP_HISTORY_PATH
globals()["RR_FAP_IMPORTANCE_PATH"] = RR_FAP_IMPORTANCE_PATH
globals()["RR_FAP_FEATURE_COLS_PATH"] = RR_FAP_FEATURE_COLS_PATH
globals()["RR_FAP_TRAIN_SCORES_PATH"] = RR_FAP_TRAIN_SCORES_PATH
globals()["RR_FAP_VAL_SCORES_PATH"] = RR_FAP_VAL_SCORES_PATH
globals()["RR_FAP_TEST_SCORES_PATH"] = RR_FAP_TEST_SCORES_PATH
globals()["LGBM_TRAIN_SCORES"] = LGBM_TRAIN_SCORES
globals()["LGBM_VAL_SCORES"] = LGBM_VAL_SCORES
globals()["LGBM_TEST_SCORES"] = LGBM_TEST_SCORES
globals()["LGBM_HAS_RR_FAP_SCORE"] = LGBM_HAS_RR_FAP_SCORE

print("✅ Cell 19D complete: MuVAP-inspired audio-text role-relative projection loaded/scored and merged into LGBM score tables.")

RR-FAP settings:
  variant: rr_fap_v1_audio_text_role_relative_projection
  force_retrain: False
  cache_safe_resume: False
  strict_model_meta_check: False
  strict_score_meta_check: False


RR-FAP target from event tables: positives=28716, matched_events=29124, rows=206097


RR-FAP target from event tables: positives=457, matched_events=458, rows=7968


Training until validation scores don't improve for 250 rounds


[100]	train's binary_logloss: 0.236014	train's auc: 0.977549	val's binary_logloss: 0.628611	val's auc: 0.864959


...

TEST RR-FAP label check — should be only -1:


,test_rr_label_count
rr_label,
-1,99307


Top RR-FAP features by gain:


,feature,importance_gain,importance_split
0,ranker_shift_score,1.069515e+06,338
1,rr_ranker_score,8.593215e+04,24
2,rr_ranker_score_minus_local_max,1.107764e+04,211
3,lgbm_binary_shift_score,1.658241e+03,118
4,wavlm_center_score,1.005014e+03,119
5,wavlm_score,6.468675e+02,74
6,rr_ranker_score_local_max,6.440657e+02,55
7,rr_ranker_score_rank_frac,5.173658e+02,30
8,rr_min_neighbor_gap_s,5.065115e+02,59
9,wavlm_p_NTT,4.905463e+02,60


✅ Cell 19D complete: MuVAP-inspired audio-text role-relative projection loaded/scored and merged into LGBM score tables.


In [28]:
# VAP / RR-FAP / Text learned meta-confirmer
import json
import time
import math
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, List, Optional
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score

assert "LGBM_SYS2_DIR" in globals(), "Run Cell 19 first so LGBM_SYS2_DIR is available."
assert "LGBM_TRAIN_SCORES" in globals(), "Run Cell 19D first so LGBM_TRAIN_SCORES is available."
assert "LGBM_VAL_SCORES" in globals(), "Run Cell 19D first so LGBM_VAL_SCORES is available."
assert "LGBM_TEST_SCORES" in globals(), "Run Cell 19D first so LGBM_TEST_SCORES is available."
assert "TEMPORAL_TRAIN_SCORES" in globals(), "Run Cell 18 first so TEMPORAL_TRAIN_SCORES is available."
assert "TEMPORAL_VAL_SCORES" in globals(), "Run Cell 18 first so TEMPORAL_VAL_SCORES is available."
assert "TEMPORAL_TEST_SCORES" in globals(), "Run Cell 18 first so TEMPORAL_TEST_SCORES is available."
assert "SYS2_FEATURES_ALL" in globals(), "Run Cell 16 first so SYS2_FEATURES_ALL is available."
assert "TOL" in globals(), "Need TOL."

META_CONFIRMER_VARIANT = "meta_confirmer_v1_vap_rrfap_text_synth_diffusion"

META_CONFIRMER_DIR = LGBM_SYS2_DIR / "meta_confirmer_vap_rrfap_text"
META_CONFIRMER_DIR.mkdir(parents=True, exist_ok=True)

META_CONFIRMER_MODEL_PATH = META_CONFIRMER_DIR / "lightgbm_meta_confirmer.txt"
META_CONFIRMER_MODEL_META_PATH = META_CONFIRMER_DIR / "lightgbm_meta_confirmer.meta.json"
META_CONFIRMER_FEATURE_COLS_PATH = META_CONFIRMER_DIR / "lightgbm_meta_confirmer_feature_cols.json"
META_CONFIRMER_IMPORTANCE_PATH = META_CONFIRMER_DIR / "lightgbm_meta_confirmer_feature_importance.csv"
META_CONFIRMER_HISTORY_PATH = META_CONFIRMER_DIR / "lightgbm_meta_confirmer_eval_history.json"

META_CONFIRMER_TRAIN_SCORES_PATH = META_CONFIRMER_DIR / "meta_confirmer_train_scores.pkl"
META_CONFIRMER_VAL_SCORES_PATH = META_CONFIRMER_DIR / "meta_confirmer_val_scores.pkl"
META_CONFIRMER_TEST_SCORES_PATH = META_CONFIRMER_DIR / "meta_confirmer_test_scores.pkl"

META_CONFIRMER_TRAIN_SCORES_META_PATH = META_CONFIRMER_DIR / "meta_confirmer_train_scores.meta.json"
META_CONFIRMER_VAL_SCORES_META_PATH = META_CONFIRMER_DIR / "meta_confirmer_val_scores.meta.json"
META_CONFIRMER_TEST_SCORES_META_PATH = META_CONFIRMER_DIR / "meta_confirmer_test_scores.meta.json"

USE_META_CONFIRMER = bool(globals().get("USE_META_CONFIRMER", True))
FORCE_RETRAIN_META_CONFIRMER = bool(globals().get("FORCE_RETRAIN_META_CONFIRMER", False))

CACHE_SAFE_RESUME_RUN = bool(globals().get("CACHE_SAFE_RESUME_RUN", True))

STRICT_META_CONFIRMER_MODEL_META_CHECK = bool(
    globals().get("STRICT_META_CONFIRMER_MODEL_META_CHECK", False)
)

STRICT_META_CONFIRMER_SCORE_META_CHECK = bool(
    globals().get("STRICT_META_CONFIRMER_SCORE_META_CHECK", False)
)

if CACHE_SAFE_RESUME_RUN:
    FORCE_RETRAIN_META_CONFIRMER = False

globals()["FORCE_RETRAIN_META_CONFIRMER"] = FORCE_RETRAIN_META_CONFIRMER
globals()["STRICT_META_CONFIRMER_MODEL_META_CHECK"] = STRICT_META_CONFIRMER_MODEL_META_CHECK
globals()["STRICT_META_CONFIRMER_SCORE_META_CHECK"] = STRICT_META_CONFIRMER_SCORE_META_CHECK

META_CONFIRMER_ATTACH_TOL_S = float(globals().get("META_CONFIRMER_ATTACH_TOL_S", 0.05))
META_CONFIRMER_NUM_BOOST_ROUND = int(globals().get("META_CONFIRMER_NUM_BOOST_ROUND", 5000))
META_CONFIRMER_EARLY_STOPPING_ROUNDS = int(globals().get("META_CONFIRMER_EARLY_STOPPING_ROUNDS", 250))
META_CONFIRMER_LEARNING_RATE = float(globals().get("META_CONFIRMER_LEARNING_RATE", 0.012))
META_CONFIRMER_NUM_LEAVES = int(globals().get("META_CONFIRMER_NUM_LEAVES", 63))
META_CONFIRMER_MIN_DATA_IN_LEAF = int(globals().get("META_CONFIRMER_MIN_DATA_IN_LEAF", 20))
META_CONFIRMER_FEATURE_FRACTION = float(globals().get("META_CONFIRMER_FEATURE_FRACTION", 0.92))
META_CONFIRMER_BAGGING_FRACTION = float(globals().get("META_CONFIRMER_BAGGING_FRACTION", 0.85))
META_CONFIRMER_BAGGING_FREQ = int(globals().get("META_CONFIRMER_BAGGING_FREQ", 1))
META_CONFIRMER_LAMBDA_L2 = float(globals().get("META_CONFIRMER_LAMBDA_L2", 2.0))
META_CONFIRMER_POS_WEIGHT_MULT = float(globals().get("META_CONFIRMER_POS_WEIGHT_MULT", 1.25))
META_CONFIRMER_SYNTH_WEIGHT_MULT = float(globals().get("META_CONFIRMER_SYNTH_WEIGHT_MULT", 1.10))
META_CONFIRMER_DIFFUSION_WEIGHT_MULT = float(globals().get("META_CONFIRMER_DIFFUSION_WEIGHT_MULT", 1.10))
META_CONFIRMER_LOG_EVERY = int(globals().get("META_CONFIRMER_LOG_EVERY", 100))

META_CONFIRMER_EXCLUDED_FEATURE_COLS = {
    "label", "action", "action_id", "audio_path", "session", "split", "time",
    "meta_label", "meta_target_source", "meta_nearest_gt_dist", "nearest_gt_dist",
    "ranker_label", "ranker_target_source", "ranker_nearest_gt_dist",
    "rr_label", "rr_target_source", "rr_nearest_gt_dist",
    "rr_gt_action", "rr_gt_prev_speaker", "rr_gt_next_speaker", "rr_gt_gap",
    "rr_gt_overlap", "rr_gt_prev_dur", "rr_gt_next_dur", "rr_cluster_id",
    "lgbm_pred_action", "lgbm_pred_action_id", "meta_pred_label",
}

META_SOURCE_FEATURE_KEYWORDS = [
    "source_", "wavlm", "pyannote", "asr", "qwen", "ecapa", "candidate_density",
    "energy", "pause", "gap", "overlap", "duration", "speech", "speaker", "cos",
]

print("Meta-confirmer settings:")
print("  use_meta_confirmer:", USE_META_CONFIRMER)
print("  variant:", META_CONFIRMER_VARIANT)
print("  force_retrain:", FORCE_RETRAIN_META_CONFIRMER)
print("  cache_safe_resume:", CACHE_SAFE_RESUME_RUN)
print("  strict_model_meta_check:", STRICT_META_CONFIRMER_MODEL_META_CHECK)
print("  strict_score_meta_check:", STRICT_META_CONFIRMER_SCORE_META_CHECK)

def _mc_json_safe(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return v if math.isfinite(v) else None
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _mc_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_mc_json_safe(v) for v in obj]
    if isinstance(obj, pd.DataFrame):
        return f"<DataFrame shape={obj.shape}>"
    if isinstance(obj, pd.Series):
        return _mc_json_safe(obj.to_dict())
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _mc_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
            "size": None,
            "mtime": None,
        }

    st = path.stat()

    return {
        "exists": True,
        "path": str(path),
        "size": int(st.st_size),
        "mtime": float(st.st_mtime),
    }

def _mc_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False

    try:
        old = json.loads(Path(path).read_text())
        return old == expected

    except Exception:
        return False

def _mc_norm_session(s: pd.Series) -> pd.Series:
    return s.astype(str).str.replace(r"\.(wav|rttm)$", "", regex=True)

def _mc_num(df: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.full(len(df), float(default)), index=df.index, dtype=np.float32)

    return (
        pd.to_numeric(df[col], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(float(default))
        .astype(np.float32)
    )

def _mc_clip01(x) -> pd.Series:
    return pd.Series(np.asarray(x, dtype=np.float32)).clip(0.0, 1.0)

def _mc_attach_cols_nearest(
    base_df: pd.DataFrame,
    score_df: pd.DataFrame,
    cols_to_attach: List[str],
    max_dist: float = META_CONFIRMER_ATTACH_TOL_S,
    suffix: str = "",
) -> pd.DataFrame:
    if base_df is None or len(base_df) == 0:
        return pd.DataFrame()

    out = base_df.copy().reset_index(drop=True)

    if score_df is None or len(score_df) == 0:
        for c in cols_to_attach:
            out[f"{c}{suffix}"] = np.nan
        return out

    if "session" not in out.columns or "time" not in out.columns:
        return out

    if "session" not in score_df.columns or "time" not in score_df.columns:
        for c in cols_to_attach:
            out[f"{c}{suffix}"] = np.nan
        return out

    attach_cols = [c for c in cols_to_attach if c in score_df.columns]

    if not attach_cols:
        return out

    left = out.copy()

    for c in attach_cols:
        target_col = f"{c}{suffix}"

        if target_col in left.columns:
            left = left.drop(columns=[target_col])

    left["_row_id_mc"] = np.arange(len(left))
    left["session_norm_mc"] = _mc_norm_session(left["session"])
    left["time"] = pd.to_numeric(left["time"], errors="coerce")
    left = left.dropna(subset=["session_norm_mc", "time"])

    right = score_df[["session", "time"] + attach_cols].copy()
    right["session_norm_mc"] = _mc_norm_session(right["session"])
    right["time"] = pd.to_numeric(right["time"], errors="coerce")
    right = right.dropna(subset=["session_norm_mc", "time"])

    rows = []

    for sid, g in left.groupby("session_norm_mc", sort=False):
        l = g.sort_values("time").copy()
        r = right[right["session_norm_mc"] == sid].sort_values("time").copy()

        if len(r) == 0:
            for c in attach_cols:
                l[f"{c}{suffix}"] = np.nan
            rows.append(l)
            continue

        keep = ["time"] + attach_cols
        merged = pd.merge_asof(
            l,
            r[keep],
            on="time",
            direction="nearest",
            tolerance=float(max_dist),
        )

        for c in attach_cols:
            if suffix:
                merged = merged.rename(columns={c: f"{c}{suffix}"})

        rows.append(merged)

    if rows:
        merged_all = pd.concat(rows, ignore_index=True)
        merged_all = (
            merged_all
            .sort_values("_row_id_mc")
            .drop(columns=["_row_id_mc", "session_norm_mc"], errors="ignore")
            .reset_index(drop=True)
        )
        return merged_all

    return out.drop(columns=["_row_id_mc", "session_norm_mc"], errors="ignore")

def _mc_source_feature_cols(df: pd.DataFrame) -> List[str]:
    cols = []

    for c in df.columns:
        cl = str(c).lower()

        if c in META_CONFIRMER_EXCLUDED_FEATURE_COLS:
            continue

        if cl in {"session", "split", "audio_path", "action"}:
            continue

        if "nearest_gt" in cl or cl.startswith("gt_") or "_gt_" in cl:
            continue

        if any(k in cl for k in META_SOURCE_FEATURE_KEYWORDS):
            if pd.api.types.is_numeric_dtype(df[c]):
                cols.append(c)

    return list(dict.fromkeys(cols))

def _mc_add_vap_text_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)

    for c in [
        "temporal_shift_score", "lgbm_shift_score", "lgbm_binary_shift_score",
        "ranker_shift_score", "rr_role_shift_score",
        "rr_current_activity_proxy", "rr_current_drop_proxy",
        "rr_next_activity_proxy", "rr_next_enters_proxy",
        "rr_role_switch_proxy", "rr_overlap_future_proxy",
        "rr_source_agree_count", "rr_source_agree_frac",
        "rr_asr_boundary_proxy", "rr_qwen_shift_proxy", "rr_qwen_hold_proxy",
        "rr_text_audio_agreement", "asr_boundary_score", "qwen_shift_score",
        "qwen_p_NTT", "qwen_p_ITT", "qwen_p_NA", "qwen_p_BC", "qwen_p_BI",
        "qwen_backchannel_score", "qwen_interruption_score",
        "pyannote_change_score", "pyannote_overlap_score", "ecapa_cos",
    ]:
        if c not in out.columns:
            out[c] = 0.0
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(np.float32)

    qwen_shift = _mc_clip01(out["qwen_shift_score"] + out["qwen_p_NTT"] + out["qwen_p_ITT"])
    qwen_hold = _mc_clip01(out["qwen_p_NA"] + out["qwen_p_BC"] + out["qwen_p_BI"])
    qwen_overlap = _mc_clip01(
        out["qwen_backchannel_score"]
        + out["qwen_interruption_score"]
        + out["qwen_p_BC"]
        + out["qwen_p_ITT"]
    )

    asr_boundary = _mc_clip01(
        np.maximum(
            out["asr_boundary_score"].to_numpy(dtype=np.float32),
            out["rr_asr_boundary_proxy"].to_numpy(dtype=np.float32),
        )
    )

    speaker_change = _mc_clip01(
        np.maximum(
            out["pyannote_change_score"].to_numpy(dtype=np.float32),
            1.0 - out["ecapa_cos"].to_numpy(dtype=np.float32),
        )
    )

    out["vap_shift_soon_proxy"] = _mc_clip01(
        0.30 * out["temporal_shift_score"]
        + 0.25 * out["rr_next_enters_proxy"]
        + 0.20 * out["rr_role_switch_proxy"]
        + 0.15 * qwen_shift
        + 0.10 * asr_boundary
    )

    out["vap_current_continue_proxy"] = _mc_clip01(
        0.45 * out["rr_current_activity_proxy"]
        + 0.30 * (1.0 - out["rr_current_drop_proxy"])
        + 0.25 * qwen_hold
    )

    out["vap_next_active_proxy"] = _mc_clip01(
        0.45 * out["rr_next_activity_proxy"]
        + 0.35 * out["rr_next_enters_proxy"]
        + 0.20 * speaker_change
    )

    out["vap_overlap_future_proxy"] = _mc_clip01(
        0.50 * out["rr_overlap_future_proxy"]
        + 0.30 * qwen_overlap
        + 0.20 * out["pyannote_overlap_score"]
    )

    out["turngpt_text_completion_proxy"] = _mc_clip01(
        0.35 * qwen_shift
        + 0.25 * asr_boundary
        + 0.20 * (1.0 - qwen_hold)
        + 0.20 * out["rr_text_audio_agreement"]
    )

    out["proposer_high_recall_score"] = _mc_clip01(
        np.maximum.reduce([
            out["temporal_shift_score"].to_numpy(dtype=np.float32),
            out["lgbm_binary_shift_score"].to_numpy(dtype=np.float32),
            out["ranker_shift_score"].to_numpy(dtype=np.float32),
            out["rr_role_shift_score"].to_numpy(dtype=np.float32),
            out["vap_shift_soon_proxy"].to_numpy(dtype=np.float32),
        ])
    )

    out["confirmer_consensus_score"] = _mc_clip01(
        0.22 * out["lgbm_binary_shift_score"]
        + 0.20 * out["ranker_shift_score"]
        + 0.20 * out["rr_role_shift_score"]
        + 0.16 * out["vap_shift_soon_proxy"]
        + 0.12 * out["turngpt_text_completion_proxy"]
        + 0.10 * out["rr_source_agree_frac"]
    )

    out["hold_vs_shift_margin"] = _mc_clip01(
        out["vap_shift_soon_proxy"] - out["vap_current_continue_proxy"] + 0.5
    )

    out["role_change_text_agreement"] = _mc_clip01(
        1.0 - np.abs(out["rr_role_switch_proxy"] - out["turngpt_text_completion_proxy"])
    )

    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return out

def _mc_build_frame(
    split_name: str,
    lgbm_scores: pd.DataFrame,
    temporal_scores: pd.DataFrame,
    rr_scores: Optional[pd.DataFrame],
) -> pd.DataFrame:
    if lgbm_scores is None or len(lgbm_scores) == 0:
        return pd.DataFrame()

    base_cols = [
        c for c in ["session", "time", "split", "label", "action", "action_id", "audio_path"]
        if c in lgbm_scores.columns
    ]

    out = lgbm_scores[base_cols].copy().reset_index(drop=True)
    out["session"] = out["session"].astype(str)
    out["time"] = pd.to_numeric(out["time"], errors="coerce")
    out = out.dropna(subset=["session", "time"]).reset_index(drop=True)

    score_cols = [
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "lgbm_p_NA",
        "lgbm_p_NTT",
        "lgbm_p_ITT",
        "lgbm_p_BC",
        "lgbm_p_BI",
    ]

    out = _mc_attach_cols_nearest(
        out,
        lgbm_scores,
        [c for c in score_cols if c in lgbm_scores.columns],
    )

    out = _mc_attach_cols_nearest(
        out,
        temporal_scores,
        ["temporal_shift_score"] if "temporal_shift_score" in temporal_scores.columns else [],
    )

    if rr_scores is not None and len(rr_scores):
        rr_cols = [
            "rr_role_shift_score",
            "rr_current_activity_proxy",
            "rr_current_drop_proxy",
            "rr_next_activity_proxy",
            "rr_next_enters_proxy",
            "rr_next_minus_current_activity",
            "rr_role_switch_proxy",
            "rr_overlap_future_proxy",
            "rr_qwen_shift_proxy",
            "rr_qwen_hold_proxy",
            "rr_qwen_overlap_proxy",
            "rr_pyannote_overlap_proxy",
            "rr_asr_boundary_proxy",
            "rr_source_agree_count",
            "rr_source_agree_frac",
            "rr_text_audio_agreement",
            "rr_cluster_size",
            "rr_cluster_span_s",
            "rr_cluster_pos_frac",
            "rr_prev_gap_s",
            "rr_next_gap_s",
            "rr_min_neighbor_gap_s",
        ]

        out = _mc_attach_cols_nearest(
            out,
            rr_scores,
            [c for c in rr_cols if c in rr_scores.columns],
        )

    if "SYS2_FEATURES_ALL" in globals() and isinstance(SYS2_FEATURES_ALL, pd.DataFrame) and len(SYS2_FEATURES_ALL):
        src = SYS2_FEATURES_ALL.copy()

        if "split" in src.columns:
            if split_name == "train":
                src = src[
                    src["split"].astype(str).isin([
                        "train",
                        "synth_train",
                        "real_diffusion",
                        "synth_diffusion",
                    ])
                ].copy()
            else:
                src = src[src["split"].astype(str) == str(split_name)].copy()

        source_cols = _mc_source_feature_cols(src)

        source_cols = source_cols[:220]

        out = _mc_attach_cols_nearest(out, src, source_cols)

    out = _mc_add_vap_text_features(out)

    if "label" not in out.columns:
        out["label"] = -1

    out["label"] = pd.to_numeric(out["label"], errors="coerce").fillna(-1).astype(int)
    out["meta_label"] = out["label"].astype(int)

    out["meta_target_source"] = np.where(
        out["meta_label"] == 1,
        "shift",
        np.where(out["meta_label"] == 0, "hold", "unknown"),
    )

    return out.reset_index(drop=True)

def _mc_infer_feature_cols(df: pd.DataFrame) -> List[str]:
    cols = []

    for c in df.columns:
        cl = str(c).lower()

        if c in META_CONFIRMER_EXCLUDED_FEATURE_COLS:
            continue

        if cl in META_CONFIRMER_EXCLUDED_FEATURE_COLS:
            continue

        if "nearest_gt" in cl or cl.startswith("gt_") or "_gt_" in cl:
            continue

        if "label" in cl or "target" in cl:
            continue

        if cl in {"session", "split", "audio_path", "action"}:
            continue

        if pd.api.types.is_numeric_dtype(df[c]):
            if (
                cl.endswith("_score")
                or "proxy" in cl
                or cl.startswith("source_")
                or cl.startswith("rr_")
                or cl.startswith("vap_")
                or cl.startswith("turngpt_")
                or cl.startswith("proposer_")
                or cl.startswith("confirmer_")
                or "qwen" in cl
                or "asr" in cl
                or "pyannote" in cl
                or "ecapa" in cl
                or "wavlm" in cl
                or "candidate_density" in cl
                or "agreement" in cl
                or "margin" in cl
                or "cluster" in cl
            ):
                cols.append(c)

    cols = list(dict.fromkeys(cols))

    assert "meta_nearest_gt_dist" not in cols, "Leakage guard failed: meta_nearest_gt_dist selected."
    assert "nearest_gt_dist" not in cols, "Leakage guard failed: nearest_gt_dist selected."
    assert "ranker_nearest_gt_dist" not in cols, "Leakage guard failed: ranker_nearest_gt_dist selected."
    assert "rr_nearest_gt_dist" not in cols, "Leakage guard failed: rr_nearest_gt_dist selected."

    return cols

def _mc_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": META_CONFIRMER_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "tol": float(TOL),
        "feature_cols": [str(c) for c in feature_cols],
        "excluded_feature_cols": sorted([str(c) for c in META_CONFIRMER_EXCLUDED_FEATURE_COLS]),
        "num_boost_round": int(META_CONFIRMER_NUM_BOOST_ROUND),
        "early_stopping_rounds": int(META_CONFIRMER_EARLY_STOPPING_ROUNDS),
        "learning_rate": float(META_CONFIRMER_LEARNING_RATE),
        "num_leaves": int(META_CONFIRMER_NUM_LEAVES),
        "min_data_in_leaf": int(META_CONFIRMER_MIN_DATA_IN_LEAF),
        "feature_fraction": float(META_CONFIRMER_FEATURE_FRACTION),
        "bagging_fraction": float(META_CONFIRMER_BAGGING_FRACTION),
        "bagging_freq": int(META_CONFIRMER_BAGGING_FREQ),
        "lambda_l2": float(META_CONFIRMER_LAMBDA_L2),
        "pos_weight_mult": float(META_CONFIRMER_POS_WEIGHT_MULT),
        "synth_weight_mult": float(META_CONFIRMER_SYNTH_WEIGHT_MULT),
        "diffusion_weight_mult": float(META_CONFIRMER_DIFFUSION_WEIGHT_MULT),
        "uses_synthetic_data": bool(globals().get("USE_SYNTH_MULTIPARTY", False)),
        "uses_diffusion_aug": bool(globals().get("USE_DIFFUSION_AUG", False)),
        "lgbm_train_scores_fp": _mc_file_fingerprint(LGBM_TRAIN_SCORES_PATH) if "LGBM_TRAIN_SCORES_PATH" in globals() else None,
        "rr_fap_train_scores_fp": _mc_file_fingerprint(RR_FAP_TRAIN_SCORES_PATH) if "RR_FAP_TRAIN_SCORES_PATH" in globals() else None,
    }

def _mc_core_model_config(feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": META_CONFIRMER_VARIANT,
        "feature_cols": [str(c) for c in feature_cols],
        "excluded_feature_cols": sorted([str(c) for c in META_CONFIRMER_EXCLUDED_FEATURE_COLS]),
        "tol": float(TOL),
    }

def _mc_score_config(split_name: str, frame: pd.DataFrame, feature_cols: List[str]) -> Dict[str, Any]:
    return {
        "variant": META_CONFIRMER_VARIANT,
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "seed": int(SEED),
        "split_name": str(split_name),
        "n_rows": int(len(frame)) if frame is not None else 0,
        "feature_cols": [str(c) for c in feature_cols],
        "excluded_feature_cols": sorted([str(c) for c in META_CONFIRMER_EXCLUDED_FEATURE_COLS]),
        "model_meta_fp": _mc_file_fingerprint(META_CONFIRMER_MODEL_META_PATH),
        "test_labels_used_for_generation": False,
        "cache_format": "pickle",
    }

def _mc_training_weights(df: pd.DataFrame) -> np.ndarray:
    label = pd.to_numeric(df["meta_label"], errors="coerce").fillna(-1).astype(int).to_numpy()
    split = df.get("split", pd.Series(["train"] * len(df))).astype(str).str.lower()

    w = np.ones(len(df), dtype=np.float32)

    pos = label == 1
    neg = label == 0

    if pos.sum() > 0 and neg.sum() > 0:
        pos_w = (float(neg.sum()) / max(1.0, float(pos.sum()))) * float(META_CONFIRMER_POS_WEIGHT_MULT)
        w[pos] *= float(pos_w)

    w[split.str.contains("synth", regex=False).to_numpy()] *= float(META_CONFIRMER_SYNTH_WEIGHT_MULT)
    w[split.str.contains("diffusion", regex=False).to_numpy()] *= float(META_CONFIRMER_DIFFUSION_WEIGHT_MULT)

    if "sample_weight" in df.columns:
        sw = pd.to_numeric(df["sample_weight"], errors="coerce").fillna(1.0).to_numpy(dtype=np.float32)
        w *= np.maximum(sw, 1e-6)

    return np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0).astype(np.float32)

def _mc_load_feature_cols_from_file_or_meta() -> Optional[List[str]]:
    if META_CONFIRMER_FEATURE_COLS_PATH.exists():
        try:
            cols = json.loads(META_CONFIRMER_FEATURE_COLS_PATH.read_text())
            cols = [str(c) for c in cols]

            leaked = [
                c for c in cols
                if c in META_CONFIRMER_EXCLUDED_FEATURE_COLS
                or "nearest_gt" in c.lower()
                or "label" in c.lower()
                or "target" in c.lower()
            ]

            if leaked:
                print("⚠️ Meta-confirmer feature file contains leaky features:", leaked)
                return None

            return cols

        except Exception as e:
            print("Could not load meta-confirmer feature columns file:", e)

    for p in [
        META_CONFIRMER_VAL_SCORES_META_PATH,
        META_CONFIRMER_TEST_SCORES_META_PATH,
        META_CONFIRMER_TRAIN_SCORES_META_PATH,
    ]:
        if not p.exists():
            continue

        try:
            meta = json.loads(p.read_text())
            cols = [str(c) for c in meta.get("feature_cols", [])]

            leaked = [
                c for c in cols
                if c in META_CONFIRMER_EXCLUDED_FEATURE_COLS
                or "nearest_gt" in c.lower()
                or "label" in c.lower()
                or "target" in c.lower()
            ]

            if leaked:
                print("⚠️ Meta-confirmer score meta contains leaky features:", leaked)
                return None

            if cols:
                return cols

        except Exception:
            continue

    return None

def _mc_model_meta_is_compatible(feature_cols: List[str]) -> bool:
    if not META_CONFIRMER_MODEL_META_PATH.exists():
        return False

    try:
        saved_meta = json.loads(META_CONFIRMER_MODEL_META_PATH.read_text())
    except Exception as e:
        print("⚠️ Could not read meta-confirmer model meta:", e)
        return False

    if STRICT_META_CONFIRMER_MODEL_META_CHECK:
        expected = _mc_model_config(feature_cols)
    else:
        expected = _mc_core_model_config(feature_cols)

    for k, expected_v in expected.items():
        saved_v = saved_meta.get(k, None)

        if saved_v != expected_v:
            print(f"⚠️ Meta-confirmer model meta mismatch at key: {k}")
            print("  expected:", expected_v)
            print("  saved:   ", saved_v)
            return False

    return True

def _mc_score_cache_core_compatible(meta_path: Path, feature_cols: List[str]) -> bool:
    if not Path(meta_path).exists():
        return False

    try:
        meta = json.loads(Path(meta_path).read_text())
    except Exception:
        return False

    if meta.get("variant", None) != META_CONFIRMER_VARIANT:
        return False

    saved_cols = [str(c) for c in meta.get("feature_cols", [])]

    if saved_cols != [str(c) for c in feature_cols]:
        return False

    leaked = [
        c for c in saved_cols
        if c in META_CONFIRMER_EXCLUDED_FEATURE_COLS
        or "nearest_gt" in c.lower()
        or "label" in c.lower()
        or "target" in c.lower()
    ]

    if leaked:
        return False

    return True

def _all_meta_confirmer_score_caches_usable(feature_cols: Optional[List[str]]) -> bool:
    if not feature_cols:
        return False

    checks = [
        (META_CONFIRMER_TRAIN_SCORES_PATH, META_CONFIRMER_TRAIN_SCORES_META_PATH),
        (META_CONFIRMER_VAL_SCORES_PATH, META_CONFIRMER_VAL_SCORES_META_PATH),
        (META_CONFIRMER_TEST_SCORES_PATH, META_CONFIRMER_TEST_SCORES_META_PATH),
    ]

    for data_path, meta_path in checks:
        if not Path(data_path).exists():
            return False

        if not _mc_score_cache_core_compatible(meta_path, feature_cols):
            return False

    return True

def _clear_stale_meta_confirmer_score_caches():
    for p in [
        META_CONFIRMER_TRAIN_SCORES_PATH,
        META_CONFIRMER_VAL_SCORES_PATH,
        META_CONFIRMER_TEST_SCORES_PATH,
        META_CONFIRMER_TRAIN_SCORES_META_PATH,
        META_CONFIRMER_VAL_SCORES_META_PATH,
        META_CONFIRMER_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale meta-confirmer score cache:", p)
        except Exception as e:
            print("Could not remove stale meta-confirmer score cache:", p, e)

def _train_meta_confirmer(
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
    feature_cols: List[str],
):
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    tr = train_frame[train_frame["meta_label"].astype(int).isin([0, 1])].copy()
    va = val_frame[val_frame["meta_label"].astype(int).isin([0, 1])].copy()

    assert len(tr) > 0, "No labelled rows for meta-confirmer training."
    assert len(va) > 0, "No labelled rows for meta-confirmer validation."
    assert int((tr["meta_label"].astype(int) == 1).sum()) > 0, "Meta-confirmer train set has no SHIFT positives."
    assert int((tr["meta_label"].astype(int) == 0).sum()) > 0, "Meta-confirmer train set has no HOLD negatives."

    Xtr = tr[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    Xva = va[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    ytr = tr["meta_label"].astype(int).to_numpy()
    yva = va["meta_label"].astype(int).to_numpy()

    wtr = _mc_training_weights(tr)
    wva = np.ones(len(va), dtype=np.float32)

    dtr = lgb.Dataset(
        Xtr,
        label=ytr,
        weight=wtr,
        feature_name=feature_cols,
        free_raw_data=False,
    )

    dva = lgb.Dataset(
        Xva,
        label=yva,
        weight=wva,
        reference=dtr,
        feature_name=feature_cols,
        free_raw_data=False,
    )

    params = {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "learning_rate": float(META_CONFIRMER_LEARNING_RATE),
        "num_leaves": int(META_CONFIRMER_NUM_LEAVES),
        "min_data_in_leaf": int(META_CONFIRMER_MIN_DATA_IN_LEAF),
        "feature_fraction": float(META_CONFIRMER_FEATURE_FRACTION),
        "bagging_fraction": float(META_CONFIRMER_BAGGING_FRACTION),
        "bagging_freq": int(META_CONFIRMER_BAGGING_FREQ),
        "lambda_l2": float(META_CONFIRMER_LAMBDA_L2),
        "verbosity": -1,
        "seed": int(SEED),
        "feature_pre_filter": False,
        "force_col_wise": True,
    }

    evals_result = {}

    model = lgb.train(
        params,
        dtr,
        num_boost_round=int(META_CONFIRMER_NUM_BOOST_ROUND),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=int(META_CONFIRMER_EARLY_STOPPING_ROUNDS),
                first_metric_only=True,
                verbose=True,
            ),
            lgb.log_evaluation(period=int(META_CONFIRMER_LOG_EVERY)),
            lgb.record_evaluation(evals_result),
        ],
    )

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        model.save_model(str(META_CONFIRMER_MODEL_PATH), num_iteration=best_iter)
    else:
        model.save_model(str(META_CONFIRMER_MODEL_PATH))

    expected_meta = _mc_model_config(feature_cols)
    expected_meta.update({
        "best_iteration": int(best_iter),
        "best_score": _mc_json_safe(getattr(model, "best_score", {})),
        "trained_time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "train_rows": int(len(tr)),
        "val_rows": int(len(va)),
        "train_positive_rows": int((tr["meta_label"].astype(int) == 1).sum()),
        "val_positive_rows": int((va["meta_label"].astype(int) == 1).sum()),
    })

    META_CONFIRMER_MODEL_META_PATH.write_text(
        json.dumps(_mc_json_safe(expected_meta), indent=2, sort_keys=True)
    )

    META_CONFIRMER_FEATURE_COLS_PATH.write_text(
        json.dumps([str(c) for c in feature_cols], indent=2)
    )

    META_CONFIRMER_HISTORY_PATH.write_text(
        json.dumps(_mc_json_safe(evals_result), indent=2, sort_keys=True)
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "importance_gain": model.feature_importance(importance_type="gain"),
        "importance_split": model.feature_importance(importance_type="split"),
    }).sort_values("importance_gain", ascending=False)

    imp.to_csv(META_CONFIRMER_IMPORTANCE_PATH, index=False)

    print("Saved meta-confirmer:", META_CONFIRMER_MODEL_PATH)
    print("Saved meta-confirmer meta:", META_CONFIRMER_MODEL_META_PATH)
    print("Saved meta-confirmer feature columns:", META_CONFIRMER_FEATURE_COLS_PATH)
    print("Saved meta-confirmer feature importance:", META_CONFIRMER_IMPORTANCE_PATH)

    _clear_stale_meta_confirmer_score_caches()

    return model

def _load_meta_confirmer_model(feature_cols: List[str]):
    try:
        import lightgbm as lgb
    except Exception as e:
        raise ImportError("LightGBM is required. Install with: pip install -U lightgbm") from e

    if not META_CONFIRMER_MODEL_PATH.exists():
        return None

    if not _mc_model_meta_is_compatible(feature_cols):
        print("⚠️ Existing meta-confirmer model is incompatible with current config.")
        return None

    model = lgb.Booster(model_file=str(META_CONFIRMER_MODEL_PATH))

    print("Loaded meta-confirmer:", META_CONFIRMER_MODEL_PATH)

    return model

def _train_or_load_meta_confirmer(
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
    feature_cols: List[str],
):
    if META_CONFIRMER_MODEL_PATH.exists() and not FORCE_RETRAIN_META_CONFIRMER:
        model = _load_meta_confirmer_model(feature_cols)

        if model is not None:
            return model

        msg = (
            "Meta-confirmer model exists but is incompatible with the current features/config:\n"
            f"{META_CONFIRMER_MODEL_PATH}\n\n"
            "In cache-safe resume mode, this cell will not retrain automatically."
        )

        if CACHE_SAFE_RESUME_RUN and _all_meta_confirmer_score_caches_usable(feature_cols):
            print("⚠️", msg)
            print("Compatible meta-confirmer score caches exist, so continuing in score-cache-only mode.")
            return None

        if CACHE_SAFE_RESUME_RUN:
            raise RuntimeError(
                msg
                + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_META_CONFIRMER=True only if you intentionally want to retrain."
            )

        print("No compatible meta-confirmer found. Training a new one.")
        return _train_meta_confirmer(train_frame, val_frame, feature_cols)

    if FORCE_RETRAIN_META_CONFIRMER:
        print("FORCE_RETRAIN_META_CONFIRMER=True, training meta-confirmer.")
        return _train_meta_confirmer(train_frame, val_frame, feature_cols)

    msg = (
        "No meta-confirmer model found:\n"
        f"{META_CONFIRMER_MODEL_PATH}\n\n"
        "In cache-safe resume mode, this cell will not train automatically."
    )

    if CACHE_SAFE_RESUME_RUN and _all_meta_confirmer_score_caches_usable(feature_cols):
        print("⚠️", msg)
        print("Compatible meta-confirmer score caches exist, so continuing in score-cache-only mode.")
        return None

    if CACHE_SAFE_RESUME_RUN:
        raise FileNotFoundError(
            msg
            + "\nSet CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_META_CONFIRMER=True only if you intentionally want to train."
        )

    return _train_meta_confirmer(train_frame, val_frame, feature_cols)

def _validate_meta_confirmer_score_cache(df: pd.DataFrame, split_name: str) -> bool:
    if df is None or len(df) == 0:
        return False

    required_cols = {
        "session",
        "time",
        "split",
        "meta_confirmer_shift_score",
    }

    missing = required_cols.difference(set(df.columns))

    if missing:
        print(f"Meta-confirmer {split_name} score cache missing columns:", sorted(missing))
        return False

    return True

def _score_meta_confirmer(
    model,
    frame: pd.DataFrame,
    feature_cols: List[str],
    split_name: str,
    out_path: Path,
    meta_path: Path,
) -> pd.DataFrame:
    expected = _mc_score_config(split_name, frame, feature_cols)

    if out_path.exists() and not FORCE_RETRAIN_META_CONFIRMER:
        meta_matches = _mc_meta_matches(meta_path, expected)
        core_compatible = _mc_score_cache_core_compatible(meta_path, feature_cols)

        if meta_matches:
            print(f"Meta-confirmer score metadata matches for {split_name}.")
        else:
            print(f"⚠️ Meta-confirmer score metadata mismatch for {split_name}.")

            if not core_compatible:
                print(
                    f"Meta-confirmer {split_name} cache is not core-compatible with the current configuration. "
                    "It will not be loaded."
                )
            elif STRICT_META_CONFIRMER_SCORE_META_CHECK:
                print(f"STRICT_META_CONFIRMER_SCORE_META_CHECK=True, so {split_name} scores will be recomputed.")
            else:
                print(
                    f"Cache-safe resume mode: loading existing compatible meta-confirmer {split_name} scores anyway.\n"
                    f"Loaded from: {out_path}"
                )

        if meta_matches or (core_compatible and not STRICT_META_CONFIRMER_SCORE_META_CHECK):
            cached = pd.read_pickle(out_path)

            if not _validate_meta_confirmer_score_cache(cached, split_name):
                raise RuntimeError(
                    f"Cached meta-confirmer {split_name} scores are incompatible:\n"
                    f"{out_path}\n\n"
                    "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_META_CONFIRMER=True only if you intentionally want to rebuild."
                )

            print(f"Loaded meta-confirmer {split_name} scores:", cached.shape)
            return cached

    if model is None:
        if CACHE_SAFE_RESUME_RUN:
            raise FileNotFoundError(
                f"No compatible meta-confirmer model is loaded and no usable cached {split_name} scores were found.\n"
                f"Expected score cache: {out_path}\n\n"
                "In cache-safe resume mode, this cell will not train/rescore automatically.\n"
                "Set CACHE_SAFE_RESUME_RUN=False and FORCE_RETRAIN_META_CONFIRMER=True only if you intentionally want to rebuild this."
            )

        return pd.DataFrame()

    if frame is None or len(frame) == 0:
        return pd.DataFrame()

    X = frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    best_iter = int(getattr(model, "best_iteration", 0) or 0)

    if best_iter > 0:
        score = model.predict(X, num_iteration=best_iter)
    else:
        score = model.predict(X)

    score = np.asarray(score, dtype=np.float32).reshape(-1)

    base_cols = [
        c for c in ["session", "time", "split", "label", "action", "action_id", "audio_path"]
        if c in frame.columns
    ]

    keep_extra = [
        "temporal_shift_score",
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "vap_shift_soon_proxy",
        "vap_current_continue_proxy",
        "vap_next_active_proxy",
        "vap_overlap_future_proxy",
        "turngpt_text_completion_proxy",
        "proposer_high_recall_score",
        "confirmer_consensus_score",
        "hold_vs_shift_margin",
        "role_change_text_agreement",
    ]

    keep_extra = [c for c in keep_extra if c in frame.columns]

    out = frame[base_cols + keep_extra].copy().reset_index(drop=True)
    out["meta_confirmer_shift_score"] = pd.Series(score).clip(0.0, 1.0).astype(np.float32)
    out["meta_pred_label"] = (out["meta_confirmer_shift_score"] >= 0.5).astype(int)
    out["meta_feature_count"] = int(len(feature_cols))
    out["meta_confirmer_variant"] = META_CONFIRMER_VARIANT

    out.to_pickle(out_path)
    meta_path.write_text(json.dumps(_mc_json_safe(expected), indent=2, sort_keys=True))

    print(f"Saved meta-confirmer {split_name} scores:", out_path)

    return out

def _merge_meta_scores_into_lgbm_scores(main_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    if main_df is None or len(main_df) == 0:
        return pd.DataFrame()

    out = main_df.copy().reset_index(drop=True)
    out = out.drop(columns=["meta_confirmer_shift_score"], errors="ignore")

    if meta_df is None or len(meta_df) == 0 or "meta_confirmer_shift_score" not in meta_df.columns:
        out["meta_confirmer_shift_score"] = np.nan
        return out

    b = meta_df[["session", "time", "meta_confirmer_shift_score"]].copy()

    out = _mc_attach_cols_nearest(
        out,
        b,
        ["meta_confirmer_shift_score"],
        max_dist=float(META_CONFIRMER_ATTACH_TOL_S),
    )

    out["meta_confirmer_shift_score"] = pd.to_numeric(
        out["meta_confirmer_shift_score"],
        errors="coerce",
    )

    return out

if FORCE_RETRAIN_META_CONFIRMER and not CACHE_SAFE_RESUME_RUN:
    for p in [
        META_CONFIRMER_MODEL_PATH,
        META_CONFIRMER_MODEL_META_PATH,
        META_CONFIRMER_FEATURE_COLS_PATH,
        META_CONFIRMER_IMPORTANCE_PATH,
        META_CONFIRMER_HISTORY_PATH,
        META_CONFIRMER_TRAIN_SCORES_PATH,
        META_CONFIRMER_VAL_SCORES_PATH,
        META_CONFIRMER_TEST_SCORES_PATH,
        META_CONFIRMER_TRAIN_SCORES_META_PATH,
        META_CONFIRMER_VAL_SCORES_META_PATH,
        META_CONFIRMER_TEST_SCORES_META_PATH,
    ]:
        try:
            if Path(p).exists():
                Path(p).unlink()
                print("Removed stale meta-confirmer artifact:", p)
        except Exception as e:
            print("Could not remove meta-confirmer artifact:", p, e)

if not USE_META_CONFIRMER:
    print("USE_META_CONFIRMER=False; creating passthrough meta scores from RR-FAP/binary/ranker.")

    META_CONFIRMER_TRAIN_FRAME = pd.DataFrame()
    META_CONFIRMER_VAL_FRAME = pd.DataFrame()
    META_CONFIRMER_TEST_FRAME = pd.DataFrame()
    META_CONFIRMER_FEATURE_COLS = []
    META_CONFIRMER_MODEL = None

    META_CONFIRMER_TRAIN_SCORES = pd.DataFrame()
    META_CONFIRMER_VAL_SCORES = pd.DataFrame()
    META_CONFIRMER_TEST_SCORES = pd.DataFrame()

else:
    print("Building meta-confirmer frames...")

    META_CONFIRMER_TRAIN_FRAME = _mc_build_frame(
        "train",
        LGBM_TRAIN_SCORES,
        TEMPORAL_TRAIN_SCORES,
        globals().get("RR_FAP_TRAIN_SCORES", None),
    )

    META_CONFIRMER_VAL_FRAME = _mc_build_frame(
        "val",
        LGBM_VAL_SCORES,
        TEMPORAL_VAL_SCORES,
        globals().get("RR_FAP_VAL_SCORES", None),
    )

    META_CONFIRMER_TEST_FRAME = _mc_build_frame(
        "test",
        LGBM_TEST_SCORES,
        TEMPORAL_TEST_SCORES,
        globals().get("RR_FAP_TEST_SCORES", None),
    )

    train_known = META_CONFIRMER_TRAIN_FRAME[
        META_CONFIRMER_TRAIN_FRAME["meta_label"].astype(int).isin([0, 1])
    ].copy()

    val_known = META_CONFIRMER_VAL_FRAME[
        META_CONFIRMER_VAL_FRAME["meta_label"].astype(int).isin([0, 1])
    ].copy()

    inferred_cols = _mc_infer_feature_cols(META_CONFIRMER_TRAIN_FRAME)
    cached_cols = _mc_load_feature_cols_from_file_or_meta()

    if META_CONFIRMER_MODEL_PATH.exists() and cached_cols:
        META_CONFIRMER_FEATURE_COLS = cached_cols
        print("Using cached meta-confirmer feature columns:", len(META_CONFIRMER_FEATURE_COLS))
    else:
        META_CONFIRMER_FEATURE_COLS = inferred_cols
        print("Using inferred meta-confirmer feature columns:", len(META_CONFIRMER_FEATURE_COLS))

    assert len(META_CONFIRMER_FEATURE_COLS) > 0, "No deployable meta-confirmer features were selected."
    assert "meta_nearest_gt_dist" not in META_CONFIRMER_FEATURE_COLS
    assert "ranker_nearest_gt_dist" not in META_CONFIRMER_FEATURE_COLS
    assert "rr_nearest_gt_dist" not in META_CONFIRMER_FEATURE_COLS

    for frame_name in [
        "META_CONFIRMER_TRAIN_FRAME",
        "META_CONFIRMER_VAL_FRAME",
        "META_CONFIRMER_TEST_FRAME",
    ]:
        frame = globals()[frame_name]

        for c in META_CONFIRMER_FEATURE_COLS:
            if c not in frame.columns:
                frame[c] = 0.0

            frame[c] = (
                pd.to_numeric(frame[c], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
            )

        globals()[frame_name] = frame

    print("META_CONFIRMER_TRAIN_FRAME:", META_CONFIRMER_TRAIN_FRAME.shape)
    print("META_CONFIRMER_VAL_FRAME:", META_CONFIRMER_VAL_FRAME.shape)
    print("META_CONFIRMER_TEST_FRAME:", META_CONFIRMER_TEST_FRAME.shape)
    print("Selected meta-confirmer feature count:", len(META_CONFIRMER_FEATURE_COLS))

    print("Synthetic/diffusion training rows by split:")
    if "split" in META_CONFIRMER_TRAIN_FRAME.columns:
        display(META_CONFIRMER_TRAIN_FRAME["split"].astype(str).value_counts(dropna=False).to_frame("rows"))

    print("Train label counts:")
    display(train_known["meta_label"].value_counts(dropna=False).sort_index().to_frame("rows"))

    print("VAL label counts:")
    display(val_known["meta_label"].value_counts(dropna=False).sort_index().to_frame("rows"))

    META_CONFIRMER_MODEL = _train_or_load_meta_confirmer(
        META_CONFIRMER_TRAIN_FRAME,
        META_CONFIRMER_VAL_FRAME,
        META_CONFIRMER_FEATURE_COLS,
    )

    META_CONFIRMER_TRAIN_SCORES = _score_meta_confirmer(
        META_CONFIRMER_MODEL,
        META_CONFIRMER_TRAIN_FRAME,
        META_CONFIRMER_FEATURE_COLS,
        "train",
        META_CONFIRMER_TRAIN_SCORES_PATH,
        META_CONFIRMER_TRAIN_SCORES_META_PATH,
    )

    META_CONFIRMER_VAL_SCORES = _score_meta_confirmer(
        META_CONFIRMER_MODEL,
        META_CONFIRMER_VAL_FRAME,
        META_CONFIRMER_FEATURE_COLS,
        "val",
        META_CONFIRMER_VAL_SCORES_PATH,
        META_CONFIRMER_VAL_SCORES_META_PATH,
    )

    META_CONFIRMER_TEST_SCORES = _score_meta_confirmer(
        META_CONFIRMER_MODEL,
        META_CONFIRMER_TEST_FRAME,
        META_CONFIRMER_FEATURE_COLS,
        "test",
        META_CONFIRMER_TEST_SCORES_PATH,
        META_CONFIRMER_TEST_SCORES_META_PATH,
    )

    LGBM_TRAIN_SCORES = _merge_meta_scores_into_lgbm_scores(
        LGBM_TRAIN_SCORES,
        META_CONFIRMER_TRAIN_SCORES,
    )

    LGBM_VAL_SCORES = _merge_meta_scores_into_lgbm_scores(
        LGBM_VAL_SCORES,
        META_CONFIRMER_VAL_SCORES,
    )

    LGBM_TEST_SCORES = _merge_meta_scores_into_lgbm_scores(
        LGBM_TEST_SCORES,
        META_CONFIRMER_TEST_SCORES,
    )

    assert "meta_confirmer_shift_score" in LGBM_TRAIN_SCORES.columns, (
        "Augmented LGBM_TRAIN_SCORES missing meta_confirmer_shift_score."
    )

    assert "meta_confirmer_shift_score" in LGBM_VAL_SCORES.columns, (
        "Augmented LGBM_VAL_SCORES missing meta_confirmer_shift_score."
    )

    assert "meta_confirmer_shift_score" in LGBM_TEST_SCORES.columns, (
        "Augmented LGBM_TEST_SCORES missing meta_confirmer_shift_score."
    )

    LGBM_TRAIN_SCORES.to_pickle(LGBM_TRAIN_SCORES_PATH)
    LGBM_VAL_SCORES.to_pickle(LGBM_VAL_SCORES_PATH)
    LGBM_TEST_SCORES.to_pickle(LGBM_TEST_SCORES_PATH)

    LGBM_HAS_META_CONFIRMER_SCORE = True

    print("META_CONFIRMER_TRAIN_SCORES:", META_CONFIRMER_TRAIN_SCORES.shape)
    print("META_CONFIRMER_VAL_SCORES:", META_CONFIRMER_VAL_SCORES.shape)
    print("META_CONFIRMER_TEST_SCORES:", META_CONFIRMER_TEST_SCORES.shape)
    print("Augmented LGBM_TRAIN_SCORES includes meta-confirmer:", "meta_confirmer_shift_score" in LGBM_TRAIN_SCORES.columns)
    print("Augmented LGBM_VAL_SCORES includes meta-confirmer:", "meta_confirmer_shift_score" in LGBM_VAL_SCORES.columns)
    print("Augmented LGBM_TEST_SCORES includes meta-confirmer:", "meta_confirmer_shift_score" in LGBM_TEST_SCORES.columns)
    print("meta nearest GT leakage feature used:", any("nearest_gt" in str(c).lower() for c in META_CONFIRMER_FEATURE_COLS))

    if len(META_CONFIRMER_VAL_SCORES):
        val_known_scores = META_CONFIRMER_VAL_SCORES[
            META_CONFIRMER_VAL_SCORES["label"].astype(int).isin([0, 1])
        ].copy()

        if len(val_known_scores):
            y_true = val_known_scores["label"].astype(int).to_numpy()
            y_score = (
                pd.to_numeric(val_known_scores["meta_confirmer_shift_score"], errors="coerce")
                .fillna(0.0)
                .to_numpy(dtype=float)
            )
            y_pred = (y_score >= 0.5).astype(int)

            p, r, f1, _ = precision_recall_fscore_support(
                y_true,
                y_pred,
                average="binary",
                zero_division=0,
            )

            ap = average_precision_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")
            auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")

            print(
                f"Meta-confirmer VAL row-level @0.50: "
                f"P={p:.4f} R={r:.4f} F1={f1:.4f} AP={ap:.4f} AUC={auc:.4f}"
            )

        display(META_CONFIRMER_VAL_SCORES.head())

    if len(META_CONFIRMER_TEST_SCORES):
        display(META_CONFIRMER_TEST_SCORES["label"].value_counts(dropna=False).to_frame("test_label_count"))

        assert not META_CONFIRMER_TEST_SCORES["label"].astype(int).isin([0, 1]).any(), (
            "TEST labels leaked into META_CONFIRMER_TEST_SCORES. They must remain -1."
        )

globals()["META_CONFIRMER_VARIANT"] = META_CONFIRMER_VARIANT
globals()["META_CONFIRMER_FEATURE_COLS"] = META_CONFIRMER_FEATURE_COLS if "META_CONFIRMER_FEATURE_COLS" in globals() else []
globals()["META_CONFIRMER_TRAIN_FRAME"] = globals().get("META_CONFIRMER_TRAIN_FRAME", pd.DataFrame())
globals()["META_CONFIRMER_VAL_FRAME"] = globals().get("META_CONFIRMER_VAL_FRAME", pd.DataFrame())
globals()["META_CONFIRMER_TEST_FRAME"] = globals().get("META_CONFIRMER_TEST_FRAME", pd.DataFrame())
globals()["META_CONFIRMER_MODEL"] = globals().get("META_CONFIRMER_MODEL", None)
globals()["META_CONFIRMER_TRAIN_SCORES"] = globals().get("META_CONFIRMER_TRAIN_SCORES", pd.DataFrame())
globals()["META_CONFIRMER_VAL_SCORES"] = globals().get("META_CONFIRMER_VAL_SCORES", pd.DataFrame())
globals()["META_CONFIRMER_TEST_SCORES"] = globals().get("META_CONFIRMER_TEST_SCORES", pd.DataFrame())
globals()["META_CONFIRMER_TRAIN_SCORES_PATH"] = META_CONFIRMER_TRAIN_SCORES_PATH
globals()["META_CONFIRMER_VAL_SCORES_PATH"] = META_CONFIRMER_VAL_SCORES_PATH
globals()["META_CONFIRMER_TEST_SCORES_PATH"] = META_CONFIRMER_TEST_SCORES_PATH
globals()["META_CONFIRMER_MODEL_PATH"] = META_CONFIRMER_MODEL_PATH
globals()["META_CONFIRMER_MODEL_META_PATH"] = META_CONFIRMER_MODEL_META_PATH
globals()["META_CONFIRMER_FEATURE_COLS_PATH"] = META_CONFIRMER_FEATURE_COLS_PATH
globals()["META_CONFIRMER_IMPORTANCE_PATH"] = META_CONFIRMER_IMPORTANCE_PATH
globals()["META_CONFIRMER_HISTORY_PATH"] = META_CONFIRMER_HISTORY_PATH
globals()["LGBM_TRAIN_SCORES"] = LGBM_TRAIN_SCORES
globals()["LGBM_VAL_SCORES"] = LGBM_VAL_SCORES
globals()["LGBM_TEST_SCORES"] = LGBM_TEST_SCORES
globals()["LGBM_HAS_META_CONFIRMER_SCORE"] = bool("meta_confirmer_shift_score" in LGBM_VAL_SCORES.columns)

print("✅ Cell 19E complete: meta-confirmer loaded/scored and merged into LGBM score tables.")

Meta-confirmer settings:
  use_meta_confirmer: True
  variant: meta_confirmer_v1_vap_rrfap_text_synth_diffusion
  force_retrain: False
  cache_safe_resume: False
  strict_model_meta_check: False
  strict_score_meta_check: False
Building meta-confirmer frames...


Using inferred meta-confirmer feature columns: 73
META_CONFIRMER_TRAIN_FRAME: (206097, 91)
META_CONFIRMER_VAL_FRAME: (7968, 91)
META_CONFIRMER_TEST_FRAME: (99307, 91)
Selected meta-confirmer feature count: 73
Synthetic/diffusion training rows by split:


,rows
split,
synth_train,168975
train,37122


Train label counts:


,rows
meta_label,
0,162129
1,43968


...

Saved meta-confirmer train scores: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/lightgbm_verifier/meta_confirmer_vap_rrfap_text/meta_confirmer_train_scores.pkl
Saved meta-confirmer val scores: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/lightgbm_verifier/meta_confirmer_vap_rrfap_text/meta_confirmer_val_scores.pkl
Saved meta-confirmer test scores: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/models/lightgbm_verifier/meta_confirmer_vap_rrfap_text/meta_confirmer_test_scores.pkl


META_CONFIRMER_TRAIN_SCORES: (206097, 25)
META_CONFIRMER_VAL_SCORES: (7968, 25)
META_CONFIRMER_TEST_SCORES: (99307, 25)
Augmented LGBM_TRAIN_SCORES includes meta-confirmer: True
Augmented LGBM_VAL_SCORES includes meta-confirmer: True
Augmented LGBM_TEST_SCORES includes meta-confirmer: True
meta nearest GT leakage feature used: False
Meta-confirmer VAL row-level @0.50: P=0.6169 R=0.4868 F1=0.5442 AP=0.5927 AUC=0.8812


,session,time,split,label,action,action_id,audio_path,temporal_shift_score,lgbm_shift_score,lgbm_binary_shift_score,...,vap_overlap_future_proxy,turngpt_text_completion_proxy,proposer_high_recall_score,confirmer_consensus_score,hold_vs_shift_margin,role_change_text_agreement,meta_confirmer_shift_score,meta_pred_label,meta_feature_count,meta_confirmer_variant
0,ahnss,0.090504,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.087005,0.002104,0.001628,...,0.500000,0.590447,0.532800,0.263768,0.582790,0.929944,0.040899,0,73,meta_confirmer_v1_vap_rrfap_text_synth_diffusion
1,ahnss,1.331692,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.087005,0.001056,0.000416,...,0.356928,0.518094,0.549575,0.257503,0.555107,0.932691,0.040899,0,73,meta_confirmer_v1_vap_rrfap_text_synth_diffusion
2,ahnss,9.000000,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.087005,0.000010,0.000015,...,0.014541,0.303520,0.313724,0.193930,0.278591,0.988839,0.040899,0,73,meta_confirmer_v1_vap_rrfap_text_synth_diffusion
3,ahnss,17.429841,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.342473,0.000307,0.000530,...,0.113768,0.490829,0.504723,0.247080,0.545793,0.948722,0.040899,0,73,meta_confirmer_v1_vap_rrfap_text_synth_diffusion
4,ahnss,18.300001,val,0,NA,0,/Documents/Turn_Taking/Tech_paper/R...,0.342473,0.040051,0.019294,...,0.054405,0.132341,0.342473,0.174954,0.410530,0.749226,0.040899,0,73,meta_confirmer_v1_vap_rrfap_text_synth_diffusion


,test_label_count
label,
-1,99307


✅ Cell 19E complete: meta-confirmer loaded/scored and merged into LGBM score tables.


In [29]:
import hashlib
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error

TIMING_CORRECTOR_VARIANT = "local_offset_huber_train_internal_val_v2"
TIMING_CORRECTOR_DIR = Path(TIMING_CORRECTOR_DIR)
TIMING_CORRECTOR_DIR.mkdir(parents=True, exist_ok=True)

TIMING_CORRECTOR_MODEL_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_lightgbm.txt"
)
TIMING_CORRECTOR_META_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_meta.json"
)
TIMING_CORRECTOR_FEATURES_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_feature_columns.json"
)
TIMING_CORRECTOR_TRAIN_PAIRS_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_train_pairs.pkl"
)
TIMING_CORRECTOR_VAL_PAIRS_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_val_pairs.pkl"
)
TIMING_CORRECTOR_TRAIN_SCORES_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_train_scores.pkl"
)
TIMING_CORRECTOR_VAL_SCORES_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_val_scores.pkl"
)
TIMING_CORRECTOR_TEST_SCORES_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_test_scores.pkl"
)
TIMING_CORRECTOR_DIAGNOSTICS_PATH = (
    TIMING_CORRECTOR_DIR
    / "timing_corrector_val_diagnostics.csv"
)

TIMING_CORRECTOR_MATCH_RADIUS_S = float(
    globals().get(
        "TIMING_CORRECTOR_MATCH_RADIUS_S",
        1.00,
    )
)
TIMING_CORRECTOR_MAX_ABS_OFFSET_S = float(
    globals().get(
        "TIMING_CORRECTOR_MAX_ABS_OFFSET_S",
        0.75,
    )
)
TIMING_CORRECTOR_MAX_CANDIDATES_PER_EVENT = int(
    globals().get(
        "TIMING_CORRECTOR_MAX_CANDIDATES_PER_EVENT",
        5,
    )
)
TIMING_CORRECTOR_NUM_BOOST_ROUND = int(
    globals().get(
        "TIMING_CORRECTOR_NUM_BOOST_ROUND",
        2500,
    )
)
TIMING_CORRECTOR_EARLY_STOPPING_ROUNDS = int(
    globals().get(
        "TIMING_CORRECTOR_EARLY_STOPPING_ROUNDS",
        150,
    )
)

TIMING_CORRECTOR_INTERNAL_VAL_FRACTION = float(
    globals().get(
        "TIMING_CORRECTOR_INTERNAL_VAL_FRACTION",
        0.15,
    )
)
if not (0.05 <= TIMING_CORRECTOR_INTERNAL_VAL_FRACTION <= 0.40):
    raise ValueError(
        "TIMING_CORRECTOR_INTERNAL_VAL_FRACTION must be between 0.05 and 0.40."
    )

assert "META_SYS2_TRAIN" in globals(), (
    "Need META_SYS2_TRAIN from Cell 17."
)
assert "META_SYS2_VAL" in globals(), (
    "Need META_SYS2_VAL from Cell 17."
)
assert "META_SYS2_TEST" in globals(), (
    "Need META_SYS2_TEST from Cell 17."
)
assert "REAL_TRAIN_BOUNDARY_EVENTS" in globals(), (
    "Need paper-ready TRAIN boundary events from Cell 4."
)
assert "REAL_VAL_EVENTS" in globals(), (
    "Need paper-ready VAL events from Cell 4."
)

def _timing_merge_key(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()

    if "session" not in output.columns or "time" not in output.columns:
        raise KeyError(
            "Timing-corrector input requires session and time."
        )

    output["session"] = output["session"].astype(str)
    output["time"] = pd.to_numeric(
        output["time"],
        errors="coerce",
    )
    output = output.dropna(
        subset=["session", "time"],
    ).copy()

    output["_time_key_us"] = np.rint(
        output["time"].to_numpy(dtype=float)
        * 1_000_000.0
    ).astype(np.int64)

    return output

def _timing_score_tables_for_split(
    split_name: str,
) -> list:
    suffix = str(split_name).upper()

    names = [
        f"TEMPORAL_{suffix}_SCORES",
        f"LGBM_{suffix}_SCORES",
        f"LGBM_BINARY_{suffix}_SCORES",
        f"RANKER_{suffix}_SCORES",
        f"RR_FAP_{suffix}_SCORES",
        f"META_CONFIRMER_{suffix}_SCORES",
    ]

    tables = []

    for name in names:
        value = globals().get(name, None)

        if isinstance(value, pd.DataFrame) and len(value):
            tables.append((name, value.copy()))

    return tables

def _build_timing_candidate_frame(
    base_frame: pd.DataFrame,
    split_name: str,
) -> pd.DataFrame:
    output = _timing_merge_key(
        base_frame
    )

    for table_name, table in _timing_score_tables_for_split(
        split_name
    ):
        table = _timing_merge_key(table)

        keep = [
            column
            for column in table.columns
            if column
            not in {
                "time",
                "label",
                "action",
                "action_id",
                "dt_to_event",
            }
            and column
            not in {
                "session",
                "_time_key_us",
            }
        ]

        table = (
            table[
                [
                    "session",
                    "_time_key_us",
                    *keep,
                ]
            ]
            .sort_values(
                [
                    "session",
                    "_time_key_us",
                ]
            )
            .drop_duplicates(
                [
                    "session",
                    "_time_key_us",
                ],
                keep="last",
            )
        )

        rename = {
            column: (
                column
                if column not in output.columns
                else f"{table_name.lower()}__{column}"
            )
            for column in keep
        }
        table = table.rename(
            columns=rename
        )

        output = output.merge(
            table,
            on=[
                "session",
                "_time_key_us",
            ],
            how="left",
            validate="many_to_one",
        )

    output = (
        output
        .sort_values(
            [
                "session",
                "time",
            ]
        )
        .drop_duplicates(
            [
                "session",
                "_time_key_us",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    return output

TIMING_FORBIDDEN_PATTERNS = (
    "label",
    "action_id",
    "dt_to_event",
    "nearest_gt",
    "gt_",
    "target",
    "oracle",
    "correct_next",
    "next_speaker",
    "prev_speaker",
    "speaker_id",
    "rttm_",
    "timing_",
)

TIMING_NON_FEATURE_COLUMNS = {
    "session",
    "split",
    "time",
    "_time_key_us",
    "action",
    "audio_path",
    "rttm_path",
    "text",
    "orig_session",
    "event_contract_version",
    "event_time_definition",
    "event_anchor",
}

def _select_timing_feature_columns(
    train_frame: pd.DataFrame,
) -> list:
    columns = []

    for column in train_frame.columns:
        if column in TIMING_NON_FEATURE_COLUMNS:
            continue

        lower = str(column).lower()

        if any(
            pattern in lower
            for pattern in TIMING_FORBIDDEN_PATTERNS
        ):
            continue

        numeric = pd.to_numeric(
            train_frame[column],
            errors="coerce",
        )

        if numeric.notna().sum() == 0:
            continue

        if numeric.nunique(dropna=True) <= 1:
            continue

        columns.append(str(column))

    forbidden_selected = [
        column
        for column in columns
        if any(
            pattern in column.lower()
            for pattern in TIMING_FORBIDDEN_PATTERNS
        )
    ]

    if forbidden_selected:
        raise RuntimeError(
            "Timing feature leakage detected: "
            f"{forbidden_selected}"
        )

    return columns

def _automatic_timing_candidates(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    if "source_train_gt_seed" in output.columns:
        gt_seed = pd.to_numeric(
            output["source_train_gt_seed"],
            errors="coerce",
        ).fillna(0)

        output = output[
            gt_seed.le(0)
        ].copy()

    return output

def _build_timing_pairs(
    candidate_frame: pd.DataFrame,
    event_frame: pd.DataFrame,
    split_name: str,
) -> pd.DataFrame:
    candidates = _automatic_timing_candidates(
        candidate_frame
    )

    events = event_frame.copy()
    events = events[
        pd.to_numeric(
            events["label"],
            errors="coerce",
        ).fillna(0).astype(int).eq(1)
    ].copy()

    events["session"] = events[
        "session"
    ].astype(str)
    events["time"] = pd.to_numeric(
        events["time"],
        errors="coerce",
    )
    events = events.dropna(
        subset=["session", "time"],
    ).copy()

    pair_parts = []

    for session, event_group in events.groupby(
        "session",
        sort=False,
    ):
        session_candidates = candidates[
            candidates["session"]
            .astype(str)
            .eq(str(session))
        ].copy()

        if len(session_candidates) == 0:
            continue

        candidate_times = session_candidates[
            "time"
        ].to_numpy(dtype=float)

        candidate_assignment = []

        for event_index, event_row in event_group.reset_index(
            drop=True
        ).iterrows():
            event_time = float(
                event_row["time"]
            )
            distance = np.abs(
                candidate_times
                - event_time
            )
            valid_indices = np.where(
                distance
                <= float(
                    TIMING_CORRECTOR_MATCH_RADIUS_S
                )
            )[0]

            if len(valid_indices) == 0:
                continue

            ordered = valid_indices[
                np.argsort(
                    distance[valid_indices],
                    kind="stable",
                )
            ][
                : int(
                    TIMING_CORRECTOR_MAX_CANDIDATES_PER_EVENT
                )
            ]

            for candidate_index in ordered:
                candidate_assignment.append({
                    "candidate_index": int(
                        candidate_index
                    ),
                    "event_index": int(
                        event_index
                    ),
                    "event_time": event_time,
                    "distance": float(
                        distance[candidate_index]
                    ),
                })

        if not candidate_assignment:
            continue

        assignment = pd.DataFrame(
            candidate_assignment
        )

        assignment = (
            assignment
            .sort_values(
                [
                    "candidate_index",
                    "distance",
                    "event_index",
                ]
            )
            .drop_duplicates(
                "candidate_index",
                keep="first",
            )
        )

        selected = session_candidates.iloc[
            assignment["candidate_index"]
            .to_numpy(dtype=int)
        ].copy()
        selected = selected.reset_index(
            drop=True
        )
        assignment = assignment.reset_index(
            drop=True
        )

        selected["timing_gt_time"] = (
            assignment["event_time"]
            .to_numpy(dtype=float)
        )
        selected["timing_raw_error_s"] = (
            selected["time"]
            .to_numpy(dtype=float)
            - selected["timing_gt_time"]
            .to_numpy(dtype=float)
        )
        selected["timing_target_offset_s"] = np.clip(
            -selected[
                "timing_raw_error_s"
            ].to_numpy(dtype=float),
            -float(
                TIMING_CORRECTOR_MAX_ABS_OFFSET_S
            ),
            float(
                TIMING_CORRECTOR_MAX_ABS_OFFSET_S
            ),
        )
        selected["timing_pair_distance_s"] = (
            assignment["distance"]
            .to_numpy(dtype=float)
        )
        selected["timing_split"] = str(
            split_name
        )
        pair_parts.append(selected)

    return (
        pd.concat(
            pair_parts,
            ignore_index=True,
        )
        if pair_parts
        else pd.DataFrame()
    )

def _prepare_timing_matrix(
    frame: pd.DataFrame,
    feature_columns: list,
) -> np.ndarray:
    matrix = frame.reindex(
        columns=feature_columns
    ).copy()

    for column in feature_columns:
        matrix[column] = pd.to_numeric(
            matrix[column],
            errors="coerce",
        )

    return (
        matrix
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

TIMING_TRAIN_FRAME = _build_timing_candidate_frame(
    META_SYS2_TRAIN,
    "train",
)
TIMING_VAL_FRAME = _build_timing_candidate_frame(
    META_SYS2_VAL,
    "val",
)
TIMING_TEST_FRAME = _build_timing_candidate_frame(
    META_SYS2_TEST,
    "test",
)

TIMING_TRAIN_PAIRS = _build_timing_pairs(
    TIMING_TRAIN_FRAME,
    REAL_TRAIN_BOUNDARY_EVENTS,
    "train",
)
TIMING_VAL_PAIRS = _build_timing_pairs(
    TIMING_VAL_FRAME,
    REAL_VAL_EVENTS,
    "val",
)

if len(TIMING_TRAIN_PAIRS) == 0:
    raise RuntimeError(
        "No TRAIN candidate/GT timing pairs were created."
    )

if len(TIMING_VAL_PAIRS) == 0:
    raise RuntimeError(
        "No VAL candidate/GT timing pairs were created."
    )

def _timing_session_hash_fraction(session: str) -> float:
    digest = hashlib.md5(
        str(session).encode("utf-8")
    ).hexdigest()
    return int(digest[:12], 16) / float(16 ** 12)

def _split_timing_train_pairs_by_session(
    pair_frame: pd.DataFrame,
) -> tuple:
    sessions = sorted(
        pair_frame["session"].astype(str).unique().tolist()
    )

    if len(sessions) < 3:
        raise RuntimeError(
            "Need at least three TRAIN sessions with timing pairs "
            "for session-held-out timing calibration."
        )

    internal_val_sessions = [
        session
        for session in sessions
        if _timing_session_hash_fraction(session)
        < float(TIMING_CORRECTOR_INTERNAL_VAL_FRACTION)
    ]

    if len(internal_val_sessions) == 0:
        internal_val_sessions = [sessions[0]]
    if len(internal_val_sessions) >= len(sessions):
        internal_val_sessions = internal_val_sessions[:-1]

    internal_val_set = set(internal_val_sessions)

    fit_pairs = pair_frame[
        ~pair_frame["session"].astype(str).isin(internal_val_set)
    ].copy()
    early_stop_pairs = pair_frame[
        pair_frame["session"].astype(str).isin(internal_val_set)
    ].copy()

    if len(fit_pairs) == 0 or len(early_stop_pairs) == 0:
        raise RuntimeError(
            "The TRAIN-only timing split produced an empty partition."
        )

    return (
        fit_pairs.reset_index(drop=True),
        early_stop_pairs.reset_index(drop=True),
        sorted(set(fit_pairs["session"].astype(str))),
        sorted(set(early_stop_pairs["session"].astype(str))),
    )

(
    TIMING_TRAIN_FIT_PAIRS,
    TIMING_TRAIN_EARLY_STOP_PAIRS,
    TIMING_TRAIN_FIT_SESSIONS,
    TIMING_TRAIN_EARLY_STOP_SESSIONS,
) = _split_timing_train_pairs_by_session(
    TIMING_TRAIN_PAIRS
)

TIMING_FEATURE_COLUMNS = _select_timing_feature_columns(
    TIMING_TRAIN_PAIRS
)

if len(TIMING_FEATURE_COLUMNS) == 0:
    raise RuntimeError(
        "No deployable timing-corrector features were selected."
    )

timing_expected_meta = {
    "variant": str(
        TIMING_CORRECTOR_VARIANT
    ),
    "run_version": str(RUN_VERSION),
    "event_contract_version": str(
        EVENT_CONTRACT_VERSION
    ),
    "primary_event_time_definition": str(
        PRIMARY_EVENT_TIME_DEFINITION
    ),
    "match_radius_s": float(
        TIMING_CORRECTOR_MATCH_RADIUS_S
    ),
    "maximum_abs_offset_s": float(
        TIMING_CORRECTOR_MAX_ABS_OFFSET_S
    ),
    "maximum_candidates_per_event": int(
        TIMING_CORRECTOR_MAX_CANDIDATES_PER_EVENT
    ),
    "feature_columns": list(
        TIMING_FEATURE_COLUMNS
    ),
    "train_rows_total": int(
        len(TIMING_TRAIN_PAIRS)
    ),
    "train_fit_rows": int(
        len(TIMING_TRAIN_FIT_PAIRS)
    ),
    "train_internal_early_stop_rows": int(
        len(TIMING_TRAIN_EARLY_STOP_PAIRS)
    ),
    "train_fit_sessions": list(
        TIMING_TRAIN_FIT_SESSIONS
    ),
    "train_internal_early_stop_sessions": list(
        TIMING_TRAIN_EARLY_STOP_SESSIONS
    ),
    "train_internal_val_fraction": float(
        TIMING_CORRECTOR_INTERNAL_VAL_FRACTION
    ),
    "official_val_rows_for_diagnostics_only": int(
        len(TIMING_VAL_PAIRS)
    ),
    "official_val_used_for_model_fitting": False,
    "test_labels_used": False,
}

load_existing_timing_model = (
    TIMING_CORRECTOR_MODEL_PATH.exists()
    and TIMING_CORRECTOR_META_PATH.exists()
    and not bool(
        globals().get(
            "FORCE_RETRAIN_TIMING_CORRECTOR",
            False,
        )
    )
)

if load_existing_timing_model:
    try:
        existing_meta = json.loads(
            TIMING_CORRECTOR_META_PATH.read_text()
        )
        load_existing_timing_model = (
            existing_meta
            == timing_expected_meta
        )
    except Exception:
        load_existing_timing_model = False

if load_existing_timing_model:
    TIMING_CORRECTOR_MODEL = lgb.Booster(
        model_file=str(
            TIMING_CORRECTOR_MODEL_PATH
        )
    )
    print(
        "Loaded compatible local timing corrector:",
        TIMING_CORRECTOR_MODEL_PATH,
    )
else:
    train_matrix = _prepare_timing_matrix(
        TIMING_TRAIN_FIT_PAIRS,
        TIMING_FEATURE_COLUMNS,
    )
    val_matrix = _prepare_timing_matrix(
        TIMING_TRAIN_EARLY_STOP_PAIRS,
        TIMING_FEATURE_COLUMNS,
    )

    train_target = TIMING_TRAIN_FIT_PAIRS[
        "timing_target_offset_s"
    ].to_numpy(dtype=np.float32)
    val_target = TIMING_TRAIN_EARLY_STOP_PAIRS[
        "timing_target_offset_s"
    ].to_numpy(dtype=np.float32)

    train_weight_series = (
        TIMING_TRAIN_FIT_PAIRS["sample_weight"]
        if "sample_weight" in TIMING_TRAIN_FIT_PAIRS.columns
        else pd.Series(
            1.0,
            index=TIMING_TRAIN_FIT_PAIRS.index,
        )
    )
    val_weight_series = (
        TIMING_TRAIN_EARLY_STOP_PAIRS["sample_weight"]
        if "sample_weight" in TIMING_TRAIN_EARLY_STOP_PAIRS.columns
        else pd.Series(
            1.0,
            index=TIMING_TRAIN_EARLY_STOP_PAIRS.index,
        )
    )

    train_weight = (
        pd.to_numeric(
            train_weight_series,
            errors="coerce",
        )
        .fillna(1.0)
        .clip(lower=1e-6)
        .to_numpy(dtype=np.float32)
    )
    val_weight = (
        pd.to_numeric(
            val_weight_series,
            errors="coerce",
        )
        .fillna(1.0)
        .clip(lower=1e-6)
        .to_numpy(dtype=np.float32)
    )

    train_dataset = lgb.Dataset(
        train_matrix,
        label=train_target,
        weight=train_weight,
        feature_name=TIMING_FEATURE_COLUMNS,
        free_raw_data=False,
    )
    val_dataset = lgb.Dataset(
        val_matrix,
        label=val_target,
        weight=val_weight,
        feature_name=TIMING_FEATURE_COLUMNS,
        reference=train_dataset,
        free_raw_data=False,
    )

    timing_parameters = {
        "objective": "huber",
        "metric": "l1",
        "learning_rate": 0.025,
        "num_leaves": 31,
        "min_data_in_leaf": 30,
        "feature_fraction": 0.80,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "lambda_l1": 0.05,
        "lambda_l2": 0.25,
        "max_depth": -1,
        "verbosity": -1,
        "seed": int(SEED),
        "feature_fraction_seed": int(SEED),
        "bagging_seed": int(SEED),
        "data_random_seed": int(SEED),
        "num_threads": max(
            1,
            int(
                globals().get(
                    "LIGHTGBM_NUM_THREADS",
                    4,
                )
            ),
        ),
    }

    TIMING_CORRECTOR_MODEL = lgb.train(
        timing_parameters,
        train_dataset,
        num_boost_round=int(
            TIMING_CORRECTOR_NUM_BOOST_ROUND
        ),
        valid_sets=[
            val_dataset,
        ],
        valid_names=[
            "train_internal_val",
        ],
        callbacks=[
            lgb.early_stopping(
                int(
                    TIMING_CORRECTOR_EARLY_STOPPING_ROUNDS
                ),
                verbose=True,
            ),
            lgb.log_evaluation(
                period=50
            ),
        ],
    )

    TIMING_CORRECTOR_MODEL.save_model(
        str(
            TIMING_CORRECTOR_MODEL_PATH
        )
    )
    TIMING_CORRECTOR_FEATURES_PATH.write_text(
        json.dumps(
            TIMING_FEATURE_COLUMNS,
            indent=2,
        )
    )
    TIMING_CORRECTOR_META_PATH.write_text(
        json.dumps(
            timing_expected_meta,
            indent=2,
            sort_keys=True,
        )
    )

def _score_timing_frame(
    frame: pd.DataFrame,
    split_name: str,
    output_path: Path,
) -> pd.DataFrame:
    output = frame[
        [
            "session",
            "time",
            "_time_key_us",
        ]
    ].copy()

    matrix = _prepare_timing_matrix(
        frame,
        TIMING_FEATURE_COLUMNS,
    )

    best_iteration = (
        int(
            TIMING_CORRECTOR_MODEL.best_iteration
        )
        if getattr(
            TIMING_CORRECTOR_MODEL,
            "best_iteration",
            0,
        )
        else None
    )

    predicted = TIMING_CORRECTOR_MODEL.predict(
        matrix,
        num_iteration=best_iteration,
    )
    predicted = np.clip(
        np.asarray(
            predicted,
            dtype=float,
        ),
        -float(
            TIMING_CORRECTOR_MAX_ABS_OFFSET_S
        ),
        float(
            TIMING_CORRECTOR_MAX_ABS_OFFSET_S
        ),
    )

    output[
        "predicted_offset_s"
    ] = predicted
    output[
        "corrected_time"
    ] = (
        output["time"].to_numpy(
            dtype=float
        )
        + predicted
    )
    output[
        "timing_corrector_variant"
    ] = str(
        TIMING_CORRECTOR_VARIANT
    )
    output["split"] = str(
        split_name
    )

    output.to_pickle(
        output_path
    )
    return output

TIMING_CORRECTOR_TRAIN_SCORES = _score_timing_frame(
    TIMING_TRAIN_FRAME,
    "train",
    TIMING_CORRECTOR_TRAIN_SCORES_PATH,
)
TIMING_CORRECTOR_VAL_SCORES = _score_timing_frame(
    TIMING_VAL_FRAME,
    "val",
    TIMING_CORRECTOR_VAL_SCORES_PATH,
)
TIMING_CORRECTOR_TEST_SCORES = _score_timing_frame(
    TIMING_TEST_FRAME,
    "test",
    TIMING_CORRECTOR_TEST_SCORES_PATH,
)

TIMING_TRAIN_PAIRS.to_pickle(
    TIMING_CORRECTOR_TRAIN_PAIRS_PATH
)
TIMING_VAL_PAIRS.to_pickle(
    TIMING_CORRECTOR_VAL_PAIRS_PATH
)

def _timing_pair_diagnostics(
    pair_frame: pd.DataFrame,
    split_name: str,
) -> Dict[str, Any]:
    lookup = (
        TIMING_CORRECTOR_TRAIN_SCORES
        if split_name == "train"
        else TIMING_CORRECTOR_VAL_SCORES
    )

    merged = _timing_merge_key(
        pair_frame
    ).merge(
        lookup[
            [
                "session",
                "_time_key_us",
                "predicted_offset_s",
                "corrected_time",
            ]
        ],
        on=[
            "session",
            "_time_key_us",
        ],
        how="left",
        validate="many_to_one",
    )

    raw_absolute_error = np.abs(
        merged["time"].to_numpy(dtype=float)
        - merged[
            "timing_gt_time"
        ].to_numpy(dtype=float)
    )
    corrected_absolute_error = np.abs(
        merged[
            "corrected_time"
        ].to_numpy(dtype=float)
        - merged[
            "timing_gt_time"
        ].to_numpy(dtype=float)
    )

    return {
        "split": str(split_name),
        "pairs": int(len(merged)),
        "raw_mae_s": float(
            np.mean(
                raw_absolute_error
            )
        ),
        "corrected_mae_s": float(
            np.mean(
                corrected_absolute_error
            )
        ),
        "mae_gain_s": float(
            np.mean(
                raw_absolute_error
            )
            - np.mean(
                corrected_absolute_error
            )
        ),
        "raw_within_0p50": float(
            np.mean(
                raw_absolute_error
                <= 0.50
            )
        ),
        "corrected_within_0p50": float(
            np.mean(
                corrected_absolute_error
                <= 0.50
            )
        ),
        "test_labels_used": False,
    }

TIMING_CORRECTOR_DIAGNOSTICS_DF = pd.DataFrame([
    _timing_pair_diagnostics(
        TIMING_TRAIN_PAIRS,
        "train",
    ),
    _timing_pair_diagnostics(
        TIMING_VAL_PAIRS,
        "val",
    ),
])

TIMING_CORRECTOR_DIAGNOSTICS_DF.to_csv(
    TIMING_CORRECTOR_DIAGNOSTICS_PATH,
    index=False,
)

print("=" * 80)
print("LOCAL TIMESTAMP CORRECTOR")
print("=" * 80)
print(
    "Feature count:",
    len(TIMING_FEATURE_COLUMNS),
)
print(
    "TRAIN timing pairs (total / fit / internal early-stop):",
    len(TIMING_TRAIN_PAIRS),
    len(TIMING_TRAIN_FIT_PAIRS),
    len(TIMING_TRAIN_EARLY_STOP_PAIRS),
)
print(
    "Official VAL timing pairs (diagnostics and final selection only):",
    len(TIMING_VAL_PAIRS),
)
display(
    TIMING_CORRECTOR_DIAGNOSTICS_DF
)

assert not any(
    pattern in column.lower()
    for column in TIMING_FEATURE_COLUMNS
    for pattern in TIMING_FORBIDDEN_PATTERNS
), "Timing-corrector feature leakage detected."

globals()["TIMING_CORRECTOR_VARIANT"] = TIMING_CORRECTOR_VARIANT
globals()["TIMING_FEATURE_COLUMNS"] = TIMING_FEATURE_COLUMNS
globals()["TIMING_CORRECTOR_MODEL"] = TIMING_CORRECTOR_MODEL
globals()["TIMING_CORRECTOR_TRAIN_SCORES"] = TIMING_CORRECTOR_TRAIN_SCORES
globals()["TIMING_CORRECTOR_VAL_SCORES"] = TIMING_CORRECTOR_VAL_SCORES
globals()["TIMING_CORRECTOR_TEST_SCORES"] = TIMING_CORRECTOR_TEST_SCORES
globals()["TIMING_CORRECTOR_DIAGNOSTICS_DF"] = TIMING_CORRECTOR_DIAGNOSTICS_DF
globals()["TIMING_CORRECTOR_MODEL_PATH"] = TIMING_CORRECTOR_MODEL_PATH
globals()["TIMING_CORRECTOR_META_PATH"] = TIMING_CORRECTOR_META_PATH
globals()["TIMING_CORRECTOR_FEATURES_PATH"] = TIMING_CORRECTOR_FEATURES_PATH

print(
    "Cell 19E-B complete: TRAIN-only local offset model "
    "scored TRAIN/VAL/TEST candidates; official VAL and TEST labels "
    "were not used for model fitting."
)

Training until validation scores don't improve for 150 rounds
[50]	train_internal_val's l1: 0.346172
[100]	train_internal_val's l1: 0.286299


[150]	train_internal_val's l1: 0.270791
[200]	train_internal_val's l1: 0.267739
[250]	train_internal_val's l1: 0.266396
[300]	train_internal_val's l1: 0.267756


[350]	train_internal_val's l1: 0.267349
[400]	train_internal_val's l1: 0.269026
Early stopping, best iteration is:
[252]	train_internal_val's l1: 0.26623


LOCAL TIMESTAMP CORRECTOR
Feature count: 105
TRAIN timing pairs (total / fit / internal early-stop): 3579 3159 420
Official VAL timing pairs (diagnostics and final selection only): 1212


,split,pairs,raw_mae_s,corrected_mae_s,mae_gain_s,raw_within_0p50,corrected_within_0p50,test_labels_used
0,train,3579,0.588662,0.204803,0.383859,0.379715,0.913663,False
1,val,1212,0.423654,0.311709,0.111945,0.603960,0.784653,False


Cell 19E-B complete: TRAIN-only local offset model scored TRAIN/VAL/TEST candidates; official VAL and TEST labels were not used for model fitting.


In [30]:
import hashlib
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

def _norm_sid(x) -> str:
    return Path(str(x)).stem

def dataframe_fingerprint(df: pd.DataFrame, cols: Optional[List[str]] = None) -> str:
    if df is None:
        return "none"
    work = df.copy()
    if cols is not None:
        use = [c for c in cols if c in work.columns]
        work = work[use].copy()
    if len(work) == 0:
        return "empty"
    payload = pd.util.hash_pandas_object(work, index=True).to_numpy(dtype=np.uint64).tobytes()
    return hashlib.sha256(payload).hexdigest()

def canonical_prepare_predictions(pred_df: pd.DataFrame) -> pd.DataFrame:
    if pred_df is None or len(pred_df) == 0:
        return pd.DataFrame(columns=["session", "time", "score", "_pred_row_id"])
    if not {"session", "time"}.issubset(pred_df.columns):
        raise ValueError("Prediction table must contain session and time columns.")
    out = pred_df.copy()
    out["session"] = out["session"].astype(str).map(_norm_sid)
    out["time"] = pd.to_numeric(out["time"], errors="coerce").astype("float64")
    if "score" not in out.columns:
        for c in ["fused_shift_score", "lgbm_binary_shift_score", "lgbm_shift_score"]:
            if c in out.columns:
                out["score"] = pd.to_numeric(out[c], errors="coerce")
                break
        else:
            out["score"] = 1.0
    out["score"] = pd.to_numeric(out["score"], errors="coerce").fillna(0.0).astype("float64")
    out = out.dropna(subset=["session", "time"])
    out = out[out["time"] >= 0].copy()
    out["_time_key"] = out["time"].round(9)
    out = (
        out.sort_values(["session", "_time_key", "score"], ascending=[True, True, False])
        .drop_duplicates(["session", "_time_key"], keep="first")
        .drop(columns=["_time_key"])
        .sort_values(["session", "time"])
        .reset_index(drop=True)
    )
    out["_pred_row_id"] = np.arange(len(out), dtype=np.int64)
    return out

def canonical_prepare_gt_events(events_df: pd.DataFrame) -> pd.DataFrame:
    if events_df is None or len(events_df) == 0:
        return pd.DataFrame(columns=["session", "time", "_gt_row_id"])
    if not {"session", "time"}.issubset(events_df.columns):
        raise ValueError("GT event table must contain session and time columns.")
    out = events_df.copy()
    out["session"] = out["session"].astype(str).map(_norm_sid)
    out["time"] = pd.to_numeric(out["time"], errors="coerce").astype("float64")
    if "label" in out.columns:
        out["label"] = pd.to_numeric(out["label"], errors="coerce").fillna(0).astype(int)
        out = out[out["label"].eq(1)].copy()
    if "source_event_type" in out.columns:
        adjacent = out["source_event_type"].astype(str).eq("adjacent_rttm_boundary")
        if adjacent.any():
            out = out[adjacent].copy()
    out = out.dropna(subset=["session", "time"])
    out["_time_key"] = out["time"].round(9)
    out = (
        out.drop_duplicates(["session", "_time_key"], keep="first")
        .drop(columns=["_time_key"])
        .sort_values(["session", "time"])
        .reset_index(drop=True)
    )
    out["_gt_row_id"] = np.arange(len(out), dtype=np.int64)
    return out

def canonical_match_indices(pred_times, gt_times, tol: float) -> List[Tuple[int, int, float]]:
    p = np.asarray(pred_times, dtype=np.float64).reshape(-1)
    g = np.asarray(gt_times, dtype=np.float64).reshape(-1)
    if len(p) == 0 or len(g) == 0:
        return []
    cost = np.abs(p[:, None] - g[None, :])
    try:
        from scipy.optimize import linear_sum_assignment
        big = 1e9
        rr, cc = linear_sum_assignment(np.where(cost <= float(tol), cost, big))
        pairs = [(int(i), int(j), float(cost[i, j])) for i, j in zip(rr, cc) if cost[i, j] <= float(tol)]
        return sorted(pairs, key=lambda z: (z[0], z[1]))
    except Exception:
        candidates = []
        for i in range(len(p)):
            for j in np.where(cost[i] <= float(tol))[0]:
                candidates.append((float(cost[i, j]), int(i), int(j)))
        candidates.sort(key=lambda z: (z[0], z[1], z[2]))
        used_p, used_g, pairs = set(), set(), []
        for d, i, j in candidates:
            if i in used_p or j in used_g:
                continue
            used_p.add(i); used_g.add(j)
            pairs.append((i, j, d))
        return sorted(pairs, key=lambda z: (z[0], z[1]))

def canonical_event_metrics_from_table(
    pred_df: pd.DataFrame,
    events_df: pd.DataFrame,
    tol: float,
    return_per_session: bool = False,
    return_matches: bool = False,
) -> Dict[str, Any]:
    pred = canonical_prepare_predictions(pred_df)
    gt = canonical_prepare_gt_events(events_df)
    sessions = sorted(set(pred["session"]).union(set(gt["session"])))
    totals = {"TP": 0, "FP": 0, "FN": 0, "N_pred": 0, "N_gt": 0}
    per_rows, match_rows = [], []
    for sid in sessions:
        ps = pred[pred["session"].eq(sid)].reset_index(drop=True)
        gs = gt[gt["session"].eq(sid)].reset_index(drop=True)
        pairs = canonical_match_indices(ps["time"].to_numpy(np.float64), gs["time"].to_numpy(np.float64), tol)
        tp, npred, ngt = len(pairs), len(ps), len(gs)
        row = {"session": sid, "TP": tp, "FP": npred - tp, "FN": ngt - tp, "N_pred": npred, "N_gt": ngt}
        for k in totals:
            totals[k] += int(row[k])
        per_rows.append({"tol": float(tol), **row})
        if return_matches:
            for pi, gi, err in pairs:
                match_rows.append({
                    "session": sid,
                    "pred_local_index": int(pi),
                    "gt_local_index": int(gi),
                    "pred_row_id": int(ps.iloc[pi]["_pred_row_id"]),
                    "gt_row_id": int(gs.iloc[gi]["_gt_row_id"]),
                    "pred_time": float(ps.iloc[pi]["time"]),
                    "gt_time": float(gs.iloc[gi]["time"]),
                    "signed_error": float(ps.iloc[pi]["time"] - gs.iloc[gi]["time"]),
                    "abs_error": float(err),
                    "tol": float(tol),
                })
    tp, fp, fn = totals["TP"], totals["FP"], totals["FN"]
    p = tp / max(1, tp + fp)
    r = tp / max(1, tp + fn)
    f1 = 2 * p * r / max(1e-12, p + r)
    out = {"P": float(p), "R": float(r), "F1": float(f1), **totals}
    if return_per_session:
        out["per_session"] = pd.DataFrame(per_rows)
    if return_matches:
        out["matches"] = pd.DataFrame(match_rows)
    return out

In [31]:
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any, List, Tuple
from IPython.display import display

FINAL_LOCKED_PATH = RUN_ROOT / "final_locked_val_settings.json"
FINAL_LOCKED_META_PATH = RUN_ROOT / "final_locked_val_settings.meta.json"
VAL_FUSION_TABLE_PATH = RUN_ROOT / "val_fusion_threshold_sweep.csv"
VAL_FUSION_TABLE_META_PATH = RUN_ROOT / "val_fusion_threshold_sweep.meta.json"

FINAL_VAL_PRED_PATH = RUN_ROOT / "final_val_predictions_locked.csv"
FINAL_VAL_METRICS_PATH = RUN_ROOT / "final_val_metrics_locked.json"

CELL20_VARIANT = "cell20_v9_paper_ready_contract_timing_robust_val"

FORCE_RETUNE_VAL = bool(globals().get("FORCE_RETUNE_VAL", False))

FUSION_ALPHA_GRID = list(globals().get("FUSION_ALPHA_GRID", [0.0, 0.25, 0.50, 0.75, 1.0]))

DEFAULT_FINAL_SCORE_MODE_GRID_V8 = [
    "meta_confirmer",
    "avg_meta_rr",
    "max_meta_binary",
    "avg_meta_rr_ranker",
    "weighted_meta_temporal",

    "rr_fap",
    "avg_rr_binary",
    "max_rr_binary",
    "avg_rr_ranker_binary",
    "weighted_rr_temporal",
    "weighted_rr_ranker_temporal",

    "ranker",
    "avg_ranker_binary",
    "max_ranker_binary",
    "weighted_ranker_temporal",
    "lgbm_binary",
    "avg_binary_multi",
    "max_binary_multi",
    "weighted_binary_temporal",
    "weighted_lgbm_multi_temporal",
    "temporal_only",
]

FINAL_SCORE_MODE_GRID = list(globals().get(
    "FINAL_SCORE_MODE_GRID_V8",
    DEFAULT_FINAL_SCORE_MODE_GRID_V8,
))

FINAL_SCORE_MODE_GRID = list(dict.fromkeys([str(x) for x in FINAL_SCORE_MODE_GRID]))

REQUIRED_RANKER_SCORE_MODES = [
    "ranker",
    "avg_ranker_binary",
    "max_ranker_binary",
    "weighted_ranker_temporal",
]

REQUIRED_RR_FAP_SCORE_MODES = [
    "rr_fap",
    "avg_rr_binary",
    "max_rr_binary",
    "avg_rr_ranker_binary",
    "weighted_rr_temporal",
    "weighted_rr_ranker_temporal",
]

REQUIRED_META_CONFIRMER_SCORE_MODES = [
    "meta_confirmer",
    "avg_meta_rr",
    "max_meta_binary",
    "avg_meta_rr_ranker",
    "weighted_meta_temporal",
]

missing_ranker_modes = [m for m in REQUIRED_RANKER_SCORE_MODES if m not in FINAL_SCORE_MODE_GRID]

if missing_ranker_modes:
    print("⚠️ FINAL_SCORE_MODE_GRID_V8 was missing ranker modes. Adding them:", missing_ranker_modes)
    FINAL_SCORE_MODE_GRID = missing_ranker_modes + FINAL_SCORE_MODE_GRID
    FINAL_SCORE_MODE_GRID = list(dict.fromkeys(FINAL_SCORE_MODE_GRID))

assert all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_RANKER_SCORE_MODES), (
    "Ranker modes are missing from FINAL_SCORE_MODE_GRID. "
    f"Current grid={FINAL_SCORE_MODE_GRID}"
)

missing_rr_fap_modes = [m for m in REQUIRED_RR_FAP_SCORE_MODES if m not in FINAL_SCORE_MODE_GRID]
if missing_rr_fap_modes:
    print("⚠️ FINAL_SCORE_MODE_GRID_V8 was missing RR-FAP modes. Adding them:", missing_rr_fap_modes)
    FINAL_SCORE_MODE_GRID = missing_rr_fap_modes + FINAL_SCORE_MODE_GRID
    FINAL_SCORE_MODE_GRID = list(dict.fromkeys(FINAL_SCORE_MODE_GRID))

assert all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_RR_FAP_SCORE_MODES), (
    "RR-FAP modes are missing from FINAL_SCORE_MODE_GRID. "
    f"Current grid={FINAL_SCORE_MODE_GRID}"
)

missing_meta_modes = [m for m in REQUIRED_META_CONFIRMER_SCORE_MODES if m not in FINAL_SCORE_MODE_GRID]
if missing_meta_modes:
    print("⚠️ FINAL_SCORE_MODE_GRID_V8 was missing meta-confirmer modes. Adding them:", missing_meta_modes)
    FINAL_SCORE_MODE_GRID = missing_meta_modes + FINAL_SCORE_MODE_GRID
    FINAL_SCORE_MODE_GRID = list(dict.fromkeys(FINAL_SCORE_MODE_GRID))

assert all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_META_CONFIRMER_SCORE_MODES), (
    "Meta-confirmer modes are missing from FINAL_SCORE_MODE_GRID. "
    f"Current grid={FINAL_SCORE_MODE_GRID}"
)

RANKER_SCORE_AVAILABLE_FOR_FUSION = bool(
    "LGBM_VAL_SCORES" in globals()
    and isinstance(LGBM_VAL_SCORES, pd.DataFrame)
    and len(LGBM_VAL_SCORES)
    and "ranker_shift_score" in LGBM_VAL_SCORES.columns
)

RR_FAP_SCORE_AVAILABLE_FOR_FUSION = bool(
    "LGBM_VAL_SCORES" in globals()
    and isinstance(LGBM_VAL_SCORES, pd.DataFrame)
    and len(LGBM_VAL_SCORES)
    and "rr_role_shift_score" in LGBM_VAL_SCORES.columns
)

META_CONFIRMER_SCORE_AVAILABLE_FOR_FUSION = bool(
    "LGBM_VAL_SCORES" in globals()
    and isinstance(LGBM_VAL_SCORES, pd.DataFrame)
    and len(LGBM_VAL_SCORES)
    and "meta_confirmer_shift_score" in LGBM_VAL_SCORES.columns
)

assert RANKER_SCORE_AVAILABLE_FOR_FUSION, (
    "ranker_shift_score is missing from LGBM_VAL_SCORES. "
    "Run Cell 19C before Cell 20."
)

assert RR_FAP_SCORE_AVAILABLE_FOR_FUSION, (
    "rr_role_shift_score is missing from LGBM_VAL_SCORES. "
    "Run Cell 19D before Cell 20. This RR-FAP notebook should not silently fall back."
)

assert META_CONFIRMER_SCORE_AVAILABLE_FOR_FUSION, (
    "meta_confirmer_shift_score is missing from LGBM_VAL_SCORES. "
    "Run Cell 19E before Cell 20. This 18thAug notebook should not silently fall back."
)

FINAL_MAIN_THR_GRID = list(
    globals().get(
        "FINAL_MAIN_THR_GRID",
        [
            0.10,
            0.15,
            0.20,
            0.25,
            0.30,
            0.35,
            0.40,
            0.45,
            0.50,
            0.55,
            0.60,
            0.65,
            0.70,
            0.75,
            0.80,
            0.85,
            0.90,
        ],
    )
)

FINAL_COOLDOWN_TUNE_GRID = list(
    globals().get(
        "FINAL_COOLDOWN_TUNE_GRID",
        [
            0.20,
            0.35,
            0.50,
        ],
    )
)
FINAL_MIN_GAP_TUNE_GRID = list(
    globals().get(
        "FINAL_MIN_GAP_TUNE_GRID",
        [
            0.10,
            0.20,
            0.35,
        ],
    )
)

SHIFT_TIME_OFFSET_TUNE_GRID = list(
    globals().get(
        "SHIFT_TIME_OFFSET_TUNE_GRID",
        [0.0],
    )
)

TIMING_CORRECTOR_AVAILABLE_FOR_FUSION = bool(
    "TIMING_CORRECTOR_VAL_SCORES"
    in globals()
    and isinstance(
        TIMING_CORRECTOR_VAL_SCORES,
        pd.DataFrame,
    )
    and len(
        TIMING_CORRECTOR_VAL_SCORES
    )
    and {
        "predicted_offset_s",
        "corrected_time",
    }.issubset(
        TIMING_CORRECTOR_VAL_SCORES.columns
    )
)

FINAL_TIME_MODE_GRID = list(
    globals().get(
        "FINAL_TIME_MODE_GRID",
        [
            "raw",
            "local_corrected",
        ]
        if TIMING_CORRECTOR_AVAILABLE_FOR_FUSION
        else [
            "raw",
        ],
    )
)
FINAL_TIME_MODE_GRID = [
    str(mode)
    for mode in FINAL_TIME_MODE_GRID
    if str(mode)
    in {
        "raw",
        "local_corrected",
    }
]
if not FINAL_TIME_MODE_GRID:
    FINAL_TIME_MODE_GRID = [
        "raw",
    ]

FINAL_RESCUE_MODE_GRID = list(
    globals().get(
        "FINAL_RESCUE_MODE_GRID",
        [
            "none",
        ],
    )
)
FINAL_RESCUE_THR_GRID = list(
    globals().get(
        "FINAL_RESCUE_THR_GRID",
        [
            0.30,
        ],
    )
)
FINAL_RESCUE_MIN_SOURCES_GRID = list(
    globals().get(
        "FINAL_RESCUE_MIN_SOURCES_GRID",
        [
            2,
        ],
    )
)

RESCUE_WAVLM_THR = float(globals().get("RESCUE_WAVLM_THR", 0.08))
RESCUE_QWEN_THR = float(globals().get("RESCUE_QWEN_THR", 0.20))
RESCUE_PYANNOTE_CHANGE_MAX_DIST = float(globals().get("RESCUE_PYANNOTE_CHANGE_MAX_DIST", 0.25))
RESCUE_ASR_MAX_DIST = float(globals().get("RESCUE_ASR_MAX_DIST", 0.35))
RESCUE_ECAPA_MAX_COS = float(globals().get("RESCUE_ECAPA_MAX_COS", 0.48))

VAL_SELECT_MIN_PRECISION = float(globals().get("VAL_SELECT_MIN_PRECISION", TARGET_FINAL_PRECISION))
VAL_SELECT_MIN_RECALL = float(globals().get("VAL_SELECT_MIN_RECALL", TARGET_FINAL_RECALL))

def _cell20_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)
    if not path.exists():
        return {"exists": False, "path": str(path), "size": None, "mtime": None}
    st = path.stat()
    return {"exists": True, "path": str(path), "size": int(st.st_size), "mtime": float(st.st_mtime)}

def _cell20_fusion_config() -> Dict[str, Any]:
    return {
        "variant": CELL20_VARIANT,
        "run_version": str(RUN_VERSION),
        "seed": int(SEED),
        "tol": float(TOL),

        "fusion_alpha_grid": [float(x) for x in FUSION_ALPHA_GRID],
        "final_score_mode_grid": [str(x) for x in FINAL_SCORE_MODE_GRID],
        "final_main_thr_grid": [float(x) for x in FINAL_MAIN_THR_GRID],
        "final_cooldown_tune_grid": [float(x) for x in FINAL_COOLDOWN_TUNE_GRID],
        "final_min_gap_tune_grid": [float(x) for x in FINAL_MIN_GAP_TUNE_GRID],
        "shift_time_offset_tune_grid": [float(x) for x in SHIFT_TIME_OFFSET_TUNE_GRID],
        "final_time_mode_grid": [str(x) for x in FINAL_TIME_MODE_GRID],
        "timing_corrector_available": bool(TIMING_CORRECTOR_AVAILABLE_FOR_FUSION),
        "timing_corrector_variant": str(globals().get("TIMING_CORRECTOR_VARIANT", "")),
        "timing_corrector_meta_fingerprint": (
            _cell20_file_fingerprint(TIMING_CORRECTOR_META_PATH)
            if "TIMING_CORRECTOR_META_PATH" in globals()
            else None
        ),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_min_s": float(PRIMARY_MIN_GAP_S),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "robust_val_folds": int(VAL_ROBUST_N_FOLDS),
        "robust_val_top_k": int(VAL_ROBUST_TOP_K),
        "robust_val_std_penalty": float(VAL_ROBUST_STD_PENALTY),
        "final_rescue_mode_grid": [str(x) for x in FINAL_RESCUE_MODE_GRID],
        "final_rescue_thr_grid": [float(x) for x in FINAL_RESCUE_THR_GRID],
        "final_rescue_min_sources_grid": [int(x) for x in FINAL_RESCUE_MIN_SOURCES_GRID],
        "rescue_wavlm_thr": float(RESCUE_WAVLM_THR),
        "rescue_qwen_thr": float(RESCUE_QWEN_THR),
        "rescue_pyannote_change_max_dist": float(RESCUE_PYANNOTE_CHANGE_MAX_DIST),
        "rescue_asr_max_dist": float(RESCUE_ASR_MAX_DIST),
        "rescue_ecapa_max_cos": float(RESCUE_ECAPA_MAX_COS),

        "target_final_precision": float(TARGET_FINAL_PRECISION),
        "target_final_recall": float(TARGET_FINAL_RECALL),
        "target_final_f1": float(TARGET_FINAL_F1),
        "val_select_min_precision": float(VAL_SELECT_MIN_PRECISION),
        "val_select_min_recall": float(VAL_SELECT_MIN_RECALL),

        "temporal_val_scores_fingerprint": _cell20_file_fingerprint(TEMPORAL_VAL_SCORES_PATH) if "TEMPORAL_VAL_SCORES_PATH" in globals() else None,
        "temporal_test_scores_fingerprint": _cell20_file_fingerprint(TEMPORAL_TEST_SCORES_PATH) if "TEMPORAL_TEST_SCORES_PATH" in globals() else None,

        "lgbm_val_scores_fingerprint": _cell20_file_fingerprint(LGBM_VAL_SCORES_PATH) if "LGBM_VAL_SCORES_PATH" in globals() else None,
        "lgbm_test_scores_fingerprint": _cell20_file_fingerprint(LGBM_TEST_SCORES_PATH) if "LGBM_TEST_SCORES_PATH" in globals() else None,

        "lgbm_binary_val_scores_fingerprint": _cell20_file_fingerprint(LGBM_BINARY_VAL_SCORES_PATH) if "LGBM_BINARY_VAL_SCORES_PATH" in globals() else None,
        "lgbm_binary_test_scores_fingerprint": _cell20_file_fingerprint(LGBM_BINARY_TEST_SCORES_PATH) if "LGBM_BINARY_TEST_SCORES_PATH" in globals() else None,

        "ranker_val_scores_fingerprint": _cell20_file_fingerprint(RANKER_VAL_SCORES_PATH) if "RANKER_VAL_SCORES_PATH" in globals() else None,
        "ranker_test_scores_fingerprint": _cell20_file_fingerprint(RANKER_TEST_SCORES_PATH) if "RANKER_TEST_SCORES_PATH" in globals() else None,
        "rr_fap_val_scores_fingerprint": _cell20_file_fingerprint(RR_FAP_VAL_SCORES_PATH) if "RR_FAP_VAL_SCORES_PATH" in globals() else None,
        "rr_fap_test_scores_fingerprint": _cell20_file_fingerprint(RR_FAP_TEST_SCORES_PATH) if "RR_FAP_TEST_SCORES_PATH" in globals() else None,
        "meta_confirmer_val_scores_fingerprint": _cell20_file_fingerprint(META_CONFIRMER_VAL_SCORES_PATH) if "META_CONFIRMER_VAL_SCORES_PATH" in globals() else None,
        "meta_confirmer_test_scores_fingerprint": _cell20_file_fingerprint(META_CONFIRMER_TEST_SCORES_PATH) if "META_CONFIRMER_TEST_SCORES_PATH" in globals() else None,
        "has_event_ranker_score": bool(globals().get("LGBM_HAS_EVENT_RANKER_SCORE", False)),
        "has_rr_fap_score": bool(globals().get("LGBM_HAS_RR_FAP_SCORE", False)),

        "n_meta_sys2_val": int(len(META_SYS2_VAL)) if "META_SYS2_VAL" in globals() else -1,
        "n_temporal_val_scores": int(len(TEMPORAL_VAL_SCORES)) if "TEMPORAL_VAL_SCORES" in globals() else 0,
        "n_lgbm_val_scores": int(len(LGBM_VAL_SCORES)) if "LGBM_VAL_SCORES" in globals() else 0,
        "n_ranker_val_scores": int(len(RANKER_VAL_SCORES)) if "RANKER_VAL_SCORES" in globals() else 0,
        "n_rr_fap_val_scores": int(len(RR_FAP_VAL_SCORES)) if "RR_FAP_VAL_SCORES" in globals() else 0,
        "n_real_val_events": int(len(REAL_VAL_EVENTS)) if "REAL_VAL_EVENTS" in globals() else -1,

        "locked_on": "real_val_event_level",
        "test_not_used_for_selection": True,
    }

def _json_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False
    try:
        old = json.loads(Path(path).read_text())
        return old == expected
    except Exception:
        return False

def _attach_nearest_score(
    base_df: pd.DataFrame,
    score_df: pd.DataFrame,
    score_col: str,
    out_col: str,
    max_dist: float = 0.05,
) -> pd.DataFrame:
    out = base_df.copy()

    if score_df is None or len(score_df) == 0 or score_col not in score_df.columns:
        out[out_col] = np.nan
        return out

    out = out.drop(columns=[out_col], errors="ignore")

    out["_row_id_tmp"] = np.arange(len(out), dtype=int)

    work_score = score_df[["session", "time", score_col]].copy()
    work_score["session"] = work_score["session"].astype(str)
    work_score["time"] = pd.to_numeric(work_score["time"], errors="coerce")
    work_score[score_col] = pd.to_numeric(work_score[score_col], errors="coerce")
    work_score = work_score.dropna(subset=["session", "time", score_col]).reset_index(drop=True)

    if len(work_score) == 0:
        out[out_col] = np.nan
        return out.drop(columns=["_row_id_tmp"], errors="ignore")

    merged_parts = []

    for sid, g in out.groupby("session", sort=False):
        left = g.sort_values("time").copy()
        right = work_score[work_score["session"] == str(sid)].copy()

        if len(right) == 0:
            left[out_col] = np.nan
            merged_parts.append(left)
            continue

        right = right.sort_values("time")

        merged = pd.merge_asof(
            left,
            right[["time", score_col]],
            on="time",
            direction="nearest",
            tolerance=float(max_dist),
        )

        merged = merged.rename(columns={score_col: out_col})
        merged_parts.append(merged)

    out = pd.concat(merged_parts, ignore_index=True)
    out = out.sort_values("_row_id_tmp").drop(columns=["_row_id_tmp"], errors="ignore").reset_index(drop=True)

    if out_col not in out.columns:
        out[out_col] = np.nan

    return out

TRACE_COLS_FOR_FUSION = [
    "source_wavlm",
    "source_pyannote",
    "source_pyannote_change",
    "source_pyannote_overlap",
    "source_asr",
    "source_qwen3",
    "wavlm_score",
    "wavlm_center_score",
    "pyannote_change_score",
    "pyannote_change_dist",
    "pyannote_any_candidate_dist",
    "pyannote_overlap_dist",
    "asr_boundary_score",
    "asr_boundary_dist",
    "qwen_shift_score",
    "qwen_backchannel_score",
    "qwen_interruption_score",
    "qwen_p_NTT",
    "qwen_p_ITT",
    "wavlm_p_NTT",
    "wavlm_p_ITT",
    "ecapa_available",
    "ecapa_cos",
    "candidate_density_1s",
    "candidate_density_2s",
]

def _score_mode_name(score_mode: str = None) -> str:
    if score_mode is None:
        if "FINAL_LOCKED" in globals() and isinstance(FINAL_LOCKED, dict):
            return str(FINAL_LOCKED.get("score_mode", "weighted_lgbm_multi_temporal"))
        return "weighted_lgbm_multi_temporal"
    return str(score_mode)

def combine_scores(
    base_meta: pd.DataFrame,
    temporal: pd.DataFrame,
    lgbm: pd.DataFrame,
    alpha: float = 0.5,
    score_mode: str = None,
    timing: pd.DataFrame = None,
) -> pd.DataFrame:
    score_mode = _score_mode_name(score_mode)

    if base_meta is None or len(base_meta) == 0:
        return pd.DataFrame(columns=[
            "session", "time", "split", "label", "action", "action_id",
            "temporal_shift_score",
            "lgbm_shift_score",
            "lgbm_binary_shift_score",
            "ranker_shift_score",
            "rr_role_shift_score",
            "meta_confirmer_shift_score",
            "predicted_offset_s",
            "corrected_time",
            "fused_shift_score",
            "score_mode",
        ])

    cols = ["session", "time", "split", "label", "action", "action_id", "audio_path"]
    cols += [c for c in TRACE_COLS_FOR_FUSION if c in base_meta.columns]
    cols = list(dict.fromkeys([c for c in cols if c in base_meta.columns]))

    out = base_meta[cols].copy().reset_index(drop=True)

    out["session"] = out["session"].astype(str)
    out["time"] = pd.to_numeric(out["time"], errors="coerce")
    out = out.dropna(subset=["session", "time"]).reset_index(drop=True)

    out = _attach_nearest_score(
        out,
        temporal,
        score_col="temporal_shift_score",
        out_col="temporal_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        lgbm,
        score_col="lgbm_shift_score",
        out_col="lgbm_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        lgbm,
        score_col="lgbm_binary_shift_score",
        out_col="lgbm_binary_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        lgbm,
        score_col="ranker_shift_score",
        out_col="ranker_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        lgbm,
        score_col="rr_role_shift_score",
        out_col="rr_role_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        lgbm,
        score_col="meta_confirmer_shift_score",
        out_col="meta_confirmer_shift_score",
        max_dist=0.05,
    )

    out = _attach_nearest_score(
        out,
        timing if timing is not None else pd.DataFrame(),
        score_col="predicted_offset_s",
        out_col="predicted_offset_s",
        max_dist=0.05,
    )
    out = _attach_nearest_score(
        out,
        timing if timing is not None else pd.DataFrame(),
        score_col="corrected_time",
        out_col="corrected_time",
        max_dist=0.05,
    )

    out["predicted_offset_s"] = (
        pd.to_numeric(
            out["predicted_offset_s"],
            errors="coerce",
        )
        .fillna(0.0)
        .clip(
            -float(globals().get("TIMING_CORRECTOR_MAX_ABS_OFFSET_S", 0.75)),
            float(globals().get("TIMING_CORRECTOR_MAX_ABS_OFFSET_S", 0.75)),
        )
    )
    out["corrected_time"] = (
        pd.to_numeric(
            out["corrected_time"],
            errors="coerce",
        )
        .fillna(
            out["time"]
            + out["predicted_offset_s"]
        )
    )

    out["temporal_shift_score"] = pd.to_numeric(out["temporal_shift_score"], errors="coerce")
    out["lgbm_shift_score"] = pd.to_numeric(out["lgbm_shift_score"], errors="coerce")
    out["lgbm_binary_shift_score"] = pd.to_numeric(out["lgbm_binary_shift_score"], errors="coerce")
    out["ranker_shift_score"] = pd.to_numeric(out["ranker_shift_score"], errors="coerce")
    out["rr_role_shift_score"] = pd.to_numeric(out["rr_role_shift_score"], errors="coerce")
    out["meta_confirmer_shift_score"] = pd.to_numeric(out["meta_confirmer_shift_score"], errors="coerce")

    out["lgbm_binary_shift_score"] = (
        out["lgbm_binary_shift_score"]
        .fillna(out["lgbm_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["lgbm_shift_score"] = (
        out["lgbm_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["temporal_shift_score"] = (
        out["temporal_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["lgbm_shift_score"])
        .fillna(0.0)
    )

    out["ranker_shift_score"] = (
        out["ranker_shift_score"]
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["lgbm_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["rr_role_shift_score"] = (
        out["rr_role_shift_score"]
        .fillna(out["ranker_shift_score"])
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["lgbm_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    out["meta_confirmer_shift_score"] = (
        out["meta_confirmer_shift_score"]
        .fillna(out["rr_role_shift_score"])
        .fillna(out["ranker_shift_score"])
        .fillna(out["lgbm_binary_shift_score"])
        .fillna(out["temporal_shift_score"])
        .fillna(0.0)
    )

    alpha = float(alpha)
    alpha = min(1.0, max(0.0, alpha))

    if score_mode == "meta_confirmer":
        fused = out["meta_confirmer_shift_score"]

    elif score_mode == "avg_meta_rr":
        fused = 0.5 * out["meta_confirmer_shift_score"] + 0.5 * out["rr_role_shift_score"]

    elif score_mode == "max_meta_binary":
        fused = np.maximum(out["meta_confirmer_shift_score"], out["lgbm_binary_shift_score"])

    elif score_mode == "avg_meta_rr_ranker":
        fused = (
            out["meta_confirmer_shift_score"]
            + out["rr_role_shift_score"]
            + out["ranker_shift_score"]
        ) / 3.0

    elif score_mode == "weighted_meta_temporal":
        fused = alpha * out["temporal_shift_score"] + (1.0 - alpha) * out["meta_confirmer_shift_score"]

    elif score_mode == "rr_fap":
        fused = out["rr_role_shift_score"]

    elif score_mode == "avg_rr_binary":
        fused = 0.5 * out["rr_role_shift_score"] + 0.5 * out["lgbm_binary_shift_score"]

    elif score_mode == "max_rr_binary":
        fused = np.maximum(out["rr_role_shift_score"], out["lgbm_binary_shift_score"])

    elif score_mode == "avg_rr_ranker_binary":
        fused = (
            out["rr_role_shift_score"]
            + out["ranker_shift_score"]
            + out["lgbm_binary_shift_score"]
        ) / 3.0

    elif score_mode == "weighted_rr_temporal":
        fused = alpha * out["temporal_shift_score"] + (1.0 - alpha) * out["rr_role_shift_score"]

    elif score_mode == "weighted_rr_ranker_temporal":
        fused = (
            0.40 * out["rr_role_shift_score"]
            + 0.40 * out["ranker_shift_score"]
            + 0.20 * out["temporal_shift_score"]
        )

    elif score_mode == "ranker":
        fused = out["ranker_shift_score"]

    elif score_mode == "avg_ranker_binary":
        fused = 0.5 * out["ranker_shift_score"] + 0.5 * out["lgbm_binary_shift_score"]

    elif score_mode == "max_ranker_binary":
        fused = np.maximum(out["ranker_shift_score"], out["lgbm_binary_shift_score"])

    elif score_mode == "weighted_ranker_temporal":
        fused = alpha * out["temporal_shift_score"] + (1.0 - alpha) * out["ranker_shift_score"]

    elif score_mode == "lgbm_binary":
        fused = out["lgbm_binary_shift_score"]

    elif score_mode == "lgbm_multiclass":
        fused = out["lgbm_shift_score"]

    elif score_mode == "avg_binary_multi":
        fused = 0.5 * out["lgbm_binary_shift_score"] + 0.5 * out["lgbm_shift_score"]

    elif score_mode == "max_binary_multi":
        fused = np.maximum(out["lgbm_binary_shift_score"], out["lgbm_shift_score"])

    elif score_mode == "weighted_binary_temporal":
        fused = alpha * out["temporal_shift_score"] + (1.0 - alpha) * out["lgbm_binary_shift_score"]

    elif score_mode == "temporal_only":
        fused = out["temporal_shift_score"]

    else:
        score_mode = "weighted_lgbm_multi_temporal"
        fused = alpha * out["temporal_shift_score"] + (1.0 - alpha) * out["lgbm_shift_score"]

    out["fused_shift_score"] = pd.to_numeric(fused, errors="coerce").fillna(0.0).clip(0.0, 1.0)
    out["score_mode"] = score_mode

    return out

def _as_float_series(df: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.full(len(df), float(default)), index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(float(default)).astype(float)

def _as_int_series(df: pd.DataFrame, col: str, default: int = 0) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.full(len(df), int(default)), index=df.index)
    return pd.to_numeric(df[col], errors="coerce").fillna(int(default)).astype(int)

def _source_agreement_count(work: pd.DataFrame) -> pd.Series:
    n = pd.Series(np.zeros(len(work), dtype=np.int32), index=work.index)

    wavlm_shift = (
        (_as_int_series(work, "source_wavlm") > 0)
        | (_as_float_series(work, "wavlm_score") >= float(RESCUE_WAVLM_THR))
        | ((_as_float_series(work, "wavlm_p_NTT") + _as_float_series(work, "wavlm_p_ITT")) >= float(RESCUE_WAVLM_THR))
    )
    n += wavlm_shift.astype(np.int32)

    pyannote_change = (
        (_as_int_series(work, "source_pyannote_change") > 0)
        | (_as_float_series(work, "pyannote_change_score") > 0)
        | (_as_float_series(work, "pyannote_change_dist", 999.0) <= float(RESCUE_PYANNOTE_CHANGE_MAX_DIST))
    )
    n += pyannote_change.astype(np.int32)

    asr_boundary = (
        (_as_int_series(work, "source_asr") > 0)
        | (_as_float_series(work, "asr_boundary_score") > 0)
        | (_as_float_series(work, "asr_boundary_dist", 999.0) <= float(RESCUE_ASR_MAX_DIST))
    )
    n += asr_boundary.astype(np.int32)

    qwen_shift = (
        (_as_int_series(work, "source_qwen3") > 0)
        & (
            (_as_float_series(work, "qwen_shift_score") >= float(RESCUE_QWEN_THR))
            | ((_as_float_series(work, "qwen_p_NTT") + _as_float_series(work, "qwen_p_ITT")) >= float(RESCUE_QWEN_THR))
        )
    )
    n += qwen_shift.astype(np.int32)

    ecapa_change = (
        (_as_int_series(work, "ecapa_available") > 0)
        & (_as_float_series(work, "ecapa_cos", 1.0) <= float(RESCUE_ECAPA_MAX_COS))
    )
    n += ecapa_change.astype(np.int32)

    return n.astype(int)

def apply_final_event_postprocess(
    score_df: pd.DataFrame,
    score_col: str,
    thr: float,
    cooldown_s: float = 0.2,
    min_gap_s: float = 0.2,
    offset_s: float = 0.0,
    rescue_mode: str = "none",
    rescue_thr: float = 0.0,
    rescue_min_sources: int = 0,
    time_mode: str = "raw",
) -> pd.DataFrame:
    if (
        score_df is None
        or len(score_df) == 0
        or score_col not in score_df.columns
    ):
        return pd.DataFrame(
            columns=[
                "session",
                "time",
                "raw_time",
                "corrected_time",
                "predicted_offset_s",
                "score",
                "time_mode",
            ]
        )

    work = score_df.copy()
    work["session"] = work["session"].astype(str)
    work["time"] = pd.to_numeric(
        work["time"],
        errors="coerce",
    )
    work[score_col] = pd.to_numeric(
        work[score_col],
        errors="coerce",
    )
    work = work.dropna(
        subset=["session", "time", score_col]
    ).copy()

    if len(work) == 0:
        return pd.DataFrame(
            columns=[
                "session",
                "time",
                "raw_time",
                "corrected_time",
                "predicted_offset_s",
                "score",
                "time_mode",
            ]
        )

    work["predicted_offset_s"] = pd.to_numeric(
        work.get(
            "predicted_offset_s",
            pd.Series(
                0.0,
                index=work.index,
            ),
        ),
        errors="coerce",
    ).fillna(0.0)

    work["corrected_time"] = pd.to_numeric(
        work.get(
            "corrected_time",
            work["time"]
            + work["predicted_offset_s"],
        ),
        errors="coerce",
    ).fillna(
        work["time"]
        + work["predicted_offset_s"]
    )

    time_mode = str(time_mode)
    if time_mode not in {
        "raw",
        "local_corrected",
    }:
        raise ValueError(
            f"Unknown final time_mode={time_mode!r}."
        )

    rescue_mode = str(
        rescue_mode or "none"
    )
    main_accept = (
        work[score_col]
        >= float(thr)
    )

    if rescue_mode == "source_agree":
        agreement = _source_agreement_count(
            work
        )
        work[
            "source_agree_count"
        ] = agreement.astype(int)

        rescue_accept = (
            work[score_col]
            >= float(rescue_thr)
        ) & (
            agreement
            >= int(rescue_min_sources)
        )

        work["accept_reason"] = np.where(
            main_accept,
            "main",
            np.where(
                rescue_accept,
                "rescue_source_agree",
                "reject",
            ),
        )
        work = work[
            main_accept | rescue_accept
        ].copy()
    else:
        work["source_agree_count"] = 0
        work["accept_reason"] = np.where(
            main_accept,
            "main",
            "reject",
        )
        work = work[
            main_accept
        ].copy()

    if len(work) == 0:
        return pd.DataFrame(
            columns=[
                "session",
                "time",
                "raw_time",
                "corrected_time",
                "predicted_offset_s",
                "score",
                "time_mode",
            ]
        )

    suppression_gap = max(
        float(cooldown_s),
        float(min_gap_s),
    )
    rows = []

    def _emit(
        session_id: str,
        cluster: list,
    ) -> Dict[str, Any]:
        best = max(
            cluster,
            key=lambda item: (
                item["score"],
                item[
                    "source_agree_count"
                ],
            ),
        )

        raw_time = float(
            best["raw_time"]
        )
        corrected_time = float(
            best["corrected_time"]
        )

        selected_time = (
            corrected_time
            if time_mode
            == "local_corrected"
            else raw_time
        )
        selected_time = float(
            selected_time
            + float(offset_s)
        )

        return {
            "session": str(session_id),
            "time": selected_time,
            "raw_time": raw_time,
            "corrected_time": corrected_time,
            "predicted_offset_s": float(
                best[
                    "predicted_offset_s"
                ]
            ),
            "score": float(
                best["score"]
            ),
            "source_agree_count": int(
                best[
                    "source_agree_count"
                ]
            ),
            "accept_reason": str(
                best["accept_reason"]
            ),
            "time_mode": str(time_mode),
            "global_offset_s": float(
                offset_s
            ),
        }

    for session_id, group in work.groupby(
        "session",
        sort=False,
    ):
        group = group.sort_values(
            "time"
        ).reset_index(drop=True)

        current_cluster = []

        for _, row in group.iterrows():
            item = {
                "raw_time": float(
                    row["time"]
                ),
                "corrected_time": float(
                    row["corrected_time"]
                ),
                "predicted_offset_s": float(
                    row[
                        "predicted_offset_s"
                    ]
                ),
                "score": float(
                    row[score_col]
                ),
                "source_agree_count": int(
                    row.get(
                        "source_agree_count",
                        0,
                    )
                ),
                "accept_reason": str(
                    row.get(
                        "accept_reason",
                        "main",
                    )
                ),
            }

            if not current_cluster:
                current_cluster = [
                    item
                ]
                continue

            previous_raw_time = float(
                current_cluster[-1][
                    "raw_time"
                ]
            )

            if (
                item["raw_time"]
                - previous_raw_time
                <= suppression_gap
            ):
                current_cluster.append(
                    item
                )
            else:
                rows.append(
                    _emit(
                        str(session_id),
                        current_cluster,
                    )
                )
                current_cluster = [
                    item
                ]

        if current_cluster:
            rows.append(
                _emit(
                    str(session_id),
                    current_cluster,
                )
            )

    output = pd.DataFrame(rows)

    if len(output):
        output["time"] = pd.to_numeric(
            output["time"],
            errors="coerce",
        )
        output = output.dropna(
            subset=["time"]
        )
        output = output[
            output["time"] >= 0
        ].copy()

        if (
            time_mode == "local_corrected"
            and len(output)
        ):
            corrected_rows = []

            for session_id, group in output.groupby(
                "session",
                sort=False,
            ):
                group = group.sort_values(
                    "time"
                ).reset_index(drop=True)
                cluster = []

                for _, row in group.iterrows():
                    record = row.to_dict()

                    if not cluster:
                        cluster = [record]
                        continue

                    if (
                        float(record["time"])
                        - float(cluster[-1]["time"])
                        <= suppression_gap
                    ):
                        cluster.append(record)
                    else:
                        corrected_rows.append(
                            max(
                                cluster,
                                key=lambda item: (
                                    float(item["score"]),
                                    int(
                                        item.get(
                                            "source_agree_count",
                                            0,
                                        )
                                    ),
                                ),
                            )
                        )
                        cluster = [record]

                if cluster:
                    corrected_rows.append(
                        max(
                            cluster,
                            key=lambda item: (
                                float(item["score"]),
                                int(
                                    item.get(
                                        "source_agree_count",
                                        0,
                                    )
                                ),
                            ),
                        )
                    )

            output = pd.DataFrame(
                corrected_rows
            )

        output = (
            output
            .sort_values(
                [
                    "session",
                    "time",
                ]
            )
            .reset_index(drop=True)
        )

    return output

def _match_event_times_safe(pred_times, gt_times, tol: float) -> Dict[str, int]:
    pred_times = np.asarray(pred_times, dtype=float).reshape(-1)
    gt_times = np.asarray(gt_times, dtype=float).reshape(-1)

    if "match_event_times" in globals():
        try:
            return match_event_times(pred_times, gt_times, tol=float(tol), prefer_min_cost=True)
        except TypeError:
            return match_event_times(pred_times, gt_times, tol=float(tol))

    pairs = []

    for i, p in enumerate(pred_times):
        d = np.abs(gt_times - float(p))
        js = np.where(d <= float(tol))[0]
        for j in js:
            pairs.append((float(d[j]), int(i), int(j)))

    pairs.sort(key=lambda x: x[0])
    used_p, used_g = set(), set()

    for _, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)

    tp = len(used_p)
    fp = len(pred_times) - tp
    fn = len(gt_times) - tp

    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "N_pred": int(len(pred_times)), "N_gt": int(len(gt_times))}

def event_metrics_against_event_table(
    pred_df: pd.DataFrame,
    events_df: pd.DataFrame,
    tol: float = TOL,
) -> Dict[str, Any]:
    if pred_df is None or len(pred_df) == 0:
        pred_df = pd.DataFrame(columns=["session", "time"])

    pred = pred_df[["session", "time"]].copy()
    pred["session"] = pred["session"].astype(str)
    pred["time"] = pd.to_numeric(pred["time"], errors="coerce")
    pred = pred.dropna(subset=["session", "time"]).reset_index(drop=True)

    ev = events_df.copy()
    ev["session"] = ev["session"].astype(str)
    ev["time"] = pd.to_numeric(ev["time"], errors="coerce")
    ev["label"] = pd.to_numeric(ev["label"], errors="coerce").fillna(0).astype(int)
    ev = ev.dropna(subset=["session", "time"]).reset_index(drop=True)

    sessions = sorted(set(ev["session"].unique()).union(set(pred["session"].unique())))

    total = {"TP": 0, "FP": 0, "FN": 0, "N_pred": 0, "N_gt": 0}

    for sid in sessions:
        pt = pred.loc[pred["session"] == sid, "time"].to_numpy(dtype=float)
        gt = ev.loc[(ev["session"] == sid) & (ev["label"] == 1), "time"].to_numpy(dtype=float)

        m = _match_event_times_safe(pt, gt, tol=float(tol))

        for k in total:
            total[k] += int(m.get(k, 0))

    tp, fp, fn = total["TP"], total["FP"], total["FN"]
    p = tp / max(1, tp + fp)
    r = tp / max(1, tp + fn)
    f1 = 2 * p * r / max(1e-12, p + r)

    return {"P": float(p), "R": float(r), "F1": float(f1), **total}

def _json_safe_final_dict(d: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in d.items():
        if isinstance(v, (np.floating, float)):
            out[k] = float(v)
        elif isinstance(v, (np.integer, int)):
            out[k] = int(v)
        elif isinstance(v, (np.bool_, bool)):
            out[k] = bool(v)
        else:
            out[k] = v
    return out

def _select_final_val_setting(
    frame: pd.DataFrame,
) -> pd.Series:
    assert (
        frame is not None
        and len(frame) > 0
    ), "Empty final VAL fusion sweep."

    work = frame.copy()

    target_all = work[
        work["P"].ge(
            float(
                TARGET_FINAL_PRECISION
            )
        )
        & work["R"].ge(
            float(
                TARGET_FINAL_RECALL
            )
        )
        & work["F1"].ge(
            float(
                TARGET_FINAL_F1
            )
        )
    ]
    target_pr = work[
        work["P"].ge(
            float(
                TARGET_FINAL_PRECISION
            )
        )
        & work["R"].ge(
            float(
                TARGET_FINAL_RECALL
            )
        )
    ]
    relaxed = work[
        work["P"].ge(
            float(
                VAL_SELECT_MIN_PRECISION
            )
        )
        & work["R"].ge(
            float(
                VAL_SELECT_MIN_RECALL
            )
        )
    ]

    if len(target_all):
        pool = target_all
        reason = (
            "met final target P/R/F1 floors"
        )
    elif len(target_pr):
        pool = target_pr
        reason = (
            "met final P/R floors but not "
            "the complete F1 target"
        )
    elif len(relaxed):
        pool = relaxed
        reason = (
            "met configured VAL P/R floors"
        )
    else:
        pool = work
        reason = (
            "no setting met P/R floors; "
            "selected robust VAL setting"
        )

    if (
        "robust_score" in pool.columns
        and pd.to_numeric(
            pool["robust_score"],
            errors="coerce",
        ).notna().any()
    ):
        pool = pool.sort_values(
            [
                "robust_score",
                "cv_f1_mean",
                "cv_f1_min",
                "F1",
                "P",
                "R",
                "N_pred",
            ],
            ascending=[
                False,
                False,
                False,
                False,
                False,
                False,
                True,
            ],
        )
        reason += (
            "; ranked by session-fold mean F1 "
            "minus variability penalty"
        )
    else:
        pool = pool.sort_values(
            [
                "F1",
                "P",
                "R",
                "N_pred",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )

    best = pool.iloc[0].copy()
    best[
        "selection_reason"
    ] = reason
    return best

def _alpha_grid_for_mode(score_mode: str) -> List[float]:
    score_mode = str(score_mode)

    if score_mode in {
        "weighted_binary_temporal",
        "weighted_lgbm_multi_temporal",
        "weighted_ranker_temporal",
        "weighted_rr_temporal",
        "weighted_meta_temporal",
    }:
        return [float(x) for x in FUSION_ALPHA_GRID]

    if score_mode == "temporal_only":
        return [1.0]

    return [0.0]

def _rescue_grid() -> List[Tuple[str, float, int]]:
    rows = [("none", 0.0, 0)]
    for mode in FINAL_RESCUE_MODE_GRID:
        mode = str(mode)
        if mode == "none":
            continue
        for rthr in FINAL_RESCUE_THR_GRID:
            for nsrc in FINAL_RESCUE_MIN_SOURCES_GRID:
                rows.append((mode, float(rthr), int(nsrc)))
    return rows

def _stable_val_fold(
    session_id: str,
    n_folds: int,
) -> int:
    import hashlib

    digest = hashlib.sha1(
        str(session_id).encode(
            "utf-8"
        )
    ).hexdigest()

    return int(
        digest[:12],
        16,
    ) % max(
        1,
        int(n_folds),
    )

def _add_robust_val_metrics(
    sweep_frame: pd.DataFrame,
    fused_cache: Dict[
        Tuple[str, float],
        pd.DataFrame,
    ],
) -> pd.DataFrame:
    output = sweep_frame.copy()

    robust_columns = [
        "cv_f1_mean",
        "cv_f1_std",
        "cv_f1_min",
        "cv_p_mean",
        "cv_r_mean",
        "robust_score",
    ]
    for column in robust_columns:
        output[column] = np.nan

    shortlist_indices = (
        output
        .sort_values(
            [
                "F1",
                "P",
                "R",
                "N_pred",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .head(
            min(
                int(
                    VAL_ROBUST_TOP_K
                ),
                len(output),
            )
        )
        .index
        .tolist()
    )

    val_sessions = sorted(
        set(
            REAL_VAL_EVENTS[
                "session"
            ].astype(str)
        )
    )
    session_fold = {
        session: _stable_val_fold(
            session,
            int(
                VAL_ROBUST_N_FOLDS
            ),
        )
        for session in val_sessions
    }

    for row_index in shortlist_indices:
        setting = output.loc[
            row_index
        ]

        key = (
            str(
                setting[
                    "score_mode"
                ]
            ),
            float(
                setting[
                    "alpha_temporal"
                ]
            ),
        )
        fused = fused_cache[key]

        prediction = apply_final_event_postprocess(
            fused,
            score_col="fused_shift_score",
            thr=float(setting["thr"]),
            cooldown_s=float(
                setting["cooldown"]
            ),
            min_gap_s=float(
                setting["min_gap"]
            ),
            offset_s=float(
                setting["offset"]
            ),
            rescue_mode=str(
                setting[
                    "rescue_mode"
                ]
            ),
            rescue_thr=float(
                setting[
                    "rescue_thr"
                ]
            ),
            rescue_min_sources=int(
                setting[
                    "rescue_min_sources"
                ]
            ),
            time_mode=str(
                setting.get(
                    "time_mode",
                    "raw",
                )
            ),
        )

        fold_metrics = []

        for fold in range(
            int(
                VAL_ROBUST_N_FOLDS
            )
        ):
            fold_sessions = {
                session
                for session, assigned
                in session_fold.items()
                if assigned == fold
            }

            if not fold_sessions:
                continue

            fold_prediction = prediction[
                prediction[
                    "session"
                ].astype(str).isin(
                    fold_sessions
                )
            ].copy()
            fold_events = REAL_VAL_EVENTS[
                REAL_VAL_EVENTS[
                    "session"
                ].astype(str).isin(
                    fold_sessions
                )
            ].copy()

            fold_metrics.append(
                event_metrics_against_event_table(
                    fold_prediction,
                    fold_events,
                    tol=float(TOL),
                )
            )

        if not fold_metrics:
            continue

        f1_values = np.asarray(
            [
                metric["F1"]
                for metric in fold_metrics
            ],
            dtype=float,
        )
        p_values = np.asarray(
            [
                metric["P"]
                for metric in fold_metrics
            ],
            dtype=float,
        )
        r_values = np.asarray(
            [
                metric["R"]
                for metric in fold_metrics
            ],
            dtype=float,
        )

        cv_f1_mean = float(
            np.mean(f1_values)
        )
        cv_f1_std = float(
            np.std(
                f1_values,
                ddof=0,
            )
        )

        output.loc[
            row_index,
            "cv_f1_mean",
        ] = cv_f1_mean
        output.loc[
            row_index,
            "cv_f1_std",
        ] = cv_f1_std
        output.loc[
            row_index,
            "cv_f1_min",
        ] = float(
            np.min(f1_values)
        )
        output.loc[
            row_index,
            "cv_p_mean",
        ] = float(
            np.mean(p_values)
        )
        output.loc[
            row_index,
            "cv_r_mean",
        ] = float(
            np.mean(r_values)
        )
        output.loc[
            row_index,
            "robust_score",
        ] = (
            cv_f1_mean
            - float(
                VAL_ROBUST_STD_PENALTY
            )
            * cv_f1_std
        )

    return output

expected_final_meta = _cell20_fusion_config()

has_any_verifier_scores = (
    ("TEMPORAL_VAL_SCORES" in globals() and TEMPORAL_VAL_SCORES is not None and len(TEMPORAL_VAL_SCORES) > 0)
    or ("LGBM_VAL_SCORES" in globals() and LGBM_VAL_SCORES is not None and len(LGBM_VAL_SCORES) > 0)
    or ("RANKER_VAL_SCORES" in globals() and RANKER_VAL_SCORES is not None and len(RANKER_VAL_SCORES) > 0)
    or ("META_CONFIRMER_VAL_SCORES" in globals() and META_CONFIRMER_VAL_SCORES is not None and len(META_CONFIRMER_VAL_SCORES) > 0)
)

assert has_any_verifier_scores, "No verifier VAL scores available. Run Cell 18 and/or Cell 19/19B/19C first."

if (
    FINAL_LOCKED_PATH.exists()
    and _json_meta_matches(FINAL_LOCKED_META_PATH, expected_final_meta)
    and not FORCE_RETUNE_VAL
):
    FINAL_LOCKED = json.loads(FINAL_LOCKED_PATH.read_text())
    print("Loaded final locked settings:", FINAL_LOCKED)

    if VAL_FUSION_TABLE_PATH.exists():
        VAL_FUSION_DF = pd.read_csv(VAL_FUSION_TABLE_PATH)
        print("Loaded VAL fusion sweep:", VAL_FUSION_TABLE_PATH)

else:
    base_val = META_SYS2_VAL.copy()

    assert len(base_val) > 0, "META_SYS2_VAL is empty. Run Cells 16–18 first."

    base_val = base_val[base_val["split"].astype(str) == "val"].copy()
    base_val = base_val[base_val["label"].astype(int).isin([0, 1])].copy()

    assert len(base_val) > 0, "No labelled real VAL rows available for final threshold tuning."

    results = []
    fused_cache = {}
    rescue_combos = _rescue_grid()

    print("VAL fusion search:")
    print("  score modes:", FINAL_SCORE_MODE_GRID)
    print("  required ranker modes present:", all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_RANKER_SCORE_MODES))
    print("  required RR-FAP modes present:", all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_RR_FAP_SCORE_MODES))
    print("  required meta-confirmer modes present:", all(m in FINAL_SCORE_MODE_GRID for m in REQUIRED_META_CONFIRMER_SCORE_MODES))
    print("  ranker score available:", RANKER_SCORE_AVAILABLE_FOR_FUSION)
    print("  RR-FAP score available:", RR_FAP_SCORE_AVAILABLE_FOR_FUSION)
    print("  meta-confirmer score available:", META_CONFIRMER_SCORE_AVAILABLE_FOR_FUSION)
    print("  main thresholds:", len(FINAL_MAIN_THR_GRID))
    print("  postprocess combos:", len(FINAL_COOLDOWN_TUNE_GRID) * len(FINAL_MIN_GAP_TUNE_GRID) * len(SHIFT_TIME_OFFSET_TUNE_GRID))
    print("  rescue combos:", len(rescue_combos))

    for score_mode in FINAL_SCORE_MODE_GRID:
        for alpha in _alpha_grid_for_mode(score_mode):
            key = (str(score_mode), float(alpha))

            fused = combine_scores(
                base_val,
                TEMPORAL_VAL_SCORES if "TEMPORAL_VAL_SCORES" in globals() else pd.DataFrame(),
                LGBM_VAL_SCORES if "LGBM_VAL_SCORES" in globals() else pd.DataFrame(),
                alpha=float(alpha),
                score_mode=str(score_mode),
                timing=(
                    TIMING_CORRECTOR_VAL_SCORES
                    if "TIMING_CORRECTOR_VAL_SCORES" in globals()
                    else pd.DataFrame()
                ),
            )

            fused_cache[key] = fused

            for thr in FINAL_MAIN_THR_GRID:
                for cd in FINAL_COOLDOWN_TUNE_GRID:
                    for mg in FINAL_MIN_GAP_TUNE_GRID:
                        for off in SHIFT_TIME_OFFSET_TUNE_GRID:
                            for time_mode in FINAL_TIME_MODE_GRID:
                                for rescue_mode, rescue_thr, rescue_min_sources in rescue_combos:
                                    pred = apply_final_event_postprocess(
                                        fused,
                                        score_col="fused_shift_score",
                                        thr=float(thr),
                                        cooldown_s=float(cd),
                                        min_gap_s=float(mg),
                                        offset_s=float(off),
                                        rescue_mode=str(rescue_mode),
                                        rescue_thr=float(rescue_thr),
                                        rescue_min_sources=int(rescue_min_sources),
                                        time_mode=str(time_mode),
                                    )

                                    m = event_metrics_against_event_table(
                                        pred,
                                        REAL_VAL_EVENTS,
                                        tol=TOL,
                                    )

                                    results.append({
                                        "score_mode": str(score_mode),
                                        "alpha_temporal": float(alpha),
                                        "thr": float(thr),
                                        "cooldown": float(cd),
                                        "min_gap": float(mg),
                                        "offset": float(off),
                                        "time_mode": str(time_mode),
                                        "rescue_mode": str(rescue_mode),
                                        "rescue_thr": float(rescue_thr),
                                        "rescue_min_sources": int(rescue_min_sources),
                                        "meets_target_precision": bool(m["P"] >= TARGET_FINAL_PRECISION),
                                        "meets_target_recall": bool(m["R"] >= TARGET_FINAL_RECALL),
                                        "meets_target_f1": bool(m["F1"] >= TARGET_FINAL_F1),
                                        "meets_all_targets": bool(
                                            (m["P"] >= TARGET_FINAL_PRECISION)
                                            and (m["R"] >= TARGET_FINAL_RECALL)
                                            and (m["F1"] >= TARGET_FINAL_F1)
                                        ),
                                        **m,
                                    })

    VAL_FUSION_DF = pd.DataFrame(results)

    assert len(VAL_FUSION_DF) > 0, "Final VAL fusion sweep produced no rows."

    VAL_FUSION_DF = _add_robust_val_metrics(
        VAL_FUSION_DF,
        fused_cache,
    )

    VAL_FUSION_DF = VAL_FUSION_DF.sort_values(
        ["meets_all_targets", "F1", "P", "R", "N_pred"],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)

    best = _select_final_val_setting(VAL_FUSION_DF)

    FINAL_LOCKED = _json_safe_final_dict(best.to_dict())
    FINAL_LOCKED["tol"] = float(TOL)
    FINAL_LOCKED["locked_on"] = "real_val_event_level"
    FINAL_LOCKED["test_not_used_for_selection"] = True
    FINAL_LOCKED["event_contract_version"] = str(EVENT_CONTRACT_VERSION)
    FINAL_LOCKED["primary_event_time_definition"] = str(PRIMARY_EVENT_TIME_DEFINITION)
    FINAL_LOCKED["primary_gap_min_s"] = float(PRIMARY_MIN_GAP_S)
    FINAL_LOCKED["primary_gap_max_s"] = float(PRIMARY_MAX_POSITIVE_GAP_S)
    FINAL_LOCKED["robust_val_folds"] = int(VAL_ROBUST_N_FOLDS)
    FINAL_LOCKED["robust_val_std_penalty"] = float(VAL_ROBUST_STD_PENALTY)
    FINAL_LOCKED["target_final_precision"] = float(TARGET_FINAL_PRECISION)
    FINAL_LOCKED["target_final_recall"] = float(TARGET_FINAL_RECALL)
    FINAL_LOCKED["target_final_f1"] = float(TARGET_FINAL_F1)
    FINAL_LOCKED["rescue_wavlm_thr"] = float(RESCUE_WAVLM_THR)
    FINAL_LOCKED["rescue_qwen_thr"] = float(RESCUE_QWEN_THR)
    FINAL_LOCKED["rescue_pyannote_change_max_dist"] = float(RESCUE_PYANNOTE_CHANGE_MAX_DIST)
    FINAL_LOCKED["rescue_asr_max_dist"] = float(RESCUE_ASR_MAX_DIST)
    FINAL_LOCKED["rescue_ecapa_max_cos"] = float(RESCUE_ECAPA_MAX_COS)
    FINAL_LOCKED["created_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    FINAL_LOCKED_PATH.write_text(json.dumps(FINAL_LOCKED, indent=2, sort_keys=True))
    FINAL_LOCKED_META_PATH.write_text(json.dumps(expected_final_meta, indent=2, sort_keys=True))

    VAL_FUSION_DF.to_csv(VAL_FUSION_TABLE_PATH, index=False)
    VAL_FUSION_TABLE_META_PATH.write_text(json.dumps(expected_final_meta, indent=2, sort_keys=True))

    print("Saved final locked settings:", FINAL_LOCKED_PATH)
    print("Saved final locked meta:", FINAL_LOCKED_META_PATH)
    print("Saved VAL fusion sweep:", VAL_FUSION_TABLE_PATH)

    display(VAL_FUSION_DF.head(20))

print("FINAL_LOCKED:")
print(json.dumps(FINAL_LOCKED, indent=2))

BASE_TRAIN_LOCKED = META_SYS2_TRAIN.copy()
BASE_TRAIN_LOCKED = BASE_TRAIN_LOCKED[
    BASE_TRAIN_LOCKED["split"].astype(str) == "train"
].copy()

FUSED_TRAIN_LOCKED = combine_scores(
    BASE_TRAIN_LOCKED,
    TEMPORAL_TRAIN_SCORES if "TEMPORAL_TRAIN_SCORES" in globals() else pd.DataFrame(),
    LGBM_TRAIN_SCORES if "LGBM_TRAIN_SCORES" in globals() else pd.DataFrame(),
    alpha=float(FINAL_LOCKED.get("alpha_temporal", 0.5)),
    score_mode=str(FINAL_LOCKED.get("score_mode", "weighted_lgbm_multi_temporal")),
    timing=(
        TIMING_CORRECTOR_TRAIN_SCORES
        if "TIMING_CORRECTOR_TRAIN_SCORES" in globals()
        else pd.DataFrame()
    ),
)

BASE_VAL_LOCKED = META_SYS2_VAL.copy()
BASE_VAL_LOCKED = BASE_VAL_LOCKED[BASE_VAL_LOCKED["split"].astype(str) == "val"].copy()

FUSED_VAL_LOCKED = combine_scores(
    BASE_VAL_LOCKED,
    TEMPORAL_VAL_SCORES if "TEMPORAL_VAL_SCORES" in globals() else pd.DataFrame(),
    LGBM_VAL_SCORES if "LGBM_VAL_SCORES" in globals() else pd.DataFrame(),
    alpha=float(FINAL_LOCKED.get("alpha_temporal", 0.5)),
    score_mode=str(FINAL_LOCKED.get("score_mode", "weighted_lgbm_multi_temporal")),
    timing=(
        TIMING_CORRECTOR_VAL_SCORES
        if "TIMING_CORRECTOR_VAL_SCORES" in globals()
        else pd.DataFrame()
    ),
)

FINAL_VAL_PRED = apply_final_event_postprocess(
    FUSED_VAL_LOCKED,
    score_col="fused_shift_score",
    thr=float(FINAL_LOCKED.get("thr", 0.5)),
    cooldown_s=float(FINAL_LOCKED.get("cooldown", 0.2)),
    min_gap_s=float(FINAL_LOCKED.get("min_gap", 0.2)),
    offset_s=float(FINAL_LOCKED.get("offset", 0.0)),
    rescue_mode=str(FINAL_LOCKED.get("rescue_mode", "none")),
    rescue_thr=float(FINAL_LOCKED.get("rescue_thr", 0.0)),
    rescue_min_sources=int(FINAL_LOCKED.get("rescue_min_sources", 0)),
    time_mode=str(FINAL_LOCKED.get("time_mode", "raw")),
)

FINAL_VAL_METRICS = event_metrics_against_event_table(
    FINAL_VAL_PRED,
    REAL_VAL_EVENTS,
    tol=TOL,
)

FINAL_VAL_PRED.to_csv(FINAL_VAL_PRED_PATH, index=False)
FINAL_VAL_METRICS_PATH.write_text(json.dumps(FINAL_VAL_METRICS, indent=2, sort_keys=True, default=str))

print("FINAL_VAL_PRED:", FINAL_VAL_PRED.shape)
print("Saved FINAL_VAL_PRED:", FINAL_VAL_PRED_PATH)
print("FINAL_VAL_METRICS:", FINAL_VAL_METRICS)
print("Saved FINAL_VAL_METRICS:", FINAL_VAL_METRICS_PATH)

if "VAL_FUSION_DF" in globals() and len(VAL_FUSION_DF):
    print("Top 10 VAL fusion settings:")
    display(VAL_FUSION_DF.head(10))

if "ranker_shift_score" in FUSED_VAL_LOCKED.columns:
    print("Locked VAL fused table includes ranker_shift_score.")
    display(
        FUSED_VAL_LOCKED[
            [
                c for c in [
                    "session",
                    "time",
                    "temporal_shift_score",
                    "lgbm_shift_score",
                    "lgbm_binary_shift_score",
                    "ranker_shift_score",
                    "rr_role_shift_score",
                    "fused_shift_score",
                    "score_mode",
                ]
                if c in FUSED_VAL_LOCKED.columns
            ]
        ].head(10)
    )
else:
    print("⚠️ ranker_shift_score not found in FUSED_VAL_LOCKED. Check Cell 19C.")

OFFICIAL_FINAL_LOCKED_20 = dict(FINAL_LOCKED)
OFFICIAL_FINAL_VAL_PRED_20 = FINAL_VAL_PRED.copy()
OFFICIAL_FINAL_VAL_METRICS_20 = dict(FINAL_VAL_METRICS)
OFFICIAL_APPLY_FINAL_EVENT_POSTPROCESS_20 = apply_final_event_postprocess
OFFICIAL_FINAL_LOCKED_PATH_20 = FINAL_LOCKED_PATH

globals()["FINAL_LOCKED"] = FINAL_LOCKED
globals()["VAL_FUSION_DF"] = VAL_FUSION_DF
globals()["FUSED_TRAIN_LOCKED"] = FUSED_TRAIN_LOCKED
globals()["FUSED_VAL_LOCKED"] = FUSED_VAL_LOCKED
globals()["FINAL_VAL_PRED"] = FINAL_VAL_PRED
globals()["FINAL_VAL_METRICS"] = FINAL_VAL_METRICS
globals()["OFFICIAL_FINAL_LOCKED_20"] = OFFICIAL_FINAL_LOCKED_20
globals()["OFFICIAL_FINAL_VAL_PRED_20"] = OFFICIAL_FINAL_VAL_PRED_20
globals()["OFFICIAL_FINAL_VAL_METRICS_20"] = OFFICIAL_FINAL_VAL_METRICS_20
globals()["OFFICIAL_APPLY_FINAL_EVENT_POSTPROCESS_20"] = (
    OFFICIAL_APPLY_FINAL_EVENT_POSTPROCESS_20
)
globals()["OFFICIAL_FINAL_LOCKED_PATH_20"] = OFFICIAL_FINAL_LOCKED_PATH_20

print(
    "Cell 20 complete: event contract, timing mode, score fusion, and "
    "post-processing locked on real VAL only; TEST remained untouched."
)

VAL fusion search:
  score modes: ['meta_confirmer', 'avg_meta_rr', 'max_meta_binary', 'avg_meta_rr_ranker', 'weighted_meta_temporal', 'rr_fap', 'avg_rr_binary', 'max_rr_binary', 'avg_rr_ranker_binary', 'weighted_rr_temporal', 'weighted_rr_ranker_temporal', 'ranker', 'avg_ranker_binary', 'max_ranker_binary', 'weighted_ranker_temporal', 'lgbm_binary', 'avg_binary_multi', 'max_binary_multi', 'weighted_binary_temporal', 'weighted_lgbm_multi_temporal', 'temporal_only']
  required ranker modes present: True
  required RR-FAP modes present: True
  required meta-confirmer modes present: True
  ranker score available: True
  RR-FAP score available: True
  meta-confirmer score available: True
  main thresholds: 17
  postprocess combos: 9
  rescue combos: 1


Saved final locked settings: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/final_locked_val_settings.json
Saved final locked meta: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/final_locked_val_settings.meta.json
Saved VAL fusion sweep: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/val_fusion_threshold_sweep.csv


,score_mode,alpha_temporal,thr,cooldown,min_gap,offset,time_mode,rescue_mode,rescue_thr,rescue_min_sources,...,FP,FN,N_pred,N_gt,cv_f1_mean,cv_f1_std,cv_f1_min,cv_p_mean,cv_r_mean,robust_score
0,weighted_binary_temporal,0.25,0.6,0.5,0.10,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
1,weighted_binary_temporal,0.25,0.6,0.5,0.20,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
2,weighted_binary_temporal,0.25,0.6,0.5,0.35,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
3,max_meta_binary,0.00,0.6,0.5,0.10,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
4,max_meta_binary,0.00,0.6,0.5,0.20,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
5,max_meta_binary,0.00,0.6,0.5,0.35,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
6,max_ranker_binary,0.00,0.5,0.5,0.10,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
7,max_ranker_binary,0.00,0.5,0.5,0.20,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
8,max_ranker_binary,0.00,0.5,0.5,0.35,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
9,lgbm_binary,0.00,0.5,0.5,0.10,0.0,local_corrected,none,0.0,0,...,231,166,532,467,0.589321,0.075955,0.495050,0.566539,0.618915,0.551344


FINAL_LOCKED:
{
  "score_mode": "weighted_binary_temporal",
  "alpha_temporal": 0.5,
  "thr": 0.55,
  "cooldown": 0.5,
  "min_gap": 0.1,
  "offset": 0.0,
  "time_mode": "local_corrected",
  "rescue_mode": "none",
  "rescue_thr": 0.0,
  "rescue_min_sources": 0,
  "meets_target_precision": 0,
  "meets_target_recall": 0,
  "meets_target_f1": 0,
  "meets_all_targets": 0,
  "P": 0.5850202429149798,
  "R": 0.6188436830835118,
  "F1": 0.6014568158168575,
  "TP": 289,
  "FP": 205,
  "FN": 178,
  "N_pred": 494,
  "N_gt": 467,
  "cv_f1_mean": 0.6052977535929185,
  "cv_f1_std": 0.04349758564777433,
  "cv_f1_min": 0.5436893203883495,
  "cv_p_mean": 0.5968893638716761,
  "cv_r_mean": 0.6216846012366947,
  "robust_score": 0.5835489607690313,
  "selection_reason": "no setting met P/R floors; selected robust VAL setting; ranked by session-fold mean F1 minus variability penalty",
  "tol": 0.5,
  "locked_on": "real_val_event_level",
  "test_not_used_for_selection": true,
  "event_contract_version": "vox

FINAL_VAL_PRED: (494, 10)
Saved FINAL_VAL_PRED: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/final_val_predictions_locked.csv
FINAL_VAL_METRICS: {'P': 0.5850202429149798, 'R': 0.6188436830835118, 'F1': 0.6014568158168575, 'TP': 289, 'FP': 205, 'FN': 178, 'N_pred': 494, 'N_gt': 467}
Saved FINAL_VAL_METRICS: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/final_val_metrics_locked.json
Top 10 VAL fusion settings:


,score_mode,alpha_temporal,thr,cooldown,min_gap,offset,time_mode,rescue_mode,rescue_thr,rescue_min_sources,...,FP,FN,N_pred,N_gt,cv_f1_mean,cv_f1_std,cv_f1_min,cv_p_mean,cv_r_mean,robust_score
0,weighted_binary_temporal,0.25,0.6,0.5,0.10,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
1,weighted_binary_temporal,0.25,0.6,0.5,0.20,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
2,weighted_binary_temporal,0.25,0.6,0.5,0.35,0.0,local_corrected,none,0.0,0,...,174,189,452,467,0.598965,0.069706,0.500000,0.624208,0.580935,0.564112
3,max_meta_binary,0.00,0.6,0.5,0.10,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
4,max_meta_binary,0.00,0.6,0.5,0.20,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
5,max_meta_binary,0.00,0.6,0.5,0.35,0.0,local_corrected,none,0.0,0,...,200,179,488,467,0.589845,0.075426,0.484848,0.590729,0.593067,0.552132
6,max_ranker_binary,0.00,0.5,0.5,0.10,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
7,max_ranker_binary,0.00,0.5,0.5,0.20,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
8,max_ranker_binary,0.00,0.5,0.5,0.35,0.0,local_corrected,none,0.0,0,...,233,165,535,467,0.590257,0.078408,0.495050,0.566060,0.621446,0.551053
9,lgbm_binary,0.00,0.5,0.5,0.10,0.0,local_corrected,none,0.0,0,...,231,166,532,467,0.589321,0.075955,0.495050,0.566539,0.618915,0.551344


Locked VAL fused table includes ranker_shift_score.


,session,time,temporal_shift_score,lgbm_shift_score,lgbm_binary_shift_score,ranker_shift_score,rr_role_shift_score,fused_shift_score,score_mode
0,ahnss,0.090504,0.087005,0.002104,0.001628,0.265338,0.271204,0.044316,weighted_binary_temporal
1,ahnss,1.331692,0.087005,0.001056,0.000416,0.265338,0.271204,0.043711,weighted_binary_temporal
2,ahnss,9.000000,0.087005,0.000010,0.000015,0.265338,0.271204,0.043510,weighted_binary_temporal
3,ahnss,17.429841,0.342473,0.000307,0.000530,0.265338,0.271204,0.171501,weighted_binary_temporal
4,ahnss,18.300001,0.342473,0.040051,0.019294,0.265338,0.271204,0.180884,weighted_binary_temporal
5,ahnss,19.059947,0.342473,0.278246,0.366779,0.272367,0.278824,0.354626,weighted_binary_temporal
6,ahnss,20.049895,0.342473,0.053296,0.037005,0.265338,0.271204,0.189739,weighted_binary_temporal
7,ahnss,20.600000,0.342473,0.015922,0.012488,0.265338,0.271204,0.177481,weighted_binary_temporal
8,ahnss,21.423317,0.007670,0.000931,0.000476,0.265338,0.271204,0.004073,weighted_binary_temporal
9,ahnss,26.360000,0.000675,0.000007,0.000018,0.265338,0.271204,0.000346,weighted_binary_temporal


Cell 20 complete: event contract, timing mode, score fusion, and post-processing locked on real VAL only; TEST remained untouched.


In [32]:
# Generate TEST predictions using locked VAL settings
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Dict, Any
from IPython.display import display

assert "PRED_DIR" in globals(), "PRED_DIR is missing. Run earlier setup/output-dir cells first."
assert "RUN_VERSION" in globals(), "Need RUN_VERSION."
assert "SEED" in globals(), "Need SEED."
assert "TOL" in globals(), "Need TOL."
assert "FINAL_LOCKED" in globals(), "Need FINAL_LOCKED from Cell 20."
assert "combine_scores" in globals(), "Need combine_scores(...) from Cell 20."
assert "apply_final_event_postprocess" in globals(), "Need apply_final_event_postprocess(...) from Cell 20."

PRED_DIR.mkdir(parents=True, exist_ok=True)

CELL21_VARIANT = "cell21_v10_paper_ready_contract_timing_prediction"

USE_TEST_INFORMED_EXPLORATORY_BRANCH = bool(
    globals().get(
        "USE_TEST_INFORMED_EXPLORATORY_BRANCH",
        False,
    )
)
if USE_TEST_INFORMED_EXPLORATORY_BRANCH:
    raise RuntimeError(
        "The paper-ready notebook forbids a TEST-informed exploratory branch "
        "from generating official predictions."
    )

FINAL_LOCKED = dict(
    globals().get(
        "OFFICIAL_FINAL_LOCKED_20",
        FINAL_LOCKED,
    )
)
apply_final_event_postprocess = globals().get(
    "OFFICIAL_APPLY_FINAL_EVENT_POSTPROCESS_20",
    apply_final_event_postprocess,
)

ACTIVE_LOCK_STAGE_21 = str(
    FINAL_LOCKED.get(
        "stage",
        FINAL_LOCKED.get(
            "locked_on",
            "cell20_official",
        ),
    )
)
ACTIVE_LOCK_PROVENANCE_21 = "val_selected_official"

TEST_PRED_CSV = PRED_DIR / f"test_predictions_{ACTIVE_LOCK_PROVENANCE_21}_{ACTIVE_LOCK_STAGE_21}.csv"
TEST_PRED_PKL = PRED_DIR / f"test_predictions_{ACTIVE_LOCK_PROVENANCE_21}_{ACTIVE_LOCK_STAGE_21}.pkl"
TEST_FUSED_SCORES_PATH = PRED_DIR / f"test_fused_candidate_scores_{ACTIVE_LOCK_PROVENANCE_21}_{ACTIVE_LOCK_STAGE_21}.pkl"
TEST_PRED_META_PATH = PRED_DIR / f"test_predictions_{ACTIVE_LOCK_PROVENANCE_21}_{ACTIVE_LOCK_STAGE_21}.meta.json"

PREDICTION_BRANCH_NAME = f"{ACTIVE_LOCK_PROVENANCE_21}:{ACTIVE_LOCK_STAGE_21}:{FINAL_LOCKED.get('score_mode', 'unknown')}"
SYS2_CHANGE_MODE_NAME = str(FINAL_LOCKED.get("score_mode", "unknown"))

FORCE_REBUILD_TEST_PRED = bool(globals().get("FORCE_REBUILD_TEST_PRED", False))

def _cell21_file_fingerprint(path: Path) -> Dict[str, Any]:
    path = Path(path)
    if not path.exists():
        return {"exists": False, "path": str(path), "size": None, "mtime": None}
    st = path.stat()
    return {"exists": True, "path": str(path), "size": int(st.st_size), "mtime": float(st.st_mtime)}

def _json_safe_for_meta(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return float(obj) if np.isfinite(obj) else None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return v if np.isfinite(v) else None
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): _json_safe_for_meta(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_json_safe_for_meta(v) for v in obj]
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    try:
        return str(obj)
    except Exception:
        return "<unserializable>"

def _locked_uses_ranker() -> bool:
    return "ranker" in str(FINAL_LOCKED.get("score_mode", "")).lower()

def _ranker_available_in_lgbm_test_scores() -> bool:
    return (
        "LGBM_TEST_SCORES" in globals()
        and isinstance(LGBM_TEST_SCORES, pd.DataFrame)
        and len(LGBM_TEST_SCORES)
        and "ranker_shift_score" in LGBM_TEST_SCORES.columns
    )

def _locked_uses_rr_fap() -> bool:
    mode = str(FINAL_LOCKED.get("score_mode", "")).lower()
    return bool("rr" in mode or "fap" in mode)

def _rr_fap_available_in_lgbm_test_scores() -> bool:
    return (
        "LGBM_TEST_SCORES" in globals()
        and isinstance(LGBM_TEST_SCORES, pd.DataFrame)
        and len(LGBM_TEST_SCORES)
        and "rr_role_shift_score" in LGBM_TEST_SCORES.columns
    )

def _locked_uses_meta_confirmer() -> bool:
    return "meta" in str(FINAL_LOCKED.get("score_mode", "")).lower()

def _meta_confirmer_available_in_lgbm_test_scores() -> bool:
    return (
        "LGBM_TEST_SCORES" in globals()
        and isinstance(LGBM_TEST_SCORES, pd.DataFrame)
        and len(LGBM_TEST_SCORES)
        and "meta_confirmer_shift_score" in LGBM_TEST_SCORES.columns
    )

def _test_prediction_config() -> Dict[str, Any]:
    return {
        "run_version": str(RUN_VERSION),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
        "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
        "primary_gap_min_s": float(PRIMARY_MIN_GAP_S),
        "primary_gap_max_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
        "seed": int(SEED),
        "cell21_variant": CELL21_VARIANT,
        "active_lock_stage": ACTIVE_LOCK_STAGE_21,
        "active_lock_provenance": ACTIVE_LOCK_PROVENANCE_21,
        "uses_test_informed_exploratory_branch": bool(USE_TEST_INFORMED_EXPLORATORY_BRANCH),
        "final_locked": _json_safe_for_meta(FINAL_LOCKED if "FINAL_LOCKED" in globals() else {}),
        "locked_score_mode": str(FINAL_LOCKED.get("score_mode", "")),
        "locked_uses_ranker": bool(_locked_uses_ranker()),
        "locked_uses_rr_fap": bool(_locked_uses_rr_fap()),
        "locked_uses_meta_confirmer": bool(_locked_uses_meta_confirmer()),
        "ranker_available_in_lgbm_test_scores": bool(_ranker_available_in_lgbm_test_scores()),
        "rr_fap_available_in_lgbm_test_scores": bool(_rr_fap_available_in_lgbm_test_scores()),
        "meta_confirmer_available_in_lgbm_test_scores": bool(_meta_confirmer_available_in_lgbm_test_scores()),
        "active_prediction_branch": PREDICTION_BRANCH_NAME,
        "sys2_change_mode": SYS2_CHANGE_MODE_NAME,
        "tol": float(TOL),

        "temporal_test_scores_fingerprint": (
            _cell21_file_fingerprint(TEMPORAL_TEST_SCORES_PATH)
            if "TEMPORAL_TEST_SCORES_PATH" in globals()
            else None
        ),
        "lgbm_test_scores_fingerprint": (
            _cell21_file_fingerprint(LGBM_TEST_SCORES_PATH)
            if "LGBM_TEST_SCORES_PATH" in globals()
            else None
        ),
        "lgbm_binary_test_scores_fingerprint": (
            _cell21_file_fingerprint(LGBM_BINARY_TEST_SCORES_PATH)
            if "LGBM_BINARY_TEST_SCORES_PATH" in globals()
            else None
        ),
        "ranker_test_scores_fingerprint": (
            _cell21_file_fingerprint(RANKER_TEST_SCORES_PATH)
            if "RANKER_TEST_SCORES_PATH" in globals()
            else None
        ),
        "rr_fap_test_scores_fingerprint": (
            _cell21_file_fingerprint(RR_FAP_TEST_SCORES_PATH)
            if "RR_FAP_TEST_SCORES_PATH" in globals()
            else None
        ),
        "meta_confirmer_test_scores_fingerprint": (
            _cell21_file_fingerprint(META_CONFIRMER_TEST_SCORES_PATH)
            if "META_CONFIRMER_TEST_SCORES_PATH" in globals()
            else None
        ),
        "timing_corrector_test_scores_fingerprint": (
            _cell21_file_fingerprint(TIMING_CORRECTOR_TEST_SCORES_PATH)
            if "TIMING_CORRECTOR_TEST_SCORES_PATH" in globals()
            else None
        ),
        "locked_time_mode": str(FINAL_LOCKED.get("time_mode", "raw")),
        "timing_corrector_variant": str(
            globals().get("TIMING_CORRECTOR_VARIANT", "")
        ),
        "has_event_ranker_score": bool(globals().get("LGBM_HAS_EVENT_RANKER_SCORE", False)),
        "has_rr_fap_score": bool(globals().get("LGBM_HAS_RR_FAP_SCORE", False)),
        "has_meta_confirmer_score": bool(globals().get("LGBM_HAS_META_CONFIRMER_SCORE", False)),

        "n_meta_sys2_test": int(len(META_SYS2_TEST)) if "META_SYS2_TEST" in globals() else -1,
        "n_temporal_test_scores": int(len(TEMPORAL_TEST_SCORES)) if "TEMPORAL_TEST_SCORES" in globals() else 0,
        "n_lgbm_test_scores": int(len(LGBM_TEST_SCORES)) if "LGBM_TEST_SCORES" in globals() else 0,
        "n_ranker_test_scores": int(len(RANKER_TEST_SCORES)) if "RANKER_TEST_SCORES" in globals() else 0,
        "n_rr_fap_test_scores": int(len(RR_FAP_TEST_SCORES)) if "RR_FAP_TEST_SCORES" in globals() else 0,
        "n_meta_confirmer_test_scores": int(len(META_CONFIRMER_TEST_SCORES)) if "META_CONFIRMER_TEST_SCORES" in globals() else 0,

        "test_labels_used_for_generation": False,
        "test_labels_attached_in_meta": False,
        "gt_columns_removed_before_generation": True,

        "csv_path": str(TEST_PRED_CSV),
        "pkl_path": str(TEST_PRED_PKL),
        "fused_scores_path": str(TEST_FUSED_SCORES_PATH),
        "created_by_cell": "CELL 21",
        "cache_format": "csv+pkl",
    }

def _test_pred_meta_matches(path: Path, expected: Dict[str, Any]) -> bool:
    if not Path(path).exists():
        return False
    try:
        old = json.loads(Path(path).read_text())
        return old == expected
    except Exception:
        return False

def _ensure_prediction_schema(df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "session",
        "time",
        "score",
        "temporal_shift_score",
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "meta_confirmer_shift_score",
        "fused_shift_score",
        "alpha_temporal",
        "thr",
        "cooldown",
        "min_gap",
        "offset",
        "score_mode",
        "rescue_mode",
        "rescue_thr",
        "rescue_min_sources",
        "source_agree_count",
        "accept_reason",
        "prediction_branch",
        "sys2_change_mode",
    ]

    if df is None or len(df) == 0:
        return pd.DataFrame(columns=cols)

    out = df.copy()

    for c in cols:
        if c not in out.columns:
            if c in ["session", "score_mode", "rescue_mode", "accept_reason", "prediction_branch", "sys2_change_mode"]:
                out[c] = ""
            else:
                out[c] = 0.0

    out["session"] = out["session"].astype(str)
    out["time"] = pd.to_numeric(out["time"], errors="coerce")
    out["score"] = pd.to_numeric(out["score"], errors="coerce")

    out = out.dropna(subset=["session", "time", "score"])
    out = out[out["time"] >= 0].reset_index(drop=True)

    for c in [
        "temporal_shift_score",
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "meta_confirmer_shift_score",
        "fused_shift_score",
        "alpha_temporal",
        "thr",
        "cooldown",
        "min_gap",
        "offset",
        "rescue_thr",
        "rescue_min_sources",
        "source_agree_count",
    ]:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    out["score_mode"] = out["score_mode"].astype(str)
    out["rescue_mode"] = out["rescue_mode"].astype(str)
    out["accept_reason"] = out["accept_reason"].astype(str)
    out["prediction_branch"] = out["prediction_branch"].astype(str)
    out["sys2_change_mode"] = out["sys2_change_mode"].astype(str)

    return out[cols].sort_values(["session", "time"]).reset_index(drop=True)

def _nearest_value_for_prediction(
    df: pd.DataFrame,
    session: str,
    tau: float,
    value_col: str,
    default=0.0,
    max_dist: float = 0.50,
):
    if df is None or len(df) == 0 or value_col not in df.columns:
        return default

    g = df[df["session"].astype(str) == str(session)].copy()
    if len(g) == 0:
        return default

    g["time"] = pd.to_numeric(g["time"], errors="coerce")
    g = g.dropna(subset=["time"])

    if len(g) == 0:
        return default

    arr = g["time"].to_numpy(dtype=float)
    d = np.abs(arr - float(tau))

    if len(d) == 0 or not np.isfinite(d).any():
        return default

    j = int(np.nanargmin(d))

    if not np.isfinite(d[j]) or d[j] > float(max_dist):
        return default

    try:
        v = g.iloc[j][value_col]
        if isinstance(default, str):
            return str(v)
        v = float(v)
        return v if np.isfinite(v) else default
    except Exception:
        return default

def _add_fused_scores_to_predictions(pred_df: pd.DataFrame, fused_df: pd.DataFrame) -> pd.DataFrame:
    if pred_df is None or len(pred_df) == 0:
        return _ensure_prediction_schema(pred_df)

    pred = pred_df.copy()

    max_lookup_dist = max(
        0.10,
        abs(float(FINAL_LOCKED.get("offset", 0.0))) + float(FINAL_LOCKED.get("min_gap", 0.2)),
        abs(float(FINAL_LOCKED.get("offset", 0.0))) + float(FINAL_LOCKED.get("cooldown", 0.2)),
    )

    temporal_scores = []
    lgbm_scores = []
    binary_scores = []
    ranker_scores = []
    rr_fap_scores = []
    meta_confirmer_scores = []
    fused_scores = []
    agree_counts = []
    accept_reasons = []

    for _, r in pred.iterrows():
        sid = str(r.session)
        tau = float(r.time)
        base_score = float(r.get("score", 0.0))

        temporal_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "temporal_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        lgbm_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "lgbm_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        binary_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "lgbm_binary_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        ranker_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "ranker_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        rr_fap_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "rr_role_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        meta_confirmer_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "meta_confirmer_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        fused_scores.append(
            _nearest_value_for_prediction(
                fused_df,
                sid,
                tau,
                "fused_shift_score",
                base_score,
                max_lookup_dist,
            )
        )

        agree_counts.append(int(float(r.get("source_agree_count", 0))))
        accept_reasons.append(str(r.get("accept_reason", "main")))

    pred["temporal_shift_score"] = temporal_scores
    pred["lgbm_shift_score"] = lgbm_scores
    pred["lgbm_binary_shift_score"] = binary_scores
    pred["ranker_shift_score"] = ranker_scores
    pred["rr_role_shift_score"] = rr_fap_scores
    pred["meta_confirmer_shift_score"] = meta_confirmer_scores
    pred["fused_shift_score"] = fused_scores

    pred["alpha_temporal"] = float(FINAL_LOCKED.get("alpha_temporal", 0.5))
    pred["thr"] = float(FINAL_LOCKED.get("thr", 0.5))
    pred["cooldown"] = float(FINAL_LOCKED.get("cooldown", 0.2))
    pred["min_gap"] = float(FINAL_LOCKED.get("min_gap", 0.2))
    pred["offset"] = float(FINAL_LOCKED.get("offset", 0.0))
    pred["score_mode"] = str(FINAL_LOCKED.get("score_mode", "weighted_lgbm_multi_temporal"))
    pred["rescue_mode"] = str(FINAL_LOCKED.get("rescue_mode", "none"))
    pred["rescue_thr"] = float(FINAL_LOCKED.get("rescue_thr", 0.0))
    pred["rescue_min_sources"] = int(FINAL_LOCKED.get("rescue_min_sources", 0))
    pred["source_agree_count"] = agree_counts
    pred["accept_reason"] = accept_reasons
    pred["prediction_branch"] = PREDICTION_BRANCH_NAME
    pred["sys2_change_mode"] = SYS2_CHANGE_MODE_NAME

    return _ensure_prediction_schema(pred)

def _test_labels_are_safe(base_test: pd.DataFrame) -> bool:
    if base_test is None or len(base_test) == 0 or "label" not in base_test.columns:
        return True

    labels = (
        pd.to_numeric(base_test["label"], errors="coerce")
        .fillna(-1)
        .astype(int)
        .unique()
        .tolist()
    )

    return set(labels).issubset({-1})

assert str(
    FINAL_LOCKED.get(
        "event_contract_version",
        EVENT_CONTRACT_VERSION,
    )
) == str(EVENT_CONTRACT_VERSION), (
    "FINAL_LOCKED was created under a different event contract."
)
assert bool(
    FINAL_LOCKED.get(
        "test_not_used_for_selection",
        False,
    )
), "Official TEST generation requires a VAL-only lock."

expected_test_pred_meta = _test_prediction_config()

if (
    TEST_PRED_CSV.exists()
    and TEST_PRED_PKL.exists()
    and _test_pred_meta_matches(TEST_PRED_META_PATH, expected_test_pred_meta)
    and not FORCE_REBUILD_TEST_PRED
):
    try:
        TEST_PRED_DF = pd.read_pickle(TEST_PRED_PKL)
        print("Loaded TEST predictions from pickle:", TEST_PRED_PKL)
    except Exception:
        TEST_PRED_DF = pd.read_csv(TEST_PRED_CSV, keep_default_na=False)
        print("Loaded TEST predictions from CSV:", TEST_PRED_CSV)

    TEST_PRED_DF = _ensure_prediction_schema(TEST_PRED_DF)

    if TEST_FUSED_SCORES_PATH.exists():
        try:
            TEST_FUSED_SCORES_DF = pd.read_pickle(TEST_FUSED_SCORES_PATH)
            print("Loaded TEST fused candidate scores:", TEST_FUSED_SCORES_PATH)
        except Exception:
            TEST_FUSED_SCORES_DF = pd.DataFrame()
    else:
        TEST_FUSED_SCORES_DF = pd.DataFrame()

else:
    assert "META_SYS2_TEST" in globals(), "Missing META_SYS2_TEST. Run Cell 17 first."

    base_test = META_SYS2_TEST.copy()

    gt_like_cols = [c for c in base_test.columns if c.lower() in {
        "label", "target", "y", "action_id", "gt_label", "gt_action",
        "gt_time", "gt_next_speaker", "next_speaker", "prev_speaker"
    } or c.lower().startswith("gt_")]
    base_test = base_test.drop(columns=gt_like_cols, errors="ignore")

    assert len(base_test) > 0, "META_SYS2_TEST is empty. Run Cells 16–17 first."

    temporal_test_for_pred = (
        TEMPORAL_TEST_SCORES
        if "TEMPORAL_TEST_SCORES" in globals() and TEMPORAL_TEST_SCORES is not None and len(TEMPORAL_TEST_SCORES)
        else pd.DataFrame()
    )

    lgbm_test_for_pred = (
        LGBM_TEST_SCORES
        if "LGBM_TEST_SCORES" in globals() and LGBM_TEST_SCORES is not None and len(LGBM_TEST_SCORES)
        else pd.DataFrame()
    )

    ranker_test_available = (
        "ranker_shift_score" in lgbm_test_for_pred.columns
        if isinstance(lgbm_test_for_pred, pd.DataFrame) and len(lgbm_test_for_pred)
        else False
    )

    rr_fap_test_available = (
        "rr_role_shift_score" in lgbm_test_for_pred.columns
        if isinstance(lgbm_test_for_pred, pd.DataFrame) and len(lgbm_test_for_pred)
        else False
    )

    meta_confirmer_test_available = (
        "meta_confirmer_shift_score" in lgbm_test_for_pred.columns
        if isinstance(lgbm_test_for_pred, pd.DataFrame) and len(lgbm_test_for_pred)
        else False
    )

    assert len(temporal_test_for_pred) or len(lgbm_test_for_pred), (
        "No TEST verifier scores available. Run Cell 18 and/or Cell 19/19B/19C first."
    )

    if _locked_uses_ranker():
        assert bool(ranker_test_available), (
            "FINAL_LOCKED selected a ranker score_mode, but ranker_shift_score is not available "
            "inside LGBM_TEST_SCORES. Rerun Cell 19C before Cell 21."
        )

    if _locked_uses_rr_fap():
        assert bool(rr_fap_test_available), (
            "FINAL_LOCKED selected an RR-FAP score_mode, but rr_role_shift_score is not available "
            "inside LGBM_TEST_SCORES. Rerun Cell 19D before Cell 21."
        )

    if _locked_uses_meta_confirmer():
        assert bool(meta_confirmer_test_available), (
            "FINAL_LOCKED selected a meta-confirmer score_mode, but meta_confirmer_shift_score is not available "
            "inside LGBM_TEST_SCORES. Rerun Cell 19E before Cell 21."
        )

    print("TEST verifier availability:")
    print("  temporal_test_for_pred:", temporal_test_for_pred.shape)
    print("  lgbm_test_for_pred:", lgbm_test_for_pred.shape)
    print("  ranker_shift_score available in LGBM_TEST_SCORES:", bool(ranker_test_available))
    print("  rr_role_shift_score available in LGBM_TEST_SCORES:", bool(rr_fap_test_available))
    print("  meta_confirmer_shift_score available in LGBM_TEST_SCORES:", bool(meta_confirmer_test_available))
    print("  locked score_mode:", str(FINAL_LOCKED.get("score_mode", "")))
    print("  locked uses ranker:", bool(_locked_uses_ranker()))
    print("  locked uses RR-FAP:", bool(_locked_uses_rr_fap()))
    print("  locked uses meta-confirmer:", bool(_locked_uses_meta_confirmer()))

    fused_test = combine_scores(
        base_test,
        temporal_test_for_pred,
        lgbm_test_for_pred,
        alpha=float(FINAL_LOCKED.get("alpha_temporal", 0.5)),
        score_mode=str(FINAL_LOCKED.get("score_mode", "weighted_lgbm_multi_temporal")),
        timing=(
            TIMING_CORRECTOR_TEST_SCORES
            if "TIMING_CORRECTOR_TEST_SCORES" in globals()
            else pd.DataFrame()
        ),
    )

    for c in [
        "temporal_shift_score",
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "meta_confirmer_shift_score",
        "predicted_offset_s",
        "corrected_time",
        "fused_shift_score",
    ]:
        if c not in fused_test.columns:
            fused_test[c] = 0.0
        fused_test[c] = pd.to_numeric(
            fused_test[c],
            errors="coerce",
        ).fillna(0.0)

    if "predicted_offset_s" not in fused_test.columns:
        fused_test["predicted_offset_s"] = 0.0
    if "corrected_time" not in fused_test.columns:
        fused_test["corrected_time"] = fused_test["time"]
    else:
        invalid_corrected = (
            ~np.isfinite(
                pd.to_numeric(
                    fused_test["corrected_time"],
                    errors="coerce",
                )
            )
        )
        fused_test.loc[
            invalid_corrected,
            "corrected_time",
        ] = (
            fused_test.loc[
                invalid_corrected,
                "time",
            ]
            + fused_test.loc[
                invalid_corrected,
                "predicted_offset_s",
            ]
        )

    fused_test.to_pickle(TEST_FUSED_SCORES_PATH)
    TEST_FUSED_SCORES_DF = fused_test.copy()

    raw_pred = apply_final_event_postprocess(
        fused_test,
        score_col="fused_shift_score",
        thr=float(FINAL_LOCKED.get("thr", 0.5)),
        cooldown_s=float(FINAL_LOCKED.get("cooldown", 0.2)),
        min_gap_s=float(FINAL_LOCKED.get("min_gap", 0.2)),
        offset_s=float(FINAL_LOCKED.get("offset", 0.0)),
        rescue_mode=str(FINAL_LOCKED.get("rescue_mode", "none")),
        rescue_thr=float(FINAL_LOCKED.get("rescue_thr", 0.0)),
        rescue_min_sources=int(FINAL_LOCKED.get("rescue_min_sources", 0)),
        time_mode=str(FINAL_LOCKED.get("time_mode", "raw")),
    )

    TEST_PRED_DF = _add_fused_scores_to_predictions(raw_pred, fused_test)

    TEST_PRED_DF.to_csv(TEST_PRED_CSV, index=False)
    TEST_PRED_DF.to_pickle(TEST_PRED_PKL)
    TEST_PRED_META_PATH.write_text(
        json.dumps(
            _json_safe_for_meta(expected_test_pred_meta),
            indent=2,
            sort_keys=True,
        )
    )

    print("Saved TEST predictions:", TEST_PRED_CSV)
    print("Saved TEST prediction pickle:", TEST_PRED_PKL)
    print("Saved TEST fused candidate scores:", TEST_FUSED_SCORES_PATH)
    print("Saved TEST prediction meta:", TEST_PRED_META_PATH)

PRED_CSV = TEST_PRED_CSV
PRED_DF = TEST_PRED_DF.copy()
TEST_PREDICTIONS_DF = TEST_PRED_DF.copy()

ACTIVE_PREDICTION_BRANCH = PREDICTION_BRANCH_NAME
SYS2_CHANGE_MODE = SYS2_CHANGE_MODE_NAME

print("ACTIVE_PREDICTION_BRANCH:", ACTIVE_PREDICTION_BRANCH)
print("SYS2_CHANGE_MODE:", SYS2_CHANGE_MODE)
print("PRED_CSV:", PRED_CSV)

display(TEST_PRED_DF.head(20))

print(
    "Predicted TEST events:",
    len(TEST_PRED_DF),
    "sessions:",
    TEST_PRED_DF["session"].nunique() if len(TEST_PRED_DF) else 0,
)

if len(TEST_PRED_DF):
    print("Prediction score summary:")
    display(TEST_PRED_DF["score"].describe().to_frame("score"))

    print("Fused score summary:")
    display(TEST_PRED_DF["fused_shift_score"].describe().to_frame("fused_shift_score"))

    if "ranker_shift_score" in TEST_PRED_DF.columns:
        print("Ranker score summary:")
        display(TEST_PRED_DF["ranker_shift_score"].describe().to_frame("ranker_shift_score"))

    if "rr_role_shift_score" in TEST_PRED_DF.columns:
        print("RR-FAP role-relative score summary:")
        display(TEST_PRED_DF["rr_role_shift_score"].describe().to_frame("rr_role_shift_score"))

    if "meta_confirmer_shift_score" in TEST_PRED_DF.columns:
        print("Meta-confirmer score summary:")
        display(TEST_PRED_DF["meta_confirmer_shift_score"].describe().to_frame("meta_confirmer_shift_score"))

    if "accept_reason" in TEST_PRED_DF.columns:
        print("Prediction accept reasons:")
        display(TEST_PRED_DF["accept_reason"].value_counts(dropna=False).to_frame("n"))

    print("Predicted events per session summary:")
    per_sess = TEST_PRED_DF.groupby("session").size().describe().to_frame("events_per_session")
    display(per_sess)
else:
    print("⚠️ No TEST events predicted. Cell 22 evaluation will report zero TP and all FN if GT exists.")

if "TEST_FUSED_SCORES_DF" in globals() and len(TEST_FUSED_SCORES_DF):
    print("TEST fused candidate score table:", TEST_FUSED_SCORES_DF.shape)
    show_cols = [
        "session",
        "time",
        "temporal_shift_score",
        "lgbm_shift_score",
        "lgbm_binary_shift_score",
        "ranker_shift_score",
        "rr_role_shift_score",
        "meta_confirmer_shift_score",
        "fused_shift_score",
        "score_mode",
    ]
    show_cols = [c for c in show_cols if c in TEST_FUSED_SCORES_DF.columns]
    display(TEST_FUSED_SCORES_DF[show_cols].head())

    if "ranker_shift_score" in TEST_FUSED_SCORES_DF.columns:
        print("✅ TEST_FUSED_SCORES_DF includes ranker_shift_score.")
    else:
        print("⚠️ TEST_FUSED_SCORES_DF does not include ranker_shift_score.")

    if "rr_role_shift_score" in TEST_FUSED_SCORES_DF.columns:
        print("✅ TEST_FUSED_SCORES_DF includes rr_role_shift_score.")
    else:
        print("⚠️ TEST_FUSED_SCORES_DF does not include rr_role_shift_score.")

    if "meta_confirmer_shift_score" in TEST_FUSED_SCORES_DF.columns:
        print("✅ TEST_FUSED_SCORES_DF includes meta_confirmer_shift_score.")
    else:
        print("⚠️ TEST_FUSED_SCORES_DF does not include meta_confirmer_shift_score.")

print("✅ Cell 21 complete: provenance-safe TEST predictions generated/loaded.")

TEST verifier availability:
  temporal_test_for_pred: (99307, 17)
  lgbm_test_for_pred: (99307, 23)
  ranker_shift_score available in LGBM_TEST_SCORES: True
  rr_role_shift_score available in LGBM_TEST_SCORES: True
  meta_confirmer_shift_score available in LGBM_TEST_SCORES: True
  locked score_mode: weighted_binary_temporal
  locked uses ranker: False
  locked uses RR-FAP: False
  locked uses meta-confirmer: False


Saved TEST predictions: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/predictions/test_predictions_val_selected_official_real_val_event_level.csv
Saved TEST prediction pickle: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/predictions/test_predictions_val_selected_official_real_val_event_level.pkl
Saved TEST fused candidate scores: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/predictions/test_fused_candidate_scores_val_selected_official_real_val_event_level.pkl
Saved TEST prediction meta: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/predictions/test_predictions_val_selected_official_real_val_event_level.meta.json
ACTIVE_PRE

,session,time,score,temporal_shift_score,lgbm_shift_score,lgbm_binary_shift_score,ranker_shift_score,rr_role_shift_score,meta_confirmer_shift_score,fused_shift_score,...,min_gap,offset,score_mode,rescue_mode,rescue_thr,rescue_min_sources,source_agree_count,accept_reason,prediction_branch,sys2_change_mode
0,aepyx,6.104183,0.767635,0.553353,0.975232,0.981918,0.774437,0.754180,0.962028,0.767635,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
1,aepyx,11.458788,0.738103,0.692508,0.745625,0.783697,0.600980,0.604149,0.345587,0.738103,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
2,aepyx,15.046393,0.789490,0.598470,0.975485,0.980510,0.777034,0.759266,0.962028,0.789490,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
3,aepyx,27.548680,0.633748,0.628734,0.771464,0.638763,0.556575,0.553647,0.375917,0.633748,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
4,aepyx,32.741810,0.847805,0.712639,0.951774,0.930720,0.588982,0.466783,0.959903,0.821680,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
5,aepyx,63.832901,0.581607,0.472913,0.747786,0.690300,0.580023,0.605880,0.411166,0.581607,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
6,aepyx,66.016976,0.739067,0.668548,0.350900,0.474998,0.315543,0.283621,0.077030,0.571773,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
7,aepyx,66.629301,0.642438,0.668548,0.865956,0.809587,0.735974,0.748556,0.922447,0.739067,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
8,aepyx,75.141129,0.819856,0.656565,0.830979,0.795776,0.518849,0.411861,0.818345,0.726171,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal
9,aepyx,79.046739,0.854581,0.715354,0.996900,0.993808,0.777034,0.759266,0.962174,0.854581,...,0.1,0.0,weighted_binary_temporal,none,0.0,0,0,main,val_selected_official:real_val_event_level:wei...,weighted_binary_temporal


Predicted TEST events: 6493 sessions: 205
Prediction score summary:


,score
count,6493.000000
mean,0.728163
std,0.107029
min,0.550038
25%,0.635863
50%,0.724631
75%,0.816250
max,0.966866


...

Predicted events per session summary:


,events_per_session
count,205.000000
mean,31.673171
std,29.672816
min,1.000000
25%,10.000000
50%,24.000000
75%,42.000000
max,147.000000


TEST fused candidate score table: (99307, 40)


,session,time,temporal_shift_score,lgbm_shift_score,lgbm_binary_shift_score,ranker_shift_score,rr_role_shift_score,meta_confirmer_shift_score,fused_shift_score,score_mode
0,aepyx,0.000000,0.553353,0.000339,0.001629,0.265338,0.271204,0.040899,0.277491,weighted_binary_temporal
1,aepyx,0.646941,0.553353,0.096875,0.044752,0.265338,0.271204,0.040899,0.299053,weighted_binary_temporal
2,aepyx,1.507531,0.553353,0.004663,0.012813,0.265338,0.271204,0.040899,0.283083,weighted_binary_temporal
3,aepyx,2.600000,0.553353,0.649730,0.534602,0.362589,0.360094,0.221538,0.543977,weighted_binary_temporal
4,aepyx,3.643190,0.553353,0.001493,0.001764,0.265338,0.271204,0.040899,0.277559,weighted_binary_temporal


✅ TEST_FUSED_SCORES_DF includes ranker_shift_score.
✅ TEST_FUSED_SCORES_DF includes rr_role_shift_score.
✅ TEST_FUSED_SCORES_DF includes meta_confirmer_shift_score.
✅ Cell 21 complete: provenance-safe TEST predictions generated/loaded.


In [33]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

assert "EVAL_DIR" in globals(), "Need EVAL_DIR."
assert "REAL_TEST_EVENTS" in globals(), (
    "Need paper-ready TEST events from Cell 4."
)
assert "REAL_TEST_EVENTS_LEGACY" in globals(), (
    "Need legacy midpoint TEST events from Cell 4."
)
assert "canonical_event_metrics_from_table" in globals(), (
    "Need canonical evaluator from Cell 19J."
)
assert "canonical_prepare_predictions" in globals(), (
    "Need canonical prediction preparation from Cell 19J."
)
assert "dataframe_fingerprint" in globals(), (
    "Need dataframe_fingerprint from Cell 19J."
)
assert "match_event_times_to_intervals" in globals(), (
    "Need interval matcher from Cell 4."
)

EVAL_DIR = Path(EVAL_DIR)
EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CELL22_VARIANT = (
    "cell22_v3_paper_ready_primary_secondary_bootstrap"
)

FINAL_TEST_METRICS_CSV = (
    EVAL_DIR
    / "final_test_metrics.csv"
)
FINAL_TEST_PER_SESSION_CSV = (
    EVAL_DIR
    / "final_test_metrics_per_session.csv"
)
FINAL_TEST_METRICS_JSON = (
    EVAL_DIR
    / "final_test_metrics.json"
)
FINAL_TEST_RUN_CARD_JSON = (
    EVAL_DIR
    / "final_test_run_card.json"
)
FINAL_TEST_PRED_USED_CSV = (
    EVAL_DIR
    / "final_test_predictions_used.csv"
)

BRANCH1_FINAL_TEST_TOLERANCE_CSV = (
    EVAL_DIR
    / "branch1_e2e_test_metrics_by_tolerance.csv"
)
BRANCH1_FINAL_TEST_PER_SESSION_CSV = (
    EVAL_DIR
    / "branch1_e2e_test_per_session_by_tolerance.csv"
)
BRANCH1_FINAL_TEST_TOLERANCE_JSON = (
    EVAL_DIR
    / "branch1_e2e_test_metrics_by_tolerance.json"
)

BRANCH1_SECONDARY_PROTOCOL_METRICS_CSV = (
    EVAL_DIR
    / "branch1_secondary_protocol_metrics.csv"
)
BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_CSV = (
    EVAL_DIR
    / "branch1_secondary_protocol_per_session.csv"
)
BRANCH1_BOOTSTRAP_CSV = (
    EVAL_DIR
    / "branch1_strict_session_bootstrap.csv"
)
BRANCH1_BOOTSTRAP_JSON = (
    EVAL_DIR
    / "branch1_strict_session_bootstrap.json"
)

BRANCH1_E2E_TOLERANCES = list(
    globals().get(
        "BRANCH1_E2E_TOLERANCES",
        [
            {
                "name": "strict",
                "label": "strict_0p50",
                "tol": 0.50,
                "description": (
                    "Strict event matching, ±0.50 s"
                ),
            },
            {
                "name": "dataset_calibrated",
                "label": (
                    "dataset_calibrated_0p70"
                ),
                "tol": 0.70,
                "description": (
                    "Dataset-calibrated event matching, ±0.70 s"
                ),
            },
            {
                "name": "relaxed",
                "label": "relaxed_1p00",
                "tol": 1.00,
                "description": (
                    "Relaxed event matching, ±1.00 s"
                ),
            },
        ],
    )
)

TEST_BOOTSTRAP_REPLICATES = int(
    globals().get(
        "TEST_BOOTSTRAP_REPLICATES",
        1000,
    )
)
TEST_BOOTSTRAP_SEED = int(
    globals().get(
        "TEST_BOOTSTRAP_SEED",
        SEED + 2200,
    )
)

def _safe(value):
    if isinstance(value, dict):
        return {
            str(key): _safe(item)
            for key, item
            in value.items()
        }
    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            _safe(item)
            for item in value
        ]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        numeric = float(value)
        return (
            numeric
            if np.isfinite(numeric)
            else None
        )
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, pd.DataFrame):
        return value.to_dict(
            "records"
        )
    if isinstance(value, pd.Series):
        return value.to_dict()
    return value

def _load_pred() -> pd.DataFrame:
    if (
        "TEST_PRED_DF" in globals()
        and isinstance(
            TEST_PRED_DF,
            pd.DataFrame,
        )
    ):
        return TEST_PRED_DF.copy()

    for variable_name in [
        "PRED_CSV",
        "TEST_PRED_CSV",
    ]:
        value = globals().get(
            variable_name,
            None,
        )
        if (
            value
            and Path(value).exists()
        ):
            return pd.read_csv(
                value
            )

    raise FileNotFoundError(
        "Run Cell 21 first."
    )

def _interval_metrics_from_table(
    prediction_frame: pd.DataFrame,
    event_frame: pd.DataFrame,
    tolerance_s: float,
) -> dict:
    prediction = (
        prediction_frame[
            [
                "session",
                "time",
            ]
        ]
        .copy()
    )
    prediction["session"] = (
        prediction["session"].astype(str)
    )
    prediction["time"] = pd.to_numeric(
        prediction["time"],
        errors="coerce",
    )
    prediction = prediction.dropna(
        subset=[
            "session",
            "time",
        ]
    )

    events = event_frame.copy()
    events["session"] = (
        events["session"].astype(str)
    )
    events["label"] = (
        pd.to_numeric(
            events["label"],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )
    events = events[
        events["label"].eq(1)
    ].copy()

    for column in [
        "transition_interval_start",
        "transition_interval_end",
    ]:
        events[column] = pd.to_numeric(
            events[column],
            errors="coerce",
        )

    events = events.dropna(
        subset=[
            "session",
            "transition_interval_start",
            "transition_interval_end",
        ]
    )

    sessions = sorted(
        set(
            prediction[
                "session"
            ].unique()
        )
        | set(
            events[
                "session"
            ].unique()
        )
    )

    totals = {
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "N_pred": 0,
        "N_gt": 0,
    }
    per_session_rows = []

    for session in sessions:
        predicted_times = prediction.loc[
            prediction["session"].eq(
                session
            ),
            "time",
        ].to_numpy(dtype=float)

        session_events = events[
            events["session"].eq(
                session
            )
        ]

        intervals = session_events[
            [
                "transition_interval_start",
                "transition_interval_end",
            ]
        ].to_numpy(dtype=float)

        metrics = match_event_times_to_intervals(
            predicted_times,
            intervals,
            tol=float(tolerance_s),
        )
        metrics.pop(
            "matches",
            None,
        )

        for key in totals:
            totals[key] += int(
                metrics.get(
                    key,
                    0,
                )
            )

        per_session_rows.append({
            "session": str(session),
            "tol": float(
                tolerance_s
            ),
            **{
                key: int(
                    metrics.get(
                        key,
                        0,
                    )
                )
                for key in totals
            },
        })

    precision = (
        totals["TP"]
        / max(
            1,
            totals["TP"]
            + totals["FP"],
        )
    )
    recall = (
        totals["TP"]
        / max(
            1,
            totals["TP"]
            + totals["FN"],
        )
    )
    f1 = (
        2.0
        * precision
        * recall
        / max(
            1e-12,
            precision + recall,
        )
    )

    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        **totals,
        "per_session": pd.DataFrame(
            per_session_rows
        ),
    }

def _bootstrap_session_metrics(
    strict_per_session: pd.DataFrame,
    replicates: int,
    seed: int,
) -> pd.DataFrame:
    required = {
        "session",
        "TP",
        "FP",
        "FN",
    }
    missing = sorted(
        required
        - set(
            strict_per_session.columns
        )
    )
    if missing:
        raise KeyError(
            "Strict per-session table is missing "
            f"bootstrap columns: {missing}"
        )

    frame = strict_per_session[
        [
            "session",
            "TP",
            "FP",
            "FN",
        ]
    ].copy()

    for column in [
        "TP",
        "FP",
        "FN",
    ]:
        frame[column] = (
            pd.to_numeric(
                frame[column],
                errors="coerce",
            )
            .fillna(0)
            .astype(int)
        )

    if len(frame) == 0:
        return pd.DataFrame(
            columns=[
                "replicate",
                "P",
                "R",
                "F1",
            ]
        )

    values = frame[
        [
            "TP",
            "FP",
            "FN",
        ]
    ].to_numpy(
        dtype=np.int64
    )

    rng = np.random.default_rng(
        int(seed)
    )
    rows = []

    for replicate in range(
        int(replicates)
    ):
        indices = rng.integers(
            0,
            len(values),
            size=len(values),
        )
        totals = values[
            indices
        ].sum(axis=0)

        true_positive = int(
            totals[0]
        )
        false_positive = int(
            totals[1]
        )
        false_negative = int(
            totals[2]
        )

        precision = (
            true_positive
            / max(
                1,
                true_positive
                + false_positive,
            )
        )
        recall = (
            true_positive
            / max(
                1,
                true_positive
                + false_negative,
            )
        )
        f1 = (
            2.0
            * precision
            * recall
            / max(
                1e-12,
                precision + recall,
            )
        )

        rows.append({
            "replicate": int(
                replicate
            ),
            "P": float(
                precision
            ),
            "R": float(
                recall
            ),
            "F1": float(f1),
            "TP": true_positive,
            "FP": false_positive,
            "FN": false_negative,
        })

    return pd.DataFrame(rows)

def _bootstrap_summary(
    bootstrap_frame: pd.DataFrame,
) -> dict:
    output = {
        "replicates": int(
            len(bootstrap_frame)
        ),
        "sampling_unit": (
            "TEST session with replacement"
        ),
        "confidence_level": 0.95,
        "test_not_used_for_selection": True,
    }

    for metric in [
        "P",
        "R",
        "F1",
    ]:
        values = pd.to_numeric(
            bootstrap_frame[metric],
            errors="coerce",
        ).dropna()

        output[metric] = {
            "mean": float(
                values.mean()
            ),
            "lower_95": float(
                values.quantile(
                    0.025
                )
            ),
            "upper_95": float(
                values.quantile(
                    0.975
                )
            ),
        }

    return output

TEST_PRED_EVAL_DF = canonical_prepare_predictions(
    _load_pred()
)
TEST_PRED_EVAL_DF = TEST_PRED_EVAL_DF.drop(
    columns=[
        "_pred_row_id",
    ],
    errors="ignore",
)
TEST_PRED_EVAL_DF.to_csv(
    FINAL_TEST_PRED_USED_CSV,
    index=False,
)

fingerprint_columns = [
    column
    for column in [
        "session",
        "time",
        "score",
    ]
    if column
    in TEST_PRED_EVAL_DF.columns
]
FINAL_TEST_PRED_FINGERPRINT = dataframe_fingerprint(
    TEST_PRED_EVAL_DF,
    fingerprint_columns,
)

lock = dict(
    globals().get(
        "FINAL_LOCKED",
        {},
    )
)

assert bool(
    lock.get(
        "test_not_used_for_selection",
        False,
    )
), (
    "The official final lock must be selected "
    "without TEST labels."
)
assert str(
    lock.get(
        "event_contract_version",
        EVENT_CONTRACT_VERSION,
    )
) == str(EVENT_CONTRACT_VERSION), (
    "The official lock and TEST labels use "
    "different event contracts."
)

primary_rows = []
primary_per_session_parts = []
secondary_rows = []
secondary_per_session_parts = []

for tolerance_config in BRANCH1_E2E_TOLERANCES:
    tolerance = float(
        tolerance_config["tol"]
    )
    tolerance_name = str(
        tolerance_config["name"]
    )
    tolerance_label = str(
        tolerance_config["label"]
    )

    primary_metrics = (
        canonical_event_metrics_from_table(
            TEST_PRED_EVAL_DF,
            REAL_TEST_EVENTS,
            tolerance,
            return_per_session=True,
        )
    )
    primary_per_session = (
        primary_metrics.pop(
            "per_session"
        )
    )
    primary_per_session[
        "tolerance_name"
    ] = tolerance_name
    primary_per_session[
        "tolerance_label"
    ] = tolerance_label
    primary_per_session[
        "protocol"
    ] = "primary_action_aware_point"
    primary_per_session_parts.append(
        primary_per_session
    )

    primary_row = {
        "branch": (
            "Branch 1: end-to-end SHIFT "
            "event detection"
        ),
        "protocol": (
            "primary_action_aware_point"
        ),
        "event_contract_version": str(
            EVENT_CONTRACT_VERSION
        ),
        "event_time_definition": str(
            PRIMARY_EVENT_TIME_DEFINITION
        ),
        "positive_gap_cap_s": float(
            PRIMARY_MAX_POSITIVE_GAP_S
        ),
        "tolerance_name": tolerance_name,
        "tolerance_label": tolerance_label,
        "tolerance_description": str(
            tolerance_config.get(
                "description",
                "",
            )
        ),
        "tol": tolerance,
        "score_mode": str(
            lock.get(
                "score_mode",
                "unknown",
            )
        ),
        "time_mode": str(
            lock.get(
                "time_mode",
                "raw",
            )
        ),
        "locked_stage": str(
            lock.get(
                "stage",
                lock.get(
                    "locked_on",
                    "unknown",
                ),
            )
        ),
        "selection_provenance": (
            "val_selected_official"
        ),
        "prediction_fingerprint": (
            FINAL_TEST_PRED_FINGERPRINT
        ),
        **primary_metrics,
    }

    for metric_name, target_name in [
        (
            "P",
            "TARGET_FINAL_PRECISION",
        ),
        (
            "R",
            "TARGET_FINAL_RECALL",
        ),
        (
            "F1",
            "TARGET_FINAL_F1",
        ),
    ]:
        target = float(
            globals().get(
                target_name,
                0.70,
            )
        )
        primary_row[
            f"{metric_name}_target"
        ] = target
        primary_row[
            f"{metric_name}_target_met"
        ] = bool(
            primary_row[metric_name]
            >= target
        )

    primary_rows.append(
        primary_row
    )

    legacy_metrics = (
        canonical_event_metrics_from_table(
            TEST_PRED_EVAL_DF,
            REAL_TEST_EVENTS_LEGACY,
            tolerance,
            return_per_session=True,
        )
    )
    legacy_per_session = (
        legacy_metrics.pop(
            "per_session"
        )
    )
    legacy_per_session[
        "tolerance_name"
    ] = tolerance_name
    legacy_per_session[
        "tolerance_label"
    ] = tolerance_label
    legacy_per_session[
        "protocol"
    ] = "secondary_legacy_midpoint"
    secondary_per_session_parts.append(
        legacy_per_session
    )

    secondary_rows.append({
        "protocol": (
            "secondary_legacy_midpoint"
        ),
        "event_contract_version": (
            "legacy_midpoint_no_positive_gap_cap"
        ),
        "event_time_definition": (
            "midpoint"
        ),
        "tolerance_name": tolerance_name,
        "tolerance_label": tolerance_label,
        "tol": tolerance,
        "selection_provenance": (
            "fixed_predictions_only"
        ),
        **legacy_metrics,
    })

    interval_metrics = (
        _interval_metrics_from_table(
            TEST_PRED_EVAL_DF,
            REAL_TEST_EVENTS,
            tolerance,
        )
    )
    interval_per_session = (
        interval_metrics.pop(
            "per_session"
        )
    )
    interval_per_session[
        "tolerance_name"
    ] = tolerance_name
    interval_per_session[
        "tolerance_label"
    ] = tolerance_label
    interval_per_session[
        "protocol"
    ] = (
        "secondary_transition_interval"
    )
    secondary_per_session_parts.append(
        interval_per_session
    )

    secondary_rows.append({
        "protocol": (
            "secondary_transition_interval"
        ),
        "event_contract_version": str(
            EVENT_CONTRACT_VERSION
        ),
        "event_time_definition": (
            "transition_interval"
        ),
        "tolerance_name": tolerance_name,
        "tolerance_label": tolerance_label,
        "tol": tolerance,
        "selection_provenance": (
            "fixed_predictions_only"
        ),
        **interval_metrics,
    })

BRANCH1_FINAL_TEST_TOLERANCE_DF = (
    pd.DataFrame(
        primary_rows
    )
)
BRANCH1_FINAL_TEST_PER_SESSION_DF = (
    pd.concat(
        primary_per_session_parts,
        ignore_index=True,
    )
)
BRANCH1_SECONDARY_PROTOCOL_METRICS_DF = (
    pd.DataFrame(
        secondary_rows
    )
)
BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_DF = (
    pd.concat(
        secondary_per_session_parts,
        ignore_index=True,
    )
)

BRANCH1_FINAL_TEST_TOLERANCE_DF.to_csv(
    BRANCH1_FINAL_TEST_TOLERANCE_CSV,
    index=False,
)
BRANCH1_FINAL_TEST_PER_SESSION_DF.to_csv(
    BRANCH1_FINAL_TEST_PER_SESSION_CSV,
    index=False,
)
BRANCH1_SECONDARY_PROTOCOL_METRICS_DF.to_csv(
    BRANCH1_SECONDARY_PROTOCOL_METRICS_CSV,
    index=False,
)
BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_DF.to_csv(
    BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_CSV,
    index=False,
)

strict = BRANCH1_FINAL_TEST_TOLERANCE_DF[
    BRANCH1_FINAL_TEST_TOLERANCE_DF[
        "tolerance_label"
    ].eq("strict_0p50")
]
FINAL_TEST_SUMMARY = (
    strict.iloc[0]
    if len(strict)
    else BRANCH1_FINAL_TEST_TOLERANCE_DF.iloc[0]
).to_dict()
FINAL_TEST_METRICS = dict(
    FINAL_TEST_SUMMARY
)

pd.DataFrame([
    FINAL_TEST_SUMMARY
]).to_csv(
    FINAL_TEST_METRICS_CSV,
    index=False,
)

strict_per_session = (
    BRANCH1_FINAL_TEST_PER_SESSION_DF[
        BRANCH1_FINAL_TEST_PER_SESSION_DF[
            "tolerance_label"
        ].eq(
            FINAL_TEST_SUMMARY[
                "tolerance_label"
            ]
        )
    ]
    .copy()
)
strict_per_session.to_csv(
    FINAL_TEST_PER_SESSION_CSV,
    index=False,
)

BRANCH1_STRICT_BOOTSTRAP_DF = (
    _bootstrap_session_metrics(
        strict_per_session,
        replicates=int(
            TEST_BOOTSTRAP_REPLICATES
        ),
        seed=int(
            TEST_BOOTSTRAP_SEED
        ),
    )
)
BRANCH1_STRICT_BOOTSTRAP_SUMMARY = (
    _bootstrap_summary(
        BRANCH1_STRICT_BOOTSTRAP_DF
    )
)

BRANCH1_STRICT_BOOTSTRAP_DF.to_csv(
    BRANCH1_BOOTSTRAP_CSV,
    index=False,
)
BRANCH1_BOOTSTRAP_JSON.write_text(
    json.dumps(
        _safe(
            BRANCH1_STRICT_BOOTSTRAP_SUMMARY
        ),
        indent=2,
        sort_keys=True,
    )
)

payload = {
    "cell22_variant": str(
        CELL22_VARIANT
    ),
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
    "matching": (
        "float64 one-to-one minimum-cost assignment"
    ),
    "prediction_fingerprint": (
        FINAL_TEST_PRED_FINGERPRINT
    ),
    "final_locked": lock,
    "primary_protocol": {
        "event_contract_version": str(
            EVENT_CONTRACT_VERSION
        ),
        "event_time_definition": str(
            PRIMARY_EVENT_TIME_DEFINITION
        ),
        "positive_gap_cap_s": float(
            PRIMARY_MAX_POSITIVE_GAP_S
        ),
        "selected_using_test": False,
    },
    "metrics_by_tolerance": (
        BRANCH1_FINAL_TEST_TOLERANCE_DF
        .to_dict("records")
    ),
    "secondary_protocol_metrics": (
        BRANCH1_SECONDARY_PROTOCOL_METRICS_DF
        .to_dict("records")
    ),
    "strict_session_bootstrap_95_ci": (
        BRANCH1_STRICT_BOOTSTRAP_SUMMARY
    ),
}

FINAL_TEST_METRICS_JSON.write_text(
    json.dumps(
        _safe(payload),
        indent=2,
        sort_keys=True,
    )
)
BRANCH1_FINAL_TEST_TOLERANCE_JSON.write_text(
    json.dumps(
        _safe(payload),
        indent=2,
        sort_keys=True,
    )
)
FINAL_TEST_RUN_CARD_JSON.write_text(
    json.dumps(
        _safe(payload),
        indent=2,
        sort_keys=True,
    )
)

FINAL_TEST_SHIFT_METRICS = {
    "precision": float(
        FINAL_TEST_SUMMARY["P"]
    ),
    "recall": float(
        FINAL_TEST_SUMMARY["R"]
    ),
    "f1": float(
        FINAL_TEST_SUMMARY["F1"]
    ),
    "tp": int(
        FINAL_TEST_SUMMARY["TP"]
    ),
    "fp": int(
        FINAL_TEST_SUMMARY["FP"]
    ),
    "fn": int(
        FINAL_TEST_SUMMARY["FN"]
    ),
    "n_pred": int(
        FINAL_TEST_SUMMARY["N_pred"]
    ),
    "n_gt": int(
        FINAL_TEST_SUMMARY["N_gt"]
    ),
    "tol": float(
        FINAL_TEST_SUMMARY["tol"]
    ),
    "event_contract_version": str(
        EVENT_CONTRACT_VERSION
    ),
    "event_time_definition": str(
        PRIMARY_EVENT_TIME_DEFINITION
    ),
}

TEST_SHIFT_METRICS = (
    FINAL_TEST_SHIFT_METRICS.copy()
)
EVAL_SHIFT_SUMMARY_DF = pd.DataFrame([
    FINAL_TEST_SUMMARY
])
EVAL_SHIFT_RES_ANY = {
    "micro": FINAL_TEST_SUMMARY,
    "per_session": strict_per_session,
}
EVAL_SHIFT_RESULTS = {
    "GT_any": EVAL_SHIFT_RES_ANY
}
BRANCH1_FINAL_TEST_METRICS_BY_TOLERANCE = {
    row["tolerance_label"]: row
    for row in primary_rows
}

for variable_name in [
    "TEST_PRED_EVAL_DF",
    "FINAL_TEST_PRED_FINGERPRINT",
    "BRANCH1_FINAL_TEST_TOLERANCE_DF",
    "BRANCH1_FINAL_TEST_PER_SESSION_DF",
    "BRANCH1_SECONDARY_PROTOCOL_METRICS_DF",
    "BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_DF",
    "BRANCH1_STRICT_BOOTSTRAP_DF",
    "BRANCH1_STRICT_BOOTSTRAP_SUMMARY",
    "FINAL_TEST_SUMMARY",
    "FINAL_TEST_METRICS",
    "FINAL_TEST_SHIFT_METRICS",
    "TEST_SHIFT_METRICS",
    "EVAL_SHIFT_SUMMARY_DF",
    "EVAL_SHIFT_RES_ANY",
    "EVAL_SHIFT_RESULTS",
    "BRANCH1_FINAL_TEST_METRICS_BY_TOLERANCE",
]:
    globals()[
        variable_name
    ] = locals()[
        variable_name
    ]

for variable_name in [
    "FINAL_TEST_METRICS_CSV",
    "FINAL_TEST_PER_SESSION_CSV",
    "FINAL_TEST_METRICS_JSON",
    "FINAL_TEST_RUN_CARD_JSON",
    "FINAL_TEST_PRED_USED_CSV",
    "BRANCH1_FINAL_TEST_TOLERANCE_CSV",
    "BRANCH1_FINAL_TEST_PER_SESSION_CSV",
    "BRANCH1_FINAL_TEST_TOLERANCE_JSON",
    "BRANCH1_SECONDARY_PROTOCOL_METRICS_CSV",
    "BRANCH1_SECONDARY_PROTOCOL_PER_SESSION_CSV",
    "BRANCH1_BOOTSTRAP_CSV",
    "BRANCH1_BOOTSTRAP_JSON",
]:
    globals()[
        variable_name
    ] = str(
        locals()[
            variable_name
        ]
    )

print("=" * 80)
print(
    "FINAL TEST METRICS — PRIMARY PAPER-READY PROTOCOL"
)
print("=" * 80)
display(
    BRANCH1_FINAL_TEST_TOLERANCE_DF[
        [
            "tolerance_name",
            "tol",
            "P",
            "R",
            "F1",
            "TP",
            "FP",
            "FN",
            "N_pred",
            "N_gt",
            "score_mode",
            "time_mode",
            "selection_provenance",
        ]
    ]
)

print(
    "\nSecondary fixed-prediction protocol report:"
)
display(
    BRANCH1_SECONDARY_PROTOCOL_METRICS_DF[
        [
            "protocol",
            "tolerance_name",
            "tol",
            "P",
            "R",
            "F1",
            "TP",
            "FP",
            "FN",
        ]
    ]
)

print(
    "\nStrict session-bootstrap 95% intervals:"
)
display(
    pd.DataFrame(
        BRANCH1_STRICT_BOOTSTRAP_SUMMARY
    ).T
)

print(
    "Prediction fingerprint:",
    FINAL_TEST_PRED_FINGERPRINT,
)
print(
    "Cell 22 complete: TEST was evaluated once under the locked VAL-selected "
    "primary contract; legacy and interval metrics are secondary reports only."
)

FINAL TEST METRICS — PRIMARY PAPER-READY PROTOCOL


,tolerance_name,tol,P,R,F1,TP,FP,FN,N_pred,N_gt,score_mode,time_mode,selection_provenance
0,strict,0.5,0.543354,0.555766,0.549490,3528,2965,2820,6493,6348,weighted_binary_temporal,local_corrected,val_selected_official
1,dataset_calibrated,0.7,0.583552,0.596881,0.590141,3789,2704,2559,6493,6348,weighted_binary_temporal,local_corrected,val_selected_official
2,relaxed,1.0,0.601571,0.615312,0.608364,3906,2587,2442,6493,6348,weighted_binary_temporal,local_corrected,val_selected_official



Secondary fixed-prediction protocol report:


,protocol,tolerance_name,tol,P,R,F1,TP,FP,FN
0,secondary_legacy_midpoint,strict,0.5,0.572463,0.441449,0.498491,3717,2776,4703
1,secondary_transition_interval,strict,0.5,0.574619,0.587559,0.581017,3731,2762,2619
2,secondary_legacy_midpoint,dataset_calibrated,0.7,0.628369,0.484561,0.547174,4080,2413,4340
3,secondary_transition_interval,dataset_calibrated,0.7,0.594640,0.608031,0.601261,3861,2632,2489
4,secondary_legacy_midpoint,relaxed,1.0,0.661790,0.510333,0.576276,4297,2196,4123
5,secondary_transition_interval,relaxed,1.0,0.606961,0.620630,0.613720,3941,2552,2409



Strict session-bootstrap 95% intervals:


,mean,lower_95,upper_95
replicates,1000,1000,1000
sampling_unit,TEST session with replacement,TEST session with replacement,TEST session with replacement
confidence_level,0.95,0.95,0.95
test_not_used_for_selection,True,True,True
P,0.542639,0.516131,0.568226
R,0.555057,0.527313,0.585163
F1,0.548684,0.524105,0.572461


Prediction fingerprint: 072d2042af2c95a1a68da4c3651e627e0837c98edfac03b0d8c4c7fa8477217a
Cell 22 complete: TEST was evaluated once under the locked VAL-selected primary contract; legacy and interval metrics are secondary reports only.


In [34]:
# Augmentation ablation completion marker
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

assert RUN_CLEAN_AUG_ABLATION is True
assert AUG_ABLATION_CONDITION in AUG_ABLATION_CONDITIONS
assert str(EVENT_CONTRACT_VERSION) == "voxconverse_action_aware_immediate_v1"
assert "FINAL_TEST_SUMMARY" in globals()
assert "FINAL_LOCKED" in globals()
assert "BRANCH1_STRICT_BOOTSTRAP_SUMMARY" in globals()

def _aug_safe(x):
    if isinstance(x, dict):
        return {str(k): _aug_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_aug_safe(v) for v in x]
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        y = float(x)
        return y if np.isfinite(y) else None
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, pd.Series):
        return _aug_safe(x.to_dict())
    return x

expected_flags = AUG_ABLATION_CONDITIONS[AUG_ABLATION_CONDITION]
assert bool(USE_SYNTH_MULTIPARTY) == bool(expected_flags["use_synthetic_multiparty"])
assert bool(USE_DIFFUSION_AUG) == bool(expected_flags["use_diffusion_aug"])

mix_rows = []
if "TRAIN_EVENTS_MIXED" in globals() and isinstance(TRAIN_EVENTS_MIXED, pd.DataFrame):
    mix = TRAIN_EVENTS_MIXED.copy()
    if len(mix):
        for col, default in [
            ("is_synthetic", False),
            ("is_diffusion_augmented", False),
            ("label", 0),
            ("sample_weight", 1.0),
        ]:
            if col not in mix.columns:
                mix[col] = default

        grouped = (
            mix.groupby(
                ["is_synthetic", "is_diffusion_augmented"],
                dropna=False,
            )
            .agg(
                rows=("session", "size"),
                shifts=("label", "sum"),
                weighted_rows=("sample_weight", "sum"),
            )
            .reset_index()
        )
        mix_rows = grouped.to_dict("records")

strict = dict(FINAL_TEST_SUMMARY)

payload = {
    "status": "complete",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "condition": str(AUG_ABLATION_CONDITION),
    "run_version": str(RUN_VERSION),
    "base_run_version": str(BASE_RUN_VERSION),
    "event_contract_version": str(EVENT_CONTRACT_VERSION),
    "primary_event_time_definition": str(PRIMARY_EVENT_TIME_DEFINITION),
    "primary_min_gap_s": float(PRIMARY_MIN_GAP_S),
    "primary_max_positive_gap_s": float(PRIMARY_MAX_POSITIVE_GAP_S),
    "action_short_turn_s": float(ACTION_SHORT_TURN_S),
    "action_overlap_min_s": float(ACTION_OVERLAP_MIN_S),
    "use_synthetic_multiparty": bool(USE_SYNTH_MULTIPARTY),
    "use_diffusion_aug": bool(USE_DIFFUSION_AUG),
    "test_used_for_selection": False,
    "selection_partition": "VAL",
    "strict_test_metrics": {
        "P": float(strict["P"]),
        "R": float(strict["R"]),
        "F1": float(strict["F1"]),
        "TP": int(strict["TP"]),
        "FP": int(strict["FP"]),
        "FN": int(strict["FN"]),
        "N_pred": int(strict["N_pred"]),
        "N_gt": int(strict["N_gt"]),
        "tol": float(strict["tol"]),
        "tolerance_label": str(strict["tolerance_label"]),
        "event_contract_version": str(EVENT_CONTRACT_VERSION),
    },
    "strict_session_bootstrap_95_ci": BRANCH1_STRICT_BOOTSTRAP_SUMMARY,
    "final_locked": FINAL_LOCKED,
    "sys1_locked": globals().get("SYS1_LOCKED", {}),
    "train_mix": mix_rows,
    "final_test_metrics_json": str(globals().get("FINAL_TEST_METRICS_JSON", "")),
    "final_test_per_session_csv": str(globals().get("FINAL_TEST_PER_SESSION_CSV", "")),
}

completion_path = Path(RUN_ROOT) / "augmentation_ablation_condition_complete.json"
completion_path.write_text(
    json.dumps(_aug_safe(payload), indent=2, sort_keys=True)
)

print("=" * 80)
print("AUGMENTATION ABLATION CONDITION COMPLETE")
print("=" * 80)
print("Condition:", AUG_ABLATION_CONDITION)
print("Run:", RUN_VERSION)
print("Contract:", EVENT_CONTRACT_VERSION)
print(
    "Strict TEST:",
    {
        "P": round(float(strict["P"]), 4),
        "R": round(float(strict["R"]), 4),
        "F1": round(float(strict["F1"]), 4),
        "TP": int(strict["TP"]),
        "FP": int(strict["FP"]),
        "FN": int(strict["FN"]),
    },
)
print("Saved completion marker:", completion_path)

AUGMENTATION ABLATION CONDITION COMPLETE
Condition: synthetic_only
Run: 18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only
Contract: voxconverse_action_aware_immediate_v1
Strict TEST: {'P': 0.5434, 'R': 0.5558, 'F1': 0.5495, 'TP': 3528, 'FP': 2965, 'FN': 2820}
Saved completion marker: <REPOSITORY_ROOT>/Experiment_1/voxconverse-dataset/runs/18thAug_VAP_RRFAP_TEXT_META_CONFIRMER_PAPER_READY_v2__aug_ablation_synthetic_only/augmentation_ablation_condition_complete.json
